In [ ]:
import scanpy as sc 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import pandas as pd
import os
import sklearn

In [ ]:
root = "/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/"

# import reference
adata_ref = sc.read_h5ad('/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/JK_Anal_ysis/haber_ref.h5ad')
# Correct order: replace longer/more specific term first
adata_ref.obs['cell_type_broad'] = (
    adata_ref.obs['cell_type_broad']
    .str.replace("Enteroendocrine", "EEC")
    .str.replace("Endocrine", "EEC")
)

In [ ]:
##############################################
# IMPORTS & PATHS
##############################################
import pandas as pd
import scanpy as sc
import numpy as np

##############################################
# LOAD RAW MATRICES
##############################################
dcm1 = pd.read_csv(f"{root}/Pool35_TSS_gene_DCM_matrix.csv", index_col=0).T
rna1 = pd.read_csv(f"{root}/Pool35_RNA_matrix.csv", index_col=0).T
meta1 = pd.read_csv(f"{root}/metadata_Pool35_DCM_RNA.csv")

dcm2 = pd.read_csv(f"{root}/Pool40_TSS_gene_DCM_matrix.csv", index_col=0).T
rna2 = pd.read_csv(f"{root}/Pool40_RNA_matrix.csv", index_col=0).T
meta2 = pd.read_csv(f"{root}/metadata_Pool40_DCM_RNA.csv")

##############################################
# PREPARE METADATA BEFORE FILTERING
##############################################
def prepare(meta, pool_name):
    m = meta.copy()

    # Keep rows with plate + well
    m = m[m["Plate"].notna() & m["Well"].notna()].copy()

    m["pool"] = pool_name
    m["plate_well"] = m["Plate"].astype(str) + "_" + m["Well"].astype(str)

    return m

m1 = prepare(meta1, "pool35")
m2 = prepare(meta2, "pool40")

meta_all = pd.concat([m1, m2], axis=0)

##############################################
# CHECK RAW DUPLICATION PER POOL/WELL
##############################################
dups = (
    meta_all
    .groupby(["pool", "plate_well"])
    .agg(
        dcm_count=("DCM_matrixID", lambda x: x.notna().sum()),
        rna_count=("Sample_RNA", lambda x: x.notna().sum())
    )
)

print("RAW duplicates (before Scanpy filtering):")
print(dups[(dups.dcm_count > 1) | (dups.rna_count > 1)])

##############################################
# BUILD RAW LINK TABLE (BEFORE ANY FILTERING)
##############################################
links = meta_all[["pool", "plate_well", "DCM_matrixID", "Sample_RNA"]].copy()

##############################################
# ALIGN GENES
##############################################
dcm_genes = sorted(set(dcm1.columns).union(dcm2.columns))
dcm1 = dcm1.reindex(columns=dcm_genes, fill_value=0)
dcm2 = dcm2.reindex(columns=dcm_genes, fill_value=0)

rna_genes = sorted(set(rna1.columns).union(rna2.columns))
rna1 = rna1.reindex(columns=rna_genes, fill_value=0)
rna2 = rna2.reindex(columns=rna_genes, fill_value=0)

##############################################
# CREATE AnnData OBJECTS
##############################################
adata_dcm1 = sc.AnnData(X=dcm1)
adata_dcm2 = sc.AnnData(X=dcm2)
adata_rna1 = sc.AnnData(X=rna1)
adata_rna2 = sc.AnnData(X=rna2)

##############################################
# FILTER CELLS
##############################################
for adata in [adata_dcm1, adata_dcm2, adata_rna1, adata_rna2]:
    sc.pp.filter_cells(adata, min_genes=200)

##############################################
# ATTACH METADATA AFTER FILTERING
##############################################
def attach_meta(adata, meta, id_col):
    m = meta.set_index(id_col)
    keep = m.index.intersection(adata.obs_names)
    adata = adata[keep].copy()
    adata.obs = adata.obs.join(m.loc[keep])
    return adata

adata_dcm1 = attach_meta(adata_dcm1, m1, "DCM_matrixID")
adata_dcm2 = attach_meta(adata_dcm2, m2, "DCM_matrixID")
adata_rna1 = attach_meta(adata_rna1, m1, "Sample_RNA")
adata_rna2 = attach_meta(adata_rna2, m2, "Sample_RNA")

# For adata_dcm
for ad in [adata_dcm1, adata_dcm2]:
    sc.pp.normalize_total(ad, target_sum=1e4)
    sc.pp.log1p(ad)
    ad.layers['lognorm'] = ad.X.copy()
    sc.pp.scale(ad)
    ad.layers['lognorm_scaled'] = ad.X

# For adata_rna
for ad in [adata_rna1, adata_rna2]:
    sc.pp.normalize_total(ad, target_sum=1e4)
    sc.pp.log1p(ad)
    ad.layers['lognorm'] = ad.X.copy()
    sc.pp.scale(ad)
    ad.layers['lognorm_scaled'] = ad.X

##############################################
# CONCATENATE POOLS
##############################################
adata_dcm = adata_dcm1.concatenate(
    adata_dcm2,
    batch_key="pool",
    batch_categories=["pool35", "pool40"]
)

adata_rna = adata_rna1.concatenate(
    adata_rna2,
    batch_key="pool",
    batch_categories=["pool35", "pool40"]
)

##############################################
# MATCH CELLS THAT SURVIVED FILTERING
##############################################
valid_dcm = adata_dcm.obs[["pool", "plate_well", "Dox"]].copy()
valid_rna = adata_rna.obs[["pool", "plate_well", "Dox"]].copy()

# join pool + plate_well
matched = (
    valid_dcm.reset_index()
    .merge(
        valid_rna.reset_index(),
        on=["pool", "plate_well"],
        suffixes=("_dcm", "_rna"),
        how="inner"
    )
)

# rename obs names to match plotting code
matched = matched.rename(columns={
    'index': 'obs_name_dcm',
    'index_rna': 'obs_name_rna',
    'Dox_dcm': 'Dox_dcm',
    'Dox_rna': 'Dox_rna'
})

print("Matched DCM→RNA pairs after filtering:", len(matched))
print(matched.head())



In [ ]:
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 8
plt.rcParams['axes.linewidth'] = 1
plt.rcParams['xtick.major.width'] = 1
plt.rcParams['ytick.major.width'] = 1
plt.rcParams['axes.grid'] = False
sns.set_style("white", {'axes.grid': False})
# add orig.ident to dcm and rna scnmt
adata_dcm.obs['orig.ident'] = 'scNMT' + '_' + adata_dcm.obs.pool.astype(str) + '_DCM'
adata_rna.obs['orig.ident'] = 'scNMT' + '_' + adata_rna.obs.pool.astype(str) + '_RNA'

# -----------------------------
# FILTER DCM FOR PLUS DOX
# -----------------------------
adata_dcm_plus = adata_dcm[adata_dcm.obs.Dox != 'Minus'].copy()

# -----------------------------
# COMBINE WITH RNA AND REFERENCE
# -----------------------------
print("\n🔹 COMBINING DCM+RNA+REFERENCE")

# Columns to preserve from all datasets
all_obs_cols = list(set().union(*[adata.obs.columns for adata in [adata_dcm_plus, adata_rna, adata_ref]]))

# Ensure all datasets have the same obs columns (fill missing with placeholder)
for ad in [adata_dcm_plus, adata_rna, adata_ref]:
    for col in all_obs_cols:
        if col not in ad.obs.columns:
            ad.obs[col] = 'Unknown'

# Perform inner join on genes but keep all obs columns
adata_combined = sc.concat(
    [adata_dcm_plus, adata_rna, adata_ref],
    join='inner',
    merge='unique',
    index_unique=None
)

# Convert selected obs columns to string and fill missing
for col in ['Dox', 'Sorted_population', 'cell_type_broad', 'celltype', 'leiden', 'orig.ident']:
    if col in adata_combined.obs.columns:
        adata_combined.obs[col] = adata_combined.obs[col].fillna('Unknown').astype(str)

# Optional: ensure categorical-like columns are string to avoid Categorical issues
for col in ['celltype', 'Sorted_population', 'Dox', 'orig.ident']:
    if col in adata_combined.obs.columns:
        adata_combined.obs[col] = adata_combined.obs[col].astype('string')

# Composite column for sorting/labels
adata_combined.obs['sort_assay_dox'] = (
    adata_combined.obs['Sorted_population'].fillna('').astype(str) + '_' +
    adata_combined.obs['orig.ident'].fillna('').astype(str) + '_' +
    adata_combined.obs['Dox'].fillna('').astype(str)
)

# -----------------------------
# PARAMETERS
# -----------------------------
sc.settings.figdir = '/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/integration2/'
sc.set_figure_params(dpi=300, dpi_save=300, vector_friendly=True, fontsize=6)

outdir = '/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/integration2/'

dimensions = [15, 20, 25, 30, 35, 45, 50, 40]
n_top_genes = 2000
neighbors_within_batch = 1
pynndescent_n_neighbors = 3

# -----------------------------
# HVG & PCA
# -----------------------------
if 'orig.ident' not in adata_combined.obs.columns:
    raise ValueError("orig.ident missing in adata_combined.obs; needed for batch-aware HVG")

if 'lognorm' not in adata_combined.layers:
    raise ValueError("adata_combined.layers must contain 'lognorm' for HVG selection")

adata_combined.X = adata_combined.layers['lognorm']

sc.pp.highly_variable_genes(
    adata_combined,
    n_top_genes=n_top_genes,
    batch_key='orig.ident'
)
print("Number of HVG selected:", int(adata_combined.var['highly_variable'].sum()))

adata_combined.X = adata_combined.layers['lognorm_scaled']
sc.tl.pca(adata_combined)

# prepare storage for labels
if 'cell_labels_across_dims' not in adata_combined.uns:
    adata_combined.uns['cell_labels_across_dims'] = pd.DataFrame(index=adata_combined.obs_names)

# -----------------------------
# LOOP OVER DIMENSIONS
# -----------------------------
for dim in dimensions:
    print(f"\n--- Running integration for dim={dim} ---")

    # -----------------------------
    # BBKNN (batch-aware neighbors)
    # -----------------------------
    try:
        sc.external.pp.bbknn(
            adata_combined,
            batch_key='orig.ident',
            n_pcs=dim,
            neighbors_within_batch=neighbors_within_batch,
            pynndescent_n_neighbors=pynndescent_n_neighbors
        )
    except Exception as e:
        print("BBKNN failed:", e)
        print("Falling back to standard neighbors()")
        sc.pp.neighbors(adata_combined, n_pcs=dim)

    # -----------------------------
    # UMAP & Leiden
    # -----------------------------
    sc.tl.umap(adata_combined)
    sc.tl.leiden(adata_combined, resolution=2, key_added='leiden', random_state=0)
# -----------------------------
    # HEATMAP: Leiden vs cell_type_broad
    # -----------------------------

    if 'cell_type_broad' in adata_combined.obs.columns:
        # Create contingency table (cell counts)
        contingency = pd.crosstab(
            adata_combined.obs['leiden'].astype(str),
            adata_combined.obs['cell_type_broad'].astype(str),
            margins=True,
            margins_name='Total'
        )
        
        # Sort index numerically where possible
        idx_order = sorted(
            [i for i in contingency.index if i != 'Total'],
            key=lambda x: int(x) if x.isdigit() else float('inf')
        ) + ['Total']
        contingency = contingency.reindex(idx_order)
        
        # Create heatmap with cell counts
        fig, ax = plt.subplots(figsize=(3, max(2, len(contingency) * 0.2)))
        
        # Plot heatmap (exclude Total row/col for color scaling)
        heatmap_data = contingency.iloc[:-1, :-1]  # Exclude margins for heatmap colors
        heatmap_data = heatmap_data.drop(columns='Unknown', errors='ignore')
        sns.heatmap(
            heatmap_data,
            annot=True,
            fmt='d',
            cmap='viridis',
            ax=ax,
            cbar_kws={'label': 'Cell Count'},
            linewidths=0,
            linecolor='white'
        )
        
        ax.set_xlabel('Reference Cell Type (cell_type_broad)', fontsize=10)
        ax.set_ylabel('Leiden Cluster', fontsize=10)
        ax.set_title(f'Leiden Clusters vs Reference Cell Types (dim={dim})\nCell Counts per Cluster-CellType Combination', fontsize=12)
        
        plt.tight_layout()
        fig.savefig(f'{outdir}heatmap_leiden_vs_celltype_dim{dim}.pdf', format='pdf', dpi=300, bbox_inches='tight')
        plt.close(fig)
        
        # Also create a normalized version (proportion within each cluster)
        fig2, ax2 = plt.subplots(figsize=(2, max(2, len(contingency) * 0.2)))
        
        # Normalize by row (cluster) to show proportion of each cell type within cluster
        heatmap_norm = heatmap_data.div(heatmap_data.sum(axis=1), axis=0) * 100
        
        sns.heatmap(
            heatmap_norm,
            annot=True,
            fmt='.1f',
            cmap='viridis',
            ax=ax2,
            cbar_kws={'label': 'Percentage (%)'},
            linewidths=0,
            linecolor='white'
        )
        
        ax2.set_xlabel('Reference Cell Type (cell_type_broad)', fontsize=10)
        ax2.set_ylabel('Leiden Cluster', fontsize=10)
        ax2.set_title(f'Leiden Clusters vs Reference Cell Types (dim={dim})\nPercentage of Each Cell Type per Cluster', fontsize=12)
        
        plt.tight_layout()
        fig2.savefig(f'{outdir}heatmap_leiden_vs_celltype_normalized_dim{dim}.pdf', format='pdf', dpi=300, bbox_inches='tight')
        plt.close(fig2)
        
        # Save the contingency table as CSV for reference
        contingency.to_csv(f'{outdir}contingency_leiden_vs_celltype_dim{dim}.csv')
        
        print(f"Saved heatmaps for dim={dim}")
    # -----------------------------
    # AUTOMATIC ANNOTATION (skip Unknown)
    # -----------------------------
    annot_col = f'leiden_annot_{dim}'
    cluster_annotations = {}
    
    for cluster in sorted(adata_combined.obs['leiden'].astype(str).unique(),
                          key=lambda x: int(x) if x.isdigit() else x):
        cluster_cells = adata_combined.obs[adata_combined.obs['leiden'].astype(str) == str(cluster)]
        
        if 'cell_type_broad' in cluster_cells.columns:
            counts = cluster_cells['cell_type_broad'].value_counts()
            
            if len(counts) > 0:
                # Filter out "Unknown" and similar values
                counts_filtered = counts[~counts.index.isin(['Unknown', 'unknown', 'NA', 'nan', ''])]
                
                if len(counts_filtered) > 0:
                    majority = counts_filtered.idxmax()
                else:
                    majority = 'Unknown'
            else:
                majority = 'Unknown'
        else:
            majority = 'Unknown'
        
        cluster_annotations[str(cluster)] = majority

    adata_combined.obs[annot_col] = (
        adata_combined.obs['leiden'].astype(str).map(cluster_annotations).fillna('Unknown')
    )
    adata_combined.obs[annot_col] = adata_combined.obs['leiden'].astype(str) + '_' + adata_combined.obs[annot_col]

    # -----------------------------
    # CREATE CLEAN ANNOTATION (without numbers)
    # -----------------------------
    annot_col_clean = f'cell_type_broad_{dim}'
    adata_combined.obs[annot_col_clean] = (
        adata_combined.obs[annot_col]
        .astype(str)
        .str.replace(r'^\d+_', '', regex=True)
    )

    # store for label consistency across dims
    if 'cell_labels_across_dims' not in adata_combined.uns:
        adata_combined.uns['cell_labels_across_dims'] = {}
    adata_combined.uns['cell_labels_across_dims'][annot_col] = adata_combined.obs[annot_col]
    adata_combined.uns['cell_labels_across_dims']['batch'] = adata_combined.obs['orig.ident']

    # -----------------------------
    # UMAP PLOT (with cluster numbers) - SVG
    # -----------------------------
    plot_cols = [c for c in ['Dox', 'Sorted_population', 'cell_type_broad', annot_col, 'leiden'] if c in adata_combined.obs.columns]
    sc.pl.umap(
        adata_combined,
        color=plot_cols,
        frameon=False,
        legend_loc='on data',
        save=f'{dim}_umap.svg'
    )

    # -----------------------------
    # UMAP PLOT (clean - without numbers) - SVG
    # -----------------------------
    plot_cols_clean = [c for c in ['Dox', 'Sorted_population', 'cell_type_broad', annot_col_clean] if c in adata_combined.obs.columns]
    sc.pl.umap(
        adata_combined,
        color=plot_cols_clean,
        frameon=False,
        legend_loc='on data',
        save=f'{dim}_umap_clean.svg'
    )

    # -----------------------------
    # DCM→RNA lines on UMAP - SVG
    # -----------------------------
    required_cols = {'index_dcm', 'obs_name_rna', 'Dox_dcm'}
    if required_cols.issubset(matched.columns) and len(matched) > 0:
        umap_coords = adata_combined.obsm['X_umap'][:, :2]

        for dox in matched['Dox_dcm'].unique():
            msub = matched[matched['Dox_dcm'] == dox]
            if len(msub) == 0:
                continue

            dcm_idx = [adata_combined.obs_names.get_loc(c) for c in msub['index_dcm'] if c in adata_combined.obs_names]
            rna_idx = [adata_combined.obs_names.get_loc(c) for c in msub['obs_name_rna'] if c in adata_combined.obs_names]

            if len(dcm_idx) == 0 or len(rna_idx) == 0:
                print(f"No valid matches for Dox {dox} (dim {dim}); skipping UMAP lines")
                continue

            fig, ax = plt.subplots(figsize=(8, 6))
            ax.scatter(umap_coords[:, 0], umap_coords[:, 1], c='lightgray', s=10, alpha=0.4)
            ax.scatter(umap_coords[dcm_idx, 0], umap_coords[dcm_idx, 1], c='red', s=20, label='DCM matched')
            ax.scatter(umap_coords[rna_idx, 0], umap_coords[rna_idx, 1], c='blue', s=20, label='RNA matched')
            for i in range(min(len(dcm_idx), len(rna_idx))):
                ax.plot(
                    [umap_coords[dcm_idx[i], 0], umap_coords[rna_idx[i], 0]],
                    [umap_coords[dcm_idx[i], 1], umap_coords[rna_idx[i], 1]],
                    c='gray', lw=0.5, alpha=0.5
                )
            ax.set_title(f'DCM→RNA connections - Dox {dox} (dim {dim})')
            ax.set_xlabel('UMAP1')
            ax.set_ylabel('UMAP2')
            ax.legend()
            plt.tight_layout()
            fig.savefig(f'{outdir}{dox}_{dim}_DCMtoRNA_umap.svg', format='svg', dpi=300, bbox_inches='tight')
            plt.show()
            plt.close(fig)
    else:
        print(f"Skipping DCM→RNA UMAP lines: matched table missing required columns or empty")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Run PCA if not already done
if 'pca' not in adata_combined.obsm:
    sc.tl.pca(adata_combined, svd_solver='arpack')

# Explained variance ratio
var_ratio = adata_combined.uns['pca']['variance_ratio']
cum_var = np.cumsum(var_ratio)

# Plot
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(cum_var) + 1), cum_var, marker='o')
plt.axhline(0.8, linestyle='--', linewidth=1)   # 80% variance (optional)
plt.axhline(0.9, linestyle='--', linewidth=1)   # 90% variance (optional)

plt.xlabel('Number of PCA dimensions')
plt.ylabel('Cumulative explained variance')
plt.title('Elbow Plot for PCA Dimensions')
plt.tight_layout()
plt.show()


In [ ]:
#Switches per dimension

In [ ]:
adata_combined.obs['orig.ident'].unique().tolist()

In [ ]:
adata_combined.obs['assay'] = adata_combined.obs['orig.ident'].map({
    'scNMT_pool35_DCM': 'pDCM',
    'scNMT_pool40_DCM': 'pDCM',
    'scNMT_pool35_RNA': 'pRNA',
    'scNMT_pool40_RNA': 'pRNA',
    'B1': 'Ref',
    'B2': 'Ref',
    'B3': 'Ref',
    'B4': 'Ref',
    'B5': 'Ref',
    'B6': 'Ref',
    'B7': 'Ref',
    'B8': 'Ref',
    'B9': 'Ref',
    'B10': 'Ref',
    'RNA_Harber': 'Ref'
})

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. LOAD DATA (if not already loaded)
# ============================================================
# adata_combined = sc.read_h5ad('reference.h5ad')

# ============================================================
# 2. SEPARATE DCM AND RNA CELLS
# ============================================================

# Identify DCM and RNA cells based on batch or assay
# Adjust this based on your actual column names
dcm_mask = adata_combined.obs['assay'].str.contains('DCM', case=False, na=False)
rna_mask = adata_combined.obs['assay'].str.contains('RNA', case=False, na=False)

adata_dcm2 = adata_combined[dcm_mask].copy()
adata_rna2 = adata_combined[rna_mask].copy()

print(f"DCM cells: {adata_dcm2.n_obs}")
print(f"RNA cells: {adata_rna2.n_obs}")

# ============================================================
# 3. MATCH DCM AND RNA CELLS
# ============================================================

# DCM cells: use obs_names
# RNA cells: match using Sample_RNA column
# Create a mapping from Sample_RNA to RNA cell annotations

# Get leiden_annot columns
leiden_cols = [col for col in adata_combined.obs.columns if col.startswith('leiden_annot_')]
print(f"Leiden annotation columns: {leiden_cols}")

# Create RNA lookup dictionary: Sample_RNA -> cell annotations
rna_df = adata_rna2.obs.copy()
rna_df['idpool'] = rna_df.index
rna_df.index = rna_df.index.str.replace("-pool35", "").str.replace("-pool40", "")
rna_df['cell_id'] = rna_df.index
rna_df['Sample_RNA'] = rna_df.index

# Create DCM dataframe
dcm_df = adata_dcm2.obs.copy()
dcm_df['cell_id'] = dcm_df['Sample_RNA']
dcm_df.index = dcm_df['Sample_RNA']

# Match: DCM obs_names should match RNA Sample_RNA
# First, check what format the matching should be
print("\nDCM obs_names examples:")
print(dcm_df.index[:5].tolist())
print("\nRNA Sample_RNA examples:")
print(rna_df['Sample_RNA'].head().tolist())

# Create matched dataframe
# RNA cells indexed by Sample_RNA for matching
rna_lookup = rna_df.set_index('Sample_RNA')

In [ ]:


# Find matching cells
matched_dcm_ids = []
matched_rna_ids = []

for dcm_id in dcm_df.index:
    if dcm_id in rna_lookup.index:
        matched_dcm_ids.append(dcm_id)
        matched_rna_ids.append(dcm_id)

print(f"\nMatched cells: {len(matched_dcm_ids)}")

# ============================================================
# 4. CREATE MATCHED DATAFRAME WITH TRANSITIONS
# ============================================================

# Get DCM annotations for matched cells
dcm_matched = dcm_df.loc[matched_dcm_ids].copy()
dcm_matched = dcm_matched.add_suffix('_DCM')

# Get RNA annotations for matched cells
rna_matched = rna_lookup.loc[matched_rna_ids].copy()
rna_matched = rna_matched.add_suffix('_RNA')
rna_matched.index = matched_dcm_ids  # Align indices

# Combine
matched_df = pd.concat([dcm_matched, rna_matched], axis=1)

print(f"Matched dataframe shape: {matched_df.shape}")
# ============================================================
# CLEAN CELL TYPE ANNOTATIONS: Remove number prefix (0_RGCs -> RGCs)
# ============================================================

# Get all leiden_annot columns (both DCM and RNA)
leiden_cols_all = [col for col in matched_df.columns if 'leiden_annot_' in col]

# Remove number prefix from cell type annotations
for col in leiden_cols_all:
    matched_df[col] = matched_df[col].str.replace(r'^\d+_', '', regex=True)

print("Cell type annotations cleaned (removed number prefix)")
print(f"Example values: {matched_df[leiden_cols_all[0]].unique()[:5]}")

print(f"Matched dataframe shape: {matched_df.shape}")
# ============================================================
# 5. CALCULATE TRANSITION PERCENTAGES PER DOX CONDITION
# ============================================================

def calculate_transitions(df, leiden_col, dox_col='Dox_DCM'):
    """
    Calculate transition percentages from DCM to RNA cell types per Dox condition.
    """
    dcm_col = f'{leiden_col}_DCM'
    rna_col = f'{leiden_col}_RNA'
    
    if dcm_col not in df.columns or rna_col not in df.columns:
        print(f"Columns {dcm_col} or {rna_col} not found")
        return None
    
    results = []
    
    for Dox in df[dox_col].unique():
        dox_df = df[df[dox_col] == Dox]
        
        # Count transitions
        transition_counts = dox_df.groupby([dcm_col, rna_col]).size().unstack(fill_value=0)
        
        # Calculate percentages (row-wise: from DCM state)
        transition_pct = transition_counts.div(transition_counts.sum(axis=1), axis=0) * 100
        
        # Store results
        for dcm_state in transition_pct.index:
            for rna_state in transition_pct.columns:
                results.append({
                    'Dox': Dox,
                    'leiden_dim': leiden_col,
                    'DCM_celltype': dcm_state,
                    'RNA_celltype': rna_state,
                    'count': transition_counts.loc[dcm_state, rna_state],
                    'percentage': transition_pct.loc[dcm_state, rna_state]
                })
    
    return pd.DataFrame(results)

# Calculate transitions for each leiden dimension
all_transitions = []

for leiden_col in leiden_cols:
    print(f"\nProcessing {leiden_col}...")
    trans_df = calculate_transitions(matched_df, leiden_col)
    if trans_df is not None:
        all_transitions.append(trans_df)

# Combine all results
transitions_df = pd.concat(all_transitions, ignore_index=True)

print(f"\nTotal transition records: {len(transitions_df)}")

# ============================================================
# 6. SUMMARY TABLES
# ============================================================

# Create summary for each leiden dimension
for leiden_col in leiden_cols:
    print(f"\n{'='*60}")
    print(f"TRANSITIONS FOR {leiden_col}")
    print('='*60)
    
    subset = transitions_df[transitions_df['leiden_dim'] == leiden_col]
    
    for Dox in sorted(subset['Dox'].unique()):
        print(f"\n--- Dox condition: {Dox} ---")
        dox_subset = subset[subset['Dox'] == Dox]
        
        # Pivot to matrix format
        pivot = dox_subset.pivot(index='DCM_celltype', columns='RNA_celltype', values='percentage')
        print(pivot.round(1).to_string())

# ============================================================
# 7. VISUALIZATION
# ============================================================

def plot_transition_heatmap(df, leiden_col, dox_condition, save=True):
    """Plot heatmap of transitions for a specific leiden dimension and Dox condition."""
    subset = df[(df['leiden_dim'] == leiden_col) & (df['Dox'] == dox_condition)]
    
    if len(subset) == 0:
        print(f"No data for {leiden_col}, Dox={dox_condition}")
        return
    
    pivot = subset.pivot(index='DCM_celltype', columns='RNA_celltype', values='percentage')
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='Blues', cbar_kws={'label': 'Percentage'})
    plt.title(f'Cell Type Transitions: {leiden_col}, Dox={dox_condition}')
    plt.xlabel('RNA Cell Type')
    plt.ylabel('DCM Cell Type')
    plt.tight_layout()
    
    if save:
        plt.savefig(f'transition_heatmap_{leiden_col}_dox{dox_condition}.png', dpi=150, bbox_inches='tight')
    plt.show()

# Plot for each combination (example: leiden_annot_20)
example_leiden = 'leiden_annot_20'
dox_conditions = matched_df['Dox_DCM'].unique()

for Dox in sorted(dox_conditions):
    plot_transition_heatmap(transitions_df, example_leiden, Dox)

# ============================================================
# 8. SAVE RESULTS
# ============================================================

# Save full transitions table
transitions_df.to_csv('cell_type_transitions_all.csv', index=False)

# Save summary per leiden dimension
for leiden_col in leiden_cols:
    subset = transitions_df[transitions_df['leiden_dim'] == leiden_col]
    subset.to_csv(f'transitions_{leiden_col}.csv', index=False)

# Save matched cell dataframe
matched_df.to_csv('matched_dcm_rna_cells.csv')

print("\n" + "="*60)
print("ANALYSIS COMPLETE")
print("="*60)
print(f"Total matched cells: {len(matched_df)}")
print(f"Dox conditions: {sorted(matched_df['Dox_DCM'].unique())}")
print(f"Leiden dimensions analyzed: {leiden_cols}")
print(f"\nOutput files:")
print("  - cell_type_transitions_all.csv")
print("  - transitions_leiden_annot_XX.csv (per dimension)")
print("  - matched_dcm_rna_cells.csv")
print("  - transition_heatmap_*.png")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. SETUP - Assuming matched_df is already created
# ============================================================

# Get leiden annotation columns (original names without _DCM/_RNA suffix)
leiden_cols = [col.replace('_DCM', '') for col in matched_df.columns if col.startswith('leiden_annot_') and col.endswith('_DCM')]
print(f"Leiden dimensions: {leiden_cols}")

# ============================================================
# 2. CALCULATE STABLE VS SWITCH PERCENTAGES
# ============================================================

def calculate_stability(df, leiden_col, dox_col='Dox_DCM'):
    """
    Calculate percentage of stable (same label) vs switch (different label) transitions.
    """
    dcm_col = f'{leiden_col}_DCM'
    rna_col = f'{leiden_col}_RNA'
    
    results = []
    
    for dox in df[dox_col].unique():
        dox_df = df[df[dox_col] == dox]
        total_cells = len(dox_df)
        
        if total_cells == 0:
            continue
        
        # Stable: DCM label == RNA label
        stable_mask = dox_df[dcm_col] == dox_df[rna_col]
        stable_count = stable_mask.sum()
        switch_count = total_cells - stable_count
        
        stable_pct = (stable_count / total_cells) * 100
        switch_pct = (switch_count / total_cells) * 100
        
        results.append({
            'Dox': dox,
            'leiden_dim': leiden_col,
            'dimension': int(leiden_col.split('_')[-1]),  # Extract number for sorting
            'stable_count': stable_count,
            'switch_count': switch_count,
            'total_cells': total_cells,
            'stable_pct': stable_pct,
            'switch_pct': switch_pct
        })
    
    return pd.DataFrame(results)

# Calculate for all dimensions
stability_results = []

for leiden_col in leiden_cols:
    print(f"Processing {leiden_col}...")
    result = calculate_stability(matched_df, leiden_col)
    stability_results.append(result)

stability_df = pd.concat(stability_results, ignore_index=True)

# Sort by dimension
stability_df = stability_df.sort_values(['Dox', 'dimension']).reset_index(drop=True)

print(f"\nStability dataframe shape: {stability_df.shape}")
print(stability_df.head(10))

# ============================================================
# 3. SUMMARY TABLE
# ============================================================

print("\n" + "="*60)
print("STABLE VS SWITCH PERCENTAGES")
print("="*60)

for dox in sorted(stability_df['Dox'].unique()):
    print(f"\n--- Dox condition: {dox} ---")
    dox_subset = stability_df[stability_df['Dox'] == dox].sort_values('dimension')
    print(dox_subset[['leiden_dim', 'total_cells', 'stable_pct', 'switch_pct']].to_string(index=False))

# ============================================================
# 4. RESHAPE DATA FOR PLOTTING
# ============================================================

# Melt the dataframe for easier plotting
plot_df = stability_df.melt(
    id_vars=['Dox', 'leiden_dim', 'dimension', 'total_cells'],
    value_vars=['stable_pct', 'switch_pct'],
    var_name='transition_type',
    value_name='percentage'
)

# Clean up labels
plot_df['transition_type'] = plot_df['transition_type'].map({
    'stable_pct': 'Stable',
    'switch_pct': 'Switch'
})

# ============================================================
# 5. PLOT: BARPLOT PER DIMENSION SEPARATED BY DOX
# ============================================================

# Get unique dox conditions
dox_conditions = sorted(stability_df['Dox'].unique())
n_dox = len(dox_conditions)

# Create figure with subplots for each dox condition
fig, axes = plt.subplots(1, n_dox, figsize=(5*n_dox, 6), sharey=True)

if n_dox == 1:
    axes = [axes]

colors = {'Stable': '#2ecc71', 'Switch': '#e74c3c'}

for ax, dox in zip(axes, dox_conditions):
    dox_data = plot_df[plot_df['Dox'] == dox].sort_values('dimension')
    
    sns.barplot(
        data=dox_data,
        x='dimension',
        y='percentage',
        hue='transition_type',
        palette=colors,
        ax=ax
    )
    
    ax.set_title(f'Dox = {dox}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Dimension', fontsize=12)
    ax.set_ylabel('Percentage (%)' if ax == axes[0] else '', fontsize=12)
    ax.set_ylim(0, 100)
    ax.legend(title='Transition', loc='upper right')
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Cell Type Stability: DCM vs RNA', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('stability_barplot_by_dox.png', dpi=150, bbox_inches='tight')
plt.show()

# ============================================================
# 6. ALTERNATIVE PLOT: STACKED BARPLOT
# ============================================================

fig, axes = plt.subplots(1, n_dox, figsize=(5*n_dox, 6), sharey=True)

if n_dox == 1:
    axes = [axes]

for ax, dox in zip(axes, dox_conditions):
    dox_data = stability_df[stability_df['Dox'] == dox].sort_values('dimension')
    
    dimensions = dox_data['dimension'].values
    stable = dox_data['stable_pct'].values
    switch = dox_data['switch_pct'].values
    
    x = np.arange(len(dimensions))
    
    ax.bar(x, stable, label='Stable', color='#2ecc71')
    ax.bar(x, switch, bottom=stable, label='Switch', color='#e74c3c')
    
    ax.set_title(f'Dox = {dox}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Dimension', fontsize=12)
    ax.set_ylabel('Percentage (%)' if ax == axes[0] else '', fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(dimensions)
    ax.set_ylim(0, 100)
    ax.legend(title='Transition', loc='upper right')

plt.suptitle('Cell Type Stability: DCM vs RNA (Stacked)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('stability_stacked_barplot_by_dox.png', dpi=150, bbox_inches='tight')
plt.show()

# ============================================================
# 7. PLOT: ALL DOX CONDITIONS IN ONE PLOT (FACETED BY DIMENSION)
# ============================================================

g = sns.catplot(
    data=plot_df,
    x='Dox',
    y='percentage',
    hue='transition_type',
    col='dimension',
    kind='bar',
    palette=colors,
    col_wrap=4,
    height=4,
    aspect=1.2
)

g.set_axis_labels('Dox Condition', 'Percentage (%)')
g.set_titles('Dimension {col_name}')
g.add_legend(title='Transition')

for ax in g.axes.flat:
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Cell Type Stability Across Dimensions and Dox Conditions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('stability_faceted_by_dimension.png', dpi=150, bbox_inches='tight')
plt.show()

# ============================================================
# 8. PLOT: LINE PLOT SHOWING STABILITY TREND ACROSS DIMENSIONS
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

for dox in dox_conditions:
    dox_data = stability_df[stability_df['Dox'] == dox].sort_values('dimension')
    ax.plot(dox_data['dimension'], dox_data['stable_pct'], marker='o', label=f'Dox {dox}', linewidth=2)

ax.set_xlabel('Dimension', fontsize=12)
ax.set_ylabel('Stable Percentage (%)', fontsize=12)
ax.set_title('Cell Type Stability Across Dimensions by Dox Condition', fontsize=14, fontweight='bold')
ax.legend(title='Dox Condition')
ax.set_ylim(0, 100)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('stability_lineplot_by_dox.png', dpi=150, bbox_inches='tight')
plt.show()

# ============================================================
# 9. SAVE RESULTS
# ============================================================

stability_df.to_csv('stability_percentages.csv', index=False)
plot_df.to_csv('stability_percentages_long.csv', index=False)

print("\n" + "="*60)
print("ANALYSIS COMPLETE")
print("="*60)
print(f"\nOutput files:")
print("  - stability_percentages.csv")
print("  - stability_percentages_long.csv")
print("  - stability_barplot_by_dox.png")
print("  - stability_stacked_barplot_by_dox.png")
print("  - stability_faceted_by_dimension.png")
print("  - stability_lineplot_by_dox.png")

In [ ]:
import os 

# -----------------------------
# LABEL CONSISTENCY ACROSS DIMENSIONS
# -----------------------------
dims_cols = [c for c in adata_combined.obs.columns if c.startswith('leiden_annot_')]
labels_df = adata_combined.obs[dims_cols].copy()

# Parse cell type from number_celltype format (e.g., "1_Tuft" -> "Tuft")
for col in dims_cols:
    # Extract just the cell type part after the underscore
    labels_df[col] = labels_df[col].str.split('_').str[-1]

labels_df['batch'] = adata_combined.obs['orig.ident']
labels_df['Dox'] = adata_combined.obs['Dox'] if 'Dox' in adata_combined.obs.columns else 'Unknown'
labels_df['Sorted_population'] = adata_combined.obs['Sorted_population'] if 'Sorted_population' in adata_combined.obs.columns else 'Unknown'

# Keep only DCM cells
dcm_mask = labels_df['batch'].str.contains('_DCM', na=False)
labels_df = labels_df[dcm_mask].copy()

print(f"Found {len(labels_df)} DCM cells for label consistency analysis")
print(f"Dimensional annotation columns: {dims_cols}")

def best_label_extended(row):
    """
    For each cell, find the most consistent label across dimensions.
    Returns detailed statistics about label consistency.
    """
    # Count occurrences of each label across dimensions
    counts = row.value_counts()
    
    if counts.empty:
        return pd.Series({
            'best_match': 'Unknown', 
            'n_matches': 0, 
            'matching_dims': [], 
            'non_best_counts': {},
            'total_dims': len(row),
            'consistency_score': 0.0
        })
    
    best = counts.idxmax()
    n_best = counts.max()
    total_dims = len(row)
    consistency_score = n_best / total_dims
    
    # Which dimensions agree with the best label
    matching_dims = row[row == best].index.tolist()
    
    # What other labels appear and how often
    non_best_counts = row[row != best].value_counts().to_dict()
    
    return pd.Series({
        'best_match': best,
        'n_matches': n_best,
        'total_dims': total_dims,
        'matching_dims': matching_dims,
        'non_best_counts': non_best_counts,
        'consistency_score': consistency_score
    })

if not labels_df.empty and dims_cols:
    # Apply the analysis to each DCM cell
    summary_df = labels_df[dims_cols].apply(best_label_extended, axis=1)
    
    # Add metadata
    summary_df['cell'] = labels_df.index
    summary_df['batch'] = labels_df['batch']
    summary_df['Dox'] = labels_df['Dox']
    summary_df['Sorted_population'] = labels_df['Sorted_population']
    
    # Calculate overall statistics
    total_dcm_cells = len(summary_df)
    fully_stable = (summary_df['n_matches'] == len(dims_cols)).sum()
    partially_stable = (summary_df['n_matches'] < len(dims_cols)).sum()
    avg_consistency = summary_df['consistency_score'].mean()
    
    print(f"\n{'='*60}")
    print("LABEL CONSISTENCY ANALYSIS - PARSED CELL TYPES")
    print(f"{'='*60}")
    print(f"Total DCM cells analyzed: {total_dcm_cells}")
    print(f"Number of dimensions: {len(dims_cols)}")
    print(f"\nConsistency Statistics:")
    print(f"  Fully stable across all dims: {fully_stable} ({fully_stable/total_dcm_cells*100:.1f}%)")
    print(f"  With label changes across dims: {partially_stable} ({partially_stable/total_dcm_cells*100:.1f}%)")
    print(f"  Average consistency score: {avg_consistency:.3f}")
    
    # Show sample of cells with inconsistencies
    inconsistent_cells = summary_df[summary_df['n_matches'] < len(dims_cols)]
    if not inconsistent_cells.empty:
        print(f"\nSample of inconsistent cells (first 5):")
        for idx, row in inconsistent_cells.head().iterrows():
            print(f"  Cell: {row['cell']}")
            print(f"    Best match: {row['best_match']} ({row['n_matches']}/{row['total_dims']} dims)")
            print(f"    Other labels: {row['non_best_counts']}")
            print(f"    Consistency score: {row['consistency_score']:.3f}")
    
    # Show distribution of best matches
    print(f"\nDistribution of best matches across DCM cells:")
    label_dist = summary_df['best_match'].value_counts()
    for label, count in label_dist.items():
        percentage = count / total_dcm_cells * 100
        print(f"  {label}: {count} cells ({percentage:.1f}%)")
    
    # Analyze by Dox treatment
    if 'Dox' in summary_df.columns:
        print(f"\nConsistency by Dox treatment:")
        for dox_val in summary_df['Dox'].unique():
            subset = summary_df[summary_df['Dox'] == dox_val]
            if len(subset) > 0:
                avg_cons = subset['consistency_score'].mean()
                stable_pct = (subset['n_matches'] == len(dims_cols)).mean() * 100
                print(f"  Dox={dox_val}: {len(subset)} cells, avg consistency={avg_cons:.3f}, {stable_pct:.1f}% fully stable")
    
    # Save detailed results
    output_file = os.path.join(outdir, 'dcm_label_consistency_analysis.csv')
    summary_df.to_csv(output_file, index=False)
    print(f"\nDetailed results saved to: {output_file}")
    
    # Create visualization of consistency scores
    plt.figure(figsize=(10, 6))
    sns.histplot(data = summary_df, x = 'consistency_score', bins=20, edgecolor='black', alpha=0.7, hue = 'best_match')
    plt.axvline(x=avg_consistency, color='red', linestyle='--', label=f'Mean: {avg_consistency:.3f}')
    plt.xlabel('Consistency Score (fraction of dimensions agreeing)')
    plt.ylabel('Number of DCM cells')
    plt.title('DCM Cell Label Consistency Across Dimensions')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    
    viz_file = os.path.join(outdir, 'dcm_label_consistency_histogram.png')
    plt.savefig(viz_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Consistency histogram saved to: {viz_file}")
    
    # Additional analysis: Which cell types are most/least consistent
    print(f"\nCell type-specific consistency analysis:")
    celltype_consistency = summary_df.groupby('best_match').agg({
        'consistency_score': 'mean',
        'cell': 'count'
    }).rename(columns={'cell': 'n_cells', 'consistency_score': 'avg_consistency'})
    celltype_consistency = celltype_consistency.sort_values('avg_consistency', ascending=False)
    print(celltype_consistency.round(3))
    
else:
    print("No DCM cells found for label consistency analysis or no dimensional annotation columns.")

In [ ]:
# -----------------------------
# LABEL CONSISTENCY ACROSS DIMENSIONS - WITH BEST DIM IDENTIFICATION
# -----------------------------
outdir = '/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/integration2'
dims_cols = [c for c in adata_combined.obs.columns if c.startswith('leiden_annot_')]
labels_df = adata_combined.obs[dims_cols].copy()

# Parse cell type from number_celltype format (e.g., "1_Tuft" -> "Tuft")
for col in dims_cols:
    # Extract just the cell type part after the underscore
    labels_df[col] = labels_df[col].str.split('_').str[-1]

labels_df['batch'] = adata_combined.obs['orig.ident']
labels_df['Dox'] = adata_combined.obs['Dox'] if 'Dox' in adata_combined.obs.columns else 'Unknown'
labels_df['Sorted_population'] = adata_combined.obs['Sorted_population'] if 'Sorted_population' in adata_combined.obs.columns else 'Unknown'

# Keep only DCM cells
dcm_mask = labels_df['batch'].str.contains('_DCM', na=False)
labels_df = labels_df[dcm_mask].copy()

print(f"Found {len(labels_df)} DCM cells for label consistency analysis")
print(f"Dimensional annotation columns: {dims_cols}")

def best_label_extended(row):
    """
    For each cell, find the most consistent label across dimensions.
    Returns detailed statistics about label consistency.
    """
    # Count occurrences of each label across dimensions
    counts = row.value_counts()
    
    if counts.empty:
        return pd.Series({
            'best_match': 'Unknown', 
            'n_matches': 0, 
            'matching_dims': [], 
            'non_best_counts': {},
            'total_dims': len(row),
            'consistency_score': 0.0
        })
    
    best = counts.idxmax()
    n_best = counts.max()
    total_dims = len(row)
    consistency_score = n_best / total_dims
    
    # Which dimensions agree with the best label
    matching_dims = row[row == best].index.tolist()
    
    # What other labels appear and how often
    non_best_counts = row[row != best].value_counts().to_dict()
    
    return pd.Series({
        'best_match': best,
        'n_matches': n_best,
        'total_dims': total_dims,
        'matching_dims': matching_dims,
        'non_best_counts': non_best_counts,
        'consistency_score': consistency_score
    })

if not labels_df.empty and dims_cols:
    # Apply the analysis to each DCM cell
    summary_df = labels_df[dims_cols].apply(best_label_extended, axis=1)
    
    # Add metadata
    summary_df['cell'] = labels_df.index
    summary_df['batch'] = labels_df['batch']
    summary_df['Dox'] = labels_df['Dox']
    summary_df['Sorted_population'] = labels_df['Sorted_population']
    
    # Calculate overall statistics
    total_dcm_cells = len(summary_df)
    fully_stable = (summary_df['n_matches'] == len(dims_cols)).sum()
    partially_stable = (summary_df['n_matches'] < len(dims_cols)).sum()
    avg_consistency = summary_df['consistency_score'].mean()
    
    print(f"\n{'='*60}")
    print("LABEL CONSISTENCY ANALYSIS - PARSED CELL TYPES")
    print(f"{'='*60}")
    print(f"Total DCM cells analyzed: {total_dcm_cells}")
    print(f"Number of dimensions: {len(dims_cols)}")
    print(f"\nConsistency Statistics:")
    print(f"  Fully stable across all dims: {fully_stable} ({fully_stable/total_dcm_cells*100:.1f}%)")
    print(f"  With label changes across dims: {partially_stable} ({partially_stable/total_dcm_cells*100:.1f}%)")
    print(f"  Average consistency score: {avg_consistency:.3f}")
    
    # -----------------------------------------------------------------
    # NEW: IDENTIFY BEST PERFORMING DIMENSION(S)
    # -----------------------------------------------------------------
    print(f"\n{'='*60}")
    print("BEST PERFORMING DIMENSION(S) ANALYSIS")
    print(f"{'='*60}")
    
    # Method 1: Agreement with consensus labels
    dim_performance = {}
    
    for dim in dims_cols:
        # For each dimension, count how often it agrees with the best_match
        agreement_count = 0
        total_cells_checked = 0
        
        for idx, row in summary_df.iterrows():
            cell_label = labels_df.loc[row['cell'], dim]
            best_match = row['best_match']
            
            if pd.notna(cell_label) and pd.notna(best_match):
                total_cells_checked += 1
                if cell_label == best_match:
                    agreement_count += 1
        
        if total_cells_checked > 0:
            agreement_rate = agreement_count / total_cells_checked
            dim_performance[dim] = {
                'agreement_with_consensus': agreement_rate,
                'cells_checked': total_cells_checked,
                'agreement_count': agreement_count
            }
    
    # Sort dimensions by agreement rate
    sorted_dims = sorted(dim_performance.items(), 
                        key=lambda x: x[1]['agreement_with_consensus'], 
                        reverse=True)
    
    print("\nDimension performance (agreement with consensus labels):")
    for dim, stats in sorted_dims:
        agreement_rate = stats['agreement_with_consensus']
        cells_checked = stats['cells_checked']
        print(f"  {dim}: {agreement_rate:.3f} ({stats['agreement_count']}/{cells_checked} cells)")
    
    # Method 2: Self-consistency within each dimension
    print("\nSelf-consistency analysis (within each dimension):")
    dim_self_consistency = {}
    
    for dim in dims_cols:
        # Get all unique labels in this dimension
        unique_labels = labels_df[dim].nunique()
        total_cells = labels_df[dim].notna().sum()
        
        # Calculate entropy or purity
        label_counts = labels_df[dim].value_counts()
        if len(label_counts) > 0:
            # Purity: fraction of cells in the largest cluster
            purity = label_counts.iloc[0] / total_cells
            
            # Entropy: measure of label distribution
            proportions = label_counts / total_cells
            entropy = -np.sum(proportions * np.log2(proportions))
            
            dim_self_consistency[dim] = {
                'purity': purity,
                'entropy': entropy,
                'unique_labels': unique_labels,
                'total_cells': total_cells
            }
            print(f"  {dim}: Purity={purity:.3f}, Entropy={entropy:.3f}, {unique_labels} unique labels")
    
    # Method 3: Agreement between dimensions (pairwise)
    print("\nPairwise agreement between dimensions:")
    pairwise_agreement = pd.DataFrame(index=dims_cols, columns=dims_cols)
    
    for dim1 in dims_cols:
        for dim2 in dims_cols:
            if dim1 == dim2:
                pairwise_agreement.loc[dim1, dim2] = 1.0
            else:
                # Calculate agreement between dim1 and dim2
                mask = labels_df[dim1].notna() & labels_df[dim2].notna()
                if mask.sum() > 0:
                    agreement = (labels_df.loc[mask, dim1] == labels_df.loc[mask, dim2]).mean()
                    pairwise_agreement.loc[dim1, dim2] = agreement
                else:
                    pairwise_agreement.loc[dim1, dim2] = np.nan
    
    print("\nPairwise agreement matrix (first 3x3):")
    print(pairwise_agreement.iloc[:3, :3].round(3))
    
    # -----------------------------------------------------------------
    # IDENTIFY BEST DIMENSION(S)
    # -----------------------------------------------------------------
    print(f"\n{'='*60}")
    print("BEST PERFORMING DIMENSION(S) RECOMMENDATION")
    print(f"{'='*60}")
    
    # Combine metrics to score each dimension
    dim_scores = {}
    
    for dim in dims_cols:
        if dim in dim_performance and dim in dim_self_consistency:
            # Weighted score: 60% agreement with consensus, 40% purity
            agreement_score = dim_performance[dim]['agreement_with_consensus']
            purity_score = dim_self_consistency[dim]['purity']
            
            # Calculate average pairwise agreement with other dims
            other_dims = [d for d in dims_cols if d != dim]
            pairwise_scores = []
            for other in other_dims:
                if pd.notna(pairwise_agreement.loc[dim, other]):
                    pairwise_scores.append(pairwise_agreement.loc[dim, other])
            
            if pairwise_scores:
                avg_pairwise = np.mean(pairwise_scores)
            else:
                avg_pairwise = 0
            
            # Final score: weighted combination
            final_score = (0.5 * agreement_score + 0.3 * purity_score + 0.2 * avg_pairwise)
            dim_scores[dim] = final_score
    
    # Sort and recommend
    if dim_scores:
        sorted_by_score = sorted(dim_scores.items(), key=lambda x: x[1], reverse=True)
        
        print("\nRanked dimensions by overall performance score:")
        for rank, (dim, score) in enumerate(sorted_by_score, 1):
            agreement = dim_performance[dim]['agreement_with_consensus']
            purity = dim_self_consistency[dim]['purity']
            print(f"  {rank}. {dim}: Score={score:.3f} (Agreement={agreement:.3f}, Purity={purity:.3f})")
        
        # Top recommendations
        top_n = min(3, len(sorted_by_score))
        top_dims = [dim for dim, score in sorted_by_score[:top_n]]
        
        print(f"\n🎯 RECOMMENDED TOP {top_n} DIMENSION(S):")
        for i, dim in enumerate(top_dims, 1):
            print(f"  {i}. {dim}")
        
        # Check if there's a clear winner
        if len(sorted_by_score) >= 2:
            score_diff = sorted_by_score[0][1] - sorted_by_score[1][1]
            if score_diff > 0.05:  # Clear winner threshold
                print(f"\n✅ CLEAR WINNER: {sorted_by_score[0][0]} (score difference: {score_diff:.3f})")
            else:
                print(f"\n⚠️ CLOSE COMPETITION: Consider using average or weighted consensus of top dimensions")
        
        # Return the best dimension(s) for use in downstream analysis
        best_dimension = sorted_by_score[0][0] if sorted_by_score else None
        top_dimensions = top_dims
        
        # Save dimension performance results
        performance_df = pd.DataFrame([
            {
                'dimension': dim,
                'overall_score': dim_scores[dim],
                'agreement_with_consensus': dim_performance[dim]['agreement_with_consensus'],
                'purity': dim_self_consistency[dim]['purity'],
                'entropy': dim_self_consistency[dim]['entropy'],
                'unique_labels': dim_self_consistency[dim]['unique_labels'],
                'total_cells': dim_self_consistency[dim]['total_cells']
            }
            for dim in dim_scores.keys()
        ]).sort_values('overall_score', ascending=False)
        
        performance_file = os.path.join(outdir, 'dimension_performance_analysis.csv')
        performance_df.to_csv(performance_file, index=False)
        print(f"\n✅ Dimension performance analysis saved to: {performance_file}")
        
    else:
        print("Could not calculate dimension scores.")
        best_dimension = None
        top_dimensions = []
    
    # -----------------------------------------------------------------
    # VISUALIZE DIMENSION PERFORMANCE
    # -----------------------------------------------------------------
    if dim_scores:
        # Create performance visualization
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # Plot 1: Overall scores
        dim_names = list(dim_scores.keys())
        scores = list(dim_scores.values())
        axes[0, 0].barh(dim_names, scores, color='skyblue')
        axes[0, 0].set_xlabel('Overall Score')
        axes[0, 0].set_title('Dimension Performance Scores')
        axes[0, 0].grid(axis='x', alpha=0.3)
        
        # Plot 2: Agreement with consensus
        agreement_scores = [dim_performance[d]['agreement_with_consensus'] for d in dim_names]
        axes[0, 1].barh(dim_names, agreement_scores, color='lightgreen')
        axes[0, 1].set_xlabel('Agreement with Consensus')
        axes[0, 1].set_title('Agreement with Consensus Labels')
        axes[0, 1].grid(axis='x', alpha=0.3)
        
        # Plot 3: Purity scores
        purity_scores = [dim_self_consistency[d]['purity'] for d in dim_names]
        axes[1, 0].barh(dim_names, purity_scores, color='lightcoral')
        axes[1, 0].set_xlabel('Purity (Largest Cluster Fraction)')
        axes[1, 0].set_title('Dimension Purity Scores')
        axes[1, 0].grid(axis='x', alpha=0.3)
        
        # Plot 4: Pairwise agreement heatmap
        im = axes[1, 1].imshow(pairwise_agreement.astype(float), cmap='YlOrRd', aspect='auto')
        axes[1, 1].set_xticks(range(len(dims_cols)))
        axes[1, 1].set_yticks(range(len(dims_cols)))
        axes[1, 1].set_xticklabels([d.replace('leiden_annot_', '') for d in dims_cols], rotation=45, ha='right')
        axes[1, 1].set_yticklabels([d.replace('leiden_annot_', '') for d in dims_cols])
        axes[1, 1].set_title('Pairwise Agreement Between Dimensions')
        plt.colorbar(im, ax=axes[1, 1])
        
        plt.suptitle('Dimension Performance Analysis', fontsize=16)
        plt.tight_layout()
        
        viz_file = os.path.join(outdir, 'dimension_performance_visualization.png')
        plt.savefig(viz_file, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"✅ Dimension performance visualization saved to: {viz_file}")
    
else:
    print("No DCM cells found for label consistency analysis or no dimensional annotation columns.")
    best_dimension = None
    top_dimensions = []

# The best dimension(s) are now available for downstream use:
# - best_dimension: Single best performing dimension
# - top_dimensions: List of top N performing dimensions
print(f"\n{'='*60}")
print("DIMENSION SELECTION COMPLETE")
print(f"{'='*60}")
if best_dimension:
    print(f"Best dimension: {best_dimension}")
    print(f"Top dimensions: {top_dimensions}")
    
    # You can now use these for downstream analysis, e.g.:
    # adata_combined.obs['best_dim_label'] = adata_combined.obs[best_dimension]
    # Or create consensus from top dimensions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# -----------------------------
# LABEL CONSISTENCY ACROSS DIMENSIONS - WITH BEST DIM IDENTIFICATION
# NO DATA OVERWRITING VERSION
# -----------------------------

# Create a copy of the relevant columns to work with (no modification of adata)
dims_cols = [c for c in adata_combined.obs.columns if c.startswith('leiden_annot_')]
labels_df = adata_combined.obs[dims_cols].copy()

# Parse cell type from number_celltype format (e.g., "1_Tuft" -> "Tuft")
for col in dims_cols:
    # Extract just the cell type part after the underscore
    labels_df[col] = labels_df[col].str.split('_').str[-1]

labels_df['batch'] = adata_combined.obs['orig.ident'].copy()
labels_df['Dox'] = adata_combined.obs['Dox'].copy() if 'Dox' in adata_combined.obs.columns else 'Unknown'
labels_df['Sorted_population'] = adata_combined.obs['Sorted_population'].copy() if 'Sorted_population' in adata_combined.obs.columns else 'Unknown'

# Keep only DCM cells
dcm_mask = labels_df['batch'].str.contains('_DCM', na=False)
labels_df = labels_df[dcm_mask].copy()

print(f"Found {len(labels_df)} DCM cells for label consistency analysis")
print(f"Dimensional annotation columns: {dims_cols}")

def best_label_extended(row):
    """
    For each cell, find the most consistent label across dimensions.
    Returns detailed statistics about label consistency.
    """
    # Count occurrences of each label across dimensions
    counts = row.value_counts()
    
    if counts.empty:
        return pd.Series({
            'best_match': 'Unknown', 
            'n_matches': 0, 
            'matching_dims': [], 
            'non_best_counts': {},
            'total_dims': len(row),
            'consistency_score': 0.0
        })
    
    best = counts.idxmax()
    n_best = counts.max()
    total_dims = len(row)
    consistency_score = n_best / total_dims
    
    # Which dimensions agree with the best label
    matching_dims = row[row == best].index.tolist()
    
    # What other labels appear and how often
    non_best_counts = row[row != best].value_counts().to_dict()
    
    return pd.Series({
        'best_match': best,
        'n_matches': n_best,
        'total_dims': total_dims,
        'matching_dims': matching_dims,
        'non_best_counts': non_best_counts,
        'consistency_score': consistency_score
    })

if not labels_df.empty and dims_cols:
    # Apply the analysis to each DCM cell
    summary_df = labels_df[dims_cols].apply(best_label_extended, axis=1)
    
    # Add metadata
    summary_df['cell'] = labels_df.index
    summary_df['batch'] = labels_df['batch']
    summary_df['Dox'] = labels_df['Dox']
    summary_df['Sorted_population'] = labels_df['Sorted_population']
    
    # Calculate overall statistics
    total_dcm_cells = len(summary_df)
    fully_stable = (summary_df['n_matches'] == len(dims_cols)).sum()
    partially_stable = (summary_df['n_matches'] < len(dims_cols)).sum()
    avg_consistency = summary_df['consistency_score'].mean()
    
    print(f"\n{'='*60}")
    print("LABEL CONSISTENCY ANALYSIS - PARSED CELL TYPES")
    print(f"{'='*60}")
    print(f"Total DCM cells analyzed: {total_dcm_cells}")
    print(f"Number of dimensions: {len(dims_cols)}")
    print(f"\nConsistency Statistics:")
    print(f"  Fully stable across all dims: {fully_stable} ({fully_stable/total_dcm_cells*100:.1f}%)")
    print(f"  With label changes across dims: {partially_stable} ({partially_stable/total_dcm_cells*100:.1f}%)")
    print(f"  Average consistency score: {avg_consistency:.3f}")
    
    # -----------------------------------------------------------------
    # IDENTIFY BEST PERFORMING DIMENSION(S)
    # -----------------------------------------------------------------
    print(f"\n{'='*60}")
    print("BEST PERFORMING DIMENSION(S) ANALYSIS")
    print(f"{'='*60}")
    
    # Method 1: Agreement with consensus labels
    dim_performance = {}
    
    for dim in dims_cols:
        # For each dimension, count how often it agrees with the best_match
        agreement_count = 0
        total_cells_checked = 0
        
        for idx, row in summary_df.iterrows():
            cell_label = labels_df.loc[row['cell'], dim]
            best_match = row['best_match']
            
            if pd.notna(cell_label) and pd.notna(best_match):
                total_cells_checked += 1
                if cell_label == best_match:
                    agreement_count += 1
        
        if total_cells_checked > 0:
            agreement_rate = agreement_count / total_cells_checked
            dim_performance[dim] = {
                'agreement_with_consensus': agreement_rate,
                'cells_checked': total_cells_checked,
                'agreement_count': agreement_count
            }
    
    # Sort dimensions by agreement rate
    sorted_dims = sorted(dim_performance.items(), 
                        key=lambda x: x[1]['agreement_with_consensus'], 
                        reverse=True)
    
    print("\nDimension performance (agreement with consensus labels):")
    for dim, stats in sorted_dims:
        agreement_rate = stats['agreement_with_consensus']
        cells_checked = stats['cells_checked']
        print(f"  {dim}: {agreement_rate:.3f} ({stats['agreement_count']}/{cells_checked} cells)")
    
    # Method 2: Self-consistency within each dimension
    print("\nSelf-consistency analysis (within each dimension):")
    dim_self_consistency = {}
    
    for dim in dims_cols:
        # Get all unique labels in this dimension
        unique_labels = labels_df[dim].nunique()
        total_cells = labels_df[dim].notna().sum()
        
        # Calculate entropy or purity
        label_counts = labels_df[dim].value_counts()
        if len(label_counts) > 0:
            # Purity: fraction of cells in the largest cluster
            purity = label_counts.iloc[0] / total_cells
            
            # Entropy: measure of label distribution
            proportions = label_counts / total_cells
            entropy = -np.sum(proportions * np.log2(proportions))
            
            dim_self_consistency[dim] = {
                'purity': purity,
                'entropy': entropy,
                'unique_labels': unique_labels,
                'total_cells': total_cells
            }
            print(f"  {dim}: Purity={purity:.3f}, Entropy={entropy:.3f}, {unique_labels} unique labels")
    
    # Method 3: Agreement between dimensions (pairwise)
    print("\nPairwise agreement between dimensions:")
    pairwise_agreement = pd.DataFrame(index=dims_cols, columns=dims_cols)
    
    for dim1 in dims_cols:
        for dim2 in dims_cols:
            if dim1 == dim2:
                pairwise_agreement.loc[dim1, dim2] = 1.0
            else:
                # Calculate agreement between dim1 and dim2
                mask = labels_df[dim1].notna() & labels_df[dim2].notna()
                if mask.sum() > 0:
                    agreement = (labels_df.loc[mask, dim1] == labels_df.loc[mask, dim2]).mean()
                    pairwise_agreement.loc[dim1, dim2] = agreement
                else:
                    pairwise_agreement.loc[dim1, dim2] = np.nan
    
    print("\nPairwise agreement matrix (first 3x3):")
    print(pairwise_agreement.iloc[:3, :3].round(3))
    
    # -----------------------------------------------------------------
    # IDENTIFY BEST DIMENSION(S)
    # -----------------------------------------------------------------
    print(f"\n{'='*60}")
    print("BEST PERFORMING DIMENSION(S) RECOMMENDATION")
    print(f"{'='*60}")
    
    # Combine metrics to score each dimension
    dim_scores = {}
    
    for dim in dims_cols:
        if dim in dim_performance and dim in dim_self_consistency:
            # Weighted score: 60% agreement with consensus, 40% purity
            agreement_score = dim_performance[dim]['agreement_with_consensus']
            purity_score = dim_self_consistency[dim]['purity']
            
            # Calculate average pairwise agreement with other dims
            other_dims = [d for d in dims_cols if d != dim]
            pairwise_scores = []
            for other in other_dims:
                if pd.notna(pairwise_agreement.loc[dim, other]):
                    pairwise_scores.append(pairwise_agreement.loc[dim, other])
            
            if pairwise_scores:
                avg_pairwise = np.mean(pairwise_scores)
            else:
                avg_pairwise = 0
            
            # Final score: weighted combination
            final_score = (0.5 * agreement_score + 0.3 * purity_score + 0.2 * avg_pairwise)
            dim_scores[dim] = final_score
    
    # Sort and recommend
    if dim_scores:
        sorted_by_score = sorted(dim_scores.items(), key=lambda x: x[1], reverse=True)
        
        print("\nRanked dimensions by overall performance score:")
        for rank, (dim, score) in enumerate(sorted_by_score, 1):
            agreement = dim_performance[dim]['agreement_with_consensus']
            purity = dim_self_consistency[dim]['purity']
            print(f"  {rank}. {dim}: Score={score:.3f} (Agreement={agreement:.3f}, Purity={purity:.3f})")
        
        # Top recommendations
        top_n = min(3, len(sorted_by_score))
        top_dims = [dim for dim, score in sorted_by_score[:top_n]]
        
        print(f"\n🎯 RECOMMENDED TOP {top_n} DIMENSION(S):")
        for i, dim in enumerate(top_dims, 1):
            print(f"  {i}. {dim}")
        
        # Check if there's a clear winner
        if len(sorted_by_score) >= 2:
            score_diff = sorted_by_score[0][1] - sorted_by_score[1][1]
            if score_diff > 0.05:  # Clear winner threshold
                print(f"\n✅ CLEAR WINNER: {sorted_by_score[0][0]} (score difference: {score_diff:.3f})")
            else:
                print(f"\n⚠️ CLOSE COMPETITION: Consider using average or weighted consensus of top dimensions")
        
        # Return the best dimension(s) for use in downstream analysis
        best_dimension = sorted_by_score[0][0] if sorted_by_score else None
        top_dimensions = top_dims
        
        # Save dimension performance results
        performance_df = pd.DataFrame([
            {
                'dimension': dim,
                'overall_score': dim_scores[dim],
                'agreement_with_consensus': dim_performance[dim]['agreement_with_consensus'],
                'purity': dim_self_consistency[dim]['purity'],
                'entropy': dim_self_consistency[dim]['entropy'],
                'unique_labels': dim_self_consistency[dim]['unique_labels'],
                'total_cells': dim_self_consistency[dim]['total_cells']
            }
            for dim in dim_scores.keys()
        ]).sort_values('overall_score', ascending=False)
        
        performance_file = os.path.join(outdir, 'dimension_performance_analysis.csv')
        performance_df.to_csv(performance_file, index=False)
        print(f"\n✅ Dimension performance analysis saved to: {performance_file}")
        
    else:
        print("Could not calculate dimension scores.")
        best_dimension = None
        top_dimensions = []
    
    # -----------------------------------------------------------------
    # VISUALIZE DIMENSION PERFORMANCE - PDF FORMAT WITH DPI=300
    # -----------------------------------------------------------------
    if dim_scores:
        # Create performance visualization
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # Plot 1: Overall scores
        dim_names = list(dim_scores.keys())
        scores = list(dim_scores.values())
        axes[0, 0].barh(dim_names, scores, color='skyblue')
        axes[0, 0].set_xlabel('Overall Score')
        axes[0, 0].set_title('Dimension Performance Scores')
        axes[0, 0].grid(axis='x', alpha=0.3)
        
        # Plot 2: Agreement with consensus
        agreement_scores = [dim_performance[d]['agreement_with_consensus'] for d in dim_names]
        axes[0, 1].barh(dim_names, agreement_scores, color='lightgreen')
        axes[0, 1].set_xlabel('Agreement with Consensus')
        axes[0, 1].set_title('Agreement with Consensus Labels')
        axes[0, 1].grid(axis='x', alpha=0.3)
        
        # Plot 3: Purity scores
        purity_scores = [dim_self_consistency[d]['purity'] for d in dim_names]
        axes[1, 0].barh(dim_names, purity_scores, color='lightcoral')
        axes[1, 0].set_xlabel('Purity (Largest Cluster Fraction)')
        axes[1, 0].set_title('Dimension Purity Scores')
        axes[1, 0].grid(axis='x', alpha=0.3)
        
        # Plot 4: Pairwise agreement heatmap
        im = axes[1, 1].imshow(pairwise_agreement.astype(float), cmap='YlOrRd', aspect='auto')
        axes[1, 1].set_xticks(range(len(dims_cols)))
        axes[1, 1].set_yticks(range(len(dims_cols)))
        axes[1, 1].set_xticklabels([d.replace('leiden_annot_', '') for d in dims_cols], rotation=45, ha='right')
        axes[1, 1].set_yticklabels([d.replace('leiden_annot_', '') for d in dims_cols])
        axes[1, 1].set_title('Pairwise Agreement Between Dimensions')
        plt.colorbar(im, ax=axes[1, 1])
        
        plt.suptitle('Dimension Performance Analysis', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        # Save as PDF with dpi=300
        viz_file = os.path.join(outdir, 'dimension_performance_visualization.pdf')
        plt.savefig(viz_file, dpi=300, bbox_inches='tight', format='pdf')
        plt.show()
        plt.close()
        print(f"✅ Dimension performance visualization saved to: {viz_file}")
    
else:
    print("No DCM cells found for label consistency analysis or no dimensional annotation columns.")
    best_dimension = None
    top_dimensions = []

# The best dimension(s) are now stored in variables (NOT written to adata):
# - best_dimension: Single best performing dimension
# - top_dimensions: List of top N performing dimensions
print(f"\n{'='*60}")
print("DIMENSION SELECTION COMPLETE")
print(f"{'='*60}")
if best_dimension:
    print(f"Best dimension: {best_dimension}")
    print(f"Top dimensions: {top_dimensions}")
    
    # NOTE: No data is written back to adata_combined
    # If you want to add these labels later, you can uncomment:
    # adata_combined.obs['best_dim_label'] = adata_combined.obs[best_dimension]

In [ ]:
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 8
plt.rcParams['axes.linewidth'] = 1
plt.rcParams['xtick.major.width'] = 1
plt.rcParams['ytick.major.width'] = 1
plt.rcParams['axes.grid'] = False
sns.set_style("white", {'axes.grid': False})

# -----------------------------
# FIXED DIMENSION
# -----------------------------
dim = 25

# -----------------------------
# CELL TYPE ORDER
# -----------------------------
celltype_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# -----------------------------
# add orig.ident
# -----------------------------
adata_dcm.obs['orig.ident'] = 'scNMT' + '_' + adata_dcm.obs.pool.astype(str) + '_DCM'
adata_rna.obs['orig.ident'] = 'scNMT' + '_' + adata_rna.obs.pool.astype(str) + '_RNA'

# -----------------------------
# FILTER DCM FOR PLUS DOX
# -----------------------------
adata_dcm_plus = adata_dcm[adata_dcm.obs.Dox != 'Minus'].copy()

# -----------------------------
# COMBINE WITH RNA AND REFERENCE
# -----------------------------
print("\n🔹 COMBINING DCM+RNA+REFERENCE")

all_obs_cols = list(set().union(*[adata.obs.columns for adata in [adata_dcm_plus, adata_rna, adata_ref]]))

for adata in [adata_dcm_plus, adata_rna, adata_ref]:
    for col in all_obs_cols:
        if col not in adata.obs.columns:
            adata.obs[col] = 'Unknown'

adata_combined = sc.concat(
    [adata_dcm_plus, adata_rna, adata_ref],
    join='inner',
    merge='unique',
    index_unique=None
)

for col in ['Dox', 'Sorted_population', 'cell_type_broad', 'celltype', 'leiden', 'orig.ident']:
    if col in adata_combined.obs.columns:
        adata_combined.obs[col] = adata_combined.obs[col].fillna('Unknown').astype(str)

for col in ['celltype', 'Sorted_population', 'Dox', 'orig.ident']:
    if col in adata_combined.obs.columns:
        adata_combined.obs[col] = adata_combined.obs[col].astype('string')

adata_combined.obs['sort_assay_dox'] = (
    adata_combined.obs['Sorted_population'] + '_' +
    adata_combined.obs['orig.ident'] + '_' +
    adata_combined.obs['Dox']
)

# -----------------------------
# PARAMETERS
# -----------------------------
sc.settings.figdir = outdir
sc.set_figure_params(dpi=300, dpi_save=300, vector_friendly=True, fontsize=6)

n_top_genes = 2000
neighbors_within_batch = 1
pynndescent_n_neighbors = 3

# -----------------------------
# HVG & PCA
# -----------------------------
adata_combined.X = adata_combined.layers['lognorm']

sc.pp.highly_variable_genes(
    adata_combined,
    n_top_genes=n_top_genes,
    batch_key='orig.ident'
)

adata_combined.X = adata_combined.layers['lognorm_scaled']
sc.tl.pca(adata_combined)

# -----------------------------
# BBKNN
# -----------------------------
print(f"\n--- Running integration for dim={dim} ---")

try:
    sc.external.pp.bbknn(
        adata_combined,
        batch_key='orig.ident',
        n_pcs=dim,
        neighbors_within_batch=neighbors_within_batch,
        pynndescent_n_neighbors=pynndescent_n_neighbors
    )
except Exception as e:
    print("BBKNN failed, falling back to neighbors():", e)
    sc.pp.neighbors(adata_combined, n_pcs=dim)

# -----------------------------
# UMAP & LEIDEN
# -----------------------------
sc.tl.umap(adata_combined)
sc.tl.leiden(adata_combined, resolution=2, key_added='leiden', random_state=0)

# -----------------------------
# HEATMAP: Leiden vs cell_type_broad
# -----------------------------
contingency = pd.crosstab(
    adata_combined.obs['leiden'],
    adata_combined.obs['cell_type_broad'],
    margins=True,
    margins_name='Total'
)

heatmap_data = contingency.iloc[:-1, :-1].drop(columns='Unknown', errors='ignore')

# Reorder columns based on celltype_order (only keep columns that exist)
cols_ordered = [col for col in celltype_order if col in heatmap_data.columns]
# Add any remaining columns not in the order
cols_remaining = [col for col in heatmap_data.columns if col not in cols_ordered]
heatmap_data = heatmap_data[cols_ordered + cols_remaining]

fig, ax = plt.subplots(figsize=(3, max(2, len(heatmap_data) * 0.2)))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='d',
    cmap='viridis',
    linewidths=0,
    ax=ax
)
plt.tight_layout()
fig.savefig(f'{outdir}heatmap_leiden_vs_celltype_dim{dim}.pdf')
plt.close(fig)

# -----------------------------
# NORMALIZED HEATMAP
# -----------------------------
heatmap_norm = heatmap_data.div(heatmap_data.sum(axis=1), axis=0) * 100

# Already ordered from above, but apply same order
heatmap_norm = heatmap_norm[cols_ordered + cols_remaining]

fig, ax = plt.subplots(figsize=(3, max(2, len(heatmap_norm) * 0.2)))
sns.heatmap(
    heatmap_norm,
    annot=True,
    fmt='.1f',
    cmap='viridis',
    linewidths=0,
    ax=ax
)
plt.tight_layout()
fig.savefig(f'{outdir}heatmap_leiden_vs_celltype_normalized_dim{dim}.pdf')
plt.close(fig)

# -----------------------------
# AUTOMATIC ANNOTATION
# -----------------------------
annot_col = f'leiden_annot_{dim}'
cluster_annotations = {}

for cl in adata_combined.obs['leiden'].astype(str).unique():
    vals = adata_combined.obs.loc[
        adata_combined.obs['leiden'].astype(str) == cl,
        'cell_type_broad'
    ].value_counts()
    vals = vals.drop(['Unknown'], errors='ignore')
    cluster_annotations[cl] = vals.idxmax() if len(vals) > 0 else 'Unknown'

adata_combined.obs[annot_col] = (
    adata_combined.obs['leiden'].astype(str).map(cluster_annotations)
)

# -----------------------------
# UMAP PLOTS
# -----------------------------
sc.pl.umap(
    adata_combined,
    color=['leiden', 'cell_type_broad', annot_col],
    frameon=False,
    legend_loc='on data',
    save=f'{dim}_umap.svg'
)

In [ ]:
adata_combined.obs['leiden']

In [ ]:
# -----------------------------
# CELL TYPE ORDER
# -----------------------------
celltype_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# -----------------------------
# HEATMAP: Leiden vs cell_type_broad
# -----------------------------
contingency = pd.crosstab(
    adata_combined.obs['leiden'],
    adata_combined.obs['leiden_annot_25'],
    margins=True,
    margins_name='Total'
)

heatmap_data = contingency.iloc[:-1, :-1].drop(columns='Unknown', errors='ignore')

# Reorder columns (cell types)
cols_ordered = [col for col in celltype_order if col in heatmap_data.columns]
cols_remaining = [col for col in heatmap_data.columns if col not in cols_ordered]
heatmap_data = heatmap_data[cols_ordered + cols_remaining]

# Reorder rows (leiden clusters) numerically
heatmap_data.index = heatmap_data.index.astype(int)
heatmap_data = heatmap_data.sort_index()

fig, ax = plt.subplots(figsize=(4, max(3, len(heatmap_data) * 0.25)))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='d',
    cmap='viridis',
    linewidths=0,
    ax=ax
)
ax.set_ylabel('Leiden Cluster', fontsize=10)
ax.set_xlabel('Cell Type', fontsize=10)
plt.tight_layout()
fig.savefig(f'{outdir}heatmap_leiden_vs_celltype_dim{dim}.pdf')
plt.close(fig)

# -----------------------------
# NORMALIZED HEATMAP
# -----------------------------
heatmap_norm = heatmap_data.div(heatmap_data.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(4, max(3, len(heatmap_norm) * 0.25)))
sns.heatmap(
    heatmap_norm,
    annot=True,
    fmt='.1f',
    cmap='viridis',
    linewidths=0,
    ax=ax
)
ax.set_ylabel('Leiden Cluster', fontsize=10)
ax.set_xlabel('Cell Type', fontsize=10)
plt.tight_layout()
fig.savefig(f'{outdir}heatmap_leiden_vs_celltype_normalized_dim{dim}.pdf')
plt.close(fig)

print(f"Total clusters: {len(heatmap_data)}")
print(f"Clusters: {heatmap_data.index.tolist()}")

In [ ]:
adata_combined.obs['leiden_annot_25'] = adata_combined.obs['leiden_annot_25'].astype(str)
adata_combined.obs.loc[
    adata_combined.obs['leiden'] == '7',
    'leiden_annot_25'
] = 'TA'

In [ ]:
adata_combined.obs['assay'] = adata_combined.obs['orig.ident'].map({
    'scNMT_pool35_DCM': 'pDCM',
    'scNMT_pool40_DCM': 'pDCM',
    'scNMT_pool35_RNA': 'pRNA',
    'scNMT_pool40_RNA': 'pRNA',
    'B1': 'Ref',
    'B2': 'Ref',
    'B3': 'Ref',
    'B4': 'Ref',
    'B5': 'Ref',
    'B6': 'Ref',
    'B7': 'Ref',
    'B8': 'Ref',
    'B9': 'Ref',
    'B10': 'Ref',
    'RNA_Harber': 'Ref'
})

In [ ]:
### -----------------------------------------
### Define all marker sets in one dictionary
### -----------------------------------------
marker_sets = {
    "Enterocyte": [
        "Alpi", "Fabp1", "Fabp2", "Apoa1", "Apoa4",
        "Slc2a2", "Adh1", "Sis", "Mttp", "Acaa1a",
        "Krt20", "Reg1", "Aspscr1", "Pla2g2a",
        "Apoa2", "ApoC3", "Cyp2c70"
    ],
    "TA": [
        "Mki67", "Top2a", "Ccnb1", "Ccnb2", "Cdk1",
        "Pcna", "Cdc20", "Cenpf", "Cdk4", "Birc5",
        "Cks1b", "Rrm2", "Tyms", "Uhrf1"
    ],
    "Stem": [
        "Lgr5", "Olfm4", "Ascl2", "Sox9", "Axin2",
        "Tnfrsf19", "Ephb3", "Ephb2", "Smoc2", "Rnf43",
        "Tmem184b", "Tspan1"
    ],
    "Goblet": [
        "Muc2", "Spdef", "Agr2", "Tff3",
        "Fcgbp", "Guca2a", "Dll1",
        "Klf4", "Muc13", "Muc4",
        "Clca1", "Tmem45b"
    ],
    "Paneth": [
        "Lyz1", "Lyz2", "Defa24", "Defa17", "Defa22",
        "Reg3b", "Reg3g", "Mmp7", "Casp4",
        "Ang4", "Pla2g2a", "Srp9", "Guca2a"
    ],
    "Tuft": [
        "Dclk1", "Pou2f3", "Trpm5", "Gfi1b",
        "Il25", "Avil", "Krt18", "Krt8",
        "Rgs13", "Alox5ap", "Ano1"
    ],
    "EEC": [
        "Chga", "Chgb", "Neurod1", "Neurog3",
        "Pax6", "Tph1", "Tac1",
        "Gcg", "Pyy", "Gip",
        "Sct", "Sst", "Ucn3",
        "Cck", "Ghrl", "Insm1"
    ]
}

### -----------------------------------------
### Score all marker sets automatically
### -----------------------------------------
for score_name, genes in marker_sets.items():
    sc.tl.score_genes(
        adata_combined,
        gene_list=genes,
        score_name=f"{score_name}_score"
    )

### -----------------------------------------
### Set vector-friendly settings with dpi
### -----------------------------------------
sc.settings.set_figure_params(vector_friendly=True, dpi=300, dpi_save=300)

### -----------------------------------------
### Plot UMAP with all scores + metadata
### -----------------------------------------
# Option 1: Use scanpy's built-in save
sc.pl.umap(
    adata_combined,
    color=[f"{k}_score" for k in marker_sets.keys()] + ["leiden_annot_25", "cell_type_broad"],
    legend_loc='on data',
    cmap="coolwarm",
    save='_marker_scores.svg'
)

# Option 2: Save each plot individually
for score_name in marker_sets.keys():
    sc.pl.umap(
        adata_combined,
        color=f"{score_name}_score",
        legend_loc='on data',
        cmap="coolwarm",
        frameon=False,
        save=f'_{score_name}_score.svg'
    )

for annot in ["leiden_annot_25", "cell_type_broad"]:
    sc.pl.umap(
        adata_combined,
        color=annot,
        legend_loc='on data',
        frameon=False,
        save=f'_{annot}.svg'
    )


In [ ]:
for batch in stability_df['orig.ident'].unique():
    dfsub = stability_df[stability_df['orig.ident'] == batch]
    sns.histplot(data= dfsub, x = 'stability', hue = 'best_label')
    plt.title(f'{batch}')
    plt.show()

In [ ]:
###Heatmap per assay and cell type##

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# -----------------------------
# SETTINGS
# -----------------------------
outdir = "/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/integration/"
min_cells = 10   # minimum cells per (assay, cell type)

plt.rcParams['font.family'] = 'Arial'
sns.set_style("white")

# -----------------------------
# DEFINE ASSAY AND COMBINED LABEL
# -----------------------------

adata_combined.obs['assay_celltype'] = (
    adata_combined.obs['assay'].astype(str) + '|' +
    adata_combined.obs['leiden_annot_25'].astype(str)
)

# -----------------------------
# PSEUDOBULK EXPRESSION
# -----------------------------
expr = adata_combined.layers['lognorm_scaled']
genes = adata_combined.var_names

pb = {}
for key in adata_combined.obs['assay_celltype'].unique():
    mask = adata_combined.obs['assay_celltype'] == key
    if mask.sum() < min_cells:
        continue
    pb[key] = np.asarray(expr[mask].mean(axis=0)).ravel()

pb_df = pd.DataFrame(pb, index=genes)

# Optional: restrict to HVGs
if 'highly_variable' in adata_combined.var.columns:
    pb_df = pb_df.loc[adata_combined.var['highly_variable']]

# -----------------------------
# CORRELATION MATRIX
# -----------------------------
corr = pb_df.corr(method='pearson')

# -----------------------------
# ORDER COLUMNS AND ROWS
# -----------------------------
celltype_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']
assay_order = ['Ref', 'pRNA', 'pDCM']

ordered_cols = []
for ct in celltype_order:
    for assay in assay_order:
        col_name = f"{assay}|{ct}"
        if col_name in corr.columns:
            ordered_cols.append(col_name)

corr = corr.loc[ordered_cols, ordered_cols]

# -----------------------------
# CREATE ANNOTATION COLORS
# -----------------------------
# Assign colors to assays
assay_palette = {'Ref': '#808080', 'pRNA': '#8B0000', 'pDCM': '#003366'}
assay_colors = [assay_palette[label.split('|')[0]] for label in ordered_cols]

# Assign colors to cell types
celltype_palette = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1'
}
celltype_colors = [celltype_palette[label.split('|')[1]] for label in ordered_cols]

# Combine into a DataFrame for seaborn clustermap
col_colors = pd.DataFrame({
    'Assay': assay_colors,
    'CellType': celltype_colors
}, index=ordered_cols)

# -----------------------------
# PLOT CLUSTERMAP (with annotations)
# -----------------------------
sns.set(font_scale=0.8)
g = sns.clustermap(
    corr,
    row_cluster=False,
    col_cluster=False,
    cmap='RdBu_r',
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0,
    cbar_kws={'label': 'Pearson correlation'},
    row_colors=col_colors,
    col_colors=col_colors,
    figsize=(0.5 * corr.shape[0], 0.5 * corr.shape[0])
)

# Adjust axis labels
for label in g.ax_heatmap.get_xticklabels():
    label.set_rotation(90)
for label in g.ax_heatmap.get_yticklabels():
    label.set_rotation(0)

plt.tight_layout()
g.savefig(f"{outdir}assay_celltype_correlation_heatmap_annotated.pdf",
          format='pdf', dpi=300, bbox_inches='tight')
plt.close(g.fig)

print("Ordered and annotated correlation heatmap saved.")


In [ ]:
##Stats plots paper#

In [ ]:
import scanpy as sc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# -------------------------
# Rank genes with both methods
# -------------------------
adata_ref.X = adata_ref.layers['lognorm']
sc.tl.rank_genes_groups(adata_ref, groupby='cell_type_broad', method='logreg')
logreg_res = adata_ref.uns['rank_genes_groups']

sc.tl.rank_genes_groups(adata_ref, groupby='cell_type_broad', method='t-test')
ttest_res = adata_ref.uns['rank_genes_groups']

# --- Helper to extract results from Scanpy's rank_genes_groups ---
def extract_results(res, method):
    groups = res['names'].dtype.names
    all_genes = []
    for ct in groups:
        df = pd.DataFrame({
            'gene': res['names'][ct],
            'score_logreg': res['scores'][ct] if method == 'logreg' else None,
            'logfc_ttest': res['logfoldchanges'][ct] if method == 't-test' else None,
            'pvals_adj': res['pvals_adj'][ct] if method == 't-test' else None,
            'cell_type': ct
        })
        all_genes.append(df)
    return pd.concat(all_genes, ignore_index=True)

# Extract results for both methods
logreg_df = extract_results(logreg_res, 'logreg')
ttest_df = extract_results(ttest_res, 't-test')

# Merge on gene + cell_type
compare_df = pd.merge(
    logreg_df[['gene','cell_type','score_logreg','pvals_adj']],
    ttest_df[['gene','cell_type','logfc_ttest','pvals_adj']],
    on=['gene','cell_type'],
    suffixes=('_logreg','_ttest')
)

# Determine significant genes per method
compare_df['significant_logreg'] = compare_df['score_logreg'] > 0.5
compare_df['significant_ttest'] = compare_df['pvals_adj_ttest'] < 0.05

# Keep only genes significant in at least one method
sig_df = compare_df[(compare_df['significant_logreg']) | (compare_df['significant_ttest'])].copy()

# --- KDE plot ---
plt.figure(figsize=(8,6))
sns.kdeplot(
    data=sig_df,
    x='score_logreg',
    hue='cell_type',
    common_norm=False,
    fill=True,
    alpha=0.3
)
sns.kdeplot(
    data=sig_df,
    x='logfc_ttest',
    hue='cell_type',
    common_norm=False,
    fill=False,
    linestyle='--',
    alpha=0.7
)
plt.xlabel("Marker Score")
plt.title("Distribution of logreg vs t-test marker scores (significant genes only)")
plt.show()

# --- Scatter plot ---
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=sig_df,
    x='score_logreg',
    y='logfc_ttest',
    hue='cell_type',
    alpha=0.7
)

plt.axhline(0, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')
plt.xlabel("logreg score")
plt.ylabel("t-test log fold change")
plt.title("Comparison of logreg vs t-test marker scores")
plt.legend(bbox_to_anchor=(1.05,1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
########################################## LOG REG MARKERS ############################################################
######### rna reference ########## clean up plot for main fig #########################################################
topgenes = compare_df[(compare_df.score_logreg >= 0.5) & (compare_df.significant_ttest == True)].gene.to_list()
sc.tl.rank_genes_groups(adata_ref, groupby='cell_type_broad', pts = True)
sc.tl.dendrogram(adata_ref, groupby='cell_type_broad')
sc.pl.rank_genes_groups_matrixplot(adata_ref, var_names=topgenes, cmap = 'coolwarm', layer = 'lognorm_scaled', standard_scale = 'var', dendrogram=False)


######### dcm ##########
# Find common indices
common_cells = adata_dcm_plus.obs_names.intersection(adata_combined.obs_names)
print(f"Common cells: {len(common_cells)} out of {len(adata_dcm_plus)}")

# Only assign for common cells
adata_dcm_plus.obs['leiden_annot_25'] = None  # Initialize with None
adata_dcm_plus.obs.loc[common_cells, 'leiden_annot_25'] = adata_combined.obs.loc[common_cells, 'leiden_annot_25']

# More robust splitting
adata_dcm_plus.obs['cell_type_broad'] = (
    adata_dcm_plus.obs['leiden_annot_25']
    .fillna('Unknown')  # Fill NaN with 'Unknown'
)
adata_dcm_plus.X = adata_dcm_plus.layers['lognorm']
sc.tl.rank_genes_groups(adata_dcm_plus, groupby='cell_type_broad', pts = True)
sc.tl.dendrogram(adata_dcm_plus, groupby='cell_type_broad')
sc.pl.rank_genes_groups_matrixplot(adata_dcm_plus, groupby='cell_type_broad', var_names = topgenes, cmap = 'coolwarm', layer = 'lognorm_scaled', standard_scale = 'var', dendrogram=False)

######### rna ##########
# Find common indices
common_cells = adata_rna.obs_names.intersection(adata_combined.obs_names)
print(f"Common cells: {len(common_cells)} out of {len(adata_rna)}")

# Only assign for common cells
adata_rna.obs['leiden_annot_25'] = None  # Initialize with None
adata_rna.obs.loc[common_cells, 'leiden_annot_25'] = adata_combined.obs.loc[common_cells, 'leiden_annot_25']

# More robust splitting
adata_rna.obs['cell_type_broad'] = (
    adata_rna.obs['leiden_annot_25']
    .fillna('Unknown')  # Fill NaN with 'Unknown'
)
adata_rna.X = adata_rna.layers['lognorm']
sc.tl.rank_genes_groups(adata_rna, groupby='cell_type_broad', pts = True)
sc.tl.dendrogram(adata_rna, groupby='cell_type_broad')
sc.pl.rank_genes_groups_matrixplot(adata_rna, groupby='cell_type_broad', var_names = topgenes, cmap = 'coolwarm', layer = 'lognorm_scaled', standard_scale = 'var', dendrogram=False)

In [ ]:
#################### classical markers  #################### 
################################################################################
# 1) DEFINE ALL MARKER SETS CLEANLY
################################################################################

#### literature markers 
enterocyte_markers = [
    "Alpi", "Fabp1", "Fabp2", "Apoa1", "Apoa4",
    "Slc2a2", "Adh1", "Sis", "Mttp", "Acaa1a",
    "Krt20", "Reg1", "Aspscr1", "Pla2g2a",
    "Apoa2", "ApoC3", "Cyp2c70"
]



ta_markers = [
    "Mki67", "Top2a", "Ccnb1", "Ccnb2", "Cdk1",
    "Pcna", "Cdc20", "Cenpf", "Cdk4", "Birc5",
    "Cks1b", "Rrm2", "Tyms", "Uhrf1"
]

stem_general_markers = [
    "Lgr5", "Olfm4", "Ascl2", "Sox9", "Axin2",
    "Tnfrsf19", "Ephb3", "Ephb2", "Smoc2", "Rnf43",
    "Tmem184b", "Tspan1"
]



goblet_markers = [
    "Muc2", "Spdef", "Agr2", "Tff3",
    "Fcgbp", "Guca2a", "Dll1",
    "Klf4", "Muc13", "Muc4",
    "Clca1", "Tmem45b"
]

paneth_markers = [
    "Lyz1", "Lyz2", "Defa24", "Defa17", "Defa22",
    "Reg3b", "Reg3g", "Mmp7", "Casp4",
    "Ang4", "Pla2g2a", "Srp9", "Guca2a"
]

tuft_markers = [
    "Dclk1", "Pou2f3", "Trpm5", "Gfi1b",
    "Il25", "Avil", "Krt18", "Krt8",
    "Rgs13", "Alox5ap", "Ano1"
]

enteroendocrine_markers = [
    "Chga", "Chgb", "Neurod1", "Neurog3",
    "Pax6", "Tph1", "Tac1",
    "Gcg", "Pyy", "Gip",
    "Sct", "Sst", "Ucn3",
    "Cck", "Ghrl", "Insm1"
]

#### top scanpy markers 
top_markers = compare_df[(compare_df.score_logreg >= 0.5) & (compare_df.significant_ttest == True)] #### contains cell_type and gene column

################################################################################
# 2) CONCATENATE ALL MARKERS FOR PLOTTING PURPOSES ONLY
################################################################################

all_marker_genes = (
    enterocyte_markers + enteroendocrine_markers    + goblet_markers    + paneth_markers    + stem_general_markers   + ta_markers    + tuft_markers) ######### add top_markers to this dict per cell type 

# Remove duplicates while preserving order
seen = set()
all_marker_genes_unique = []
for g in all_marker_genes:
    if g not in seen:
        seen.add(g)
        all_marker_genes_unique.append(g)





In [ ]:
#################### classical markers  #################### 
################################################################################
# 1) DEFINE ALL MARKER SETS IN DESIRED ORDER
################################################################################

# Define cell type order
cell_type_order = ["Stem", "TA", "Enterocyte", "Paneth", "Goblet", "Tuft", "EEC"]

# Define markers in the same order
marker_dict_ordered = {
    "Stem": [
        "Lgr5", "Olfm4", "Ascl2", "Sox9", "Axin2",
        "Tnfrsf19", "Ephb3", "Ephb2", "Smoc2", "Rnf43",
        "Tmem184b", "Tspan1"
    ],
    "TA": [
        "Mki67", "Top2a", "Ccnb1", "Ccnb2", "Cdk1",
        "Pcna", "Cdc20", "Cenpf", "Cdk4", "Birc5",
        "Cks1b", "Rrm2", "Tyms", "Uhrf1"
    ],
    "Enterocyte": [
        "Alpi", "Fabp1", "Fabp2", "Apoa1", "Apoa4",
        "Slc2a2", "Adh1", "Sis", "Mttp", "Acaa1a",
        "Krt20", "Reg1", "Aspscr1", "Pla2g2a",
        "Apoa2", "ApoC3", "Cyp2c70"
    ],
    "Paneth": [
        "Lyz1", "Lyz2", "Defa24", "Defa17", "Defa22",
        "Reg3b", "Reg3g", "Mmp7", "Casp4",
        "Ang4", "Pla2g2a", "Srp9", "Guca2a"
    ],
    "Goblet": [
        "Muc2", "Spdef", "Agr2", "Tff3",
        "Fcgbp", "Guca2a", "Dll1",
        "Klf4", "Muc13", "Muc4",
        "Clca1", "Tmem45b"
    ],
    "Tuft": [
        "Dclk1", "Pou2f3", "Trpm5", "Gfi1b",
        "Il25", "Avil", "Krt18", "Krt8",
        "Rgs13", "Alox5ap", "Ano1"
    ],
    "EEC": [
        "Chga", "Chgb", "Neurod1", "Neurog3",
        "Pax6", "Tph1", "Tac1",
        "Gcg", "Pyy", "Gip",
        "Sct", "Sst", "Ucn3",
        "Cck", "Ghrl", "Insm1"
    ]
}

################################################################################
# 2) FIND OVERLAPPING MARKERS ACROSS ALL THREE DATASETS
################################################################################

# Get genes present in all datasets
common_genes = (
    set(adata_ref.var_names) & 
    set(adata_dcm_plus.var_names) & 
    set(adata_rna.var_names)
)

print(f"Total common genes across datasets: {len(common_genes)}")

################################################################################
# 3) FIND TOP MARKERS IN DCM FOR EACH CELL TYPE
################################################################################

# Run differential expression on DCM
sc.tl.rank_genes_groups(adata_dcm_plus, groupby="leiden_annot_25", method='wilcoxon', pts=True)

# Extract top markers per cell type from DCM
def get_top_markers_from_dcm(adata, cell_type, candidate_genes, n_top=10):
    """
    Get genes that are highest expressed in the specified cell type in DCM.
    Only consider genes from the candidate list.
    """
    # Get the ranked genes for this cell type
    try:
        result = sc.get.rank_genes_groups_df(adata, group=cell_type)
        # Filter to only candidate genes
        result = result[result['names'].isin(candidate_genes)]
        # Sort by score (highest first)
        result = result.sort_values('scores', ascending=False)
        # Return top n genes
        return result['names'].head(n_top).tolist()
    except:
        return []

# Build filtered marker dict with only top DCM markers
marker_dict_dcm_top = {}
n_top_per_celltype = 5  # Adjust this number as needed

for cell_type in cell_type_order:
    # Get candidate genes for this cell type (from literature)
    candidate_genes = marker_dict_ordered.get(cell_type, [])
    # Filter to genes present in all datasets
    candidate_genes = [g for g in candidate_genes if g in common_genes]
    
    if candidate_genes:
        # Get top markers from DCM for this cell type
        top_genes = get_top_markers_from_dcm(
            adata_dcm_plus, 
            cell_type, 
            candidate_genes, 
            n_top=n_top_per_celltype
        )
        
        if top_genes:
            marker_dict_dcm_top[cell_type] = top_genes
        else:
            # Fallback: use all candidate genes if ranking fails
            marker_dict_dcm_top[cell_type] = candidate_genes[:n_top_per_celltype]
    
    print(f"{cell_type}: {marker_dict_dcm_top.get(cell_type, [])}")

# Create flat list in order
valid_markers_ordered = []
for cell_type in cell_type_order:
    if cell_type in marker_dict_dcm_top:
        valid_markers_ordered.extend(marker_dict_dcm_top[cell_type])

print(f"\nTotal markers to plot: {len(valid_markers_ordered)}")
print(f"Markers: {valid_markers_ordered}")

################################################################################
# 4) SET CATEGORY ORDER FOR CELL TYPES
################################################################################

# Reorder cell types in all datasets
for adata, groupby_col in [
    (adata_ref, "cell_type_broad"),
    (adata_dcm_plus, "leiden_annot_25"),
    (adata_rna, "leiden_annot_25")
]:
    # Get existing categories
    existing_cats = adata.obs[groupby_col].unique().tolist()
    # Order: first the ones in our list, then any others
    ordered_cats = [c for c in cell_type_order if c in existing_cats]
    ordered_cats += [c for c in existing_cats if c not in ordered_cats]
    
    # Set as ordered categorical
    adata.obs[groupby_col] = pd.Categorical(
        adata.obs[groupby_col],
        categories=ordered_cats,
        ordered=True
    )

################################################################################
# 5) PLOT SETTINGS
################################################################################

sc.set_figure_params(dpi=300, dpi_save=300, vector_friendly=True, fontsize=8)

################################################################################
# 6) RNA REFERENCE PLOT
################################################################################

sc.pl.matrixplot(
    adata_ref,
    var_names=marker_dict_dcm_top,  # Grouped by cell type
    groupby="cell_type_broad",
    cmap="coolwarm",
    layer="lognorm_scaled",
    standard_scale="var",
    dendrogram=False,
    title="RNA Reference — Top DCM markers",
    save="_ref_dcm_top_markers.svg"
)

################################################################################
# 7) DCM PLOT
################################################################################

sc.pl.matrixplot(
    adata_dcm_plus,
    var_names=marker_dict_dcm_top,
    groupby="leiden_annot_25",
    cmap="coolwarm",
    layer="lognorm_scaled",
    standard_scale="var",
    dendrogram=False,
    title="DCM — Top DCM markers",
    save="_dcm_top_markers.svg"
)

################################################################################
# 8) pRNA PLOT
################################################################################

sc.pl.matrixplot(
    adata_rna,
    var_names=marker_dict_dcm_top,
    groupby="leiden_annot_25",
    cmap="coolwarm",
    layer="lognorm_scaled",
    standard_scale="var",
    dendrogram=False,
    title="pRNA — Top DCM markers",
    save="_prna_dcm_top_markers.svg"
)

################################################################################
# 9) OPTIONAL: DOTPLOT VERSION
################################################################################

for adata, groupby_col, name in [
    (adata_ref, "cell_type_broad", "ref"),
    (adata_dcm_plus, "leiden_annot_25", "dcm"),
    (adata_rna, "leiden_annot_25", "prna")
]:
    sc.pl.dotplot(
        adata,
        var_names=marker_dict_dcm_top,
        groupby=groupby_col,
        cmap="coolwarm",
        layer="lognorm_scaled",
        standard_scale="var",
        dendrogram=False,
        title=f"{name.upper()} — Top DCM markers",
        save=f"_{name}_dcm_top_markers_dotplot.svg"
    )

################################################################################
# 10) OPTIONAL: SIDE-BY-SIDE COMPARISON
################################################################################

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

datasets = [
    (adata_ref, "RNA Reference", "cell_type_broad"),
    (adata_dcm_plus, "DCM", "leiden_annot_25"),
    (adata_rna, "pRNA", "leiden_annot_25")
]

for ax, (adata, title, groupby) in zip(axes, datasets):
    sc.pl.matrixplot(
        adata,
        var_names=marker_dict_dcm_top,
        groupby=groupby,
        cmap="coolwarm",
        layer="lognorm_scaled",
        standard_scale="var",
        dendrogram=False,
        title=title,
        ax=ax,
        show=False
    )

plt.tight_layout()
plt.savefig(f"{outdir}combined_dcm_top_markers.svg", format='svg', dpi=300, bbox_inches='tight')
plt.show()

################################################################################
# 11) PRINT SUMMARY
################################################################################

print("\n" + "="*60)
print("SUMMARY: Top DCM markers per cell type")
print("="*60)
for cell_type in cell_type_order:
    if cell_type in marker_dict_dcm_top:
        genes = marker_dict_dcm_top[cell_type]
        print(f"\n{cell_type}:")
        print(f"  {', '.join(genes)}")

In [ ]:
################################################### STRICT MARKER SET (LITERATURE + TOP DE MARKERS) ########################################
################################################################################
# 1) DEFINE ALL MARKER SETS CLEANLY
################################################################################

#### literature markers 
literature_markers = {
    'Enterocyte': [
        "Alpi", "Fabp1", "Fabp2", "Apoa1", "Apoa4",
        "Slc2a2", "Adh1", "Sis", "Mttp", "Acaa1a",
        "Krt20", "Reg1", "Aspscr1", "Pla2g2a",
        "Apoa2", "ApoC3", "Cyp2c70"
    ],
    # 'Immature_enterocyte': [
    #     "Slc52a3", "Guca2b", "Rprm", "Aldh3a1", "Anxa2",
    #     "Dmbt1", "Apoa1", "Krt8", "Krt18", "Mmp7",
    #     "Gstm3", "Cldn7", "Lgals3", "Klk1",
    #     "Pmp22", "Muc1"
    # ],
    'TA': [
        "Mki67", "Top2a", "Ccnb1", "Ccnb2", "Cdk1",
        "Pcna", "Cdc20", "Cenpf", "Cdk4", "Birc5",
        "Cks1b", "Rrm2", "Tyms", "Uhrf1"
    ],
    'Stem': [
        "Lgr5", "Olfm4", "Ascl2", "Sox9", "Axin2",
        "Tnfrsf19", "Ephb3", "Ephb2", "Smoc2", "Rnf43",
        "Tmem184b", "Tspan1"
    ],
    # 'Stem_quiescent': [
    #     "Hopx", "Bmi1", "Clu", "Mex3a", "Tert",
    #     "Areg", "Gadd45g", "Fos", "Jun",
    #     "Anxa1", "Anxa2"
    # ],
    'Goblet': [
        "Muc2", "Spdef", "Agr2", "Tff3",
        "Fcgbp", "Guca2a", "Dll1",
        "Klf4", "Muc13", "Muc4",
        "Clca1", "Tmem45b"
    ],
    'Paneth': [
        "Lyz1", "Lyz2", "Defa24", "Defa17", "Defa22",
        "Reg3b", "Reg3g", "Mmp7", "Casp4",
        "Ang4", "Pla2g2a", "Srp9", "Guca2a"
    ],
    'Tuft': [
        "Dclk1", "Pou2f3", "Trpm5", "Gfi1b",
        "Il25", "Avil", "Krt18", "Krt8",
        "Rgs13", "Alox5ap", "Ano1"
    ],
    'EEC': [
        "Chga", "Chgb", "Neurod1", "Neurog3",
        "Pax6", "Tph1", "Tac1",
        "Gcg", "Pyy", "Gip",
        "Sct", "Sst", "Ucn3",
        "Cck", "Ghrl", "Insm1"
    ]
}

#### top scanpy markers 
top_markers_df = compare_df[(compare_df.score_logreg >= 0.5) & (compare_df.significant_ttest == True)]

################################################################################
# 2) IDENTIFY AND REMOVE NON-SPECIFIC MARKERS
################################################################################

def create_specific_marker_dict(literature_dict, top_markers_df):
    """
    Combine literature markers and top markers, but EXCLUDE genes
    that appear in multiple cell types (non-specific markers).
    """
    
    print("=== PROCESSING MARKERS ===")
    
    # First, combine all markers to identify duplicates
    all_markers_flat = []
    gene_to_celltypes = {}
    
    # Process literature markers
    for cell_type, genes in literature_dict.items():
        for gene in genes:
            all_markers_flat.append(gene)
            if gene not in gene_to_celltypes:
                gene_to_celltypes[gene] = []
            if cell_type not in gene_to_celltypes[gene]:
                gene_to_celltypes[gene].append(cell_type)
    
    # Process top markers
    top_markers_by_celltype = top_markers_df.groupby('cell_type')['gene'].apply(list).to_dict()
    for cell_type, genes in top_markers_by_celltype.items():
        for gene in genes:
            all_markers_flat.append(gene)
            if gene not in gene_to_celltypes:
                gene_to_celltypes[gene] = []
            if cell_type not in gene_to_celltypes[gene]:
                gene_to_celltypes[gene].append(cell_type)
    
    # Identify non-specific markers (genes in multiple cell types)
    non_specific_genes = {gene: cts for gene, cts in gene_to_celltypes.items() if len(cts) > 1}
    
    print(f"\nFound {len(non_specific_genes)} non-specific markers (appear in multiple cell types):")
    for gene, cts in sorted(non_specific_genes.items()):
        print(f"  '{gene}' -> {cts}")
    
    # Create final dictionary with only specific markers
    specific_markers = {}
    
    # Process literature markers first (priority)
    for cell_type, genes in literature_dict.items():
        specific_genes = [g for g in genes if g not in non_specific_genes]
        specific_markers[cell_type] = {'literature': specific_genes, 'top_markers': []}
    
    # Process top markers, adding to existing cell types or creating new ones
    for cell_type, genes in top_markers_by_celltype.items():
        # Ensure cell type exists in dictionary
        if cell_type not in specific_markers:
            specific_markers[cell_type] = {'literature': [], 'top_markers': []}
        
        # Add only specific top markers
        specific_top_genes = [g for g in genes if g not in non_specific_genes]
        specific_markers[cell_type]['top_markers'].extend(specific_top_genes)
    
    # Remove cell types with no markers
    specific_markers = {k: v for k, v in specific_markers.items() 
                       if v['literature'] or v['top_markers']}
    
    # Print statistics
    print(f"\n=== SPECIFIC MARKER STATISTICS ===")
    total_specific_genes = 0
    for cell_type in sorted(specific_markers.keys()):
        lit_count = len(specific_markers[cell_type]['literature'])
        top_count = len(specific_markers[cell_type]['top_markers'])
        total = lit_count + top_count
        total_specific_genes += total
        print(f"  {cell_type}: {total} specific markers ({lit_count} literature, {top_count} top)")
    
    print(f"\nTotal specific markers: {total_specific_genes}")
    print(f"Non-specific markers excluded: {len(non_specific_genes)}")
    
    return specific_markers, non_specific_genes

# Create the specific marker dictionary
specific_markers, non_specific_genes = create_specific_marker_dict(literature_markers, top_markers_df)

################################################################################
# 3) CREATE FLAT LIST FOR PLOTTING (ONLY SPECIFIC MARKERS)
################################################################################

# Get all specific unique genes for plotting
all_specific_genes = set()
for cell_type, sources in specific_markers.items():
    all_specific_genes.update(sources['literature'])
    all_specific_genes.update(sources['top_markers'])

all_specific_genes_unique = sorted(list(all_specific_genes))
print(f"\nUnique specific markers for plotting: {len(all_specific_genes_unique)}")

################################################################################
# 4) CREATE MARKER ASSIGNMENT DATAFRAME
################################################################################

# Create comprehensive dataframe
marker_rows = []
for cell_type, sources in specific_markers.items():
    # Literature markers
    for gene in sources['literature']:
        marker_rows.append({
            'gene': gene,
            'cell_type': cell_type,
            'source': 'literature',
            'is_specific': True
        })
    
    # Top markers
    for gene in sources['top_markers']:
        marker_rows.append({
            'gene': gene,
            'cell_type': cell_type,
            'source': 'top_markers',
            'is_specific': True
        })

# Also track non-specific markers
for gene, cell_types in non_specific_genes.items():
    marker_rows.append({
        'gene': gene,
        'cell_type': '; '.join(cell_types),  # All cell types it appears in
        'source': 'multiple',
        'is_specific': False
    })

marker_assignments_df = pd.DataFrame(marker_rows)

# Separate dataframes
specific_markers_df = marker_assignments_df[marker_assignments_df['is_specific']].copy()
non_specific_markers_df = marker_assignments_df[~marker_assignments_df['is_specific']].copy()

print(f"\nSpecific markers dataframe: {specific_markers_df.shape}")
print(f"Non-specific markers dataframe: {non_specific_markers_df.shape}")

# Save to files
specific_markers_df.to_csv('specific_markers.csv', index=False)
non_specific_markers_df.to_csv('non_specific_markers_excluded.csv', index=False)

print("\nSaved to files:")
print("  - specific_markers.csv: Only markers specific to one cell type")
print("  - non_specific_markers_excluded.csv: Markers excluded (appear in multiple cell types)")

################################################################################
# 5) FILTER TO VALID GENES PER ASSAY
################################################################################

valid_markers_ref = [g for g in all_specific_genes_unique if g in adata_ref.var_names]
valid_markers_dcm = [g for g in all_specific_genes_unique if g in adata_dcm_plus.var_names]
valid_markers_rna = [g for g in all_specific_genes_unique if g in adata_rna.var_names]

print(f"\n=== VALIDATION ===")
print(f"RNA reference valid specific markers: {len(valid_markers_ref)} / {len(all_specific_genes_unique)}")
print(f"DCM valid specific markers: {len(valid_markers_dcm)} / {len(all_specific_genes_unique)}")

# Also check non-specific markers (for information)
non_specific_list = list(non_specific_genes.keys())
valid_non_specific_ref = [g for g in non_specific_list if g in adata_ref.var_names]
valid_non_specific_dcm = [g for g in non_specific_list if g in adata_dcm_plus.var_names]
valid_non_specific_rna = [g for g in non_specific_list if g in adata_rna.var_names]

print(f"\nNon-specific markers (excluded):")
print(f"  Total: {len(non_specific_list)}")
print(f"  Valid in RNA: {len(valid_non_specific_ref)}")
print(f"  Valid in DCM: {len(valid_non_specific_dcm)}")

# Show missing specific genes if any
missing_ref = set(all_specific_genes_unique) - set(valid_markers_ref)
missing_dcm = set(all_specific_genes_unique) - set(valid_markers_dcm)

if missing_ref:
    print(f"\nSpecific markers missing from RNA reference ({len(missing_ref)}):")
    for gene in sorted(missing_ref)[:10]:  # Show first 10
        print(f"  {gene}")
    if len(missing_ref) > 10:
        print(f"  ... and {len(missing_ref) - 10} more")

if missing_dcm:
    print(f"\nSpecific markers missing from DCM ({len(missing_dcm)}):")
    for gene in sorted(missing_dcm)[:10]:
        print(f"  {gene}")
    if len(missing_dcm) > 10:
        print(f"  ... and {len(missing_dcm) - 10} more")

################################################################################
# 6) RNA PLOT — All lineages with ONLY SPECIFIC markers
################################################################################

# Reorder genes by cell type for the matrixplot
valid_markers_ref = [g for ct in specific_markers.keys() for g in specific_markers[ct]['literature'] + specific_markers[ct]['top_markers'] if g in adata_ref.var_names]
valid_markers_dcm = [g for ct in specific_markers.keys() for g in specific_markers[ct]['literature'] + specific_markers[ct]['top_markers'] if g in adata_dcm_plus.var_names]
valid_markers_rna = [g for ct in specific_markers.keys() for g in specific_markers[ct]['literature'] + specific_markers[ct]['top_markers'] if g in adata_rna.var_names]

print("\n=== RNA REFERENCE MATRIXPLOT (SPECIFIC MARKERS ONLY) ===")

# Compute ranking and dendrogram for RNA reference
sc.tl.rank_genes_groups(adata_ref, groupby="cell_type_broad", pts=True)
sc.tl.dendrogram(adata_ref, groupby="cell_type_broad")

# Create matrixplot for RNA reference
sc.pl.rank_genes_groups_matrixplot(
    adata_ref,
    var_names=valid_markers_ref,
    cmap="coolwarm",
    layer="lognorm_scaled",
    standard_scale="var",
    dendrogram=False,
    title=f"RNA reference — Specific markers only\n({len(valid_markers_ref)} genes, {len(non_specific_genes)} non-specific excluded)",
    show=False
)

plt.gcf().set_size_inches(15, 10)
plt.tight_layout()
plt.show()

################################################################################
# 7) DCM PLOT — All lineages with ONLY SPECIFIC markers
################################################################################

print("\n=== DCM MATRIXPLOT (SPECIFIC MARKERS ONLY) ===")

# Compute ranking and dendrogram for DCM
sc.tl.rank_genes_groups(adata_dcm_plus, groupby="leiden_annot_25", pts=True)
sc.tl.dendrogram(adata_dcm_plus, groupby="leiden_annot_25")

# Create matrixplot for DCM
sc.pl.rank_genes_groups_matrixplot(
    adata_dcm_plus,
    var_names=valid_markers_dcm,
    cmap="coolwarm",
    layer="lognorm_scaled",
    standard_scale="var",
    dendrogram=False,
    title=f"DCM — Specific markers only\n({len(valid_markers_dcm)} genes, {len(non_specific_genes)} non-specific excluded)",
    show=False
)

plt.gcf().set_size_inches(15, 10)
plt.tight_layout()
plt.show()

################################################################################
# 7) RNA PLOT — All lineages with ONLY SPECIFIC markers
################################################################################

print("\n=== DCM MATRIXPLOT (SPECIFIC MARKERS ONLY) ===")

# Compute ranking and dendrogram for DCM
sc.tl.rank_genes_groups(adata_rna, groupby="leiden_annot_25", pts=True)
sc.tl.dendrogram(adata_rna, groupby="leiden_annot_25")

# Create matrixplot for DCM
sc.pl.rank_genes_groups_matrixplot(
    adata_rna,
    var_names=valid_markers_rna,
    cmap="coolwarm",
    layer="lognorm_scaled",
    standard_scale="var",
    dendrogram=False,
    title=f"pRNA — Specific markers only\n({len(valid_markers_dcm)} genes, {len(non_specific_genes)} non-specific excluded)",
    show=False
)

plt.gcf().set_size_inches(15, 10)
plt.tight_layout()
plt.show()


In [ ]:
################################################################################
# STRICT MARKER FILTERING: Overlap across assays + DCM validation
################################################################################

import pandas as pd
import numpy as np
#### literature markers 
literature_markers = {
    'Enterocyte': [
        "Alpi", "Fabp1", "Fabp2", "Apoa1", "Apoa4",
        "Slc2a2", "Adh1", "Sis", "Mttp", "Acaa1a",
        "Krt20", "Reg1", "Aspscr1", "Pla2g2a",
        "Apoa2", "ApoC3", "Cyp2c70"
    ],

    'TA': [
        "Mki67", "Top2a", "Ccnb1", "Ccnb2", "Cdk1",
        "Pcna", "Cdc20", "Cenpf", "Cdk4", "Birc5",
        "Cks1b", "Rrm2", "Tyms", "Uhrf1"
    ],
    'Stem': [
        "Lgr5", "Olfm4", "Ascl2", "Sox9", "Axin2",
        "Tnfrsf19", "Ephb3", "Ephb2", "Smoc2", "Rnf43",
        "Tmem184b", "Tspan1"
    ],

    'Goblet': [
        "Muc2", "Spdef", "Agr2", "Tff3",
        "Fcgbp", "Guca2a", "Dll1",
        "Klf4", "Muc13", "Muc4",
        "Clca1", "Tmem45b"
    ],
    'Paneth': [
        "Lyz1", "Lyz2", "Defa24", "Defa17", "Defa22",
        "Reg3b", "Reg3g", "Mmp7", "Casp4",
        "Ang4", "Pla2g2a", "Srp9", "Guca2a"
    ],
    'Tuft': [
        "Dclk1", "Pou2f3", "Trpm5", "Gfi1b",
        "Il25", "Avil", "Krt18", "Krt8",
        "Rgs13", "Alox5ap", "Ano1"
    ],
    'EEC': [
        "Chga", "Chgb", "Neurod1", "Neurog3",
        "Pax6", "Tph1", "Tac1",
        "Gcg", "Pyy", "Gip",
        "Sct", "Sst", "Ucn3",
        "Cck", "Ghrl", "Insm1"
    ]
}

################################################################################
# 1) FIND GENES OVERLAPPING IN ALL THREE ASSAYS
################################################################################
if 'Enteroendocrine' in literature_markers:
    literature_markers['EEC'] = literature_markers.pop('Enteroendocrine')
# Get valid genes per assay
genes_ref = set(adata_ref.var_names)
genes_dcm = set(adata_dcm_plus.var_names)
genes_rna = set(adata_rna.var_names)

# Find overlap across all three
genes_overlap = genes_ref & genes_dcm & genes_rna

print(f"Genes in RNA reference: {len(genes_ref)}")
print(f"Genes in DCM: {len(genes_dcm)}")
print(f"Genes in pRNA: {len(genes_rna)}")
print(f"Genes overlapping all three: {len(genes_overlap)}")

# Filter specific markers to only overlapping genes
overlapping_specific_markers = {}
for cell_type, sources in specific_markers.items():
    lit_genes = [g for g in sources['literature'] if g in genes_overlap]
    top_genes = [g for g in sources['top_markers'] if g in genes_overlap]
    if lit_genes or top_genes:
        overlapping_specific_markers[cell_type] = {
            'literature': lit_genes,
            'top_markers': top_genes
        }

# Print statistics
print(f"\n=== OVERLAPPING SPECIFIC MARKERS ===")
total_overlap = 0
for ct in sorted(overlapping_specific_markers.keys()):
    lit_count = len(overlapping_specific_markers[ct]['literature'])
    top_count = len(overlapping_specific_markers[ct]['top_markers'])
    total = lit_count + top_count
    total_overlap += total
    print(f"  {ct}: {total} markers ({lit_count} literature, {top_count} top)")
print(f"Total overlapping specific markers: {total_overlap}")

################################################################################
# 2) VALIDATE IN DCM: KEEP ONLY GENES HIGHEST IN EXPECTED CELL TYPE
################################################################################

def get_mean_expression_by_group(adata, genes, groupby, layer='lognorm'):
    """
    Calculate mean expression of genes per group.
    Returns a DataFrame with genes as rows, groups as columns.
    """
    # Use specified layer
    if layer and layer in adata.layers:
        X = adata[:, genes].layers[layer]
    else:
        X = adata[:, genes].X
    
    # Convert to dense if sparse
    if hasattr(X, 'toarray'):
        X = X.toarray()
    
    # Create DataFrame
    expr_df = pd.DataFrame(X, index=adata.obs_names, columns=genes)
    expr_df['group'] = adata.obs[groupby].values
    
    # Calculate mean per group
    mean_expr = expr_df.groupby('group').mean().T
    
    return mean_expr

# Map cell types to DCM cluster annotations
# First, extract base cell type from leiden_annot_25 (e.g., "0_Stem" -> "Stem")
def extract_celltype(annot):
    """Extract cell type from annotation like '0_Stem' -> 'Stem'"""
    import re
    match = re.sub(r'^\d+_', '', str(annot))
    return match

# Get unique cluster annotations and their base cell types
dcm_clusters = adata_dcm_plus.obs['leiden_annot_25'].unique()
cluster_to_celltype = {c: extract_celltype(c) for c in dcm_clusters}
print(f"\nDCM cluster to cell type mapping:")
for c, ct in sorted(cluster_to_celltype.items(), key=lambda x: int(x[0].split('_')[0]) if x[0].split('_')[0].isdigit() else 999):
    print(f"  {c} -> {ct}")

# Get all genes to validate
all_genes_to_validate = []
gene_to_expected_celltype = {}
for cell_type, sources in overlapping_specific_markers.items():
    for gene in sources['literature'] + sources['top_markers']:
        if gene not in gene_to_expected_celltype:
            all_genes_to_validate.append(gene)
            gene_to_expected_celltype[gene] = cell_type

print(f"\nValidating {len(all_genes_to_validate)} genes in DCM...")

# Calculate mean expression per cluster in DCM
mean_expr_dcm = get_mean_expression_by_group(
    adata_dcm_plus, 
    all_genes_to_validate, 
    groupby='leiden_annot_25',
    layer='lognorm'
)

# For each gene, check if highest expression is in expected cell type
validated_genes = []
failed_genes = []

for gene in all_genes_to_validate:
    expected_celltype = gene_to_expected_celltype[gene]
    
    # Get expression across clusters
    gene_expr = mean_expr_dcm.loc[gene]
    
    # Find cluster with highest expression
    max_cluster = gene_expr.idxmax()
    max_celltype = extract_celltype(max_cluster)
    
    # Check if it matches expected cell type
    if max_celltype == expected_celltype:
        validated_genes.append(gene)
    else:
        failed_genes.append({
            'gene': gene,
            'expected': expected_celltype,
            'highest_in': max_celltype,
            'highest_cluster': max_cluster,
            'expected_expr': gene_expr[[c for c in gene_expr.index if extract_celltype(c) == expected_celltype]].max() if any(extract_celltype(c) == expected_celltype for c in gene_expr.index) else 0,
            'highest_expr': gene_expr[max_cluster]
        })

print(f"\n=== DCM VALIDATION RESULTS ===")
print(f"Validated (highest in expected cell type): {len(validated_genes)}")
print(f"Failed (highest in different cell type): {len(failed_genes)}")

# Show failed genes
if failed_genes:
    failed_df = pd.DataFrame(failed_genes)
    print(f"\nFailed genes:")
    print(failed_df.to_string())
    failed_df.to_csv('failed_marker_validation.csv', index=False)

################################################################################
# 3) CREATE FINAL VALIDATED MARKER DICTIONARY
################################################################################

# Define cell type order
cell_type_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

validated_markers = {}
for cell_type, sources in overlapping_specific_markers.items():
    lit_validated = [g for g in sources['literature'] if g in validated_genes]
    top_validated = [g for g in sources['top_markers'] if g in validated_genes]
    if lit_validated or top_validated:
        validated_markers[cell_type] = {
            'literature': lit_validated,
            'top_markers': top_validated
        }

print(f"\n=== FINAL VALIDATED MARKERS ===")
total_validated = 0
for ct in cell_type_order:
    if ct not in validated_markers:
        continue
    lit_count = len(validated_markers[ct]['literature'])
    top_count = len(validated_markers[ct]['top_markers'])
    total = lit_count + top_count
    total_validated += total
    print(f"  {ct}: {total} markers ({lit_count} literature, {top_count} top)")
    # Print the actual genes
    all_genes = validated_markers[ct]['literature'] + validated_markers[ct]['top_markers']
    print(f"    Genes: {', '.join(all_genes)}")
print(f"\nTotal validated markers: {total_validated}")

# Create ordered gene list for plotting - IN SPECIFIED ORDER
validated_genes_ordered = []
for ct in cell_type_order:
    if ct in validated_markers:
        validated_genes_ordered.extend(validated_markers[ct]['literature'])
        validated_genes_ordered.extend(validated_markers[ct]['top_markers'])

# Remove duplicates while preserving order
validated_genes_ordered = list(dict.fromkeys(validated_genes_ordered))

print(f"\nFinal gene list for plotting: {len(validated_genes_ordered)} genes")
print(f"Order: {cell_type_order}")

# Save validated markers
validated_markers_rows = []
for cell_type, sources in validated_markers.items():
    for gene in sources['literature']:
        validated_markers_rows.append({'gene': gene, 'cell_type': cell_type, 'source': 'literature'})
    for gene in sources['top_markers']:
        validated_markers_rows.append({'gene': gene, 'cell_type': cell_type, 'source': 'top_markers'})

validated_markers_df = pd.DataFrame(validated_markers_rows)
validated_markers_df.to_csv('validated_markers_final.csv', index=False)
print(f"\nSaved validated markers to 'validated_markers_final.csv'")

###############################################################################
# 4) PLOT RNA REFERENCE WITH VALIDATED MARKERS
################################################################################

print("\n=== RNA REFERENCE MATRIXPLOT (VALIDATED MARKERS) ===")

sc.tl.rank_genes_groups(adata_ref, groupby="cell_type_broad", pts=True)
sc.tl.dendrogram(adata_ref, groupby="cell_type_broad")

sc.pl.rank_genes_groups_matrixplot(
    adata_ref,
    var_names=validated_genes_ordered,
    cmap="coolwarm",
    layer="lognorm_scaled",
    standard_scale="var",
    dendrogram=False,
    title=f"RNA reference — Validated markers ({len(validated_genes_ordered)} genes)",
    show=False
)

plt.gcf().set_size_inches(8, 5)
plt.tight_layout()
plt.savefig('matrixplot_ref_validated.pdf', dpi=300, bbox_inches='tight')
plt.close()

################################################################################
# 5) PLOT DCM WITH VALIDATED MARKERS
################################################################################

print("\n=== DCM MATRIXPLOT (VALIDATED MARKERS) ===")

sc.tl.rank_genes_groups(adata_dcm_plus, groupby="leiden_annot_25", pts=True)
sc.tl.dendrogram(adata_dcm_plus, groupby="leiden_annot_25")

sc.pl.rank_genes_groups_matrixplot(
    adata_dcm_plus,
    var_names=validated_genes_ordered,
    cmap="coolwarm",
    layer="lognorm_scaled",
    standard_scale="var",
    dendrogram=False,
    title=f"DCM — Validated markers ({len(validated_genes_ordered)} genes)",
    show=False
)

plt.gcf().set_size_inches(8, 5)
plt.tight_layout()
plt.savefig('matrixplot_dcm_validated.pdf', dpi=300, bbox_inches='tight')
plt.close()

################################################################################
# 6) PLOT pRNA WITH VALIDATED MARKERS
################################################################################

print("\n=== pRNA MATRIXPLOT (VALIDATED MARKERS) ===")

sc.tl.rank_genes_groups(adata_rna, groupby="leiden_annot_25", pts=True)
sc.tl.dendrogram(adata_rna, groupby="leiden_annot_25")

sc.pl.rank_genes_groups_matrixplot(
    adata_rna,
    var_names=validated_genes_ordered,
    cmap="coolwarm",
    layer="lognorm_scaled",
    standard_scale="var",
    dendrogram=False,
    title=f"pRNA — Validated markers ({len(validated_genes_ordered)} genes)",
    show=False
)

plt.gcf().set_size_inches(8, 5)
plt.tight_layout()
plt.savefig('matrixplot_rna_validated.pdf', dpi=300, bbox_inches='tight')
plt.close()

print("\nAll plots saved as PDF (300 dpi)")

In [ ]:
########################### GENE SCORING ###########################

In [ ]:
# First get significant genes
ttest_df_sig = ttest_df[(ttest_df.pvals_adj < 0.0001) & (ttest_df.logfc_ttest > 1)]

# Count genes per cell type before filtering duplicates
print("Before removing duplicates:")
print(ttest_df_sig.cell_type.value_counts())

# Find genes that appear in only ONE cell type
gene_counts = ttest_df_sig['gene'].value_counts()
unique_genes = gene_counts[gene_counts == 1].index

# Filter to keep only genes that are unique to one cell type
ttest_df_unique = ttest_df_sig[ttest_df_sig['gene'].isin(unique_genes)]

print("\nAfter removing duplicates:")
print(ttest_df_unique.cell_type.value_counts())


################################################################################
# 3) FILTER TO TOP 100 GENES PER CELL TYPE FROM TTEST_DF_UNIQUE
################################################################################

# Get top 100 genes per cell type based on logfc_ttest
top_genes_per_celltype = (
    ttest_df_unique
    .sort_values(['cell_type', 'logfc_ttest'], ascending=[True, False])
    .groupby('cell_type')
    .head(100)
)

# Get unique genes from the top 100 per cell type
all_marker_genes_unique = top_genes_per_celltype['gene'].unique()
print(f"Total unique marker genes from top 100 per cell type: {len(all_marker_genes_unique)}")

# Check which cell types are represented
cell_types_represented = top_genes_per_celltype['cell_type'].unique()
print(f"Cell types represented: {sorted(cell_types_represented)}")

# Show gene count per cell type
genes_per_celltype = top_genes_per_celltype.groupby('cell_type').size()
print("Genes per cell type:")
for cell_type, count in genes_per_celltype.items():
    print(f"  {cell_type}: {count} genes")

################################################################################
# 4) FILTER TO VALID GENES PER ASSAY
################################################################################

valid_markers_ref = [g for g in all_marker_genes_unique if g in adata_ref.var_names]
valid_markers_dcm = [g for g in all_marker_genes_unique if g in adata_dcm_plus.var_names]
valid_markers_rna = [g for g in all_marker_genes_unique if g in adata_rna.var_names]

print(f"RNA reference valid markers: {len(valid_markers_ref)} / {len(all_marker_genes_unique)}")
print(f"DCM valid markers: {len(valid_markers_dcm)} / {len(all_marker_genes_unique)}")
print(f"pRNA valid markers: {len(valid_markers_rna)} / {len(all_marker_genes_unique)}")

# Show missing genes if any
missing_ref = set(all_marker_genes_unique) - set(valid_markers_ref)
missing_dcm = set(all_marker_genes_unique) - set(valid_markers_dcm)
missing_rna = set(all_marker_genes_unique) - set(valid_markers_rna)

if missing_ref:
    print(f"Genes missing from RNA reference: {len(missing_ref)}")
if missing_dcm:
    print(f"Genes missing from DCM: {len(missing_dcm)}")
if missing_rna:
    print(f"Genes missing from pRNA: {len(missing_rna)}")

################################################################################
# 5) RNA PLOT — All lineages in a single matrixplot
################################################################################

print("\n=== RNA REFERENCE MATRIXPLOT ===")

# Compute ranking and dendrogram for RNA reference
sc.tl.rank_genes_groups(adata_ref, groupby="cell_type_broad", pts=True)
sc.tl.dendrogram(adata_ref, groupby="cell_type_broad")

# Create matrixplot for RNA reference
sc.pl.rank_genes_groups_matrixplot(
    adata_ref,
    var_names=valid_markers_ref,
    cmap="coolwarm",
    layer="lognorm_scaled",  # or use None for .X
    standard_scale="var",    # scale per gene (column)
    dendrogram=True,
    title=f"RNA reference — Top markers from t-test\n({len(valid_markers_ref)} genes, top 100 per cell type)",
    show=False
)

# Optional: Save the figure
plt.gcf().set_size_inches(12, 8)
plt.tight_layout()
plt.show()

################################################################################
# 6) DCM PLOT — All lineages in a single matrixplot
################################################################################

print("\n=== DCM MATRIXPLOT ===")

# Compute ranking and dendrogram for DCM
sc.tl.rank_genes_groups(adata_dcm_plus, groupby="leiden_annot_25", pts=True)
sc.tl.dendrogram(adata_dcm_plus, groupby="leiden_annot_25")

# Create matrixplot for DCM
sc.pl.rank_genes_groups_matrixplot(
    adata_dcm_plus,
    var_names=valid_markers_dcm,
    cmap="coolwarm",
    layer="lognorm_scaled",  # or use None for .X
    standard_scale="var",    # scale per gene (column)
    dendrogram=True,
    title=f"DCM — Top markers from t-test\n({len(valid_markers_dcm)} genes, top 100 per cell type)",
    show=False
)

# Optional: Save the figure
plt.gcf().set_size_inches(12, 8)
plt.tight_layout()
plt.show()
################################################################################
# 6) RNA PLOT — All lineages in a single matrixplot
################################################################################

print("\n=== pRNA MATRIXPLOT ===")

# Compute ranking and dendrogram for DCM
sc.tl.rank_genes_groups(adata_rna, groupby="leiden_annot_25", pts=True)
sc.tl.dendrogram(adata_rna, groupby="leiden_annot_25")

# Create matrixplot for DCM
sc.pl.rank_genes_groups_matrixplot(
    adata_rna,
    var_names=valid_markers_rna,
    cmap="coolwarm",
    layer="lognorm_scaled",  # or use None for .X
    standard_scale="var",    # scale per gene (column)
    dendrogram=True,
    title=f"pRNA — Top markers from t-test\n({len(valid_markers_rna)} genes, top 100 per cell type)",
    show=False
)

# Optional: Save the figure
plt.gcf().set_size_inches(12, 8)
plt.tight_layout()
plt.show()


In [ ]:
###########Filtered genes#################

In [ ]:
# First get significant genes
ttest_df_sig = ttest_df[(ttest_df.pvals_adj < 0.0001) & (ttest_df.logfc_ttest > 1)]

# Count genes per cell type before filtering duplicates
print("Before removing duplicates:")
print(ttest_df_sig.cell_type.value_counts())

# Find genes that appear in only ONE cell type
gene_counts = ttest_df_sig['gene'].value_counts()
unique_genes = gene_counts[gene_counts == 1].index

# Filter to keep only genes that are unique to one cell type
ttest_df_unique = ttest_df_sig[ttest_df_sig['gene'].isin(unique_genes)]

print("\nAfter removing duplicates:")
print(ttest_df_unique.cell_type.value_counts())

################################################################################
# 3) FILTER TO TOP 100 GENES PER CELL TYPE FROM TTEST_DF_UNIQUE
################################################################################

# Get top 100 genes per cell type based on logfc_ttest
top_genes_per_celltype = (
    ttest_df_unique
    .sort_values(['cell_type', 'logfc_ttest'], ascending=[True, False])
    .groupby('cell_type')
    .head(100)
)

# Get unique genes from the top 100 per cell type
all_marker_genes_unique = top_genes_per_celltype['gene'].unique()
print(f"Total unique marker genes from top 100 per cell type: {len(all_marker_genes_unique)}")

# Check which cell types are represented
cell_types_represented = top_genes_per_celltype['cell_type'].unique()
print(f"Cell types represented: {sorted(cell_types_represented)}")

# Show gene count per cell type
genes_per_celltype = top_genes_per_celltype.groupby('cell_type').size()
print("Genes per cell type:")
for cell_type, count in genes_per_celltype.items():
    print(f"  {cell_type}: {count} genes")

################################################################################
# 4) FIND GENES OVERLAPPING IN ALL THREE ASSAYS
################################################################################

genes_ref = set(adata_ref.var_names)
genes_dcm = set(adata_dcm_plus.var_names)
genes_rna = set(adata_rna.var_names)

# Find overlap across all three
genes_overlap = genes_ref & genes_dcm & genes_rna

# Filter to overlapping genes only
overlapping_marker_genes = [g for g in all_marker_genes_unique if g in genes_overlap]
print(f"\nGenes overlapping all three assays: {len(overlapping_marker_genes)} / {len(all_marker_genes_unique)}")

################################################################################
# 5) VALIDATE IN DCM: KEEP ONLY GENES HIGHEST IN EXPECTED CELL TYPE
################################################################################

def get_mean_expression_by_group(adata, genes, groupby, layer='lognorm'):
    """
    Calculate mean expression of genes per group.
    Returns a DataFrame with genes as rows, groups as columns.
    """
    if layer and layer in adata.layers:
        X = adata[:, genes].layers[layer]
    else:
        X = adata[:, genes].X
    
    if hasattr(X, 'toarray'):
        X = X.toarray()
    
    expr_df = pd.DataFrame(X, index=adata.obs_names, columns=genes)
    expr_df['group'] = adata.obs[groupby].values
    
    mean_expr = expr_df.groupby('group').mean().T
    
    return mean_expr

def extract_celltype(annot):
    """Extract cell type from annotation like '0_Stem' -> 'Stem'"""
    import re
    match = re.sub(r'^\d+_', '', str(annot))
    return match

# Get unique cluster annotations and their base cell types
dcm_clusters = adata_dcm_plus.obs['leiden_annot_25'].unique()
cluster_to_celltype = {c: extract_celltype(c) for c in dcm_clusters}
print(f"\nDCM cluster to cell type mapping:")
for c, ct in sorted(cluster_to_celltype.items(), key=lambda x: int(x[0].split('_')[0]) if x[0].split('_')[0].isdigit() else 999):
    print(f"  {c} -> {ct}")

# Create gene to expected cell type mapping from ttest results
gene_to_expected_celltype = dict(zip(top_genes_per_celltype['gene'], top_genes_per_celltype['cell_type']))

# Filter to only overlapping genes
genes_to_validate = [g for g in overlapping_marker_genes if g in gene_to_expected_celltype]
print(f"\nValidating {len(genes_to_validate)} genes in DCM...")

# Calculate mean expression per cluster in DCM
mean_expr_dcm = get_mean_expression_by_group(
    adata_dcm_plus, 
    genes_to_validate, 
    groupby='leiden_annot_25',
    layer='lognorm'
)

# For each gene, check if highest expression is in expected cell type
validated_genes = []
failed_genes = []

for gene in genes_to_validate:
    expected_celltype = gene_to_expected_celltype[gene]
    
    # Get expression across clusters
    gene_expr = mean_expr_dcm.loc[gene]
    
    # Find cluster with highest expression
    max_cluster = gene_expr.idxmax()
    max_celltype = extract_celltype(max_cluster)
    
    # Check if it matches expected cell type
    if max_celltype == expected_celltype:
        validated_genes.append(gene)
    else:
        failed_genes.append({
            'gene': gene,
            'expected': expected_celltype,
            'highest_in': max_celltype,
            'highest_cluster': max_cluster,
            'expected_expr': gene_expr[[c for c in gene_expr.index if extract_celltype(c) == expected_celltype]].max() if any(extract_celltype(c) == expected_celltype for c in gene_expr.index) else 0,
            'highest_expr': gene_expr[max_cluster]
        })

print(f"\n=== DCM VALIDATION RESULTS ===")
print(f"Validated (highest in expected cell type): {len(validated_genes)}")
print(f"Failed (highest in different cell type): {len(failed_genes)}")

# Show failed genes
if failed_genes:
    failed_df = pd.DataFrame(failed_genes)
    print(f"\nFailed genes summary:")
    print(failed_df['expected'].value_counts())
    failed_df.to_csv('failed_marker_validation_ttest.csv', index=False)

################################################################################
# 6) CREATE ORDERED VALIDATED GENE LIST
################################################################################

# Define cell type order
cell_type_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# Create validated markers per cell type
validated_markers_per_ct = {ct: [] for ct in cell_type_order}
for gene in validated_genes:
    ct = gene_to_expected_celltype[gene]
    if ct in validated_markers_per_ct:
        validated_markers_per_ct[ct].append(gene)

# Print statistics
print(f"\n=== FINAL VALIDATED MARKERS ===")
total_validated = 0
for ct in cell_type_order:
    count = len(validated_markers_per_ct[ct])
    total_validated += count
    print(f"  {ct}: {count} markers")
print(f"Total validated markers: {total_validated}")

# Create ordered gene list
validated_genes_ordered = []
for ct in cell_type_order:
    validated_genes_ordered.extend(validated_markers_per_ct[ct])

print(f"\nFinal gene list for plotting: {len(validated_genes_ordered)} genes")

# Save validated markers
validated_markers_df = pd.DataFrame([
    {'gene': gene, 'cell_type': gene_to_expected_celltype[gene]}
    for gene in validated_genes_ordered
])
validated_markers_df.to_csv('validated_markers_ttest_final.csv', index=False)

################################################################################
# 7) PLOT RNA REFERENCE WITH VALIDATED MARKERS
################################################################################

save_dir = '/media/erlantz/71DE167A75E99BB4/Pool32/Combi/Python/integration/'

print("\n=== RNA REFERENCE MATRIXPLOT (VALIDATED MARKERS) ===")

sc.tl.rank_genes_groups(adata_ref, groupby="cell_type_broad", pts=True)
sc.tl.dendrogram(adata_ref, groupby="cell_type_broad")

sc.pl.rank_genes_groups_matrixplot(
    adata_ref,
    var_names=validated_genes_ordered,
    cmap="coolwarm",
    layer="lognorm_scaled",
    standard_scale="var",
    dendrogram=False,
    title=f"RNA reference — Validated markers ({len(validated_genes_ordered)} genes)",
    show=False
)

plt.gcf().set_size_inches(8, 5)
plt.tight_layout()
plt.savefig(f'{save_dir}/matrixplot_ref_ttest_validated.pdf', dpi=300, bbox_inches='tight')
plt.close()

################################################################################
# 8) PLOT DCM WITH VALIDATED MARKERS
################################################################################

print("\n=== DCM MATRIXPLOT (VALIDATED MARKERS) ===")

sc.tl.rank_genes_groups(adata_dcm_plus, groupby="leiden_annot_25", pts=True)
sc.tl.dendrogram(adata_dcm_plus, groupby="leiden_annot_25")

sc.pl.rank_genes_groups_matrixplot(
    adata_dcm_plus,
    var_names=validated_genes_ordered,
    cmap="coolwarm",
    layer="lognorm_scaled",
    standard_scale="var",
    dendrogram=False,
    title=f"DCM — Validated markers ({len(validated_genes_ordered)} genes)",
    show=False
)

plt.gcf().set_size_inches(8, 5)
plt.tight_layout()
plt.savefig(f'{save_dir}/matrixplot_dcm_ttest_validated.pdf', dpi=300, bbox_inches='tight')
plt.close()

################################################################################
# 9) PLOT pRNA WITH VALIDATED MARKERS
################################################################################

print("\n=== pRNA MATRIXPLOT (VALIDATED MARKERS) ===")

sc.tl.rank_genes_groups(adata_rna, groupby="leiden_annot_25", pts=True)
sc.tl.dendrogram(adata_rna, groupby="leiden_annot_25")

sc.pl.rank_genes_groups_matrixplot(
    adata_rna,
    var_names=validated_genes_ordered,
    cmap="coolwarm",
    layer="lognorm_scaled",
    standard_scale="var",
    dendrogram=False,
    title=f"pRNA — Validated markers ({len(validated_genes_ordered)} genes)",
    show=False
)

plt.gcf().set_size_inches(8, 5)
plt.tight_layout()
plt.savefig(f'{save_dir}/matrixplot_rna_ttest_validated.pdf', dpi=300, bbox_inches='tight')
plt.close()

print("\nAll plots saved as PDF (300 dpi)")

In [ ]:
#### Module scoring for literature and ttest markers########

In [ ]:
########################################################## SIMPLE GENE SCORING, Z-SCORING ##########################################################

In [ ]:
########################################################## SIMPLE GENE SCORING, Z-SCORING ##########################################################
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
import scanpy as sc
import json

# Adjust label in adata_combined
adata_combined.obs['cell_type_broad'] = adata_combined.obs['leiden_annot_25']

# Set directory
score_dir = f'{root}/gene_scoring/'
os.makedirs(score_dir, exist_ok=True)

# Define label order and color palette
label_order = ['Stem', 'TA', 'Enterocyte', 'Goblet', 'Paneth', 'EEC', 'Tuft', 'ambiguous']

palette_dict = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': 'lightgray'
}
################################################################################
# 1) DEFINE MARKER SETS
################################################################################

# Use the specific_markers we just created (literature + top markers)
literature_top_markers = specific_markers  # From previous code

# Create ttest markers (filtering with your criteria)
ttest_markers_by_ct = {}
for ct in adata_combined.obs['cell_type_broad'].unique():
    if ct == 'ambiguous':
        continue
    ttest_ct = ttest_df_unique[
        (ttest_df_unique.cell_type == ct) & 
        (ttest_df_unique.logfc_ttest > 1) & 
        (ttest_df_unique.pvals_adj < 0.0001)
    ]
    if not ttest_ct.empty:
        ttest_markers_by_ct[ct] = ttest_ct['gene'].tolist()

# Create combined dictionary with all marker sets
all_marker_sets = {
    'literature_top': literature_top_markers,
    'ttest': ttest_markers_by_ct
}

print("=== MARKER SETS TO PROCESS ===")
print(f"1. Literature + Top markers: {len(literature_top_markers)} cell types")
print(f"2. T-test markers: {len(ttest_markers_by_ct)} cell types")

def process_marker_set(marker_set_name, gene_sets_dict, output_suffix):
    """
    Process a marker set through the scoring pipeline
    """
    print(f"\n{'='*60}")
    print(f"PROCESSING: {marker_set_name.upper()}")
    print(f"{'='*60}")
    
    # Create output directory
    set_dir = os.path.join(score_dir, marker_set_name)
    os.makedirs(set_dir, exist_ok=True)
    
    # SET .X TO LOGNORM
    adata_combined.X = adata_combined.layers['lognorm']
    
    # Subset to RNA + DCM cells only
    subset_mask = (adata_combined.obs['orig.ident'].str.contains('scNMT')) & \
                  (adata_combined.obs['Dox'] != 'Minus')
    adata_subset = adata_combined[subset_mask].copy()
    
    # Filter out genes not in data
    genes_in_data = set(adata_subset.var_names)
    filtered_gene_sets = {}
    for ct, genes in gene_sets_dict.items():
        if isinstance(genes, dict):  # Handle nested structure for literature_top
            lit_genes = [g for g in genes.get('literature', []) if g in genes_in_data]
            top_genes = [g for g in genes.get('top_markers', []) if g in genes_in_data]
            filtered_gene_sets[ct] = lit_genes + top_genes
        else:  # Simple list for ttest
            filtered_gene_sets[ct] = [g for g in genes if g in genes_in_data]
        
        if len(filtered_gene_sets[ct]) == 0:
            print(f"⚠️ No matching genes for {ct} after filtering!")
        else:
            print(f"  {ct}: {len(filtered_gene_sets[ct])} genes")
    
    # Initialize empty columns
    for ct in filtered_gene_sets.keys():
        adata_subset.obs[f'{ct}_score_raw'] = np.nan
        adata_subset.obs[f'{ct}_module_score_raw'] = np.nan
    
    # Compute both scoring types WITHIN each assay
    for assay in adata_subset.obs['orig.ident'].unique():
        print(f"  Processing assay: {assay}")
        sub = adata_subset[adata_subset.obs['orig.ident'] == assay].copy()
        
        for ct, genes in filtered_gene_sets.items():
            if len(genes) == 0:
                continue
            
            # (1) Custom mean expression score within assay
            sub.obs[f'{ct}_score_raw'] = np.asarray(sub[:, genes].X.mean(axis=1)).ravel()
            
            # (2) Scanpy module score within assay
            sc.tl.score_genes(
                sub,
                gene_list=genes,
                score_name=f'{ct}_module_score_raw',
                use_raw=False
            )
            
            # Merge results back
            adata_subset.obs.loc[sub.obs_names, f'{ct}_score_raw'] = sub.obs[f'{ct}_score_raw']
            adata_subset.obs.loc[sub.obs_names, f'{ct}_module_score_raw'] = sub.obs[f'{ct}_module_score_raw']
    
    # Z-score BOTH types of scores per assay
    for ct in filtered_gene_sets.keys():
        for prefix in ['_score_raw', '_module_score_raw']:
            raw_col = f'{ct}{prefix}'
            if raw_col not in adata_subset.obs:
                continue
            
            z_col = raw_col.replace('_raw', '_Z')
            adata_subset.obs[z_col] = np.nan
            
            for assay in adata_subset.obs['orig.ident'].unique():
                mask = adata_subset.obs['orig.ident'] == assay
                adata_subset.obs.loc[mask, z_col] = zscore(
                    adata_subset.obs.loc[mask, raw_col],
                    nan_policy='omit'
                )
    
    # Derive new annotation from z-scored module scores
    module_score_z_cols = [c for c in adata_subset.obs.columns 
                          if c.endswith('_module_score_Z')]
    
    print(f"  Found {len(module_score_z_cols)} module score Z columns")
    
    # Function to assign per-cell max module label and value
    def assign_module_label_per_cell(row):
        scores = row[module_score_z_cols]
        max_module = scores.idxmax()
        max_value = scores[max_module]
        label = max_module.replace('_module_score_Z', '')
        return pd.Series({'module_max_label': label, 'module_max_value': max_value})
    
    # Apply within each assay
    adata_subset.obs[['module_max_label', 'module_max_value']] = (
        adata_subset.obs.groupby('orig.ident')
        .apply(lambda df: df.apply(assign_module_label_per_cell, axis=1))
        .reset_index(level=0, drop=True)
    )
    
    # Apply confidence threshold (z > 1.65 → confident)
    confidence_threshold = 1.65
    adata_subset.obs['module_label_confident'] = np.where(
        adata_subset.obs['module_max_value'] >= confidence_threshold,
        adata_subset.obs['module_max_label'],
        'ambiguous'
    )
    
    print(f"  Added 'module_label_confident' (z < {confidence_threshold} → ambiguous)")
    
    # FIX: Clean up any problematic columns before saving
    # Remove any columns that might cause issues with H5AD saving
    problematic_cols = []
    for col in adata_subset.obs.columns:
        # Check for columns that might have mixed types
        try:
            # Try to convert to string to test
            _ = adata_subset.obs[col].astype(str)
        except:
            problematic_cols.append(col)
            print(f"  Warning: Column '{col}' might have data type issues")
    
    # If there are problematic columns, handle them
    if problematic_cols:
        print(f"  Found {len(problematic_cols)} problematic columns:")
        for col in problematic_cols:
            print(f"    - {col}")
            # Try to convert to string
            try:
                adata_subset.obs[col] = adata_subset.obs[col].astype(str)
                print(f"      Converted to string")
            except:
                print(f"      Could not convert, removing column")
                adata_subset.obs = adata_subset.obs.drop(columns=[col])
    
    # Visualize confusion matrices
    for assay in adata_subset.obs['orig.ident'].unique():
        mask = adata_subset.obs['orig.ident'] == assay
        cm = pd.crosstab(
            adata_subset.obs.loc[mask, 'cell_type_broad'],
            adata_subset.obs.loc[mask, 'module_label_confident'],
            rownames=['cell_type_broad'],
            colnames=['module_label_confident']
        )
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Cell count'})
        plt.title(f"{assay}: {marker_set_name} markers vs cell_type_broad_25")
        plt.tight_layout()
        
        save_path = os.path.join(set_dir, f"{assay}_confusion_matrix_{output_suffix}.png")
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
    
    # Create violin plots for each cell type
    for ct in filtered_gene_sets.keys():
        score_col = f'{ct}_score_Z'
        mod_col = f'{ct}_module_score_Z'
        
        if score_col not in adata_subset.obs or mod_col not in adata_subset.obs:
            continue
        
        for assay in adata_subset.obs['orig.ident'].unique():
            subset = adata_subset.obs[adata_subset.obs['orig.ident'] == assay]
            
            # Violin plots for each annotation
            for anno_col, label_name in [
                ('cell_type_broad', 'cell_type_broad'),
                ('module_label_confident', 'module_label_confident')
            ]:
                # Avg expression violin plot
                plt.figure(figsize=(6, 5))
                sns.violinplot(
                    data=subset,
                    x=anno_col,
                    y=score_col,
                    palette=palette_dict,
                    inner=None,
                    order=label_order
                )
                sns.stripplot(
                    data=subset,
                    x=anno_col,
                    y=score_col,
                    color='black',
                    size=2,
                    alpha=0.6,
                    order=label_order
                )
                plt.title(f"{ct} — {assay} ({marker_set_name} avg expr by {label_name})")
                plt.xticks(rotation=45)
                plt.tight_layout()
                
                save_path = os.path.join(
                    set_dir, f"{assay}_{ct}_avgExpr_{label_name}_{output_suffix}.png"
                )
                plt.savefig(save_path, dpi=300, bbox_inches='tight')
                plt.close()
                
                # Module score violin plot
                plt.figure(figsize=(6, 5))
                sns.violinplot(
                    data=subset,
                    x=anno_col,
                    y=mod_col,
                    palette=palette_dict,
                    inner=None,
                    order=label_order
                )
                sns.stripplot(
                    data=subset,
                    x=anno_col,
                    y=mod_col,
                    color='black',
                    size=2,
                    alpha=0.6,
                    order=label_order
                )
                plt.title(f"{ct} — {assay} ({marker_set_name} module score by {label_name})")
                plt.xticks(rotation=45)
                plt.tight_layout()
                
                save_path = os.path.join(
                    set_dir, f"{assay}_{ct}_moduleScore_{label_name}_{output_suffix}.png"
                )
                plt.savefig(save_path, dpi=300, bbox_inches='tight')
                plt.show()
                plt.close()
    
    # Store results for later use
    results = {
        'marker_set_name': marker_set_name,
        'gene_sets': filtered_gene_sets,
        'confusion_matrices': {},
        'scores_summary': {}
    }
    
    # Calculate confusion matrices for storage
    for assay in adata_subset.obs['orig.ident'].unique():
        mask = adata_subset.obs['orig.ident'] == assay
        cm = pd.crosstab(
            adata_subset.obs.loc[mask, 'cell_type_broad'],
            adata_subset.obs.loc[mask, 'module_label_confident']
        )
        results['confusion_matrices'][assay] = cm
        
        # Calculate accuracy
        total_cells = cm.values.sum()
        correct = cm.values.diagonal().sum()
        accuracy = correct / total_cells if total_cells > 0 else 0
        
        results['scores_summary'][assay] = {
            'total_cells': int(total_cells),
            'correct_assignments': int(correct),
            'accuracy': float(accuracy)
        }
    
    # FIX: Create a clean copy for saving (without problematic columns)
    adata_to_save = adata_subset.copy()
    
    # Remove any columns that still might cause issues
    columns_to_keep = []
    for col in adata_to_save.obs.columns:
        # Keep string, numeric, and categorical columns
        dtype = adata_to_save.obs[col].dtype
        if pd.api.types.is_string_dtype(dtype) or \
           pd.api.types.is_numeric_dtype(dtype) or \
           pd.api.types.is_categorical_dtype(dtype):
            columns_to_keep.append(col)
        else:
            print(f"  Removing column '{col}' with dtype {dtype} for saving")
    
    adata_to_save.obs = adata_to_save.obs[columns_to_keep]
    
    # Save the processed AnnData
    save_path = os.path.join(set_dir, f'processed_data_{output_suffix}.h5ad')
    print(f"  Saving to: {save_path}")
    try:
        adata_to_save.write_h5ad(save_path)
        print("  ✓ Successfully saved AnnData")
    except Exception as e:
        print(f"  ✗ Error saving AnnData: {e}")
        print("  Trying alternative save method...")
        # Try saving with different parameters
        try:
            adata_to_save.write_h5ad(
                save_path,
                compression='gzip',
                compression_opts={'level': 1}
            )
            print("  ✓ Successfully saved with compression")
        except Exception as e2:
            print(f"  ✗ Could not save AnnData: {e2}")
            print("  Will save results without AnnData object")
    
    # Save summary statistics
    summary_path = os.path.join(set_dir, f'summary_{output_suffix}.json')
    with open(summary_path, 'w') as f:
        json.dump({
            'marker_set_name': marker_set_name,
            'cell_types_processed': list(filtered_gene_sets.keys()),
            'genes_per_celltype': {ct: len(genes) for ct, genes in filtered_gene_sets.items()},
            'total_genes': sum(len(genes) for genes in filtered_gene_sets.values()),
            'confidence_threshold': confidence_threshold,
            'scores_summary': results['scores_summary']
        }, f, indent=2)
    
    print(f"✓ Completed processing {marker_set_name}")
    print(f"  Results saved to: {set_dir}")
    
    return results

################################################################################
# 3) PROCESS ALL MARKER SETS
################################################################################

all_results = {}

# Process literature + top markers
try:
    lit_top_results = process_marker_set(
        'literature_top',
        literature_top_markers,
        'specific_markers'
    )
    all_results['literature_top'] = lit_top_results
except Exception as e:
    print(f"Error processing literature_top markers: {e}")
    print("Skipping literature_top markers...")

# Process t-test markers
try:
    ttest_results = process_marker_set(
        'ttest',
        ttest_markers_by_ct,
        'ttest_markers'
    )
    all_results['ttest'] = ttest_results
except Exception as e:
    print(f"Error processing ttest markers: {e}")
    print("Skipping ttest markers...")

################################################################################
# 4) COMPARE RESULTS BETWEEN MARKER SETS (if both succeeded)
################################################################################

if len(all_results) >= 1:
    print(f"\n{'='*60}")
    print("COMPARISON BETWEEN MARKER SETS")
    print(f"{'='*60}")
    
    # Create comparison summary
    comparison_df = pd.DataFrame()
    
    for marker_set_name, results in all_results.items():
        for assay, summary in results['scores_summary'].items():
            comparison_df = pd.concat([
                comparison_df,
                pd.DataFrame({
                    'marker_set': [marker_set_name],
                    'assay': [assay],
                    'accuracy': [summary['accuracy']],
                    'total_cells': [summary['total_cells']],
                    'correct_assignments': [summary['correct_assignments']]
                })
            ])
    
    if not comparison_df.empty:
        # Pivot for easier comparison
        accuracy_comparison = comparison_df.pivot(
            index='assay', 
            columns='marker_set', 
            values='accuracy'
        )
        
        print("\nAccuracy comparison:")
        print(accuracy_comparison)
        
        # Create comparison plot
        if len(accuracy_comparison.columns) > 1:
            plt.figure(figsize=(10, 6))
            x = np.arange(len(accuracy_comparison))
            width = 0.35
            
            for i, marker_set in enumerate(accuracy_comparison.columns):
                plt.bar(x + i*width, accuracy_comparison[marker_set], 
                        width=width, label=marker_set, alpha=0.8)
            
            plt.xlabel('Assay')
            plt.ylabel('Accuracy')
            plt.title('Marker Set Performance Comparison')
            plt.xticks(x + width/2, accuracy_comparison.index)
            plt.legend()
            plt.grid(axis='y', alpha=0.3)
            plt.tight_layout()
            
            save_path = os.path.join(score_dir, 'marker_set_comparison.png')
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.show()
        else:
            print("Only one marker set processed, skipping comparison plot")
        
        ################################################################################
        # 5) SAVE COMPREHENSIVE RESULTS
        ################################################################################
        
        # Save all results to a pickle file for later analysis
        import pickle
        
        with open(os.path.join(score_dir, 'all_marker_set_results.pkl'), 'wb') as f:
            pickle.dump(all_results, f)
        
        # Create final summary report
        summary_report = {
            'timestamp': pd.Timestamp.now().isoformat(),
            'marker_sets_processed': list(all_results.keys()),
            'comparison': accuracy_comparison.to_dict() if not comparison_df.empty else {},
            'parameters': {
                'confidence_threshold': 1.65,
                'logfc_threshold': 1,
                'pval_threshold': 0.0001
            }
        }
        
        with open(os.path.join(score_dir, 'final_summary_report.json'), 'w') as f:
            json.dump(summary_report, f, indent=2)
        
        print(f"\n{'='*60}")
        print("PROCESSING COMPLETE")
        print(f"{'='*60}")
        print(f"✓ Processed {len(all_results)} marker set(s)")
        print(f"✓ All results saved to: {score_dir}")
        print(f"✓ Final summary: {os.path.join(score_dir, 'final_summary_report.json')}")
        print(f"✓ Raw results: {os.path.join(score_dir, 'all_marker_set_results.pkl')}")
        
        # Display final comparison
        if not comparison_df.empty:
            print("\nFINAL ACCURACY COMPARISON:")
            print(accuracy_comparison)
else:
    print("\nNo marker sets were successfully processed!")

print("\nDone!")

In [ ]:
########################################################## SIMPLE GENE SCORING, Z-SCORING ##########################################################
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
import scanpy as sc
import json

# Adjust label in adata_combined
adata_combined.obs['cell_type_broad'] = adata_combined.obs['leiden_annot_25']

# Set directory
score_dir = f'{root}/gene_scoring/'
os.makedirs(score_dir, exist_ok=True)

# Define label order and color palette
label_order = ['Stem', 'TA', 'Enterocyte', 'Goblet', 'Paneth', 'EEC', 'Tuft', 'ambiguous']
cell_type_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

palette_dict = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': 'lightgray'
}

################################################################################
# 1) DEFINE MARKER SETS
################################################################################

# Use the specific_markers from previous code
literature_top_markers = specific_markers

# Create ttest markers
ttest_markers_by_ct = {}
for ct in cell_type_order:
    ttest_ct = ttest_df_unique[
        (ttest_df_unique.cell_type == ct) & 
        (ttest_df_unique.logfc_ttest > 1) & 
        (ttest_df_unique.pvals_adj < 0.0001)
    ]
    if not ttest_ct.empty:
        ttest_markers_by_ct[ct] = ttest_ct['gene'].tolist()

# Create combined dictionary
all_marker_sets = {
    'literature': literature_top_markers,
    'ttest': ttest_markers_by_ct
}

print("=== MARKER SETS TO PROCESS ===")
for name, markers in all_marker_sets.items():
    print(f"{name}: {len(markers)} cell types")

################################################################################
# 2) INITIALIZE COLUMNS IN ADATA_COMBINED FOR ALL SCORES
################################################################################

# Initialize all score columns in adata_combined
for marker_set_name in all_marker_sets.keys():
    for ct in cell_type_order:
        # Module score columns
        adata_combined.obs[f'{marker_set_name}_{ct}_module'] = np.nan
        adata_combined.obs[f'{marker_set_name}_{ct}_zscore'] = np.nan
    
    # Max label columns
    adata_combined.obs[f'{marker_set_name}_max_celltype'] = 'Unknown'
    adata_combined.obs[f'{marker_set_name}_max_zscore'] = np.nan
    adata_combined.obs[f'{marker_set_name}_label_confident'] = 'Unknown'

print("Initialized score columns in adata_combined")

################################################################################
# 3) PROCESSING FUNCTION
################################################################################

def process_marker_set(marker_set_name, gene_sets_dict, output_suffix):
    """
    Process a marker set and save scores back to adata_combined
    """
    print(f"\n{'='*60}")
    print(f"PROCESSING: {marker_set_name.upper()}")
    print(f"{'='*60}")
    
    # Create output directory
    set_dir = os.path.join(score_dir, marker_set_name)
    os.makedirs(set_dir, exist_ok=True)
    
    # SET .X TO LOGNORM
    adata_combined.X = adata_combined.layers['lognorm']
    
    # Subset to scNMT cells only
    subset_mask = (adata_combined.obs['orig.ident'].str.contains('scNMT')) & \
                  (adata_combined.obs['Dox'] != 'Minus')
    adata_subset = adata_combined[subset_mask].copy()
    
    print(f"  Subset: {adata_subset.n_obs} cells")
    
    # Filter genes to those in data
    genes_in_data = set(adata_subset.var_names)
    filtered_gene_sets = {}
    
    for ct in cell_type_order:
        if ct not in gene_sets_dict:
            continue
        
        genes = gene_sets_dict[ct]
        if isinstance(genes, dict):  # Handle nested structure for literature
            lit_genes = [g for g in genes.get('literature', []) if g in genes_in_data]
            top_genes = [g for g in genes.get('top_markers', []) if g in genes_in_data]
            valid_genes = lit_genes + top_genes
        else:  # Simple list for ttest
            valid_genes = [g for g in genes if g in genes_in_data]
        
        if len(valid_genes) >= 2:
            filtered_gene_sets[ct] = valid_genes
            print(f"  {ct}: {len(valid_genes)} genes")
        else:
            print(f"  {ct}: Skipping - only {len(valid_genes)} valid genes")
    
    # Initialize columns in subset
    for ct in filtered_gene_sets.keys():
        adata_subset.obs[f'{marker_set_name}_{ct}_module'] = np.nan
    
    # Compute module scores WITHIN each assay
    for assay in adata_subset.obs['orig.ident'].unique():
        print(f"  Processing assay: {assay}")
        mask = adata_subset.obs['orig.ident'] == assay
        sub = adata_subset[mask].copy()
        
        for ct, genes in filtered_gene_sets.items():
            # Calculate module score
            score_col = f'{marker_set_name}_{ct}_module'
            sc.tl.score_genes(
                sub,
                gene_list=genes,
                score_name=score_col,
                use_raw=False
            )
            
            # Merge back to subset
            adata_subset.obs.loc[sub.obs_names, score_col] = sub.obs[score_col]
    
    # Z-score within each assay
    for ct in filtered_gene_sets.keys():
        module_col = f'{marker_set_name}_{ct}_module'
        zscore_col = f'{marker_set_name}_{ct}_zscore'
        adata_subset.obs[zscore_col] = np.nan
        
        for assay in adata_subset.obs['orig.ident'].unique():
            mask = adata_subset.obs['orig.ident'] == assay
            adata_subset.obs.loc[mask, zscore_col] = zscore(
                adata_subset.obs.loc[mask, module_col],
                nan_policy='omit'
            )
    
    # Get Z-score columns
    zscore_cols = [f'{marker_set_name}_{ct}_zscore' for ct in filtered_gene_sets.keys()]
    
    # Assign max label per cell
    def assign_max_label(row):
        scores = row[zscore_cols]
        if scores.isna().all():
            return pd.Series({
                f'{marker_set_name}_max_celltype': 'ambiguous',
                f'{marker_set_name}_max_zscore': np.nan
            })
        max_col = scores.idxmax()
        max_value = scores[max_col]
        label = max_col.replace(f'{marker_set_name}_', '').replace('_zscore', '')
        return pd.Series({
            f'{marker_set_name}_max_celltype': label,
            f'{marker_set_name}_max_zscore': max_value
        })
    
    max_labels = adata_subset.obs.apply(assign_max_label, axis=1)
    adata_subset.obs[f'{marker_set_name}_max_celltype'] = max_labels[f'{marker_set_name}_max_celltype']
    adata_subset.obs[f'{marker_set_name}_max_zscore'] = max_labels[f'{marker_set_name}_max_zscore']
    
    # Apply confidence threshold
    confidence_threshold = 1.65
    adata_subset.obs[f'{marker_set_name}_label_confident'] = np.where(
        adata_subset.obs[f'{marker_set_name}_max_zscore'] >= confidence_threshold,
        adata_subset.obs[f'{marker_set_name}_max_celltype'],
        'ambiguous'
    )
    
    print(f"  Added '{marker_set_name}_label_confident' (z < {confidence_threshold} → ambiguous)")
    
    ############################################################################
    # SAVE SCORES BACK TO ADATA_COMBINED
    ############################################################################
    
    # Get all score columns to transfer
    cols_to_transfer = []
    for ct in filtered_gene_sets.keys():
        cols_to_transfer.append(f'{marker_set_name}_{ct}_module')
        cols_to_transfer.append(f'{marker_set_name}_{ct}_zscore')
    cols_to_transfer.extend([
        f'{marker_set_name}_max_celltype',
        f'{marker_set_name}_max_zscore',
        f'{marker_set_name}_label_confident'
    ])
    
    # Transfer scores to adata_combined
    for col in cols_to_transfer:
        if col in adata_subset.obs.columns:
            adata_combined.obs.loc[adata_subset.obs_names, col] = adata_subset.obs[col]
    
    print(f"  Transferred {len(cols_to_transfer)} columns to adata_combined")
    
    ############################################################################
    # CONFUSION MATRICES
    ############################################################################
    
    results = {'confusion_matrices': {}, 'scores_summary': {}}
    
    for assay in adata_subset.obs['orig.ident'].unique():
        mask = adata_subset.obs['orig.ident'] == assay
        
        if mask.sum() == 0:
            print(f"  {assay}: No cells, skipping")
            continue
        
        cm = pd.crosstab(
            adata_subset.obs.loc[mask, 'cell_type_broad'],
            adata_subset.obs.loc[mask, f'{marker_set_name}_label_confident'],
            rownames=['cell_type_broad'],
            colnames=[f'{marker_set_name}_label_confident']
        )
        
        if cm.empty or cm.values.sum() == 0:
            print(f"  {assay}: Empty confusion matrix, skipping")
            continue
        
        results['confusion_matrices'][assay] = cm
        
        # Calculate accuracy
        total_cells = cm.values.sum()
        correct = 0
        for ct in cm.index:
            if ct in cm.columns:
                correct += cm.loc[ct, ct]
        accuracy = correct / total_cells if total_cells > 0 else 0
        
        results['scores_summary'][assay] = {
            'total_cells': int(total_cells),
            'correct': int(correct),
            'accuracy': float(accuracy)
        }
        
        # Plot confusion matrix
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', cbar_kws={'label': 'Cell count'})
        plt.title(f"{assay}: {marker_set_name} vs cell_type_broad (Acc: {accuracy:.2%})")
        plt.tight_layout()
        plt.savefig(os.path.join(set_dir, f'{assay}_confusion_matrix.pdf'), dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"  {assay}: Accuracy = {accuracy:.2%} ({correct}/{total_cells})")
    
    ############################################################################
    # VIOLIN PLOTS
    ############################################################################
    
    for ct in filtered_gene_sets.keys():
        zscore_col = f'{marker_set_name}_{ct}_zscore'
        
        if zscore_col not in adata_subset.obs.columns:
            continue
        
        for assay in adata_subset.obs['orig.ident'].unique():
            subset = adata_subset.obs[adata_subset.obs['orig.ident'] == assay]
            
            # Filter to present categories
            present_labels = [l for l in label_order if l in subset['cell_type_broad'].unique()]
            
            if len(present_labels) == 0:
                continue
            
            # Violin plot by cell_type_broad
            plt.figure(figsize=(6, 5))
            sns.violinplot(
                data=subset,
                x='cell_type_broad',
                y=zscore_col,
                palette=palette_dict,
                inner=None,
                order=present_labels
            )
            sns.stripplot(
                data=subset,
                x='cell_type_broad',
                y=zscore_col,
                color='black',
                size=2,
                alpha=0.6,
                order=present_labels
            )
            plt.axhline(y=1.65, color='red', linestyle='--', label='Threshold (1.65)')
            plt.title(f"{ct} Z-score — {assay} ({marker_set_name})")
            plt.xticks(rotation=45, ha='right')
            plt.legend()
            plt.tight_layout()
            plt.savefig(os.path.join(set_dir, f'{assay}_{ct}_zscore.pdf'), dpi=300, bbox_inches='tight')
            plt.close()
    
    # Save summary
    summary_path = os.path.join(set_dir, f'summary_{output_suffix}.json')
    with open(summary_path, 'w') as f:
        json.dump({
            'marker_set_name': marker_set_name,
            'cell_types_processed': list(filtered_gene_sets.keys()),
            'genes_per_celltype': {ct: len(genes) for ct, genes in filtered_gene_sets.items()},
            'confidence_threshold': confidence_threshold,
            'scores_summary': results['scores_summary']
        }, f, indent=2)
    
    print(f"✓ Completed {marker_set_name}")
    
    return results

################################################################################
# 4) PROCESS ALL MARKER SETS
################################################################################

all_results = {}

# Process literature markers
try:
    lit_results = process_marker_set(
        'literature',
        literature_top_markers,
        'literature_markers'
    )
    all_results['literature'] = lit_results
except Exception as e:
    print(f"Error processing literature markers: {e}")
    import traceback
    traceback.print_exc()

# Process ttest markers
try:
    ttest_results = process_marker_set(
        'ttest',
        ttest_markers_by_ct,
        'ttest_markers'
    )
    all_results['ttest'] = ttest_results
except Exception as e:
    print(f"Error processing ttest markers: {e}")
    import traceback
    traceback.print_exc()

################################################################################
# 5) PRINT SUMMARY OF COLUMNS ADDED TO ADATA_COMBINED
################################################################################

print(f"\n{'='*60}")
print("COLUMNS ADDED TO ADATA_COMBINED")
print(f"{'='*60}")

# List all new columns
new_cols = [col for col in adata_combined.obs.columns if 'literature_' in col or 'ttest_' in col]

print(f"\nTotal new columns: {len(new_cols)}")
print("\nModule score columns:")
module_cols = [col for col in new_cols if '_module' in col]
for col in sorted(module_cols):
    print(f"  - {col}")

print("\nZ-score columns:")
zscore_cols = [col for col in new_cols if '_zscore' in col and '_max_' not in col]
for col in sorted(zscore_cols):
    print(f"  - {col}")

print("\nLabel columns:")
label_cols = [col for col in new_cols if '_max_' in col or '_confident' in col]
for col in sorted(label_cols):
    print(f"  - {col}")

################################################################################
# 6) SAVE ADATA_COMBINED
################################################################################

print(f"\n{'='*60}")
print("SAVING ADATA_COMBINED")
print(f"{'='*60}")

# Save to h5ad
save_path = os.path.join(score_dir, 'adata_combined_with_scores.h5ad')
try:
    adata_combined.write_h5ad(save_path)
    print(f"✓ Saved adata_combined to: {save_path}")
except Exception as e:
    print(f"✗ Error saving: {e}")

# Save scores to CSV
scores_df = adata_combined.obs[['orig.ident', 'cell_type_broad'] + new_cols].copy()
scores_df.to_csv(os.path.join(score_dir, 'all_scores.csv'))
print(f"✓ Saved scores to: {os.path.join(score_dir, 'all_scores.csv')}")

################################################################################
# 7) COMPARISON BETWEEN MARKER SETS
################################################################################

if len(all_results) >= 2:
    print(f"\n{'='*60}")
    print("COMPARISON BETWEEN MARKER SETS")
    print(f"{'='*60}")
    
    comparison_rows = []
    for marker_set_name, results in all_results.items():
        for assay, summary in results['scores_summary'].items():
            comparison_rows.append({
                'marker_set': marker_set_name,
                'assay': assay,
                'accuracy': summary['accuracy'],
                'total_cells': summary['total_cells'],
                'correct': summary['correct']
            })
    
    comparison_df = pd.DataFrame(comparison_rows)
    
    if not comparison_df.empty:
        accuracy_pivot = comparison_df.pivot(index='assay', columns='marker_set', values='accuracy')
        print("\nAccuracy comparison:")
        print(accuracy_pivot)
        
        # Save comparison
        comparison_df.to_csv(os.path.join(score_dir, 'marker_set_comparison.csv'), index=False)
        
        # Plot comparison
        plt.figure(figsize=(10, 6))
        x = np.arange(len(accuracy_pivot))
        width = 0.35
        
        for i, marker_set in enumerate(accuracy_pivot.columns):
            plt.bar(x + i*width, accuracy_pivot[marker_set], width=width, label=marker_set, alpha=0.8)
        
        plt.xlabel('Assay')
        plt.ylabel('Accuracy')
        plt.title('Marker Set Performance Comparison')
        plt.xticks(x + width/2, accuracy_pivot.index, rotation=45, ha='right')
        plt.legend()
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(score_dir, 'marker_set_comparison.pdf'), dpi=300, bbox_inches='tight')
        plt.close()

print("\nDone!")

In [ ]:
ttest_markers = pd.read_csv(
    "/media/erlantz/71DE167A75E99BB4/Pool32/Combi/Python/integration/gene_scoring/marker_gene_sets/ttest_markers_13012026.csv"
)

In [ ]:
########################################################## 
# MODULE SCORING ACROSS ALL CELLS + UMAP PLOTS + BOXPLOTS
##########################################################
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import scanpy as sc

# Set directory
score_dir = f'{root}/gene_scoring_allcells/'
os.makedirs(score_dir, exist_ok=True)

cell_type_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

palette_dict = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
}

################################################################################
# 1) COMPUTE MODULE SCORES ACROSS ALL CELLS
################################################################################

# Convert ttest_markers DataFrame to dict: {cell_type: [gene1, gene2, ...]}
ttest_markers_dict = ttest_markers.groupby('cell_type')['gene'].apply(list).to_dict()
print(f"Converted ttest_markers to dict: {list(ttest_markers_dict.keys())}")

# Set .X to lognorm
adata_combined.X = adata_combined.layers['lognorm']

genes_in_data = set(adata_combined.var_names)

for ct in cell_type_order:
    if ct not in ttest_markers_dict:
        print(f"  {ct}: Not in ttest_markers_dict, skipping")
        continue
    
    genes = ttest_markers_dict[ct]
    valid_genes = [g for g in genes if g in genes_in_data]
    
    if len(valid_genes) < 2:
        print(f"  {ct}: Only {len(valid_genes)} valid genes, skipping")
        continue
    
    score_col = f'ttest_{ct}_allassays_MS'
    
    sc.tl.score_genes(
        adata_combined,
        gene_list=valid_genes,
        score_name=score_col,
        use_raw=False
    )
    
    print(f"  {ct}: Scored with {len(valid_genes)} genes -> {score_col}")

# Collect score columns that were actually computed
score_cols = [f'ttest_{ct}_allassays_MS' for ct in cell_type_order 
              if f'ttest_{ct}_allassays_MS' in adata_combined.obs.columns]

################################################################################
# HELPER: Build a blurred gradient colormap (light gray -> cell type color)
################################################################################

def make_gradient_cmap(hex_color, name='custom'):
    """Create a smooth light-to-color colormap for blurred gradient effect."""
    rgb = mcolors.to_rgb(hex_color)
    cdict = {
        'red':   [(0, 0.95, 0.95), (0.5, (0.95 + rgb[0]) / 2, (0.95 + rgb[0]) / 2), (1, rgb[0], rgb[0])],
        'green': [(0, 0.95, 0.95), (0.5, (0.95 + rgb[1]) / 2, (0.95 + rgb[1]) / 2), (1, rgb[1], rgb[1])],
        'blue':  [(0, 0.95, 0.95), (0.5, (0.95 + rgb[2]) / 2, (0.95 + rgb[2]) / 2), (1, rgb[2], rgb[2])],
    }
    return mcolors.LinearSegmentedColormap(name, cdict)

################################################################################
# 2) UMAP PLOTS — VECTOR-FRIENDLY SVG WITH BLURRED COLOR GRADIENT
################################################################################

print(f"\nPlotting UMAPs for {len(score_cols)} scores...")

# --- Individual UMAPs ---
for col in score_cols:
    ct_name = col.replace('ttest_', '').replace('_allassays_MS', '')
    
    fig, ax = plt.subplots(figsize=(5, 4.5))
    sc.pl.umap(
        adata_combined, color=col, cmap='RdBu_r', frameon=False,
        title=f'{ct_name} module score', show=False, ax=ax,
        sort_order=True,   # plot high scores on top
        size=15
    )
    fig.savefig(os.path.join(score_dir, f'umap_{col}.svg'), bbox_inches='tight')
    fig.savefig(os.path.join(score_dir, f'umap_{col}.pdf'), dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved umap_{col}.svg / .pdf")

# --- Combined panel ---
n_scores = len(score_cols)
ncols_panel = 4
nrows_panel = int(np.ceil(n_scores / ncols_panel))

fig, axes = plt.subplots(nrows_panel, ncols_panel,
                         figsize=(5 * ncols_panel, 4.5 * nrows_panel))
axes = np.atleast_2d(axes).flatten()

for i, col in enumerate(score_cols):
    ct_name = col.replace('ttest_', '').replace('_allassays_MS', '')
    sc.pl.umap(
        adata_combined, color=col, cmap='RdBu_r', frameon=False,
        title=ct_name, show=False, ax=axes[i],
        sort_order=True, size=10
    )

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
fig.savefig(os.path.join(score_dir, 'umap_all_scores_panel.svg'), bbox_inches='tight')
fig.savefig(os.path.join(score_dir, 'umap_all_scores_panel.pdf'), dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"✓ Saved combined UMAP panel (.svg + .pdf)")

################################################################################
# 3) BOXPLOTS — HIGHLIGHTED MATCHING CELL TYPE, REST GRAY
################################################################################

print(f"\nPlotting boxplots...")

# Determine which cell types are present in the data
present_cts = [ct for ct in cell_type_order 
               if ct in adata_combined.obs['cell_type_broad'].unique()]

# --- Individual boxplots ---
for col in score_cols:
    ct_name = col.replace('ttest_', '').replace('_allassays_MS', '')
    
    # Build color palette: matching ct gets its color, rest are gray
    box_palette = {ct: palette_dict[ct] if ct == ct_name else '#D3D3D3' 
                   for ct in present_cts}
    
    fig, ax = plt.subplots(figsize=(7, 5))
    
    sns.boxplot(
        data=adata_combined.obs,
        x='cell_type_broad',
        y=col,
        order=present_cts,
        palette=box_palette,
        fliersize=0,
        linewidth=1.2,
        ax=ax
    )
    sns.stripplot(
        data=adata_combined.obs,
        x='cell_type_broad',
        y=col,
        order=present_cts,
        color='black',
        size=1.5,
        alpha=0.3,
        jitter=True,
        ax=ax
    )
    
    ax.set_title(f'{ct_name} module score by cell type', fontsize=13)
    ax.set_xlabel('')
    ax.set_ylabel('Module score')
    ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    fig.savefig(os.path.join(score_dir, f'boxplot_{col}.svg'), bbox_inches='tight')
    fig.savefig(os.path.join(score_dir, f'boxplot_{col}.pdf'), dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved boxplot_{col}.svg / .pdf")

# --- Combined boxplot panel ---
fig, axes = plt.subplots(nrows_panel, ncols_panel,
                         figsize=(7 * ncols_panel, 5 * nrows_panel))
axes = np.atleast_2d(axes).flatten()

for i, col in enumerate(score_cols):
    ct_name = col.replace('ttest_', '').replace('_allassays_MS', '')
    box_palette = {ct: palette_dict[ct] if ct == ct_name else '#D3D3D3' 
                   for ct in present_cts}
    
    sns.boxplot(
        data=adata_combined.obs,
        x='cell_type_broad',
        y=col,
        order=present_cts,
        palette=box_palette,
        fliersize=0,
        linewidth=1,
        ax=axes[i]
    )
    sns.stripplot(
        data=adata_combined.obs,
        x='cell_type_broad',
        y=col,
        order=present_cts,
        color='black',
        size=1,
        alpha=0.2,
        jitter=True,
        ax=axes[i]
    )
    
    axes[i].set_title(ct_name, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Module score')
    axes[i].tick_params(axis='x', rotation=45)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
fig.savefig(os.path.join(score_dir, 'boxplot_all_scores_panel.svg'), bbox_inches='tight')
fig.savefig(os.path.join(score_dir, 'boxplot_all_scores_panel.pdf'), dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"✓ Saved combined boxplot panel (.svg + .pdf)")

################################################################################
# 4) SUMMARY
################################################################################

print(f"\n{'='*60}")
print("New columns added to adata_combined:")
for col in score_cols:
    print(f"  - {col}")
print(f"\nAll outputs saved to: {score_dir}")
print("Done!")

In [ ]:
##########################################################
# ISC (Stem) MODULE SCORE UMAP — ADJUSTABLE SCALE
##########################################################
import os
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc

score_dir = f'{root}/gene_scoring_allcells/'
os.makedirs(score_dir, exist_ok=True)

col = 'ttest_Stem_allassays_MS'

# ============================================================
# ADJUST THESE VALUES TO CHANGE THE COLOR SCALE
# ============================================================
vmin = 0.2    # lower bound of color scale
vmax = 1.0     # upper bound of color scale
# ============================================================

fig, ax = plt.subplots(figsize=(5, 4.5))
sc.pl.umap(
    adata_combined, color=col, cmap='RdBu_r', frameon=False,
    title='Stem module score',
    show=False, ax=ax,
    sort_order=True,
    size=15,
    vmin=vmin,
    vmax=vmax
)
fig.savefig(os.path.join(score_dir, f'umap_{col}_custom_scale.svg'), bbox_inches='tight')
fig.savefig(os.path.join(score_dir, f'umap_{col}_custom_scale.pdf'), dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"Saved with scale [{vmin}, {vmax}]")

In [ ]:
########################################################################### GMM ###########################################################################

In [ ]:
root = '/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/integration/GMM'

In [ ]:
########################################################################### GMM ###########################################################################
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
import os
from scipy.stats import zscore

# --- Parameters ---
posterior_threshold = 0.85  # threshold for discrete label
assay_col = 'assay'    # column defining assays (changed from 'assay')
jb_anno_col = 'leiden_annot_25'     # integration label

# Define color palette for consistency
palette_dict = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': 'lightgray'
}

# Cell type order
cell_type_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# Set directory
gmm_dir = f'{root}/gene_scoring/gmm_085/'
os.makedirs(gmm_dir, exist_ok=True)

################################################################################
# FUNCTION TO RUN GMM ANALYSIS ON DATA
################################################################################

def run_gmm_analysis(adata_subset, marker_set_name, output_dir, module_score_suffix, posterior_threshold=0.85):
    """
    Run GMM analysis on provided AnnData object
    
    Parameters:
    -----------
    adata_subset : AnnData
        Subset of data to analyze
    marker_set_name : str
        Name of marker set ('literature' or 'ttest')
    output_dir : str
        Directory to save outputs
    module_score_suffix : str
        Suffix to identify module score columns (e.g., '_module' or '_zscore')
    posterior_threshold : float
        Threshold for confident label assignment
    """
    print(f"\n    🔧 RUNNING GMM FOR: {marker_set_name.upper()}")
    print("    " + "=" * 40)
    
    # --- Identify module columns ---
    # Look for columns like 'literature_Stem_zscore' or 'ttest_Stem_zscore'
    module_cols = [c for c in adata_subset.obs.columns 
                   if c.startswith(f'{marker_set_name}_') and c.endswith(module_score_suffix)]
    
    print(f"    Found {len(module_cols)} module score columns:")
    for col in module_cols:
        print(f"      - {col}")
    
    if len(module_cols) == 0:
        print(f"    ⚠️ No module columns found with pattern '{marker_set_name}_*{module_score_suffix}'")
        return None
    
    # --- Function to fit mixture per module ---
    def fit_mixture_posterior(scores, n_components=2, random_state=0):
        """
        Fit 2-component Gaussian mixture to scores.
        Returns posterior probability of active component (higher mean).
        """
        # Remove NaN values
        valid_mask = ~np.isnan(scores)
        if valid_mask.sum() < 10:
            return np.full(len(scores), np.nan), None
        
        scores_clean = scores[valid_mask]
        scores_2d = scores_clean.reshape(-1, 1)
        
        gmm = GaussianMixture(n_components=n_components, random_state=random_state)
        gmm.fit(scores_2d)
        
        active_idx = np.argmax(gmm.means_.ravel())
        posterior_clean = gmm.predict_proba(scores_2d)[:, active_idx]
        
        # Map back to original indices
        posterior = np.full(len(scores), np.nan)
        posterior[valid_mask] = posterior_clean
        
        return posterior, gmm
    
    # --- Store assay-specific results ---
    assay_posterior_cols = {}
    assay_label_cols = {}
    all_assay_results = {}
    
    # --- Loop per assay ---
    for assay in adata_subset.obs[assay_col].unique():
        print(f"\n      🔹 PROCESSING ASSAY: {assay}")
        print("      " + "-" * 30)
        
        mask = adata_subset.obs[assay_col] == assay
        assay_data = adata_subset.obs.loc[mask]
        assay_posterior_cols[assay] = []
        assay_label_cols[assay] = []
        all_assay_results[assay] = {}
        
        # Pre-check module score distributions
        print(f"      Module score distributions:")
        for module_col in module_cols:
            scores = assay_data[module_col]
            print(f"        {module_col}: mean={scores.mean():.3f}, std={scores.std():.3f}, n={scores.notna().sum()}")
        
        for module_col in module_cols:
            # Extract cell type from column name (e.g., 'literature_Stem_zscore' -> 'Stem')
            module_name = module_col.replace(f'{marker_set_name}_', '').replace(module_score_suffix, '')
            print(f"\n        📊 Processing {module_name}...")
            
            # Fit mixture and compute posterior
            scores = assay_data[module_col].values
            posterior, gmm = fit_mixture_posterior(scores)
            
            if gmm is None:
                print(f"          ⚠️ Not enough data for GMM, skipping")
                continue
            
            # Create column names
            post_col = f"{marker_set_name}_{module_name}_posterior"
            label_col = f"{marker_set_name}_{module_name}_gmm_label"
            
            # Store results
            adata_subset.obs.loc[mask, post_col] = posterior
            adata_subset.obs.loc[mask, label_col] = np.where(
                posterior >= posterior_threshold,
                module_name,
                'ambiguous'
            )
            
            assay_posterior_cols[assay].append(post_col)
            assay_label_cols[assay].append(label_col)
            
            # Store GMM statistics
            all_assay_results[assay][module_name] = {
                'gmm_means': gmm.means_.ravel().tolist(),
                'gmm_covariances': gmm.covariances_.ravel().tolist(),
                'mean_posterior': np.nanmean(posterior),
                'std_posterior': np.nanstd(posterior),
                'n_above_threshold': int(np.nansum(posterior >= posterior_threshold)),
                'pct_above_threshold': float(np.nanmean(posterior >= posterior_threshold) * 100)
            }
            
            print(f"          GMM means: {gmm.means_.ravel()}")
            print(f"          Mean posterior: {np.nanmean(posterior):.3f}")
            print(f"          Cells above {posterior_threshold}: {all_assay_results[assay][module_name]['n_above_threshold']} ({all_assay_results[assay][module_name]['pct_above_threshold']:.1f}%)")
    
    # --- Assign max posterior labels ---
    print(f"\n      🎯 ASSIGNING MAX POSTERIOR LABELS")
    print("      " + "=" * 30)
    
    # Get all posterior columns
    all_posterior_cols = [f"{marker_set_name}_{ct}_posterior" for ct in cell_type_order 
                          if f"{marker_set_name}_{ct}_posterior" in adata_subset.obs.columns]
    
    print(f"      Posterior columns: {all_posterior_cols}")
    
    # Initialize columns
    adata_subset.obs[f'{marker_set_name}_gmm_max_celltype'] = 'ambiguous'
    adata_subset.obs[f'{marker_set_name}_gmm_max_posterior'] = np.nan
    adata_subset.obs[f'{marker_set_name}_gmm_label_confident'] = 'ambiguous'
    
    for assay in adata_subset.obs[assay_col].unique():
        print(f"\n        🔹 {assay}:")
        mask = adata_subset.obs[assay_col] == assay
        
        # Get posterior columns that exist for this marker set
        assay_post_cols = [col for col in all_posterior_cols if col in adata_subset.obs.columns]
        
        if len(assay_post_cols) == 0:
            print(f"          No posterior columns found, skipping")
            continue
        
        def assign_max_posterior(row):
            post_vals = row[assay_post_cols]
            if post_vals.isna().all():
                return pd.Series({
                    'max_celltype': 'ambiguous',
                    'max_posterior': np.nan,
                    'label_confident': 'ambiguous'
                })
            
            max_col = post_vals.idxmax()
            max_prob = post_vals.max()
            
            # Extract cell type from column name
            celltype = max_col.replace(f'{marker_set_name}_', '').replace('_posterior', '')
            
            if max_prob >= posterior_threshold:
                label = celltype
            else:
                label = 'ambiguous'
            
            return pd.Series({
                'max_celltype': celltype,
                'max_posterior': max_prob,
                'label_confident': label
            })
        
        # Apply to each cell
        results = adata_subset.obs.loc[mask].apply(assign_max_posterior, axis=1)
        
        adata_subset.obs.loc[mask, f'{marker_set_name}_gmm_max_celltype'] = results['max_celltype']
        adata_subset.obs.loc[mask, f'{marker_set_name}_gmm_max_posterior'] = results['max_posterior']
        adata_subset.obs.loc[mask, f'{marker_set_name}_gmm_label_confident'] = results['label_confident']
        
        # Print assignment statistics
        labels = adata_subset.obs.loc[mask, f'{marker_set_name}_gmm_label_confident']
        print(f"          Total cells: {mask.sum()}")
        print(f"          Label distribution:")
        for label, count in labels.value_counts().items():
            pct = (count / mask.sum()) * 100
            print(f"            {label}: {count} cells ({pct:.1f}%)")
    
    # --- Confusion matrices ---
    print(f"\n      📊 CONFUSION MATRICES")
    print("      " + "=" * 30)
    
    accuracy_results = {}
    
    for assay in adata_subset.obs[assay_col].unique():
        print(f"\n        🔹 {assay}: GMM labels vs {jb_anno_col}")
        mask = adata_subset.obs[assay_col] == assay
        
        if mask.sum() == 0:
            print(f"          No cells, skipping")
            continue
        
        cm = pd.crosstab(
            adata_subset.obs.loc[mask, jb_anno_col],
            adata_subset.obs.loc[mask, f'{marker_set_name}_gmm_label_confident'],
            rownames=[jb_anno_col],
            colnames=[f'{marker_set_name}_gmm_label']
        )
        
        if cm.empty or cm.values.sum() == 0:
            print(f"          Empty confusion matrix, skipping")
            continue
        
        # Calculate accuracy
        total_cells = cm.values.sum()
        correct = 0
        for ct in cm.index:
            if ct in cm.columns:
                correct += cm.loc[ct, ct]
        accuracy = (correct / total_cells * 100) if total_cells > 0 else 0
        
        accuracy_results[assay] = accuracy
        
        print(f"          Accuracy: {accuracy:.1f}% ({correct}/{total_cells})")
        
        # Plot confusion matrix
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', cbar_kws={'label': 'Number of cells'})
        plt.title(f"{marker_set_name.upper()} | {assay}: GMM Labels vs {jb_anno_col}\nAccuracy: {accuracy:.1f}% | Threshold: {posterior_threshold}")
        plt.tight_layout()
        
        save_path = os.path.join(output_dir, f'{assay}_GMM_confusion_matrix.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        # Save as CSV
        cm.to_csv(os.path.join(output_dir, f'{assay}_confusion_matrix.csv'))
    
    # --- Posterior distributions ---
    print(f"\n      📈 POSTERIOR DISTRIBUTIONS")
    print("      " + "=" * 30)
    
    for assay in adata_subset.obs[assay_col].unique():
        mask = adata_subset.obs[assay_col] == assay
        
        if mask.sum() == 0:
            continue
        
        for ct in cell_type_order:
            post_col = f"{marker_set_name}_{ct}_posterior"
            
            if post_col not in adata_subset.obs.columns:
                continue
            
            # Filter to present labels
            present_labels = [l for l in cell_type_order + ['ambiguous'] 
                             if l in adata_subset.obs.loc[mask, f'{marker_set_name}_gmm_label_confident'].unique()]
            
            if len(present_labels) == 0:
                continue
            
            plt.figure(figsize=(10, 6))
            sns.boxplot(
                data=adata_subset.obs.loc[mask],
                x=f'{marker_set_name}_gmm_label_confident',
                y=post_col,
                palette=palette_dict,
                order=present_labels
            )
            sns.stripplot(
                data=adata_subset.obs.loc[mask],
                x=f'{marker_set_name}_gmm_label_confident',
                y=post_col,
                color='black',
                size=2,
                alpha=0.3,
                order=present_labels
            )
            plt.axhline(y=posterior_threshold, color='red', linestyle='--', alpha=0.7, 
                       label=f'Threshold ({posterior_threshold})')
            plt.title(f"{marker_set_name.upper()} | {assay}: {ct} Posterior Distribution")
            plt.xticks(rotation=45, ha='right')
            plt.legend()
            plt.tight_layout()
            
            save_path = os.path.join(output_dir, f'{assay}_{ct}_posterior.pdf')
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close()
    
    # --- Save summary ---
    summary_stats = []
    
    for assay, modules in all_assay_results.items():
        for module_name, stats in modules.items():
            summary_stats.append({
                'marker_set': marker_set_name,
                'assay': assay,
                'module': module_name,
                **stats
            })
    
    if summary_stats:
        summary_df = pd.DataFrame(summary_stats)
        summary_df.to_csv(os.path.join(output_dir, 'GMM_summary.csv'), index=False)
        print(f"\n      ✅ Summary saved to: {os.path.join(output_dir, 'GMM_summary.csv')}")
    
    return {
        'adata_subset': adata_subset,
        'accuracy_results': accuracy_results,
        'all_assay_results': all_assay_results
    }

################################################################################
# MAIN PROCESSING
################################################################################

print("=" * 80)
print("RUNNING GAUSSIAN MIXTURE MODEL ANALYSIS")
print("=" * 80)

# Subset to scNMT cells only
subset_mask = (adata_combined.obs['orig.ident'].str.contains('scNMT')) & \
              (adata_combined.obs['Dox'] != 'Minus')
adata_gmm = adata_combined[subset_mask].copy()

print(f"Subset: {adata_gmm.n_obs} cells")

# Check available columns
print("\nAvailable score columns:")
lit_cols = [c for c in adata_gmm.obs.columns if c.startswith('literature_')]
ttest_cols = [c for c in adata_gmm.obs.columns if c.startswith('ttest_')]
print(f"  Literature columns: {len(lit_cols)}")
print(f"  T-test columns: {len(ttest_cols)}")

all_gmm_results = {}

################################################################################
# 1) LITERATURE MARKERS GMM
################################################################################

print(f"\n{'='*60}")
print("PROCESSING: LITERATURE MARKERS GMM")
print(f"{'='*60}")

lit_gmm_dir = os.path.join(gmm_dir, 'literature')
os.makedirs(lit_gmm_dir, exist_ok=True)

lit_results = run_gmm_analysis(
    adata_gmm,
    marker_set_name='literature',
    output_dir=lit_gmm_dir,
    module_score_suffix='_zscore',
    posterior_threshold=posterior_threshold
)

if lit_results:
    all_gmm_results['literature'] = lit_results

################################################################################
# 2) TTEST MARKERS GMM
################################################################################

print(f"\n{'='*60}")
print("PROCESSING: TTEST MARKERS GMM")
print(f"{'='*60}")

ttest_gmm_dir = os.path.join(gmm_dir, 'ttest')
os.makedirs(ttest_gmm_dir, exist_ok=True)

ttest_results = run_gmm_analysis(
    adata_gmm,
    marker_set_name='ttest',
    output_dir=ttest_gmm_dir,
    module_score_suffix='_zscore',
    posterior_threshold=posterior_threshold
)

if ttest_results:
    all_gmm_results['ttest'] = ttest_results

################################################################################
# 3) TRANSFER GMM RESULTS BACK TO ADATA_COMBINED
################################################################################

print(f"\n{'='*60}")
print("TRANSFERRING GMM RESULTS TO ADATA_COMBINED")
print(f"{'='*60}")

# Get all GMM-related columns
gmm_cols = [col for col in adata_gmm.obs.columns 
            if '_posterior' in col or '_gmm_' in col]

print(f"GMM columns to transfer: {len(gmm_cols)}")
for col in gmm_cols:
    print(f"  - {col}")

# Initialize columns in adata_combined
for col in gmm_cols:
    if col not in adata_combined.obs.columns:
        if 'posterior' in col or 'max_posterior' in col:
            adata_combined.obs[col] = np.nan
        else:
            adata_combined.obs[col] = 'Unknown'

# Transfer values
for col in gmm_cols:
    adata_combined.obs.loc[adata_gmm.obs_names, col] = adata_gmm.obs[col]

print(f"✅ Transferred {len(gmm_cols)} columns to adata_combined")

################################################################################
# 4) COMPARISON BETWEEN MARKER SETS
################################################################################

if len(all_gmm_results) >= 2:
    print(f"\n{'='*60}")
    print("COMPARISON BETWEEN MARKER SETS")
    print(f"{'='*60}")
    
    comparison_rows = []
    
    for marker_set_name, results in all_gmm_results.items():
        if results and 'accuracy_results' in results:
            for assay, accuracy in results['accuracy_results'].items():
                comparison_rows.append({
                    'marker_set': marker_set_name,
                    'assay': assay,
                    'accuracy': accuracy
                })
    
    if comparison_rows:
        comparison_df = pd.DataFrame(comparison_rows)
        
        # Pivot table
        accuracy_pivot = comparison_df.pivot(
            index='assay',
            columns='marker_set',
            values='accuracy'
        )
        
        print("\nGMM Accuracy Comparison:")
        print(accuracy_pivot.round(1))
        
        # Plot comparison
        plt.figure(figsize=(10, 6))
        x = np.arange(len(accuracy_pivot))
        width = 0.35
        
        for i, marker_set in enumerate(accuracy_pivot.columns):
            plt.bar(x + i*width, accuracy_pivot[marker_set], width=width, label=marker_set, alpha=0.8)
        
        plt.xlabel('Assay')
        plt.ylabel('Accuracy (%)')
        plt.title(f'GMM Accuracy Comparison (Threshold: {posterior_threshold})')
        plt.xticks(x + width/2, accuracy_pivot.index, rotation=45, ha='right')
        plt.legend()
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        
        plt.savefig(os.path.join(gmm_dir, 'GMM_comparison.pdf'), dpi=300, bbox_inches='tight')
        plt.close()
        
        # Save comparison
        comparison_df.to_csv(os.path.join(gmm_dir, 'GMM_comparison.csv'), index=False)

################################################################################
# 5) SAVE RESULTS
################################################################################

print(f"\n{'='*60}")
print("SAVING RESULTS")
print(f"{'='*60}")

# Save adata_gmm with all GMM results
try:
    save_path = os.path.join(gmm_dir, 'adata_gmm.h5ad')
    adata_gmm.write_h5ad(save_path)
    print(f"✅ Saved adata_gmm to: {save_path}")
except Exception as e:
    print(f"⚠️ Could not save adata_gmm: {e}")

# Save adata_combined with GMM columns
try:
    save_path = os.path.join(gmm_dir, 'adata_combined_with_gmm.h5ad')
    adata_combined.write_h5ad(save_path)
    print(f"✅ Saved adata_combined to: {save_path}")
except Exception as e:
    print(f"⚠️ Could not save adata_combined: {e}")

# Save GMM scores to CSV
gmm_scores_df = adata_combined.obs[['assay', 'cell_type_broad'] + gmm_cols].copy()
gmm_scores_df.to_csv(os.path.join(gmm_dir, 'gmm_scores.csv'))
print(f"✅ Saved GMM scores to: {os.path.join(gmm_dir, 'gmm_scores.csv')}")

# Print summary of columns added
print(f"\n{'='*60}")
print("COLUMNS ADDED TO ADATA_COMBINED")
print(f"{'='*60}")

print("\nPosterior columns:")
post_cols = [c for c in gmm_cols if '_posterior' in c and '_max_' not in c]
for col in sorted(post_cols):
    print(f"  - {col}")

print("\nGMM label columns:")
label_cols = [c for c in gmm_cols if '_gmm_' in c]
for col in sorted(label_cols):
    print(f"  - {col}")

print(f"\n{'='*60}")
print("GMM ANALYSIS COMPLETE!")
print(f"{'='*60}")

In [ ]:
############ GMM 085 ################

In [ ]:
########################################################################### GMM ###########################################################################
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
import os
from scipy.stats import zscore

# --- Parameters ---
posterior_threshold = 0.85  # threshold for discrete label
assay_col = 'assay'    # column defining assays (changed from 'assay')
jb_anno_col = 'leiden_annot_25'     # integration label

# Define color palette for consistency
palette_dict = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': 'lightgray'
}

# Cell type order
cell_type_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# Set directory
gmm_dir = f'{root}/gene_scoring/gmm_085/'
os.makedirs(gmm_dir, exist_ok=True)

################################################################################
# FUNCTION TO RUN GMM ANALYSIS ON DATA
################################################################################

def run_gmm_analysis(adata_subset, marker_set_name, output_dir, module_score_suffix, posterior_threshold=0.85):
    """
    Run GMM analysis on provided AnnData object
    
    Parameters:
    -----------
    adata_subset : AnnData
        Subset of data to analyze
    marker_set_name : str
        Name of marker set ('literature' or 'ttest')
    output_dir : str
        Directory to save outputs
    module_score_suffix : str
        Suffix to identify module score columns (e.g., '_module' or '_zscore')
    posterior_threshold : float
        Threshold for confident label assignment
    """
    print(f"\n    🔧 RUNNING GMM FOR: {marker_set_name.upper()}")
    print("    " + "=" * 40)
    
    # --- Identify module columns ---
    module_cols = [c for c in adata_subset.obs.columns 
                   if c.startswith(f'{marker_set_name}_') and c.endswith(module_score_suffix)]
    
    print(f"    Found {len(module_cols)} module score columns:")
    for col in module_cols:
        print(f"      - {col}")
    
    if len(module_cols) == 0:
        print(f"    ⚠️ No module columns found with pattern '{marker_set_name}_*{module_score_suffix}'")
        return None
    
    # --- Function to fit mixture per module ---
    def fit_mixture_posterior(scores, n_components=2, random_state=0):
        """
        Fit 2-component Gaussian mixture to scores.
        Returns posterior probability of active component (higher mean).
        """
        valid_mask = ~np.isnan(scores)
        if valid_mask.sum() < 10:
            return np.full(len(scores), np.nan), None
        
        scores_clean = scores[valid_mask]
        scores_2d = scores_clean.reshape(-1, 1)
        
        gmm = GaussianMixture(n_components=n_components, random_state=random_state)
        gmm.fit(scores_2d)
        
        active_idx = np.argmax(gmm.means_.ravel())
        posterior_clean = gmm.predict_proba(scores_2d)[:, active_idx]
        
        posterior = np.full(len(scores), np.nan)
        posterior[valid_mask] = posterior_clean
        
        return posterior, gmm
    
    # --- Store assay-specific results ---
    assay_posterior_cols = {}
    assay_label_cols = {}
    all_assay_results = {}
    
    # --- Loop per assay ---
    for assay in adata_subset.obs[assay_col].unique():
        print(f"\n      🔹 PROCESSING ASSAY: {assay}")
        print("      " + "-" * 30)
        
        mask = adata_subset.obs[assay_col] == assay
        assay_data = adata_subset.obs.loc[mask]
        assay_posterior_cols[assay] = []
        assay_label_cols[assay] = []
        all_assay_results[assay] = {}
        
        print(f"      Module score distributions:")
        for module_col in module_cols:
            scores = assay_data[module_col]
            print(f"        {module_col}: mean={scores.mean():.3f}, std={scores.std():.3f}, n={scores.notna().sum()}")
        
        for module_col in module_cols:
            module_name = module_col.replace(f'{marker_set_name}_', '').replace(module_score_suffix, '')
            print(f"\n        📊 Processing {module_name}...")
            
            scores = assay_data[module_col].values
            posterior, gmm = fit_mixture_posterior(scores)
            
            if gmm is None:
                print(f"          ⚠️ Not enough data for GMM, skipping")
                continue
            
            # Create column names with 085 suffix
            post_col = f"{marker_set_name}_{module_name}_posterior_085"
            label_col = f"{marker_set_name}_{module_name}_gmm_label_085"
            
            # Store results
            adata_subset.obs.loc[mask, post_col] = posterior
            adata_subset.obs.loc[mask, label_col] = np.where(
                posterior >= posterior_threshold,
                module_name,
                'ambiguous'
            )
            
            assay_posterior_cols[assay].append(post_col)
            assay_label_cols[assay].append(label_col)
            
            # Store GMM statistics
            all_assay_results[assay][module_name] = {
                'gmm_means': gmm.means_.ravel().tolist(),
                'gmm_covariances': gmm.covariances_.ravel().tolist(),
                'mean_posterior': np.nanmean(posterior),
                'std_posterior': np.nanstd(posterior),
                'n_above_threshold': int(np.nansum(posterior >= posterior_threshold)),
                'pct_above_threshold': float(np.nanmean(posterior >= posterior_threshold) * 100)
            }
            
            print(f"          GMM means: {gmm.means_.ravel()}")
            print(f"          Mean posterior: {np.nanmean(posterior):.3f}")
            print(f"          Cells above {posterior_threshold}: {all_assay_results[assay][module_name]['n_above_threshold']} ({all_assay_results[assay][module_name]['pct_above_threshold']:.1f}%)")
    
    # --- Assign max posterior labels ---
    print(f"\n      🎯 ASSIGNING MAX POSTERIOR LABELS")
    print("      " + "=" * 30)
    
    # Get all posterior columns with 085 suffix
    all_posterior_cols = [f"{marker_set_name}_{ct}_posterior_085" for ct in cell_type_order 
                          if f"{marker_set_name}_{ct}_posterior_085" in adata_subset.obs.columns]
    
    print(f"      Posterior columns: {all_posterior_cols}")
    
    # Initialize columns with 085 suffix
    adata_subset.obs[f'{marker_set_name}_gmm_max_celltype_085'] = 'ambiguous'
    adata_subset.obs[f'{marker_set_name}_gmm_max_posterior_085'] = np.nan
    adata_subset.obs[f'{marker_set_name}_gmm_label_confident_085'] = 'ambiguous'
    
    for assay in adata_subset.obs[assay_col].unique():
        print(f"\n        🔹 {assay}:")
        mask = adata_subset.obs[assay_col] == assay
        
        assay_post_cols = [col for col in all_posterior_cols if col in adata_subset.obs.columns]
        
        if len(assay_post_cols) == 0:
            print(f"          No posterior columns found, skipping")
            continue
        
        def assign_max_posterior(row):
            post_vals = row[assay_post_cols]
            if post_vals.isna().all():
                return pd.Series({
                    'max_celltype': 'ambiguous',
                    'max_posterior': np.nan,
                    'label_confident': 'ambiguous'
                })
            
            max_col = post_vals.idxmax()
            max_prob = post_vals.max()
            
            # Extract cell type from column name with 085 suffix
            celltype = max_col.replace(f'{marker_set_name}_', '').replace('_posterior_085', '')
            
            if max_prob >= posterior_threshold:
                label = celltype
            else:
                label = 'ambiguous'
            
            return pd.Series({
                'max_celltype': celltype,
                'max_posterior': max_prob,
                'label_confident': label
            })
        
        results = adata_subset.obs.loc[mask].apply(assign_max_posterior, axis=1)
        
        adata_subset.obs.loc[mask, f'{marker_set_name}_gmm_max_celltype_085'] = results['max_celltype']
        adata_subset.obs.loc[mask, f'{marker_set_name}_gmm_max_posterior_085'] = results['max_posterior']
        adata_subset.obs.loc[mask, f'{marker_set_name}_gmm_label_confident_085'] = results['label_confident']
        
        labels = adata_subset.obs.loc[mask, f'{marker_set_name}_gmm_label_confident_085']
        print(f"          Total cells: {mask.sum()}")
        print(f"          Label distribution:")
        for label, count in labels.value_counts().items():
            pct = (count / mask.sum()) * 100
            print(f"            {label}: {count} cells ({pct:.1f}%)")
    
    # --- Confusion matrices ---
    print(f"\n      📊 CONFUSION MATRICES")
    print("      " + "=" * 30)
    
    accuracy_results = {}
    
    for assay in adata_subset.obs[assay_col].unique():
        print(f"\n        🔹 {assay}: GMM labels vs {jb_anno_col}")
        mask = adata_subset.obs[assay_col] == assay
        
        if mask.sum() == 0:
            print(f"          No cells, skipping")
            continue
        
        cm = pd.crosstab(
            adata_subset.obs.loc[mask, jb_anno_col],
            adata_subset.obs.loc[mask, f'{marker_set_name}_gmm_label_confident_085'],
            rownames=[jb_anno_col],
            colnames=[f'{marker_set_name}_gmm_label_085']
        )
        
        if cm.empty or cm.values.sum() == 0:
            print(f"          Empty confusion matrix, skipping")
            continue
        
        total_cells = cm.values.sum()
        correct = 0
        for ct in cm.index:
            if ct in cm.columns:
                correct += cm.loc[ct, ct]
        accuracy = (correct / total_cells * 100) if total_cells > 0 else 0
        
        accuracy_results[assay] = accuracy
        
        print(f"          Accuracy: {accuracy:.1f}% ({correct}/{total_cells})")
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', cbar_kws={'label': 'Number of cells'})
        plt.title(f"{marker_set_name.upper()} | {assay}: GMM Labels vs {jb_anno_col}\nAccuracy: {accuracy:.1f}% | Threshold: {posterior_threshold}")
        plt.tight_layout()
        
        save_path = os.path.join(output_dir, f'{assay}_GMM_confusion_matrix.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        cm.to_csv(os.path.join(output_dir, f'{assay}_confusion_matrix.csv'))
    
    # --- Posterior distributions ---
    print(f"\n      📈 POSTERIOR DISTRIBUTIONS")
    print("      " + "=" * 30)
    
    for assay in adata_subset.obs[assay_col].unique():
        mask = adata_subset.obs[assay_col] == assay
        
        if mask.sum() == 0:
            continue
        
        for ct in cell_type_order:
            post_col = f"{marker_set_name}_{ct}_posterior_085"
            
            if post_col not in adata_subset.obs.columns:
                continue
            
            present_labels = [l for l in cell_type_order + ['ambiguous'] 
                             if l in adata_subset.obs.loc[mask, f'{marker_set_name}_gmm_label_confident_085'].unique()]
            
            if len(present_labels) == 0:
                continue
            
            plt.figure(figsize=(10, 6))
            sns.boxplot(
                data=adata_subset.obs.loc[mask],
                x=f'{marker_set_name}_gmm_label_confident_085',
                y=post_col,
                palette=palette_dict,
                order=present_labels
            )
            sns.stripplot(
                data=adata_subset.obs.loc[mask],
                x=f'{marker_set_name}_gmm_label_confident_085',
                y=post_col,
                color='black',
                size=2,
                alpha=0.3,
                order=present_labels
            )
            plt.axhline(y=posterior_threshold, color='red', linestyle='--', alpha=0.7, 
                       label=f'Threshold ({posterior_threshold})')
            plt.title(f"{marker_set_name.upper()} | {assay}: {ct} Posterior Distribution")
            plt.xticks(rotation=45, ha='right')
            plt.legend()
            plt.tight_layout()
            
            save_path = os.path.join(output_dir, f'{assay}_{ct}_posterior.pdf')
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close()
    
    # --- Save summary ---
    summary_stats = []
    
    for assay, modules in all_assay_results.items():
        for module_name, stats in modules.items():
            summary_stats.append({
                'marker_set': marker_set_name,
                'assay': assay,
                'module': module_name,
                **stats
            })
    
    if summary_stats:
        summary_df = pd.DataFrame(summary_stats)
        summary_df.to_csv(os.path.join(output_dir, 'GMM_summary.csv'), index=False)
        print(f"\n      ✅ Summary saved to: {os.path.join(output_dir, 'GMM_summary.csv')}")
    
    return {
        'adata_subset': adata_subset,
        'accuracy_results': accuracy_results,
        'all_assay_results': all_assay_results
    }

################################################################################
# MAIN PROCESSING
################################################################################

print("=" * 80)
print("RUNNING GAUSSIAN MIXTURE MODEL ANALYSIS")
print("=" * 80)

subset_mask = (adata_combined.obs['orig.ident'].str.contains('scNMT')) & \
              (adata_combined.obs['Dox'] != 'Minus')
adata_gmm = adata_combined[subset_mask].copy()

print(f"Subset: {adata_gmm.n_obs} cells")

print("\nAvailable score columns:")
lit_cols = [c for c in adata_gmm.obs.columns if c.startswith('literature_')]
ttest_cols = [c for c in adata_gmm.obs.columns if c.startswith('ttest_')]
print(f"  Literature columns: {len(lit_cols)}")
print(f"  T-test columns: {len(ttest_cols)}")

all_gmm_results = {}

################################################################################
# 1) LITERATURE MARKERS GMM
################################################################################

print(f"\n{'='*60}")
print("PROCESSING: LITERATURE MARKERS GMM")
print(f"{'='*60}")

lit_gmm_dir = os.path.join(gmm_dir, 'literature')
os.makedirs(lit_gmm_dir, exist_ok=True)

lit_results = run_gmm_analysis(
    adata_gmm,
    marker_set_name='literature',
    output_dir=lit_gmm_dir,
    module_score_suffix='_zscore',
    posterior_threshold=posterior_threshold
)

if lit_results:
    all_gmm_results['literature'] = lit_results

################################################################################
# 2) TTEST MARKERS GMM
################################################################################

print(f"\n{'='*60}")
print("PROCESSING: TTEST MARKERS GMM")
print(f"{'='*60}")

ttest_gmm_dir = os.path.join(gmm_dir, 'ttest')
os.makedirs(ttest_gmm_dir, exist_ok=True)

ttest_results = run_gmm_analysis(
    adata_gmm,
    marker_set_name='ttest',
    output_dir=ttest_gmm_dir,
    module_score_suffix='_zscore',
    posterior_threshold=posterior_threshold
)

if ttest_results:
    all_gmm_results['ttest'] = ttest_results

################################################################################
# 3) TRANSFER GMM RESULTS BACK TO ADATA_COMBINED
################################################################################

print(f"\n{'='*60}")
print("TRANSFERRING GMM RESULTS TO ADATA_COMBINED")
print(f"{'='*60}")

# Get all GMM-related columns with 085 suffix
gmm_cols = [col for col in adata_gmm.obs.columns 
            if ('_posterior_085' in col or '_gmm_' in col and '_085' in col)]

print(f"GMM columns to transfer: {len(gmm_cols)}")
for col in gmm_cols:
    print(f"  - {col}")

# Initialize columns in adata_combined
for col in gmm_cols:
    if col not in adata_combined.obs.columns:
        if 'posterior' in col:
            adata_combined.obs[col] = np.nan
        else:
            adata_combined.obs[col] = 'Unknown'

# Transfer values
for col in gmm_cols:
    adata_combined.obs.loc[adata_gmm.obs_names, col] = adata_gmm.obs[col]

print(f"✅ Transferred {len(gmm_cols)} columns to adata_combined")

################################################################################
# 4) COMPARISON BETWEEN MARKER SETS
################################################################################

if len(all_gmm_results) >= 2:
    print(f"\n{'='*60}")
    print("COMPARISON BETWEEN MARKER SETS")
    print(f"{'='*60}")
    
    comparison_rows = []
    
    for marker_set_name, results in all_gmm_results.items():
        if results and 'accuracy_results' in results:
            for assay, accuracy in results['accuracy_results'].items():
                comparison_rows.append({
                    'marker_set': marker_set_name,
                    'assay': assay,
                    'accuracy': accuracy
                })
    
    if comparison_rows:
        comparison_df = pd.DataFrame(comparison_rows)
        
        accuracy_pivot = comparison_df.pivot(
            index='assay',
            columns='marker_set',
            values='accuracy'
        )
        
        print("\nGMM Accuracy Comparison:")
        print(accuracy_pivot.round(1))
        
        plt.figure(figsize=(10, 6))
        x = np.arange(len(accuracy_pivot))
        width = 0.35
        
        for i, marker_set in enumerate(accuracy_pivot.columns):
            plt.bar(x + i*width, accuracy_pivot[marker_set], width=width, label=marker_set, alpha=0.8)
        
        plt.xlabel('Assay')
        plt.ylabel('Accuracy (%)')
        plt.title(f'GMM Accuracy Comparison (Threshold: {posterior_threshold})')
        plt.xticks(x + width/2, accuracy_pivot.index, rotation=45, ha='right')
        plt.legend()
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        
        plt.savefig(os.path.join(gmm_dir, 'GMM_comparison_085.pdf'), dpi=300, bbox_inches='tight')
        plt.close()
        
        comparison_df.to_csv(os.path.join(gmm_dir, 'GMM_comparison_085.csv'), index=False)

################################################################################
# 5) SAVE RESULTS
################################################################################

print(f"\n{'='*60}")
print("SAVING RESULTS")
print(f"{'='*60}")

try:
    save_path = os.path.join(gmm_dir, 'adata_gmm_085.h5ad')
    adata_gmm.write_h5ad(save_path)
    print(f"✅ Saved adata_gmm to: {save_path}")
except Exception as e:
    print(f"⚠️ Could not save adata_gmm: {e}")

try:
    save_path = os.path.join(gmm_dir, 'adata_combined_with_gmm_085.h5ad')
    adata_combined.write_h5ad(save_path)
    print(f"✅ Saved adata_combined to: {save_path}")
except Exception as e:
    print(f"⚠️ Could not save adata_combined: {e}")

gmm_scores_df = adata_combined.obs[['assay', 'cell_type_broad'] + gmm_cols].copy()
gmm_scores_df.to_csv(os.path.join(gmm_dir, 'gmm_scores_085.csv'))
print(f"✅ Saved GMM scores to: {os.path.join(gmm_dir, 'gmm_scores_085.csv')}")

print(f"\n{'='*60}")
print("COLUMNS ADDED TO ADATA_COMBINED")
print(f"{'='*60}")

print("\nPosterior columns:")
post_cols = [c for c in gmm_cols if '_posterior_085' in c and '_max_' not in c]
for col in sorted(post_cols):
    print(f"  - {col}")

print("\nGMM label columns:")
label_cols = [c for c in gmm_cols if '_gmm_' in c and '_085' in c]
for col in sorted(label_cols):
    print(f"  - {col}")

print(f"\n{'='*60}")
print("GMM ANALYSIS COMPLETE!")
print(f"{'='*60}")

In [ ]:
###################Posterior plots################
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scanpy as sc
from matplotlib.colors import LinearSegmentedColormap

# Set save directory
save_dir = f'{root}/gene_scoring/gmm_085/'

# Define cell type order
cell_type_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

################################################################################
# FUNCTION TO CREATE POSTERIOR HEATMAP
################################################################################

def plot_posterior_heatmap(adata, marker_set_name, assay_name, save_dir):
    """
    Plot posterior probability heatmap for a given marker set and assay
    """
    print(f"\n{'='*60}")
    print(f"PLOTTING: {marker_set_name.upper()} - {assay_name}")
    print(f"{'='*60}")
    
    # Subset to specific assay
    mask = adata.obs['assay'] == assay_name
    adata_assay = adata[mask].copy()
    
    print(f"Cells in {assay_name}: {adata_assay.n_obs}")
    
    # Define posterior columns
    columns = [f'{marker_set_name}_{ct}_posterior_085' for ct in cell_type_order]
    
    # Check which columns exist
    existing_cols = [col for col in columns if col in adata_assay.obs.columns]
    print(f"Found {len(existing_cols)} posterior columns")
    
    if len(existing_cols) == 0:
        print("No posterior columns found, skipping")
        return
    
    # Extract the data
    df = adata_assay.obs[existing_cols].copy()
    
    # Remove rows with NaN values
    df_clean = df.dropna()
    print(f"Original number of cells: {len(df)}")
    print(f"Cells after removing NaN: {len(df_clean)}")
    
    if len(df_clean) == 0:
        print("No valid cells after removing NaN, skipping")
        return
    
    # Rename columns for cleaner labels (remove prefix and suffix)
    clean_names = [col.replace(f'{marker_set_name}_', '').replace('_posterior_085', '') 
                   for col in existing_cols]
    df_clean.columns = clean_names
    
    # Reorder columns to match cell_type_order
    ordered_cols = [ct for ct in cell_type_order if ct in df_clean.columns]
    df_clean = df_clean[ordered_cols]
    
    # Sort cells by their maximum score category
    df_clean['max_type'] = df_clean[ordered_cols].idxmax(axis=1)
    
    # Within each cell type, sort by the score of that type (descending)
    df_sorted = pd.DataFrame()
    for cell_type in ordered_cols:
        subset = df_clean[df_clean['max_type'] == cell_type].copy()
        subset = subset.sort_values(by=cell_type, ascending=False)
        df_sorted = pd.concat([df_sorted, subset])
    
    # Remove helper column
    df_sorted = df_sorted.drop(columns=['max_type'])
    
    # Store cell type boundaries for annotation
    cell_counts = df_clean.groupby('max_type').size()
    cell_counts = cell_counts.reindex(ordered_cols)
    cell_counts = cell_counts.fillna(0).astype(int)
    
    print(f"\nCells per type:")
    for ct, count in cell_counts.items():
        print(f"  {ct}: {count}")
    
    # Check data range
    print(f"\nData range: {df_sorted.min().min():.3f} to {df_sorted.max().max():.3f}")
    
    # Count cells above 0.85
    print(f"\nCells with posterior > 0.85:")
    for col in ordered_cols:
        count_above = (df_sorted[col] > 0.85).sum()
        pct = (count_above / len(df_sorted)) * 100
        print(f"  {col}: {count_above} ({pct:.1f}%)")
    
    # Create custom colormap with emphasis on values > 0.85
    colors = plt.cm.RdBu_r(np.linspace(0, 1, 256))
    threshold_idx = int(0.85 * 256)
    for i in range(threshold_idx, 256):
        colors[i] = [0.4, 0.0, 0.1, 1.0]  # Dark maroon color
    custom_cmap = LinearSegmentedColormap.from_list('RdBu_r_highlight', colors)
    
    # Create publication-ready figure
    n_celltypes = len(ordered_cols)
    fig, ax = plt.subplots(figsize=(12, max(3, n_celltypes * 0.5)))
    
    # Create heatmap
    heatmap = sns.heatmap(
        df_sorted.T,
        cmap=custom_cmap,
        vmin=0,
        vmax=1,
        xticklabels=False,
        yticklabels=True,
        cbar_kws={
            'label': 'Posterior probability',
            'shrink': 0.8,
            'aspect': 20
        },
        ax=ax,
        rasterized=True
    )
    
    # Add marker on colorbar to indicate 0.85 threshold
    cbar = heatmap.collections[0].colorbar
    cbar.ax.axhline(y=0.85, color='black', linewidth=1.5, linestyle='--')
    cbar.ax.text(1.5, 0.85, '0.85', va='center', ha='left', fontsize=9)
    
    # Style adjustments
    ax.set_xlabel('Cells', fontsize=12)
    ax.set_ylabel('')
    ax.tick_params(axis='y', labelsize=11, rotation=0)
    
    # Add vertical lines to separate cell type groups
    cumsum = 0
    for i, cell_type in enumerate(ordered_cols):
        if cell_type in cell_counts.index and cell_counts[cell_type] > 0:
            cumsum += cell_counts[cell_type]
            if i < len(ordered_cols) - 1:
                ax.axvline(x=cumsum, color='black', linewidth=1.5)
    
    # Add cell type labels at the bottom
    cumsum = 0
    for cell_type in ordered_cols:
        if cell_type in cell_counts.index and cell_counts[cell_type] > 0:
            count = cell_counts[cell_type]
            mid_point = cumsum + count / 2
            ax.text(mid_point, len(ordered_cols) + 0.3, cell_type, 
                   ha='center', va='top', fontsize=10, fontweight='bold')
            cumsum += count
    
    # Title
    ax.set_title(f'{marker_set_name.upper()} Posterior Probabilities - {assay_name}', 
                fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    
    # Save the plot
    save_path = os.path.join(save_dir, f'{assay_name}_{marker_set_name}_posterior_heatmap.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    
    print(f"\n✅ Saved: {save_path}")
    
    return df_sorted

################################################################################
# PLOT FOR EACH ASSAY AND MARKER SET
################################################################################

# Get unique assays in the GMM data
assays = adata_gmm.obs['assay'].unique()
print(f"Assays to plot: {assays}")

# Plot for literature markers
print("\n" + "="*80)
print("LITERATURE MARKERS POSTERIOR HEATMAPS")
print("="*80)

for assay in assays:
    plot_posterior_heatmap(
        adata_gmm, 
        marker_set_name='literature', 
        assay_name=assay, 
        save_dir=gmm_dir
    )

# Plot for ttest markers
print("\n" + "="*80)
print("TTEST MARKERS POSTERIOR HEATMAPS")
print("="*80)

for assay in assays:
    plot_posterior_heatmap(
        adata_gmm, 
        marker_set_name='ttest', 
        assay_name=assay, 
        save_dir=gmm_dir
    )

################################################################################
# COMBINED PLOT (ALL ASSAYS TOGETHER)
################################################################################

def plot_combined_posterior_heatmap(adata, marker_set_name, save_dir):
    """
    Plot combined posterior heatmap for all assays
    """
    print(f"\n{'='*60}")
    print(f"PLOTTING COMBINED: {marker_set_name.upper()}")
    print(f"{'='*60}")
    
    # Define posterior columns
    columns = [f'{marker_set_name}_{ct}_posterior_085' for ct in cell_type_order]
    existing_cols = [col for col in columns if col in adata.obs.columns]
    
    if len(existing_cols) == 0:
        print("No posterior columns found, skipping")
        return
    
    # Extract data
    df = adata.obs[existing_cols + ['assay']].copy()
    df_clean = df.dropna()
    
    print(f"Total cells: {len(df_clean)}")
    
    # Rename columns
    clean_names = [col.replace(f'{marker_set_name}_', '').replace('_posterior_085', '') 
                   for col in existing_cols]
    rename_dict = {old: new for old, new in zip(existing_cols, clean_names)}
    df_clean = df_clean.rename(columns=rename_dict)
    
    ordered_cols = [ct for ct in cell_type_order if ct in clean_names]
    
    # Sort by assay first, then by max type
    df_clean['max_type'] = df_clean[ordered_cols].idxmax(axis=1)
    
    df_sorted = pd.DataFrame()
    for assay in adata.obs['assay'].unique():
        assay_data = df_clean[df_clean['assay'] == assay].copy()
        for cell_type in ordered_cols:
            subset = assay_data[assay_data['max_type'] == cell_type].copy()
            subset = subset.sort_values(by=cell_type, ascending=False)
            df_sorted = pd.concat([df_sorted, subset])
    
    # Store assay info before dropping
    assay_info = df_sorted['assay'].copy()
    df_sorted = df_sorted.drop(columns=['max_type', 'assay'])
    
    # Create custom colormap
    colors = plt.cm.RdBu_r(np.linspace(0, 1, 256))
    threshold_idx = int(0.85 * 256)
    for i in range(threshold_idx, 256):
        colors[i] = [0.4, 0.0, 0.1, 1.0]
    custom_cmap = LinearSegmentedColormap.from_list('RdBu_r_highlight', colors)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(14, max(3, len(ordered_cols) * 0.5)))
    
    heatmap = sns.heatmap(
        df_sorted.T,
        cmap=custom_cmap,
        vmin=0,
        vmax=1,
        xticklabels=False,
        yticklabels=True,
        cbar_kws={
            'label': 'Posterior probability',
            'shrink': 0.8,
            'aspect': 20
        },
        ax=ax,
        rasterized=True
    )
    
    cbar = heatmap.collections[0].colorbar
    cbar.ax.axhline(y=0.85, color='black', linewidth=1.5, linestyle='--')
    cbar.ax.text(1.5, 0.85, '0.85', va='center', ha='left', fontsize=9)
    
    ax.set_xlabel('Cells', fontsize=12)
    ax.set_ylabel('')
    ax.tick_params(axis='y', labelsize=11, rotation=0)
    ax.set_title(f'{marker_set_name.upper()} Posterior Probabilities - All Assays', 
                fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    
    save_path = os.path.join(save_dir, f'combined_{marker_set_name}_posterior_085_heatmap.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    
    print(f"\n✅ Saved: {save_path}")

# Plot combined heatmaps
plot_combined_posterior_heatmap(adata_gmm, 'literature', gmm_dir)
plot_combined_posterior_heatmap(adata_gmm, 'ttest', gmm_dir)

print("\n" + "="*80)
print("ALL POSTERIOR HEATMAPS COMPLETE!")
print("="*80)

In [ ]:
###################Posterior plots################
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scanpy as sc
from matplotlib.colors import LinearSegmentedColormap

# Set save directory
save_dir = f'{root}/gene_scoring/gmm_085/'

# Define cell type order
cell_type_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

################################################################################
# FUNCTION TO CREATE POSTERIOR HEATMAP
################################################################################

def plot_posterior_heatmap(adata, marker_set_name, assay_name, save_dir):
    """
    Plot posterior probability heatmap for a given marker set and assay
    """
    print(f"\n{'='*60}")
    print(f"PLOTTING: {marker_set_name.upper()} - {assay_name}")
    print(f"{'='*60}")
    
    # Subset to specific assay
    mask = adata.obs['assay'] == assay_name
    adata_assay = adata[mask].copy()
    
    print(f"Cells in {assay_name}: {adata_assay.n_obs}")
    
    # Define posterior columns
    columns = [f'{marker_set_name}_{ct}_posterior' for ct in cell_type_order]
    
    # Check which columns exist
    existing_cols = [col for col in columns if col in adata_assay.obs.columns]
    print(f"Found {len(existing_cols)} posterior columns")
    
    if len(existing_cols) == 0:
        print("No posterior columns found, skipping")
        return
    
    # Extract the data
    df = adata_assay.obs[existing_cols].copy()
    
    # Remove rows with NaN values
    df_clean = df.dropna()
    print(f"Original number of cells: {len(df)}")
    print(f"Cells after removing NaN: {len(df_clean)}")
    
    if len(df_clean) == 0:
        print("No valid cells after removing NaN, skipping")
        return
    
    # Rename columns for cleaner labels (remove prefix and suffix)
    clean_names = [col.replace(f'{marker_set_name}_', '').replace('_posterior', '') 
                   for col in existing_cols]
    df_clean.columns = clean_names
    
    # Reorder columns to match cell_type_order
    ordered_cols = [ct for ct in cell_type_order if ct in df_clean.columns]
    df_clean = df_clean[ordered_cols]
    
    # Sort cells by their maximum score category
    df_clean['max_type'] = df_clean[ordered_cols].idxmax(axis=1)
    
    # Within each cell type, sort by the score of that type (descending)
    df_sorted = pd.DataFrame()
    for cell_type in ordered_cols:
        subset = df_clean[df_clean['max_type'] == cell_type].copy()
        subset = subset.sort_values(by=cell_type, ascending=False)
        df_sorted = pd.concat([df_sorted, subset])
    
    # Remove helper column
    df_sorted = df_sorted.drop(columns=['max_type'])
    
    # Store cell type boundaries for annotation
    cell_counts = df_clean.groupby('max_type').size()
    cell_counts = cell_counts.reindex(ordered_cols)
    cell_counts = cell_counts.fillna(0).astype(int)
    
    print(f"\nCells per type:")
    for ct, count in cell_counts.items():
        print(f"  {ct}: {count}")
    
    # Check data range
    print(f"\nData range: {df_sorted.min().min():.3f} to {df_sorted.max().max():.3f}")
    
    # Count cells above 0.95
    print(f"\nCells with posterior > 0.95:")
    for col in ordered_cols:
        count_above = (df_sorted[col] > 0.95).sum()
        pct = (count_above / len(df_sorted)) * 100
        print(f"  {col}: {count_above} ({pct:.1f}%)")
    
    # Create custom colormap with emphasis on values > 0.95
    colors = plt.cm.RdBu_r(np.linspace(0, 1, 256))
    threshold_idx = int(0.95 * 256)
    for i in range(threshold_idx, 256):
        colors[i] = [0.4, 0.0, 0.1, 1.0]  # Dark maroon color
    custom_cmap = LinearSegmentedColormap.from_list('RdBu_r_highlight', colors)
    
    # Create publication-ready figure
    n_celltypes = len(ordered_cols)
    fig, ax = plt.subplots(figsize=(12, max(3, n_celltypes * 0.5)))
    
    # Create heatmap
    heatmap = sns.heatmap(
        df_sorted.T,
        cmap=custom_cmap,
        vmin=0,
        vmax=1,
        xticklabels=False,
        yticklabels=True,
        cbar_kws={
            'label': 'Posterior probability',
            'shrink': 0.8,
            'aspect': 20
        },
        ax=ax,
        rasterized=True
    )
    
    # Add marker on colorbar to indicate 0.95 threshold
    cbar = heatmap.collections[0].colorbar
    cbar.ax.axhline(y=0.95, color='black', linewidth=1.5, linestyle='--')
    cbar.ax.text(1.5, 0.95, '0.95', va='center', ha='left', fontsize=9)
    
    # Style adjustments
    ax.set_xlabel('Cells', fontsize=12)
    ax.set_ylabel('')
    ax.tick_params(axis='y', labelsize=11, rotation=0)
    
    # Add vertical lines to separate cell type groups
    cumsum = 0
    for i, cell_type in enumerate(ordered_cols):
        if cell_type in cell_counts.index and cell_counts[cell_type] > 0:
            cumsum += cell_counts[cell_type]
            if i < len(ordered_cols) - 1:
                ax.axvline(x=cumsum, color='black', linewidth=1.5)
    
    # Add cell type labels at the bottom
    cumsum = 0
    for cell_type in ordered_cols:
        if cell_type in cell_counts.index and cell_counts[cell_type] > 0:
            count = cell_counts[cell_type]
            mid_point = cumsum + count / 2
            ax.text(mid_point, len(ordered_cols) + 0.3, cell_type, 
                   ha='center', va='top', fontsize=10, fontweight='bold')
            cumsum += count
    
    # Title
    ax.set_title(f'{marker_set_name.upper()} Posterior Probabilities - {assay_name}', 
                fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    
    # Save the plot
    save_path = os.path.join(save_dir, f'{assay_name}_{marker_set_name}_posterior_heatmap.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    
    print(f"\n✅ Saved: {save_path}")
    
    return df_sorted

################################################################################
# PLOT FOR EACH ASSAY AND MARKER SET
################################################################################

# Get unique assays in the GMM data
assays = adata_gmm.obs['assay'].unique()
print(f"Assays to plot: {assays}")

# Plot for literature markers
print("\n" + "="*80)
print("LITERATURE MARKERS POSTERIOR HEATMAPS")
print("="*80)

for assay in assays:
    plot_posterior_heatmap(
        adata_gmm, 
        marker_set_name='literature', 
        assay_name=assay, 
        save_dir=gmm_dir
    )

# Plot for ttest markers
print("\n" + "="*80)
print("TTEST MARKERS POSTERIOR HEATMAPS")
print("="*80)

for assay in assays:
    plot_posterior_heatmap(
        adata_gmm, 
        marker_set_name='ttest', 
        assay_name=assay, 
        save_dir=gmm_dir
    )

################################################################################
# COMBINED PLOT (ALL ASSAYS TOGETHER)
################################################################################

def plot_combined_posterior_heatmap(adata, marker_set_name, save_dir):
    """
    Plot combined posterior heatmap for all assays
    """
    print(f"\n{'='*60}")
    print(f"PLOTTING COMBINED: {marker_set_name.upper()}")
    print(f"{'='*60}")
    
    # Define posterior columns
    columns = [f'{marker_set_name}_{ct}_posterior' for ct in cell_type_order]
    existing_cols = [col for col in columns if col in adata.obs.columns]
    
    if len(existing_cols) == 0:
        print("No posterior columns found, skipping")
        return
    
    # Extract data
    df = adata.obs[existing_cols + ['assay']].copy()
    df_clean = df.dropna()
    
    print(f"Total cells: {len(df_clean)}")
    
    # Rename columns
    clean_names = [col.replace(f'{marker_set_name}_', '').replace('_posterior', '') 
                   for col in existing_cols]
    rename_dict = {old: new for old, new in zip(existing_cols, clean_names)}
    df_clean = df_clean.rename(columns=rename_dict)
    
    ordered_cols = [ct for ct in cell_type_order if ct in clean_names]
    
    # Sort by assay first, then by max type
    df_clean['max_type'] = df_clean[ordered_cols].idxmax(axis=1)
    
    df_sorted = pd.DataFrame()
    for assay in adata.obs['assay'].unique():
        assay_data = df_clean[df_clean['assay'] == assay].copy()
        for cell_type in ordered_cols:
            subset = assay_data[assay_data['max_type'] == cell_type].copy()
            subset = subset.sort_values(by=cell_type, ascending=False)
            df_sorted = pd.concat([df_sorted, subset])
    
    # Store assay info before dropping
    assay_info = df_sorted['assay'].copy()
    df_sorted = df_sorted.drop(columns=['max_type', 'assay'])
    
    # Create custom colormap
    colors = plt.cm.RdBu_r(np.linspace(0, 1, 256))
    threshold_idx = int(0.95 * 256)
    for i in range(threshold_idx, 256):
        colors[i] = [0.4, 0.0, 0.1, 1.0]
    custom_cmap = LinearSegmentedColormap.from_list('RdBu_r_highlight', colors)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(14, max(3, len(ordered_cols) * 0.5)))
    
    heatmap = sns.heatmap(
        df_sorted.T,
        cmap=custom_cmap,
        vmin=0,
        vmax=1,
        xticklabels=False,
        yticklabels=True,
        cbar_kws={
            'label': 'Posterior probability',
            'shrink': 0.8,
            'aspect': 20
        },
        ax=ax,
        rasterized=True
    )
    
    cbar = heatmap.collections[0].colorbar
    cbar.ax.axhline(y=0.95, color='black', linewidth=1.5, linestyle='--')
    cbar.ax.text(1.5, 0.95, '0.95', va='center', ha='left', fontsize=9)
    
    ax.set_xlabel('Cells', fontsize=12)
    ax.set_ylabel('')
    ax.tick_params(axis='y', labelsize=11, rotation=0)
    ax.set_title(f'{marker_set_name.upper()} Posterior Probabilities - All Assays', 
                fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    
    save_path = os.path.join(save_dir, f'combined_{marker_set_name}_posterior_heatmap.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    
    print(f"\n✅ Saved: {save_path}")

# Plot combined heatmaps
plot_combined_posterior_heatmap(adata_gmm, 'literature', gmm_dir)
plot_combined_posterior_heatmap(adata_gmm, 'ttest', gmm_dir)

print("\n" + "="*80)
print("ALL POSTERIOR HEATMAPS COMPLETE!")
print("="*80)

In [ ]:
for col in adata_combined.obs.columns:
    if pd.api.types.is_string_dtype(adata_combined.obs[col]):
        adata_combined.obs[col] = adata_combined.obs[col].astype(str)

# Drop any unnamed CSV index columns just in case
adata_combined.obs = adata_combined.obs.loc[:, ~adata_combined.obs.columns.str.contains("^Unnamed")]

# Now save
adata_combined.write_h5ad(
    "/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/integration/adata_combined_gmm_23032026.h5ad"
)

In [ ]:
print("\n🧹 ADDING CLEAN GMM RESULTS TO ADATA_COMBINED")
print("=" * 60)
score_dir = os.path.join(root, 'gene_scoring') 

# --------------------------------------------------------------
# 1. INITIAL CLEANUP
# --------------------------------------------------------------
print("Cleaning up existing GMM columns...")
cols_to_remove = [col for col in adata_combined.obs.columns 
                  if any(x in col for x in ['posterior', '_score_', 'module_max_label', 'module_label_'])]
if cols_to_remove:
    adata_combined.obs = adata_combined.obs.drop(columns=cols_to_remove)
    print(f"  Removed {len(cols_to_remove)} columns")

# Also clean up any problematic columns that might cause save issues
print("Checking for problematic columns...")
problematic_patterns = ['Unnamed: 0', 'index', 'level_0']
for pattern in problematic_patterns:
    if pattern in adata_combined.obs.columns:
        print(f"  Removing '{pattern}' column")
        adata_combined.obs = adata_combined.obs.drop(columns=[pattern])

# --------------------------------------------------------------
# 2. PREPARE DATA
# --------------------------------------------------------------
# Create a mask for scNMT cells (RNA + DCM)
scnmt_mask = (adata_combined.obs['orig.ident'].str.contains('scNMT')) & \
             (adata_combined.obs['Dox'] != 'Minus')
print(f"\n📊 Processing {scnmt_mask.sum()} scNMT cells...")

# Define cell types
cell_types = ['Stem', 'TA', 'Enterocyte', 'Goblet', 'Paneth', 'Enteroendocrine', 'Tuft']

# --------------------------------------------------------------
# 3. ADD CLEAN GMM RESULTS
# --------------------------------------------------------------
for marker_set_name, results_dict in all_gmm_results.items():
    print(f"\n🔹 Processing {marker_set_name} results...")
    
    adata_subset = results_dict['adata_subset']
    common_cells = adata_combined.obs_names[scnmt_mask].intersection(adata_subset.obs_names)
    print(f"  Matching {len(common_cells)} cells")
    
    # 1. Add FINAL LABEL
    print("  Adding final label...")
    label_col_name = f'{marker_set_name}_gmm_label'
    adata_combined.obs[label_col_name] = np.nan
    
    if 'module_max_label' in adata_subset.obs.columns:
        adata_combined.obs.loc[common_cells, label_col_name] = \
            adata_subset.obs.loc[common_cells, 'module_max_label']
    
    # 2. Add PER-CELLTYPE POSTERIORS (clean, assay-agnostic)
    print(f"  Adding per-celltype posteriors...")
    
    for ct in cell_types:
        # Create clean column name
        posterior_col_name = f'{marker_set_name}_{ct}_posterior'
        adata_combined.obs[posterior_col_name] = np.nan
        
        # For each cell, get the posterior from its specific assay
        for cell_id in common_cells:
            assay = adata_combined.obs.loc[cell_id, 'orig.ident']
            
            # Try to find the assay-specific posterior column
            assay_posterior_col = f'{assay}_{ct}_module_score_Z_posterior'
            if assay_posterior_col in adata_subset.obs.columns:
                posterior_val = adata_subset.obs.loc[cell_id, assay_posterior_col]
                adata_combined.obs.loc[cell_id, posterior_col_name] = posterior_val
    
    # 3. Add CONFIDENCE (posterior of assigned label)
    print("  Adding confidence scores...")
    confidence_col_name = f'{marker_set_name}_gmm_confidence'
    adata_combined.obs[confidence_col_name] = np.nan
    
    for cell_id in common_cells:
        label = adata_combined.obs.loc[cell_id, label_col_name]
        if pd.isna(label) or label == 'ambiguous':
            continue
            
        # Get the posterior for the assigned label
        posterior_col = f'{marker_set_name}_{label}_posterior'
        if posterior_col in adata_combined.obs.columns:
            adata_combined.obs.loc[cell_id, confidence_col_name] = \
                adata_combined.obs.loc[cell_id, posterior_col]
    
    # 4. Add RAW Z-SCORES (clean, assay-agnostic)
    print("  Adding Z-scores...")
    for ct in cell_types:
        # Create clean column name for Z-scores
        zscore_col_name = f'{marker_set_name}_{ct}_zscore'
        adata_combined.obs[zscore_col_name] = np.nan
        
        # For each cell, get the Z-score from its specific assay
        for cell_id in common_cells:
            assay = adata_combined.obs.loc[cell_id, 'orig.ident']
            
            # Try to find the assay-specific Z-score column
            assay_zscore_col = f'{ct}_module_score_Z'
            if assay_zscore_col in adata_subset.obs.columns:
                zscore_val = adata_subset.obs.loc[cell_id, assay_zscore_col]
                adata_combined.obs.loc[cell_id, zscore_col_name] = zscore_val

print(f"\n✅ Added clean GMM results to adata_combined")
print(f"   Total columns now: {adata_combined.obs.shape[1]}")

# --------------------------------------------------------------
# 4. SUMMARY
# --------------------------------------------------------------
print("\n📋 CLEAN COLUMNS ADDED:")
for marker_set_name in all_gmm_results.keys():
    print(f"\n  {marker_set_name}:")
    print(f"    - {marker_set_name}_gmm_label (final assignment)")
    print(f"    - {marker_set_name}_gmm_confidence (0-1)")
    for ct in cell_types:
        print(f"    - {marker_set_name}_{ct}_posterior (posterior probability)")
        print(f"    - {marker_set_name}_{ct}_zscore (Z-score)")

print(f"\n🎯 GMM label distribution:")
for marker_set_name in all_gmm_results.keys():
    label_col = f'{marker_set_name}_gmm_label'
    print(f"\n  {marker_set_name}:")
    counts = adata_combined.obs[label_col].value_counts(dropna=False)
    for label, count in counts.items():
        print(f"    {label}: {count} cells")

# --------------------------------------------------------------
# 5. FINAL CLEANUP & SAVING (FIXED)
# --------------------------------------------------------------
print(f"\n🧹 FINAL CLEANUP BEFORE SAVING")

# Fix 1: Convert StringArray columns to regular strings
for col in adata_combined.obs.columns:
    dtype_str = str(adata_combined.obs[col].dtype)
    
    if dtype_str == 'string' or 'String' in dtype_str:
        print(f"  Converting StringArray: {col}")
        adata_combined.obs[col] = adata_combined.obs[col].astype('object')
    
    elif adata_combined.obs[col].dtype == 'object':
        # Clean up object columns
        try:
            adata_combined.obs[col] = adata_combined.obs[col].astype(str)
            adata_combined.obs[col] = adata_combined.obs[col].replace({'nan': np.nan, '<NA>': np.nan, 'None': np.nan})
        except:
            print(f"  ⚠️ Removing problematic column: {col}")
            adata_combined.obs = adata_combined.obs.drop(columns=[col])

# Fix 2: Also check .var
if 'Unnamed: 0' in adata_combined.var.columns:
    adata_combined.var = adata_combined.var.drop(columns=['Unnamed: 0'])

for col in adata_combined.var.columns:
    dtype_str = str(adata_combined.var[col].dtype)
    if dtype_str == 'string' or 'String' in dtype_str:
        adata_combined.var[col] = adata_combined.var[col].astype('object')

print(f"  Final shape: {adata_combined.shape}")
print(f"  .obs columns: {len(adata_combined.obs.columns)}")

# --------------------------------------------------------------
# 6. SAVE WITH ERROR HANDLING (FIXED)
# --------------------------------------------------------------
print(f"\n💾 Saving updated adata_combined...")
output_path = os.path.join(score_dir, 'adata_combined_with_clean_gmm_13012026.h5ad')

try:
    # First try normal save
    adata_combined.write_h5ad(output_path)
    print(f"✅ Saved to: {output_path}")
    
except Exception as e:
    print(f"❌ Error saving: {e}")
    
    # Try enabling nullable strings
    print("  Enabling nullable strings setting...")
    import anndata
    anndata.settings.allow_write_nullable_strings = True
    
    try:
        adata_combined.write_h5ad(output_path)
        print(f"✅ Saved to: {output_path}")
    except Exception as e2:
        print(f"❌ Still failed: {e2}")
        
        # Try with compression (FIXED syntax)
        try:
            adata_combined.write_h5ad(output_path, compression='gzip')
            print(f"✅ Saved with compression to: {output_path}")
        except Exception as e3:
            print(f"❌ All save attempts failed: {e3}")
            
            # Save metadata as CSV as last resort
            csv_path = os.path.join(score_dir, 'adata_combined_metadata_13012026.csv')
            adata_combined.obs.to_csv(csv_path)
            print(f"  Saved metadata to: {csv_path}")

# --------------------------------------------------------------
# 7. OPTIONAL: ASSAY-SPECIFIC COLUMNS
# --------------------------------------------------------------
print("\n🔧 OPTIONAL: Add assay-specific columns with cleaner names?")
add_assay_specific = False  # Set to True if you want them

if add_assay_specific:
    print("  Adding assay-specific columns...")
    for marker_set_name, results_dict in all_gmm_results.items():
        adata_subset = results_dict['adata_subset']
        common_cells = adata_combined.obs_names[scnmt_mask].intersection(adata_subset.obs_names)
        
        # Add assay-specific posteriors with cleaner names
        for assay in ['scNMT_pool35_RNA', 'scNMT_pool35_DCM', 'scNMT_pool40_RNA', 'scNMT_pool40_DCM']:
            assay_short = assay.replace('scNMT_', '').replace('pool', 'p')
            for ct in cell_types:
                # Original column in subset
                orig_col = f'{assay}_{ct}_module_score_Z_posterior'
                if orig_col in adata_subset.obs.columns:
                    # Cleaner column name
                    clean_col = f'{marker_set_name}_{assay_short}_{ct}_posterior'
                    adata_combined.obs[clean_col] = np.nan
                    adata_combined.obs.loc[common_cells, clean_col] = \
                        adata_subset.obs.loc[common_cells, orig_col]
    
    print(f"  Added assay-specific columns. Total columns: {adata_combined.obs.shape[1]}")

print(f"\n🎉 CLEAN GMM ADDITION COMPLETE!")

In [ ]:
##### save marker sets used ####
print("\n💾 SAVING MARKER GENE SETS (SIMPLE CSV)")
print("=" * 60)

# Create directory
marker_sets_dir = os.path.join(score_dir, 'marker_gene_sets')
os.makedirs(marker_sets_dir, exist_ok=True)

# 1. Save literature_top markers as CSV
if 'specific_markers' in locals():
    print("Saving literature_top markers...")
    lit_rows = []
    for cell_type, sources in specific_markers.items():
        # Literature genes
        for gene in sources.get('literature', []):
            lit_rows.append([cell_type, gene, 'literature', 'literature_top'])
        # Top marker genes  
        for gene in sources.get('top_markers', []):
            lit_rows.append([cell_type, gene, 'top_marker', 'literature_top'])
    
    if lit_rows:
        lit_df = pd.DataFrame(lit_rows, columns=['cell_type', 'gene', 'source', 'marker_set'])
        lit_path = os.path.join(marker_sets_dir, 'literature_top_markers_13012026.csv')
        lit_df.to_csv(lit_path, index=False)
        print(f"  ✓ {len(lit_df)} genes saved to {lit_path}")

# 2. Save ttest markers as CSV
if 'ttest_markers_by_ct' in locals():
    print("Saving ttest markers...")
    ttest_rows = []
    for cell_type, genes in ttest_markers_by_ct.items():
        for gene in genes:
            ttest_rows.append([cell_type, gene, 'ttest', 'ttest'])
    
    if ttest_rows:
        ttest_df = pd.DataFrame(ttest_rows, columns=['cell_type', 'gene', 'source', 'marker_set'])
        ttest_path = os.path.join(marker_sets_dir, 'ttest_markers_13012026.csv')
        ttest_df.to_csv(ttest_path, index=False)
        print(f"  ✓ {len(ttest_df)} genes saved to {ttest_path}")

print("\n✅ Done. Marker genes saved as simple CSV files.")

In [ ]:
print("\n🔀 DCM vs RNA GMM LABEL COMPARISON (Pool + Plate + Well matching)")
print("=" * 70)

for marker_set_name in all_gmm_results.keys():
    print(f"\n🔹 {marker_set_name.upper()}")
    
    adata_subset = all_gmm_results[marker_set_name]['adata_subset']
    
    # Check for required columns
    required_cols = ['Plate', 'Well']
    missing_cols = [col for col in required_cols if col not in adata_subset.obs.columns]
    
    if missing_cols:
        print(f"  ⚠️ Missing columns: {missing_cols}")
        continue
    
    # Extract pool number from assay name
    def extract_pool(assay_name):
        if 'pool35' in assay_name.lower():
            return '35'
        elif 'pool40' in assay_name.lower():
            return '40'
        return None
    
    # Create unique cell identifier: Pool_Plate_Well
    adata_subset.obs['pool_plate_well'] = adata_subset.obs.apply(
        lambda row: f"{extract_pool(row['orig.ident'])}_{row['Plate']}_{row['Well']}", 
        axis=1
    )
    
    # Separate DCM and RNA cells
    dcm_cells = adata_subset.obs[adata_subset.obs['orig.ident'].str.contains('DCM')].copy()
    rna_cells = adata_subset.obs[adata_subset.obs['orig.ident'].str.contains('RNA')].copy()
    
    print(f"  DCM cells: {len(dcm_cells)}")
    print(f"  RNA cells: {len(rna_cells)}")
    
    # Find the CORRECT label columns - assay-specific final labels
    dcm_label_cols = []
    rna_label_cols = []
    
    # Look for assay-specific module_max_label columns
    for col in adata_subset.obs.columns:
        if 'module_max_label' in col:
            if any(x in col for x in ['DCM_module_max_label', '_DCM_']):
                dcm_label_cols.append(col)
            elif any(x in col for x in ['RNA_module_max_label', '_RNA_']):
                rna_label_cols.append(col)
    
    print(f"  Available DCM final labels: {dcm_label_cols}")
    print(f"  Available RNA final labels: {rna_label_cols}")
    
    # For merging, we need to match DCM and RNA labels appropriately
    # We'll create a mapping dictionary for each assay
    label_mapping = {
        'scNMT_pool35_DCM': None,
        'scNMT_pool40_DCM': None,
        'scNMT_pool35_RNA': None,
        'scNMT_pool40_RNA': None
    }
    
    # Find the appropriate label column for each assay
    for assay in label_mapping.keys():
        # Look for exact match first
        exact_col = f'{assay}_module_max_label'
        if exact_col in adata_subset.obs.columns:
            label_mapping[assay] = exact_col
        else:
            # Look for any column containing the assay name and module_max_label
            for col in adata_subset.obs.columns:
                if assay in col and 'module_max_label' in col:
                    label_mapping[assay] = col
                    break
    
    print(f"\n  Label column mapping:")
    for assay, col in label_mapping.items():
        if col:
            print(f"    {assay} → {col}")
    
    # Prepare dataframes with appropriate labels
    dcm_data_list = []
    for assay in ['scNMT_pool35_DCM', 'scNMT_pool40_DCM']:
        if label_mapping[assay]:
            assay_cells = dcm_cells[dcm_cells['orig.ident'] == assay].copy()
            assay_cells['final_label'] = assay_cells[label_mapping[assay]]
            dcm_data_list.append(assay_cells[['pool_plate_well', 'final_label', 'Dox', 'orig.ident']])
    
    rna_data_list = []
    for assay in ['scNMT_pool35_RNA', 'scNMT_pool40_RNA']:
        if label_mapping[assay]:
            assay_cells = rna_cells[rna_cells['orig.ident'] == assay].copy()
            assay_cells['final_label'] = assay_cells[label_mapping[assay]]
            rna_data_list.append(assay_cells[['pool_plate_well', 'final_label', 'orig.ident']])
    
    if not dcm_data_list or not rna_data_list:
        print(f"  ⚠️ No label data for merging")
        continue
    
    dcm_combined = pd.concat(dcm_data_list, ignore_index=True)
    rna_combined = pd.concat(rna_data_list, ignore_index=True)
    
    # Merge based on pool_plate_well
    meta_merged = pd.merge(
        dcm_combined, 
        rna_combined, 
        on='pool_plate_well', 
        suffixes=['_dcm', '_rna'],
        how='inner'
    )
    
    print(f"\n  Matched pairs found: {len(meta_merged)}")
    
    if len(meta_merged) > 0:
        # Show pool distribution
        print(f"  Pool distribution:")
        meta_merged['pool'] = meta_merged['orig.ident_dcm'].apply(extract_pool)
        for pool in sorted(meta_merged['pool'].unique()):
            pool_count = (meta_merged['pool'] == pool).sum()
            print(f"    Pool {pool}: {pool_count} pairs")
        
        # Check if we have final_label columns
        if 'final_label_dcm' in meta_merged.columns and 'final_label_rna' in meta_merged.columns:
            # Process by Dox treatment
            if 'Dox' in meta_merged.columns:
                for dox in sorted(meta_merged['Dox'].dropna().unique()):
                    metasub = meta_merged[meta_merged['Dox'] == dox]
                    
                    print(f"\n  📊 Dox Treatment: {dox}")
                    print(f"    Matched pairs: {len(metasub)}")
                    
                    # Create cross-tabulation
                    cross_tab = pd.crosstab(
                        metasub['final_label_dcm'],
                        metasub['final_label_rna'],
                        margins=True
                    )
                    
                    # Create heatmap
                    plt.figure(figsize=(12, 8))
                    heatmap_data = cross_tab.iloc[:-1, :-1]  # Exclude margins
                    
                    # Only plot if we have data
                    if not heatmap_data.empty and heatmap_data.sum().sum() > 0:
                        sns.heatmap(
                            data=heatmap_data,
                            annot=True, 
                            fmt='d', 
                            cmap='YlOrRd',
                            cbar_kws={'label': 'Number of Cell Pairs'},
                            square=True
                        )
                        
                        # Calculate consistency
                        consistent_pairs = (metasub['final_label_dcm'] == metasub['final_label_rna']).sum()
                        consistency_rate = (consistent_pairs / len(metasub)) * 100
                        
                        plt.title(f'{marker_set_name} | DCM → RNA Label Transitions\n'
                                 f'Dox: {dox} | Pairs: {len(metasub)} | Consistency: {consistency_rate:.1f}%',
                                 fontsize=14, fontweight='bold')
                        plt.xlabel('RNA Labels', fontsize=12)
                        plt.ylabel('DCM Labels', fontsize=12)
                        plt.xticks(rotation=45, ha='right')
                        plt.yticks(rotation=0)
                        plt.tight_layout()
                        
                        # Save plot
                        output_dir = os.path.join(score_dir, f'{marker_set_name}_gmm')
                        os.makedirs(output_dir, exist_ok=True)
                        output_path = os.path.join(output_dir, f'Dox_{dox}_DCM_RNA_transitions.png')
                        plt.savefig(output_path, dpi=300, bbox_inches='tight')
                        plt.show()
                        
                        print(f"    Label consistency: {consistent_pairs}/{len(metasub)} ({consistency_rate:.1f}%)")
                        
                        # Print transition matrix
                        print(f"    Transition matrix:")
                        print(heatmap_data.to_string())
                    else:
                        print(f"    ⚠️ No data for heatmap (empty matrix)")
                        print(f"    DCM labels: {metasub['final_label_dcm'].unique()}")
                        print(f"    RNA labels: {metasub['final_label_rna'].unique()}")
            else:
                print(f"  ⚠️ Dox column not found in merged data")
        else:
            print(f"  ⚠️ Final label columns not found after merge")
    else:
        print(f"  ⚠️ No matched pairs found")

print(f"\n✅ DCM-RNA COMPARISON COMPLETE")

In [ ]:
print("\n📊 KDE PLOTS: ACTIVE vs INACTIVE GROUPS WITH THRESHOLD")
print("=" * 60)
output_dir = '/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/gene_scoring/gmm_085/'
# Define cell types
cell_types = ['Stem', 'TA', 'Enterocyte', 'Goblet', 'Paneth', 'Enteroendocrine', 'Tuft']

for marker_set_name in all_gmm_results.keys():
    print(f"\n🔹 {marker_set_name.upper()} KDE Plots")
    
    adata_subset = all_gmm_results[marker_set_name]['adata_subset']
    
    # Get all assays in this dataset
    assays = sorted(adata_subset.obs['orig.ident'].unique())
    print(f"  Assays found: {assays}")
    
    for assay in assays:
        print(f"\n  Assay: {assay}")
        
        # Filter for this assay
        assay_mask = adata_subset.obs['orig.ident'] == assay
        assay_data = adata_subset.obs.loc[assay_mask]
        
        if len(assay_data) == 0:
            continue
        
        # Process each cell type
        for ct in cell_types:
            # Find the module score column for this assay
            score_col = None
            # Try different naming patterns
            possible_score_cols = [
                f'{ct}_module_score_Z',  # Clean column if you added it
                f'{assay}_{ct}_module_score_Z'  # Assay-specific column
            ]
            
            for col in possible_score_cols:
                if col in assay_data.columns:
                    score_col = col
                    break
            
            if score_col is None:
                # Try to find any column with this cell type and module score
                for col in assay_data.columns:
                    if ct in col and 'module_score_Z' in col and 'posterior' not in col:
                        score_col = col
                        break
            
            # Find the posterior column
            post_col = None
            possible_post_cols = [
                f'{assay}_{ct}_module_score_Z_posterior',  # Most likely
                f'{marker_set_name}_{assay}_{ct}_module_score_Z_posterior'  # If prefixed
            ]
            
            for col in possible_post_cols:
                if col in assay_data.columns:
                    post_col = col
                    break
            
            if post_col is None:
                # Try to find any posterior column for this cell type
                for col in assay_data.columns:
                    if ct in col and 'posterior' in col:
                        post_col = col
                        break
            
            if score_col is None or post_col is None:
                # Skip if we can't find both columns
                continue
            
            # Get scores and posteriors
            scores = assay_data[score_col].dropna()
            posteriors = assay_data[post_col].dropna()
            
            if len(scores) == 0 or len(posteriors) == 0:
                continue
            
            # Align indices (just in case)
            common_idx = scores.index.intersection(posteriors.index)
            scores = scores.loc[common_idx]
            posteriors = posteriors.loc[common_idx]
            
            # Identify active vs inactive groups
            active_mask = posteriors >= posterior_threshold
            active_scores = scores[active_mask]
            inactive_scores = scores[~active_mask]
            
            active_count = len(active_scores)
            inactive_count = len(inactive_scores)
            
            print(f"    {ct}: {active_count} active, {inactive_count} inactive")
            
            # Skip if not enough data for KDE
            if active_count < 3 or inactive_count < 3:
                print(f"      ⚠️ Not enough data for KDE plot")
                continue
            
            # Create KDE plot
            plt.figure(figsize=(10, 6))
            
            # Plot KDE for inactive group
            if inactive_count > 0:
                sns.kdeplot(data=inactive_scores, label='Inactive', 
                           color='blue', linewidth=2, fill=True, alpha=0.3)
            
            # Plot KDE for active group
            if active_count > 0:
                sns.kdeplot(data=active_scores, label='Active', 
                           color='red', linewidth=2, fill=True, alpha=0.3)
            
            # Add threshold line (approximate)
            if active_count > 0 and inactive_count > 0:
                # Find approximate threshold score (min of active group)
                threshold_score = active_scores.min()
                plt.axvline(x=threshold_score, color='black', linestyle='--', 
                           linewidth=2, alpha=0.7, 
                           label=f'Threshold (posterior > {posterior_threshold})')
            
            plt.xlabel(f'{ct} Module Score (Z)')
            plt.ylabel('Density')
            plt.title(f'{marker_set_name} | {assay}: {ct}\n'
                     f'Active: {active_count}, Inactive: {inactive_count}')
            plt.legend()
            plt.grid(True, alpha=0.3)
            
            # Save figure
            output_dir = os.path.join(score_dir, f'{marker_set_name}_gmm')
            os.makedirs(output_dir, exist_ok=True)
            output_path = os.path.join(output_dir, f'{assay}_{ct}_KDE.png')
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            plt.show()
            
            # Also create a scatter plot of scores vs posteriors
            plt.figure(figsize=(10, 6))
            plt.scatter(scores, posteriors, alpha=0.5, s=10)
            plt.axhline(y=posterior_threshold, color='red', linestyle='--', 
                       label=f'Threshold ({posterior_threshold})')
            plt.xlabel(f'{ct} Module Score (Z)')
            plt.ylabel('Posterior Probability')
            plt.title(f'{marker_set_name} | {assay}: {ct}\nScore vs Posterior')
            plt.legend()
            plt.grid(True, alpha=0.3)
            
            scatter_path = os.path.join(output_dir, f'{assay}_{ct}_scatter.png')
            plt.savefig(scatter_path, dpi=300, bbox_inches='tight')
            plt.show()

print(f"\n✅ KDE ANALYSIS COMPLETE")

# Plots for paper

In [ ]:
sc.settings.figdir = "/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/integration/"

In [ ]:
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

# -----------------------------
# SETTINGS
# -----------------------------
dim = 25
annot_col = f'leiden_annot_{dim}'
temp_col = f'{annot_col}_clean'

# Cell type colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1'
}

desired_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# -----------------------------
# CLEAN ANNOTATION COLUMN
# -----------------------------
if annot_col in adata_combined.obs.columns:
    adata_combined.obs[temp_col] = (
        adata_combined.obs[annot_col]
        .astype(str)
        .apply(lambda x: re.sub(r'^\d+_', '', x))
    )

    # Build ordered categories
    present = adata_combined.obs[temp_col].unique().tolist()
    categories = [ct for ct in desired_order if ct in present]
    categories += [ct for ct in present if ct not in categories]

    adata_combined.obs[temp_col] = pd.Categorical(
        adata_combined.obs[temp_col],
        categories=categories,
        ordered=True
    )

# -----------------------------
# UMAP PLOTS
# -----------------------------
plot_cols = [temp_col, 'leiden']
plot_cols = [c for c in plot_cols if c in adata_combined.obs.columns]

for col in plot_cols:
    if col == temp_col:
        palette = [cell_type_colors.get(ct, '#808080')
                   for ct in adata_combined.obs[col].cat.categories]

        sc.pl.umap(
            adata_combined,
            color=col,
            palette=palette,
            frameon=False,
            legend_loc='on data',
            save=f'_{dim}_{col}_umap.pdf'
        )
    else:
        sc.pl.umap(
            adata_combined,
            color=col,
            frameon=False,
            legend_loc='on data',
            save=f'_{dim}_{col}_umap.pdf'
        )

# -----------------------------
# HIGHLIGHT ASSAYS ON UMAP
# -----------------------------
if 'assay' in adata_combined.obs.columns:

    print("Unique assay values:", adata_combined.obs['assay'].unique())

    umap_coords = adata_combined.obsm['X_umap']

    fig, ax = plt.subplots(figsize=(8, 6))

    # Background (reference)
    ref_mask = adata_combined.obs['assay'] == 'Ref'
    ax.scatter(
        umap_coords[ref_mask, 0],
        umap_coords[ref_mask, 1],
        s=5, c='gray', alpha=0.2,
        label='Ref', rasterized=True
    )

    # pDCM
    dcm_mask = adata_combined.obs['assay'] == 'pDCM'
    ax.scatter(
        umap_coords[dcm_mask, 0],
        umap_coords[dcm_mask, 1],
        s=15, c='#003366', alpha=0.8,
        label='pDCM', rasterized=True
    )

    # pRNA
    rna_mask = adata_combined.obs['assay'] == 'pRNA'
    ax.scatter(
        umap_coords[rna_mask, 0],
        umap_coords[rna_mask, 1],
        s=15, c='#8B0000', alpha=0.8,
        label='pRNA', rasterized=True
    )

    ax.set_title('sciMeDseq cells highlighted (dim 25)')
    ax.set_aspect('equal')
    ax.axis('off')
    ax.legend(loc='upper right')

    plt.tight_layout()

    save_path = (
        f'{sc.settings.figdir}/umap_sciMedseq_highlight_dim25.pdf'
        if sc.settings.figdir else
        'umap_sciMedseq_highlight_dim25.pdf'
    )

    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

    print(f"Saved: {save_path}")

else:
    print("'assay' column not found in adata_combined.obs")

# -----------------------------
# CLEANUP
# -----------------------------
if temp_col in adata_combined.obs.columns:
    adata_combined.obs.drop(columns=[temp_col], inplace=True)

print("All dim=25 UMAPs saved!")


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import re

# -----------------------------
# LOAD MARKER GENES
# -----------------------------
ttest_markers = pd.read_csv(
    "/media/erlantz/71DE167A75E99BB4/Pool32/Combi/Python/integration/gene_scoring/marker_gene_sets/ttest_markers_13012026.csv"
)
lit_markers = pd.read_csv(
    "/media/erlantz/71DE167A75E99BB4/Pool32/Combi/Python/integration/gene_scoring/marker_gene_sets/literature_top_markers_13012026.csv"
)

print("Marker table columns:", lit_markers.columns.tolist())
print(lit_markers.head())

# -----------------------------
# TOP 10 GENES PER CELL TYPE
# -----------------------------
top_genes_per_type = {}

group_col = 'cell_type' if 'cell_type' in lit_markers.columns else 'cluster'
gene_col = 'gene'

for ct in lit_markers[group_col].unique():
    subset = lit_markers[lit_markers[group_col] == ct]
    top_genes_per_type[ct] = subset[gene_col].head(10).tolist()

print("\nTop genes per cell type:")
for ct, genes in top_genes_per_type.items():
    print(ct, genes)

# -----------------------------
# SMALL INTESTINE SETUP
# -----------------------------
si_cell_order = [
    'Stem', 'TA', 'Enterocyte',
    'Paneth', 'Goblet', 'Tuft', 'EEC'
]

assay_config = {
    'Ref':  {'cell_type_col': 'cell_type_broad'},
    'pRNA': {'cell_type_col': 'cell_type_broad'},
    'pDCM': {'cell_type_col': 'cell_type_broad'}
}

# -----------------------------
# LOOP OVER ASSAYS
# -----------------------------
for assay, config in assay_config.items():

    print(f"\n{'='*60}")
    print(f"Processing Small Intestine – {assay}")
    print('='*60)

    mask = adata_combined.obs['assay'] == assay
    if mask.sum() == 0:
        print("No cells found")
        continue

    adata_assay = adata_combined[mask].copy()

    cell_type_col = config['cell_type_col']
    if cell_type_col not in adata_assay.obs.columns:
        print(f"Missing column: {cell_type_col}")
        continue

    # -----------------------------
    # CLEAN CELL TYPE COLUMN
    # -----------------------------
    adata_assay.obs['cell_type_plot'] = (
        adata_assay.obs[cell_type_col]
        .astype(str)
        .str.replace(r'^\d+_', '', regex=True)
    )

    adata_assay = adata_assay[
        adata_assay.obs['cell_type_plot'].isin(si_cell_order)
    ].copy()

    print("Cell counts:")
    print(adata_assay.obs['cell_type_plot'].value_counts())

    if adata_assay.n_obs == 0:
        print("No cells after filtering")
        continue

    adata_assay.obs['cell_type_plot'] = pd.Categorical(
        adata_assay.obs['cell_type_plot'],
        categories=si_cell_order,
        ordered=True
    )

    # -----------------------------
    # USE LOGNORMALIZED LAYER
    # -----------------------------
    original_X = None
    if 'lognorm' in adata_assay.layers:
        original_X = adata_assay.X.copy()
        adata_assay.X = adata_assay.layers['lognorm']
        print("Using lognorm layer")

    # -----------------------------
    # PSEUDOBULK MEANS
    # -----------------------------
    pseudobulk = {}
    for ct in adata_assay.obs['cell_type_plot'].cat.categories:
        cells = adata_assay.obs['cell_type_plot'] == ct
        if cells.sum() > 0:
            pseudobulk[ct] = np.asarray(
                adata_assay[cells].X.mean(axis=0)
            ).flatten()

    pseudobulk_df = pd.DataFrame(
        pseudobulk, index=adata_assay.var_names
    )

    # -----------------------------
    # FILTER GENES BY SPECIFICITY
    # -----------------------------
    filtered_genes = []

    for ct, genes in top_genes_per_type.items():
        if ct not in pseudobulk_df.columns:
            continue

        for g in genes:
            if g not in pseudobulk_df.index:
                continue

            if pseudobulk_df.loc[g].idxmax() == ct:
                filtered_genes.append(g)

    filtered_genes = list(dict.fromkeys(filtered_genes))

    print(f"Selected {len(filtered_genes)} clean marker genes")

    if len(filtered_genes) == 0:
        print("No valid genes found")
        if original_X is not None:
            adata_assay.X = original_X
        continue

    # -----------------------------
    # HEATMAP
    # -----------------------------
    sc.pl.heatmap(
        adata_assay,
        var_names=filtered_genes,
        groupby='cell_type_plot',
        standard_scale='var',
        cmap='RdBu_r',
        show_gene_labels=True,
        save=f'_SI_{assay}_filtered_lit_markers_heatmap.pdf'
    )

    # -----------------------------
    # DOTPLOT
    # -----------------------------
    sc.pl.dotplot(
        adata_assay,
        var_names=filtered_genes,
        groupby='cell_type_plot',
        standard_scale='var',
        dot_max=0.6,
        title=f'Small intestine – {assay}',
        save=f'_SI_{assay}_filtered_lit_markers_dotplot.pdf'
    )

    # Restore X
    if original_X is not None:
        adata_assay.X = original_X

    print(f"Saved plots for {assay}")

print("\nAll Small Intestine plots saved!")


In [ ]:
ttest_markers = pd.read_csv(
    "/media/erlantz/71DE167A75E99BB4/Pool32/Combi/Python/integration/gene_scoring/marker_gene_sets/ttest_markers_13012026.csv"
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming your dataframe is called 'df' with columns: 'cell_type' and 'gene'
df = ttest_markers
# Get genes for each cell type
ta_genes = set(df[df['cell_type'] == 'TA']['gene'].unique())
print(f"Total TA genes: {len(ta_genes)}")

# Calculate overlap with each cell type
overlap_results = []

for cell_type in df['cell_type'].unique():
    if cell_type == 'TA':
        continue  # Skip TA itself
    
    # Get genes for this cell type
    ct_genes = set(df[df['cell_type'] == cell_type]['gene'].unique())
    
    # Find overlap
    overlapping_genes = ta_genes.intersection(ct_genes)
    
    overlap_results.append({
        'cell_type': cell_type,
        'total_genes': len(ct_genes),
        'overlap_with_TA': len(overlapping_genes),
        'overlap_percentage': (len(overlapping_genes) / len(ta_genes) * 100) if len(ta_genes) > 0 else 0,
        'percentage_of_ct_genes': (len(overlapping_genes) / len(ct_genes) * 100) if len(ct_genes) > 0 else 0,
        'overlapping_genes': list(overlapping_genes)
    })

# Create results dataframe
overlap_df = pd.DataFrame(overlap_results)
overlap_df = overlap_df.sort_values('overlap_with_TA', ascending=False)

# Display results
print("\n" + "="*80)
print("TA GENE OVERLAP WITH OTHER CELL TYPES")
print("="*80)
print(overlap_df[['cell_type', 'total_genes', 'overlap_with_TA', 'overlap_percentage', 'percentage_of_ct_genes']].to_string(index=False))

# Save results
overlap_df.to_csv('TA_gene_overlap_analysis.csv', index=False)
print(f"\n✅ Results saved to: TA_gene_overlap_analysis.csv")

# -----------------------------
# VISUALIZATION
# -----------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Number of overlapping genes
axes[0].barh(overlap_df['cell_type'], overlap_df['overlap_with_TA'], color='steelblue')
axes[0].set_xlabel('Number of Overlapping Genes with TA', fontweight='bold')
axes[0].set_ylabel('Cell Type', fontweight='bold')
axes[0].set_title(f'TA Gene Overlap Counts\n(Total TA genes: {len(ta_genes)})', fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Add values on bars
for i, (idx, row) in enumerate(overlap_df.iterrows()):
    axes[0].text(row['overlap_with_TA'] + 0.5, i, f"{int(row['overlap_with_TA'])}", 
                va='center', fontsize=9)

# Plot 2: Percentage of TA genes found in each cell type
axes[1].barh(overlap_df['cell_type'], overlap_df['overlap_percentage'], color='coral')
axes[1].set_xlabel('% of TA Genes Found in Cell Type', fontweight='bold')
axes[1].set_ylabel('Cell Type', fontweight='bold')
axes[1].set_title('TA Gene Overlap Percentage', fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)
axes[1].set_xlim(0, 100)

# Add percentage values on bars
for i, (idx, row) in enumerate(overlap_df.iterrows()):
    axes[1].text(row['overlap_percentage'] + 1, i, f"{row['overlap_percentage']:.1f}%", 
                va='center', fontsize=9)

plt.tight_layout()
plt.savefig('TA_gene_overlap_visualization.pdf', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Visualization saved to: TA_gene_overlap_visualization.pdf")

# -----------------------------
# SUMMARY STATISTICS
# -----------------------------
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

# Genes unique to TA (not found in any other cell type)
all_other_genes = set()
for cell_type in df['cell_type'].unique():
    if cell_type != 'TA':
        all_other_genes.update(df[df['cell_type'] == cell_type]['gene'].unique())

unique_ta_genes = ta_genes - all_other_genes
print(f"Genes unique to TA (not in any other cell type): {len(unique_ta_genes)}")
print(f"Percentage of TA genes that are unique: {len(unique_ta_genes)/len(ta_genes)*100:.1f}%")

# Genes shared with at least one other cell type
shared_genes = ta_genes - unique_ta_genes
print(f"\nGenes shared with at least one other cell type: {len(shared_genes)}")
print(f"Percentage of TA genes that are shared: {len(shared_genes)/len(ta_genes)*100:.1f}%")

# Most promiscuous genes (found in most cell types)
gene_counts = df.groupby('gene')['cell_type'].nunique().reset_index()
gene_counts.columns = ['gene', 'n_cell_types']
promiscuous_ta_genes = gene_counts[gene_counts['gene'].isin(ta_genes)].sort_values('n_cell_types', ascending=False)

print(f"\n🔝 Top 10 most promiscuous TA genes (found in most cell types):")
print(promiscuous_ta_genes.head(10).to_string(index=False))

# Save unique and shared gene lists
pd.DataFrame({'gene': list(unique_ta_genes)}).to_csv('TA_unique_genes.csv', index=False)
pd.DataFrame({'gene': list(shared_genes)}).to_csv('TA_shared_genes.csv', index=False)
promiscuous_ta_genes.to_csv('TA_promiscuous_genes.csv', index=False)

print("\n✅ Gene lists saved:")
print("   - TA_unique_genes.csv")
print("   - TA_shared_genes.csv")
print("   - TA_promiscuous_genes.csv")

# -----------------------------
# OPTIONAL: HEATMAP OF GENE SHARING
# -----------------------------
print("\n" + "="*80)
print("CREATING GENE SHARING HEATMAP")
print("="*80)

# Create a matrix of gene sharing between all cell types
cell_types = df['cell_type'].unique()
overlap_matrix = pd.DataFrame(0, index=cell_types, columns=cell_types)

for ct1 in cell_types:
    genes1 = set(df[df['cell_type'] == ct1]['gene'].unique())
    for ct2 in cell_types:
        genes2 = set(df[df['cell_type'] == ct2]['gene'].unique())
        overlap_matrix.loc[ct1, ct2] = len(genes1.intersection(genes2))

# Plot heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(overlap_matrix.astype(int), annot=True, fmt='d', cmap='YlOrRd', 
            cbar_kws={'label': 'Number of Shared Genes'},
            linewidths=0.5, linecolor='gray', ax=ax)
ax.set_xlabel('Cell Type', fontweight='bold', fontsize=12)
ax.set_ylabel('Cell Type', fontweight='bold', fontsize=12)
ax.set_title('Gene Sharing Matrix Between Cell Types\n(Diagonal = Total genes per cell type)', 
             fontweight='bold', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('gene_sharing_heatmap.pdf', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Gene sharing heatmap saved to: gene_sharing_heatmap.pdf")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import re

# -----------------------------
# LOAD MARKER GENES
# -----------------------------
ttest_markers = pd.read_csv(
    "/media/erlantz/71DE167A75E99BB4/Pool32/Combi/Python/integration/gene_scoring/marker_gene_sets/ttest_markers_13012026.csv"
)
lit_markers = pd.read_csv(
    "/media/erlantz/71DE167A75E99BB4/Pool32/Combi/Python/integration/gene_scoring/marker_gene_sets/literature_top_markers_13012026.csv"
)

group_col = 'cell_type' if 'cell_type' in lit_markers.columns else 'cluster'
gene_col = 'gene'

# -----------------------------
# SMALL INTESTINE CELL TYPES
# -----------------------------
si_cell_types = [
    'Stem', 'TA', 'Enterocyte',
    'Paneth', 'Goblet', 'Tuft', 'EEC'
]

# -----------------------------
# ASSAY -> LABEL COLUMN MAPPING
# -----------------------------
assay_label_col = {
    'Ref':  'cell_type_broad',
    'pRNA': 'cell_type_broad',
    'pDCM': 'ttest_gmm_label_confident_085',
}

# -----------------------------
# TAKE MANY GENES FIRST
# -----------------------------
top_genes_per_type = {}

for ct in lit_markers[group_col].unique():
    subset = lit_markers[lit_markers[group_col] == ct]
    top_genes_per_type[ct] = subset[gene_col].head(50).tolist()

# -----------------------------
# ASSAYS
# -----------------------------
assays = ['Ref', 'pRNA', 'pDCM']

# -----------------------------
# FIRST PASS: FILTER PER ASSAY
# -----------------------------
filtered_genes_per_assay = {}

for assay in assays:
    print(f"\nProcessing {assay}")

    mask = adata_combined.obs['assay'] == assay
    if mask.sum() == 0:
        continue

    adata_assay = adata_combined[mask].copy()

    label_col = assay_label_col[assay]

    # Clean cell types (strip numeric prefix, e.g. "3_Stem" -> "Stem")
    adata_assay.obs['cell_type_plot'] = (
        adata_assay.obs[label_col]
        .astype(str)
        .str.replace(r'^\d+_', '', regex=True)
    )

    adata_assay = adata_assay[
        adata_assay.obs['cell_type_plot'].isin(si_cell_types)
    ].copy()

    if adata_assay.n_obs == 0:
        continue

    # Use lognorm if present
    if 'lognorm' in adata_assay.layers:
        adata_assay.X = adata_assay.layers['lognorm']

    # Pseudobulk
    pseudobulk = {}
    for ct in si_cell_types:
        cells = adata_assay.obs['cell_type_plot'] == ct
        if cells.sum() > 0:
            pseudobulk[ct] = np.asarray(
                adata_assay[cells].X.mean(axis=0)
            ).flatten()

    pseudobulk_df = pd.DataFrame(
        pseudobulk, index=adata_assay.var_names
    )

    # Filter genes by specificity
    filtered_by_ct = {}

    for ct in si_cell_types:
        if ct not in top_genes_per_type:
            continue

        genes_ok = []
        for g in top_genes_per_type[ct]:
            if g not in pseudobulk_df.index:
                continue
            if pseudobulk_df.loc[g].idxmax() == ct:
                genes_ok.append(g)

        filtered_by_ct[ct] = genes_ok
        print(f"  {ct}: {len(genes_ok)} genes")

    filtered_genes_per_assay[assay] = filtered_by_ct

# -----------------------------
# INTERSECTION ACROSS ASSAYS
# -----------------------------
common_genes_per_ct = {}

for ct in si_cell_types:
    gene_sets = []
    for assay in assays:
        if assay in filtered_genes_per_assay:
            gene_sets.append(set(filtered_genes_per_assay[assay].get(ct, [])))

    if len(gene_sets) == 0:
        continue

    common = set.intersection(*gene_sets)
    ordered = [g for g in top_genes_per_type.get(ct, []) if g in common][:10]
    common_genes_per_ct[ct] = ordered

    print(f"{ct}: {len(common)} common genes → keeping {len(ordered)}")

# Flatten gene list
all_common_genes = []
for ct in si_cell_types:
    all_common_genes.extend(common_genes_per_ct.get(ct, []))

all_common_genes = list(dict.fromkeys(all_common_genes))

print(f"\nTotal genes to plot: {len(all_common_genes)}")

# -----------------------------
# SECOND PASS: PLOTTING
# -----------------------------
for assay in assays:
    print(f"\nPlotting {assay}")

    mask = adata_combined.obs['assay'] == assay
    if mask.sum() == 0:
        continue

    adata_assay = adata_combined[mask].copy()

    label_col = assay_label_col[assay]

    adata_assay.obs['cell_type_plot'] = (
        adata_assay.obs[label_col]
        .astype(str)
        .str.replace(r'^\d+_', '', regex=True)
    )

    adata_assay = adata_assay[
        adata_assay.obs['cell_type_plot'].isin(si_cell_types)
    ].copy()

    adata_assay.obs['cell_type_plot'] = pd.Categorical(
        adata_assay.obs['cell_type_plot'],
        categories=si_cell_types,
        ordered=True
    )

    if 'lognorm' in adata_assay.layers:
        original_X = adata_assay.X.copy()
        adata_assay.X = adata_assay.layers['lognorm']
    else:
        original_X = None

    genes_to_plot = [g for g in all_common_genes if g in adata_assay.var_names]

    if len(genes_to_plot) == 0:
        continue

    sc.pl.heatmap(
        adata_assay,
        var_names=genes_to_plot,
        groupby='cell_type_plot',
        standard_scale='var',
        cmap='RdBu_r',
        show_gene_labels=True,
        save=f'_SI_{assay}_lit_markers_heatmap_GMM.pdf'
    )

    sc.pl.matrixplot(
        adata_assay,
        var_names=genes_to_plot,
        groupby='cell_type_plot',
        standard_scale='var',
        cmap='RdBu_r',
        save=f'_SI_{assay}_lit_markers_matrixplot_GMM.pdf'
    )

    sc.pl.dotplot(
        adata_assay,
        var_names=genes_to_plot,
        groupby='cell_type_plot',
        standard_scale='var',
        dot_max=0.5,
        cmap='RdBu_r',
        save=f'_SI_{assay}_lit_markers_dotplot_GMM.pdf'
    )

    if original_X is not None:
        adata_assay.X = original_X

print("\nAll Small Intestine plots saved!")

In [ ]:
os.chdir("/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/gene_scoring/gmm_085/")

In [ ]:
#Boxplot module score Z

In [ ]:
root = '/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py//media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/integration/'

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# USER SETTINGS
# -----------------------------
score_mode = "ttest"  # "ttest" or "literature"
adata_plot = adata_combined     # your AnnData object with precomputed module z-scores
outdir = f"{root}/gene_scoring/gmm_085/zscore_boxplots_per_assay/"
os.makedirs(outdir, exist_ok=True)

# Cell type palette and order
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1'
}
desired_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# Assays
assay_list = ['pDCM', 'pRNA', 'Ref']

# -----------------------------
# GET ZSCORE COLUMNS
# -----------------------------
if score_mode == "ttest":
    zscore_cols = [c for c in adata_plot.obs.columns if c.startswith("ttest_") and c.endswith("_zscore")]
    label_col = "ttest_gmm_label_confident_085"
elif score_mode == "literature":
    zscore_cols = [c for c in adata_plot.obs.columns if c.startswith("literature_") and c.endswith("_zscore")]
    label_col = "literature_gmm_label_confident_085"
else:
    raise ValueError("score_mode must be 'ttest' or 'literature'")

print(f"Found {len(zscore_cols)} module score columns.")

# -----------------------------
# LOOP PER MODULE AND ASSAY
# -----------------------------
for zcol in zscore_cols:
    module = zcol.split("_")[1]  # e.g., ttest_Stem_zscore → Stem

    for assay in assay_list:
        # Subset
        df = adata_plot.obs[[zcol, label_col, "assay"]].copy()
        df.columns = ["zscore", "cell_type", "assay"]
        df = df[df["assay"] == assay]
        df = df.dropna()
        df = df[df["cell_type"].isin(desired_order)]
        df["cell_type"] = pd.Categorical(df["cell_type"], categories=desired_order, ordered=True)

        if df.empty:
            print(f"Skipping {module} for {assay} - no data")
            continue

        # Plot
        fig, ax = plt.subplots(figsize=(4, 4))
        sns.boxplot(
            data=df,
            x="cell_type",
            y="zscore",
            palette=[cell_type_colors[ct] for ct in desired_order if ct in df["cell_type"].unique()],
            linewidth=1.2,
            fliersize=2,
            ax=ax
        )

        ax.set_title(f"{module} Module Z-score ({assay})", fontsize=14, fontweight="bold")
        ax.set_xlabel("Cell type")
        ax.set_ylabel("Z-score")
        ax.axhline(0, color="black", linestyle="--", alpha=0.5, linewidth=1)

        plt.tight_layout()
        save_path = os.path.join(outdir, f"{module}_{assay}_{score_mode}_zscore_boxplot.pdf")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.show()
        plt.close()

        print(f"Saved: {save_path}")


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/gmm_085'
os.makedirs(score_dir, exist_ok=True)

adata = adata_combined  # Use your AnnData object here

# Cell type palette and order
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': 'lightgray'
}
label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# Assay to plot
assay_to_plot = 'pDCM'
extra_filter_col = 'Dox'
extra_filter_value = 'Minus'

# -----------------------------
# GET ZSCORE COLUMNS
# -----------------------------
zscore_cols = [col for col in adata.obs.columns if col.endswith('_zscore')]
print(f"Found Z-score columns: {zscore_cols}")

# -----------------------------
# SUBSET CELLS
# -----------------------------
mask = adata.obs['assay'] == assay_to_plot
if extra_filter_col in adata.obs.columns:
    mask &= adata.obs[extra_filter_col] != extra_filter_value

adata_dcm = adata[mask].copy()
print(f"{assay_to_plot} cells (excluding {extra_filter_value}): {adata_dcm.n_obs}")

# -----------------------------
# PER MODULE BOXPLOTS
# -----------------------------
cell_type_col = 'ttest_gmm_label_confident_085'

for zscore_col in zscore_cols:
    module_ct = zscore_col.replace('_zscore', '')
    print(f"\nProcessing {module_ct} module...")

    # Prepare data
    plot_data = pd.DataFrame({
        'zscore': adata_dcm.obs[zscore_col].values,
        'cell_type': adata_dcm.obs[cell_type_col].astype(str).values
    })

    # Filter out NaN / ambiguous / unknown
    plot_data = plot_data.dropna()
    plot_data = plot_data[~plot_data['cell_type'].str.contains('nan|ambiguous|unknown', case=False, na=True)]
    plot_data = plot_data[plot_data['cell_type'].isin(label_order)]

    if plot_data.empty:
        print(f"  Skipping {module_ct} - no data")
        continue

    # Set categorical order
    plot_data['cell_type'] = pd.Categorical(plot_data['cell_type'], categories=label_order, ordered=True)

    # Palette: highlight corresponding cell type if you want, otherwise just use color mapping
    palette = [cell_type_colors[ct] for ct in label_order if ct in plot_data['cell_type'].unique()]

    # Plot
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.boxplot(
        data=plot_data,
        x='cell_type',
        y='zscore',
        palette=palette,
        order=label_order,
        linewidth=1.5,
        fliersize=3,
        ax=ax
    )

    ax.set_title(f'{module_ct} Module Z-score ({assay_to_plot})', fontsize=14, fontweight='bold')
    ax.set_xlabel('Cell Type', fontsize=12)
    ax.set_ylabel('Z-score', fontsize=12)
    ax.axhline(0, color='black', linestyle='--', alpha=0.5, linewidth=1)
    ax.axhline(1.65, color='red', linestyle=':', alpha=0.5, linewidth=1, label='Confidence threshold (1.65)')

    plt.tight_layout()
    save_path = os.path.join(score_dir, f'{module_ct}_zscore_boxplot_{assay_to_plot}.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

    print(f"  Saved {save_path}")

# -----------------------------
# COMBINED FIGURE (ALL MODULES)
# -----------------------------
n_modules = len(zscore_cols)
if n_modules > 0:
    fig, axes = plt.subplots(1, n_modules, figsize=(5 * n_modules, 5))
    if n_modules == 1:
        axes = [axes]

    for idx, zscore_col in enumerate(zscore_cols):
        ax = axes[idx]
        module_ct = zscore_col.replace('_module_score_Z_allassays', '')

        plot_data = pd.DataFrame({
            'zscore': adata_dcm.obs[zscore_col].values,
            'cell_type': adata_dcm.obs[cell_type_col].astype(str).values
        })
        plot_data = plot_data.dropna()
        plot_data = plot_data[~plot_data['cell_type'].str.contains('nan|ambiguous|unknown', case=False, na=True)]
        plot_data = plot_data[plot_data['cell_type'].isin(label_order)]
        if plot_data.empty:
            continue

        plot_data['cell_type'] = pd.Categorical(plot_data['cell_type'], categories=label_order, ordered=True)
        palette = [cell_type_colors[ct] for ct in label_order if ct in plot_data['cell_type'].unique()]

        sns.boxplot(
            data=plot_data,
            x='cell_type',
            y='zscore',
            palette=palette,
            order=label_order,
            linewidth=1.5,
            fliersize=3,
            ax=ax
        )

        ax.set_title(f'{module_ct}', fontsize=14, fontweight='bold')
        ax.set_xlabel('Cell Type', fontsize=12)
        ax.set_ylabel('Z-score', fontsize=12)
        ax.axhline(0, color='black', linestyle='--', alpha=0.5, linewidth=1)
        ax.axhline(1.65, color='red', linestyle=':', alpha=0.5, linewidth=1)

    plt.tight_layout()
    save_path = os.path.join(score_dir, f'all_modules_zscore_boxplot_{assay_to_plot}.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

    print(f"Saved {save_path}")

print("\nAll boxplots saved!")


In [ ]:
zscore_cols = [col for col in adata_combined.obs.columns if col.endswith('_zscore')]
zscore_cols

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, mannwhitneyu
from itertools import combinations
import scanpy as sc

# Set directory
score_dir = f'{root}/gene_scoring/'
os.makedirs(score_dir, exist_ok=True)

# Define color palette for cell types
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': 'lightgray'
}
label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# Assay to plot
assay_to_plot = 'pDCM'
extra_filter_col = 'Dox'
extra_filter_value = 'Minus'

# Get Z-score columns from adata_all_assays
zscore_cols = [col for col in adata_combined.obs.columns if col.endswith('_zscore')]
print(f"Found Z-score columns: {zscore_cols}")

# Filter to scM_DCM only, exclude Minus from Dox
dcm_mask = (adata_combined.obs['assay'] == 'pDCM')
if 'Dox' in adata_combined.obs.columns:
    dcm_mask = dcm_mask & (adata_combined.obs['Dox'] != 'Minus')

adata_dcm = adata_combined[dcm_mask].copy()
print(f"scM_DCM cells (excluding Minus): {adata_dcm.n_obs}")

# Use module_max_label_allassays for cell type
cell_type_col = 'literature_gmm_label_confident'

# Store all statistical results
all_stats_results = []

# Create one boxplot per module score with statistics
for zscore_col in zscore_cols:
    # Extract cell type name from column
    module_ct = zscore_col.replace('_zscore', '')
    
    print(f"\n{'='*60}")
    print(f"Processing {module_ct} module...")
    print('='*60)
    
    # Prepare data for plotting
    plot_data = pd.DataFrame({
        'zscore': adata_dcm.obs[zscore_col].values,
        'cell_type': adata_dcm.obs[cell_type_col].astype(str).values
    })
    
    # Remove NaN and ambiguous
    plot_data = plot_data.dropna()
    plot_data = plot_data[~plot_data['cell_type'].str.contains('nan|ambiguous', case=False, na=True)]
    plot_data = plot_data[plot_data['cell_type'].isin(label_order)]
    
    if len(plot_data) == 0:
        print(f"  Skipping {module_ct} - no data")
        continue
    
    # Get available cell types
    available_types = [ct for ct in label_order if ct in plot_data['cell_type'].unique()]
    
    # Perform pairwise t-tests and Mann-Whitney U tests
    print(f"\n  Pairwise comparisons for {module_ct} module:")
    
    for ct1, ct2 in combinations(available_types, 2):
        group1 = plot_data[plot_data['cell_type'] == ct1]['zscore'].values
        group2 = plot_data[plot_data['cell_type'] == ct2]['zscore'].values
        
        # T-test (parametric)
        t_stat, t_pval = ttest_ind(group1, group2)
        
        # Mann-Whitney U test (non-parametric)
        u_stat, u_pval = mannwhitneyu(group1, group2, alternative='two-sided')
        
        # Calculate means and standard deviations
        mean1, std1 = np.mean(group1), np.std(group1)
        mean2, std2 = np.mean(group2), np.std(group2)
        
        # Effect size (Cohen's d)
        pooled_std = np.sqrt(((len(group1) - 1) * std1**2 + (len(group2) - 1) * std2**2) / 
                            (len(group1) + len(group2) - 2))
        cohens_d = (mean1 - mean2) / pooled_std if pooled_std > 0 else 0
        
        # Store results
        result = {
            'module': module_ct,
            'group1': ct1,
            'group2': ct2,
            'n_group1': len(group1),
            'n_group2': len(group2),
            'mean_group1': mean1,
            'mean_group2': mean2,
            'std_group1': std1,
            'std_group2': std2,
            'mean_diff': mean1 - mean2,
            't_statistic': t_stat,
            't_pvalue': t_pval,
            'u_statistic': u_stat,
            'u_pvalue': u_pval,
            'cohens_d': cohens_d
        }
        all_stats_results.append(result)
        
        # Print results
        sig_t = '***' if t_pval < 0.001 else '**' if t_pval < 0.01 else '*' if t_pval < 0.05 else 'ns'
        sig_u = '***' if u_pval < 0.001 else '**' if u_pval < 0.01 else '*' if u_pval < 0.05 else 'ns'
        
        print(f"\n    {ct1} vs {ct2}:")
        print(f"      {ct1}: n={len(group1)}, mean={mean1:.3f} ± {std1:.3f}")
        print(f"      {ct2}: n={len(group2)}, mean={mean2:.3f} ± {std2:.3f}")
        print(f"      T-test: t={t_stat:.3f}, p={t_pval:.2e} ({sig_t})")
        print(f"      Mann-Whitney U: U={u_stat:.1f}, p={u_pval:.2e} ({sig_u})")
        print(f"      Cohen's d: {cohens_d:.3f}")
    
    # Set categorical order
    plot_data['cell_type'] = pd.Categorical(
        plot_data['cell_type'],
        categories=available_types,
        ordered=True
    )
    
    # Create custom palette: corresponding cell type gets its color, others are gray
    palette = []
    for ct in available_types:
        if ct == module_ct:
            palette.append(cell_type_colors[ct])
        else:
            palette.append('#D3D3D3')
    
    # Create boxplot with significance annotations
    fig, ax = plt.subplots(figsize=(7, 6))
    
    sns.boxplot(
        data=plot_data,
        x='cell_type',
        y='zscore',
        hue='cell_type',
        order=available_types,
        hue_order=available_types,
        palette=palette,
        ax=ax,
        linewidth=1.5,
        fliersize=3,
        legend=False
    )
    
    # Add significance annotations
    y_max = plot_data['zscore'].max()
    y_range = plot_data['zscore'].max() - plot_data['zscore'].min()
    
    # Get relevant comparisons for this module
    module_comparisons = [r for r in all_stats_results if r['module'] == module_ct]
    
    # Add significance bars
    for i, comp in enumerate(module_comparisons):
        ct1, ct2 = comp['group1'], comp['group2']
        pval = comp['t_pvalue']
        
        # Determine significance symbol
        if pval < 0.001:
            sig = '***'
        elif pval < 0.01:
            sig = '**'
        elif pval < 0.05:
            sig = '*'
        else:
            sig = 'ns'
        
        # Get x positions
        x1 = available_types.index(ct1)
        x2 = available_types.index(ct2)
        
        # Calculate y position for the bar
        y_bar = y_max + (i + 1) * y_range * 0.12
        
        # Draw the significance bar
        ax.plot([x1, x1, x2, x2], [y_bar - y_range*0.02, y_bar, y_bar, y_bar - y_range*0.02], 
                color='black', linewidth=1)
        ax.text((x1 + x2) / 2, y_bar + y_range*0.01, sig, ha='center', va='bottom', fontsize=10)
    
    # Customize
    ax.set_title(f'{module_ct} Module Z-score (scM_DCM)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Cell Type', fontsize=12)
    ax.set_ylabel('Z-score', fontsize=12)
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.5, linewidth=1)
    ax.axhline(y=1.65, color='red', linestyle=':', alpha=0.5, linewidth=1, label='Confidence threshold (1.65)')
    
    # Adjust y-axis to accommodate significance bars
    ax.set_ylim(plot_data['zscore'].min() - y_range*0.1, y_max + (len(module_comparisons) + 1) * y_range * 0.15)
    
    plt.tight_layout()
    
    # Save
    save_path = os.path.join(score_dir, f'{module_ct}_zscore_boxplot_DCM_with_stats.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    
    print(f"\n  Saved {save_path}")

# Create DataFrame with all results
stats_df = pd.DataFrame(all_stats_results)

# Add multiple testing correction (Bonferroni and FDR)
from scipy.stats import false_discovery_control

if len(stats_df) > 0:
    # Bonferroni correction
    stats_df['t_pvalue_bonferroni'] = stats_df['t_pvalue'] * len(stats_df)
    stats_df['t_pvalue_bonferroni'] = stats_df['t_pvalue_bonferroni'].clip(upper=1.0)
    
    stats_df['u_pvalue_bonferroni'] = stats_df['u_pvalue'] * len(stats_df)
    stats_df['u_pvalue_bonferroni'] = stats_df['u_pvalue_bonferroni'].clip(upper=1.0)
    
    # FDR correction (Benjamini-Hochberg)
    try:
        stats_df['t_pvalue_fdr'] = false_discovery_control(stats_df['t_pvalue'].values, method='bh')
        stats_df['u_pvalue_fdr'] = false_discovery_control(stats_df['u_pvalue'].values, method='bh')
    except:
        # Fallback manual FDR calculation
        from scipy.stats import rankdata
        n = len(stats_df)
        
        # For t-test
        ranks = rankdata(stats_df['t_pvalue'])
        stats_df['t_pvalue_fdr'] = stats_df['t_pvalue'] * n / ranks
        stats_df['t_pvalue_fdr'] = stats_df['t_pvalue_fdr'].clip(upper=1.0)
        
        # For Mann-Whitney
        ranks = rankdata(stats_df['u_pvalue'])
        stats_df['u_pvalue_fdr'] = stats_df['u_pvalue'] * n / ranks
        stats_df['u_pvalue_fdr'] = stats_df['u_pvalue_fdr'].clip(upper=1.0)
    
    # Add significance columns
    def get_sig(pval):
        if pval < 0.001:
            return '***'
        elif pval < 0.01:
            return '**'
        elif pval < 0.05:
            return '*'
        else:
            return 'ns'
    
    stats_df['t_significance'] = stats_df['t_pvalue'].apply(get_sig)
    stats_df['t_significance_fdr'] = stats_df['t_pvalue_fdr'].apply(get_sig)
    stats_df['u_significance'] = stats_df['u_pvalue'].apply(get_sig)
    stats_df['u_significance_fdr'] = stats_df['u_pvalue_fdr'].apply(get_sig)

# Save statistical results
stats_path = os.path.join(score_dir, 'module_zscore_statistics_DCM.csv')
stats_df.to_csv(stats_path, index=False)
print(f"\n{'='*60}")
print(f"Statistical results saved to: {stats_path}")
print('='*60)

# Print summary table
print("\n\nSUMMARY TABLE:")
print("="*100)
summary_cols = ['module', 'group1', 'group2', 'mean_diff', 't_pvalue', 't_significance', 'u_pvalue', 'u_significance', 'cohens_d']
print(stats_df[summary_cols].to_string(index=False))

# Save summary table as well
summary_path = os.path.join(score_dir, 'module_zscore_statistics_DCM_summary.csv')
stats_df[summary_cols].to_csv(summary_path, index=False)
print(f"\nSummary saved to: {summary_path}")

# Create combined figure with all modules
print("\n\nCreating combined figure...")

n_modules = len(zscore_cols)
if n_modules > 0:
    fig, axes = plt.subplots(1, n_modules, figsize=(6*n_modules, 6))
    
    if n_modules == 1:
        axes = [axes]
    
    for idx, zscore_col in enumerate(zscore_cols):
        ax = axes[idx]
        module_ct = zscore_col.replace('_module_score_Z_allassays', '')
        
        # Prepare data
        plot_data = pd.DataFrame({
            'zscore': adata_dcm.obs[zscore_col].values,
            'cell_type': adata_dcm.obs[cell_type_col].astype(str).values
        })
        
        plot_data = plot_data.dropna()
        plot_data = plot_data[~plot_data['cell_type'].str.contains('nan|ambiguous', case=False, na=True)]
        plot_data = plot_data[plot_data['cell_type'].isin(label_order)]
        
        if len(plot_data) == 0:
            continue
        
        available_types = [ct for ct in label_order if ct in plot_data['cell_type'].unique()]
        
        plot_data['cell_type'] = pd.Categorical(
            plot_data['cell_type'],
            categories=available_types,
            ordered=True
        )
        
        # Create custom palette
        palette = []
        for ct in available_types:
            if ct == module_ct:
                palette.append(cell_type_colors[ct])
            else:
                palette.append('#D3D3D3')
        
        sns.boxplot(
            data=plot_data,
            x='cell_type',
            y='zscore',
            hue='cell_type',
            order=available_types,
            hue_order=available_types,
            palette=palette,
            ax=ax,
            linewidth=1.5,
            fliersize=3,
            legend=False
        )
        
        # Add significance annotations
        y_max = plot_data['zscore'].max()
        y_range = plot_data['zscore'].max() - plot_data['zscore'].min()
        
        module_comparisons = [r for r in all_stats_results if r['module'] == module_ct]
        
        for i, comp in enumerate(module_comparisons):
            ct1, ct2 = comp['group1'], comp['group2']
            pval = comp['t_pvalue']
            
            if pval < 0.001:
                sig = '***'
            elif pval < 0.01:
                sig = '**'
            elif pval < 0.05:
                sig = '*'
            else:
                sig = 'ns'
            
            x1 = available_types.index(ct1)
            x2 = available_types.index(ct2)
            y_bar = y_max + (i + 1) * y_range * 0.12
            
            ax.plot([x1, x1, x2, x2], [y_bar - y_range*0.02, y_bar, y_bar, y_bar - y_range*0.02], 
                    color='black', linewidth=1)
            ax.text((x1 + x2) / 2, y_bar + y_range*0.01, sig, ha='center', va='bottom', fontsize=10)
        
        ax.set_title(f'{module_ct}', fontsize=14, fontweight='bold')
        ax.set_xlabel('Cell Type', fontsize=12)
        ax.set_ylabel('Z-score', fontsize=12)
        ax.axhline(y=0, color='black', linestyle='--', alpha=0.5, linewidth=1)
        ax.axhline(y=1.65, color='red', linestyle=':', alpha=0.5, linewidth=1)
        
        ax.set_ylim(plot_data['zscore'].min() - y_range*0.1, y_max + (len(module_comparisons) + 1) * y_range * 0.15)
    
    plt.tight_layout()
    save_path = os.path.join(score_dir, 'all_modules_zscore_boxplot_DCM_with_stats.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    
    print(f"Saved {save_path}")

print("\nAll boxplots and statistics saved!")

In [ ]:
#UMAP connecting dots#

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/gmm_085/'
os.makedirs(score_dir, exist_ok=True)

adata = adata_combined

# Small intestine cell types and colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': '#7f7f7f'
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata.obs['batch_plate_well'] = (
    adata.obs['Batch'].astype(str) + '_' + 
    adata.obs['plate_well'].astype(str)
)

# -----------------------------
# EXTRACT UMAP AND METADATA
# -----------------------------
umap_coords = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1': umap_coords[:, 0],
    'UMAP2': umap_coords[:, 1],
    'assay': adata.obs['assay'].values,
    'plate_well': adata.obs['plate_well'].values,
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'Batch': adata.obs['Batch'].values,
    'cell_type_broad': adata.obs['leiden_annot_25'].astype(str).values,
    'module_max_label': adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'cell_index': adata.obs.index.values
})

if 'Dox' in adata.obs.columns:
    plot_df['Dox'] = adata.obs['Dox'].values

# -----------------------------
# FILTER DCM CELLS WITH KNOWN LABELS (EXCLUDE MINUS DOX)
# -----------------------------
print("\n" + "="*60)
print("STEP 1: FILTERING DCM CELLS")
print("="*60)

dcm_mask = plot_df['assay'] == 'pDCM'
dcm_known_mask = dcm_mask & plot_df['module_max_label'].isin(label_order)

# Exclude Minus Dox if Dox column exists
if 'Dox' in plot_df.columns:
    dcm_known_mask = dcm_known_mask & (plot_df['Dox'] != 'Minus')

dcm_cells = plot_df[dcm_known_mask].copy()

print(f"Total scM_DCM cells: {dcm_mask.sum()}")
print(f"DCM cells with known labels (excluding Minus Dox): {dcm_known_mask.sum()}")
print(dcm_cells['module_max_label'].value_counts())

# -----------------------------
# FILTER RNA CELLS (EXCLUDE MINUS DOX)
# -----------------------------
print("\n" + "="*60)
print("STEP 2: FILTERING RNA CELLS")
print("="*60)

rna_mask = plot_df['assay'] == 'pRNA'

# Exclude Minus Dox if Dox column exists
if 'Dox' in plot_df.columns:
    rna_mask = rna_mask & (plot_df['Dox'] != 'Minus')

rna_cells = plot_df[rna_mask].copy()
print(f"Total scM_RNA cells (excluding Minus Dox): {rna_mask.sum()}")
print(rna_cells['cell_type_broad'].value_counts())

# -----------------------------
# CHECK DUPLICATES WITH BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("STEP 3: CHECKING DUPLICATES")
print("="*60)

dcm_bpw_counts = dcm_cells['batch_plate_well'].value_counts()
rna_bpw_counts = rna_cells['batch_plate_well'].value_counts()

print(f"DCM batch_plate_wells with >1 cell: {(dcm_bpw_counts > 1).sum()}")
print(f"RNA batch_plate_wells with >1 cell: {(rna_bpw_counts > 1).sum()}")

# -----------------------------
# MATCH BY BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("STEP 4: FINDING CONNECTIONS USING BATCH_PLATE_WELL")
print("="*60)

dcm_bpw = set(dcm_cells['batch_plate_well'])
rna_bpw = set(rna_cells['batch_plate_well'])
common_bpw = dcm_bpw.intersection(rna_bpw)
print(f"Common batch_plate_wells: {len(common_bpw)}")

# Filter to common
dcm_common = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
rna_common = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]

# Find 1:1 matches
dcm_counts = dcm_common['batch_plate_well'].value_counts()
rna_counts = rna_common['batch_plate_well'].value_counts()

dcm_single = set(dcm_counts[dcm_counts == 1].index)
rna_single = set(rna_counts[rna_counts == 1].index)
both_single = dcm_single.intersection(rna_single)

print(f"1:1 matched batch_plate_wells: {len(both_single)}")

# Build connections
connections = []
for bpw in both_single:
    dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
    rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
    
    if len(dcm_cell) == 1 and len(rna_cell) == 1:
        dcm_row = dcm_cell.iloc[0]
        rna_row = rna_cell.iloc[0]
        
        connections.append({
            'batch_plate_well': bpw,
            'batch': dcm_row['Batch'],
            'plate_well': dcm_row['plate_well'],
            'dcm_umap1': dcm_row['UMAP1'],
            'dcm_umap2': dcm_row['UMAP2'],
            'rna_umap1': rna_row['UMAP1'],
            'rna_umap2': rna_row['UMAP2'],
            'dcm_cell_type': dcm_row['module_max_label'],
            'rna_cell_type': rna_row['cell_type_broad'],
            'transition': f"{dcm_row['module_max_label']} -> {rna_row['cell_type_broad']}"
        })

connections_df = pd.DataFrame(connections)
print(f"Total valid connections: {len(connections_df)}")

# -----------------------------
# PLOT TRANSITIONS
# -----------------------------
if len(connections_df) > 0:
    unique_transitions = connections_df['transition'].unique()
    
    for transition in unique_transitions:
        parts = transition.split(' -> ')
        dcm_ct = parts[0]
        rna_ct = parts[1]
        trans_connections = connections_df[connections_df['transition'] == transition]
        trans_bpw = set(trans_connections['batch_plate_well'])
        
        n_cells = len(trans_connections)
        print(f"\nPlotting transition: {transition} ({n_cells} cells)")

        fig, ax = plt.subplots(figsize=(10, 8))

        # Reference background
        for ct in label_order:
            ref_cells = plot_df[(plot_df['assay'] == 'Ref') & (plot_df['cell_type_broad'] == ct)]
            if len(ref_cells) > 0:
                ax.scatter(ref_cells['UMAP1'], ref_cells['UMAP2'],
                           s=8, c=cell_type_colors[ct], alpha=0.05, marker='o', zorder=1, rasterized=True)

        # Ambiguous reference
        ref_ambig = plot_df[(plot_df['assay'] == 'Ref') & (~plot_df['cell_type_broad'].isin(label_order))]
        if len(ref_ambig) > 0:
            ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                       s=8, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)

        # Arrows
        for _, conn in trans_connections.iterrows():
            arrow = FancyArrowPatch((conn['dcm_umap1'], conn['dcm_umap2']),
                                    (conn['rna_umap1'], conn['rna_umap2']),
                                    arrowstyle='-|>', mutation_scale=20,
                                    color='black', alpha=0.8, linewidth=2.5, zorder=3)
            ax.add_patch(arrow)

        # DCM cells - squares (using batch_plate_well)
        dcm_cells_plot = plot_df[(plot_df['assay'] == 'pDCM') &
                                 (plot_df['batch_plate_well'].isin(trans_bpw)) &
                                 (plot_df['module_max_label'] == dcm_ct)]
        if len(dcm_cells_plot) > 0:
            ax.scatter(dcm_cells_plot['UMAP1'], dcm_cells_plot['UMAP2'],
                       s=100, c=cell_type_colors[dcm_ct], alpha=0.9,
                       edgecolors='black', linewidths=1, marker='s', zorder=4, rasterized=True)

        # RNA cells - circles (using batch_plate_well)
        rna_cells_plot = plot_df[(plot_df['assay'] == 'pRNA') &
                                 (plot_df['batch_plate_well'].isin(trans_bpw)) &
                                 (plot_df['cell_type_broad'] == rna_ct)]
        if len(rna_cells_plot) > 0:
            ax.scatter(rna_cells_plot['UMAP1'], rna_cells_plot['UMAP2'],
                       s=100, c=cell_type_colors.get(rna_ct, '#7f7f7f'), alpha=0.9,
                       edgecolors='black', linewidths=1, marker='o', zorder=4, rasterized=True)

        # Legend
        legend_elements = [
            mlines.Line2D([], [], color=cell_type_colors[dcm_ct], marker='s', linestyle='None', markersize=12,
                          markeredgecolor='black', label=f'DCM ({dcm_ct})'),
            mlines.Line2D([], [], color=cell_type_colors.get(rna_ct, '#7f7f7f'), marker='o', linestyle='None', markersize=12,
                          markeredgecolor='black', label=f'RNA ({rna_ct})'),
            mlines.Line2D([], [], color='black', linestyle='-', linewidth=3, alpha=0.8,
                          marker='>', markersize=10, label='DCM → RNA')
        ]
        ax.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=10)

        ax.set_title(f'Transition: {transition}\n(n = {n_cells} cells, Matched using Batch + Plate_Well)', 
                    fontsize=14, fontweight='bold')
        ax.set_aspect('equal')
        ax.axis('off')

        plt.tight_layout()
        safe_name = transition.replace(' -> ', '_to_').replace(' ', '_')
        save_path = os.path.join(score_dir, f'umap_transition_{safe_name}.png')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f"Saved: {save_path}")

    # Save connections summary
    connections_df.to_csv(os.path.join(score_dir, 'DCM_RNA_connections_small_intestine_batch_plate_well.csv'), index=False)
    print(f"\nSaved connections: {os.path.join(score_dir, 'DCM_RNA_connections_small_intestine_batch_plate_well.csv')}")

else:
    print("No connections found!")

# -----------------------------
# SUMMARY BY TRANSITION
# -----------------------------
print("\n" + "="*60)
print("SUMMARY BY TRANSITION")
print("="*60)

if len(connections_df) > 0:
    transition_counts = connections_df['transition'].value_counts()
    print(transition_counts)
    
    # Stable vs changed
    connections_df['stable'] = connections_df['dcm_cell_type'] == connections_df['rna_cell_type']
    n_stable = connections_df['stable'].sum()
    n_changed = (~connections_df['stable']).sum()
    
    print(f"\nStable (same type): {n_stable} ({n_stable/len(connections_df)*100:.1f}%)")
    print(f"Changed (different type): {n_changed} ({n_changed/len(connections_df)*100:.1f}%)")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
######Big groups######

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/'
os.makedirs(score_dir, exist_ok=True)

adata = adata_combined

# Small intestine cell types and colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': '#7f7f7f'
}

# Lineage colors
lineage_colors = {
    'Stem': '#9370DB',
    'Absorptive': '#FF8C00',
    'Secretory': '#2E8B57',
    'ambiguous': '#7f7f7f'
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']
lineage_order = ['Stem', 'Absorptive', 'Secretory']

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata.obs['batch_plate_well'] = (
    adata.obs['Batch'].astype(str) + '_' + 
    adata.obs['plate_well'].astype(str)
)

# -----------------------------
# FUNCTION TO MAP CELL TYPE TO LINEAGE
# -----------------------------
def map_to_lineage(cell_type):
    if cell_type in ['Stem', 'TA']:
        return 'Stem'
    elif cell_type in ['Enterocyte']:
        return 'Absorptive'
    elif cell_type in ['Paneth', 'Goblet', 'Tuft', 'EEC']:
        return 'Secretory'
    else:
        return 'ambiguous'

# -----------------------------
# EXTRACT UMAP AND METADATA
# -----------------------------
umap_coords = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1': umap_coords[:, 0],
    'UMAP2': umap_coords[:, 1],
    'assay': adata.obs['assay'].values,
    'plate_well': adata.obs['plate_well'].values,
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'Batch': adata.obs['Batch'].values,
    'cell_type_broad': adata.obs['leiden_annot_25'].astype(str).values,
    'module_max_label': adata.obs['ttest_gmm_label_confident'].astype(str).values,
    'cell_index': adata.obs.index.values
})

if 'Dox' in adata.obs.columns:
    plot_df['Dox'] = adata.obs['Dox'].values

# -----------------------------
# ADD LINEAGE COLUMNS
# -----------------------------
print("\n" + "="*60)
print("ADDING LINEAGE COLUMNS")
print("="*60)

plot_df['cell_type_lineage'] = plot_df['cell_type_broad'].apply(map_to_lineage)
plot_df['module_lineage'] = plot_df['module_max_label'].apply(map_to_lineage)

print("\nCell type broad -> Lineage distribution:")
print(plot_df.groupby(['cell_type_broad', 'cell_type_lineage']).size())

# -----------------------------
# FILTER DCM CELLS WITH KNOWN LABELS (EXCLUDE MINUS DOX)
# -----------------------------
print("\n" + "="*60)
print("STEP 1: FILTERING DCM CELLS")
print("="*60)

dcm_mask = plot_df['assay'] == 'pDCM'
dcm_known_mask = dcm_mask & plot_df['module_max_label'].isin(label_order)

# Exclude Minus Dox
if 'Dox' in plot_df.columns:
    dcm_known_mask = dcm_known_mask & (plot_df['Dox'] != 'Minus')

dcm_cells = plot_df[dcm_known_mask].copy()

print(f"Total scM_DCM cells: {dcm_mask.sum()}")
print(f"DCM cells with known labels (excluding Minus Dox): {dcm_known_mask.sum()}")
print("\nDCM cell type distribution:")
print(dcm_cells['module_max_label'].value_counts())
print("\nDCM lineage distribution:")
print(dcm_cells['module_lineage'].value_counts())

# -----------------------------
# FILTER RNA CELLS (EXCLUDE MINUS DOX)
# -----------------------------
print("\n" + "="*60)
print("STEP 2: FILTERING RNA CELLS")
print("="*60)

rna_mask = plot_df['assay'] == 'pRNA'

# Exclude Minus Dox
if 'Dox' in plot_df.columns:
    rna_mask = rna_mask & (plot_df['Dox'] != 'Minus')

rna_cells = plot_df[rna_mask].copy()
print(f"Total scM_RNA cells (excluding Minus Dox): {rna_mask.sum()}")
print("\nRNA cell type distribution:")
print(rna_cells['cell_type_broad'].value_counts())
print("\nRNA lineage distribution:")
print(rna_cells['cell_type_lineage'].value_counts())

# -----------------------------
# CHECK DUPLICATES WITH BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("STEP 3: CHECKING DUPLICATES")
print("="*60)

dcm_bpw_counts = dcm_cells['batch_plate_well'].value_counts()
rna_bpw_counts = rna_cells['batch_plate_well'].value_counts()

print(f"DCM batch_plate_wells with >1 cell: {(dcm_bpw_counts > 1).sum()}")
print(f"RNA batch_plate_wells with >1 cell: {(rna_bpw_counts > 1).sum()}")

# -----------------------------
# MATCH BY BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("STEP 4: FINDING CONNECTIONS USING BATCH_PLATE_WELL")
print("="*60)

dcm_bpw = set(dcm_cells['batch_plate_well'])
rna_bpw = set(rna_cells['batch_plate_well'])
common_bpw = dcm_bpw.intersection(rna_bpw)
print(f"Common batch_plate_wells: {len(common_bpw)}")

# Filter to common
dcm_common = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
rna_common = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]

# Find 1:1 matches
dcm_counts = dcm_common['batch_plate_well'].value_counts()
rna_counts = rna_common['batch_plate_well'].value_counts()

dcm_single = set(dcm_counts[dcm_counts == 1].index)
rna_single = set(rna_counts[rna_counts == 1].index)
both_single = dcm_single.intersection(rna_single)

print(f"1:1 matched batch_plate_wells: {len(both_single)}")

# Build connections
connections = []
for bpw in both_single:
    dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
    rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
    
    if len(dcm_cell) == 1 and len(rna_cell) == 1:
        dcm_row = dcm_cell.iloc[0]
        rna_row = rna_cell.iloc[0]
        
        connections.append({
            'batch_plate_well': bpw,
            'batch': dcm_row['Batch'],
            'plate_well': dcm_row['plate_well'],
            'dcm_umap1': dcm_row['UMAP1'],
            'dcm_umap2': dcm_row['UMAP2'],
            'rna_umap1': rna_row['UMAP1'],
            'rna_umap2': rna_row['UMAP2'],
            # Cell type level
            'dcm_cell_type': dcm_row['module_max_label'],
            'rna_cell_type': rna_row['cell_type_broad'],
            'transition': f"{dcm_row['module_max_label']} -> {rna_row['cell_type_broad']}",
            # Lineage level
            'dcm_lineage': dcm_row['module_lineage'],
            'rna_lineage': rna_row['cell_type_lineage'],
            'lineage_transition': f"{dcm_row['module_lineage']} -> {rna_row['cell_type_lineage']}"
        })

connections_df = pd.DataFrame(connections)
print(f"Total valid connections: {len(connections_df)}")

# -----------------------------
# PRINT TRANSITION SUMMARIES
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("CELL TYPE TRANSITIONS")
    print("="*60)
    print(connections_df['transition'].value_counts())
    
    print("\n" + "="*60)
    print("LINEAGE TRANSITIONS")
    print("="*60)
    print(connections_df['lineage_transition'].value_counts())

# -----------------------------
# PLOT CELL TYPE TRANSITIONS
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("PLOTTING CELL TYPE TRANSITIONS")
    print("="*60)
    
    unique_transitions = connections_df['transition'].unique()
    
    for transition in unique_transitions:
        parts = transition.split(' -> ')
        dcm_ct = parts[0]
        rna_ct = parts[1]
        trans_connections = connections_df[connections_df['transition'] == transition]
        trans_bpw = set(trans_connections['batch_plate_well'])
        
        n_cells = len(trans_connections)
        print(f"\nPlotting transition: {transition} ({n_cells} cells)")

        fig, ax = plt.subplots(figsize=(10, 8))

        # Reference background
        for ct in label_order:
            ref_cells = plot_df[(plot_df['assay'] == 'Ref') & (plot_df['cell_type_broad'] == ct)]
            if len(ref_cells) > 0:
                ax.scatter(ref_cells['UMAP1'], ref_cells['UMAP2'],
                           s=8, c=cell_type_colors[ct], alpha=0.05, marker='o', zorder=1, rasterized=True)

        # Ambiguous reference
        ref_ambig = plot_df[(plot_df['assay'] == 'Ref') & (~plot_df['cell_type_broad'].isin(label_order))]
        if len(ref_ambig) > 0:
            ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                       s=8, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)

        # Arrows
        for _, conn in trans_connections.iterrows():
            arrow = FancyArrowPatch((conn['dcm_umap1'], conn['dcm_umap2']),
                                    (conn['rna_umap1'], conn['rna_umap2']),
                                    arrowstyle='-|>', mutation_scale=20,
                                    color='black', alpha=0.8, linewidth=2.5, zorder=3)
            ax.add_patch(arrow)

        # DCM cells - squares (using batch_plate_well)
        dcm_cells_plot = plot_df[(plot_df['assay'] == 'pDCM') &
                                 (plot_df['batch_plate_well'].isin(trans_bpw)) &
                                 (plot_df['module_max_label'] == dcm_ct)]
        if len(dcm_cells_plot) > 0:
            ax.scatter(dcm_cells_plot['UMAP1'], dcm_cells_plot['UMAP2'],
                       s=100, c=cell_type_colors[dcm_ct], alpha=0.9,
                       edgecolors='black', linewidths=1, marker='s', zorder=4, rasterized=True)

        # RNA cells - circles (using batch_plate_well)
        rna_cells_plot = plot_df[(plot_df['assay'] == 'pRNA') &
                                 (plot_df['batch_plate_well'].isin(trans_bpw)) &
                                 (plot_df['cell_type_broad'] == rna_ct)]
        if len(rna_cells_plot) > 0:
            ax.scatter(rna_cells_plot['UMAP1'], rna_cells_plot['UMAP2'],
                       s=100, c=cell_type_colors.get(rna_ct, '#7f7f7f'), alpha=0.9,
                       edgecolors='black', linewidths=1, marker='o', zorder=4, rasterized=True)

        # Legend
        legend_elements = [
            mlines.Line2D([], [], color=cell_type_colors[dcm_ct], marker='s', linestyle='None', markersize=12,
                          markeredgecolor='black', label=f'DCM ({dcm_ct})'),
            mlines.Line2D([], [], color=cell_type_colors.get(rna_ct, '#7f7f7f'), marker='o', linestyle='None', markersize=12,
                          markeredgecolor='black', label=f'RNA ({rna_ct})'),
            mlines.Line2D([], [], color='black', linestyle='-', linewidth=3, alpha=0.8,
                          marker='>', markersize=10, label='DCM → RNA')
        ]
        ax.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=10)

        ax.set_title(f'Transition: {transition}\n(n = {n_cells} cells, Matched using Batch + Plate_Well)', 
                    fontsize=14, fontweight='bold')
        ax.set_aspect('equal')
        ax.axis('off')

        plt.tight_layout()
        safe_name = transition.replace(' -> ', '_to_').replace(' ', '_')
        save_path = os.path.join(score_dir, f'umap_transition_{safe_name}.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f"Saved: {save_path}")

# -----------------------------
# PLOT LINEAGE TRANSITIONS
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("PLOTTING LINEAGE TRANSITIONS")
    print("="*60)
    
    unique_lineage_transitions = connections_df['lineage_transition'].unique()
    
    for lineage_transition in unique_lineage_transitions:
        parts = lineage_transition.split(' -> ')
        dcm_lineage = parts[0]
        rna_lineage = parts[1]
        
        # Skip ambiguous transitions
        if dcm_lineage == 'ambiguous' or rna_lineage == 'ambiguous':
            print(f"Skipping ambiguous transition: {lineage_transition}")
            continue
        
        trans_connections = connections_df[connections_df['lineage_transition'] == lineage_transition]
        trans_bpw = set(trans_connections['batch_plate_well'])
        
        n_cells = len(trans_connections)
        print(f"\nPlotting lineage transition: {lineage_transition} ({n_cells} cells)")

        fig, ax = plt.subplots(figsize=(10, 8))

        # Reference background - colored by lineage
        for ct in label_order:
            ref_cells = plot_df[(plot_df['assay'] == 'Ref') & (plot_df['cell_type_broad'] == ct)]
            if len(ref_cells) > 0:
                ct_lineage = map_to_lineage(ct)
                ax.scatter(ref_cells['UMAP1'], ref_cells['UMAP2'],
                           s=8, c=lineage_colors.get(ct_lineage, 'lightgray'), 
                           alpha=0.05, marker='o', zorder=1, rasterized=True)

        # Ambiguous reference
        ref_ambig = plot_df[(plot_df['assay'] == 'Ref') & (~plot_df['cell_type_broad'].isin(label_order))]
        if len(ref_ambig) > 0:
            ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                       s=8, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)

        # Arrows
        for _, conn in trans_connections.iterrows():
            arrow = FancyArrowPatch((conn['dcm_umap1'], conn['dcm_umap2']),
                                    (conn['rna_umap1'], conn['rna_umap2']),
                                    arrowstyle='-|>', mutation_scale=20,
                                    color='black', alpha=0.8, linewidth=2.5, zorder=3)
            ax.add_patch(arrow)

        # DCM cells - squares (using batch_plate_well)
        dcm_cells_plot = plot_df[(plot_df['assay'] == 'pDCM') &
                                 (plot_df['batch_plate_well'].isin(trans_bpw)) &
                                 (plot_df['module_lineage'] == dcm_lineage)]
        if len(dcm_cells_plot) > 0:
            ax.scatter(dcm_cells_plot['UMAP1'], dcm_cells_plot['UMAP2'],
                       s=100, c=lineage_colors[dcm_lineage], alpha=0.9,
                       edgecolors='black', linewidths=1, marker='s', zorder=4, rasterized=True)

        # RNA cells - circles (using batch_plate_well)
        rna_cells_plot = plot_df[(plot_df['assay'] == 'pRNA') &
                                 (plot_df['batch_plate_well'].isin(trans_bpw)) &
                                 (plot_df['cell_type_lineage'] == rna_lineage)]
        if len(rna_cells_plot) > 0:
            ax.scatter(rna_cells_plot['UMAP1'], rna_cells_plot['UMAP2'],
                       s=100, c=lineage_colors.get(rna_lineage, '#7f7f7f'), alpha=0.9,
                       edgecolors='black', linewidths=1, marker='o', zorder=4, rasterized=True)

        # Legend
        legend_elements = [
            mlines.Line2D([], [], color=lineage_colors[dcm_lineage], marker='s', linestyle='None', markersize=12,
                          markeredgecolor='black', label=f'DCM ({dcm_lineage})'),
            mlines.Line2D([], [], color=lineage_colors.get(rna_lineage, '#7f7f7f'), marker='o', linestyle='None', markersize=12,
                          markeredgecolor='black', label=f'RNA ({rna_lineage})'),
            mlines.Line2D([], [], color='black', linestyle='-', linewidth=3, alpha=0.8,
                          marker='>', markersize=10, label='DCM → RNA')
        ]
        
        # Add cell type breakdown
        dcm_ct_counts = trans_connections['dcm_cell_type'].value_counts()
        rna_ct_counts = trans_connections['rna_cell_type'].value_counts()
        
        ax.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=10)

        # Title with cell type breakdown
        dcm_breakdown = ', '.join([f"{ct}: {count}" for ct, count in dcm_ct_counts.items()])
        rna_breakdown = ', '.join([f"{ct}: {count}" for ct, count in rna_ct_counts.items()])
        
        ax.set_title(f'Lineage Transition: {lineage_transition}\n(n = {n_cells} cells, Matched using Batch + Plate_Well)\nDCM: {dcm_breakdown} | RNA: {rna_breakdown}', 
                    fontsize=12, fontweight='bold')
        ax.set_aspect('equal')
        ax.axis('off')

        plt.tight_layout()
        safe_name = lineage_transition.replace(' -> ', '_to_').replace(' ', '_')
        save_path = os.path.join(score_dir, f'umap_lineage_transition_{safe_name}.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f"Saved: {save_path}")

# -----------------------------
# SUMMARY HEATMAP OF LINEAGE TRANSITIONS
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("CREATING LINEAGE TRANSITION SUMMARY")
    print("="*60)
    
    # Create transition matrix
    lineage_transition_matrix = pd.crosstab(
        connections_df['dcm_lineage'],
        connections_df['rna_lineage'],
        rownames=['DCM Lineage'],
        colnames=['RNA Lineage']
    )
    
    # Reorder
    ordered_lineages = ['Stem', 'Absorptive', 'Secretory', 'ambiguous']
    lineage_transition_matrix = lineage_transition_matrix.reindex(
        index=[l for l in ordered_lineages if l in lineage_transition_matrix.index],
        columns=[l for l in ordered_lineages if l in lineage_transition_matrix.columns]
    )
    
    print("\nLineage Transition Matrix:")
    print(lineage_transition_matrix)
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(8, 6))
    
    sns.heatmap(lineage_transition_matrix, annot=True, fmt='d', cmap='viridis',
                cbar_kws={'label': 'Number of cells'}, ax=ax)
    
    ax.set_title('Lineage Transitions: DCM → RNA\n(Matched using Batch + Plate_Well)', 
                fontsize=14, fontweight='bold')
    ax.set_xlabel('RNA Lineage', fontsize=12)
    ax.set_ylabel('DCM Lineage', fontsize=12)
    
    plt.tight_layout()
    save_path = os.path.join(score_dir, 'lineage_transition_heatmap.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved: {save_path}")
    
    # Cell type transition matrix
    cell_type_transition_matrix = pd.crosstab(
        connections_df['dcm_cell_type'],
        connections_df['rna_cell_type'],
        rownames=['DCM Cell Type'],
        colnames=['RNA Cell Type']
    )
    
    # Reorder
    cell_type_transition_matrix = cell_type_transition_matrix.reindex(
        index=[l for l in label_order if l in cell_type_transition_matrix.index],
        columns=[l for l in label_order if l in cell_type_transition_matrix.columns]
    )
    
    print("\nCell Type Transition Matrix:")
    print(cell_type_transition_matrix)
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(10, 8))
    
    sns.heatmap(cell_type_transition_matrix, annot=True, fmt='d', cmap='viridis',
                cbar_kws={'label': 'Number of cells'}, ax=ax)
    
    ax.set_title('Cell Type Transitions: DCM → RNA\n(Matched using Batch + Plate_Well)', 
                fontsize=14, fontweight='bold')
    ax.set_xlabel('RNA Cell Type', fontsize=12)
    ax.set_ylabel('DCM Cell Type', fontsize=12)
    
    plt.tight_layout()
    save_path = os.path.join(score_dir, 'celltype_transition_heatmap.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved: {save_path}")

# -----------------------------
# SAVE CONNECTIONS WITH LINEAGE INFO
# -----------------------------
if len(connections_df) > 0:
    connections_df.to_csv(os.path.join(score_dir, 'DCM_RNA_connections_with_lineage_batch_plate_well.csv'), index=False)
    print(f"\nSaved connections to: {os.path.join(score_dir, 'DCM_RNA_connections_with_lineage_batch_plate_well.csv')}")

# -----------------------------
# ADD LINEAGE TO ADATA_COMBINED
# -----------------------------
print("\n" + "="*60)
print("ADDING LINEAGE COLUMNS TO ADATA_COMBINED")
print("="*60)

adata_combined.obs['cell_type_lineage'] = adata_combined.obs['leiden_annot_25'].astype(str).apply(map_to_lineage)
adata_combined.obs['module_lineage'] = adata_combined.obs['ttest_gmm_label_confident'].astype(str).apply(map_to_lineage)

print("\nLineage distribution in adata_combined:")
print("\ncell_type_lineage:")
print(adata_combined.obs['cell_type_lineage'].value_counts())
print("\nmodule_lineage:")
print(adata_combined.obs['module_lineage'].value_counts())

# -----------------------------
# SUMMARY
# -----------------------------
print("\n" + "="*60)
print("SUMMARY")
print("="*60)

if len(connections_df) > 0:
    connections_df['stable'] = connections_df['dcm_cell_type'] == connections_df['rna_cell_type']
    connections_df['lineage_stable'] = connections_df['dcm_lineage'] == connections_df['rna_lineage']
    
    n_stable = connections_df['stable'].sum()
    n_lineage_stable = connections_df['lineage_stable'].sum()
    
    print(f"""
Total connections: {len(connections_df)}

Cell Type Level:
  Stable (same type): {n_stable} ({n_stable/len(connections_df)*100:.1f}%)
  Changed (different type): {len(connections_df) - n_stable} ({(len(connections_df) - n_stable)/len(connections_df)*100:.1f}%)

Lineage Level:
  Stable (same lineage): {n_lineage_stable} ({n_lineage_stable/len(connections_df)*100:.1f}%)
  Changed (different lineage): {len(connections_df) - n_lineage_stable} ({(len(connections_df) - n_lineage_stable)/len(connections_df)*100:.1f}%)
""")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
## rest to Stem or TA

In [ ]:
# Add this section after the lineage transitions plotting and before the summary heatmap
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/'
os.makedirs(score_dir, exist_ok=True)

adata = adata_combined

# Small intestine cell types and colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': '#7f7f7f'
}

# Lineage colors
lineage_colors = {
    'Stem': '#9370DB',
    'Absorptive': '#FF8C00',
    'Secretory': '#2E8B57',
    'ambiguous': '#7f7f7f'
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']
lineage_order = ['Stem', 'Absorptive', 'Secretory']

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata.obs['batch_plate_well'] = (
    adata.obs['Batch'].astype(str) + '_' + 
    adata.obs['plate_well'].astype(str)
)

# -----------------------------
# FUNCTION TO MAP CELL TYPE TO LINEAGE
# -----------------------------
def map_to_lineage(cell_type):
    if cell_type in ['Stem', 'TA']:
        return 'Stem'
    elif cell_type in ['Enterocyte']:
        return 'Absorptive'
    elif cell_type in ['Paneth', 'Goblet', 'Tuft', 'EEC']:
        return 'Secretory'
    else:
        return 'ambiguous'

# -----------------------------
# EXTRACT UMAP AND METADATA
# -----------------------------
umap_coords = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1': umap_coords[:, 0],
    'UMAP2': umap_coords[:, 1],
    'assay': adata.obs['assay'].values,
    'plate_well': adata.obs['plate_well'].values,
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'Batch': adata.obs['Batch'].values,
    'cell_type_broad': adata.obs['leiden_annot_25'].astype(str).values,
    'module_max_label': adata.obs['ttest_gmm_label_confident'].astype(str).values,
    'cell_index': adata.obs.index.values
})

if 'Dox' in adata.obs.columns:
    plot_df['Dox'] = adata.obs['Dox'].values

# -----------------------------
# ADD LINEAGE COLUMNS
# -----------------------------
print("\n" + "="*60)
print("ADDING LINEAGE COLUMNS")
print("="*60)

plot_df['cell_type_lineage'] = plot_df['cell_type_broad'].apply(map_to_lineage)
plot_df['module_lineage'] = plot_df['module_max_label'].apply(map_to_lineage)

print("\nCell type broad -> Lineage distribution:")
print(plot_df.groupby(['cell_type_broad', 'cell_type_lineage']).size())

# -----------------------------
# FILTER DCM CELLS WITH KNOWN LABELS (EXCLUDE MINUS DOX)
# -----------------------------
print("\n" + "="*60)
print("STEP 1: FILTERING DCM CELLS")
print("="*60)

dcm_mask = plot_df['assay'] == 'pDCM'
dcm_known_mask = dcm_mask & plot_df['module_max_label'].isin(label_order)

# Exclude Minus Dox
if 'Dox' in plot_df.columns:
    dcm_known_mask = dcm_known_mask & (plot_df['Dox'] != 'Minus')

dcm_cells = plot_df[dcm_known_mask].copy()

print(f"Total scM_DCM cells: {dcm_mask.sum()}")
print(f"DCM cells with known labels (excluding Minus Dox): {dcm_known_mask.sum()}")
print("\nDCM cell type distribution:")
print(dcm_cells['module_max_label'].value_counts())
print("\nDCM lineage distribution:")
print(dcm_cells['module_lineage'].value_counts())

# -----------------------------
# FILTER RNA CELLS (EXCLUDE MINUS DOX)
# -----------------------------
print("\n" + "="*60)
print("STEP 2: FILTERING RNA CELLS")
print("="*60)

rna_mask = plot_df['assay'] == 'pRNA'

# Exclude Minus Dox
if 'Dox' in plot_df.columns:
    rna_mask = rna_mask & (plot_df['Dox'] != 'Minus')

rna_cells = plot_df[rna_mask].copy()
print(f"Total scM_RNA cells (excluding Minus Dox): {rna_mask.sum()}")
print("\nRNA cell type distribution:")
print(rna_cells['cell_type_broad'].value_counts())
print("\nRNA lineage distribution:")
print(rna_cells['cell_type_lineage'].value_counts())

# -----------------------------
# CHECK DUPLICATES WITH BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("STEP 3: CHECKING DUPLICATES")
print("="*60)

dcm_bpw_counts = dcm_cells['batch_plate_well'].value_counts()
rna_bpw_counts = rna_cells['batch_plate_well'].value_counts()

print(f"DCM batch_plate_wells with >1 cell: {(dcm_bpw_counts > 1).sum()}")
print(f"RNA batch_plate_wells with >1 cell: {(rna_bpw_counts > 1).sum()}")

# -----------------------------
# MATCH BY BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("STEP 4: FINDING CONNECTIONS USING BATCH_PLATE_WELL")
print("="*60)

dcm_bpw = set(dcm_cells['batch_plate_well'])
rna_bpw = set(rna_cells['batch_plate_well'])
common_bpw = dcm_bpw.intersection(rna_bpw)
print(f"Common batch_plate_wells: {len(common_bpw)}")

# Filter to common
dcm_common = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
rna_common = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]

# Find 1:1 matches
dcm_counts = dcm_common['batch_plate_well'].value_counts()
rna_counts = rna_common['batch_plate_well'].value_counts()

dcm_single = set(dcm_counts[dcm_counts == 1].index)
rna_single = set(rna_counts[rna_counts == 1].index)
both_single = dcm_single.intersection(rna_single)

print(f"1:1 matched batch_plate_wells: {len(both_single)}")

# Build connections
connections = []
for bpw in both_single:
    dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
    rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
    
    if len(dcm_cell) == 1 and len(rna_cell) == 1:
        dcm_row = dcm_cell.iloc[0]
        rna_row = rna_cell.iloc[0]
        
        connections.append({
            'batch_plate_well': bpw,
            'batch': dcm_row['Batch'],
            'plate_well': dcm_row['plate_well'],
            'dcm_umap1': dcm_row['UMAP1'],
            'dcm_umap2': dcm_row['UMAP2'],
            'rna_umap1': rna_row['UMAP1'],
            'rna_umap2': rna_row['UMAP2'],
            # Cell type level
            'dcm_cell_type': dcm_row['module_max_label'],
            'rna_cell_type': rna_row['cell_type_broad'],
            'transition': f"{dcm_row['module_max_label']} -> {rna_row['cell_type_broad']}",
            # Lineage level
            'dcm_lineage': dcm_row['module_lineage'],
            'rna_lineage': rna_row['cell_type_lineage'],
            'lineage_transition': f"{dcm_row['module_lineage']} -> {rna_row['cell_type_lineage']}"
        })

connections_df = pd.DataFrame(connections)
print(f"Total valid connections: {len(connections_df)}")

# -----------------------------
# PRINT TRANSITION SUMMARIES
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("CELL TYPE TRANSITIONS")
    print("="*60)
    print(connections_df['transition'].value_counts())
    
    print("\n" + "="*60)
    print("LINEAGE TRANSITIONS")
    print("="*60)
    print(connections_df['lineage_transition'].value_counts())

# -----------------------------
# PLOT CELL TYPE TRANSITIONS
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("PLOTTING CELL TYPE TRANSITIONS")
    print("="*60)
    
    unique_transitions = connections_df['transition'].unique()
    
    for transition in unique_transitions:
        parts = transition.split(' -> ')
        dcm_ct = parts[0]
        rna_ct = parts[1]
        trans_connections = connections_df[connections_df['transition'] == transition]
        trans_bpw = set(trans_connections['batch_plate_well'])
        
        n_cells = len(trans_connections)
        print(f"\nPlotting transition: {transition} ({n_cells} cells)")

        fig, ax = plt.subplots(figsize=(10, 8))

        # Reference background
        for ct in label_order:
            ref_cells = plot_df[(plot_df['assay'] == 'Ref') & (plot_df['cell_type_broad'] == ct)]
            if len(ref_cells) > 0:
                ax.scatter(ref_cells['UMAP1'], ref_cells['UMAP2'],
                           s=8, c=cell_type_colors[ct], alpha=0.05, marker='o', zorder=1, rasterized=True)

        # Ambiguous reference
        ref_ambig = plot_df[(plot_df['assay'] == 'Ref') & (~plot_df['cell_type_broad'].isin(label_order))]
        if len(ref_ambig) > 0:
            ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                       s=8, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)

        # Arrows
        for _, conn in trans_connections.iterrows():
            arrow = FancyArrowPatch((conn['dcm_umap1'], conn['dcm_umap2']),
                                    (conn['rna_umap1'], conn['rna_umap2']),
                                    arrowstyle='-|>', mutation_scale=20,
                                    color='black', alpha=0.8, linewidth=2.5, zorder=3)
            ax.add_patch(arrow)

        # DCM cells - squares (using batch_plate_well)
        dcm_cells_plot = plot_df[(plot_df['assay'] == 'pDCM') &
                                 (plot_df['batch_plate_well'].isin(trans_bpw)) &
                                 (plot_df['module_max_label'] == dcm_ct)]
        if len(dcm_cells_plot) > 0:
            ax.scatter(dcm_cells_plot['UMAP1'], dcm_cells_plot['UMAP2'],
                       s=100, c=cell_type_colors[dcm_ct], alpha=0.9,
                       edgecolors='black', linewidths=1, marker='s', zorder=4, rasterized=True)

        # RNA cells - circles (using batch_plate_well)
        rna_cells_plot = plot_df[(plot_df['assay'] == 'pRNA') &
                                 (plot_df['batch_plate_well'].isin(trans_bpw)) &
                                 (plot_df['cell_type_broad'] == rna_ct)]
        if len(rna_cells_plot) > 0:
            ax.scatter(rna_cells_plot['UMAP1'], rna_cells_plot['UMAP2'],
                       s=100, c=cell_type_colors.get(rna_ct, '#7f7f7f'), alpha=0.9,
                       edgecolors='black', linewidths=1, marker='o', zorder=4, rasterized=True)

        # Legend
        legend_elements = [
            mlines.Line2D([], [], color=cell_type_colors[dcm_ct], marker='s', linestyle='None', markersize=12,
                          markeredgecolor='black', label=f'DCM ({dcm_ct})'),
            mlines.Line2D([], [], color=cell_type_colors.get(rna_ct, '#7f7f7f'), marker='o', linestyle='None', markersize=12,
                          markeredgecolor='black', label=f'RNA ({rna_ct})'),
            mlines.Line2D([], [], color='black', linestyle='-', linewidth=3, alpha=0.8,
                          marker='>', markersize=10, label='DCM → RNA')
        ]
        ax.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=10)

        ax.set_title(f'Transition: {transition}\n(n = {n_cells} cells, Matched using Batch + Plate_Well)', 
                    fontsize=14, fontweight='bold')
        ax.set_aspect('equal')
        ax.axis('off')

        plt.tight_layout()
        safe_name = transition.replace(' -> ', '_to_').replace(' ', '_')
        save_path = os.path.join(score_dir, f'umap_transition_{safe_name}.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f"Saved: {save_path}")

# -----------------------------
# PLOT LINEAGE TRANSITIONS
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("PLOTTING LINEAGE TRANSITIONS")
    print("="*60)
    
    unique_lineage_transitions = connections_df['lineage_transition'].unique()
    
    for lineage_transition in unique_lineage_transitions:
        parts = lineage_transition.split(' -> ')
        dcm_lineage = parts[0]
        rna_lineage = parts[1]
        
        # Skip ambiguous transitions
        if dcm_lineage == 'ambiguous' or rna_lineage == 'ambiguous':
            print(f"Skipping ambiguous transition: {lineage_transition}")
            continue
        
        trans_connections = connections_df[connections_df['lineage_transition'] == lineage_transition]
        trans_bpw = set(trans_connections['batch_plate_well'])
        
        n_cells = len(trans_connections)
        print(f"\nPlotting lineage transition: {lineage_transition} ({n_cells} cells)")

        fig, ax = plt.subplots(figsize=(10, 8))

        # Reference background - colored by lineage
        for ct in label_order:
            ref_cells = plot_df[(plot_df['assay'] == 'Ref') & (plot_df['cell_type_broad'] == ct)]
            if len(ref_cells) > 0:
                ct_lineage = map_to_lineage(ct)
                ax.scatter(ref_cells['UMAP1'], ref_cells['UMAP2'],
                           s=8, c=lineage_colors.get(ct_lineage, 'lightgray'), 
                           alpha=0.05, marker='o', zorder=1, rasterized=True)

        # Ambiguous reference
        ref_ambig = plot_df[(plot_df['assay'] == 'Ref') & (~plot_df['cell_type_broad'].isin(label_order))]
        if len(ref_ambig) > 0:
            ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                       s=8, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)

        # Arrows
        for _, conn in trans_connections.iterrows():
            arrow = FancyArrowPatch((conn['dcm_umap1'], conn['dcm_umap2']),
                                    (conn['rna_umap1'], conn['rna_umap2']),
                                    arrowstyle='-|>', mutation_scale=20,
                                    color='black', alpha=0.8, linewidth=2.5, zorder=3)
            ax.add_patch(arrow)

        # DCM cells - squares (using batch_plate_well)
        dcm_cells_plot = plot_df[(plot_df['assay'] == 'pDCM') &
                                 (plot_df['batch_plate_well'].isin(trans_bpw)) &
                                 (plot_df['module_lineage'] == dcm_lineage)]
        if len(dcm_cells_plot) > 0:
            ax.scatter(dcm_cells_plot['UMAP1'], dcm_cells_plot['UMAP2'],
                       s=100, c=lineage_colors[dcm_lineage], alpha=0.9,
                       edgecolors='black', linewidths=1, marker='s', zorder=4, rasterized=True)

        # RNA cells - circles (using batch_plate_well)
        rna_cells_plot = plot_df[(plot_df['assay'] == 'pRNA') &
                                 (plot_df['batch_plate_well'].isin(trans_bpw)) &
                                 (plot_df['cell_type_lineage'] == rna_lineage)]
        if len(rna_cells_plot) > 0:
            ax.scatter(rna_cells_plot['UMAP1'], rna_cells_plot['UMAP2'],
                       s=100, c=lineage_colors.get(rna_lineage, '#7f7f7f'), alpha=0.9,
                       edgecolors='black', linewidths=1, marker='o', zorder=4, rasterized=True)

        # Legend
        legend_elements = [
            mlines.Line2D([], [], color=lineage_colors[dcm_lineage], marker='s', linestyle='None', markersize=12,
                          markeredgecolor='black', label=f'DCM ({dcm_lineage})'),
            mlines.Line2D([], [], color=lineage_colors.get(rna_lineage, '#7f7f7f'), marker='o', linestyle='None', markersize=12,
                          markeredgecolor='black', label=f'RNA ({rna_lineage})'),
            mlines.Line2D([], [], color='black', linestyle='-', linewidth=3, alpha=0.8,
                          marker='>', markersize=10, label='DCM → RNA')
        ]
        
        # Add cell type breakdown
        dcm_ct_counts = trans_connections['dcm_cell_type'].value_counts()
        rna_ct_counts = trans_connections['rna_cell_type'].value_counts()
        
        ax.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=10)

        # Title with cell type breakdown
        dcm_breakdown = ', '.join([f"{ct}: {count}" for ct, count in dcm_ct_counts.items()])
        rna_breakdown = ', '.join([f"{ct}: {count}" for ct, count in rna_ct_counts.items()])
        
        ax.set_title(f'Lineage Transition: {lineage_transition}\n(n = {n_cells} cells, Matched using Batch + Plate_Well)\nDCM: {dcm_breakdown} | RNA: {rna_breakdown}', 
                    fontsize=12, fontweight='bold')
        ax.set_aspect('equal')
        ax.axis('off')

        plt.tight_layout()
        safe_name = lineage_transition.replace(' -> ', '_to_').replace(' ', '_')
        save_path = os.path.join(score_dir, f'umap_lineage_transition_{safe_name}.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f"Saved: {save_path}")
# -----------------------------
# PLOT TRANSITIONS TO STEM OR TA (FROM NON-STEM/TA CELLS)
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("PLOTTING TRANSITIONS TO STEM OR TA (FROM OTHER CELL TYPES)")
    print("="*60)
    
    # Filter for transitions TO Stem or TA, but NOT FROM Stem or TA
    to_stem_or_ta = connections_df[
        (connections_df['rna_cell_type'].isin(['Stem', 'TA'])) &
        (~connections_df['dcm_cell_type'].isin(['Stem', 'TA']))
    ].copy()
    
    print(f"\nTotal transitions to Stem or TA from other cell types: {len(to_stem_or_ta)}")
    print("\nBreakdown:")
    print(to_stem_or_ta['transition'].value_counts())
    
    if len(to_stem_or_ta) > 0:
        # Plot for each target (Stem and TA)
        for target_ct in ['Stem', 'TA']:
            target_transitions = to_stem_or_ta[to_stem_or_ta['rna_cell_type'] == target_ct]
            
            if len(target_transitions) == 0:
                print(f"\nNo transitions to {target_ct} from other cell types")
                continue
            
            n_cells = len(target_transitions)
            print(f"\nPlotting all transitions to {target_ct} ({n_cells} cells)")
            print("Source cell types:")
            print(target_transitions['dcm_cell_type'].value_counts())
            
            trans_bpw = set(target_transitions['batch_plate_well'])
            
            fig, ax = plt.subplots(figsize=(12, 10))
            
            # Reference background
            for ct in label_order:
                ref_cells = plot_df[(plot_df['assay'] == 'Ref') & (plot_df['cell_type_broad'] == ct)]
                if len(ref_cells) > 0:
                    ax.scatter(ref_cells['UMAP1'], ref_cells['UMAP2'],
                               s=8, c=cell_type_colors[ct], alpha=0.05, marker='o', zorder=1, rasterized=True)
            
            # Ambiguous reference
            ref_ambig = plot_df[(plot_df['assay'] == 'Ref') & (~plot_df['cell_type_broad'].isin(label_order))]
            if len(ref_ambig) > 0:
                ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                           s=8, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)
            
            # Arrows - color by source cell type
            for _, conn in target_transitions.iterrows():
                source_color = cell_type_colors.get(conn['dcm_cell_type'], '#7f7f7f')
                arrow = FancyArrowPatch((conn['dcm_umap1'], conn['dcm_umap2']),
                                        (conn['rna_umap1'], conn['rna_umap2']),
                                        arrowstyle='-|>', mutation_scale=20,
                                        color=source_color, alpha=0.6, linewidth=2, zorder=3)
                ax.add_patch(arrow)
            
            # DCM cells - squares colored by their original cell type
            dcm_cells_plot = plot_df[(plot_df['assay'] == 'pDCM') &
                                     (plot_df['batch_plate_well'].isin(trans_bpw))]
            
            for source_ct in target_transitions['dcm_cell_type'].unique():
                dcm_ct_cells = dcm_cells_plot[dcm_cells_plot['module_max_label'] == source_ct]
                if len(dcm_ct_cells) > 0:
                    ax.scatter(dcm_ct_cells['UMAP1'], dcm_ct_cells['UMAP2'],
                               s=100, c=cell_type_colors.get(source_ct, '#7f7f7f'), alpha=0.9,
                               edgecolors='black', linewidths=1, marker='s', zorder=4, rasterized=True)
            
            # RNA cells - circles (all target cell type)
            rna_cells_plot = plot_df[(plot_df['assay'] == 'pRNA') &
                                     (plot_df['batch_plate_well'].isin(trans_bpw)) &
                                     (plot_df['cell_type_broad'] == target_ct)]
            if len(rna_cells_plot) > 0:
                ax.scatter(rna_cells_plot['UMAP1'], rna_cells_plot['UMAP2'],
                           s=100, c=cell_type_colors[target_ct], alpha=0.9,
                           edgecolors='black', linewidths=1.5, marker='o', zorder=4, rasterized=True)
            
            # Legend
            legend_elements = []
            
            # Add source cell types
            for source_ct in sorted(target_transitions['dcm_cell_type'].unique()):
                count = (target_transitions['dcm_cell_type'] == source_ct).sum()
                legend_elements.append(
                    mlines.Line2D([], [], color=cell_type_colors.get(source_ct, '#7f7f7f'), 
                                  marker='s', linestyle='None', markersize=10,
                                  markeredgecolor='black', label=f'DCM {source_ct} (n={count})')
                )
            
            # Add target
            legend_elements.append(
                mlines.Line2D([], [], color=cell_type_colors[target_ct], 
                              marker='o', linestyle='None', markersize=10,
                              markeredgecolor='black', markeredgewidth=1.5, 
                              label=f'RNA {target_ct} (n={n_cells})')
            )
            
            ax.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=9, ncol=1)
            
            ax.set_title(f'All Transitions to {target_ct}\n(n = {n_cells} cells, Matched using Batch + Plate_Well)', 
                        fontsize=14, fontweight='bold')
            ax.set_aspect('equal')
            ax.axis('off')
            
            plt.tight_layout()
            save_path = os.path.join(score_dir, f'umap_all_to_{target_ct}.pdf')
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.show()
            plt.close()
            print(f"Saved: {save_path}")
        
        # Create summary heatmap for these specific transitions
        print("\n" + "="*60)
        print("CREATING SUMMARY HEATMAP: ALL → STEM/TA")
        print("="*60)
        
        transition_matrix = pd.crosstab(
            to_stem_or_ta['dcm_cell_type'],
            to_stem_or_ta['rna_cell_type'],
            rownames=['DCM Cell Type'],
            colnames=['RNA Cell Type']
        )
        
        # Reorder - exclude Stem and TA from DCM rows
        dcm_order = [ct for ct in label_order if ct not in ['Stem', 'TA'] and ct in transition_matrix.index]
        rna_order = ['Stem', 'TA']
        
        transition_matrix = transition_matrix.reindex(
            index=dcm_order,
            columns=[ct for ct in rna_order if ct in transition_matrix.columns]
        ).fillna(0).astype(int)
        
        print("\nTransition Matrix (DCM → Stem/TA):")
        print(transition_matrix)
        
        # Plot heatmap
        fig, ax = plt.subplots(figsize=(6, 8))
        
        sns.heatmap(transition_matrix, annot=True, fmt='d', cmap='YlOrRd',
                    cbar_kws={'label': 'Number of cells'}, ax=ax,
                    linewidths=0.5, linecolor='white')
        
        ax.set_title('Transitions to Stem/TA from Other Cell Types\n(Matched using Batch + Plate_Well)', 
                    fontsize=13, fontweight='bold')
        ax.set_xlabel('RNA Cell Type', fontsize=11, fontweight='bold')
        ax.set_ylabel('DCM Cell Type', fontsize=11, fontweight='bold')
        
        plt.tight_layout()
        save_path = os.path.join(score_dir, 'heatmap_to_stem_ta.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f"Saved: {save_path}")
        
        # Print statistics
        print("\n" + "="*60)
        print("STATISTICS: TRANSITIONS TO STEM/TA")
        print("="*60)
        
        to_stem = to_stem_or_ta[to_stem_or_ta['rna_cell_type'] == 'Stem']
        to_ta = to_stem_or_ta[to_stem_or_ta['rna_cell_type'] == 'TA']
        
        print(f"\nTo Stem: {len(to_stem)} cells")
        print("  From:")
        print(to_stem['dcm_cell_type'].value_counts().to_string())
        
        print(f"\nTo TA: {len(to_ta)} cells")
        print("  From:")
        print(to_ta['dcm_cell_type'].value_counts().to_string())
    
    else:
        print("\nNo transitions found from other cell types to Stem or TA")

        # -----------------------------
# SUMMARY HEATMAP OF LINEAGE TRANSITIONS
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("CREATING LINEAGE TRANSITION SUMMARY")
    print("="*60)
    
    # Create transition matrix
    lineage_transition_matrix = pd.crosstab(
        connections_df['dcm_lineage'],
        connections_df['rna_lineage'],
        rownames=['DCM Lineage'],
        colnames=['RNA Lineage']
    )
    
    # Reorder
    ordered_lineages = ['Stem', 'Absorptive', 'Secretory', 'ambiguous']
    lineage_transition_matrix = lineage_transition_matrix.reindex(
        index=[l for l in ordered_lineages if l in lineage_transition_matrix.index],
        columns=[l for l in ordered_lineages if l in lineage_transition_matrix.columns]
    )
    
    print("\nLineage Transition Matrix:")
    print(lineage_transition_matrix)
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(8, 6))
    
    sns.heatmap(lineage_transition_matrix, annot=True, fmt='d', cmap='viridis',
                cbar_kws={'label': 'Number of cells'}, ax=ax)
    
    ax.set_title('Lineage Transitions: DCM → RNA\n(Matched using Batch + Plate_Well)', 
                fontsize=14, fontweight='bold')
    ax.set_xlabel('RNA Lineage', fontsize=12)
    ax.set_ylabel('DCM Lineage', fontsize=12)
    
    plt.tight_layout()
    save_path = os.path.join(score_dir, 'lineage_transition_heatmap.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved: {save_path}")
    
    # Cell type transition matrix
    cell_type_transition_matrix = pd.crosstab(
        connections_df['dcm_cell_type'],
        connections_df['rna_cell_type'],
        rownames=['DCM Cell Type'],
        colnames=['RNA Cell Type']
    )
    
    # Reorder
    cell_type_transition_matrix = cell_type_transition_matrix.reindex(
        index=[l for l in label_order if l in cell_type_transition_matrix.index],
        columns=[l for l in label_order if l in cell_type_transition_matrix.columns]
    )
    
    print("\nCell Type Transition Matrix:")
    print(cell_type_transition_matrix)
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(10, 8))
    
    sns.heatmap(cell_type_transition_matrix, annot=True, fmt='d', cmap='viridis',
                cbar_kws={'label': 'Number of cells'}, ax=ax)
    
    ax.set_title('Cell Type Transitions: DCM → RNA\n(Matched using Batch + Plate_Well)', 
                fontsize=14, fontweight='bold')
    ax.set_xlabel('RNA Cell Type', fontsize=12)
    ax.set_ylabel('DCM Cell Type', fontsize=12)
    
    plt.tight_layout()
    save_path = os.path.join(score_dir, 'celltype_transition_heatmap.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved: {save_path}")

# -----------------------------
# SAVE CONNECTIONS WITH LINEAGE INFO
# -----------------------------
if len(connections_df) > 0:
    connections_df.to_csv(os.path.join(score_dir, 'DCM_RNA_connections_with_lineage_batch_plate_well.csv'), index=False)
    print(f"\nSaved connections to: {os.path.join(score_dir, 'DCM_RNA_connections_with_lineage_batch_plate_well.csv')}")

# -----------------------------
# ADD LINEAGE TO ADATA_COMBINED
# -----------------------------
print("\n" + "="*60)
print("ADDING LINEAGE COLUMNS TO ADATA_COMBINED")
print("="*60)

adata_combined.obs['cell_type_lineage'] = adata_combined.obs['leiden_annot_25'].astype(str).apply(map_to_lineage)
adata_combined.obs['module_lineage'] = adata_combined.obs['ttest_gmm_label_confident'].astype(str).apply(map_to_lineage)

print("\nLineage distribution in adata_combined:")
print("\ncell_type_lineage:")
print(adata_combined.obs['cell_type_lineage'].value_counts())
print("\nmodule_lineage:")
print(adata_combined.obs['module_lineage'].value_counts())

# -----------------------------
# SUMMARY
# -----------------------------
print("\n" + "="*60)
print("SUMMARY")
print("="*60)

if len(connections_df) > 0:
    connections_df['stable'] = connections_df['dcm_cell_type'] == connections_df['rna_cell_type']
    connections_df['lineage_stable'] = connections_df['dcm_lineage'] == connections_df['rna_lineage']
    
    n_stable = connections_df['stable'].sum()
    n_lineage_stable = connections_df['lineage_stable'].sum()
    
    print(f"""
Total connections: {len(connections_df)}

Cell Type Level:
  Stable (same type): {n_stable} ({n_stable/len(connections_df)*100:.1f}%)
  Changed (different type): {len(connections_df) - n_stable} ({(len(connections_df) - n_stable)/len(connections_df)*100:.1f}%)

Lineage Level:
  Stable (same lineage): {n_lineage_stable} ({n_lineage_stable/len(connections_df)*100:.1f}%)
  Changed (different lineage): {len(connections_df) - n_lineage_stable} ({(len(connections_df) - n_lineage_stable)/len(connections_df)*100:.1f}%)
""")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
## lineage umap transition with cell type coloring

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
import os

# -----------------------------
# SETTINGS
# -----------------------------
adata = adata_combined
score_dir = "/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/integration/plots_paper/"
# Small intestine cell types and colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': '#7f7f7f'
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']
lineage_order = ['Stem', 'Absorptive', 'Secretory']

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata.obs['batch_plate_well'] = (
    adata.obs['Batch'].astype(str) + '_' + 
    adata.obs['plate_well'].astype(str)
)

# -----------------------------
# FUNCTION TO MAP CELL TYPE TO LINEAGE
# -----------------------------
def map_to_lineage(cell_type):
    if cell_type in ['Stem', 'TA']:
        return 'Stem'
    elif cell_type in ['Enterocyte']:
        return 'Absorptive'
    elif cell_type in ['Paneth', 'Goblet', 'Tuft', 'EEC']:
        return 'Secretory'
    else:
        return 'ambiguous'

# -----------------------------
# EXTRACT UMAP AND METADATA
# -----------------------------
umap_coords = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1': umap_coords[:, 0],
    'UMAP2': umap_coords[:, 1],
    'assay': adata.obs['assay'].values,
    'plate_well': adata.obs['plate_well'].values,
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'Batch': adata.obs['Batch'].values,
    'cell_type_broad': adata.obs['leiden_annot_25'].astype(str).values,
    'module_max_label': adata.obs['ttest_gmm_label_confident'].astype(str).values,
    'cell_index': adata.obs.index.values
})

if 'Dox' in adata.obs.columns:
    plot_df['Dox'] = adata.obs['Dox'].values

# -----------------------------
# ADD LINEAGE COLUMNS
# -----------------------------
print("\n" + "="*60)
print("ADDING LINEAGE COLUMNS")
print("="*60)

plot_df['cell_type_lineage'] = plot_df['cell_type_broad'].apply(map_to_lineage)
plot_df['module_lineage'] = plot_df['module_max_label'].apply(map_to_lineage)

print("\nCell type broad -> Lineage distribution:")
print(plot_df.groupby(['cell_type_broad', 'cell_type_lineage']).size())

# -----------------------------
# FILTER DCM CELLS WITH KNOWN LABELS (EXCLUDE MINUS DOX)
# -----------------------------
print("\n" + "="*60)
print("STEP 1: FILTERING DCM CELLS")
print("="*60)

dcm_mask = plot_df['assay'] == 'pDCM'
dcm_known_mask = dcm_mask & plot_df['module_max_label'].isin(label_order)

# Exclude Minus Dox
if 'Dox' in plot_df.columns:
    dcm_known_mask = dcm_known_mask & (plot_df['Dox'] != 'Minus')

dcm_cells = plot_df[dcm_known_mask].copy()

print(f"Total scM_DCM cells: {dcm_mask.sum()}")
print(f"DCM cells with known labels (excluding Minus Dox): {dcm_known_mask.sum()}")
print("\nDCM cell type distribution:")
print(dcm_cells['module_max_label'].value_counts())
print("\nDCM lineage distribution:")
print(dcm_cells['module_lineage'].value_counts())

# -----------------------------
# FILTER RNA CELLS (EXCLUDE MINUS DOX)
# -----------------------------
print("\n" + "="*60)
print("STEP 2: FILTERING RNA CELLS")
print("="*60)

rna_mask = plot_df['assay'] == 'pRNA'

# Exclude Minus Dox
if 'Dox' in plot_df.columns:
    rna_mask = rna_mask & (plot_df['Dox'] != 'Minus')

rna_cells = plot_df[rna_mask].copy()
print(f"Total scM_RNA cells (excluding Minus Dox): {rna_mask.sum()}")
print("\nRNA cell type distribution:")
print(rna_cells['cell_type_broad'].value_counts())
print("\nRNA lineage distribution:")
print(rna_cells['cell_type_lineage'].value_counts())

# -----------------------------
# CHECK DUPLICATES WITH BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("STEP 3: CHECKING DUPLICATES")
print("="*60)

dcm_bpw_counts = dcm_cells['batch_plate_well'].value_counts()
rna_bpw_counts = rna_cells['batch_plate_well'].value_counts()

print(f"DCM batch_plate_wells with >1 cell: {(dcm_bpw_counts > 1).sum()}")
print(f"RNA batch_plate_wells with >1 cell: {(rna_bpw_counts > 1).sum()}")

# -----------------------------
# MATCH BY BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("STEP 4: FINDING CONNECTIONS USING BATCH_PLATE_WELL")
print("="*60)

dcm_bpw = set(dcm_cells['batch_plate_well'])
rna_bpw = set(rna_cells['batch_plate_well'])
common_bpw = dcm_bpw.intersection(rna_bpw)
print(f"Common batch_plate_wells: {len(common_bpw)}")

# Filter to common
dcm_common = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
rna_common = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]

# Find 1:1 matches
dcm_counts = dcm_common['batch_plate_well'].value_counts()
rna_counts = rna_common['batch_plate_well'].value_counts()

dcm_single = set(dcm_counts[dcm_counts == 1].index)
rna_single = set(rna_counts[rna_counts == 1].index)
both_single = dcm_single.intersection(rna_single)

print(f"1:1 matched batch_plate_wells: {len(both_single)}")

# Build connections
connections = []
for bpw in both_single:
    dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
    rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
    
    if len(dcm_cell) == 1 and len(rna_cell) == 1:
        dcm_row = dcm_cell.iloc[0]
        rna_row = rna_cell.iloc[0]
        
        connections.append({
            'batch_plate_well': bpw,
            'batch': dcm_row['Batch'],
            'plate_well': dcm_row['plate_well'],
            'dcm_umap1': dcm_row['UMAP1'],
            'dcm_umap2': dcm_row['UMAP2'],
            'rna_umap1': rna_row['UMAP1'],
            'rna_umap2': rna_row['UMAP2'],
            # Cell type level
            'dcm_cell_type': dcm_row['module_max_label'],
            'rna_cell_type': rna_row['cell_type_broad'],
            'transition': f"{dcm_row['module_max_label']} -> {rna_row['cell_type_broad']}",
            # Lineage level
            'dcm_lineage': dcm_row['module_lineage'],
            'rna_lineage': rna_row['cell_type_lineage'],
            'lineage_transition': f"{dcm_row['module_lineage']} -> {rna_row['cell_type_lineage']}"
        })

connections_df = pd.DataFrame(connections)
print(f"Total valid connections: {len(connections_df)}")

# -----------------------------
# PRINT TRANSITION SUMMARIES
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("CELL TYPE TRANSITIONS")
    print("="*60)
    print(connections_df['transition'].value_counts())
    
    print("\n" + "="*60)
    print("LINEAGE TRANSITIONS")
    print("="*60)
    print(connections_df['lineage_transition'].value_counts())

# -----------------------------
# PLOT CELL TYPE TRANSITIONS
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("PLOTTING CELL TYPE TRANSITIONS")
    print("="*60)
    
    unique_transitions = connections_df['transition'].unique()
    
    for transition in unique_transitions:
        parts = transition.split(' -> ')
        dcm_ct = parts[0]
        rna_ct = parts[1]
        trans_connections = connections_df[connections_df['transition'] == transition]
        trans_bpw = set(trans_connections['batch_plate_well'])
        
        n_cells = len(trans_connections)
        print(f"\nPlotting transition: {transition} ({n_cells} cells)")

        fig, ax = plt.subplots(figsize=(10, 8))

        # Reference background - CELL TYPE COLORS
        for ct in label_order:
            ref_cells = plot_df[(plot_df['assay'] == 'Ref') & (plot_df['cell_type_broad'] == ct)]
            if len(ref_cells) > 0:
                ax.scatter(ref_cells['UMAP1'], ref_cells['UMAP2'],
                           s=8, c=cell_type_colors[ct], alpha=0.3, marker='o', zorder=1, rasterized=True)

        # Ambiguous reference
        ref_ambig = plot_df[(plot_df['assay'] == 'Ref') & (~plot_df['cell_type_broad'].isin(label_order))]
        if len(ref_ambig) > 0:
            ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                       s=8, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)

        # Arrows - colored by RNA cell type
        arrow_color = cell_type_colors.get(rna_ct, '#7f7f7f')
        for _, conn in trans_connections.iterrows():
            arrow = FancyArrowPatch((conn['dcm_umap1'], conn['dcm_umap2']),
                                    (conn['rna_umap1'], conn['rna_umap2']),
                                    arrowstyle='-|>', mutation_scale=20,
                                    color=arrow_color, alpha=0.8, linewidth=1.5, zorder=3)
            ax.add_patch(arrow)

        # DCM cells - squares (using batch_plate_well)
        dcm_cells_plot = plot_df[(plot_df['assay'] == 'pDCM') &
                                 (plot_df['batch_plate_well'].isin(trans_bpw)) &
                                 (plot_df['module_max_label'] == dcm_ct)]
        if len(dcm_cells_plot) > 0:
            ax.scatter(dcm_cells_plot['UMAP1'], dcm_cells_plot['UMAP2'],
                       s=100, c=cell_type_colors[dcm_ct], alpha=0.9,
                       edgecolors='black', linewidths=1, marker='s', zorder=4, rasterized=True)

        # RNA cells - circles (using batch_plate_well)
        rna_cells_plot = plot_df[(plot_df['assay'] == 'pRNA') &
                                 (plot_df['batch_plate_well'].isin(trans_bpw)) &
                                 (plot_df['cell_type_broad'] == rna_ct)]
        if len(rna_cells_plot) > 0:
            ax.scatter(rna_cells_plot['UMAP1'], rna_cells_plot['UMAP2'],
                       s=100, c=cell_type_colors.get(rna_ct, '#7f7f7f'), alpha=0.9,
                       edgecolors='black', linewidths=1, marker='o', zorder=4, rasterized=True)

        # Legend
        legend_elements = [
            mlines.Line2D([], [], color=cell_type_colors[dcm_ct], marker='s', linestyle='None', markersize=12,
                          markeredgecolor='black', label=f'DCM ({dcm_ct})'),
            mlines.Line2D([], [], color=cell_type_colors.get(rna_ct, '#7f7f7f'), marker='o', linestyle='None', markersize=12,
                          markeredgecolor='black', label=f'RNA ({rna_ct})'),
            mlines.Line2D([], [], color=arrow_color, linestyle='-', linewidth=3, alpha=0.8,
                          marker='>', markersize=10, label='DCM → RNA')
        ]
        ax.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=10)

        ax.set_title(f'Transition: {transition}\n(n = {n_cells} cells, Matched using Batch + Plate_Well)', 
                    fontsize=14, fontweight='bold')
        ax.set_aspect('equal')
        ax.axis('off')

        plt.tight_layout()
        safe_name = transition.replace(' -> ', '_to_').replace(' ', '_')
        save_path = os.path.join(score_dir, f'umap_transition_{safe_name}.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f"Saved: {save_path}")

# -----------------------------
# PLOT LINEAGE TRANSITIONS
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("PLOTTING LINEAGE TRANSITIONS")
    print("="*60)
    
    unique_lineage_transitions = connections_df['lineage_transition'].unique()
    
    for lineage_transition in unique_lineage_transitions:
        parts = lineage_transition.split(' -> ')
        dcm_lineage = parts[0]
        rna_lineage = parts[1]
        
        # Skip ambiguous transitions
        if dcm_lineage == 'ambiguous' or rna_lineage == 'ambiguous':
            print(f"Skipping ambiguous transition: {lineage_transition}")
            continue
        
        trans_connections = connections_df[connections_df['lineage_transition'] == lineage_transition]
        trans_bpw = set(trans_connections['batch_plate_well'])
        
        n_cells = len(trans_connections)
        print(f"\nPlotting lineage transition: {lineage_transition} ({n_cells} cells)")

        fig, ax = plt.subplots(figsize=(10, 8))

        # Reference background - CELL TYPE COLORS (not lineage colors)
        for ct in label_order:
            ref_cells = plot_df[(plot_df['assay'] == 'Ref') & (plot_df['cell_type_broad'] == ct)]
            if len(ref_cells) > 0:
                ax.scatter(ref_cells['UMAP1'], ref_cells['UMAP2'],
                           s=8, c=cell_type_colors[ct],  # CELL TYPE COLOR
                           alpha=0.5, marker='o', zorder=1, rasterized=True)

        # Ambiguous reference
        ref_ambig = plot_df[(plot_df['assay'] == 'Ref') & (~plot_df['cell_type_broad'].isin(label_order))]
        if len(ref_ambig) > 0:
            ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                       s=8, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)

        # Arrows - colored by RNA cell type
        for _, conn in trans_connections.iterrows():
            rna_ct = conn['rna_cell_type']
            arrow_color = cell_type_colors.get(rna_ct, '#7f7f7f')
            arrow = FancyArrowPatch((conn['dcm_umap1'], conn['dcm_umap2']),
                                    (conn['rna_umap1'], conn['rna_umap2']),
                                    arrowstyle='-|>', mutation_scale=20,
                                    color=arrow_color, alpha=0.8, linewidth=2.5, zorder=3)
            ax.add_patch(arrow)

        # DCM cells - squares colored by CELL TYPE
        dcm_cells_plot = plot_df[(plot_df['assay'] == 'pDCM') &
                                 (plot_df['batch_plate_well'].isin(trans_bpw)) &
                                 (plot_df['module_lineage'] == dcm_lineage)]
        
        # Plot each cell type separately with its own color
        for ct in label_order:
            dcm_ct_cells = dcm_cells_plot[dcm_cells_plot['module_max_label'] == ct]
            if len(dcm_ct_cells) > 0:
                ax.scatter(dcm_ct_cells['UMAP1'], dcm_ct_cells['UMAP2'],
                           s=100, c=cell_type_colors[ct], alpha=0.9,
                           edgecolors='black', linewidths=1, marker='s', zorder=4, rasterized=True)

        # RNA cells - circles colored by CELL TYPE
        rna_cells_plot = plot_df[(plot_df['assay'] == 'pRNA') &
                                 (plot_df['batch_plate_well'].isin(trans_bpw)) &
                                 (plot_df['cell_type_lineage'] == rna_lineage)]
        
        # Plot each cell type separately with its own color
        for ct in label_order:
            rna_ct_cells = rna_cells_plot[rna_cells_plot['cell_type_broad'] == ct]
            if len(rna_ct_cells) > 0:
                ax.scatter(rna_ct_cells['UMAP1'], rna_ct_cells['UMAP2'],
                           s=100, c=cell_type_colors[ct], alpha=0.9,
                           edgecolors='black', linewidths=1, marker='o', zorder=4, rasterized=True)

        # Legend - show cell types that are actually present
        legend_elements = []
        
        # DCM cell types present in this lineage transition
        dcm_cell_types_present = trans_connections['dcm_cell_type'].unique()
        for ct in dcm_cell_types_present:
            count = (trans_connections['dcm_cell_type'] == ct).sum()
            legend_elements.append(
                mlines.Line2D([], [], color=cell_type_colors[ct], marker='s', linestyle='None', markersize=12,
                              markeredgecolor='black', label=f'DCM {ct} (n={count})')
            )
        
        # RNA cell types present in this lineage transition
        rna_cell_types_present = trans_connections['rna_cell_type'].unique()
        for ct in rna_cell_types_present:
            count = (trans_connections['rna_cell_type'] == ct).sum()
            legend_elements.append(
                mlines.Line2D([], [], color=cell_type_colors[ct], marker='o', linestyle='None', markersize=12,
                              markeredgecolor='black', label=f'RNA {ct} (n={count})')
            )
        
        # Add note about arrows in legend (since they vary by RNA cell type)
        legend_elements.append(
            mlines.Line2D([], [], color='black', linestyle='-', linewidth=3, alpha=0.8,
                          marker='>', markersize=10, label='DCM → RNA (colored by RNA cell type)')
        )
        
        ax.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=9)

        ax.set_title(f'Lineage Transition: {lineage_transition}\n(n = {n_cells} cells, Matched using Batch + Plate_Well)', 
                    fontsize=14, fontweight='bold')
        ax.set_aspect('equal')
        ax.axis('off')

        plt.tight_layout()
        safe_name = lineage_transition.replace(' -> ', '_to_').replace(' ', '_')
        save_path = os.path.join(score_dir, f'umap_lineage_transition_{safe_name}.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f"Saved: {save_path}")

# -----------------------------
# SUMMARY HEATMAP OF LINEAGE TRANSITIONS
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("CREATING LINEAGE TRANSITION SUMMARY")
    print("="*60)
    
    # Create transition matrix
    lineage_transition_matrix = pd.crosstab(
        connections_df['dcm_lineage'],
        connections_df['rna_lineage'],
        rownames=['DCM Lineage'],
        colnames=['RNA Lineage']
    )
    
    # Reorder
    ordered_lineages = ['Stem', 'Absorptive', 'Secretory', 'ambiguous']
    lineage_transition_matrix = lineage_transition_matrix.reindex(
        index=[l for l in ordered_lineages if l in lineage_transition_matrix.index],
        columns=[l for l in ordered_lineages if l in lineage_transition_matrix.columns]
    )
    
    print("\nLineage Transition Matrix:")
    print(lineage_transition_matrix)
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(8, 6))
    
    sns.heatmap(lineage_transition_matrix, annot=True, fmt='d', cmap='viridis',
                cbar_kws={'label': 'Number of cells'}, ax=ax)
    
    ax.set_title('Lineage Transitions: DCM → RNA\n(Matched using Batch + Plate_Well)', 
                fontsize=14, fontweight='bold')
    ax.set_xlabel('RNA Lineage', fontsize=12)
    ax.set_ylabel('DCM Lineage', fontsize=12)
    
    plt.tight_layout()
    save_path = os.path.join(score_dir, 'lineage_transition_heatmap.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved: {save_path}")
    
    # Cell type transition matrix
    cell_type_transition_matrix = pd.crosstab(
        connections_df['dcm_cell_type'],
        connections_df['rna_cell_type'],
        rownames=['DCM Cell Type'],
        colnames=['RNA Cell Type']
    )
    
    # Reorder
    cell_type_transition_matrix = cell_type_transition_matrix.reindex(
        index=[l for l in label_order if l in cell_type_transition_matrix.index],
        columns=[l for l in label_order if l in cell_type_transition_matrix.columns]
    )
    
    print("\nCell Type Transition Matrix:")
    print(cell_type_transition_matrix)
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(10, 8))
    
    sns.heatmap(cell_type_transition_matrix, annot=True, fmt='d', cmap='viridis',
                cbar_kws={'label': 'Number of cells'}, ax=ax)
    
    ax.set_title('Cell Type Transitions: DCM → RNA\n(Matched using Batch + Plate_Well)', 
                fontsize=14, fontweight='bold')
    ax.set_xlabel('RNA Cell Type', fontsize=12)
    ax.set_ylabel('DCM Cell Type', fontsize=12)
    
    plt.tight_layout()
    save_path = os.path.join(score_dir, 'celltype_transition_heatmap.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved: {save_path}")

# -----------------------------
# SAVE CONNECTIONS WITH LINEAGE INFO
# -----------------------------
if len(connections_df) > 0:
    connections_df.to_csv(os.path.join(score_dir, 'DCM_RNA_connections_with_lineage_batch_plate_well.csv'), index=False)
    print(f"\nSaved connections to: {os.path.join(score_dir, 'DCM_RNA_connections_with_lineage_batch_plate_well.csv')}")

# -----------------------------
# ADD LINEAGE TO ADATA_COMBINED
# -----------------------------
print("\n" + "="*60)
print("ADDING LINEAGE COLUMNS TO adata_combined")
print("="*60)

adata_combined.obs['cell_type_lineage'] = adata_combined.obs['leiden_annot_25'].astype(str).apply(map_to_lineage)
adata_combined.obs['module_lineage'] = adata_combined.obs['ttest_gmm_label_confident'].astype(str).apply(map_to_lineage)

print("\nLineage distribution in adata_combined:")
print("\ncell_type_lineage:")
print(adata_combined.obs['cell_type_lineage'].value_counts())
print("\nmodule_lineage:")
print(adata_combined.obs['module_lineage'].value_counts())

# -----------------------------
# SUMMARY
# -----------------------------
print("\n" + "="*60)
print("SUMMARY")
print("="*60)

if len(connections_df) > 0:
    connections_df['stable'] = connections_df['dcm_cell_type'] == connections_df['rna_cell_type']
    connections_df['lineage_stable'] = connections_df['dcm_lineage'] == connections_df['rna_lineage']
    
    n_stable = connections_df['stable'].sum()
    n_lineage_stable = connections_df['lineage_stable'].sum()
    
    print(f"""
Total connections: {len(connections_df)}

Cell Type Level:
  Stable (same type): {n_stable} ({n_stable/len(connections_df)*100:.1f}%)
  Changed (different type): {len(connections_df) - n_stable} ({(len(connections_df) - n_stable)/len(connections_df)*100:.1f}%)

Lineage Level:
  Stable (same lineage): {n_lineage_stable} ({n_lineage_stable/len(connections_df)*100:.1f}%)
  Changed (different lineage): {len(connections_df) - n_lineage_stable} ({(len(connections_df) - n_lineage_stable)/len(connections_df)*100:.1f}%)
""")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
##Heatmap based on Dox

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/gmm_085/'
transition_dir = os.path.join(score_dir, 'transition_umaps')
os.makedirs(transition_dir, exist_ok=True)

adata = adata_combined

# Small intestine cell types and colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': '#7f7f7f'
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata.obs['batch_plate_well'] = (
    adata.obs['Batch'].astype(str) + '_' + 
    adata.obs['plate_well'].astype(str)
)

# -----------------------------
# EXTRACT UMAP AND METADATA
# -----------------------------
print("\n" + "="*60)
print("PREPARING DATA")
print("="*60)

umap_coords = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1': umap_coords[:, 0],
    'UMAP2': umap_coords[:, 1],
    'assay': adata.obs['assay'].values,
    'plate_well': adata.obs['plate_well'].values,
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'Batch': adata.obs['Batch'].values,
    'cell_type_broad': adata.obs['leiden_annot_25'].astype(str).values,
    'module_max_label': adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'cell_index': adata.obs.index.values
})

if 'Dox' in adata.obs.columns:
    plot_df['Dox'] = adata.obs['Dox'].values

# -----------------------------
# FILTER DCM AND RNA CELLS (EXCLUDE MINUS DOX)
# -----------------------------
print("\n" + "="*60)
print("FILTERING CELLS")
print("="*60)

# DCM cells with known labels
dcm_mask = plot_df['assay'] == 'pDCM'
dcm_known_mask = dcm_mask & plot_df['module_max_label'].isin(label_order)

# Exclude Minus Dox
if 'Dox' in plot_df.columns:
    dcm_known_mask = dcm_known_mask & (plot_df['Dox'] != 'Minus')

dcm_cells = plot_df[dcm_known_mask].copy()

# RNA cells with known labels
rna_mask = plot_df['assay'] == 'pRNA'
rna_known_mask = rna_mask & plot_df['cell_type_broad'].isin(label_order)

# Exclude Minus Dox
if 'Dox' in plot_df.columns:
    rna_known_mask = rna_known_mask & (plot_df['Dox'] != 'Minus')

rna_cells = plot_df[rna_known_mask].copy()

print(f"DCM cells with known labels (excluding Minus Dox): {len(dcm_cells)}")
print(f"RNA cells with known labels (excluding Minus Dox): {len(rna_cells)}")

# -----------------------------
# CHECK DUPLICATES WITH BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("CHECKING DUPLICATES")
print("="*60)

dcm_bpw_counts = dcm_cells['batch_plate_well'].value_counts()
rna_bpw_counts = rna_cells['batch_plate_well'].value_counts()

print(f"DCM batch_plate_wells with >1 cell: {(dcm_bpw_counts > 1).sum()}")
print(f"RNA batch_plate_wells with >1 cell: {(rna_bpw_counts > 1).sum()}")

# -----------------------------
# MATCH BY BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("FINDING CONNECTIONS USING BATCH_PLATE_WELL")
print("="*60)

dcm_bpw = set(dcm_cells['batch_plate_well'])
rna_bpw = set(rna_cells['batch_plate_well'])
common_bpw = dcm_bpw.intersection(rna_bpw)
print(f"Common batch_plate_wells: {len(common_bpw)}")

# Filter to common
dcm_common = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
rna_common = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]

# Find 1:1 matches
dcm_counts = dcm_common['batch_plate_well'].value_counts()
rna_counts = rna_common['batch_plate_well'].value_counts()

dcm_single = set(dcm_counts[dcm_counts == 1].index)
rna_single = set(rna_counts[rna_counts == 1].index)
both_single = dcm_single.intersection(rna_single)

print(f"1:1 matched batch_plate_wells: {len(both_single)}")

# Build connections
connections = []
for bpw in both_single:
    dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
    rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
    
    if len(dcm_cell) == 1 and len(rna_cell) == 1:
        dcm_row = dcm_cell.iloc[0]
        rna_row = rna_cell.iloc[0]
        
        connections.append({
            'batch_plate_well': bpw,
            'batch': dcm_row['Batch'],
            'plate_well': dcm_row['plate_well'],
            'dcm_umap1': dcm_row['UMAP1'],
            'dcm_umap2': dcm_row['UMAP2'],
            'rna_umap1': rna_row['UMAP1'],
            'rna_umap2': rna_row['UMAP2'],
            'dcm_cell_type': dcm_row['module_max_label'],
            'rna_cell_type': rna_row['cell_type_broad'],
            'transition': f"{dcm_row['module_max_label']} -> {rna_row['cell_type_broad']}",
            'is_stable': dcm_row['module_max_label'] == rna_row['cell_type_broad']
        })

connections_df = pd.DataFrame(connections)
print(f"Total valid connections: {len(connections_df)}")

# -----------------------------
# FUNCTION TO PLOT TRANSITIONS FROM ONE DCM CELL TYPE
# -----------------------------
def plot_transitions_from_dcm_type(dcm_cell_type, connections_df, plot_df, 
                                    cell_type_colors, label_order, save_dir):
    """
    Plot all transitions from a specific DCM cell type to all RNA cell types
    """
    print(f"\n{'='*60}")
    print(f"PLOTTING TRANSITIONS FROM DCM {dcm_cell_type.upper()}")
    print(f"{'='*60}")
    
    # Filter connections from this DCM cell type
    dcm_connections = connections_df[connections_df['dcm_cell_type'] == dcm_cell_type].copy()
    
    if len(dcm_connections) == 0:
        print(f"  No connections found for DCM {dcm_cell_type}")
        return None
    
    print(f"  Total connections: {len(dcm_connections)}")
    print(f"  RNA destinations:")
    for rna_ct, count in dcm_connections['rna_cell_type'].value_counts().items():
        print(f"    -> {rna_ct}: {count}")
    
    # Get batch_plate_wells for this DCM type
    trans_bpw = set(dcm_connections['batch_plate_well'])
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Reference background - colored by cell type
    for ct in label_order:
        ref_cells = plot_df[(plot_df['assay'] == 'Ref') & (plot_df['cell_type_broad'] == ct)]
        if len(ref_cells) > 0:
            ax.scatter(ref_cells['UMAP1'], ref_cells['UMAP2'],
                       s=8, c=cell_type_colors[ct], alpha=0.08, marker='o', 
                       zorder=1, rasterized=True, label=None)
    
    # Ambiguous reference
    ref_ambig = plot_df[(plot_df['assay'] == 'Ref') & (~plot_df['cell_type_broad'].isin(label_order))]
    if len(ref_ambig) > 0:
        ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                   s=8, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)
    
    # Draw arrows with colors based on RNA destination
    for _, conn in dcm_connections.iterrows():
        rna_ct = conn['rna_cell_type']
        arrow_color = cell_type_colors.get(rna_ct, '#7f7f7f')
        
        arrow = FancyArrowPatch(
            (conn['dcm_umap1'], conn['dcm_umap2']),
            (conn['rna_umap1'], conn['rna_umap2']),
            arrowstyle='-|>', mutation_scale=15,
            color=arrow_color, alpha=0.7, linewidth=2, zorder=3
        )
        ax.add_patch(arrow)
    
    # DCM cells - squares (using batch_plate_well)
    dcm_cells_plot = plot_df[(plot_df['assay'] == 'pDCM') &
                             (plot_df['batch_plate_well'].isin(trans_bpw)) &
                             (plot_df['module_max_label'] == dcm_cell_type)]
    if len(dcm_cells_plot) > 0:
        ax.scatter(dcm_cells_plot['UMAP1'], dcm_cells_plot['UMAP2'],
                   s=120, c=cell_type_colors[dcm_cell_type], alpha=0.9,
                   edgecolors='black', linewidths=1.5, marker='s', zorder=4, rasterized=True)
    
    # RNA cells - circles (using batch_plate_well)
    for rna_ct in label_order:
        rna_ct_connections = dcm_connections[dcm_connections['rna_cell_type'] == rna_ct]
        if len(rna_ct_connections) == 0:
            continue
        
        rna_bpw = set(rna_ct_connections['batch_plate_well'])
        rna_cells_plot = plot_df[(plot_df['assay'] == 'pRNA') &
                                 (plot_df['batch_plate_well'].isin(rna_bpw)) &
                                 (plot_df['cell_type_broad'] == rna_ct)]
        
        if len(rna_cells_plot) > 0:
            ax.scatter(rna_cells_plot['UMAP1'], rna_cells_plot['UMAP2'],
                       s=120, c=cell_type_colors[rna_ct], alpha=0.9,
                       edgecolors='black', linewidths=1.5, marker='o', zorder=4, rasterized=True)
    
    # Legend
    legend_elements = []
    
    # DCM source
    legend_elements.append(
        mlines.Line2D([], [], color=cell_type_colors[dcm_cell_type], marker='s', 
                     linestyle='None', markersize=12, markeredgecolor='black',
                     label=f'DCM: {dcm_cell_type} (n={len(dcm_connections)})')
    )
    
    # RNA destinations
    for rna_ct in label_order:
        count = (dcm_connections['rna_cell_type'] == rna_ct).sum()
        if count > 0:
            legend_elements.append(
                mlines.Line2D([], [], color=cell_type_colors[rna_ct], marker='o', 
                             linestyle='None', markersize=10, markeredgecolor='black',
                             label=f'RNA: {rna_ct} (n={count})')
            )
    
    # Arrow legend
    legend_elements.append(
        mlines.Line2D([], [], color='gray', linestyle='-', linewidth=2, alpha=0.7,
                     marker='>', markersize=8, label='DCM → RNA')
    )
    
    ax.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=9,
             fancybox=True, framealpha=0.9)
    
    # Title with statistics
    n_stable = dcm_connections['is_stable'].sum()
    n_unstable = len(dcm_connections) - n_stable
    pct_stable = (n_stable / len(dcm_connections) * 100) if len(dcm_connections) > 0 else 0
    
    ax.set_title(f'Transitions from DCM {dcm_cell_type}\n'
                f'Total: {len(dcm_connections)} | Stable: {n_stable} ({pct_stable:.1f}%) | Unstable: {n_unstable}\n'
                f'(Matched using Batch + Plate_Well)', 
                fontsize=14, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')
    
    plt.tight_layout()
    
    save_path = os.path.join(save_dir, f'umap_from_DCM_{dcm_cell_type}_to_all.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  Saved: {save_path}")
    
    return dcm_connections

# -----------------------------
# FUNCTION TO PLOT TRANSITIONS TO ONE RNA CELL TYPE
# -----------------------------
def plot_transitions_to_rna_type(rna_cell_type, connections_df, plot_df, 
                                  cell_type_colors, label_order, save_dir):
    """
    Plot all transitions to a specific RNA cell type from all DCM cell types
    """
    print(f"\n{'='*60}")
    print(f"PLOTTING TRANSITIONS TO RNA {rna_cell_type.upper()}")
    print(f"{'='*60}")
    
    # Filter connections to this RNA cell type
    rna_connections = connections_df[connections_df['rna_cell_type'] == rna_cell_type].copy()
    
    if len(rna_connections) == 0:
        print(f"  No connections found for RNA {rna_cell_type}")
        return None
    
    print(f"  Total connections: {len(rna_connections)}")
    print(f"  DCM sources:")
    for dcm_ct, count in rna_connections['dcm_cell_type'].value_counts().items():
        print(f"    <- {dcm_ct}: {count}")
    
    # Get batch_plate_wells for this RNA type
    trans_bpw = set(rna_connections['batch_plate_well'])
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Reference background
    for ct in label_order:
        ref_cells = plot_df[(plot_df['assay'] == 'Ref') & (plot_df['cell_type_broad'] == ct)]
        if len(ref_cells) > 0:
            ax.scatter(ref_cells['UMAP1'], ref_cells['UMAP2'],
                       s=8, c=cell_type_colors[ct], alpha=0.08, marker='o', 
                       zorder=1, rasterized=True)
    
    ref_ambig = plot_df[(plot_df['assay'] == 'Ref') & (~plot_df['cell_type_broad'].isin(label_order))]
    if len(ref_ambig) > 0:
        ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                   s=8, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)
    
    # Draw arrows with colors based on DCM source
    for _, conn in rna_connections.iterrows():
        dcm_ct = conn['dcm_cell_type']
        arrow_color = cell_type_colors.get(dcm_ct, '#7f7f7f')
        
        arrow = FancyArrowPatch(
            (conn['dcm_umap1'], conn['dcm_umap2']),
            (conn['rna_umap1'], conn['rna_umap2']),
            arrowstyle='-|>', mutation_scale=15,
            color=arrow_color, alpha=0.7, linewidth=2, zorder=3
        )
        ax.add_patch(arrow)
    
    # DCM cells - squares (using batch_plate_well)
    for dcm_ct in label_order:
        dcm_ct_connections = rna_connections[rna_connections['dcm_cell_type'] == dcm_ct]
        if len(dcm_ct_connections) == 0:
            continue
        
        dcm_bpw = set(dcm_ct_connections['batch_plate_well'])
        dcm_cells_plot = plot_df[(plot_df['assay'] == 'pDCM') &
                                 (plot_df['batch_plate_well'].isin(dcm_bpw)) &
                                 (plot_df['module_max_label'] == dcm_ct)]
        
        if len(dcm_cells_plot) > 0:
            ax.scatter(dcm_cells_plot['UMAP1'], dcm_cells_plot['UMAP2'],
                       s=120, c=cell_type_colors[dcm_ct], alpha=0.9,
                       edgecolors='black', linewidths=1.5, marker='s', zorder=4, rasterized=True)
    
    # RNA cells - circles (using batch_plate_well)
    rna_cells_plot = plot_df[(plot_df['assay'] == 'pRNA') &
                             (plot_df['batch_plate_well'].isin(trans_bpw)) &
                             (plot_df['cell_type_broad'] == rna_cell_type)]
    if len(rna_cells_plot) > 0:
        ax.scatter(rna_cells_plot['UMAP1'], rna_cells_plot['UMAP2'],
                   s=120, c=cell_type_colors[rna_cell_type], alpha=0.9,
                   edgecolors='black', linewidths=1.5, marker='o', zorder=4, rasterized=True)
    
    # Legend
    legend_elements = []
    
    # DCM sources
    for dcm_ct in label_order:
        count = (rna_connections['dcm_cell_type'] == dcm_ct).sum()
        if count > 0:
            legend_elements.append(
                mlines.Line2D([], [], color=cell_type_colors[dcm_ct], marker='s', 
                             linestyle='None', markersize=10, markeredgecolor='black',
                             label=f'DCM: {dcm_ct} (n={count})')
            )
    
    # RNA destination
    legend_elements.append(
        mlines.Line2D([], [], color=cell_type_colors[rna_cell_type], marker='o', 
                     linestyle='None', markersize=12, markeredgecolor='black',
                     label=f'RNA: {rna_cell_type} (n={len(rna_connections)})')
    )
    
    # Arrow legend
    legend_elements.append(
        mlines.Line2D([], [], color='gray', linestyle='-', linewidth=2, alpha=0.7,
                     marker='>', markersize=8, label='DCM → RNA')
    )
    
    ax.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=9,
             fancybox=True, framealpha=0.9)
    
    # Title with statistics
    n_stable = rna_connections['is_stable'].sum()
    n_unstable = len(rna_connections) - n_stable
    pct_stable = (n_stable / len(rna_connections) * 100) if len(rna_connections) > 0 else 0
    
    ax.set_title(f'Transitions to RNA {rna_cell_type}\n'
                f'Total: {len(rna_connections)} | Stable: {n_stable} ({pct_stable:.1f}%) | Unstable: {n_unstable}\n'
                f'(Matched using Batch + Plate_Well)', 
                fontsize=14, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')
    
    plt.tight_layout()
    
    save_path = os.path.join(save_dir, f'umap_to_RNA_{rna_cell_type}_from_all.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  Saved: {save_path}")
    
    return rna_connections

# -----------------------------
# CREATE UMAPS FOR EACH DCM CELL TYPE
# -----------------------------
print("\n" + "="*60)
print("CREATING UMAPS: FROM EACH DCM CELL TYPE TO ALL RNA")
print("="*60)

dcm_summary = {}

for dcm_ct in label_order:
    if dcm_ct in connections_df['dcm_cell_type'].unique():
        result = plot_transitions_from_dcm_type(
            dcm_ct, connections_df, plot_df, 
            cell_type_colors, label_order, transition_dir
        )
        if result is not None:
            dcm_summary[dcm_ct] = result['rna_cell_type'].value_counts().to_dict()

# -----------------------------
# CREATE UMAPS FOR EACH RNA CELL TYPE
# -----------------------------
print("\n" + "="*60)
print("CREATING UMAPS: TO EACH RNA CELL TYPE FROM ALL DCM")
print("="*60)

rna_summary = {}

for rna_ct in label_order:
    if rna_ct in connections_df['rna_cell_type'].unique():
        result = plot_transitions_to_rna_type(
            rna_ct, connections_df, plot_df, 
            cell_type_colors, label_order, transition_dir
        )
        if result is not None:
            rna_summary[rna_ct] = result['dcm_cell_type'].value_counts().to_dict()

# -----------------------------
# SUMMARY TABLE
# -----------------------------
print("\n" + "="*60)
print("TRANSITION SUMMARY")
print("="*60)

print("\n--- From DCM to RNA ---")
for dcm_ct, destinations in dcm_summary.items():
    total = sum(destinations.values())
    print(f"\n{dcm_ct} (n={total}):")
    for rna_ct, count in sorted(destinations.items(), key=lambda x: -x[1]):
        pct = (count / total * 100) if total > 0 else 0
        stable = "STABLE" if dcm_ct == rna_ct else ""
        print(f"  -> {rna_ct}: {count} ({pct:.1f}%) {stable}")

print("\n--- To RNA from DCM ---")
for rna_ct, sources in rna_summary.items():
    total = sum(sources.values())
    print(f"\n{rna_ct} (n={total}):")
    for dcm_ct, count in sorted(sources.items(), key=lambda x: -x[1]):
        pct = (count / total * 100) if total > 0 else 0
        stable = "STABLE" if dcm_ct == rna_ct else ""
        print(f"  <- {dcm_ct}: {count} ({pct:.1f}%) {stable}")

# -----------------------------
# CREATE COMBINED SUMMARY PLOT
# -----------------------------
print("\n" + "="*60)
print("CREATING SUMMARY HEATMAPS")
print("="*60)

# Create transition matrix
transition_matrix = pd.crosstab(
    connections_df['dcm_cell_type'],
    connections_df['rna_cell_type'],
    rownames=['DCM Cell Type'],
    colnames=['RNA Cell Type']
)

# Reorder
transition_matrix = transition_matrix.reindex(
    index=[l for l in label_order if l in transition_matrix.index],
    columns=[l for l in label_order if l in transition_matrix.columns]
)

# Plot counts heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Counts
ax1 = axes[0]
sns.heatmap(transition_matrix, annot=True, fmt='d', cmap='viridis',
            cbar_kws={'label': 'Number of cells'}, ax=ax1)
ax1.set_title('Cell Type Transitions: DCM → RNA (Counts)\n(Matched using Batch + Plate_Well)', 
             fontsize=12, fontweight='bold')
ax1.set_xlabel('RNA Cell Type', fontsize=11)
ax1.set_ylabel('DCM Cell Type', fontsize=11)

# Percentages (row-normalized)
ax2 = axes[1]
transition_pct = transition_matrix.div(transition_matrix.sum(axis=1), axis=0) * 100
sns.heatmap(transition_pct, annot=True, fmt='.1f', cmap='viridis',
            cbar_kws={'label': 'Percentage (%)'}, ax=ax2)
ax2.set_title('Cell Type Transitions: DCM → RNA (% of DCM type)\n(Matched using Batch + Plate_Well)', 
             fontsize=12, fontweight='bold')
ax2.set_xlabel('RNA Cell Type', fontsize=11)
ax2.set_ylabel('DCM Cell Type', fontsize=11)

plt.tight_layout()
save_path = os.path.join(transition_dir, 'transition_summary_heatmaps.pdf')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print(f"Saved: {save_path}")

# -----------------------------
# SAVE SUMMARY DATA
# -----------------------------
connections_df.to_csv(os.path.join(transition_dir, 'all_transitions_batch_plate_well.csv'), index=False)
transition_matrix.to_csv(os.path.join(transition_dir, 'transition_matrix_counts.csv'))
transition_pct.to_csv(os.path.join(transition_dir, 'transition_matrix_percentages.csv'))

print(f"\nSaved:")
print(f"  - {os.path.join(transition_dir, 'all_transitions_batch_plate_well.csv')}")
print(f"  - {os.path.join(transition_dir, 'transition_matrix_counts.csv')}")
print(f"  - {os.path.join(transition_dir, 'transition_matrix_percentages.csv')}")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
```

## Summary of outputs:

### 1. UMAPs from each DCM cell type
`umap_from_DCM_{CellType}_to_all.pdf`

For each DCM cell type (Stem, TA, Enterocyte, etc.), shows:
- All transitions FROM that DCM type to ANY RNA type
- DCM cells as squares (same color)
- RNA cells as circles (colored by their type)
- Arrows colored by RNA destination
- Legend with counts per destination

### 2. UMAPs to each RNA cell type
`umap_to_RNA_{CellType}_from_all.pdf`

For each RNA cell type, shows:
- All transitions TO that RNA type from ANY DCM type
- DCM cells as squares (colored by their type)
- RNA cells as circles (same color)
- Arrows colored by DCM source
- Legend with counts per source

### 3. Summary heatmaps
`transition_summary_heatmaps.pdf`
- Left: Count matrix
- Right: Percentage matrix (row-normalized, % of each DCM type)

### 4. Output files
| File | Description |
|------|-------------|
| `all_transitions.csv` | All individual transitions |
| `transition_matrix_counts.csv` | Count matrix |
| `transition_matrix_percentages.csv` | Percentage matrix |

### Visual example:
```
From DCM Tuft to all RNA:
┌────────────────────────────────────┐
│  [Reference cells in background]   │
│                                    │
│    ■ Tuft (DCM)                   │
│     ↘ → ● Tuft (stable)          │
│     ↘ → ● EEC                     │
│     ↘ → ● Goblet                  │
│                                    │
│  Legend:                           │
│  ■ DCM: Tuft (n=X)                │
│  ● RNA: Tuft (n=Y)                │
│  ● RNA: EEC (n=Z)                 │
│  ...                               │

In [ ]:
##heatmap per dox

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/gmm_085/'
heatmap_dir = os.path.join(score_dir, 'transition_heatmaps_by_dox')
os.makedirs(heatmap_dir, exist_ok=True)

adata = adata_combined

# Cell types and colors
label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata.obs['batch_plate_well'] = (
    adata.obs['Batch'].astype(str) + '_' + 
    adata.obs['plate_well'].astype(str)
)

# -----------------------------
# PREPARE DATA
# -----------------------------
print("\n" + "="*60)
print("PREPARING DATA")
print("="*60)

plot_df = pd.DataFrame({
    'assay': adata.obs['assay'].values,
    'plate_well': adata.obs['plate_well'].values,
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'Batch': adata.obs['Batch'].values,
    'cell_type_broad': adata.obs['leiden_annot_25'].astype(str).values,  # BEFORE GMM
    'module_max_label': adata.obs['ttest_gmm_label_confident_085'].astype(str).values,  # AFTER GMM
    'Dox': adata.obs['Dox'].astype(str).values if 'Dox' in adata.obs.columns else 'Unknown'
})

# Get unique Dox conditions (excluding Minus)
dox_conditions = [d for d in plot_df['Dox'].unique() if d != 'Minus' and d != 'nan' and d != 'Unknown']
print(f"Dox conditions (excluding Minus): {dox_conditions}")

# -----------------------------
# HELPER FUNCTION FOR 1:1 MATCHING
# -----------------------------
def get_matched_connections(dcm_cells, rna_cells, dcm_label_col, rna_label_col):
    """
    Find 1:1 matched cells using batch_plate_well
    Returns DataFrame with connections
    """
    # Find common batch_plate_wells
    dcm_bpw = set(dcm_cells['batch_plate_well'].unique())
    rna_bpw = set(rna_cells['batch_plate_well'].unique())
    common_bpw = dcm_bpw.intersection(rna_bpw)
    
    # Filter to common
    dcm_common = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
    rna_common = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]
    
    # Find 1:1 matches
    dcm_counts = dcm_common['batch_plate_well'].value_counts()
    rna_counts = rna_common['batch_plate_well'].value_counts()
    
    dcm_single = set(dcm_counts[dcm_counts == 1].index)
    rna_single = set(rna_counts[rna_counts == 1].index)
    both_single = dcm_single.intersection(rna_single)
    
    # Build connections
    connections = []
    for bpw in both_single:
        dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
        rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
        
        if len(dcm_cell) == 1 and len(rna_cell) == 1:
            connections.append({
                'batch_plate_well': bpw,
                'batch': dcm_cell.iloc[0]['Batch'],
                'plate_well': dcm_cell.iloc[0]['plate_well'],
                'dcm_label': dcm_cell.iloc[0][dcm_label_col],
                'rna_label': rna_cell.iloc[0][rna_label_col]
            })
    
    return pd.DataFrame(connections), len(common_bpw), len(both_single)

################################################################################
# HEATMAP 1: AFTER GMM - DCM (module_max_label) vs RNA (cell_type_broad)
################################################################################

print("\n" + "="*60)
print("HEATMAP SET 1: AFTER GMM")
print("DCM (ttest_gmm_label_confident) vs RNA (leiden_annot_25)")
print("="*60)

after_gmm_summary = {}

for dox in dox_conditions:
    print(f"\n--- Processing Dox = {dox} ---")
    
    # Filter DCM cells: known labels only, specific Dox
    dcm_mask = (plot_df['assay'] == 'pDCM') & \
               (plot_df['module_max_label'].isin(label_order)) & \
               (plot_df['Dox'] == dox)
    dcm_cells = plot_df[dcm_mask].copy()
    
    # Filter RNA cells: known labels only, specific Dox
    rna_mask = (plot_df['assay'] == 'pRNA') & \
               (plot_df['cell_type_broad'].isin(label_order)) & \
               (plot_df['Dox'] == dox)
    rna_cells = plot_df[rna_mask].copy()
    
    print(f"  DCM cells (known GMM labels): {len(dcm_cells)}")
    print(f"  RNA cells (known labels): {len(rna_cells)}")
    
    if len(dcm_cells) == 0 or len(rna_cells) == 0:
        print(f"  Skipping Dox = {dox} - no data")
        continue
    
    # Get matched connections using batch_plate_well
    connections_df, n_common, n_matched = get_matched_connections(
        dcm_cells, rna_cells, 
        dcm_label_col='module_max_label', 
        rna_label_col='cell_type_broad'
    )
    
    print(f"  Common batch_plate_wells: {n_common}")
    print(f"  1:1 matched batch_plate_wells: {n_matched}")
    print(f"  Total connections: {len(connections_df)}")
    
    if len(connections_df) == 0:
        print(f"  No connections found for Dox = {dox}")
        continue
    
    # Store for summary
    after_gmm_summary[dox] = connections_df
    
    # Save connections
    connections_df.to_csv(os.path.join(heatmap_dir, f'connections_AFTER_GMM_Dox_{dox}.csv'), index=False)
    
    # Create crosstab
    crosstab = pd.crosstab(
        connections_df['dcm_label'], 
        connections_df['rna_label'],
        margins=True
    )
    
    # Reorder to match label_order
    row_order = [l for l in label_order if l in crosstab.index] + ['All']
    col_order = [l for l in label_order if l in crosstab.columns] + ['All']
    crosstab = crosstab.reindex(index=row_order, columns=col_order, fill_value=0)
    
    print(f"\n  Crosstab (counts):")
    print(crosstab)
    
    # Create heatmap (without margins for visualization)
    crosstab_no_margins = crosstab.drop('All', axis=0).drop('All', axis=1)
    
    # Calculate percentages (row-normalized)
    crosstab_pct = crosstab_no_margins.div(crosstab_no_margins.sum(axis=1), axis=0) * 100
    crosstab_pct = crosstab_pct.fillna(0)
    
    # --- Plot 1: Counts heatmap ---
    fig, ax = plt.subplots(figsize=(10, 8))
    
    sns.heatmap(
        crosstab_no_margins,
        annot=True,
        fmt='d',
        cmap='RdBu_r',
        cbar_kws={'label': 'Cell count'},
        ax=ax,
        linewidths=0.5,
        linecolor='white'
    )
    
    ax.set_xlabel('RNA Cell Type (leiden_annot_25)', fontsize=12)
    ax.set_ylabel('DCM Cell Type (GMM label)', fontsize=12)
    ax.set_title(f'DCM → RNA Transition Matrix - AFTER GMM\nDox = {dox} (n = {len(connections_df)} paired cells)\n(Matched using Batch + Plate_Well)', 
                 fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    
    save_path = os.path.join(heatmap_dir, f'heatmap_AFTER_GMM_counts_Dox_{dox}.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  Saved: {save_path}")
    
    # --- Plot 2: Percentages heatmap ---
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create annotation with both percentage and count
    annot_matrix = crosstab_pct.copy().astype(str)
    for i in range(len(crosstab_pct.index)):
        for j in range(len(crosstab_pct.columns)):
            pct = crosstab_pct.iloc[i, j]
            count = crosstab_no_margins.iloc[i, j]
            annot_matrix.iloc[i, j] = f'{pct:.1f}%\n({count})'
    
    sns.heatmap(
        crosstab_pct,
        annot=annot_matrix,
        fmt='',
        cmap='RdBu_r',
        cbar_kws={'label': 'Percentage (%)'},
        ax=ax,
        linewidths=0.5,
        linecolor='white',
        vmin=0,
        vmax=100
    )
    
    ax.set_xlabel('RNA Cell Type (leiden_annot_25)', fontsize=12)
    ax.set_ylabel('DCM Cell Type (GMM label)', fontsize=12)
    ax.set_title(f'DCM → RNA Transition Matrix - AFTER GMM (% of DCM type)\nDox = {dox} (n = {len(connections_df)} paired cells)\n(Matched using Batch + Plate_Well)', 
                 fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    
    save_path = os.path.join(heatmap_dir, f'heatmap_AFTER_GMM_percentages_Dox_{dox}.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  Saved: {save_path}")
    
    # Save crosstabs
    crosstab.to_csv(os.path.join(heatmap_dir, f'crosstab_AFTER_GMM_counts_Dox_{dox}.csv'))
    crosstab_pct.to_csv(os.path.join(heatmap_dir, f'crosstab_AFTER_GMM_percentages_Dox_{dox}.csv'))

################################################################################
# HEATMAP 2: BEFORE GMM - DCM (cell_type_broad) vs RNA (cell_type_broad)
################################################################################

print("\n" + "="*60)
print("HEATMAP SET 2: BEFORE GMM")
print("DCM (leiden_annot_25) vs RNA (leiden_annot_25)")
print("="*60)

before_gmm_summary = {}

for dox in dox_conditions:
    print(f"\n--- Processing Dox = {dox} ---")
    
    # Filter DCM cells: use cell_type_broad (leiden_annot_25), specific Dox
    dcm_mask = (plot_df['assay'] == 'pDCM') & \
               (plot_df['cell_type_broad'].isin(label_order)) & \
               (plot_df['Dox'] == dox)
    dcm_cells = plot_df[dcm_mask].copy()
    
    # Filter RNA cells: use cell_type_broad, specific Dox
    rna_mask = (plot_df['assay'] == 'pRNA') & \
               (plot_df['cell_type_broad'].isin(label_order)) & \
               (plot_df['Dox'] == dox)
    rna_cells = plot_df[rna_mask].copy()
    
    print(f"  DCM cells (known leiden labels): {len(dcm_cells)}")
    print(f"  RNA cells (known leiden labels): {len(rna_cells)}")
    
    if len(dcm_cells) == 0 or len(rna_cells) == 0:
        print(f"  Skipping Dox = {dox} - no data")
        continue
    
    # Get matched connections using batch_plate_well
    connections_df, n_common, n_matched = get_matched_connections(
        dcm_cells, rna_cells, 
        dcm_label_col='cell_type_broad', 
        rna_label_col='cell_type_broad'
    )
    
    print(f"  Common batch_plate_wells: {n_common}")
    print(f"  1:1 matched batch_plate_wells: {n_matched}")
    print(f"  Total connections: {len(connections_df)}")
    
    if len(connections_df) == 0:
        print(f"  No connections found for Dox = {dox}")
        continue
    
    # Store for summary
    before_gmm_summary[dox] = connections_df
    
    # Save connections
    connections_df.to_csv(os.path.join(heatmap_dir, f'connections_BEFORE_GMM_Dox_{dox}.csv'), index=False)
    
    # Create crosstab
    crosstab = pd.crosstab(
        connections_df['dcm_label'], 
        connections_df['rna_label'],
        margins=True
    )
    
    # Reorder to match label_order
    row_order = [l for l in label_order if l in crosstab.index] + ['All']
    col_order = [l for l in label_order if l in crosstab.columns] + ['All']
    crosstab = crosstab.reindex(index=row_order, columns=col_order, fill_value=0)
    
    print(f"\n  Crosstab (counts):")
    print(crosstab)
    
    # Create heatmap (without margins for visualization)
    crosstab_no_margins = crosstab.drop('All', axis=0).drop('All', axis=1)
    
    # Calculate percentages (row-normalized)
    crosstab_pct = crosstab_no_margins.div(crosstab_no_margins.sum(axis=1), axis=0) * 100
    crosstab_pct = crosstab_pct.fillna(0)
    
    # --- Plot 1: Counts heatmap ---
    fig, ax = plt.subplots(figsize=(10, 8))
    
    sns.heatmap(
        crosstab_no_margins,
        annot=True,
        fmt='d',
        cmap='RdBu_r',
        cbar_kws={'label': 'Cell count'},
        ax=ax,
        linewidths=0.5,
        linecolor='white'
    )
    
    ax.set_xlabel('RNA Cell Type (leiden_annot_25)', fontsize=12)
    ax.set_ylabel('DCM Cell Type (leiden_annot_25)', fontsize=12)
    ax.set_title(f'DCM → RNA Transition Matrix - BEFORE GMM\nDox = {dox} (n = {len(connections_df)} paired cells)\n(Matched using Batch + Plate_Well)', 
                 fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    
    save_path = os.path.join(heatmap_dir, f'heatmap_BEFORE_GMM_counts_Dox_{dox}.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  Saved: {save_path}")
    
    # --- Plot 2: Percentages heatmap ---
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create annotation with both percentage and count
    annot_matrix = crosstab_pct.copy().astype(str)
    for i in range(len(crosstab_pct.index)):
        for j in range(len(crosstab_pct.columns)):
            pct = crosstab_pct.iloc[i, j]
            count = crosstab_no_margins.iloc[i, j]
            annot_matrix.iloc[i, j] = f'{pct:.1f}%\n({count})'
    
    sns.heatmap(
        crosstab_pct,
        annot=annot_matrix,
        fmt='',
        cmap='RdBu_r',
        cbar_kws={'label': 'Percentage (%)'},
        ax=ax,
        linewidths=0.5,
        linecolor='white',
        vmin=0,
        vmax=100
    )
    
    ax.set_xlabel('RNA Cell Type (leiden_annot_25)', fontsize=12)
    ax.set_ylabel('DCM Cell Type (leiden_annot_25)', fontsize=12)
    ax.set_title(f'DCM → RNA Transition Matrix - BEFORE GMM (% of DCM type)\nDox = {dox} (n = {len(connections_df)} paired cells)\n(Matched using Batch + Plate_Well)', 
                 fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    
    save_path = os.path.join(heatmap_dir, f'heatmap_BEFORE_GMM_percentages_Dox_{dox}.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  Saved: {save_path}")
    
    # Save crosstabs
    crosstab.to_csv(os.path.join(heatmap_dir, f'crosstab_BEFORE_GMM_counts_Dox_{dox}.csv'))
    crosstab_pct.to_csv(os.path.join(heatmap_dir, f'crosstab_BEFORE_GMM_percentages_Dox_{dox}.csv'))

################################################################################
# SIDE-BY-SIDE COMPARISON: BEFORE vs AFTER GMM per Dox
################################################################################

print("\n" + "="*60)
print("SIDE-BY-SIDE COMPARISON: BEFORE vs AFTER GMM")
print("="*60)

for dox in dox_conditions:
    if dox not in before_gmm_summary or dox not in after_gmm_summary:
        print(f"Skipping Dox = {dox} - missing data")
        continue
    
    print(f"\n--- Creating comparison for Dox = {dox} ---")
    
    before_df = before_gmm_summary[dox]
    after_df = after_gmm_summary[dox]
    
    # Create crosstabs
    crosstab_before = pd.crosstab(before_df['dcm_label'], before_df['rna_label'])
    crosstab_after = pd.crosstab(after_df['dcm_label'], after_df['rna_label'])
    
    # Reorder
    crosstab_before = crosstab_before.reindex(
        index=[l for l in label_order if l in crosstab_before.index],
        columns=[l for l in label_order if l in crosstab_before.columns],
        fill_value=0
    )
    crosstab_after = crosstab_after.reindex(
        index=[l for l in label_order if l in crosstab_after.index],
        columns=[l for l in label_order if l in crosstab_after.columns],
        fill_value=0
    )
    
    # Calculate percentages
    crosstab_before_pct = crosstab_before.div(crosstab_before.sum(axis=1), axis=0) * 100
    crosstab_before_pct = crosstab_before_pct.fillna(0)
    
    crosstab_after_pct = crosstab_after.div(crosstab_after.sum(axis=1), axis=0) * 100
    crosstab_after_pct = crosstab_after_pct.fillna(0)
    
    # Create side-by-side figure
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    
    # BEFORE GMM
    ax1 = axes[0]
    sns.heatmap(
        crosstab_before_pct,
        annot=True,
        fmt='.1f',
        cmap='RdBu_r',
        cbar_kws={'label': 'Percentage (%)'},
        ax=ax1,
        linewidths=0.5,
        linecolor='white',
        vmin=0,
        vmax=100
    )
    ax1.set_xlabel('RNA Cell Type', fontsize=11)
    ax1.set_ylabel('DCM Cell Type', fontsize=11)
    ax1.set_title(f'BEFORE GMM (leiden_annot_25)\nn = {len(before_df)} cells', fontsize=12, fontweight='bold')
    
    # AFTER GMM
    ax2 = axes[1]
    sns.heatmap(
        crosstab_after_pct,
        annot=True,
        fmt='.1f',
        cmap='RdBu_r',
        cbar_kws={'label': 'Percentage (%)'},
        ax=ax2,
        linewidths=0.5,
        linecolor='white',
        vmin=0,
        vmax=100
    )
    ax2.set_xlabel('RNA Cell Type', fontsize=11)
    ax2.set_ylabel('DCM Cell Type (GMM)', fontsize=11)
    ax2.set_title(f'AFTER GMM (ttest_gmm_label_confident)\nn = {len(after_df)} cells', fontsize=12, fontweight='bold')
    
    # Calculate accuracy (diagonal sum)
    accuracy_before = np.diag(crosstab_before).sum() / crosstab_before.values.sum() * 100 if crosstab_before.values.sum() > 0 else 0
    accuracy_after = np.diag(crosstab_after).sum() / crosstab_after.values.sum() * 100 if crosstab_after.values.sum() > 0 else 0
    
    fig.suptitle(f'DCM → RNA Transition Comparison - Dox = {dox}\n'
                f'Diagonal Accuracy: BEFORE = {accuracy_before:.1f}% | AFTER = {accuracy_after:.1f}%\n'
                f'(Matched using Batch + Plate_Well)', 
                fontsize=14, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    
    save_path = os.path.join(heatmap_dir, f'comparison_BEFORE_vs_AFTER_GMM_Dox_{dox}.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  Saved: {save_path}")

################################################################################
# SUMMARY STATISTICS TABLE
################################################################################

print("\n" + "="*60)
print("SUMMARY STATISTICS: BEFORE vs AFTER GMM by Dox")
print("="*60)

summary_data = []

for dox in dox_conditions:
    row = {'Dox': dox}
    
    # BEFORE GMM
    if dox in before_gmm_summary:
        before_df = before_gmm_summary[dox]
        n_before = len(before_df)
        n_stable_before = (before_df['dcm_label'] == before_df['rna_label']).sum()
        accuracy_before = (n_stable_before / n_before * 100) if n_before > 0 else 0
        
        row['n_cells_BEFORE'] = n_before
        row['n_stable_BEFORE'] = n_stable_before
        row['accuracy_BEFORE'] = accuracy_before
    else:
        row['n_cells_BEFORE'] = 0
        row['n_stable_BEFORE'] = 0
        row['accuracy_BEFORE'] = np.nan
    
    # AFTER GMM
    if dox in after_gmm_summary:
        after_df = after_gmm_summary[dox]
        n_after = len(after_df)
        n_stable_after = (after_df['dcm_label'] == after_df['rna_label']).sum()
        accuracy_after = (n_stable_after / n_after * 100) if n_after > 0 else 0
        
        row['n_cells_AFTER'] = n_after
        row['n_stable_AFTER'] = n_stable_after
        row['accuracy_AFTER'] = accuracy_after
    else:
        row['n_cells_AFTER'] = 0
        row['n_stable_AFTER'] = 0
        row['accuracy_AFTER'] = np.nan
    
    summary_data.append(row)
    
    print(f"\nDox = {dox}:")
    print(f"  BEFORE GMM: {row['n_stable_BEFORE']}/{row['n_cells_BEFORE']} stable ({row['accuracy_BEFORE']:.1f}%)")
    print(f"  AFTER GMM:  {row['n_stable_AFTER']}/{row['n_cells_AFTER']} stable ({row['accuracy_AFTER']:.1f}%)")

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(os.path.join(heatmap_dir, 'summary_BEFORE_vs_AFTER_GMM_by_Dox.csv'), index=False)
print(f"\nSaved summary: {os.path.join(heatmap_dir, 'summary_BEFORE_vs_AFTER_GMM_by_Dox.csv')}")

################################################################################
# BARPLOT: ACCURACY COMPARISON BEFORE vs AFTER GMM
################################################################################

print("\n" + "="*60)
print("CREATING ACCURACY COMPARISON BARPLOT")
print("="*60)

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(summary_df))
width = 0.35

bars_before = ax.bar(x - width/2, summary_df['accuracy_BEFORE'], width, 
                     label='BEFORE GMM (leiden)', color='#7f7f7f', edgecolor='black')
bars_after = ax.bar(x + width/2, summary_df['accuracy_AFTER'], width, 
                    label='AFTER GMM (ttest_gmm)', color='#2E8B57', edgecolor='black')

# Add value labels
for bar, n in zip(bars_before, summary_df['n_cells_BEFORE']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}%\n(n={n})', ha='center', va='bottom', fontsize=8)

for bar, n in zip(bars_after, summary_df['n_cells_AFTER']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}%\n(n={n})', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Dox Condition', fontsize=12, fontweight='bold')
ax.set_ylabel('Diagonal Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('DCM → RNA Transition Accuracy: BEFORE vs AFTER GMM\n(% of cells where DCM label = RNA label)\n(Matched using Batch + Plate_Well)', 
            fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(summary_df['Dox'], fontsize=11)
ax.set_ylim(0, 110)
ax.legend(loc='upper right', fontsize=10)
ax.axhline(y=50, color='gray', linestyle='--', linewidth=1, alpha=0.5)

plt.tight_layout()

save_path = os.path.join(heatmap_dir, 'barplot_accuracy_BEFORE_vs_AFTER_GMM.pdf')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print(f"Saved: {save_path}")

################################################################################
# DETAILED TRANSITION BREAKDOWN PER DOX
################################################################################

print("\n" + "="*60)
print("DETAILED TRANSITION BREAKDOWN PER DOX")
print("="*60)

for dox in dox_conditions:
    if dox not in after_gmm_summary:
        continue
    
    print(f"\n{'='*40}")
    print(f"Dox = {dox}")
    print(f"{'='*40}")
    
    after_df = after_gmm_summary[dox]
    
    for dcm_ct in label_order:
        dcm_subset = after_df[after_df['dcm_label'] == dcm_ct]
        if len(dcm_subset) == 0:
            continue
        
        print(f"\n  DCM {dcm_ct} (n={len(dcm_subset)}):")
        for rna_ct, count in dcm_subset['rna_label'].value_counts().items():
            pct = (count / len(dcm_subset) * 100)
            stable = "★ STABLE" if dcm_ct == rna_ct else ""
            print(f"    -> {rna_ct}: {count} ({pct:.1f}%) {stable}")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
##Schematic transition proportions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/gmm_085'
schematic_dir = os.path.join(score_dir, 'transition_schematics')
os.makedirs(schematic_dir, exist_ok=True)

adata = adata_combined

# Cell types and colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': '#7f7f7f'
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata.obs['batch_plate_well'] = (
    adata.obs['Batch'].astype(str) + '_' + 
    adata.obs['plate_well'].astype(str)
)

# -----------------------------
# LINEAGE MAPPING
# -----------------------------
def get_lineage(cell_type):
    """Map cell type to lineage"""
    if cell_type in ['Stem', 'TA']:
        return 'Stem'
    elif cell_type in ['Enterocyte']:
        return 'Absorptive'
    elif cell_type in ['Paneth', 'Goblet', 'Tuft', 'EEC']:
        return 'Secretory'
    else:
        return 'ambiguous'

# Define distinct colors for each specific lineage transition
transition_colors = {
    ('Secretory', 'Secretory'): ('#2E8B57', 'Secretory → Secretory'),
    ('Secretory', 'Absorptive'): ('#20B2AA', 'Secretory → Absorptive'),
    ('Secretory', 'Stem'): ('#008B8B', 'Secretory → Stem'),
    ('Absorptive', 'Absorptive'): ('#FF8C00', 'Absorptive → Absorptive'),
    ('Absorptive', 'Secretory'): ('#FF4500', 'Absorptive → Secretory'),
    ('Absorptive', 'Stem'): ('#DC143C', 'Absorptive → Stem'),
}

gray_color = '#CCCCCC'

# -----------------------------
# PREPARE DATA
# -----------------------------
print("\n" + "="*60)
print("PREPARING DATA")
print("="*60)

plot_df = pd.DataFrame({
    'assay': adata.obs['assay'].values,
    'plate_well': adata.obs['plate_well'].values,
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'Batch': adata.obs['Batch'].values,
    'cell_type_broad': adata.obs['leiden_annot_25'].astype(str).values,
    'module_max_label': adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'Dox': adata.obs['Dox'].astype(str).values if 'Dox' in adata.obs.columns else 'Unknown'
})

dox_conditions = [d for d in plot_df['Dox'].unique() if d != 'Minus' and d != 'nan' and d != 'Unknown']
print(f"Dox conditions: {dox_conditions}")

# -----------------------------
# GET TRANSITIONS PER DOX CONDITION (USING BATCH_PLATE_WELL)
# -----------------------------
print("\n" + "="*60)
print("GETTING TRANSITIONS USING BATCH_PLATE_WELL")
print("="*60)

transitions_by_dox = {}

for dox in dox_conditions:
    dcm_mask = (plot_df['assay'] == 'pDCM') & \
               (plot_df['module_max_label'].isin(label_order)) & \
               (plot_df['Dox'] == dox)
    dcm_cells = plot_df[dcm_mask].copy()
    
    rna_mask = (plot_df['assay'] == 'pRNA') & \
               (plot_df['cell_type_broad'].isin(label_order)) & \
               (plot_df['Dox'] == dox)
    rna_cells = plot_df[rna_mask].copy()
    
    print(f"\n{dox}:")
    print(f"  DCM cells with confident label: {len(dcm_cells)}")
    print(f"  RNA cells with known label: {len(rna_cells)}")
    
    dcm_bpw = set(dcm_cells['batch_plate_well'].unique())
    rna_bpw = set(rna_cells['batch_plate_well'].unique())
    common_bpw = dcm_bpw.intersection(rna_bpw)
    
    print(f"  Common batch_plate_wells: {len(common_bpw)}")
    
    dcm_common = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
    rna_common = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]
    
    dcm_counts = dcm_common['batch_plate_well'].value_counts()
    rna_counts = rna_common['batch_plate_well'].value_counts()
    
    dcm_single = set(dcm_counts[dcm_counts == 1].index)
    rna_single = set(rna_counts[rna_counts == 1].index)
    both_single = dcm_single.intersection(rna_single)
    
    print(f"  1:1 matched batch_plate_wells: {len(both_single)}")
    
    connections = []
    for bpw in both_single:
        dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
        rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
        
        if len(dcm_cell) == 1 and len(rna_cell) == 1:
            dcm_ct = dcm_cell.iloc[0]['module_max_label']
            rna_ct = rna_cell.iloc[0]['cell_type_broad']
            
            connections.append({
                'batch_plate_well': bpw,
                'batch': dcm_cell.iloc[0]['Batch'],
                'plate_well': dcm_cell.iloc[0]['plate_well'],
                'dcm_label': dcm_ct,
                'rna_label': rna_ct,
                'dcm_lineage': get_lineage(dcm_ct),
                'rna_lineage': get_lineage(rna_ct)
            })
    
    if len(connections) > 0:
        transitions_by_dox[dox] = pd.DataFrame(connections)
        print(f"  Final transitions: {len(connections)}")

# -----------------------------
# CIRCULAR DIAGRAM FUNCTION (NO NUMBERS ON ARROWS)
# -----------------------------
def create_circular_schematic(transitions_df, dox_name, label_order, 
                               cell_type_colors, transition_colors, gray_color, save_dir):
    """
    Create circular diagram with all transitions
    NO numbers on arrows - cleaner look
    """
    print(f"\n{'='*60}")
    print(f"CREATING CIRCULAR SCHEMATIC FOR {dox_name}")
    print(f"{'='*60}")
    
    trans_matrix = pd.crosstab(
        transitions_df['dcm_label'],
        transitions_df['rna_label']
    )
    
    for ct in label_order:
        if ct not in trans_matrix.index:
            trans_matrix.loc[ct] = 0
        if ct not in trans_matrix.columns:
            trans_matrix[ct] = 0
    
    trans_matrix = trans_matrix.reindex(index=label_order, columns=label_order, fill_value=0)
    
    fig, ax = plt.subplots(figsize=(16, 14))
    
    n_types = len(label_order)
    angles = np.linspace(0, 2 * np.pi, n_types, endpoint=False)
    radius = 0.30
    center = (0.5, 0.5)
    
    node_positions = {}
    for i, ct in enumerate(label_order):
        x = center[0] + radius * np.cos(angles[i] - np.pi/2)
        y = center[1] + radius * np.sin(angles[i] - np.pi/2)
        node_positions[ct] = (x, y)
    
    node_radius = 0.042
    for ct in label_order:
        x, y = node_positions[ct]
        
        total_from = trans_matrix.loc[ct].sum()
        total_to = trans_matrix[ct].sum()
        
        circle = Circle((x, y), node_radius, 
                        facecolor=cell_type_colors[ct], 
                        edgecolor='black', linewidth=2.5, zorder=10,
                        alpha=0.9 if (total_from > 0 or total_to > 0) else 0.3)
        ax.add_patch(circle)
        
        label_radius = radius + 0.085
        label_x = center[0] + label_radius * np.cos(angles[label_order.index(ct)] - np.pi/2)
        label_y = center[1] + label_radius * np.sin(angles[label_order.index(ct)] - np.pi/2)
        
        lineage = get_lineage(ct)
        ax.text(label_x, label_y, f'{ct}\n({lineage})', 
               fontsize=10, fontweight='bold', ha='center', va='center',
               color=cell_type_colors[ct])
    
    max_count = trans_matrix.values.max()
    min_width = 0.5
    max_width = 8
    
    transition_counts = {key: 0 for key in transition_colors.keys()}
    gray_count = 0
    
    # First pass: draw gray arrows
    for dcm_ct in label_order:
        for rna_ct in label_order:
            count = trans_matrix.loc[dcm_ct, rna_ct]
            if count == 0:
                continue
            
            dcm_lineage = get_lineage(dcm_ct)
            rna_lineage = get_lineage(rna_ct)
            
            if (dcm_lineage, rna_lineage) in transition_colors:
                continue
            
            gray_count += count
            
            start_pos = node_positions[dcm_ct]
            end_pos = node_positions[rna_ct]
            arrow_width = min_width + (count / max_count) * (max_width - min_width)
            
            if dcm_ct == rna_ct:
                angle = angles[label_order.index(dcm_ct)] - np.pi/2
                loop_radius = 0.065
                arc_angles = np.linspace(angle - 0.7, angle + 0.7, 50)
                arc_x = start_pos[0] + loop_radius * np.cos(arc_angles)
                arc_y = start_pos[1] + loop_radius * np.sin(arc_angles)
                
                ax.plot(arc_x, arc_y, color=gray_color, 
                       linewidth=arrow_width, alpha=0.4, zorder=3)
            else:
                dx = end_pos[0] - start_pos[0]
                dy = end_pos[1] - start_pos[1]
                dist = np.sqrt(dx**2 + dy**2)
                
                start_offset = (start_pos[0] + node_radius * dx/dist,
                               start_pos[1] + node_radius * dy/dist)
                end_offset = (end_pos[0] - node_radius * dx/dist,
                             end_pos[1] - node_radius * dy/dist)
                
                style = "arc3,rad=0.15"
                ax.annotate('', xy=end_offset, xytext=start_offset,
                           arrowprops=dict(arrowstyle='-|>',
                                          connectionstyle=style,
                                          color=gray_color,
                                          lw=arrow_width,
                                          alpha=0.4,
                                          mutation_scale=10 + arrow_width * 0.5),
                           zorder=3)
    
    # Second pass: draw colored arrows
    for dcm_ct in label_order:
        for rna_ct in label_order:
            count = trans_matrix.loc[dcm_ct, rna_ct]
            if count == 0:
                continue
            
            dcm_lineage = get_lineage(dcm_ct)
            rna_lineage = get_lineage(rna_ct)
            
            if (dcm_lineage, rna_lineage) not in transition_colors:
                continue
            
            transition_counts[(dcm_lineage, rna_lineage)] += count
            arrow_color, _ = transition_colors[(dcm_lineage, rna_lineage)]
            
            start_pos = node_positions[dcm_ct]
            end_pos = node_positions[rna_ct]
            arrow_width = min_width + (count / max_count) * (max_width - min_width)
            
            if dcm_ct == rna_ct:
                angle = angles[label_order.index(dcm_ct)] - np.pi/2
                loop_radius = 0.065
                arc_angles = np.linspace(angle - 0.7, angle + 0.7, 50)
                arc_x = start_pos[0] + loop_radius * np.cos(arc_angles)
                arc_y = start_pos[1] + loop_radius * np.sin(arc_angles)
                
                ax.plot(arc_x, arc_y, color=arrow_color, 
                       linewidth=arrow_width, alpha=0.85, zorder=6)
            else:
                dx = end_pos[0] - start_pos[0]
                dy = end_pos[1] - start_pos[1]
                dist = np.sqrt(dx**2 + dy**2)
                
                start_offset = (start_pos[0] + node_radius * dx/dist,
                               start_pos[1] + node_radius * dy/dist)
                end_offset = (end_pos[0] - node_radius * dx/dist,
                             end_pos[1] - node_radius * dy/dist)
                
                style = "arc3,rad=0.15"
                ax.annotate('', xy=end_offset, xytext=start_offset,
                           arrowprops=dict(arrowstyle='-|>',
                                          connectionstyle=style,
                                          color=arrow_color,
                                          lw=arrow_width,
                                          alpha=0.85,
                                          mutation_scale=10 + arrow_width * 0.5),
                           zorder=6)
    
    # Legend
    legend_elements = []
    
    for (from_lin, to_lin), (color, label) in transition_colors.items():
        count = transition_counts[(from_lin, to_lin)]
        legend_elements.append(
            plt.Line2D([0], [0], color=color, linewidth=6, alpha=0.85,
                       label=f'{label} (n={count})')
        )
    
    legend_elements.append(
        plt.Line2D([0], [0], color=gray_color, linewidth=4, alpha=0.4,
                   label=f'Stem → any (n={gray_count})')
    )
    
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9, 
             frameon=True, fancybox=True, framealpha=0.95,
             title='Lineage Transitions', title_fontsize=10)
    
    total_cells = trans_matrix.values.sum()
    colored_total = sum(transition_counts.values())
    
    ax.set_title(f'Cell Type Transitions: {dox_name}\n'
                f'Total: {total_cells} cells | Highlighted: {colored_total} | Gray: {gray_count}\n'
                f'(Matched using Batch + Plate_Well)', 
                fontsize=16, fontweight='bold')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.axis('off')
    
    plt.tight_layout()
    
    save_path = os.path.join(save_dir, f'circular_transitions_clean_{dox_name}.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved: {save_path}")
    
    print(f"\n  Transition counts:")
    for (from_lin, to_lin), (color, label) in transition_colors.items():
        count = transition_counts[(from_lin, to_lin)]
        print(f"    {label}: {count}")
    print(f"    Stem → any (gray): {gray_count}")

# -----------------------------
# CREATE ONE PLOT PER DOX CONDITION
# -----------------------------
print("\n" + "="*60)
print("CREATING CIRCULAR DIAGRAMS (ONE PER DOX CONDITION)")
print("="*60)

for dox in dox_conditions:
    if dox in transitions_by_dox:
        create_circular_schematic(
            transitions_by_dox[dox], 
            dox, 
            label_order, 
            cell_type_colors,
            transition_colors,
            gray_color,
            schematic_dir
        )

# -----------------------------
# SUMMARY TABLE
# -----------------------------
print("\n" + "="*60)
print("SUMMARY TABLE")
print("="*60)

summary_data = []

for dox in dox_conditions:
    if dox not in transitions_by_dox:
        continue
    
    df = transitions_by_dox[dox]
    total = len(df)
    
    row = {'Dox': dox, 'Total': total}
    
    for (from_lin, to_lin), (color, label) in transition_colors.items():
        count = ((df['dcm_lineage'] == from_lin) & (df['rna_lineage'] == to_lin)).sum()
        row[label] = count
    
    gray = (df['dcm_lineage'] == 'Stem').sum()
    row['Stem → any'] = gray
    
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print("\n")
print(summary_df.to_string(index=False))

summary_df.to_csv(os.path.join(schematic_dir, 'transition_counts_by_lineage_batch_plate_well.csv'), index=False)
print(f"\nSaved: {os.path.join(schematic_dir, 'transition_counts_by_lineage_batch_plate_well.csv')}")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# -----------------------------
# SETTINGS
# -----------------------------
#score_dir = '/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/integration/plots_paper/'
score_dir = '/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/gene_scoring/gmm_085/'

schematic_dir = os.path.join(score_dir, 'transition_schematics')
os.makedirs(schematic_dir, exist_ok=True)

adata = adata_combined

# Cell types and colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': '#7f7f7f'
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata.obs['batch_plate_well'] = (
    adata.obs['Batch'].astype(str) + '_' + 
    adata.obs['plate_well'].astype(str)
)

# -----------------------------
# LINEAGE MAPPING
# -----------------------------
def get_lineage(cell_type):
    """Map cell type to lineage"""
    if cell_type in ['Stem', 'TA']:
        return 'Stem'
    elif cell_type in ['Enterocyte']:
        return 'Absorptive'
    elif cell_type in ['Paneth', 'Goblet', 'Tuft', 'EEC']:
        return 'Secretory'
    else:
        return 'ambiguous'

# -----------------------------
# PREPARE DATA
# -----------------------------
print("\n" + "="*60)
print("PREPARING DATA")
print("="*60)

plot_df = pd.DataFrame({
    'assay': adata.obs['assay'].values,
    'plate_well': adata.obs['plate_well'].values,
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'Batch': adata.obs['Batch'].values,
    'cell_type_broad': adata.obs['leiden_annot_25'].astype(str).values,
    'module_max_label': adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'Dox': adata.obs['Dox'].astype(str).values if 'Dox' in adata.obs.columns else 'Unknown'
})

dox_conditions = [d for d in plot_df['Dox'].unique() if d != 'Minus' and d != 'nan' and d != 'Unknown']
print(f"Dox conditions: {dox_conditions}")

# -----------------------------
# GET TRANSITIONS PER DOX CONDITION (USING BATCH_PLATE_WELL)
# -----------------------------
print("\n" + "="*60)
print("GETTING TRANSITIONS USING BATCH_PLATE_WELL")
print("="*60)

transitions_by_dox = {}

for dox in dox_conditions:
    dcm_mask = (plot_df['assay'] == 'pDCM') & \
               (plot_df['module_max_label'].isin(label_order)) & \
               (plot_df['Dox'] == dox)
    dcm_cells = plot_df[dcm_mask].copy()
    
    rna_mask = (plot_df['assay'] == 'pRNA') & \
               (plot_df['cell_type_broad'].isin(label_order)) & \
               (plot_df['Dox'] == dox)
    rna_cells = plot_df[rna_mask].copy()
    
    print(f"\n{dox}:")
    print(f"  DCM cells with confident label: {len(dcm_cells)}")
    print(f"  RNA cells with known label: {len(rna_cells)}")
    
    dcm_bpw = set(dcm_cells['batch_plate_well'].unique())
    rna_bpw = set(rna_cells['batch_plate_well'].unique())
    common_bpw = dcm_bpw.intersection(rna_bpw)
    
    print(f"  Common batch_plate_wells: {len(common_bpw)}")
    
    dcm_common = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
    rna_common = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]
    
    dcm_counts = dcm_common['batch_plate_well'].value_counts()
    rna_counts = rna_common['batch_plate_well'].value_counts()
    
    dcm_single = set(dcm_counts[dcm_counts == 1].index)
    rna_single = set(rna_counts[rna_counts == 1].index)
    both_single = dcm_single.intersection(rna_single)
    
    print(f"  1:1 matched batch_plate_wells: {len(both_single)}")
    
    connections = []
    for bpw in both_single:
        dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
        rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
        
        if len(dcm_cell) == 1 and len(rna_cell) == 1:
            dcm_ct = dcm_cell.iloc[0]['module_max_label']
            rna_ct = rna_cell.iloc[0]['cell_type_broad']
            
            connections.append({
                'batch_plate_well': bpw,
                'batch': dcm_cell.iloc[0]['Batch'],
                'plate_well': dcm_cell.iloc[0]['plate_well'],
                'dcm_label': dcm_ct,
                'rna_label': rna_ct,
                'dcm_lineage': get_lineage(dcm_ct),
                'rna_lineage': get_lineage(rna_ct)
            })
    
    if len(connections) > 0:
        transitions_by_dox[dox] = pd.DataFrame(connections)
        print(f"  Final transitions: {len(connections)}")

# -----------------------------
# HEATMAP FUNCTION
# -----------------------------
def create_transition_heatmap(transitions_df, dox_name, label_order, save_dir):
    """
    Create heatmap showing transitions from DCM labels (rows) to RNA labels (columns)
    """
    print(f"\n{'='*60}")
    print(f"CREATING HEATMAP FOR {dox_name}")
    print(f"{'='*60}")
    
    # Create transition matrix
    trans_matrix = pd.crosstab(
        transitions_df['dcm_label'],
        transitions_df['rna_label']
    )
    
    # Ensure all cell types are present
    for ct in label_order:
        if ct not in trans_matrix.index:
            trans_matrix.loc[ct] = 0
        if ct not in trans_matrix.columns:
            trans_matrix[ct] = 0
    
    trans_matrix = trans_matrix.reindex(index=label_order, columns=label_order, fill_value=0)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create heatmap with blue-to-red gradient
    sns.heatmap(trans_matrix, annot=True, fmt='d', cmap='RdBu_r', 
                cbar_kws={'label': 'Number of cells'},
                linewidths=0.5, linecolor='gray',
                ax=ax, vmin=0)
    
    ax.set_xlabel('RNA Cell Type (After)', fontsize=13, fontweight='bold')
    ax.set_ylabel('DCM Cell Type (Before)', fontsize=13, fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    
    # Calculate statistics
    total_cells = trans_matrix.values.sum()
    diagonal_sum = np.diag(trans_matrix.values).sum()
    stability_pct = (diagonal_sum / total_cells * 100) if total_cells > 0 else 0
    
    ax.set_title(f'Cell Type Transitions: {dox_name}\n'
                 f'Total: {total_cells} cells | Same type: {diagonal_sum} ({stability_pct:.1f}%)\n'
                 f'(Matched using Batch + Plate_Well)',
                 fontsize=14, fontweight='bold', pad=15)
    
    plt.tight_layout()
    
    # Save figure
    save_path = os.path.join(save_dir, f'heatmap_transitions_{dox_name}.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved: {save_path}")
    
    # Print transition statistics
    print(f"\n  Transition statistics:")
    print(f"    Total cells: {total_cells}")
    print(f"    Cells maintaining type: {diagonal_sum} ({stability_pct:.1f}%)")
    print(f"    Cells changing type: {total_cells - diagonal_sum} ({100-stability_pct:.1f}%)")
    
    # Save matrix as CSV
    count_csv = os.path.join(save_dir, f'transition_counts_{dox_name}.csv')
    trans_matrix.to_csv(count_csv)
    print(f"    Saved counts: {count_csv}")

# -----------------------------
# CREATE ONE HEATMAP PER DOX CONDITION
# -----------------------------
print("\n" + "="*60)
print("CREATING HEATMAPS (ONE PER DOX CONDITION)")
print("="*60)

for dox in dox_conditions:
    if dox in transitions_by_dox:
        create_transition_heatmap(
            transitions_by_dox[dox], 
            dox, 
            label_order,
            schematic_dir
        )

# -----------------------------
# SUMMARY TABLE
# -----------------------------
print("\n" + "="*60)
print("SUMMARY TABLE")
print("="*60)

summary_data = []

for dox in dox_conditions:
    if dox not in transitions_by_dox:
        continue
    
    df = transitions_by_dox[dox]
    total = len(df)
    
    trans_matrix = pd.crosstab(df['dcm_label'], df['rna_label'])
    diagonal_sum = np.diag(trans_matrix.reindex(index=label_order, columns=label_order, fill_value=0).values).sum()
    
    row = {
        'Dox': dox,
        'Total': total,
        'Same_type': diagonal_sum,
        'Changed_type': total - diagonal_sum,
        'Stability_%': (diagonal_sum / total * 100) if total > 0 else 0
    }
    
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print("\n")
print(summary_df.to_string(index=False))

summary_path = os.path.join(schematic_dir, 'transition_summary_by_dox.csv')
summary_df.to_csv(summary_path, index=False)
print(f"\nSaved: {summary_path}")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
###########Multiple labels############

In [ ]:
adata_combined

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.collections import PatchCollection
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/gmm_085/'
os.makedirs(score_dir, exist_ok=True)

adata = adata_combined
posterior_threshold = 0.85

# Small intestine cell types and colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': '#7f7f7f'
}

# Lineage colors
lineage_colors = {
    'Stem': '#9370DB',
    'Absorptive': '#FF8C00',
    'Secretory': '#2E8B57',
    'ambiguous': '#7f7f7f'
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']
lineage_order = ['Stem', 'Absorptive', 'Secretory']

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata.obs['batch_plate_well'] = (
    adata.obs['Batch'].astype(str) + '_' + 
    adata.obs['plate_well'].astype(str)
)

# -----------------------------
# FUNCTION TO MAP CELL TYPE TO LINEAGE
# -----------------------------
def map_to_lineage(cell_type):
    if cell_type in ['Stem', 'TA']:
        return 'Stem'
    elif cell_type in ['Enterocyte']:
        return 'Absorptive'
    elif cell_type in ['Paneth', 'Goblet', 'Tuft', 'EEC']:
        return 'Secretory'
    else:
        return 'ambiguous'

# -----------------------------
# IDENTIFY POSTERIOR COLUMNS
# -----------------------------
print("\n" + "="*60)
print("STEP 1: IDENTIFYING POSTERIOR COLUMNS")
print("="*60)

# Columns to exclude
exclude_cols = ['ttest_max_posterior_085', 'ttest_gmm_max_posterior_085']

# Find all ttest posterior columns (excluding the max posterior columns)
posterior_cols = [col for col in adata.obs.columns 
                  if col.startswith('ttest_') 
                  and col.endswith('_posterior_085')
                  and col not in exclude_cols]

print(f"Found {len(posterior_cols)} posterior columns (excluding {exclude_cols}):")
for col in posterior_cols:
    ct = col.replace('ttest_', '').replace('_posterior_085', '')
    print(f"  - {col} -> {ct}")

# Extract cell types from column names
cell_types_from_cols = [col.replace('ttest_', '').replace('_posterior_085', '') for col in posterior_cols]

# -----------------------------
# COUNT SIGNIFICANT MODULES PER CELL
# -----------------------------
print("\n" + "="*60)
print("STEP 2: COUNTING SIGNIFICANT MODULES PER CELL")
print("="*60)

# Initialize columns
adata.obs['n_significant_modules'] = 0
adata.obs['significant_celltypes'] = ''
adata.obs['significant_lineages'] = ''
adata.obs['is_multipotent'] = False

# Process each cell
for idx in adata.obs.index:
    significant_cts = []
    
    for col in posterior_cols:
        posterior_val = adata.obs.loc[idx, col]
        
        if pd.notna(posterior_val) and posterior_val >= posterior_threshold:
            ct = col.replace('ttest_', '').replace('_posterior_085', '')
            significant_cts.append(ct)
    
    # Store results
    adata.obs.loc[idx, 'n_significant_modules'] = len(significant_cts)
    adata.obs.loc[idx, 'significant_celltypes'] = ';'.join(sorted(significant_cts)) if significant_cts else 'none'
    
    # Map to lineages
    significant_lineages = list(set([map_to_lineage(ct) for ct in significant_cts]))
    significant_lineages = [l for l in significant_lineages if l != 'ambiguous']
    adata.obs.loc[idx, 'significant_lineages'] = ';'.join(sorted(significant_lineages)) if significant_lineages else 'none'
    
    # Is multipotent (more than one significant module)
    adata.obs.loc[idx, 'is_multipotent'] = len(significant_cts) > 1

# Convert to appropriate types
adata.obs['n_significant_modules'] = adata.obs['n_significant_modules'].astype(int)
adata.obs['is_multipotent'] = adata.obs['is_multipotent'].astype(bool)

# Print summary
print(f"\nNumber of significant modules distribution:")
print(adata.obs['n_significant_modules'].value_counts().sort_index())

print(f"\nMultipotent cells (>1 significant module):")
print(adata.obs['is_multipotent'].value_counts())

print(f"\nTop 20 significant cell type combinations:")
print(adata.obs['significant_celltypes'].value_counts().head(20))

print(f"\nSignificant lineage combinations:")
print(adata.obs['significant_lineages'].value_counts())

# -----------------------------
# EXTRACT UMAP AND METADATA
# -----------------------------
print("\n" + "="*60)
print("STEP 3: PREPARING PLOT DATA")
print("="*60)

umap_coords = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1': umap_coords[:, 0],
    'UMAP2': umap_coords[:, 1],
    'assay': adata.obs['assay'].values,
    'plate_well': adata.obs['plate_well'].values,
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'Batch': adata.obs['Batch'].values,
    'cell_type_broad': adata.obs['leiden_annot_25'].astype(str).values,
    'module_max_label': adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'n_significant_modules': adata.obs['n_significant_modules'].values,
    'significant_celltypes': adata.obs['significant_celltypes'].values,
    'significant_lineages': adata.obs['significant_lineages'].values,
    'is_multipotent': adata.obs['is_multipotent'].values,
    'cell_index': adata.obs.index.values
})

# Add posterior values (only the cell-type specific ones)
for col in posterior_cols:
    ct = col.replace('ttest_', '').replace('_posterior', '')
    plot_df[f'{ct}_posterior'] = adata.obs[col].values

if 'Dox' in adata.obs.columns:
    plot_df['Dox'] = adata.obs['Dox'].values

# Add lineage columns
plot_df['cell_type_lineage'] = plot_df['cell_type_broad'].apply(map_to_lineage)
plot_df['module_lineage'] = plot_df['module_max_label'].apply(map_to_lineage)

# -----------------------------
# FILTER DCM AND RNA CELLS (EXCLUDE MINUS DOX)
# -----------------------------
print("\n" + "="*60)
print("STEP 4: FILTERING CELLS")
print("="*60)

# DCM cells with at least one significant module (exclude Minus Dox)
dcm_mask = plot_df['assay'] == 'pDCM'
if 'Dox' in plot_df.columns:
    dcm_mask = dcm_mask & (plot_df['Dox'] != 'Minus')
dcm_significant_mask = dcm_mask & (plot_df['n_significant_modules'] >= 1)
dcm_cells = plot_df[dcm_significant_mask].copy()

print(f"Total DCM cells (excluding Minus Dox): {dcm_mask.sum()}")
print(f"DCM cells with ≥1 significant module: {dcm_significant_mask.sum()}")
print(f"DCM multipotent cells: {(dcm_cells['is_multipotent']).sum()}")

print("\nDCM cells by number of significant modules:")
print(dcm_cells['n_significant_modules'].value_counts().sort_index())

# RNA cells (exclude Minus Dox)
rna_mask = plot_df['assay'] == 'pRNA'
if 'Dox' in plot_df.columns:
    rna_mask = rna_mask & (plot_df['Dox'] != 'Minus')
rna_cells = plot_df[rna_mask].copy()
print(f"\nTotal RNA cells (excluding Minus Dox): {rna_mask.sum()}")

# -----------------------------
# CHECK DUPLICATES WITH BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("CHECKING DUPLICATES")
print("="*60)

dcm_bpw_counts = dcm_cells['batch_plate_well'].value_counts()
rna_bpw_counts = rna_cells['batch_plate_well'].value_counts()

print(f"DCM batch_plate_wells with >1 cell: {(dcm_bpw_counts > 1).sum()}")
print(f"RNA batch_plate_wells with >1 cell: {(rna_bpw_counts > 1).sum()}")

# -----------------------------
# MATCH BY BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("STEP 5: FINDING CONNECTIONS USING BATCH_PLATE_WELL")
print("="*60)

dcm_bpw = set(dcm_cells['batch_plate_well'])
rna_bpw = set(rna_cells['batch_plate_well'])
common_bpw = dcm_bpw.intersection(rna_bpw)
print(f"Common batch_plate_wells: {len(common_bpw)}")

# Filter to common
dcm_common = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
rna_common = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]

# Find 1:1 matches
dcm_counts = dcm_common['batch_plate_well'].value_counts()
rna_counts = rna_common['batch_plate_well'].value_counts()

dcm_single = set(dcm_counts[dcm_counts == 1].index)
rna_single = set(rna_counts[rna_counts == 1].index)
both_single = dcm_single.intersection(rna_single)

print(f"1:1 matched batch_plate_wells: {len(both_single)}")

connections = []
for bpw in both_single:
    dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
    rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
    
    if len(dcm_cell) == 1 and len(rna_cell) == 1:
        dcm_row = dcm_cell.iloc[0]
        rna_row = rna_cell.iloc[0]
        
        # Parse significant cell types
        sig_cts = dcm_row['significant_celltypes'].split(';') if dcm_row['significant_celltypes'] != 'none' else []
        sig_lineages = dcm_row['significant_lineages'].split(';') if dcm_row['significant_lineages'] != 'none' else []
        
        connections.append({
            'batch_plate_well': bpw,
            'batch': dcm_row['Batch'],
            'plate_well': dcm_row['plate_well'],
            'dcm_umap1': dcm_row['UMAP1'],
            'dcm_umap2': dcm_row['UMAP2'],
            'rna_umap1': rna_row['UMAP1'],
            'rna_umap2': rna_row['UMAP2'],
            # DCM info
            'dcm_cell_type': dcm_row['module_max_label'],
            'dcm_n_significant': dcm_row['n_significant_modules'],
            'dcm_significant_celltypes': dcm_row['significant_celltypes'],
            'dcm_significant_lineages': dcm_row['significant_lineages'],
            'dcm_is_multipotent': dcm_row['is_multipotent'],
            'dcm_sig_ct_list': sig_cts,
            'dcm_sig_lineage_list': sig_lineages,
            # RNA info
            'rna_cell_type': rna_row['cell_type_broad'],
            'rna_lineage': rna_row['cell_type_lineage'],
            # Transitions
            'transition': f"{dcm_row['module_max_label']} -> {rna_row['cell_type_broad']}",
            'lineage_transition': f"{dcm_row['module_lineage']} -> {rna_row['cell_type_lineage']}"
        })

connections_df = pd.DataFrame(connections)
print(f"Total valid connections: {len(connections_df)}")

if len(connections_df) > 0:
    print(f"\nConnections with multipotent DCM cells: {connections_df['dcm_is_multipotent'].sum()}")
    print("\nDCM significant cell type combinations:")
    print(connections_df['dcm_significant_celltypes'].value_counts().head(20))

# -----------------------------
# FUNCTION TO DRAW MULTI-COLORED SQUARE
# -----------------------------
def draw_multicolor_square(ax, x, y, cell_types, size=4, zorder=4):
    """
    Draw a square divided into sections, each colored by cell type
    """
    n_types = len(cell_types)
    
    if n_types == 0:
        return
    elif n_types == 1:
        # Single color square
        color = cell_type_colors.get(cell_types[0], '#7f7f7f')
        rect = plt.Rectangle((x - size/2, y - size/2), size, size,
                             facecolor=color, edgecolor='black', linewidth=1.5, zorder=zorder)
        ax.add_patch(rect)
    else:
        # Multi-colored square - divide horizontally
        section_width = size / n_types
        for i, ct in enumerate(sorted(cell_types)):
            color = cell_type_colors.get(ct, '#7f7f7f')
            rect = plt.Rectangle((x - size/2 + i * section_width, y - size/2), 
                                 section_width, size,
                                 facecolor=color, edgecolor='none', zorder=zorder)
            ax.add_patch(rect)
        
        # Add border
        border = plt.Rectangle((x - size/2, y - size/2), size, size,
                               facecolor='none', edgecolor='black', linewidth=1.5, zorder=zorder+1)
        ax.add_patch(border)

# -----------------------------
# PLOT ALL MULTIPOTENT TRANSITIONS
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("PLOTTING MULTIPOTENT CELL TRANSITIONS")
    print("="*60)
    
    # Filter to multipotent DCM cells
    multipotent_connections = connections_df[connections_df['dcm_is_multipotent']].copy()
    
    if len(multipotent_connections) > 0:
        print(f"\nPlotting {len(multipotent_connections)} multipotent cell transitions")
        
        # Get UMAP limits for consistent square sizes
        umap1_range = plot_df['UMAP1'].max() - plot_df['UMAP1'].min()
        umap2_range = plot_df['UMAP2'].max() - plot_df['UMAP2'].min()
        square_size = min(umap1_range, umap2_range) * 0.02
        
        fig, ax = plt.subplots(figsize=(12, 10))
        
        # Reference background
        for ct in label_order:
            ref_cells = plot_df[(plot_df['assay'] == 'Ref') & (plot_df['cell_type_broad'] == ct)]
            if len(ref_cells) > 0:
                ax.scatter(ref_cells['UMAP1'], ref_cells['UMAP2'],
                           s=8, c=cell_type_colors[ct], alpha=0.3, marker='o', zorder=1, rasterized=True)
        
        ref_ambig = plot_df[(plot_df['assay'] == 'Ref') & (~plot_df['cell_type_broad'].isin(label_order))]
        if len(ref_ambig) > 0:
            ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                       s=8, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)
        
        # Draw arrows and cells - ARROWS COLORED BY RNA CELL TYPE
        for _, conn in multipotent_connections.iterrows():
            arrow_color = cell_type_colors.get(conn['rna_cell_type'], '#7f7f7f')
            arrow = FancyArrowPatch((conn['dcm_umap1'], conn['dcm_umap2']),
                                    (conn['rna_umap1'], conn['rna_umap2']),
                                    arrowstyle='-|>', mutation_scale=15,
                                    color=arrow_color, alpha=0.7, linewidth=2, zorder=3)
            ax.add_patch(arrow)
            
            draw_multicolor_square(ax, conn['dcm_umap1'], conn['dcm_umap2'], 
                                  conn['dcm_sig_ct_list'], size=square_size, zorder=4)
            
            rna_color = cell_type_colors.get(conn['rna_cell_type'], '#7f7f7f')
            ax.scatter(conn['rna_umap1'], conn['rna_umap2'],
                      s=100, c=rna_color, alpha=0.9,
                      edgecolors='black', linewidths=1, marker='o', zorder=4)
        
        # Legend
        legend_elements = []
        for ct in label_order:
            legend_elements.append(
                mlines.Line2D([], [], color=cell_type_colors[ct], marker='s', linestyle='None', 
                             markersize=10, markeredgecolor='black', label=ct)
            )
        legend_elements.append(
            mlines.Line2D([], [], color='black', linestyle='-', linewidth=2, alpha=0.7,
                         marker='>', markersize=8, label='DCM → RNA (colored by RNA type)')
        )
        
        ax.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=9, ncol=2)
        
        ax.set_title(f'Multipotent DCM Cell Transitions\n(n = {len(multipotent_connections)} cells with >1 significant module)\n(Matched using Batch + Plate_Well)', 
                    fontsize=14, fontweight='bold')
        ax.set_aspect('equal')
        ax.axis('off')
        
        plt.tight_layout()
        save_path = os.path.join(score_dir, 'umap_multipotent_transitions_all.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f"Saved: {save_path}")
    else:
        print("No multipotent cells found")
    
    # -----------------------------
    # PLOT BY SPECIFIC MULTIPOTENT COMBINATIONS
    # -----------------------------
    if len(multipotent_connections) > 0:
        print("\n" + "="*60)
        print("PLOTTING BY MULTIPOTENT COMBINATION")
        print("="*60)
        
        unique_combinations = multipotent_connections['dcm_significant_celltypes'].unique()
        
        for combo in unique_combinations:
            combo_connections = multipotent_connections[multipotent_connections['dcm_significant_celltypes'] == combo]
            n_cells = len(combo_connections)
            
            if n_cells < 1:
                continue
            
            print(f"\nPlotting combination: {combo} ({n_cells} cells)")
            
            fig, ax = plt.subplots(figsize=(12, 10))
            
            # Reference background
            for ct in label_order:
                ref_cells = plot_df[(plot_df['assay'] == 'Ref') & (plot_df['cell_type_broad'] == ct)]
                if len(ref_cells) > 0:
                    ax.scatter(ref_cells['UMAP1'], ref_cells['UMAP2'],
                               s=8, c=cell_type_colors[ct], alpha=0.3, marker='o', zorder=1, rasterized=True)
            
            ref_ambig = plot_df[(plot_df['assay'] == 'Ref') & (~plot_df['cell_type_broad'].isin(label_order))]
            if len(ref_ambig) > 0:
                ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                           s=8, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)
            
            # Draw arrows and cells - ARROWS COLORED BY RNA CELL TYPE
            for _, conn in combo_connections.iterrows():
                arrow_color = cell_type_colors.get(conn['rna_cell_type'], '#7f7f7f')
                arrow = FancyArrowPatch((conn['dcm_umap1'], conn['dcm_umap2']),
                                        (conn['rna_umap1'], conn['rna_umap2']),
                                        arrowstyle='-|>', mutation_scale=15,
                                        color=arrow_color, alpha=0.7, linewidth=2, zorder=3)
                ax.add_patch(arrow)
                
                draw_multicolor_square(ax, conn['dcm_umap1'], conn['dcm_umap2'], 
                                      conn['dcm_sig_ct_list'], size=square_size, zorder=4)
                
                rna_color = cell_type_colors.get(conn['rna_cell_type'], '#7f7f7f')
                ax.scatter(conn['rna_umap1'], conn['rna_umap2'],
                          s=100, c=rna_color, alpha=0.9,
                          edgecolors='black', linewidths=1, marker='o', zorder=4)
            
            # Legend
            combo_cts = combo.split(';')
            rna_cts = combo_connections['rna_cell_type'].unique()
            relevant_cts = list(set(combo_cts) | set(rna_cts))
            
            legend_elements = []
            for ct in label_order:
                if ct in relevant_cts:
                    legend_elements.append(
                        mlines.Line2D([], [], color=cell_type_colors[ct], marker='s', linestyle='None', 
                                     markersize=10, markeredgecolor='black', label=ct)
                    )
            legend_elements.append(
                mlines.Line2D([], [], color='black', linestyle='-', linewidth=2, alpha=0.7,
                             marker='>', markersize=8, label='DCM → RNA (colored by RNA type)')
            )
            
            ax.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=10)
            
            rna_dist = combo_connections['rna_cell_type'].value_counts()
            rna_str = ', '.join([f"{ct}: {count}" for ct, count in rna_dist.items()])
            
            ax.set_title(f'Multipotent DCM: {combo}\n(n = {n_cells} cells) → RNA: {rna_str}\n(Matched using Batch + Plate_Well)', 
                        fontsize=12, fontweight='bold')
            ax.set_aspect('equal')
            ax.axis('off')
            
            plt.tight_layout()
            safe_name = combo.replace(';', '_')
            save_path = os.path.join(score_dir, f'umap_multipotent_{safe_name}.pdf')
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.show()
            plt.close()
            print(f"Saved: {save_path}")

# -----------------------------
# PLOT SINGLE VS MULTIPOTENT COMPARISON
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("PLOTTING SINGLE VS MULTIPOTENT COMPARISON")
    print("="*60)
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    umap1_range = plot_df['UMAP1'].max() - plot_df['UMAP1'].min()
    umap2_range = plot_df['UMAP2'].max() - plot_df['UMAP2'].min()
    square_size = min(umap1_range, umap2_range) * 0.015
    
    for ax_idx, (is_multi, title) in enumerate([(False, 'Single Module'), (True, 'Multipotent (>1 Module)')]):
        ax = axes[ax_idx]
        
        subset = connections_df[connections_df['dcm_is_multipotent'] == is_multi]
        
        # Reference background
        for ct in label_order:
            ref_cells = plot_df[(plot_df['assay'] == 'Ref') & (plot_df['cell_type_broad'] == ct)]
            if len(ref_cells) > 0:
                ax.scatter(ref_cells['UMAP1'], ref_cells['UMAP2'],
                           s=5, c=cell_type_colors[ct], alpha=0.3, marker='o', zorder=1, rasterized=True)
        
        ref_ambig = plot_df[(plot_df['assay'] == 'Ref') & (~plot_df['cell_type_broad'].isin(label_order))]
        if len(ref_ambig) > 0:
            ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                       s=5, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)
        
        # Draw arrows - COLORED BY RNA CELL TYPE
        for _, conn in subset.iterrows():
            arrow_color = cell_type_colors.get(conn['rna_cell_type'], '#7f7f7f')
            arrow = FancyArrowPatch((conn['dcm_umap1'], conn['dcm_umap2']),
                                    (conn['rna_umap1'], conn['rna_umap2']),
                                    arrowstyle='-|>', mutation_scale=12,
                                    color=arrow_color, alpha=0.7, linewidth=1.5, zorder=3)
            ax.add_patch(arrow)
            
            if is_multi:
                draw_multicolor_square(ax, conn['dcm_umap1'], conn['dcm_umap2'], 
                                      conn['dcm_sig_ct_list'], size=square_size, zorder=4)
            else:
                dcm_color = cell_type_colors.get(conn['dcm_cell_type'], '#7f7f7f')
                ax.scatter(conn['dcm_umap1'], conn['dcm_umap2'],
                          s=80, c=dcm_color, alpha=0.9,
                          edgecolors='black', linewidths=1, marker='s', zorder=4)
            
            rna_color = cell_type_colors.get(conn['rna_cell_type'], '#7f7f7f')
            ax.scatter(conn['rna_umap1'], conn['rna_umap2'],
                      s=80, c=rna_color, alpha=0.9,
                      edgecolors='black', linewidths=1, marker='o', zorder=4)
        
        ax.set_title(f'{title}\n(n = {len(subset)} cells)', fontsize=14, fontweight='bold')
        ax.set_aspect('equal')
        ax.axis('off')
    
    legend_elements = [
        mlines.Line2D([], [], color=cell_type_colors[ct], marker='s', linestyle='None', 
                     markersize=10, markeredgecolor='black', label=ct)
        for ct in label_order
    ]
    legend_elements.append(
        mlines.Line2D([], [], color='black', linestyle='-', linewidth=2,
                     marker='>', markersize=8, label='DCM → RNA (colored by RNA type)')
    )
    
    fig.legend(handles=legend_elements, loc='center right', frameon=True, fontsize=10, 
              bbox_to_anchor=(0.98, 0.5))
    
    fig.suptitle('Single vs Multipotent DCM Cell Transitions\n(Matched using Batch + Plate_Well)', 
                fontsize=16, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    plt.subplots_adjust(right=0.88)
    save_path = os.path.join(score_dir, 'umap_single_vs_multipotent.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved: {save_path}")

# -----------------------------
# SUMMARY STATISTICS
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("SUMMARY STATISTICS")
    print("="*60)
    
    print(f"\nTotal connections: {len(connections_df)}")
    print(f"Single module DCM cells: {(~connections_df['dcm_is_multipotent']).sum()}")
    print(f"Multipotent DCM cells: {connections_df['dcm_is_multipotent'].sum()}")
    
    print("\n--- Multipotent Cell Outcomes ---")
    multi_df = connections_df[connections_df['dcm_is_multipotent']]
    if len(multi_df) > 0:
        print("\nDCM combination -> RNA outcome:")
        for combo in multi_df['dcm_significant_celltypes'].unique():
            combo_data = multi_df[multi_df['dcm_significant_celltypes'] == combo]
            rna_dist = combo_data['rna_cell_type'].value_counts()
            print(f"\n  {combo} (n={len(combo_data)}):")
            for rna_ct, count in rna_dist.items():
                pct = (count / len(combo_data)) * 100
                print(f"    -> {rna_ct}: {count} ({pct:.1f}%)")
    
    connections_df.to_csv(os.path.join(score_dir, 'DCM_RNA_connections_multipotent_batch_plate_well.csv'), index=False)
    print(f"\nSaved connections to: {os.path.join(score_dir, 'DCM_RNA_connections_multipotent_batch_plate_well.csv')}")

# -----------------------------
# SAVE ADATA WITH NEW COLUMNS
# -----------------------------
print("\n" + "="*60)
print("SAVING UPDATED ADATA")
print("="*60)

print("\nNew columns added to adata.obs:")
print("  - batch_plate_well: Combined Batch + plate_well identifier")
print("  - n_significant_modules: Number of modules with posterior ≥ 0.95")
print("  - significant_celltypes: Cell types with significant posteriors (semicolon-separated)")
print("  - significant_lineages: Lineages with significant posteriors")
print("  - is_multipotent: Boolean, True if >1 significant module")

print(f"\nExcluded columns from analysis: {exclude_cols}")

try:
    save_path = os.path.join(score_dir, 'adata_with_multipotent_info.h5ad')
    adata.write_h5ad(save_path)
    print(f"\n✅ Saved adata to: {save_path}")
except Exception as e:
    print(f"\n⚠️ Could not save adata: {e}")
    csv_path = os.path.join(score_dir, 'multipotent_metadata.csv')
    adata.obs[['batch_plate_well', 'n_significant_modules', 'significant_celltypes', 'significant_lineages', 'is_multipotent']].to_csv(csv_path)
    print(f"  Saved metadata to: {csv_path}")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:

## Summary of what this code does:

### 1. New Columns Added to `adata.obs`:

| Column | Description |
|--------|-------------|
| `n_significant_modules` | Number of cell types with posterior ≥ 0.95 |
| `significant_celltypes` | Semicolon-separated list of significant cell types (e.g., "Stem;TA") |
| `significant_lineages` | Semicolon-separated list of significant lineages |
| `is_multipotent` | Boolean: True if cell has >1 significant module |

### 2. Visualizations:

| Plot | Description |
|------|-------------|
| `umap_multipotent_transitions_all.pdf` | All multipotent cells with multi-colored squares |
| `umap_multipotent_{combination}.pdf` | Individual plots per cell type combination |
| `umap_single_vs_multipotent.pdf` | Side-by-side comparison |

### 3. Multi-colored Squares:

The `draw_multicolor_square()` function creates squares divided into horizontal sections, each colored by the significant cell types:
```
Single (Stem):     Multi (Stem;TA):     Multi (Stem;TA;Enterocyte):
┌─────────┐        ┌────┬────┐          ┌───┬───┬───┐
│  Purple │        │Pur │Tan │          │Pur│Tan│Org│
└─────────┘        └────┴────┘          └───┴───┴───┘

In [ ]:
##Heatmap module score per transition

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/'
os.makedirs(score_dir, exist_ok=True)

adata = adata_combined

# Cell types and colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': '#7f7f7f'
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata.obs['batch_plate_well'] = (
    adata.obs['Batch'].astype(str) + '_' + 
    adata.obs['plate_well'].astype(str)
)

# -----------------------------
# IDENTIFY MODULE SCORE COLUMNS
# -----------------------------
print("\n" + "="*60)
print("STEP 1: IDENTIFYING MODULE SCORE COLUMNS")
print("="*60)

module_cols = []
for ct in label_order:
    col_name = f'ttest_{ct}_module'
    if col_name in adata.obs.columns:
        module_cols.append(col_name)
        print(f"  Found: {col_name}")
    else:
        alt_names = [f'ttest_{ct}_zscore', f'ttest_{ct}_score']
        for alt in alt_names:
            if alt in adata.obs.columns:
                module_cols.append(alt)
                print(f"  Found (alt): {alt}")
                break

print(f"\nTotal module columns found: {len(module_cols)}")

# -----------------------------
# PREPARE DATA
# -----------------------------
print("\n" + "="*60)
print("STEP 2: PREPARING DATA")
print("="*60)

umap_coords = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1': umap_coords[:, 0],
    'UMAP2': umap_coords[:, 1],
    'assay': adata.obs['assay'].values,
    'plate_well': adata.obs['plate_well'].values,
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'Batch': adata.obs['Batch'].values,
    'cell_type_broad': adata.obs['leiden_annot_25'].astype(str).values,
    'module_max_label': adata.obs['ttest_gmm_label_confident'].astype(str).values,
    'cell_index': adata.obs.index.values
})

if 'Dox' in adata.obs.columns:
    plot_df['Dox'] = adata.obs['Dox'].values

# Add module scores
for col in module_cols:
    plot_df[col] = adata.obs[col].values

# -----------------------------
# FILTER AND MATCH CELLS (EXCLUDE MINUS DOX)
# -----------------------------
print("\n" + "="*60)
print("STEP 3: FILTERING AND MATCHING CELLS")
print("="*60)

dcm_mask = plot_df['assay'] == 'pDCM'
dcm_known_mask = dcm_mask & plot_df['module_max_label'].isin(label_order)

if 'Dox' in plot_df.columns:
    dcm_known_mask = dcm_known_mask & (plot_df['Dox'] != 'Minus')

dcm_cells = plot_df[dcm_known_mask].copy()

rna_mask = plot_df['assay'] == 'pRNA'
rna_known_mask = rna_mask & plot_df['cell_type_broad'].isin(label_order)

if 'Dox' in plot_df.columns:
    rna_known_mask = rna_known_mask & (plot_df['Dox'] != 'Minus')

rna_cells = plot_df[rna_known_mask].copy()

print(f"DCM cells with known labels (excluding Minus Dox): {len(dcm_cells)}")
print(f"RNA cells with known labels (excluding Minus Dox): {len(rna_cells)}")

# -----------------------------
# CHECK DUPLICATES WITH BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("CHECKING DUPLICATES")
print("="*60)

dcm_bpw_counts = dcm_cells['batch_plate_well'].value_counts()
rna_bpw_counts = rna_cells['batch_plate_well'].value_counts()

print(f"DCM batch_plate_wells with >1 cell: {(dcm_bpw_counts > 1).sum()}")
print(f"RNA batch_plate_wells with >1 cell: {(rna_bpw_counts > 1).sum()}")

# -----------------------------
# MATCH BY BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("FINDING CONNECTIONS USING BATCH_PLATE_WELL")
print("="*60)

dcm_bpw = set(dcm_cells['batch_plate_well'])
rna_bpw = set(rna_cells['batch_plate_well'])
common_bpw = dcm_bpw.intersection(rna_bpw)
print(f"Common batch_plate_wells: {len(common_bpw)}")

dcm_common = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
rna_common = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]

dcm_counts = dcm_common['batch_plate_well'].value_counts()
rna_counts = rna_common['batch_plate_well'].value_counts()

dcm_single = set(dcm_counts[dcm_counts == 1].index)
rna_single = set(rna_counts[rna_counts == 1].index)
both_single = dcm_single.intersection(rna_single)

print(f"1:1 matched batch_plate_wells: {len(both_single)}")

# Create connections with module scores
connections = []
for bpw in both_single:
    dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
    rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
    
    if len(dcm_cell) == 1 and len(rna_cell) == 1:
        dcm_row = dcm_cell.iloc[0]
        rna_row = rna_cell.iloc[0]
        
        conn = {
            'batch_plate_well': bpw,
            'batch': dcm_row['Batch'],
            'plate_well': dcm_row['plate_well'],
            'dcm_cell_type': dcm_row['module_max_label'],
            'rna_cell_type': rna_row['cell_type_broad'],
            'transition': f"{dcm_row['module_max_label']} -> {rna_row['cell_type_broad']}",
            'dcm_index': dcm_row['cell_index'],
            'rna_index': rna_row['cell_index']
        }
        
        for col in module_cols:
            ct = col.replace('ttest_', '').replace('_module', '').replace('_zscore', '').replace('_score', '')
            conn[f'dcm_{ct}_score'] = dcm_row[col]
        
        for col in module_cols:
            ct = col.replace('ttest_', '').replace('_module', '').replace('_zscore', '').replace('_score', '')
            conn[f'rna_{ct}_score'] = rna_row[col]
        
        connections.append(conn)

connections_df = pd.DataFrame(connections)
print(f"Total valid connections: {len(connections_df)}")

print("\nTransition counts:")
print(connections_df['transition'].value_counts())

# -----------------------------
# Z-SCORE TRANSFORMATION
# -----------------------------
print("\n" + "="*60)
print("STEP 4: Z-SCORE TRANSFORMATION")
print("="*60)

dcm_score_cols = [col for col in connections_df.columns if col.startswith('dcm_') and col.endswith('_score')]
rna_score_cols = [col for col in connections_df.columns if col.startswith('rna_') and col.endswith('_score')]

# Get raw values
dcm_all_values = connections_df[dcm_score_cols].values.flatten()
dcm_all_values = dcm_all_values[~np.isnan(dcm_all_values)]

rna_all_values = connections_df[rna_score_cols].values.flatten()
rna_all_values = rna_all_values[~np.isnan(rna_all_values)]

print(f"\nRaw DCM scores: mean={dcm_all_values.mean():.2f}, std={dcm_all_values.std():.2f}")
print(f"Raw RNA scores: mean={rna_all_values.mean():.2f}, std={rna_all_values.std():.2f}")

# Calculate Z-score parameters for DCM and RNA separately
dcm_mean = dcm_all_values.mean()
dcm_std = dcm_all_values.std()

rna_mean = rna_all_values.mean()
rna_std = rna_all_values.std()

# Create Z-scored columns
for col in dcm_score_cols:
    zscore_col = col.replace('_score', '_zscore')
    connections_df[zscore_col] = (connections_df[col] - dcm_mean) / dcm_std

for col in rna_score_cols:
    zscore_col = col.replace('_score', '_zscore')
    connections_df[zscore_col] = (connections_df[col] - rna_mean) / rna_std

# Get Z-scored column names
dcm_zscore_cols = [col for col in connections_df.columns if col.startswith('dcm_') and col.endswith('_zscore')]
rna_zscore_cols = [col for col in connections_df.columns if col.startswith('rna_') and col.endswith('_zscore')]

# Verify Z-score transformation
dcm_z_values = connections_df[dcm_zscore_cols].values.flatten()
dcm_z_values = dcm_z_values[~np.isnan(dcm_z_values)]

rna_z_values = connections_df[rna_zscore_cols].values.flatten()
rna_z_values = rna_z_values[~np.isnan(rna_z_values)]

print(f"\nZ-scored DCM: mean={dcm_z_values.mean():.4f}, std={dcm_z_values.std():.4f}")
print(f"Z-scored RNA: mean={rna_z_values.mean():.4f}, std={rna_z_values.std():.4f}")

# Set common Z-score scale (symmetric around 0)
z_limit = 3  # Standard: +/- 3 SD covers 99.7% of data
vmin = -z_limit
vmax = z_limit

print(f"\nZ-score scale: [{vmin}, {vmax}]")

# -----------------------------
# FUNCTION TO CREATE HEATMAP WITH Z-SCORE SCALING
# -----------------------------
def create_transition_heatmap_zscore(connections_df, dcm_cell_type, label_order, 
                                      cell_type_colors, score_dir, vmin, vmax):
    """
    Create heatmap for all transitions starting from a specific DCM cell type
    using Z-SCORE scaling (same scale for DCM and RNA)
    """
    print(f"\n{'='*60}")
    print(f"CREATING Z-SCORE HEATMAP FOR DCM {dcm_cell_type.upper()} TRANSITIONS")
    print(f"{'='*60}")
    
    subset = connections_df[connections_df['dcm_cell_type'] == dcm_cell_type].copy()
    
    if len(subset) == 0:
        print(f"  No transitions found for DCM {dcm_cell_type}")
        return
    
    print(f"  Total cells: {len(subset)}")
    print(f"  Transitions: {subset['transition'].value_counts().to_dict()}")
    
    all_dcm_data = []
    all_rna_data = []
    transition_boundaries = []
    transition_info = []
    
    current_pos = 0
    
    for rna_ct in label_order:
        trans_subset = subset[subset['rna_cell_type'] == rna_ct]
        
        if len(trans_subset) == 0:
            continue
        
        dcm_zscore_col = f'dcm_{dcm_cell_type}_zscore'
        rna_zscore_col = f'rna_{rna_ct}_zscore'
        
        trans_subset = trans_subset.sort_values(
            by=[dcm_zscore_col, rna_zscore_col], 
            ascending=[False, False]
        )
        
        for _, row in trans_subset.iterrows():
            dcm_zscores = [row[f'dcm_{ct}_zscore'] for ct in label_order]
            all_dcm_data.append(dcm_zscores)
        
        for _, row in trans_subset.iterrows():
            rna_zscores = [row[f'rna_{ct}_zscore'] for ct in label_order]
            all_rna_data.append(rna_zscores)
        
        transition_info.append((rna_ct, len(trans_subset), current_pos))
        current_pos += len(trans_subset)
        transition_boundaries.append(current_pos)
    
    if len(all_dcm_data) == 0:
        print(f"  No data to plot")
        return
    
    dcm_matrix = np.array(all_dcm_data).T
    rna_matrix = np.array(all_rna_data).T
    
    print(f"  DCM matrix shape: {dcm_matrix.shape}")
    print(f"  RNA matrix shape: {rna_matrix.shape}")
    
    fig = plt.figure(figsize=(max(14, len(all_dcm_data) * 0.15), 12))
    
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 0.03], height_ratios=[1, 1],
                         hspace=0.25, wspace=0.05)
    
    cmap = plt.cm.RdBu_r
    
    # DCM Heatmap
    ax_dcm = fig.add_subplot(gs[0, 0])
    im_dcm = ax_dcm.imshow(dcm_matrix, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    
    for boundary in transition_boundaries[:-1]:
        ax_dcm.axvline(x=boundary - 0.5, color='white', linewidth=2)
    
    dcm_row_idx = label_order.index(dcm_cell_type)
    ax_dcm.axhline(y=dcm_row_idx - 0.5, color='black', linewidth=2, linestyle='--', alpha=0.8)
    ax_dcm.axhline(y=dcm_row_idx + 0.5, color='black', linewidth=2, linestyle='--', alpha=0.8)
    
    ax_dcm.set_yticks(range(len(label_order)))
    ax_dcm.set_yticklabels(label_order, fontsize=10)
    ax_dcm.set_xticks([])
    ax_dcm.set_title(f'DCM Module Z-Scores (cells starting as {dcm_cell_type})', fontsize=12, fontweight='bold')
    ax_dcm.set_ylabel('Module', fontsize=11)
    
    for rna_ct, n_cells, start_pos in transition_info:
        mid_pos = start_pos + n_cells / 2
        ax_dcm.text(mid_pos, -0.8, f'→{rna_ct}\n(n={n_cells})', 
                   ha='center', va='bottom', fontsize=9, fontweight='bold',
                   color=cell_type_colors[rna_ct])
    
    ax_dcm_cbar = fig.add_subplot(gs[0, 1])
    cbar_dcm = fig.colorbar(im_dcm, cax=ax_dcm_cbar)
    cbar_dcm.set_label('Z-Score', fontsize=10)
    
    # RNA Heatmap
    ax_rna = fig.add_subplot(gs[1, 0])
    im_rna = ax_rna.imshow(rna_matrix, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    
    for boundary in transition_boundaries[:-1]:
        ax_rna.axvline(x=boundary - 0.5, color='white', linewidth=2)
    
    ax_rna.set_yticks(range(len(label_order)))
    ax_rna.set_yticklabels(label_order, fontsize=10)
    ax_rna.set_xticks([])
    ax_rna.set_xlabel('Cells (sorted by transition and Z-score)', fontsize=11)
    ax_rna.set_title(f'RNA Module Z-Scores (after transition)', fontsize=12, fontweight='bold')
    ax_rna.set_ylabel('Module', fontsize=11)
    
    ax_rna_cbar = fig.add_subplot(gs[1, 1])
    cbar_rna = fig.colorbar(im_rna, cax=ax_rna_cbar)
    cbar_rna.set_label('Z-Score', fontsize=10)
    
    fig.suptitle(f'Z-Score Module Scores for {dcm_cell_type} DCM Cells → RNA Transitions\n'
                f'(n = {len(all_dcm_data)} cells, Matched using Batch + Plate_Well)\n'
                f'Common Z-score scale: [{vmin}, {vmax}]', 
                fontsize=13, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.90)
    
    save_path = os.path.join(score_dir, f'heatmap_zscore_transitions_from_{dcm_cell_type}.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  Saved: {save_path}")

# -----------------------------
# CREATE HEATMAPS FOR EACH DCM CELL TYPE
# -----------------------------
print("\n" + "="*60)
print("CREATING Z-SCORE HEATMAPS FOR EACH DCM CELL TYPE")
print("="*60)

for dcm_ct in label_order:
    if dcm_ct in connections_df['dcm_cell_type'].unique():
        create_transition_heatmap_zscore(
            connections_df, 
            dcm_ct, 
            label_order,
            cell_type_colors, 
            score_dir,
            vmin, vmax
        )

# -----------------------------
# COMBINED HEATMAP WITH Z-SCORE SCALING
# -----------------------------
print("\n" + "="*60)
print("CREATING COMBINED Z-SCORE HEATMAP")
print("="*60)

def create_combined_heatmap_zscore(connections_df, label_order, cell_type_colors, score_dir, vmin, vmax):
    """
    Create a combined heatmap with Z-score scaling (same scale for DCM and RNA)
    """
    connections_df['dcm_order'] = connections_df['dcm_cell_type'].map({ct: i for i, ct in enumerate(label_order)})
    connections_df['rna_order'] = connections_df['rna_cell_type'].map({ct: i for i, ct in enumerate(label_order)})
    
    sorted_df = connections_df.sort_values(['dcm_order', 'rna_order']).copy()
    
    dcm_data = []
    rna_data = []
    dcm_labels = []
    rna_labels = []
    
    for _, row in sorted_df.iterrows():
        dcm_zscores = [row[f'dcm_{ct}_zscore'] for ct in label_order]
        rna_zscores = [row[f'rna_{ct}_zscore'] for ct in label_order]
        dcm_data.append(dcm_zscores)
        rna_data.append(rna_zscores)
        dcm_labels.append(row['dcm_cell_type'])
        rna_labels.append(row['rna_cell_type'])
    
    dcm_matrix = np.array(dcm_data).T
    rna_matrix = np.array(rna_data).T
    
    fig = plt.figure(figsize=(max(16, len(dcm_data) * 0.08), 14))
    
    gs = fig.add_gridspec(2, 3, width_ratios=[0.03, 1, 0.03], height_ratios=[1, 1],
                         hspace=0.2, wspace=0.02)
    
    cmap = plt.cm.RdBu_r
    
    # DCM section
    ax_dcm_color = fig.add_subplot(gs[0, 0])
    dcm_colors = [cell_type_colors[label] for label in dcm_labels]
    for i, color in enumerate(dcm_colors):
        ax_dcm_color.add_patch(plt.Rectangle((0, len(dcm_colors) - i - 1), 1, 1, 
                                             facecolor=color, edgecolor='none'))
    ax_dcm_color.set_xlim(0, 1)
    ax_dcm_color.set_ylim(0, len(dcm_colors))
    ax_dcm_color.set_xticks([])
    ax_dcm_color.set_yticks([])
    ax_dcm_color.set_ylabel('DCM\nCell Type', fontsize=10, rotation=0, ha='right', va='center')
    
    ax_dcm = fig.add_subplot(gs[0, 1])
    im_dcm = ax_dcm.imshow(dcm_matrix, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    ax_dcm.set_yticks(range(len(label_order)))
    ax_dcm.set_yticklabels(label_order, fontsize=10)
    ax_dcm.set_xticks([])
    ax_dcm.set_title(f'DCM Module Z-Scores', fontsize=12, fontweight='bold')
    
    ax_dcm_cbar = fig.add_subplot(gs[0, 2])
    cbar_dcm = fig.colorbar(im_dcm, cax=ax_dcm_cbar)
    cbar_dcm.set_label('Z-Score', fontsize=10)
    
    # RNA section
    ax_rna_color = fig.add_subplot(gs[1, 0])
    rna_colors = [cell_type_colors[label] for label in rna_labels]
    for i, color in enumerate(rna_colors):
        ax_rna_color.add_patch(plt.Rectangle((0, len(rna_colors) - i - 1), 1, 1, 
                                             facecolor=color, edgecolor='none'))
    ax_rna_color.set_xlim(0, 1)
    ax_rna_color.set_ylim(0, len(rna_colors))
    ax_rna_color.set_xticks([])
    ax_rna_color.set_yticks([])
    ax_rna_color.set_ylabel('RNA\nCell Type', fontsize=10, rotation=0, ha='right', va='center')
    
    ax_rna = fig.add_subplot(gs[1, 1])
    im_rna = ax_rna.imshow(rna_matrix, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    ax_rna.set_yticks(range(len(label_order)))
    ax_rna.set_yticklabels(label_order, fontsize=10)
    ax_rna.set_xticks([])
    ax_rna.set_xlabel('Cells (sorted by transition)', fontsize=11)
    ax_rna.set_title(f'RNA Module Z-Scores', fontsize=12, fontweight='bold')
    
    ax_rna_cbar = fig.add_subplot(gs[1, 2])
    cbar_rna = fig.colorbar(im_rna, cax=ax_rna_cbar)
    cbar_rna.set_label('Z-Score', fontsize=10)
    
    legend_elements = [Patch(facecolor=cell_type_colors[ct], edgecolor='black', label=ct) 
                      for ct in label_order]
    fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(0.99, 0.99),
              fontsize=9, title='Cell Type', title_fontsize=10)
    
    fig.suptitle(f'All DCM → RNA Transitions (n = {len(dcm_data)} cells)\n'
                f'Z-Score Scaling [{vmin}, {vmax}] (Matched using Batch + Plate_Well)', 
                fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    
    save_path = os.path.join(score_dir, 'heatmap_zscore_all_transitions.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved: {save_path}")

create_combined_heatmap_zscore(connections_df, label_order, cell_type_colors, score_dir, vmin, vmax)

# -----------------------------
# DETAILED HEATMAP PER TRANSITION WITH Z-SCORE
# -----------------------------
print("\n" + "="*60)
print("CREATING DETAILED Z-SCORE HEATMAPS PER SPECIFIC TRANSITION")
print("="*60)

def create_single_transition_heatmap_zscore(connections_df, dcm_ct, rna_ct, label_order, 
                                            cell_type_colors, score_dir, vmin, vmax):
    """
    Create detailed heatmap for a specific transition with Z-score scaling
    """
    subset = connections_df[(connections_df['dcm_cell_type'] == dcm_ct) & 
                           (connections_df['rna_cell_type'] == rna_ct)].copy()
    
    if len(subset) == 0:
        return
    
    print(f"\n  {dcm_ct} -> {rna_ct}: {len(subset)} cells")
    
    dcm_zscore_col = f'dcm_{dcm_ct}_zscore'
    subset = subset.sort_values(dcm_zscore_col, ascending=False)
    
    dcm_data = []
    rna_data = []
    
    for _, row in subset.iterrows():
        dcm_zscores = [row[f'dcm_{ct}_zscore'] for ct in label_order]
        rna_zscores = [row[f'rna_{ct}_zscore'] for ct in label_order]
        dcm_data.append(dcm_zscores)
        rna_data.append(rna_zscores)
    
    dcm_matrix = np.array(dcm_data).T
    rna_matrix = np.array(rna_data).T
    
    fig = plt.figure(figsize=(max(12, len(dcm_data) * 0.4), 7))
    
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.05], wspace=0.15)
    
    cmap = plt.cm.RdBu_r
    
    ax_dcm = fig.add_subplot(gs[0, 0])
    im_dcm = ax_dcm.imshow(dcm_matrix, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    
    dcm_idx = label_order.index(dcm_ct)
    ax_dcm.axhline(y=dcm_idx - 0.5, color='black', linewidth=2, linestyle='--')
    ax_dcm.axhline(y=dcm_idx + 0.5, color='black', linewidth=2, linestyle='--')
    
    ax_dcm.set_yticks(range(len(label_order)))
    ax_dcm.set_yticklabels(label_order, fontsize=10)
    ax_dcm.set_xticks([])
    ax_dcm.set_xlabel('Cells', fontsize=10)
    ax_dcm.set_title(f'DCM ({dcm_ct})', fontsize=11, fontweight='bold', color=cell_type_colors[dcm_ct])
    
    ax_rna = fig.add_subplot(gs[0, 1])
    im_rna = ax_rna.imshow(rna_matrix, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    
    rna_idx = label_order.index(rna_ct)
    ax_rna.axhline(y=rna_idx - 0.5, color='black', linewidth=2, linestyle='--')
    ax_rna.axhline(y=rna_idx + 0.5, color='black', linewidth=2, linestyle='--')
    
    ax_rna.set_yticks(range(len(label_order)))
    ax_rna.set_yticklabels(label_order, fontsize=10)
    ax_rna.set_xticks([])
    ax_rna.set_xlabel('Cells', fontsize=10)
    ax_rna.set_title(f'RNA ({rna_ct})', fontsize=11, fontweight='bold', color=cell_type_colors[rna_ct])
    
    # Single shared colorbar
    ax_cbar = fig.add_subplot(gs[0, 2])
    cbar = fig.colorbar(im_rna, cax=ax_cbar)
    cbar.set_label('Z-Score', fontsize=10)
    
    fig.suptitle(f'Transition: {dcm_ct} → {rna_ct} (n = {len(subset)} cells)\n'
                f'Z-Score Scale: [{vmin}, {vmax}] (Matched using Batch + Plate_Well)', 
                fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    
    save_path = os.path.join(score_dir, f'heatmap_zscore_transition_{dcm_ct}_to_{rna_ct}.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

unique_transitions = connections_df['transition'].unique()
for trans in unique_transitions:
    dcm_ct, rna_ct = trans.split(' -> ')
    if dcm_ct in label_order and rna_ct in label_order:
        create_single_transition_heatmap_zscore(
            connections_df, dcm_ct, rna_ct, 
            label_order, cell_type_colors, score_dir,
            vmin, vmax
        )

# -----------------------------
# PRINT Z-SCORE SUMMARY
# -----------------------------
print("\n" + "="*60)
print("Z-SCORE SUMMARY")
print("="*60)

print(f"\nDCM Z-Scores:")
print(f"  Min: {dcm_z_values.min():.2f}")
print(f"  Max: {dcm_z_values.max():.2f}")
print(f"  Mean: {dcm_z_values.mean():.4f}")
print(f"  Std: {dcm_z_values.std():.4f}")
print(f"  % within [{vmin}, {vmax}]: {((dcm_z_values >= vmin) & (dcm_z_values <= vmax)).mean() * 100:.1f}%")

print(f"\nRNA Z-Scores:")
print(f"  Min: {rna_z_values.min():.2f}")
print(f"  Max: {rna_z_values.max():.2f}")
print(f"  Mean: {rna_z_values.mean():.4f}")
print(f"  Std: {rna_z_values.std():.4f}")
print(f"  % within [{vmin}, {vmax}]: {((rna_z_values >= vmin) & (rna_z_values <= vmax)).mean() * 100:.1f}%")

# -----------------------------
# SAVE DATA
# -----------------------------
print("\n" + "="*60)
print("SAVING DATA")
print("="*60)

connections_df.to_csv(os.path.join(score_dir, 'transitions_with_zscores_batch_plate_well.csv'), index=False)
print(f"Saved: {os.path.join(score_dir, 'transitions_with_zscores_batch_plate_well.csv')}")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
## Summary of what this code creates:

### 1. Per DCM Cell Type Heatmaps
`heatmap_transitions_from_{CellType}.pdf`

Shows all transitions starting from a specific cell type (e.g., all Stem → X transitions):
- **Top panel**: DCM module scores
- **Bottom panel**: RNA module scores
- Cells grouped by RNA destination
- Sorted by module score within each group
- Red dashed lines highlight the relevant cell type row

### 2. Combined Heatmap
`heatmap_all_transitions_combined.pdf`

Shows all transitions in one plot with:
- Color bars indicating DCM and RNA cell types
- All cells sorted by transition type

### 3. Individual Transition Heatmaps
`heatmap_transition_{DCM}_to_{RNA}.pdf`

Detailed view of each specific transition (e.g., Stem → Enterocyte):
- Side-by-side DCM and RNA heatmaps
- Cells sorted by DCM module score
- Red dashed lines highlight source and destination cell types

### Visual Layout:
```
Per DCM Cell Type (e.g., Stem):
┌─────────────────────────────────────────────────┐
│  DCM Module Scores                              │
│  ┌──────┬──────┬──────┬──────┬──────┐          │
│  │→Stem │→TA   │→Ent  │→Pan  │→Gob  │ ...      │
│  ├──────┼──────┼──────┼──────┼──────┤          │
│Stem ████│██    │█     │      │      │ ← highlighted
│TA   ██  │████  │██    │      │      │          │
│Ent  █   │██    │████  │█     │      │          │
│...      │      │      │      │      │          │
│  └──────┴──────┴──────┴──────┴──────┘          │
├─────────────────────────────────────────────────┤
│  RNA Module Scores                              │
│  (same layout, showing destination scores)      │
└─────────────────────────────────────────────────┘

In [ ]:
#own Zscores

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap, TwoSlopeNorm
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/gmm_085/'
os.makedirs(score_dir, exist_ok=True)

# Create subdirectory for individual transition heatmaps
transition_heatmap_dir = os.path.join(score_dir, 'transition_heatmaps_2')
os.makedirs(transition_heatmap_dir, exist_ok=True)

# Extract columns that contain ttest z-scores, EXCLUDING 'max' columns
ttest_cols = [col for col in adata_combined.obs.columns 
              if 'ttest_' in col and '_zscore' in col and 'max' not in col.lower()]
print(f"Found {len(ttest_cols)} ttest z-score columns: {ttest_cols}")

# Define cell type order
cell_type_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# Cell type colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1'
}

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("\n" + "="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata_combined.obs['batch_plate_well'] = (
    adata_combined.obs['Batch'].astype(str) + '_' + 
    adata_combined.obs['plate_well'].astype(str)
)

# -----------------------------
# PREPARE DATA
# -----------------------------
print("\n" + "="*60)
print("PREPARING DATA FOR TRANSITIONS")
print("="*60)

plot_df = pd.DataFrame({
    'assay': adata_combined.obs['assay'].values,
    'plate_well': adata_combined.obs['plate_well'].values,
    'batch_plate_well': adata_combined.obs['batch_plate_well'].values,
    'Batch': adata_combined.obs['Batch'].values,
    'cell_type_broad': adata_combined.obs['leiden_annot_25'].astype(str).values,
    'module_max_label': adata_combined.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'Dox': adata_combined.obs['Dox'].astype(str).values if 'Dox' in adata_combined.obs.columns else 'Unknown',
    'cell_index': adata_combined.obs.index.values
})

# Extract cell type from leiden_annot_25
plot_df['cell_type_broad'] = plot_df['cell_type_broad'].str.split('_').str[-1]
plot_df['module_max_label'] = plot_df['module_max_label'].str.split('_').str[-1]

# Add z-score columns
for col in ttest_cols:
    plot_df[col] = adata_combined.obs[col].values

# -----------------------------
# FILTER FOR DCM AND RNA CELLS - ALL CELL TYPES
# -----------------------------
print("\n" + "="*60)
print("FILTERING DCM AND RNA CELLS")
print("="*60)

# DCM cells - all known labels, excluding Minus Dox
dcm_mask = (plot_df['assay'] == 'pDCM') & \
           (plot_df['module_max_label'].isin(cell_type_order)) & \
           (plot_df['Dox'] != 'Minus')
dcm_cells = plot_df[dcm_mask].copy()

# RNA cells - all known labels, excluding Minus Dox
rna_mask = (plot_df['assay'] == 'pRNA') & \
           (plot_df['cell_type_broad'].isin(cell_type_order)) & \
           (plot_df['Dox'] != 'Minus')
rna_cells = plot_df[rna_mask].copy()

print(f"DCM cells (all types, excluding Minus): {len(dcm_cells)}")
print(f"RNA cells (all types, excluding Minus): {len(rna_cells)}")

# -----------------------------
# MATCH BY BATCH_PLATE_WELL (1:1)
# -----------------------------
print("\n" + "="*60)
print("MATCHING CELLS BY BATCH_PLATE_WELL")
print("="*60)

dcm_bpw = set(dcm_cells['batch_plate_well'])
rna_bpw = set(rna_cells['batch_plate_well'])
common_bpw = dcm_bpw.intersection(rna_bpw)

print(f"Common batch_plate_wells: {len(common_bpw)}")

dcm_common = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
rna_common = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]

dcm_counts = dcm_common['batch_plate_well'].value_counts()
rna_counts = rna_common['batch_plate_well'].value_counts()

dcm_single = set(dcm_counts[dcm_counts == 1].index)
rna_single = set(rna_counts[rna_counts == 1].index)
both_single = dcm_single.intersection(rna_single)

print(f"1:1 matched batch_plate_wells: {len(both_single)}")

# Create connections
connections = []
for bpw in both_single:
    dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
    rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
    
    if len(dcm_cell) == 1 and len(rna_cell) == 1:
        dcm_row = dcm_cell.iloc[0]
        rna_row = rna_cell.iloc[0]
        
        conn = {
            'batch_plate_well': bpw,
            'batch': dcm_row['Batch'],
            'plate_well': dcm_row['plate_well'],
            'dcm_cell_type': dcm_row['module_max_label'],
            'rna_cell_type': rna_row['cell_type_broad'],
            'transition': f"{dcm_row['module_max_label']} -> {rna_row['cell_type_broad']}",
            'dcm_index': dcm_row['cell_index'],
            'rna_index': rna_row['cell_index']
        }
        
        # Add z-scores for both DCM and RNA
        for col in ttest_cols:
            ct = col.replace('ttest_', '').replace('_zscore', '')
            conn[f'dcm_{ct}_zscore'] = dcm_row[col]
            conn[f'rna_{ct}_zscore'] = rna_row[col]
        
        connections.append(conn)

connections_df = pd.DataFrame(connections)
print(f"\nTotal transitions found: {len(connections_df)}")
print("\nTransition counts:")
print(connections_df['transition'].value_counts())

# -----------------------------
# HEATMAP FUNCTION FOR EACH INDIVIDUAL TRANSITION PAIR
# -----------------------------
def create_pairwise_transition_heatmap(connections_df, dcm_ct, rna_ct,
                                        cell_type_order, cell_type_colors, save_dir):
    """
    Create heatmap for one specific DCM -> RNA transition pair
    """
    subset = connections_df[
        (connections_df['dcm_cell_type'] == dcm_ct) &
        (connections_df['rna_cell_type'] == rna_ct)
    ].copy()

    if len(subset) == 0:
        print(f"  No cells for {dcm_ct} -> {rna_ct}, skipping")
        return

    print(f"\n  Plotting: {dcm_ct} -> {rna_ct} (n={len(subset)})")

    # Sort by DCM source z-score descending
    dcm_zscore_col = f'dcm_{dcm_ct}_zscore'
    rna_zscore_col = f'rna_{rna_ct}_zscore'

    if dcm_zscore_col in subset.columns:
        subset = subset.sort_values(by=[dcm_zscore_col, rna_zscore_col], ascending=[False, False])

    # Build matrices
    dcm_matrix = np.array([[row[f'dcm_{ct}_zscore'] for ct in cell_type_order]
                            for _, row in subset.iterrows()]).T
    rna_matrix = np.array([[row[f'rna_{ct}_zscore'] for ct in cell_type_order]
                            for _, row in subset.iterrows()]).T

    # Color scale centered at 1.65
    vcenter = 1
    vmin = vcenter - 3
    vmax = vcenter + 3
    norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
    cmap = plt.cm.RdBu_r

    # Highlight rows
    dcm_highlight_idx = cell_type_order.index(dcm_ct)
    rna_highlight_idx = cell_type_order.index(rna_ct)

    fig = plt.figure(figsize=(max(8, len(subset) * 0.2), 10))
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 0.03], height_ratios=[1, 1],
                          hspace=0.3, wspace=0.05)

    # --- DCM heatmap ---
    ax_dcm = fig.add_subplot(gs[0, 0])
    im_dcm = ax_dcm.imshow(dcm_matrix, aspect='auto', cmap=cmap, norm=norm)

    # Highlight DCM source row
    ax_dcm.axhline(y=dcm_highlight_idx - 0.5, color='black', linewidth=2, linestyle='--', alpha=0.8)
    ax_dcm.axhline(y=dcm_highlight_idx + 0.5, color='black', linewidth=2, linestyle='--', alpha=0.8)

    ax_dcm.set_yticks(range(len(cell_type_order)))
    ax_dcm.set_yticklabels(cell_type_order, fontsize=10)
    ax_dcm.set_xticks([])
    ax_dcm.set_title(f'DCM Module Z-Scores\n(source: {dcm_ct}, n={len(subset)})',
                     fontsize=11, fontweight='bold', color=cell_type_colors[dcm_ct])
    ax_dcm.set_ylabel('Module', fontsize=11, fontweight='bold')

    ax_dcm_cbar = fig.add_subplot(gs[0, 1])
    cbar_dcm = fig.colorbar(im_dcm, cax=ax_dcm_cbar)
    cbar_dcm.set_label('Z-Score', fontsize=9)
    cbar_dcm.ax.axhline(y=vcenter, color='black', linewidth=1, linestyle='--')

    # --- RNA heatmap ---
    ax_rna = fig.add_subplot(gs[1, 0])
    im_rna = ax_rna.imshow(rna_matrix, aspect='auto', cmap=cmap, norm=norm)

    # Highlight RNA destination row
    ax_rna.axhline(y=rna_highlight_idx - 0.5, color='black', linewidth=2, linestyle='--', alpha=0.8)
    ax_rna.axhline(y=rna_highlight_idx + 0.5, color='black', linewidth=2, linestyle='--', alpha=0.8)

    ax_rna.set_yticks(range(len(cell_type_order)))
    ax_rna.set_yticklabels(cell_type_order, fontsize=10)
    ax_rna.set_xticks([])
    ax_rna.set_xlabel('Cells (sorted by DCM z-score)', fontsize=11, fontweight='bold')
    ax_rna.set_title(f'RNA Module Z-Scores\n(destination: {rna_ct})',
                     fontsize=11, fontweight='bold', color=cell_type_colors[rna_ct])
    ax_rna.set_ylabel('Module', fontsize=11, fontweight='bold')

    ax_rna_cbar = fig.add_subplot(gs[1, 1])
    cbar_rna = fig.colorbar(im_rna, cax=ax_rna_cbar)
    cbar_rna.set_label('Z-Score', fontsize=9)
    cbar_rna.ax.axhline(y=vcenter, color='black', linewidth=1, linestyle='--')

    is_stable = dcm_ct == rna_ct
    stable_label = "STABLE" if is_stable else "TRANSITION"

    fig.suptitle(f'{stable_label}: {dcm_ct} → {rna_ct}\n'
                 f'(n = {len(subset)} cells | Z-score center: {vcenter}, scale: [{vmin:.2f}, {vmax:.2f}])',
                 fontsize=13, fontweight='bold', y=1.01)

    plt.tight_layout()

    save_path = os.path.join(save_dir, f'heatmap_{dcm_ct}_to_{rna_ct}.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  ✅ Saved: {save_path}")


# -----------------------------
# RUN FOR ALL TRANSITION PAIRS
# -----------------------------
if len(connections_df) == 0:
    print("\nNo transitions found!")
else:
    print("\n" + "="*60)
    print("CREATING HEATMAPS FOR ALL TRANSITION PAIRS")
    print("="*60)

    # Get all unique transition pairs that actually exist in the data
    existing_transitions = connections_df[['dcm_cell_type', 'rna_cell_type']].drop_duplicates()
    print(f"Found {len(existing_transitions)} unique transition pairs")

    for _, row in existing_transitions.iterrows():
        dcm_ct = row['dcm_cell_type']
        rna_ct = row['rna_cell_type']
        create_pairwise_transition_heatmap(
            connections_df, dcm_ct, rna_ct,
            cell_type_order, cell_type_colors,
            transition_heatmap_dir
        )

    # -----------------------------
    # SUMMARY HEATMAP (ALL TRANSITIONS)
    # -----------------------------
    print("\n" + "="*60)
    print("CREATING SUMMARY HEATMAP")
    print("="*60)

    transition_matrix = pd.crosstab(
        connections_df['dcm_cell_type'],
        connections_df['rna_cell_type']
    )

    transition_matrix = transition_matrix.reindex(
        index=[ct for ct in cell_type_order if ct in transition_matrix.index],
        columns=[ct for ct in cell_type_order if ct in transition_matrix.columns]
    ).fillna(0).astype(int)

    print("\nFull Transition Matrix:")
    print(transition_matrix)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    sns.heatmap(transition_matrix, annot=True, fmt='d', cmap='viridis',
                cbar_kws={'label': 'Number of cells'}, ax=axes[0],
                linewidths=0.5, linecolor='white')
    axes[0].set_title('All Transitions: DCM → RNA (Counts)\n(Matched using Batch + Plate_Well)',
                      fontsize=12, fontweight='bold')
    axes[0].set_xlabel('RNA Cell Type', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('DCM Cell Type', fontsize=11, fontweight='bold')

    transition_pct = transition_matrix.div(transition_matrix.sum(axis=1), axis=0) * 100
    sns.heatmap(transition_pct, annot=True, fmt='.1f', cmap='viridis',
                cbar_kws={'label': 'Percentage (%)'}, ax=axes[1],
                linewidths=0.5, linecolor='white')
    axes[1].set_title('All Transitions: DCM → RNA (% of DCM type)\n(Matched using Batch + Plate_Well)',
                      fontsize=12, fontweight='bold')
    axes[1].set_xlabel('RNA Cell Type', fontsize=11, fontweight='bold')
    axes[1].set_ylabel('DCM Cell Type', fontsize=11, fontweight='bold')

    plt.tight_layout()
    save_path = os.path.join(score_dir, 'heatmap_summary_all_transitions.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"✅ Saved: {save_path}")

    # -----------------------------
    # SAVE DATA
    # -----------------------------
    connections_df.to_csv(os.path.join(score_dir, 'all_transitions_with_zscores.csv'), index=False)
    transition_matrix.to_csv(os.path.join(score_dir, 'transition_matrix_counts.csv'))
    transition_pct.to_csv(os.path.join(score_dir, 'transition_matrix_percentages.csv'))
    print(f"✅ Saved CSVs to: {score_dir}")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
#Going back to Stem or TA

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/gmm_085/'
os.makedirs(score_dir, exist_ok=True)

adata = adata_combined

# Cell types and colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': '#7f7f7f'
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata.obs['batch_plate_well'] = (
    adata.obs['Batch'].astype(str) + '_' + 
    adata.obs['plate_well'].astype(str)
)

# -----------------------------
# IDENTIFY MODULE SCORE COLUMNS
# -----------------------------
print("\n" + "="*60)
print("STEP 1: IDENTIFYING MODULE SCORE COLUMNS")
print("="*60)

module_cols = []
for ct in label_order:
    col_name = f'ttest_{ct}_module'
    if col_name in adata.obs.columns:
        module_cols.append(col_name)
        print(f"  Found: {col_name}")
    else:
        alt_names = [f'ttest_{ct}_zscore', f'ttest_{ct}_score']
        for alt in alt_names:
            if alt in adata.obs.columns:
                module_cols.append(alt)
                print(f"  Found (alt): {alt}")
                break

print(f"\nTotal module columns found: {len(module_cols)}")

# -----------------------------
# PREPARE DATA
# -----------------------------
print("\n" + "="*60)
print("STEP 2: PREPARING DATA")
print("="*60)

umap_coords = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1': umap_coords[:, 0],
    'UMAP2': umap_coords[:, 1],
    'assay': adata.obs['assay'].values,
    'plate_well': adata.obs['plate_well'].values,
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'Batch': adata.obs['Batch'].values,
    'cell_type_broad': adata.obs['leiden_annot_25'].astype(str).values,
    'module_max_label': adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'cell_index': adata.obs.index.values
})

if 'Dox' in adata.obs.columns:
    plot_df['Dox'] = adata.obs['Dox'].values

# Add module scores
for col in module_cols:
    plot_df[col] = adata.obs[col].values

# -----------------------------
# FILTER AND MATCH CELLS (EXCLUDE MINUS DOX)
# -----------------------------
print("\n" + "="*60)
print("STEP 3: FILTERING AND MATCHING CELLS")
print("="*60)

dcm_mask = plot_df['assay'] == 'pDCM'
dcm_known_mask = dcm_mask & plot_df['module_max_label'].isin(label_order)

if 'Dox' in plot_df.columns:
    dcm_known_mask = dcm_known_mask & (plot_df['Dox'] != 'Minus')

dcm_cells = plot_df[dcm_known_mask].copy()

rna_mask = plot_df['assay'] == 'pRNA'
rna_known_mask = rna_mask & plot_df['cell_type_broad'].isin(label_order)

if 'Dox' in plot_df.columns:
    rna_known_mask = rna_known_mask & (plot_df['Dox'] != 'Minus')

rna_cells = plot_df[rna_known_mask].copy()

print(f"DCM cells with known labels (excluding Minus Dox): {len(dcm_cells)}")
print(f"RNA cells with known labels (excluding Minus Dox): {len(rna_cells)}")

# -----------------------------
# CHECK DUPLICATES WITH BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("CHECKING DUPLICATES")
print("="*60)

dcm_bpw_counts = dcm_cells['batch_plate_well'].value_counts()
rna_bpw_counts = rna_cells['batch_plate_well'].value_counts()

print(f"DCM batch_plate_wells with >1 cell: {(dcm_bpw_counts > 1).sum()}")
print(f"RNA batch_plate_wells with >1 cell: {(rna_bpw_counts > 1).sum()}")

# -----------------------------
# MATCH BY BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("FINDING CONNECTIONS USING BATCH_PLATE_WELL")
print("="*60)

dcm_bpw = set(dcm_cells['batch_plate_well'])
rna_bpw = set(rna_cells['batch_plate_well'])
common_bpw = dcm_bpw.intersection(rna_bpw)
print(f"Common batch_plate_wells: {len(common_bpw)}")

dcm_common = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
rna_common = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]

dcm_counts = dcm_common['batch_plate_well'].value_counts()
rna_counts = rna_common['batch_plate_well'].value_counts()

dcm_single = set(dcm_counts[dcm_counts == 1].index)
rna_single = set(rna_counts[rna_counts == 1].index)
both_single = dcm_single.intersection(rna_single)

print(f"1:1 matched batch_plate_wells: {len(both_single)}")

# Create connections with module scores
connections = []
for bpw in both_single:
    dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
    rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
    
    if len(dcm_cell) == 1 and len(rna_cell) == 1:
        dcm_row = dcm_cell.iloc[0]
        rna_row = rna_cell.iloc[0]
        
        conn = {
            'batch_plate_well': bpw,
            'batch': dcm_row['Batch'],
            'plate_well': dcm_row['plate_well'],
            'dcm_cell_type': dcm_row['module_max_label'],
            'rna_cell_type': rna_row['cell_type_broad'],
            'transition': f"{dcm_row['module_max_label']} -> {rna_row['cell_type_broad']}",
            'dcm_index': dcm_row['cell_index'],
            'rna_index': rna_row['cell_index']
        }
        
        for col in module_cols:
            ct = col.replace('ttest_', '').replace('_module', '').replace('_zscore', '').replace('_score', '')
            conn[f'dcm_{ct}_score'] = dcm_row[col]
        
        for col in module_cols:
            ct = col.replace('ttest_', '').replace('_module', '').replace('_zscore', '').replace('_score', '')
            conn[f'rna_{ct}_score'] = rna_row[col]
        
        connections.append(conn)

connections_df = pd.DataFrame(connections)
print(f"Total valid connections: {len(connections_df)}")

# -----------------------------
# FILTER FOR TRANSITIONS TO STEM OR TA ONLY
# -----------------------------
print("\n" + "="*60)
print("FILTERING FOR TRANSITIONS TO STEM OR TA")
print("="*60)

# Keep only transitions TO Stem or TA, but NOT FROM Stem or TA
connections_df = connections_df[
    (connections_df['rna_cell_type'].isin(['Stem', 'TA'])) &
    (~connections_df['dcm_cell_type'].isin(['Stem', 'TA']))
].copy()

print(f"Connections to Stem or TA (from other cell types): {len(connections_df)}")
print("\nTransition counts:")
print(connections_df['transition'].value_counts())

if len(connections_df) == 0:
    print("\nNo transitions to Stem or TA found from other cell types!")
    print("="*60)
    print("DONE!")
    print("="*60)
else:
    # -----------------------------
    # Z-SCORE TRANSFORMATION
    # -----------------------------
    print("\n" + "="*60)
    print("STEP 4: Z-SCORE TRANSFORMATION")
    print("="*60)
    
    dcm_score_cols = [col for col in connections_df.columns if col.startswith('dcm_') and col.endswith('_score')]
    rna_score_cols = [col for col in connections_df.columns if col.startswith('rna_') and col.endswith('_score')]
    
    # Get raw values
    dcm_all_values = connections_df[dcm_score_cols].values.flatten()
    dcm_all_values = dcm_all_values[~np.isnan(dcm_all_values)]
    
    rna_all_values = connections_df[rna_score_cols].values.flatten()
    rna_all_values = rna_all_values[~np.isnan(rna_all_values)]
    
    print(f"\nRaw DCM scores: mean={dcm_all_values.mean():.2f}, std={dcm_all_values.std():.2f}")
    print(f"Raw RNA scores: mean={rna_all_values.mean():.2f}, std={rna_all_values.std():.2f}")
    
    # Calculate Z-score parameters for DCM and RNA separately
    dcm_mean = dcm_all_values.mean()
    dcm_std = dcm_all_values.std()
    
    rna_mean = rna_all_values.mean()
    rna_std = rna_all_values.std()
    
    # Create Z-scored columns
    for col in dcm_score_cols:
        zscore_col = col.replace('_score', '_zscore')
        connections_df[zscore_col] = (connections_df[col] - dcm_mean) / dcm_std
    
    for col in rna_score_cols:
        zscore_col = col.replace('_score', '_zscore')
        connections_df[zscore_col] = (connections_df[col] - rna_mean) / rna_std
    
    # Get Z-scored column names
    dcm_zscore_cols = [col for col in connections_df.columns if col.startswith('dcm_') and col.endswith('_zscore')]
    rna_zscore_cols = [col for col in connections_df.columns if col.startswith('rna_') and col.endswith('_zscore')]
    
    # Verify Z-score transformation
    dcm_z_values = connections_df[dcm_zscore_cols].values.flatten()
    dcm_z_values = dcm_z_values[~np.isnan(dcm_z_values)]
    
    rna_z_values = connections_df[rna_zscore_cols].values.flatten()
    rna_z_values = rna_z_values[~np.isnan(rna_z_values)]
    
    print(f"\nZ-scored DCM: mean={dcm_z_values.mean():.4f}, std={dcm_z_values.std():.4f}")
    print(f"\nZ-scored RNA: mean={rna_z_values.mean():.4f}, std={rna_z_values.std():.4f}")
    
    # Set common Z-score scale (symmetric around 0)
    z_limit = 3  # Standard: +/- 3 SD covers 99.7% of data
    vmin = -z_limit
    vmax = z_limit
    
    print(f"\nZ-score scale: [{vmin}, {vmax}]")
    
    # -----------------------------
    # CREATE HEATMAPS FOR TRANSITIONS TO STEM AND TA
    # -----------------------------
    print("\n" + "="*60)
    print("CREATING Z-SCORE HEATMAPS FOR TRANSITIONS TO STEM/TA")
    print("="*60)
    
    def create_transition_to_target_heatmap(connections_df, target_ct, label_order, 
                                           cell_type_colors, score_dir, vmin, vmax):
        """
        Create heatmap for all transitions TO a specific target (Stem or TA)
        FROM other cell types (excluding Stem and TA)
        """
        print(f"\n{'='*60}")
        print(f"CREATING Z-SCORE HEATMAP FOR TRANSITIONS TO {target_ct.upper()}")
        print(f"{'='*60}")
        
        subset = connections_df[connections_df['rna_cell_type'] == target_ct].copy()
        
        if len(subset) == 0:
            print(f"  No transitions found to {target_ct}")
            return
        
        print(f"  Total cells: {len(subset)}")
        print(f"  Source cell types: {subset['dcm_cell_type'].value_counts().to_dict()}")
        
        all_dcm_data = []
        all_rna_data = []
        transition_boundaries = []
        transition_info = []
        
        current_pos = 0
        
        # Sort by source cell type
        source_order = [ct for ct in label_order if ct not in ['Stem', 'TA'] and ct in subset['dcm_cell_type'].unique()]
        
        for dcm_ct in source_order:
            trans_subset = subset[subset['dcm_cell_type'] == dcm_ct]
            
            if len(trans_subset) == 0:
                continue
            
            dcm_zscore_col = f'dcm_{dcm_ct}_zscore'
            rna_zscore_col = f'rna_{target_ct}_zscore'
            
            trans_subset = trans_subset.sort_values(
                by=[dcm_zscore_col, rna_zscore_col], 
                ascending=[False, False]
            )
            
            for _, row in trans_subset.iterrows():
                dcm_zscores = [row[f'dcm_{ct}_zscore'] for ct in label_order]
                all_dcm_data.append(dcm_zscores)
            
            for _, row in trans_subset.iterrows():
                rna_zscores = [row[f'rna_{ct}_zscore'] for ct in label_order]
                all_rna_data.append(rna_zscores)
            
            transition_info.append((dcm_ct, len(trans_subset), current_pos))
            current_pos += len(trans_subset)
            transition_boundaries.append(current_pos)
        
        if len(all_dcm_data) == 0:
            print(f"  No data to plot")
            return
        
        dcm_matrix = np.array(all_dcm_data).T
        rna_matrix = np.array(all_rna_data).T
        
        print(f"  DCM matrix shape: {dcm_matrix.shape}")
        print(f"  RNA matrix shape: {rna_matrix.shape}")
        
        fig = plt.figure(figsize=(max(14, len(all_dcm_data) * 0.15), 12))
        
        gs = fig.add_gridspec(2, 2, width_ratios=[1, 0.03], height_ratios=[1, 1],
                             hspace=0.25, wspace=0.05)
        
        cmap = plt.cm.RdBu_r
        
        # DCM Heatmap
        ax_dcm = fig.add_subplot(gs[0, 0])
        im_dcm = ax_dcm.imshow(dcm_matrix, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
        
        for boundary in transition_boundaries[:-1]:
            ax_dcm.axvline(x=boundary - 0.5, color='white', linewidth=2)
        
        ax_dcm.set_yticks(range(len(label_order)))
        ax_dcm.set_yticklabels(label_order, fontsize=10)
        ax_dcm.set_xticks([])
        ax_dcm.set_title(f'DCM Module Z-Scores (cells transitioning to {target_ct})', fontsize=12, fontweight='bold')
        ax_dcm.set_ylabel('Module', fontsize=11)
        
        for dcm_ct, n_cells, start_pos in transition_info:
            mid_pos = start_pos + n_cells / 2
            ax_dcm.text(mid_pos, -0.8, f'{dcm_ct}\n(n={n_cells})', 
                       ha='center', va='bottom', fontsize=9, fontweight='bold',
                       color=cell_type_colors[dcm_ct])
        
        ax_dcm_cbar = fig.add_subplot(gs[0, 1])
        cbar_dcm = fig.colorbar(im_dcm, cax=ax_dcm_cbar)
        cbar_dcm.set_label('Z-Score', fontsize=10)
        
        # RNA Heatmap
        ax_rna = fig.add_subplot(gs[1, 0])
        im_rna = ax_rna.imshow(rna_matrix, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
        
        for boundary in transition_boundaries[:-1]:
            ax_rna.axvline(x=boundary - 0.5, color='white', linewidth=2)
        
        # Highlight target row
        target_idx = label_order.index(target_ct)
        ax_rna.axhline(y=target_idx - 0.5, color='black', linewidth=2, linestyle='--', alpha=0.8)
        ax_rna.axhline(y=target_idx + 0.5, color='black', linewidth=2, linestyle='--', alpha=0.8)
        
        ax_rna.set_yticks(range(len(label_order)))
        ax_rna.set_yticklabels(label_order, fontsize=10)
        ax_rna.set_xticks([])
        ax_rna.set_xlabel('Cells (sorted by source cell type and Z-score)', fontsize=11)
        ax_rna.set_title(f'RNA Module Z-Scores (after transition to {target_ct})', fontsize=12, fontweight='bold')
        ax_rna.set_ylabel('Module', fontsize=11)
        
        ax_rna_cbar = fig.add_subplot(gs[1, 1])
        cbar_rna = fig.colorbar(im_rna, cax=ax_rna_cbar)
        cbar_rna.set_label('Z-Score', fontsize=10)
        
        fig.suptitle(f'Z-Score Module Scores for Transitions to {target_ct}\n'
                    f'(n = {len(all_dcm_data)} cells, Matched using Batch + Plate_Well)\n'
                    f'Common Z-score scale: [{vmin}, {vmax}]', 
                    fontsize=13, fontweight='bold', y=1.02)
        
        plt.tight_layout()
        plt.subplots_adjust(top=0.90)
        
        save_path = os.path.join(score_dir, f'heatmap_zscore_transitions_to_{target_ct}.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f"  Saved: {save_path}")
    
    # Create heatmaps for Stem and TA
    for target in ['Stem', 'TA']:
        if target in connections_df['rna_cell_type'].unique():
            create_transition_to_target_heatmap(
                connections_df, 
                target, 
                label_order,
                cell_type_colors, 
                score_dir,
                vmin, vmax
            )
    
    # -----------------------------
    # SUMMARY HEATMAP
    # -----------------------------
    print("\n" + "="*60)
    print("CREATING SUMMARY HEATMAP")
    print("="*60)
    
    # Create transition matrix
    transition_matrix = pd.crosstab(
        connections_df['dcm_cell_type'],
        connections_df['rna_cell_type'],
        rownames=['DCM Cell Type'],
        colnames=['RNA Cell Type']
    )
    
    # Reorder - exclude Stem and TA from DCM rows
    dcm_order = [ct for ct in label_order if ct not in ['Stem', 'TA'] and ct in transition_matrix.index]
    rna_order = ['Stem', 'TA']
    
    transition_matrix = transition_matrix.reindex(
        index=dcm_order,
        columns=[ct for ct in rna_order if ct in transition_matrix.columns]
    ).fillna(0).astype(int)
    
    print("\nTransition Matrix (to Stem/TA):")
    print(transition_matrix)
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(6, 8))
    
    sns.heatmap(transition_matrix, annot=True, fmt='d', cmap='YlOrRd',
                cbar_kws={'label': 'Number of cells'}, ax=ax,
                linewidths=0.5, linecolor='white')
    
    ax.set_title('Transitions to Stem/TA from Other Cell Types\n(Matched using Batch + Plate_Well)', 
                fontsize=13, fontweight='bold')
    ax.set_xlabel('RNA Cell Type', fontsize=11, fontweight='bold')
    ax.set_ylabel('DCM Cell Type', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    save_path = os.path.join(score_dir, 'heatmap_summary_to_stem_ta.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved: {save_path}")
    
    # -----------------------------
    # PRINT Z-SCORE SUMMARY
    # -----------------------------
    print("\n" + "="*60)
    print("Z-SCORE SUMMARY")
    print("="*60)
    
    print(f"\nDCM Z-Scores:")
    print(f"  Min: {dcm_z_values.min():.2f}")
    print(f"  Max: {dcm_z_values.max():.2f}")
    print(f"  Mean: {dcm_z_values.mean():.4f}")
    print(f"  Std: {dcm_z_values.std():.4f}")
    print(f"  % within [{vmin}, {vmax}]: {((dcm_z_values >= vmin) & (dcm_z_values <= vmax)).mean() * 100:.1f}%")
    
    print(f"\nRNA Z-Scores:")
    print(f"  Min: {rna_z_values.min():.2f}")
    print(f"  Max: {rna_z_values.max():.2f}")
    print(f"  Mean: {rna_z_values.mean():.4f}")
    print(f"  Std: {rna_z_values.std():.4f}")
    print(f"  % within [{vmin}, {vmax}]: {((rna_z_values >= vmin) & (rna_z_values <= vmax)).mean() * 100:.1f}%")
    
    # -----------------------------
    # SAVE DATA
    # -----------------------------
    print("\n" + "="*60)
    print("SAVING DATA")
    print("="*60)
    
    connections_df.to_csv(os.path.join(score_dir, 'transitions_to_stem_ta_with_zscores.csv'), index=False)
    print(f"Saved: {os.path.join(score_dir, 'transitions_to_stem_ta_with_zscores.csv')}")
    
    print("\n" + "="*60)
    print("DONE!")
    print("="*60)

In [ ]:
#own Zscores

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/gmm_085/backtoStem'
os.makedirs(score_dir, exist_ok=True)

# Extract columns that contain ttest z-scores, EXCLUDING 'max' columns
ttest_cols = [col for col in adata_combined.obs.columns 
              if 'ttest_' in col and '_zscore' in col and 'max' not in col.lower()]
print(f"Found {len(ttest_cols)} ttest z-score columns: {ttest_cols}")

# Define cell type order
cell_type_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# Cell type colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1'
}

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("\n" + "="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata_combined.obs['batch_plate_well'] = (
    adata_combined.obs['Batch'].astype(str) + '_' + 
    adata_combined.obs['plate_well'].astype(str)
)

# -----------------------------
# PREPARE DATA
# -----------------------------
print("\n" + "="*60)
print("PREPARING DATA FOR TRANSITIONS")
print("="*60)

plot_df = pd.DataFrame({
    'assay': adata_combined.obs['assay'].values,
    'plate_well': adata_combined.obs['plate_well'].values,
    'batch_plate_well': adata_combined.obs['batch_plate_well'].values,
    'Batch': adata_combined.obs['Batch'].values,
    'cell_type_broad': adata_combined.obs['leiden_annot_25'].astype(str).values,
    'module_max_label': adata_combined.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'Dox': adata_combined.obs['Dox'].astype(str).values if 'Dox' in adata_combined.obs.columns else 'Unknown',
    'cell_index': adata_combined.obs.index.values
})

# Extract cell type from leiden_annot_25
plot_df['cell_type_broad'] = plot_df['cell_type_broad'].str.split('_').str[-1]
plot_df['module_max_label'] = plot_df['module_max_label'].str.split('_').str[-1]

# Add z-score columns
for col in ttest_cols:
    plot_df[col] = adata_combined.obs[col].values

# -----------------------------
# FILTER FOR DCM AND RNA CELLS
# -----------------------------
print("\n" + "="*60)
print("FILTERING DCM AND RNA CELLS")
print("="*60)

# DCM cells with known labels (excluding Minus Dox and excluding Stem/TA)
dcm_mask = (plot_df['assay'] == 'pDCM') & \
           (plot_df['module_max_label'].isin(cell_type_order)) & \
           (~plot_df['module_max_label'].isin(['Stem', 'TA'])) & \
           (plot_df['Dox'] != 'Minus')
dcm_cells = plot_df[dcm_mask].copy()

# RNA cells with Stem or TA labels (excluding Minus Dox)
rna_mask = (plot_df['assay'] == 'pRNA') & \
           (plot_df['cell_type_broad'].isin(['Stem', 'TA'])) & \
           (plot_df['Dox'] != 'Minus')
rna_cells = plot_df[rna_mask].copy()

print(f"DCM cells (non-Stem/TA, excluding Minus): {len(dcm_cells)}")
print(f"RNA cells (Stem/TA only, excluding Minus): {len(rna_cells)}")

# -----------------------------
# MATCH BY BATCH_PLATE_WELL (1:1)
# -----------------------------
print("\n" + "="*60)
print("MATCHING CELLS BY BATCH_PLATE_WELL")
print("="*60)

dcm_bpw = set(dcm_cells['batch_plate_well'])
rna_bpw = set(rna_cells['batch_plate_well'])
common_bpw = dcm_bpw.intersection(rna_bpw)

print(f"Common batch_plate_wells: {len(common_bpw)}")

dcm_common = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
rna_common = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]

dcm_counts = dcm_common['batch_plate_well'].value_counts()
rna_counts = rna_common['batch_plate_well'].value_counts()

dcm_single = set(dcm_counts[dcm_counts == 1].index)
rna_single = set(rna_counts[rna_counts == 1].index)
both_single = dcm_single.intersection(rna_single)

print(f"1:1 matched batch_plate_wells: {len(both_single)}")

# Create connections
connections = []
for bpw in both_single:
    dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
    rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
    
    if len(dcm_cell) == 1 and len(rna_cell) == 1:
        dcm_row = dcm_cell.iloc[0]
        rna_row = rna_cell.iloc[0]
        
        conn = {
            'batch_plate_well': bpw,
            'batch': dcm_row['Batch'],
            'plate_well': dcm_row['plate_well'],
            'dcm_cell_type': dcm_row['module_max_label'],
            'rna_cell_type': rna_row['cell_type_broad'],
            'transition': f"{dcm_row['module_max_label']} -> {rna_row['cell_type_broad']}",
            'dcm_index': dcm_row['cell_index'],
            'rna_index': rna_row['cell_index']
        }
        
        # Add z-scores for both DCM and RNA
        for col in ttest_cols:
            ct = col.replace('ttest_', '').replace('_zscore', '')
            conn[f'dcm_{ct}_zscore'] = dcm_row[col]
            conn[f'rna_{ct}_zscore'] = rna_row[col]
        
        connections.append(conn)

connections_df = pd.DataFrame(connections)
print(f"\nTotal transitions to Stem/TA: {len(connections_df)}")
print("\nTransition counts:")
print(connections_df['transition'].value_counts())

if len(connections_df) == 0:
    print("\nNo transitions found!")
else:
    # -----------------------------
    # CREATE HEATMAP FUNCTION
    # -----------------------------
    def create_transition_to_target_heatmap(connections_df, target_ct, cell_type_order, 
                                           cell_type_colors, score_dir):
        """
        Create heatmap for transitions TO a specific target (Stem or TA)
        Using existing z-scores from the data
        """
        print(f"\n{'='*60}")
        print(f"CREATING HEATMAP FOR TRANSITIONS TO {target_ct.upper()}")
        print(f"{'='*60}")
        
        subset = connections_df[connections_df['rna_cell_type'] == target_ct].copy()
        
        if len(subset) == 0:
            print(f"  No transitions found to {target_ct}")
            return
        
        print(f"  Total cells: {len(subset)}")
        print(f"  Source cell types: {subset['dcm_cell_type'].value_counts().to_dict()}")
        
        all_dcm_data = []
        all_rna_data = []
        transition_boundaries = []
        transition_info = []
        
        current_pos = 0
        
        # Sort by source cell type (excluding Stem and TA)
        source_order = [ct for ct in cell_type_order 
                       if ct not in ['Stem', 'TA'] and ct in subset['dcm_cell_type'].unique()]
        
        for dcm_ct in source_order:
            trans_subset = subset[subset['dcm_cell_type'] == dcm_ct].copy()
            
            if len(trans_subset) == 0:
                continue
            
            dcm_zscore_col = f'dcm_{dcm_ct}_zscore'
            rna_zscore_col = f'rna_{target_ct}_zscore'
            
            # Sort by DCM and RNA z-scores
            trans_subset = trans_subset.sort_values(
                by=[dcm_zscore_col, rna_zscore_col], 
                ascending=[False, False]
            )
            
            # Collect DCM z-scores
            for _, row in trans_subset.iterrows():
                dcm_zscores = [row[f'dcm_{ct}_zscore'] for ct in cell_type_order]
                all_dcm_data.append(dcm_zscores)
            
            # Collect RNA z-scores
            for _, row in trans_subset.iterrows():
                rna_zscores = [row[f'rna_{ct}_zscore'] for ct in cell_type_order]
                all_rna_data.append(rna_zscores)
            
            transition_info.append((dcm_ct, len(trans_subset), current_pos))
            current_pos += len(trans_subset)
            transition_boundaries.append(current_pos)
        
        if len(all_dcm_data) == 0:
            print(f"  No data to plot")
            return
        
        dcm_matrix = np.array(all_dcm_data).T
        rna_matrix = np.array(all_rna_data).T
        
        print(f"  DCM matrix shape: {dcm_matrix.shape}")
        print(f"  RNA matrix shape: {rna_matrix.shape}")
        
        # Z-score limits
        vmin, vmax = -3, 3
        
        # Create figure
        fig = plt.figure(figsize=(max(14, len(all_dcm_data) * 0.15), 12))
        
        gs = fig.add_gridspec(2, 2, width_ratios=[1, 0.03], height_ratios=[1, 1],
                             hspace=0.25, wspace=0.05)
        
        cmap = plt.cm.RdBu_r
        
        # DCM Heatmap
        ax_dcm = fig.add_subplot(gs[0, 0])
        im_dcm = ax_dcm.imshow(dcm_matrix, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
        
        for boundary in transition_boundaries[:-1]:
            ax_dcm.axvline(x=boundary - 0.5, color='white', linewidth=2)
        
        ax_dcm.set_yticks(range(len(cell_type_order)))
        ax_dcm.set_yticklabels(cell_type_order, fontsize=10)
        ax_dcm.set_xticks([])
        ax_dcm.set_title(f'DCM Module Z-Scores (cells transitioning to {target_ct})', 
                        fontsize=12, fontweight='bold')
        ax_dcm.set_ylabel('Module', fontsize=11, fontweight='bold')
        
        for dcm_ct, n_cells, start_pos in transition_info:
            mid_pos = start_pos + n_cells / 2
            ax_dcm.text(mid_pos, -0.8, f'{dcm_ct}\n(n={n_cells})', 
                       ha='center', va='bottom', fontsize=9, fontweight='bold',
                       color=cell_type_colors[dcm_ct])
        
        ax_dcm_cbar = fig.add_subplot(gs[0, 1])
        cbar_dcm = fig.colorbar(im_dcm, cax=ax_dcm_cbar)
        cbar_dcm.set_label('Z-Score', fontsize=10, fontweight='bold')
        
        # RNA Heatmap
        ax_rna = fig.add_subplot(gs[1, 0])
        im_rna = ax_rna.imshow(rna_matrix, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
        
        for boundary in transition_boundaries[:-1]:
            ax_rna.axvline(x=boundary - 0.5, color='white', linewidth=2)
        
        # Highlight target row
        target_idx = cell_type_order.index(target_ct)
        ax_rna.axhline(y=target_idx - 0.5, color='black', linewidth=2, linestyle='--', alpha=0.8)
        ax_rna.axhline(y=target_idx + 0.5, color='black', linewidth=2, linestyle='--', alpha=0.8)
        
        ax_rna.set_yticks(range(len(cell_type_order)))
        ax_rna.set_yticklabels(cell_type_order, fontsize=10)
        ax_rna.set_xticks([])
        ax_rna.set_xlabel('Cells (sorted by source cell type and Z-score)', 
                         fontsize=11, fontweight='bold')
        ax_rna.set_title(f'RNA Module Z-Scores (after transition to {target_ct})', 
                        fontsize=12, fontweight='bold')
        ax_rna.set_ylabel('Module', fontsize=11, fontweight='bold')
        
        ax_rna_cbar = fig.add_subplot(gs[1, 1])
        cbar_rna = fig.colorbar(im_rna, cax=ax_rna_cbar)
        cbar_rna.set_label('Z-Score', fontsize=10, fontweight='bold')
        
        fig.suptitle(f'Module Z-Scores for Transitions to {target_ct}\n'
                    f'(n = {len(all_dcm_data)} cells, Matched using Batch + Plate_Well)\n'
                    f'Z-score scale: [{vmin}, {vmax}]', 
                    fontsize=14, fontweight='bold', y=1.00)
        
        plt.tight_layout()
        
        save_path = os.path.join(score_dir, f'heatmap_zscore_transitions_to_{target_ct}.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f"  ✅ Saved: {save_path}")
    
    # -----------------------------
    # CREATE HEATMAPS FOR STEM AND TA
    # -----------------------------
    print("\n" + "="*60)
    print("CREATING HEATMAPS")
    print("="*60)
    
    for target in ['Stem', 'TA']:
        if target in connections_df['rna_cell_type'].unique():
            create_transition_to_target_heatmap(
                connections_df, 
                target, 
                cell_type_order,
                cell_type_colors, 
                score_dir
            )
    
    # -----------------------------
    # SUMMARY HEATMAP
    # -----------------------------
    print("\n" + "="*60)
    print("CREATING SUMMARY HEATMAP")
    print("="*60)
    
    transition_matrix = pd.crosstab(
        connections_df['dcm_cell_type'],
        connections_df['rna_cell_type']
    )
    
    # Reorder
    dcm_order = [ct for ct in cell_type_order 
                 if ct not in ['Stem', 'TA'] and ct in transition_matrix.index]
    rna_order = ['Stem', 'TA']
    
    transition_matrix = transition_matrix.reindex(
        index=dcm_order,
        columns=[ct for ct in rna_order if ct in transition_matrix.columns]
    ).fillna(0).astype(int)
    
    print("\nTransition Matrix (to Stem/TA):")
    print(transition_matrix)
    
    fig, ax = plt.subplots(figsize=(6, 8))
    
    sns.heatmap(transition_matrix, annot=True, fmt='d', cmap='RdBu_r',
                cbar_kws={'label': 'Number of cells'}, ax=ax,
                linewidths=0.5, linecolor='white')
    
    ax.set_title('Transitions to Stem/TA from Other Cell Types\n(Matched using Batch + Plate_Well)', 
                fontsize=13, fontweight='bold')
    ax.set_xlabel('RNA Cell Type', fontsize=11, fontweight='bold')
    ax.set_ylabel('DCM Cell Type', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    save_path = os.path.join(score_dir, 'heatmap_summary_to_stem_ta.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"✅ Saved: {save_path}")
    
    # -----------------------------
    # SAVE DATA
    # -----------------------------
    print("\n" + "="*60)
    print("SAVING DATA")
    print("="*60)
    
    connections_df.to_csv(os.path.join(score_dir, 'transitions_to_stem_ta_with_zscores.csv'), index=False)
    print(f"✅ Saved: {os.path.join(score_dir, 'transitions_to_stem_ta_with_zscores.csv')}")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
## Stem or TA going forward

In [ ]:
label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']
source_order = ['Stem', 'TA']

# Extract relevant data
df = adata_combined.obs.loc[
    adata_combined.obs['ttest_gmm_label_confident_085'].isin(source_order),
    ['batch_plate_well', 'ttest_gmm_label_confident_085', 'leiden_annot_25']
].drop_duplicates('batch_plate_well').copy()

# Convert to ordered categoricals
df['ttest_gmm_label_confident_085'] = pd.Categorical(
    df['ttest_gmm_label_confident_085'],
    categories=source_order,
    ordered=True
)

df['leiden_annot_25'] = pd.Categorical(
    df['leiden_annot_25'],
    categories=label_order,
    ordered=True
)

# Sort properly
df = df.sort_values(
    ['ttest_gmm_label_confident_085', 'leiden_annot_25']
)

# Build dictionary
cell_annotations = {
    row['batch_plate_well']: (
        row['ttest_gmm_label_confident_085'],
        row['leiden_annot_25']
    )
    for _, row in df.iterrows()
}

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from matplotlib.colors import TwoSlopeNorm

# -----------------------------
# SETTINGS
# -----------------------------
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['savefig.dpi'] = 300
root =  '/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/'
score_dir = f'{root}/gene_scoring/gmm_085/backtoStem'
os.makedirs(score_dir, exist_ok=True)

cell_type_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']
source_order = ['Stem', 'TA']

cell_type_colors = {
    'Stem': '#9370DB', 'TA': '#D2B48C', 'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C', 'Goblet': '#2E8B57', 'Tuft': '#FF1493', 'EEC': '#4169E1'
}

# -----------------------------
# ENSURE batch_plate_well EXISTS
# -----------------------------
if 'batch_plate_well' not in adata_combined.obs.columns:
    adata_combined.obs['batch_plate_well'] = (
        adata_combined.obs['Batch'].astype(str) + '_' +
        adata_combined.obs['plate_well'].astype(str)
    )

# -----------------------------
# BUILD BASE DF
# -----------------------------
df = adata_combined.obs.loc[
    adata_combined.obs['ttest_gmm_label_confident_085'].isin(source_order),
    ['batch_plate_well', 'ttest_gmm_label_confident_085', 'leiden_annot_25']
].drop_duplicates('batch_plate_well').copy()

# Ordered categories
df['ttest_gmm_label_confident_085'] = pd.Categorical(
    df['ttest_gmm_label_confident_085'],
    categories=source_order,
    ordered=True
)

df['leiden_annot_25'] = pd.Categorical(
    df['leiden_annot_25'],
    categories=cell_type_order,
    ordered=True
)

# -----------------------------
# FILTER: RNA → Stem or TA
# -----------------------------
df = df[df['leiden_annot_25'].isin(['Stem', 'TA'])].copy()

# Sort
df = df.sort_values(
    ['ttest_gmm_label_confident_085', 'leiden_annot_25']
)

# -----------------------------
# TRANSITION TABLE
# -----------------------------
df['transition'] = (
    df['ttest_gmm_label_confident_085'].astype(str) + ' → ' +
    df['leiden_annot_25'].astype(str)
)

transition_table = df['transition'].value_counts().reset_index()
transition_table.columns = ['transition', 'count']

transition_table.to_csv(
    os.path.join(score_dir, 'transition_table_stem_TA.csv'),
    index=False
)

print("\nTransition table:")
print(transition_table)

# -----------------------------
# BUILD cell_annotations (NOW DEFINED!)
# -----------------------------
cell_annotations = {
    row['batch_plate_well']: (
        row['ttest_gmm_label_confident_085'],
        row['leiden_annot_25']
    )
    for _, row in df.iterrows()
}

# -----------------------------
# GET Z-SCORE DATA
# -----------------------------
ttest_cols = [col for col in adata_combined.obs.columns 
              if 'ttest_' in col and '_zscore' in col and 'max' not in col.lower()]

plot_df = pd.DataFrame({
    'assay': adata_combined.obs['assay'].values,
    'batch_plate_well': adata_combined.obs['batch_plate_well'].values
})

for col in ttest_cols:
    plot_df[col] = adata_combined.obs[col].values

# -----------------------------
# FILTER CELLS
# -----------------------------
target_bpws = list(cell_annotations.keys())

dcm_cells = plot_df[(plot_df['assay'] == 'pDCM') & (plot_df['batch_plate_well'].isin(target_bpws))]
rna_cells = plot_df[(plot_df['assay'] == 'pRNA') & (plot_df['batch_plate_well'].isin(target_bpws))]

# -----------------------------
# SORT CELLS
# -----------------------------
def get_sort_key(bpw):
    dcm_type, rna_type = cell_annotations[bpw]
    return (
        source_order.index(dcm_type),
        cell_type_order.index(rna_type)
    )

sorted_bpws = sorted(target_bpws, key=get_sort_key)

# -----------------------------
# BUILD MATRICES
# -----------------------------
dcm_data, rna_data, valid_bpws = [], [], []

for bpw in sorted_bpws:
    dcm_row = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
    rna_row = rna_cells[rna_cells['batch_plate_well'] == bpw]
    
    if len(dcm_row) and len(rna_row):
        dcm_data.append([dcm_row.iloc[0][f'ttest_{ct}_zscore'] for ct in cell_type_order])
        rna_data.append([rna_row.iloc[0][f'ttest_{ct}_zscore'] for ct in cell_type_order])
        valid_bpws.append(bpw)

dcm_matrix = np.array(dcm_data).T
rna_matrix = np.array(rna_data).T

print(f"\nCells used in heatmap: {len(valid_bpws)}")

# -----------------------------
# SAVE ORDERED CELL CSV
# -----------------------------
ordered_cells_df = pd.DataFrame({
    'batch_plate_well': valid_bpws,
    'DCM_label': [cell_annotations[bpw][0] for bpw in valid_bpws],
    'RNA_label': [cell_annotations[bpw][1] for bpw in valid_bpws],
})

ordered_cells_df['transition'] = (
    ordered_cells_df['DCM_label'] + ' → ' + ordered_cells_df['RNA_label']
)

ordered_cells_df['heatmap_position'] = np.arange(len(valid_bpws))

ordered_cells_df.to_csv(
    os.path.join(score_dir, 'ordered_cells_stem_TA_transitions.csv'),
    index=False
)

# -----------------------------
# GROUP BOUNDARIES
# -----------------------------
dcm_labels = [cell_annotations[bpw][0] for bpw in valid_bpws]

boundaries = []
current = dcm_labels[0]

for i, label in enumerate(dcm_labels):
    if label != current:
        boundaries.append(i)
        current = label

# -----------------------------
# COLOR NORMALIZATION
# -----------------------------
vmin, vmax = -3, 3
norm_dcm = TwoSlopeNorm(vmin=vmin, vcenter=1.0, vmax=vmax)
norm_rna = TwoSlopeNorm(vmin=vmin, vcenter=0.5, vmax=vmax)
cmap = plt.cm.RdBu_r

# -----------------------------
# PLOT
# -----------------------------
fig = plt.figure(figsize=(max(14, len(valid_bpws) * 0.4), 12))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 0.03], height_ratios=[1, 1])

# DCM
ax_dcm = fig.add_subplot(gs[0, 0])
im_dcm = ax_dcm.imshow(dcm_matrix, aspect='auto', cmap=cmap, norm=norm_dcm)

for b in boundaries:
    ax_dcm.axvline(b - 0.5, color='white', linewidth=2)

ax_dcm.set_yticks(range(len(cell_type_order)))
ax_dcm.set_yticklabels(cell_type_order)
ax_dcm.set_title('DCM Z-Scores')

# RNA
ax_rna = fig.add_subplot(gs[1, 0])
im_rna = ax_rna.imshow(rna_matrix, aspect='auto', cmap=cmap, norm=norm_rna)

for b in boundaries:
    ax_rna.axvline(b - 0.5, color='white', linewidth=2)

ax_rna.set_yticks(range(len(cell_type_order)))
ax_rna.set_yticklabels(cell_type_order)
ax_rna.set_title('RNA Z-Scores')

# Colorbars
cbar_dcm = fig.colorbar(im_dcm, ax=ax_dcm)
cbar_dcm.ax.axhline(1.0, color='black', linestyle='--')

cbar_rna = fig.colorbar(im_rna, ax=ax_rna)
cbar_rna.ax.axhline(0.5, color='black', linestyle='--')

plt.tight_layout()

save_path = os.path.join(score_dir, 'heatmap_stem_TA_transitions.pdf')
plt.savefig(save_path)
plt.show()

print(f"\n✅ Saved heatmap: {save_path}")

In [ ]:
obs_names = adata_combined.obs_names

# X counts
x_pool35 = sum(('X' in name) and ('Pool35' in name or 'pool35' in name) for name in obs_names)
x_pool40 = sum(('X' in name) and ('Pool40' in name or 'pool40' in name) for name in obs_names)

# RNA counts
rna_pool35 = sum(('RNA' in name) and ('Pool35' in name or 'pool35' in name) for name in obs_names)
rna_pool40 = sum(('RNA' in name) and ('Pool40' in name or 'pool40' in name) for name in obs_names)

print(f"X + Pool35: {x_pool35}")
print(f"X + Pool40: {x_pool40}")
print(f"RNA + Pool35: {rna_pool35}")
print(f"RNA + Pool40: {rna_pool40}")

In [ ]:
#Module scores from tuft to TA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Filter for Tuft -> TA transitions
tuft_to_ta = connections_df[
    (connections_df['dcm_cell_type'] == 'Tuft') & 
    (connections_df['rna_cell_type'] == 'TA')
].copy()

print("="*60)
print("TUFT -> TA TRANSITIONS")
print("="*60)
print(f"\nTotal cells: {len(tuft_to_ta)}")

if len(tuft_to_ta) > 0:
    # Get the batch_plate_well identifiers for these transitions
    batch_plate_wells = tuft_to_ta['batch_plate_well'].values
    
    # Extract module scores from adata_combined for DCM cells
    dcm_data = adata_combined.obs[
        (adata_combined.obs['assay'] == 'pDCM') & 
        (adata_combined.obs['batch_plate_well'].isin(batch_plate_wells))
    ].copy()
    
    # Extract module scores from adata_combined for RNA cells
    rna_data = adata_combined.obs[
        (adata_combined.obs['assay'] == 'pRNA') & 
        (adata_combined.obs['batch_plate_well'].isin(batch_plate_wells))
    ].copy()
    
    # Merge with transitions to get matched pairs
    dcm_module_scores = {}
    rna_module_scores = {}
    
    for bpw in batch_plate_wells:
        dcm_cell = dcm_data[dcm_data['batch_plate_well'] == bpw]
        rna_cell = rna_data[rna_data['batch_plate_well'] == bpw]
        
        if len(dcm_cell) == 1 and len(rna_cell) == 1:
            for ct in label_order:
                col_name = f'ttest_{ct}_module'
                if col_name in adata_combined.obs.columns:
                    if f'dcm_{ct}_score' not in dcm_module_scores:
                        dcm_module_scores[f'dcm_{ct}_score'] = []
                        rna_module_scores[f'rna_{ct}_score'] = []
                    
                    dcm_module_scores[f'dcm_{ct}_score'].append(dcm_cell[col_name].values[0])
                    rna_module_scores[f'rna_{ct}_score'].append(rna_cell[col_name].values[0])
    
    # Add to tuft_to_ta dataframe
    for col, values in dcm_module_scores.items():
        tuft_to_ta[col] = values
    for col, values in rna_module_scores.items():
        tuft_to_ta[col] = values
    
    # Display module scores
    print("\n" + "="*60)
    print("MODULE SCORES (RAW VALUES)")
    print("="*60)
    
    # Get score columns
    dcm_score_cols = [col for col in tuft_to_ta.columns if col.startswith('dcm_') and col.endswith('_score')]
    rna_score_cols = [col for col in tuft_to_ta.columns if col.startswith('rna_') and col.endswith('_score')]
    
    # Create summary statistics
    print("\nDCM Module Scores:")
    print("-" * 60)
    dcm_summary = tuft_to_ta[dcm_score_cols].describe().T
    dcm_summary.index = [col.replace('dcm_', '').replace('_score', '') for col in dcm_summary.index]
    print(dcm_summary)
    
    print("\nRNA Module Scores:")
    print("-" * 60)
    rna_summary = tuft_to_ta[rna_score_cols].describe().T
    rna_summary.index = [col.replace('rna_', '').replace('_score', '') for col in rna_summary.index]
    print(rna_summary)
    
    # Create visualization
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Prepare data for plotting
    dcm_scores_plot = tuft_to_ta[dcm_score_cols].copy()
    dcm_scores_plot.columns = [col.replace('dcm_', '').replace('_score', '') for col in dcm_scores_plot.columns]
    # Reorder columns to match label_order
    existing_cols = [ct for ct in label_order if ct in dcm_scores_plot.columns]
    dcm_scores_plot = dcm_scores_plot[existing_cols]
    
    rna_scores_plot = tuft_to_ta[rna_score_cols].copy()
    rna_scores_plot.columns = [col.replace('rna_', '').replace('_score', '') for col in rna_scores_plot.columns]
    # Reorder columns to match label_order
    existing_cols = [ct for ct in label_order if ct in rna_scores_plot.columns]
    rna_scores_plot = rna_scores_plot[existing_cols]
    
    # Plot 1: DCM Raw Scores
    dcm_scores_melt = dcm_scores_plot.melt(var_name='Cell Type', value_name='Score')
    sns.boxplot(data=dcm_scores_melt, x='Cell Type', y='Score', ax=axes[0], palette='Set2')
    axes[0].set_title('DCM Module Scores (Raw)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Cell Type Module', fontsize=10)
    axes[0].set_ylabel('Module Score', fontsize=10)
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    # Highlight Tuft module
    if 'Tuft' in existing_cols:
        tuft_idx = existing_cols.index('Tuft')
        axes[0].get_xticklabels()[tuft_idx].set_color('red')
        axes[0].get_xticklabels()[tuft_idx].set_weight('bold')
    
    # Plot 2: RNA Raw Scores
    rna_scores_melt = rna_scores_plot.melt(var_name='Cell Type', value_name='Score')
    sns.boxplot(data=rna_scores_melt, x='Cell Type', y='Score', ax=axes[1], palette='Set2')
    axes[1].set_title('RNA Module Scores (Raw)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Cell Type Module', fontsize=10)
    axes[1].set_ylabel('Module Score', fontsize=10)
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    # Highlight TA module
    if 'TA' in existing_cols:
        ta_idx = existing_cols.index('TA')
        axes[1].get_xticklabels()[ta_idx].set_color('red')
        axes[1].get_xticklabels()[ta_idx].set_weight('bold')
    
    plt.suptitle(f'Module Scores for Tuft → TA Transitions (n={len(tuft_to_ta)} cells)', 
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(score_dir, 'tuft_to_ta_module_scores.pdf'), dpi=300, bbox_inches='tight')
    plt.show()
    
    # Display individual cell data
    print("\n" + "="*60)
    print("INDIVIDUAL CELL DATA (First 10 cells)")
    print("="*60)
    
    display_cols = ['batch_plate_well', 'batch', 'plate_well', 'dcm_cell_type', 'rna_cell_type'] + dcm_score_cols[:3] + rna_score_cols[:3]
    print(tuft_to_ta[display_cols].head(10).to_string())
    
    # Calculate mean changes
    print("\n" + "="*60)
    print("MEAN MODULE SCORE CHANGES (RNA - DCM)")
    print("="*60)
    
    for ct in label_order:
        dcm_col = f'dcm_{ct}_score'
        rna_col = f'rna_{ct}_score'
        if dcm_col in tuft_to_ta.columns and rna_col in tuft_to_ta.columns:
            dcm_mean = tuft_to_ta[dcm_col].mean()
            rna_mean = tuft_to_ta[rna_col].mean()
            mean_change = rna_mean - dcm_mean
            print(f"{ct:15s}: DCM={dcm_mean:+.4f}, RNA={rna_mean:+.4f}, Change={mean_change:+.4f}")
    
    # Create a heatmap showing average scores
    print("\n" + "="*60)
    print("CREATING HEATMAP OF MEAN SCORES")
    print("="*60)
    
    fig, ax = plt.subplots(figsize=(10, 4))
    
    # Prepare data for heatmap
    mean_scores = []
    for ct in label_order:
        if f'dcm_{ct}_score' in tuft_to_ta.columns:
            dcm_mean = tuft_to_ta[f'dcm_{ct}_score'].mean()
            rna_mean = tuft_to_ta[f'rna_{ct}_score'].mean()
            mean_scores.append([dcm_mean, rna_mean])
    
    heatmap_data = np.array(mean_scores).T
    
    sns.heatmap(heatmap_data, 
                annot=True, 
                fmt='.3f',
                xticklabels=label_order,
                yticklabels=['DCM', 'RNA'],
                cmap='RdBu_r',
                center=0,
                cbar_kws={'label': 'Mean Module Score'},
                ax=ax)
    
    ax.set_title(f'Mean Module Scores: Tuft → TA Transitions (n={len(tuft_to_ta)})', 
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Cell Type Module', fontsize=11)
    ax.set_ylabel('Assay', fontsize=11)
    
    plt.tight_layout()
    plt.savefig(os.path.join(score_dir, 'tuft_to_ta_mean_scores_heatmap.pdf'), dpi=300, bbox_inches='tight')
    plt.show()
    
else:
    print("\nNo Tuft -> TA transitions found!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Filter for confident cells (non-ambiguous) from both assays
confident_dcm = adata_combined.obs[
    (adata_combined.obs['assay'] == 'pDCM') &
    (adata_combined.obs['ttest_gmm_label_confident'] != 'ambiguous')
].copy()

confident_rna = adata_combined.obs[
    (adata_combined.obs['assay'] == 'pRNA') &
    (adata_combined.obs['leiden_annot_25'] != 'ambiguous')
].copy()

print("="*60)
print("TUFT AND TA MODULE SCORES BY ASSAY")
print("="*60)
print(f"\npDCM confident cells: {len(confident_dcm)}")
print(f"pRNA confident cells: {len(confident_rna)}")

# Extract data for pDCM
dcm_data = []
for idx, row in confident_dcm.iterrows():
    cell_type = row['ttest_gmm_label_confident']
    tuft_score = row['ttest_Tuft_module'] if 'ttest_Tuft_module' in confident_dcm.columns else np.nan
    ta_score = row['ttest_TA_module'] if 'ttest_TA_module' in confident_dcm.columns else np.nan
    
    dcm_data.append({
        'Cell_Type': cell_type,
        'Tuft_Score': tuft_score,
        'TA_Score': ta_score,
        'Assay': 'pDCM'
    })

# Extract data for pRNA
rna_data = []
for idx, row in confident_rna.iterrows():
    cell_type = row['leiden_annot_25']
    tuft_score = row['ttest_Tuft_module'] if 'ttest_Tuft_module' in confident_rna.columns else np.nan
    ta_score = row['ttest_TA_module'] if 'ttest_TA_module' in confident_rna.columns else np.nan
    
    rna_data.append({
        'Cell_Type': cell_type,
        'Tuft_Score': tuft_score,
        'TA_Score': ta_score,
        'Assay': 'pRNA'
    })

# Combine data
plot_df = pd.DataFrame(dcm_data + rna_data)

# Order cell types
cell_type_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']
existing_cell_types = [ct for ct in cell_type_order if ct in plot_df['Cell_Type'].unique()]

# Print summary statistics
print("\n" + "="*60)
print("CELL TYPE DISTRIBUTION")
print("="*60)
print("\npDCM:")
print(plot_df[plot_df['Assay'] == 'pDCM']['Cell_Type'].value_counts())
print("\npRNA:")
print(plot_df[plot_df['Assay'] == 'pRNA']['Cell_Type'].value_counts())

# Create figure with 4 subplots (2x2)
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Plot 1: Tuft Module Score - pDCM
dcm_df = plot_df[plot_df['Assay'] == 'pDCM']
sns.boxplot(data=dcm_df, x='Cell_Type', y='Tuft_Score', 
            order=existing_cell_types, 
            palette=[cell_type_colors.get(ct, '#cccccc') for ct in existing_cell_types],
            ax=axes[0, 0])
axes[0, 0].set_title('pDCM - Tuft Module Score', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Cell Type', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Tuft Module Score', fontsize=11, fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Highlight Tuft cell type
if 'Tuft' in existing_cell_types:
    tuft_idx = existing_cell_types.index('Tuft')
    axes[0, 0].get_xticklabels()[tuft_idx].set_color('red')
    axes[0, 0].get_xticklabels()[tuft_idx].set_weight('bold')

# Add sample sizes
for i, ct in enumerate(existing_cell_types):
    n = len(dcm_df[dcm_df['Cell_Type'] == ct])
    y_pos = axes[0, 0].get_ylim()[0] + (axes[0, 0].get_ylim()[1] - axes[0, 0].get_ylim()[0]) * 0.02
    axes[0, 0].text(i, y_pos, f'n={n}', ha='center', fontsize=8)

# Plot 2: Tuft Module Score - pRNA
rna_df = plot_df[plot_df['Assay'] == 'pRNA']
sns.boxplot(data=rna_df, x='Cell_Type', y='Tuft_Score', 
            order=existing_cell_types,
            palette=[cell_type_colors.get(ct, '#cccccc') for ct in existing_cell_types],
            ax=axes[0, 1])
axes[0, 1].set_title('pRNA - Tuft Module Score', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Cell Type', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Tuft Module Score', fontsize=11, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Highlight Tuft cell type
if 'Tuft' in existing_cell_types:
    tuft_idx = existing_cell_types.index('Tuft')
    axes[0, 1].get_xticklabels()[tuft_idx].set_color('red')
    axes[0, 1].get_xticklabels()[tuft_idx].set_weight('bold')

# Add sample sizes
for i, ct in enumerate(existing_cell_types):
    n = len(rna_df[rna_df['Cell_Type'] == ct])
    y_pos = axes[0, 1].get_ylim()[0] + (axes[0, 1].get_ylim()[1] - axes[0, 1].get_ylim()[0]) * 0.02
    axes[0, 1].text(i, y_pos, f'n={n}', ha='center', fontsize=8)

# Plot 3: TA Module Score - pDCM
sns.boxplot(data=dcm_df, x='Cell_Type', y='TA_Score', 
            order=existing_cell_types,
            palette=[cell_type_colors.get(ct, '#cccccc') for ct in existing_cell_types],
            ax=axes[1, 0])
axes[1, 0].set_title('pDCM - TA Module Score', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Cell Type', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('TA Module Score', fontsize=11, fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Highlight TA cell type
if 'TA' in existing_cell_types:
    ta_idx = existing_cell_types.index('TA')
    axes[1, 0].get_xticklabels()[ta_idx].set_color('red')
    axes[1, 0].get_xticklabels()[ta_idx].set_weight('bold')

# Add sample sizes
for i, ct in enumerate(existing_cell_types):
    n = len(dcm_df[dcm_df['Cell_Type'] == ct])
    y_pos = axes[1, 0].get_ylim()[0] + (axes[1, 0].get_ylim()[1] - axes[1, 0].get_ylim()[0]) * 0.02
    axes[1, 0].text(i, y_pos, f'n={n}', ha='center', fontsize=8)

# Plot 4: TA Module Score - pRNA
sns.boxplot(data=rna_df, x='Cell_Type', y='TA_Score', 
            order=existing_cell_types,
            palette=[cell_type_colors.get(ct, '#cccccc') for ct in existing_cell_types],
            ax=axes[1, 1])
axes[1, 1].set_title('pRNA - TA Module Score', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Cell Type', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('TA Module Score', fontsize=11, fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Highlight TA cell type
if 'TA' in existing_cell_types:
    ta_idx = existing_cell_types.index('TA')
    axes[1, 1].get_xticklabels()[ta_idx].set_color('red')
    axes[1, 1].get_xticklabels()[ta_idx].set_weight('bold')

# Add sample sizes
for i, ct in enumerate(existing_cell_types):
    n = len(rna_df[rna_df['Cell_Type'] == ct])
    y_pos = axes[1, 1].get_ylim()[0] + (axes[1, 1].get_ylim()[1] - axes[1, 1].get_ylim()[0]) * 0.02
    axes[1, 1].text(i, y_pos, f'n={n}', ha='center', fontsize=8)

plt.suptitle(f'Tuft and TA Module Scores by Assay and Cell Type\n(pDCM: n={len(dcm_df)}, pRNA: n={len(rna_df)})', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(os.path.join(score_dir, 'tuft_ta_module_scores_by_assay_and_celltype.pdf'), dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

print("\npDCM - Tuft Module Score by Cell Type:")
print(dcm_df.groupby('Cell_Type')['Tuft_Score'].describe())

print("\npRNA - Tuft Module Score by Cell Type:")
print(rna_df.groupby('Cell_Type')['Tuft_Score'].describe())

print("\npDCM - TA Module Score by Cell Type:")
print(dcm_df.groupby('Cell_Type')['TA_Score'].describe())

print("\npRNA - TA Module Score by Cell Type:")
print(rna_df.groupby('Cell_Type')['TA_Score'].describe())

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
##Transitions per condition

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binomtest, chi2_contingency, fisher_exact
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/gmm_085/stability'
os.makedirs(score_dir, exist_ok=True)

adata = adata_combined

# Cell types
label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# Colors for stable/unstable
stability_colors = {
    'Stable': '#2E8B57',    # Green
    'Unstable': '#DC143C'   # Red
}

# Define Dox condition order (48hDox first, Chase second)
dox_order = ['48hDox', '48hDox-72hChase']

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata.obs['batch_plate_well'] = (
    adata.obs['Batch'].astype(str) + '_' +
    adata.obs['plate_well'].astype(str)
)

# -----------------------------
# PREPARE DATA
# -----------------------------
print("\n" + "="*60)
print("PREPARING DATA")
print("="*60)

plot_df = pd.DataFrame({
    'assay':            adata.obs['assay'].values,
    'plate_well':       adata.obs['plate_well'].values,
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'Batch':            adata.obs['Batch'].values,
    'cell_type_broad':  adata.obs['leiden_annot_25'].astype(str).values,
    'module_max_label': adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'Dox':              adata.obs['Dox'].values if 'Dox' in adata.obs.columns else 'Unknown',
    'cell_index':       adata.obs.index.values
})

# -----------------------------
# FILTER AND MATCH CELLS
# -----------------------------
print("\n" + "="*60)
print("FILTERING AND MATCHING CELLS")
print("="*60)

dcm_mask       = (plot_df['assay'] == 'pDCM') & (plot_df['Dox'] != 'Minus')
dcm_known_mask = dcm_mask & plot_df['module_max_label'].isin(label_order)
dcm_cells      = plot_df[dcm_known_mask].copy()

rna_mask       = (plot_df['assay'] == 'pRNA') & (plot_df['Dox'] != 'Minus')
rna_known_mask = rna_mask & plot_df['cell_type_broad'].isin(label_order)
rna_cells      = plot_df[rna_known_mask].copy()

print(f"DCM cells (excluding Minus Dox): {len(dcm_cells)}")
print(f"RNA cells (excluding Minus Dox): {len(rna_cells)}")

# -----------------------------
# CHECK DUPLICATES
# -----------------------------
print("\n" + "="*60)
print("CHECKING DUPLICATES")
print("="*60)

dcm_bpw_counts = dcm_cells['batch_plate_well'].value_counts()
rna_bpw_counts = rna_cells['batch_plate_well'].value_counts()
print(f"DCM batch_plate_wells with >1 cell: {(dcm_bpw_counts > 1).sum()}")
print(f"RNA batch_plate_wells with >1 cell: {(rna_bpw_counts > 1).sum()}")

# -----------------------------
# MATCH BY BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("FINDING CONNECTIONS USING BATCH_PLATE_WELL")
print("="*60)

common_bpw  = set(dcm_cells['batch_plate_well']) & set(rna_cells['batch_plate_well'])
dcm_common  = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
rna_common  = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]

dcm_single  = set(dcm_common['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
rna_single  = set(rna_common['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
both_single = dcm_single & rna_single
print(f"1:1 matched batch_plate_wells: {len(both_single)}")

connections = []
for bpw in both_single:
    dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
    rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
    if len(dcm_cell) == 1 and len(rna_cell) == 1:
        dcm_row   = dcm_cell.iloc[0]
        rna_row   = rna_cell.iloc[0]
        is_stable = dcm_row['module_max_label'] == rna_row['cell_type_broad']
        connections.append({
            'batch_plate_well': bpw,
            'batch':            dcm_row['Batch'],
            'plate_well':       dcm_row['plate_well'],
            'Dox':              dcm_row['Dox'],
            'dcm_cell_type':    dcm_row['module_max_label'],
            'rna_cell_type':    rna_row['cell_type_broad'],
            'transition':       f"{dcm_row['module_max_label']} -> {rna_row['cell_type_broad']}",
            'is_stable':        is_stable,
            'stability':        'Stable' if is_stable else 'Unstable',
        })

connections_df = pd.DataFrame(connections)
print(f"Total valid connections: {len(connections_df)}")

# -----------------------------
# STATISTICS PER DOX CONDITION
# -----------------------------
print("\n" + "="*60)
print("CALCULATING STATISTICS PER DOX CONDITION")
print("="*60)

dox_conditions = [d for d in dox_order if d in connections_df['Dox'].unique()]
print(f"Dox conditions (ordered): {dox_conditions}")

stats_list = []
for dox in dox_conditions:
    dox_data   = connections_df[connections_df['Dox'] == dox]
    n_total    = len(dox_data)
    n_stable   = int(dox_data['is_stable'].sum())
    n_unstable = n_total - n_stable
    pct_stable   = n_stable   / n_total * 100 if n_total > 0 else 0
    pct_unstable = n_unstable / n_total * 100 if n_total > 0 else 0
    binom_pvalue = (binomtest(n_stable, n_total, p=0.5, alternative='two-sided').pvalue
                    if n_total > 0 else np.nan)
    stats_list.append({
        'Dox':          dox,
        'n_total':      n_total,
        'n_stable':     n_stable,
        'n_unstable':   n_unstable,
        'pct_stable':   pct_stable,
        'pct_unstable': pct_unstable,
        'binom_pvalue': binom_pvalue,
    })
    print(f"\n{dox}:  Total={n_total}  Stable={n_stable} ({pct_stable:.1f}%)  "
          f"Unstable={n_unstable} ({pct_unstable:.1f}%)  "
          f"Binom p={binom_pvalue:.4e}")

stats_df = pd.DataFrame(stats_list)
stats_df['Dox'] = pd.Categorical(stats_df['Dox'],
                                  categories=dox_conditions, ordered=True)
stats_df = stats_df.sort_values('Dox').reset_index(drop=True)

# -----------------------------
# PAIRWISE DOX COMPARISON
# Fisher's exact test on the 2×2 contingency table:
#   rows = Dox conditions, cols = Stable / Unstable
# This directly answers "is the stable/unstable ratio
# significantly different between the two Dox conditions?"
# -----------------------------
print("\n" + "="*60)
print("PAIRWISE DOX COMPARISON (Fisher's exact test)")
print("="*60)

pairwise_results = []

if len(dox_conditions) >= 2:
    from itertools import combinations
    for d1, d2 in combinations(dox_conditions, 2):
        r1 = stats_df[stats_df['Dox'] == d1].iloc[0]
        r2 = stats_df[stats_df['Dox'] == d2].iloc[0]
        # 2×2 table: [[stable_d1, unstable_d1], [stable_d2, unstable_d2]]
        table = np.array([[r1['n_stable'],   r1['n_unstable']],
                          [r2['n_stable'],   r2['n_unstable']]])
        odds_ratio, fisher_p = fisher_exact(table, alternative='two-sided')
        pairwise_results.append({
            'Dox_1':       d1,
            'Dox_2':       d2,
            'n_d1':        int(r1['n_total']),
            'n_d2':        int(r2['n_total']),
            'pct_stable_d1': r1['pct_stable'],
            'pct_stable_d2': r2['pct_stable'],
            'odds_ratio':  odds_ratio,
            'fisher_p':    fisher_p,
        })
        print(f"\n{d1} vs {d2}:")
        print(f"  Contingency table: {table.tolist()}")
        print(f"  Odds ratio: {odds_ratio:.3f}")
        print(f"  Fisher exact p-value: {fisher_p:.4e}")

pairwise_df = pd.DataFrame(pairwise_results)

# Also run chi-square for completeness
contingency_table = pd.crosstab(connections_df['Dox'],
                                 connections_df['stability'])
contingency_table = contingency_table.reindex(dox_conditions)
print("\nContingency Table (chi-square):")
print(contingency_table)

if contingency_table.shape[0] > 1 and contingency_table.shape[1] > 1:
    chi2, chi2_p, dof, expected = chi2_contingency(contingency_table)
    print(f"Chi-square: {chi2:.4f}  df={dof}  p={chi2_p:.4e}")
    pairwise_df['chi2']   = chi2
    pairwise_df['chi2_p'] = chi2_p

# -----------------------------
# SIGNIFICANCE HELPER
# -----------------------------
def get_significance_stars(pvalue):
    if pd.isna(pvalue): return ''
    if pvalue < 0.001:  return '***'
    if pvalue < 0.01:   return '**'
    if pvalue < 0.05:   return '*'
    return 'ns'

stats_df['significance'] = stats_df['binom_pvalue'].apply(get_significance_stars)

# Convenience lookups
stable_pcts    = [stats_df[stats_df['Dox'] == d]['pct_stable'].values[0]   for d in dox_conditions]
unstable_pcts  = [stats_df[stats_df['Dox'] == d]['pct_unstable'].values[0] for d in dox_conditions]
stable_counts  = [stats_df[stats_df['Dox'] == d]['n_stable'].values[0]     for d in dox_conditions]
unstable_counts= [stats_df[stats_df['Dox'] == d]['n_unstable'].values[0]   for d in dox_conditions]
total_counts   = [stats_df[stats_df['Dox'] == d]['n_total'].values[0]      for d in dox_conditions]

# -----------------------------
# GROUPED BARPLOT
# -----------------------------
print("\n" + "="*60)
print("CREATING GROUPED BARPLOT")
print("="*60)

x     = np.arange(len(dox_conditions))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 8))

bars_stable   = ax.bar(x - width/2, stable_pcts,   width, label='Stable',
                       color=stability_colors['Stable'],
                       edgecolor='black', linewidth=1)
bars_unstable = ax.bar(x + width/2, unstable_pcts, width, label='Unstable',
                       color=stability_colors['Unstable'],
                       edgecolor='black', linewidth=1)

# n labels on bars
for i, (bs, bu) in enumerate(zip(bars_stable, bars_unstable)):
    ax.text(bs.get_x() + bs.get_width()/2, bs.get_height() + 1,
            f'n={stable_counts[i]}', ha='center', va='bottom',
            fontsize=9, fontweight='bold')
    ax.text(bu.get_x() + bu.get_width()/2, bu.get_height() + 1,
            f'n={unstable_counts[i]}', ha='center', va='bottom',
            fontsize=9, fontweight='bold')

# ── Within-condition binomial brackets ───────────────────────
for i, dox in enumerate(dox_conditions):
    pvalue = stats_df[stats_df['Dox'] == dox]['binom_pvalue'].values[0]
    stars  = get_significance_stars(pvalue)
    max_h  = max(stable_pcts[i], unstable_pcts[i])
    bh     = max_h + 8
    ax.plot([x[i] - width/2, x[i] - width/2,
             x[i] + width/2, x[i] + width/2],
            [bh - 2, bh, bh, bh - 2],
            color='black', linewidth=1.5)
    ax.text(x[i], bh + 1, stars,
            ha='center', va='bottom', fontsize=12, fontweight='bold')
    pval_txt = f'p={pvalue:.2e}' if pvalue < 0.001 else f'p={pvalue:.3f}'
    ax.text(x[i], bh + 6, pval_txt,
            ha='center', va='bottom', fontsize=8, style='italic')

# ── Between-condition Fisher bracket (stable bars only) ──────
if len(pairwise_results) > 0 and len(dox_conditions) == 2:
    row      = pairwise_df.iloc[0]
    fish_p   = row['fisher_p']
    fish_str = get_significance_stars(fish_p)
    pval_txt = f'p={fish_p:.2e}' if fish_p < 0.001 else f'p={fish_p:.3f}'

    # Bracket spanning the two Stable bars
    y_cross = max(stable_pcts) + 22
    ax.plot([x[0] - width/2, x[0] - width/2,
             x[1] - width/2, x[1] - width/2],
            [y_cross - 2, y_cross, y_cross, y_cross - 2],
            color='#2E8B57', linewidth=1.5)
    ax.text((x[0] + x[1]) / 2 - width/2, y_cross + 1,
            f'{fish_str} (Stable: Fisher p={fish_p:.2e})',
            ha='center', va='bottom', fontsize=9,
            fontweight='bold', color='#2E8B57')

ax.set_xlabel('Dox Condition', fontsize=12, fontweight='bold')
ax.set_ylabel('Percentage of Cells (%)', fontsize=12, fontweight='bold')
ax.set_title('Stability of Cell Type Transitions by Dox Condition\n'
             '(Within-bar: Binomial test vs 50% | '
             'Between-bar bracket: Fisher exact test)',
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(dox_conditions, fontsize=11)
ax.set_ylim(0, 115)
ax.axhline(y=50, color='gray', linestyle='--', linewidth=1, alpha=0.7,
           label='50% threshold')
ax.legend(loc='upper right', fontsize=10)

for i, dox in enumerate(dox_conditions):
    ax.text(x[i], -5, f'Total: {total_counts[i]}',
            ha='center', va='top', fontsize=10, fontweight='bold')

plt.tight_layout()
save_path = os.path.join(score_dir, 'barplot_stability_by_dox_batch_plate_well.pdf')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: {save_path}")

# -----------------------------
# STACKED BARPLOT
# -----------------------------
print("\n" + "="*60)
print("CREATING STACKED BARPLOT")
print("="*60)

fig, ax = plt.subplots(figsize=(8, 8))

bars_stable   = ax.bar(dox_conditions, stable_pcts,   label='Stable',
                       color=stability_colors['Stable'],
                       edgecolor='black', linewidth=1)
bars_unstable = ax.bar(dox_conditions, unstable_pcts, bottom=stable_pcts,
                       label='Unstable',
                       color=stability_colors['Unstable'],
                       edgecolor='black', linewidth=1)

for i, dox in enumerate(dox_conditions):
    if stable_pcts[i] > 10:
        ax.text(i, stable_pcts[i]/2,
                f'{stable_pcts[i]:.1f}%\n(n={stable_counts[i]})',
                ha='center', va='center', fontsize=10,
                fontweight='bold', color='white')
    if unstable_pcts[i] > 10:
        ax.text(i, stable_pcts[i] + unstable_pcts[i]/2,
                f'{unstable_pcts[i]:.1f}%\n(n={unstable_counts[i]})',
                ha='center', va='center', fontsize=10,
                fontweight='bold', color='white')

# Within-condition binomial stars
for i, dox in enumerate(dox_conditions):
    pvalue = stats_df[stats_df['Dox'] == dox]['binom_pvalue'].values[0]
    stars  = get_significance_stars(pvalue)
    ax.text(i, 102, stars,
            ha='center', va='bottom', fontsize=14, fontweight='bold')
    pval_txt = f'p={pvalue:.2e}' if pvalue < 0.001 else f'p={pvalue:.3f}'
    ax.text(i, 107, pval_txt,
            ha='center', va='bottom', fontsize=8, style='italic')

# Between-condition Fisher bracket
if len(pairwise_results) > 0 and len(dox_conditions) == 2:
    row      = pairwise_df.iloc[0]
    fish_p   = row['fisher_p']
    fish_str = get_significance_stars(fish_p)
    y_cross  = 114
    ax.plot([0, 0, 1, 1],
            [y_cross - 1, y_cross, y_cross, y_cross - 1],
            color='black', linewidth=1.5)
    pval_txt = f'p={fish_p:.2e}' if fish_p < 0.001 else f'p={fish_p:.3f}'
    ax.text(0.5, y_cross + 0.5,
            f'{fish_str}  Fisher: {pval_txt}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('Dox Condition', fontsize=12, fontweight='bold')
ax.set_ylabel('Percentage of Cells (%)', fontsize=12, fontweight='bold')
ax.set_title('Stability of Cell Type Transitions by Dox Condition\n'
             '(Within: Binomial vs 50% | Between: Fisher exact test)',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 125)
ax.axhline(y=50, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
ax.legend(loc='upper right', fontsize=10)

for i, dox in enumerate(dox_conditions):
    ax.text(i, -3, f'n={total_counts[i]}',
            ha='center', va='top', fontsize=10, fontweight='bold')

plt.tight_layout()
save_path = os.path.join(score_dir, 'barplot_stability_by_dox_stacked_batch_plate_well.pdf')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: {save_path}")

# -----------------------------
# SUMMARY TABLE + CSV OUTPUTS
# -----------------------------
print("\n" + "="*60)
print("SUMMARY TABLE")
print("="*60)

print(stats_df[['Dox', 'n_total', 'n_stable', 'n_unstable',
                 'pct_stable', 'pct_unstable',
                 'binom_pvalue', 'significance']].to_string(index=False))

stats_df.to_csv(
    os.path.join(score_dir, 'stability_statistics_by_dox_batch_plate_well.csv'),
    index=False)
print(f"\nSaved: stability_statistics_by_dox_batch_plate_well.csv")

if not pairwise_df.empty:
    pairwise_df['significance'] = pairwise_df['fisher_p'].apply(get_significance_stars)
    pairwise_df.to_csv(
        os.path.join(score_dir, 'stability_pairwise_dox_comparison.csv'),
        index=False)
    print(f"Saved: stability_pairwise_dox_comparison.csv")
    print("\nPairwise comparison summary:")
    print(pairwise_df.to_string(index=False))

# -----------------------------
# DETAILED TRANSITION BREAKDOWN
# -----------------------------
print("\n" + "="*60)
print("DETAILED TRANSITION BREAKDOWN BY DOX")
print("="*60)

for dox in dox_conditions:
    dox_data = connections_df[connections_df['Dox'] == dox]
    print(f"\n{dox} (n={len(dox_data)}):")
    print("  Stable transitions:")
    for trans, count in dox_data[dox_data['is_stable']]['transition'].value_counts().items():
        print(f"    {trans}: {count}")
    print("  Unstable transitions (top 10):")
    for trans, count in dox_data[~dox_data['is_stable']]['transition'].value_counts().head(10).items():
        print(f"    {trans}: {count}")

connections_df.to_csv(
    os.path.join(score_dir, 'transitions_stability_by_dox_batch_plate_well.csv'),
    index=False)
print(f"\nSaved: transitions_stability_by_dox_batch_plate_well.csv")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
# -----------------------------
# DETAILED ANALYSIS USING BATCH + PLATE_WELL
# -----------------------------

print("="*60)
print("DETAILED ANALYSIS: USING BATCH + PLATE_WELL")
print("="*60)

# Create a unique identifier combining Batch and plate_well
adata_combined.obs['batch_plate_well'] = (
    adata_combined.obs['Batch'].astype(str) + '_' + 
    adata_combined.obs['plate_well'].astype(str)
)

# Starting point
label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

dcm_with_gmm = adata_combined.obs[
    (adata_combined.obs['assay'] == 'pDCM') & 
    (adata_combined.obs['ttest_gmm_label_confident'].isin(label_order))
].copy()

rna_with_label = adata_combined.obs[
    (adata_combined.obs['assay'] == 'pRNA') & 
    (adata_combined.obs['leiden_annot_25'].isin(label_order))
].copy()

print(f"\n1. DCM cells with GMM label: {len(dcm_with_gmm)}")
print(f"   Unique plate_wells: {dcm_with_gmm['plate_well'].nunique()}")
print(f"   Unique batch_plate_wells: {dcm_with_gmm['batch_plate_well'].nunique()}")

print(f"\n2. RNA cells with leiden label: {len(rna_with_label)}")
print(f"   Unique plate_wells: {rna_with_label['plate_well'].nunique()}")
print(f"   Unique batch_plate_wells: {rna_with_label['batch_plate_well'].nunique()}")

# -----------------------------
# CHECK DUPLICATES WITH BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("CHECKING DUPLICATES WITH BATCH_PLATE_WELL")
print("="*60)

dcm_bpw_counts = dcm_with_gmm['batch_plate_well'].value_counts()
dcm_bpw_duplicates = dcm_bpw_counts[dcm_bpw_counts > 1]
print(f"\nDCM batch_plate_wells with >1 cell: {len(dcm_bpw_duplicates)}")
if len(dcm_bpw_duplicates) > 0:
    print(dcm_bpw_duplicates)

rna_bpw_counts = rna_with_label['batch_plate_well'].value_counts()
rna_bpw_duplicates = rna_bpw_counts[rna_bpw_counts > 1]
print(f"\nRNA batch_plate_wells with >1 cell: {len(rna_bpw_duplicates)}")
if len(rna_bpw_duplicates) > 0:
    print(rna_bpw_duplicates.head(20))

# -----------------------------
# FIND MATCHES USING BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("MATCHING USING BATCH_PLATE_WELL")
print("="*60)

dcm_bpw = set(dcm_with_gmm['batch_plate_well'].unique())
rna_bpw = set(rna_with_label['batch_plate_well'].unique())

common_bpw = dcm_bpw.intersection(rna_bpw)
print(f"\n3. Common batch_plate_wells (DCM ∩ RNA): {len(common_bpw)}")

# DCM cells with matching RNA
dcm_with_match = dcm_with_gmm[dcm_with_gmm['batch_plate_well'].isin(common_bpw)]
print(f"4. DCM cells with matching RNA batch_plate_well: {len(dcm_with_match)}")

# -----------------------------
# CHECK FOR 1:1 MATCHES
# -----------------------------
print("\n" + "="*60)
print("1:1 MATCHES")
print("="*60)

dcm_common = dcm_with_gmm[dcm_with_gmm['batch_plate_well'].isin(common_bpw)]
rna_common = rna_with_label[rna_with_label['batch_plate_well'].isin(common_bpw)]

dcm_counts = dcm_common['batch_plate_well'].value_counts()
rna_counts = rna_common['batch_plate_well'].value_counts()

dcm_single = set(dcm_counts[dcm_counts == 1].index)
rna_single = set(rna_counts[rna_counts == 1].index)
both_single = dcm_single.intersection(rna_single)

print(f"\n5. batch_plate_wells with exactly 1 DCM cell: {len(dcm_single)}")
print(f"   batch_plate_wells with exactly 1 RNA cell: {len(rna_single)}")
print(f"   batch_plate_wells with 1:1 match: {len(both_single)}")

# Final matched cells
dcm_final = dcm_with_gmm[dcm_with_gmm['batch_plate_well'].isin(both_single)]
print(f"\n6. Final DCM cells used in transitions: {len(dcm_final)}")

# -----------------------------
# BREAKDOWN OF LOST CELLS
# -----------------------------
print("\n" + "="*60)
print("BREAKDOWN OF LOST DCM CELLS")
print("="*60)

# Lost: no RNA match
no_rna_match = dcm_bpw - rna_bpw
dcm_lost_no_rna = dcm_with_gmm[dcm_with_gmm['batch_plate_well'].isin(no_rna_match)]

# Lost: DCM duplicates
dcm_duplicate_bpw = set(dcm_counts[dcm_counts > 1].index) if len(dcm_counts[dcm_counts > 1]) > 0 else set()
dcm_lost_dcm_dup = dcm_with_gmm[dcm_with_gmm['batch_plate_well'].isin(dcm_duplicate_bpw)]

# Lost: RNA duplicates (DCM is single but RNA has duplicates)
rna_duplicate_bpw = set(rna_counts[rna_counts > 1].index) if len(rna_counts[rna_counts > 1]) > 0 else set()
dcm_lost_rna_dup = dcm_with_gmm[
    (dcm_with_gmm['batch_plate_well'].isin(dcm_single)) & 
    (dcm_with_gmm['batch_plate_well'].isin(rna_duplicate_bpw))
]

print(f"""
Starting DCM cells (with GMM label):          {len(dcm_with_gmm)}

LOST:
  - No RNA cell for batch_plate_well:         {len(dcm_lost_no_rna)}
  - DCM has duplicate cells in same well:     {len(dcm_lost_dcm_dup)}
  - RNA has duplicate cells in same well:     {len(dcm_lost_rna_dup)}

REMAINING (1:1 matches):                      {len(dcm_final)}
""")

# -----------------------------
# COMPARE WITH PLATE_WELL ONLY
# -----------------------------
print("\n" + "="*60)
print("COMPARISON: PLATE_WELL vs BATCH_PLATE_WELL")
print("="*60)

# Using plate_well only
dcm_pw = set(dcm_with_gmm['plate_well'].unique())
rna_pw = set(rna_with_label['plate_well'].unique())
common_pw = dcm_pw.intersection(rna_pw)

dcm_pw_counts = dcm_with_gmm[dcm_with_gmm['plate_well'].isin(common_pw)]['plate_well'].value_counts()
rna_pw_counts = rna_with_label[rna_with_label['plate_well'].isin(common_pw)]['plate_well'].value_counts()

pw_both_single = set(dcm_pw_counts[dcm_pw_counts == 1].index).intersection(
    set(rna_pw_counts[rna_pw_counts == 1].index)
)

print(f"""
Using plate_well only:
  - Common plate_wells: {len(common_pw)}
  - 1:1 matches: {len(pw_both_single)}

Using batch_plate_well:
  - Common batch_plate_wells: {len(common_bpw)}
  - 1:1 matches: {len(both_single)}

Difference: {len(both_single) - len(pw_both_single)} additional matches with batch info
""")

# -----------------------------
# CREATE TRANSITION TABLE WITH BATCH
# -----------------------------
print("\n" + "="*60)
print("CREATING TRANSITIONS WITH BATCH_PLATE_WELL")
print("="*60)

connections = []
for bpw in both_single:
    dcm_cell = dcm_with_gmm[dcm_with_gmm['batch_plate_well'] == bpw]
    rna_cell = rna_with_label[rna_with_label['batch_plate_well'] == bpw]
    
    if len(dcm_cell) == 1 and len(rna_cell) == 1:
        connections.append({
            'batch_plate_well': bpw,
            'batch': dcm_cell.iloc[0]['Batch'],
            'plate_well': dcm_cell.iloc[0]['plate_well'],
            'dcm_label': dcm_cell.iloc[0]['ttest_gmm_label_confident'],
            'rna_label': rna_cell.iloc[0]['leiden_annot_25'],
            'transition': f"{dcm_cell.iloc[0]['ttest_gmm_label_confident']} -> {rna_cell.iloc[0]['leiden_annot_25']}"
        })

connections_df = pd.DataFrame(connections)
print(f"Total valid connections: {len(connections_df)}")

print("\nTransition counts:")
print(connections_df['transition'].value_counts())

print("\nTransitions by Batch:")
print(connections_df.groupby('batch')['transition'].value_counts())

# Save
connections_df.to_csv(f'{outdir}transitions_with_batch.csv', index=False)
print(f"\nSaved: {outdir}transitions_with_batch.csv")

In [ ]:
adata_combined.obs.to_csv("/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/JK_Anal_ysis/metadata_GMM_labels_17032026.csv")

In [ ]:
#Cell cycle from hybrid secretory towards enterocytes and secretory towards hybrid enterocyte (cell cycle, umap and heatmaps)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scanpy as sc
from scipy import stats
import os

# -----------------------------
# SETTINGS
# -----------------------------
#score_dir  = f'{root}/gene_scoring/'
score_dir  = f'{root}/gene_scoring/gmm_085/cell_cycle3'

output_dir = score_dir
os.makedirs(output_dir, exist_ok=True)

# Cell cycle genes from Tirosh et al. 2016
s_genes = ['MCM5', 'PCNA', 'TYMS', 'FEN1', 'MCM2', 'MCM4', 'RRM1', 'UNG', 'GINS2', 'MCM6',
           'CDCA7', 'DTL', 'PRIM1', 'UHRF1', 'MLF1IP', 'HELLS', 'RFC2', 'RPA2', 'NASP',
           'RAD51AP1', 'GMNN', 'WDR76', 'SLBP', 'CCNE2', 'UBR7', 'POLD3', 'MSH2', 'ATAD2',
           'RAD51', 'RRM2', 'CDC45', 'CDC6', 'EXO1', 'TIPIN', 'DSCC1', 'BLM', 'CASP8AP2',
           'USP1', 'CLSPN', 'POLA1', 'CHAF1B', 'BRIP1', 'E2F8']

g2m_genes = ['HMGB2', 'CDK1', 'NUSAP1', 'UBE2C', 'BIRC5', 'TPX2', 'TOP2A', 'NDC80', 'CKS2',
             'NUF2', 'CKS1B', 'MKI67', 'TMPO', 'CENPF', 'TACC3', 'FAM64A', 'SMC4', 'CCNB2',
             'CKAP2L', 'CKAP2', 'AURKB', 'BUB1', 'KIF11', 'ANP32E', 'TUBB4B', 'GTSE1', 'KIF20B',
             'HJURP', 'CDCA3', 'HN1', 'CDC20', 'TTK', 'CDC25C', 'KIF2C', 'RANGAP1', 'NCAPD2',
             'DLGAP5', 'CDCA2', 'CDCA8', 'ECT2', 'KIF23', 'HMMR', 'AURKA', 'PSRC1', 'ANLN',
             'LBR', 'CKAP5', 'CENPE', 'CTCF', 'NEK2', 'G2E3', 'GAS2L3', 'CBX5', 'CENPA']

s_genes_mouse   = [g.capitalize() for g in s_genes]
g2m_genes_mouse = [g.capitalize() for g in g2m_genes]

secretory_types = ['Paneth', 'Goblet', 'Tuft', 'EEC']

# ─────────────────────────────────────────────
# STEP 1: SCORE pDCM CELLS SEPARATELY
# Doc-9 approach: score each assay on its own
# distribution so bins are not contaminated by
# the other assay.
# ─────────────────────────────────────────────
print("=" * 70)
print("STEP 1: SCORING pDCM CELLS")
print("=" * 70)

adata_dcm = adata_combined[adata_combined.obs['assay'] == 'pDCM'].copy()
print(f"Total pDCM cells: {len(adata_dcm)}")

s_use   = [g for g in s_genes_mouse   if g in adata_dcm.var_names]
g2m_use = [g for g in g2m_genes_mouse if g in adata_dcm.var_names]
print(f"S genes found   : {len(s_use)}/{len(s_genes_mouse)}")
print(f"G2M genes found : {len(g2m_use)}/{len(g2m_genes_mouse)}")

sc.tl.score_genes_cell_cycle(adata_dcm, s_genes=s_use, g2m_genes=g2m_use)
print("pDCM scoring complete.")

# ─────────────────────────────────────────────
# STEP 2: BUILD pDCM DATAFRAME
# ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 2: BUILDING pDCM DATAFRAME")
print("=" * 70)

pDCM_df = pd.DataFrame({
    'batch_plate_well': adata_dcm.obs['batch_plate_well'].values,
    'Dox':              adata_dcm.obs['Dox'].astype(str).values,
    'dcm_celltype':     adata_dcm.obs['ttest_gmm_label_confident_085'].astype(str)
                                    .str.split('_').str[-1].values,
    'is_multipotent':   adata_dcm.obs['is_multipotent'].astype(str).values,
    'sig_celltypes':    adata_dcm.obs['significant_celltypes'].astype(str).values,
    'S_score':          adata_dcm.obs['S_score'].values,
    'G2M_score':        adata_dcm.obs['G2M_score'].values,
    'phase':            adata_dcm.obs['phase'].values,
}, index=adata_dcm.obs_names)
pDCM_df['combined_score'] = pDCM_df['S_score'] + pDCM_df['G2M_score']

# ─────────────────────────────────────────────
# STEP 3: BUILD pRNA DATAFRAME
# ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 3: BUILDING pRNA DATAFRAME")
print("=" * 70)

rna_mask = adata_combined.obs['assay'] == 'pRNA'
pRNA_df  = pd.DataFrame({
    'batch_plate_well': adata_combined.obs[rna_mask]['batch_plate_well'].values,
    'rna_celltype':     adata_combined.obs[rna_mask]['leiden_annot_25'].astype(str)
                                       .str.split('_').str[-1].values,
    'rna_is_multipotent': adata_combined.obs[rna_mask]['is_multipotent'].astype(str).values,
    'rna_sig_celltypes':  adata_combined.obs[rna_mask]['significant_celltypes'].astype(str).values,
})

# ─────────────────────────────────────────────
# STEP 4: PAIR AND CLASSIFY
# ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 4: PAIRING AND CLASSIFYING TRANSITIONS")
print("=" * 70)

paired = pDCM_df.merge(pRNA_df, on='batch_plate_well', how='inner')
print(f"Total paired cells: {len(paired)}")

def has_secretory(sig_str):
    if pd.isna(sig_str) or str(sig_str).strip() in ('', 'nan', 'none', 'None'):
        return False
    return any(s in str(sig_str) for s in secretory_types)

def classify(row):
    dcm      = row['dcm_celltype']
    rna      = row['rna_celltype']
    dcm_multi = str(row['is_multipotent']).lower() in ('true', '1', 'yes')
    rna_multi = str(row['rna_is_multipotent']).lower() in ('true', '1', 'yes')
    sig       = str(row['sig_celltypes'])

    # ── Clear single-identity DCM cells ──────────────────────────
    if not dcm_multi:
        if dcm in secretory_types and rna == 'Enterocyte':
            return f'{dcm}→Enterocyte'
        if dcm in secretory_types and rna in secretory_types:
            return 'Secretory→Secretory'
        if dcm == 'TA' and rna == 'Enterocyte':
            return 'TA→Enterocyte'

    # ── Hybrid / multipotent DCM with secretory signature ────────
    if dcm_multi and has_secretory(sig) and rna == 'Enterocyte':
        return 'Hybrid(Sec)→Enterocyte'

    return None

paired['group'] = paired.apply(classify, axis=1)
paired = paired[paired['group'].notna()].copy()

print("\nGroup counts (ALL Dox):")
print(paired['group'].value_counts().to_string())

DOX_VALUES = sorted(paired['Dox'].dropna().unique())
print(f"\nDox conditions: {DOX_VALUES}")

# ─────────────────────────────────────────────
# PLOT + STATS SETTINGS
# ─────────────────────────────────────────────
NEG_CTRL  = 'Secretory→Secretory'   # reference for t-tests
POS_CTRL  = 'TA→Enterocyte'

# Order matches the transition_analysis convention:
# neg ctrl first, then specific secretory→ent, hybrid, pos ctrl
GROUP_ORDER = (
    [NEG_CTRL] +
    [f'{s}→Enterocyte' for s in secretory_types] +
    ['Hybrid(Sec)→Enterocyte', POS_CTRL]
)

GROUP_COLORS = {
    NEG_CTRL:                    '#95a5a6',
    'Paneth→Enterocyte':         '#DC143C',
    'Goblet→Enterocyte':         '#2E8B57',
    'Tuft→Enterocyte':           '#FF1493',
    'EEC→Enterocyte':            '#4169E1',
    'Hybrid(Sec)→Enterocyte':    '#FFD700',
    POS_CTRL:                    '#3498db',
}

SCORES  = ['combined_score', 'S_score', 'G2M_score']
YLABELS = ['Combined Cell Cycle Score\n(S + G2M)', 'S Phase Score', 'G2M Phase Score']
TITLES  = ['Combined (S+G2M)', 'S Phase Score', 'G2M Phase Score']

def get_sig(p):
    if np.isnan(p):   return ''
    if p < 0.001:     return '***'
    if p < 0.01:      return '**'
    if p < 0.05:      return '*'
    return 'ns'

# ─────────────────────────────────────────────
# HELPER: compute stats vs NEG_CTRL
# ─────────────────────────────────────────────
def compute_stats(df, group_order, dox_label, context):
    rows     = []
    ref_data = {s: df[df['group'] == NEG_CTRL][s].dropna() for s in SCORES}
    for grp in group_order:
        sub = df[df['group'] == grp]
        for score in SCORES:
            vals = sub[score].dropna()
            ref  = ref_data[score]
            pval = (stats.ttest_ind(vals, ref)[1]
                    if grp != NEG_CTRL and len(vals) > 1 and len(ref) > 1
                    else np.nan)
            rows.append({
                'context':               context,
                'Dox':                   dox_label,
                'group':                 grp,
                'score':                 score,
                'N':                     len(vals),
                'mean':                  vals.mean()   if len(vals) else np.nan,
                'median':                vals.median() if len(vals) else np.nan,
                'std':                   vals.std()    if len(vals) else np.nan,
                'ttest_vs_SecSec_pval':  pval,
                'significance':          get_sig(pval),
            })
    return pd.DataFrame(rows)

# ─────────────────────────────────────────────
# HELPER: draw t-test brackets on one axis
# ─────────────────────────────────────────────
def draw_brackets(ax, df, group_order, score, y_top):
    if NEG_CTRL not in group_order or NEG_CTRL not in df['group'].values:
        return y_top
    ref_vals = df[df['group'] == NEG_CTRL][score].dropna()
    if len(ref_vals) <= 1:
        return y_top

    ref_i    = group_order.index(NEG_CTRL)
    y_levels = [y_top * 1.08, y_top * 1.16]
    y_max    = y_levels[0]

    for k, grp in enumerate(g for g in group_order if g != NEG_CTRL):
        i  = group_order.index(grp)
        gv = df[df['group'] == grp][score].dropna()
        if len(gv) <= 1:
            continue
        _, pval = stats.ttest_ind(gv, ref_vals)
        stars   = get_sig(pval)
        y_lvl   = y_levels[k % 2]
        y_max   = max(y_max, y_lvl)
        x_mid   = (i + ref_i) / 2
        ax.annotate('', xy=(i, y_lvl), xytext=(ref_i, y_lvl),
                    arrowprops=dict(arrowstyle='-', color='#444444', lw=0.9))
        ax.text(x_mid, y_lvl, stars,
                ha='center', va='bottom', fontsize=10, fontweight='bold',
                color='black' if stars not in ('ns', '') else '#888888')
    return y_max

# ─────────────────────────────────────────────
# HELPER: make one boxplot figure
# ─────────────────────────────────────────────
def make_plot(df, title_str, save_path, dox_label, context):
    present = [g for g in GROUP_ORDER if g in df['group'].values]
    if not present:
        print(f"  Skipping — no groups present")
        return pd.DataFrame()

    fig, axes = plt.subplots(1, 3, figsize=(22, 8))

    for ax, score, ttl, ylabel in zip(axes, SCORES, TITLES, YLABELS):
        sns.boxplot(
            data=df, x='group', y=score,
            hue='group', order=present,
            palette=GROUP_COLORS, legend=False, ax=ax,
        )
        ax.set_xlabel('Transition', fontsize=11, fontweight='bold')
        ax.set_ylabel(ylabel,       fontsize=11, fontweight='bold')
        ax.set_title(ttl,           fontsize=12, fontweight='bold')
        ax.tick_params(axis='x', rotation=45, labelsize=9)

        ylim  = ax.get_ylim()
        y_top = ylim[1]

        for i, grp in enumerate(present):
            n = len(df[df['group'] == grp])
            ax.text(i, y_top * 1.01, f'n={n}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

        y_max = draw_brackets(ax, df, present, score, y_top)
        ax.set_ylim(ylim[0], y_max * 1.10)

        if NEG_CTRL in present:
            ax.text(0.01, 0.99, f'— t-test vs {NEG_CTRL}',
                    transform=ax.transAxes, fontsize=6,
                    color='gray', va='top', ha='left')

    plt.suptitle(title_str, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

    return compute_stats(df, present, dox_label, context)

# ─────────────────────────────────────────────
# HELPER: safe filename tag
# ─────────────────────────────────────────────
def dox_tag(dox):
    return str(dox).replace(' ', '_').replace('/', '-')

# ─────────────────────────────────────────────
# STEP 5: GENERATE PLOTS — ALL + per-Dox
# Controls (NEG_CTRL, POS_CTRL) always use ALL
# cells; experimental groups are Dox-filtered.
# ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 5: GENERATING PLOTS")
print("=" * 70)

# Pre-extract control pools (all Dox)
ctrl_neg_all = paired[paired['group'] == NEG_CTRL].copy()
ctrl_pos_all = paired[paired['group'] == POS_CTRL].copy()

all_stats = []

for dox_label in ['ALL'] + list(DOX_VALUES):
    tag = dox_tag(dox_label)
    print(f"\n  Dox = {dox_label}")

    if dox_label == 'ALL':
        exp_df = paired
    else:
        # Experimental groups filtered to this Dox;
        # controls kept from all conditions
        exp_groups = [g for g in GROUP_ORDER
                      if g not in (NEG_CTRL, POS_CTRL)]
        exp_subset = paired[
            (paired['group'].isin(exp_groups)) &
            (paired['Dox'] == dox_label)
        ]
        exp_df = pd.concat([exp_subset, ctrl_neg_all, ctrl_pos_all],
                           ignore_index=True).drop_duplicates()

    _stats = make_plot(
        exp_df,
        title_str = (f'Cell Cycle — Secretory → Enterocyte Transitions | Dox = {dox_label}\n'
                     f'pDCM scored independently | t-tests vs {NEG_CTRL}'),
        save_path = os.path.join(output_dir,
                                 f'cell_cycle_dcm_scored_Dox_{tag}.pdf'),
        dox_label = dox_label,
        context   = 'SecretoryToEnterocyte_DCMscored',
    )
    all_stats.append(_stats)

# ─────────────────────────────────────────────
# STEP 6: SAVE CSV OUTPUTS
# ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 6: SAVING CSV OUTPUTS")
print("=" * 70)

# Stats summary (t-tests vs neg ctrl, all Dox conditions × groups × scores)
stats_df  = pd.concat([s for s in all_stats if not s.empty], ignore_index=True)
stats_csv = os.path.join(output_dir, 'cell_cycle_dcm_scored_stats_vs_SecSec.csv')
stats_df.to_csv(stats_csv, index=False)
print(f"Stats CSV saved : {stats_csv}")

# Full paired data with scores
data_csv = os.path.join(output_dir, 'cell_cycle_dcm_scored_data.csv')
paired.to_csv(data_csv, index=False)
print(f"Data CSV saved  : {data_csv}")

# Gene lists actually used
gene_df = pd.DataFrame({
    'gene':  s_use   + g2m_use,
    'phase': ['S'] * len(s_use) + ['G2M'] * len(g2m_use),
})
gene_csv = os.path.join(output_dir, 'cell_cycle_dcm_scored_genes_used.csv')
gene_df.to_csv(gene_csv, index=False)
print(f"Gene list saved : {gene_csv}")

print("\nDone!")
print(f"\nOutputs in: {output_dir}")

In [ ]:
##############################################
# TRANSITION ANALYSIS — adata_combined
# Full port of the reference code adapted to
# the new adata_combined object.
#
# All needed columns are already in obs:
#   assay, batch_plate_well, leiden_annot_25,
#   ttest_gmm_label_confident, is_multipotent,
#   significant_celltypes, ttest_*_zscore,
#   S_score, G2M_score, phase
#
# NOTE: cell cycle scores are pre-computed —
# sc.tl.score_genes_cell_cycle is NOT re-run.
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns
from scipy import stats
from scipy.stats import gaussian_kde

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
root       = "/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py"
score_dir  = os.path.join(root, 'gene_scoring')
output_dir_umap = os.path.join(score_dir, 'gmm_085/cell_cycle3/umap_transitions')
output_dir_heat = os.path.join(score_dir, 'gmm_085/cell_cycle3/heatmap_transitions')
output_dir_cc   = os.path.join(score_dir, 'gmm_085/cell_cycle3/cellcycle_transitions')
for d in [output_dir_umap, output_dir_heat, output_dir_cc]:
    os.makedirs(d, exist_ok=True)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
cell_type_colors = {
    'Stem'      : '#9370DB',
    'TA'        : '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth'    : '#DC143C',
    'Goblet'    : '#2E8B57',
    'Tuft'      : '#FF1493',
    'EEC'       : '#4169E1',
    'ambiguous' : '#7f7f7f',
    'hybrid'    : '#FFD700',
}

label_order      = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']
secretory_types  = ['Paneth', 'Goblet', 'Tuft', 'EEC']

BW_METHOD  = 0.5
LEVEL_FRAC = 0.08

# ─────────────────────────────────────────────
# STEP 1: BUILD plot_df
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: BUILDING PLOT DATAFRAME")
print("=" * 60)

adata = adata_combined
umap  = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1':                   umap[:, 0],
    'UMAP2':                   umap[:, 1],
    'assay':                   adata.obs['assay'].values,
    'batch_plate_well':        adata.obs['batch_plate_well'].values,
    'leiden_annot_25':         adata.obs['leiden_annot_25'].astype(str).values,
    'ttest_gmm_label_confident': adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'is_multipotent':          adata.obs['is_multipotent'].astype(str).values,
    'significant_celltypes':   adata.obs['significant_celltypes'].astype(str).values,
    'cell_index':              adata.obs.index.values,
}, index=adata.obs_names)

# Strip numeric prefix from leiden/gmm labels (e.g. '3_Enterocyte' → 'Enterocyte')
plot_df['leiden_annot_25']           = plot_df['leiden_annot_25'].str.split('_').str[-1]
plot_df['ttest_gmm_label_confident'] = plot_df['ttest_gmm_label_confident'].str.split('_').str[-1]

# Attach ttest zscore columns
ttest_cols = [c for c in adata.obs.columns
              if c.startswith('ttest_') and c.endswith('_zscore')
              and 'max' not in c.lower()]
for col in ttest_cols:
    plot_df[col] = adata.obs[col].values

print(f"ttest zscore columns: {ttest_cols}")
print(f"Assay values: {sorted(plot_df['assay'].unique())}")

# ─────────────────────────────────────────────
# STEP 2: CELL CYCLE — already pre-scored
# Just pull S_score, G2M_score, phase from obs
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: CELL CYCLE (pre-scored, reading from obs)")
print("=" * 60)

dcm_obs = adata.obs[adata.obs['assay'] == 'pDCM']
cc_df   = pd.DataFrame({
    'batch_plate_well': dcm_obs['batch_plate_well'].values,
    'S_score':          dcm_obs['S_score'].values,
    'G2M_score':        dcm_obs['G2M_score'].values,
    'phase':            dcm_obs['phase'].values,
}, index=dcm_obs.index)
cc_df['combined_score'] = cc_df['S_score'] + cc_df['G2M_score']
print(f"DCM cells for cell cycle: {len(cc_df)}")

# Add combined_score as a new column in adata_combined for all cells
adata.obs['combined_score'] = adata.obs['S_score'].astype(float) + adata.obs['G2M_score'].astype(float)
print(f"Added 'combined_score' to adata.obs (all {adata.n_obs} cells)")

# ─────────────────────────────────────────────
# STEP 3: SEPARATE CLEAR vs HYBRID DCM
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: SEPARATING CLEAR vs HYBRID DCM CELLS")
print("=" * 60)

dcm_clear_mask = (
    (plot_df['assay'] == 'pDCM') &
    (plot_df['ttest_gmm_label_confident'].isin(label_order)) &
    (plot_df['is_multipotent'].str.lower().isin(['false', '0', 'no']))
)
dcm_clear  = plot_df[dcm_clear_mask].copy()

dcm_hybrid_mask = (
    (plot_df['assay'] == 'pDCM') &
    (plot_df['is_multipotent'].str.lower().isin(['true', '1', 'yes']))
)
dcm_hybrid = plot_df[dcm_hybrid_mask].copy()

rna_mask   = (
    (plot_df['assay'] == 'pRNA') &
    (plot_df['leiden_annot_25'].isin(label_order))
)
rna_cells  = plot_df[rna_mask].copy()

print(f"Clear DCM  : {len(dcm_clear)}")
print(f"Hybrid DCM : {len(dcm_hybrid)}")
print(f"RNA cells  : {len(rna_cells)}")

# ─────────────────────────────────────────────
# STEP 4: 1:1 MATCHING
# ─────────────────────────────────────────────
def match_one_to_one(dcm_df, rna_df):
    common = set(dcm_df['batch_plate_well']) & set(rna_df['batch_plate_well'])
    dcm_s  = set(dcm_df[dcm_df['batch_plate_well'].isin(common)]
                 ['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
    rna_s  = set(rna_df[rna_df['batch_plate_well'].isin(common)]
                 ['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
    return dcm_s & rna_s

def build_connections(dcm_df, rna_df, matched_wells, is_hybrid=False):
    rows = []
    for bpw in matched_wells:
        dr = dcm_df[dcm_df['batch_plate_well'] == bpw].iloc[0]
        rr = rna_df[rna_df['batch_plate_well'] == bpw].iloc[0]
        row = {
            'batch_plate_well': bpw,
            'dcm_umap1': dr['UMAP1'], 'dcm_umap2': dr['UMAP2'],
            'rna_umap1': rr['UMAP1'], 'rna_umap2': rr['UMAP2'],
            'dcm_cell_type':      dr['ttest_gmm_label_confident'],
            'rna_cell_type':      rr['leiden_annot_25'],
            'dcm_sig_celltypes':  dr['significant_celltypes'],
            'is_hybrid':          is_hybrid,
            'transition': f"{dr['ttest_gmm_label_confident']} -> {rr['leiden_annot_25']}",
        }
        for col in ttest_cols:
            ct = col.replace('ttest_', '').replace('_zscore', '')
            row[f'dcm_{ct}_zscore'] = dr[col]
            row[f'rna_{ct}_zscore'] = rr[col]
        rows.append(row)
    return pd.DataFrame(rows)

both_clear  = match_one_to_one(dcm_clear,  rna_cells)
both_hybrid = match_one_to_one(dcm_hybrid, rna_cells)
print(f"\nClear 1:1 matched wells : {len(both_clear)}")
print(f"Hybrid 1:1 matched wells: {len(both_hybrid)}")

conn_clear_df  = build_connections(dcm_clear,  rna_cells, both_clear,  is_hybrid=False)
conn_hybrid_df = build_connections(dcm_hybrid, rna_cells, both_hybrid, is_hybrid=True)

# ─────────────────────────────────────────────
# STEP 5: TRANSITION GROUPS
# ─────────────────────────────────────────────
sec_to_ent_clear = conn_clear_df[
    conn_clear_df['dcm_cell_type'].isin(secretory_types) &
    (conn_clear_df['rna_cell_type'] == 'Enterocyte')
].copy()
sec_to_ent_clear['group']      = sec_to_ent_clear['dcm_cell_type'] + '→Enterocyte'
sec_to_ent_clear['cell_class'] = 'clear'

ta_ent_clear = conn_clear_df[
    (conn_clear_df['dcm_cell_type'] == 'TA') &
    (conn_clear_df['rna_cell_type'] == 'Enterocyte')
].copy()
ta_ent_clear['group']      = 'TA→Enterocyte'
ta_ent_clear['cell_class'] = 'clear_control'

def has_secretory(sig_str):
    if pd.isna(sig_str) or str(sig_str) in ['none', 'None', '']:
        return False
    return any(s in str(sig_str) for s in secretory_types)

hybrid_sec_to_ent = conn_hybrid_df[
    conn_hybrid_df['dcm_sig_celltypes'].apply(has_secretory) &
    (conn_hybrid_df['rna_cell_type'] == 'Enterocyte')
].copy()
hybrid_sec_to_ent['group']      = 'Hybrid(Sec)→Enterocyte'
hybrid_sec_to_ent['cell_class'] = 'hybrid_dcm'

secretory_zscore_threshold = 1.5
ct_names = [c.replace('ttest_', '').replace('_zscore', '') for c in ttest_cols]

def rna_is_enterocyte_hybrid(row):
    if row['rna_cell_type'] != 'Enterocyte':
        return False
    return any(row.get(f'rna_{s}_zscore', 0) >= secretory_zscore_threshold
               for s in secretory_types)

sec_to_ent_hybrid_rna = conn_clear_df[
    conn_clear_df['dcm_cell_type'].isin(secretory_types)
].copy()
sec_to_ent_hybrid_rna = sec_to_ent_hybrid_rna[
    sec_to_ent_hybrid_rna.apply(rna_is_enterocyte_hybrid, axis=1)
].copy()
sec_to_ent_hybrid_rna['group']      = sec_to_ent_hybrid_rna['dcm_cell_type'] + '→EntHybrid'
sec_to_ent_hybrid_rna['cell_class'] = 'hybrid_rna'

print(f"\nTransition group sizes:")
print(f"  Clear secretory→Enterocyte     : {len(sec_to_ent_clear)}")
print(f"  TA→Enterocyte (control)        : {len(ta_ent_clear)}")
print(f"  Hybrid DCM sec→Enterocyte      : {len(hybrid_sec_to_ent)}")
print(f"  Clear sec→Enterocyte hybrid RNA: {len(sec_to_ent_hybrid_rna)}")

plot1_cc_clear = pd.concat([sec_to_ent_clear, ta_ent_clear], ignore_index=True)
plot1_cc_clear = plot1_cc_clear.merge(cc_df, on='batch_plate_well', how='left')

# ─────────────────────────────────────────────
# STEP 6: KDE STORE FOR BACKGROUND
# Use all cells for contours (pDCM + pRNA)
# ─────────────────────────────────────────────
ref_mask = plot_df['assay'].isin(['pDCM', 'pRNA'])

def build_kde(ct, bw=BW_METHOD):
    sub = plot_df[ref_mask & (plot_df['leiden_annot_25'] == ct)]
    if len(sub) < 10:
        return None
    return gaussian_kde(np.vstack([sub['UMAP1'].values, sub['UMAP2'].values]),
                        bw_method=bw)

kde_store = {ct: build_kde(ct) for ct in label_order}

# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────
def draw_background(ax):
    ref_df = plot_df[ref_mask]
    for ct in label_order:
        sub = ref_df[ref_df['leiden_annot_25'] == ct]
        if len(sub):
            ax.scatter(sub['UMAP1'], sub['UMAP2'], s=8,
                       c=cell_type_colors[ct], alpha=0.15,
                       marker='o', zorder=1, rasterized=True)
    other = ref_df[~ref_df['leiden_annot_25'].isin(label_order)]
    if len(other):
        ax.scatter(other['UMAP1'], other['UMAP2'], s=8,
                   c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)

    for ct in label_order:
        kde = kde_store.get(ct)
        if kde is None:
            continue
        sub = ref_df[ref_df['leiden_annot_25'] == ct]
        x, y = sub['UMAP1'].values, sub['UMAP2'].values
        xg = np.linspace(x.min()-0.5, x.max()+0.5, 200)
        yg = np.linspace(y.min()-0.5, y.max()+0.5, 200)
        xx, yy = np.meshgrid(xg, yg)
        zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        ax.contour(xx, yy, zz, levels=[zz.max() * LEVEL_FRAC],
                   colors=[cell_type_colors[ct]],
                   linewidths=2, linestyles='--', alpha=0.8, zorder=2)
        peak = np.unravel_index(zz.argmax(), zz.shape)
        ax.text(xx[peak], yy[peak], ct, ha='center', va='center',
                fontsize=10, fontweight='bold', color=cell_type_colors[ct], zorder=5)


def plot_umap(subset_df, title_str, save_path, hybrid_dcm=False):
    if len(subset_df) == 0:
        print(f"  Skipping UMAP — no cells"); return

    fig, ax = plt.subplots(figsize=(12, 10))

    # ── Layer 1: ALL cells coloured by cluster (background) ──
    for ct in label_order:
        all_ct = plot_df[plot_df['leiden_annot_25'] == ct]
        if len(all_ct):
            ax.scatter(all_ct['UMAP1'], all_ct['UMAP2'],
                       s=6, c=cell_type_colors.get(ct, '#7f7f7f'),
                       alpha=0.15, marker='o', zorder=1, rasterized=True)
    other_all = plot_df[~plot_df['leiden_annot_25'].isin(label_order)]
    if len(other_all):
        ax.scatter(other_all['UMAP1'], other_all['UMAP2'],
                   s=6, c='lightgray', alpha=0.08,
                   marker='o', zorder=1, rasterized=True)

    # ── Layer 2: KDE contours + labels ──
    for ct in label_order:
        kde = kde_store.get(ct)
        if kde is None:
            continue
        sub = plot_df[plot_df['leiden_annot_25'] == ct]
        x, y = sub['UMAP1'].values, sub['UMAP2'].values
        xg = np.linspace(x.min()-0.5, x.max()+0.5, 200)
        yg = np.linspace(y.min()-0.5, y.max()+0.5, 200)
        xx, yy = np.meshgrid(xg, yg)
        zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        ax.contour(xx, yy, zz, levels=[zz.max() * LEVEL_FRAC],
                   colors=[cell_type_colors[ct]],
                   linewidths=2, linestyles='--', alpha=0.8, zorder=2)
        peak = np.unravel_index(zz.argmax(), zz.shape)
        ax.text(xx[peak], yy[peak], ct, ha='center', va='center',
                fontsize=10, fontweight='bold', color=cell_type_colors[ct], zorder=5)

    # ── Layer 3: Transition arrows (colour = RNA destination) ──
    all_bpw = set(subset_df['batch_plate_well'])
    for _, conn in subset_df.iterrows():
        color = cell_type_colors.get(conn['rna_cell_type'], 'black')
        ax.add_patch(FancyArrowPatch(
            (conn['dcm_umap1'], conn['dcm_umap2']),
            (conn['rna_umap1'], conn['rna_umap2']),
            arrowstyle='-|>', mutation_scale=15,
            color=color, alpha=0.7, linewidth=1.5, zorder=3,
        ))

    # ── Layer 4: Highlighted DCM cells (squares) ──
    for ct in label_order:
        if hybrid_dcm:
            pts = plot_df[(plot_df['assay'] == 'pDCM') &
                          (plot_df['batch_plate_well'].isin(all_bpw)) &
                          (plot_df['is_multipotent'].str.lower().isin(['true','1','yes']))]
            c, m = cell_type_colors['hybrid'], 'D'
        else:
            pts = plot_df[(plot_df['assay'] == 'pDCM') &
                          (plot_df['batch_plate_well'].isin(all_bpw)) &
                          (plot_df['ttest_gmm_label_confident'] == ct)]
            c, m = cell_type_colors.get(ct, '#7f7f7f'), 's'
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100, c=c, alpha=1.0,
                       edgecolors='black', linewidths=1.5,
                       marker=m, zorder=4, rasterized=True)
        if hybrid_dcm:
            break

    # ── Layer 5: Highlighted RNA cells (circles) ──
    dest_cts = subset_df['rna_cell_type'].unique()
    for ct in dest_cts:
        pts = plot_df[(plot_df['assay'] == 'pRNA') &
                      (plot_df['batch_plate_well'].isin(all_bpw)) &
                      (plot_df['leiden_annot_25'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, '#7f7f7f'), alpha=1.0,
                       edgecolors='black', linewidths=1.5,
                       marker='o', zorder=4, rasterized=True)

    # ── Legend ──
    legend = []
    # All-cells background entries
    for ct in label_order:
        legend.append(mlines.Line2D([], [], color=cell_type_colors.get(ct, '#7f7f7f'),
                                     marker='o', linestyle='None', markersize=7,
                                     alpha=0.4, label=f'All cells ({ct})'))
    if hybrid_dcm:
        legend.append(mlines.Line2D([], [], color=cell_type_colors['hybrid'],
                                     marker='D', linestyle='None', markersize=10,
                                     markeredgecolor='black', label='DCM Hybrid'))
    else:
        for ct in subset_df['dcm_cell_type'].unique():
            legend.append(mlines.Line2D([], [], color=cell_type_colors.get(ct,'gray'),
                                         marker='s', linestyle='None', markersize=10,
                                         markeredgecolor='black', label=f'DCM {ct}'))
    for ct in dest_cts:
        c = cell_type_colors.get(ct, '#7f7f7f')
        legend += [
            mlines.Line2D([], [], color=c, marker='o', linestyle='None',
                           markersize=10, markeredgecolor='black', label=f'RNA {ct}'),
            mlines.Line2D([], [], color=c, linestyle='-', linewidth=2,
                           alpha=0.8, marker='>', markersize=8, label=f'→ {ct}'),
        ]
    ax.legend(handles=legend, loc='center left',
              bbox_to_anchor=(1, 0.5), frameon=True, fontsize=8)
    ax.set_title(title_str, fontsize=13, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


def plot_umap_cellcycle(title_str, save_path):
    """
    UMAP coloured by combined cell cycle score (S + G2M) for ALL cells.
    Scores are read directly from adata_combined.obs which has
    S_score, G2M_score, phase pre-computed for every cell.
    """
    # Use the pre-computed combined_score column added to adata.obs
    scores_aligned = (adata.obs.reindex(plot_df.index)['combined_score']
                      .fillna(0).values)

    fig, ax = plt.subplots(figsize=(12, 10))

    sc = ax.scatter(
        plot_df['UMAP1'], plot_df['UMAP2'],
        s=8, c=scores_aligned, cmap='RdYlBu_r',
        vmin=np.percentile(scores_aligned, 2),
        vmax=np.percentile(scores_aligned, 98),
        alpha=0.7, rasterized=True, zorder=2,
    )
    cbar = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02, shrink=0.6)
    cbar.set_label('Combined cell cycle score (S + G2M)', fontsize=9)

    ax.set_title(title_str, fontsize=13, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


def plot_heatmap(subset_df, title_str, save_path, group_col='dcm_cell_type'):
    if len(subset_df) == 0:
        print(f"  Skipping heatmap — no cells"); return

    norm_dcm  = TwoSlopeNorm(vmin=-2.5, vcenter=1.0, vmax=3.5)  # DCM: active threshold ~1
    norm_rna  = TwoSlopeNorm(vmin=-2.5, vcenter=0.0, vmax=3.5)  # RNA: centred on 0
    n_cells   = len(subset_df)
    fig_width = max(14, n_cells * 0.18)

    df = subset_df.copy()
    df['_sort'] = df[group_col].map({ct: i for i, ct in enumerate(label_order)}).fillna(99)
    df['_self_z'] = df.apply(
        lambda r: r.get(f'dcm_{r["dcm_cell_type"]}_zscore', 0)
        if r['dcm_cell_type'] in ct_names else 0, axis=1
    )
    df = df.sort_values(['_sort', '_self_z'], ascending=[True, False]).reset_index(drop=True)

    dcm_mat = np.array([[row.get(f'dcm_{ct}_zscore', np.nan) for ct in ct_names]
                         for _, row in df.iterrows()]).T
    rna_mat = np.array([[row.get(f'rna_{ct}_zscore', np.nan) for ct in ct_names]
                         for _, row in df.iterrows()]).T

    fig, axes = plt.subplots(2, 1, figsize=(fig_width, 10))
    for ax, matrix, label, norm, vcenter_val in [
        (axes[0], dcm_mat, 'DCM (GMM label)',         norm_dcm, 1.0),
        (axes[1], rna_mat, 'RNA (leiden_annot_25)',    norm_rna, 0.0),
    ]:
        im = ax.imshow(matrix, aspect='auto', cmap=plt.cm.RdBu_r, norm=norm)
        ax.set_yticks(range(len(ct_names)))
        ax.set_yticklabels(ct_names, fontsize=9)
        ax.set_xticks([])
        ax.set_ylabel('Module', fontsize=11, fontweight='bold')
        ax.set_title(label, fontsize=12, fontweight='bold')

        changes = df[group_col].ne(df[group_col].shift()).to_numpy().nonzero()[0]
        for x in changes:
            ax.axvline(x - 0.5, color='white', lw=1.5, alpha=0.8)
        prev = 0
        for x in list(changes) + [n_cells]:
            mid = (prev + x) / 2
            ct  = df.iloc[prev][group_col]
            ax.text(mid, -0.8, ct, ha='center', va='top', fontsize=7,
                    fontweight='bold', color=cell_type_colors.get(ct, 'black'))
            prev = x
        cbar = plt.colorbar(im, ax=ax, fraction=0.01, pad=0.01)
        cbar.set_label('Z-score', fontsize=9)
        cbar.ax.axhline(y=vcenter_val, color='black', lw=1, ls='--',
                        label=f'vcenter={vcenter_val}')

    fig.suptitle(title_str, fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


def get_significance(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'


def plot_cellcycle(df, group_col, group_order, palette,
                   control_group, title_str, save_path):
    if df.empty:
        print(f"  Skipping cell cycle — no cells"); return
    group_order = [g for g in group_order if g in df[group_col].values]
    if not group_order:
        return

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    scores  = ['combined_score', 'S_score', 'G2M_score']
    titles  = ['Combined (S+G2M)', 'S Phase Score', 'G2M Phase Score']
    ylabels = ['Combined Cell Cycle Score\n(S + G2M)', 'S Phase Score', 'G2M Phase Score']
    ctrl_vals = (df[df[group_col] == control_group]['combined_score']
                 if control_group in df[group_col].values else pd.Series(dtype=float))

    for ax, score, ttl, ylabel in zip(axes, scores, titles, ylabels):
        sns.boxplot(data=df, x=group_col, y=score,
                    order=group_order, palette=palette, ax=ax)
        ax.set_xlabel('Transition', fontsize=11, fontweight='bold')
        ax.set_ylabel(ylabel,       fontsize=11, fontweight='bold')
        ax.set_title(ttl,           fontsize=12, fontweight='bold')
        ax.tick_params(axis='x', rotation=45, labelsize=9)
        if score == 'combined_score':
            ymax = ax.get_ylim()[1]
            for i, grp in enumerate(group_order):
                n = len(df[df[group_col] == grp])
                ax.text(i, ymax * 0.97, f'n={n}', ha='center', fontsize=8, fontweight='bold')
                if grp != control_group and len(ctrl_vals) > 0:
                    gv = df[df[group_col] == grp][score]
                    if len(gv) > 1:
                        _, pval = stats.ttest_ind(gv, ctrl_vals)
                        if pval < 0.05:
                            ax.text(i, ymax * 0.89, get_significance(pval),
                                    ha='center', fontsize=12, fontweight='bold')
    plt.suptitle(title_str, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

# ─────────────────────────────────────────────
# STEP 7: CLEAR SECRETORY → ENTEROCYTE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: CLEAR SECRETORY → ENTEROCYTE")
print("=" * 60)

plot_umap(
    sec_to_ent_clear,
    title_str = f'Clear secretory → Enterocyte  (n={len(sec_to_ent_clear)})\nArrow colour = RNA destination',
    save_path = os.path.join(output_dir_umap, 'umap_clear_secretory_to_enterocyte.pdf'),
)
plot_umap_cellcycle(
    title_str = 'All cells — cell cycle score',
    save_path = os.path.join(output_dir_umap, 'umap_cellcycle_all_cells.pdf'),
)
# ── Combined heatmap: secretory→Enterocyte + TA→Enterocyte together ──
# Pool sec_to_ent_clear and ta_ent_clear; cells are sorted by dcm_cell_type
# so secretory groups appear first, then TA (positive control) at the end.
sec_plus_ta = pd.concat([sec_to_ent_clear, ta_ent_clear], ignore_index=True)
plot_heatmap(
    sec_plus_ta,
    title_str = (f'Z-score — secretory->Enterocyte + TA->Enterocyte  '
                 f'(n={len(sec_to_ent_clear)} + {len(ta_ent_clear)})'),
    save_path = os.path.join(output_dir_heat, 'heatmap_secretory_and_TA_to_enterocyte.pdf'),
    group_col = 'dcm_cell_type',
)

sec_to_sec = conn_clear_df[
    conn_clear_df['dcm_cell_type'].isin(secretory_types) &
    conn_clear_df['rna_cell_type'].isin(secretory_types)
].copy()
sec_to_sec['group'] = 'Secretory->Secretory'

# -- TA->Enterocyte UMAP (positive control) --
plot_umap(
    ta_ent_clear,
    title_str = 'TA -> Enterocyte (positive control, n=' + str(len(ta_ent_clear)) + ')\nArrow colour = RNA destination',
    save_path = os.path.join(output_dir_umap, 'umap_TA_to_enterocyte.pdf'),
)

# -- Secretory->Secretory UMAP + heatmap (negative control) --
plot_umap(
    sec_to_sec,
    title_str = 'Secretory -> Secretory (negative control, n=' + str(len(sec_to_sec)) + ')\nArrow colour = RNA destination',
    save_path = os.path.join(output_dir_umap, 'umap_secretory_to_secretory.pdf'),
)
plot_heatmap(
    sec_to_sec,
    title_str = 'Z-score -- Secretory -> Secretory (negative control, n=' + str(len(sec_to_sec)) + ')',
    save_path = os.path.join(output_dir_heat, 'heatmap_secretory_to_secretory.pdf'),
    group_col = 'dcm_cell_type',
)

cc_combined = pd.concat([
    plot1_cc_clear[plot1_cc_clear['group'].isin(
        [f'{s}→Enterocyte' for s in secretory_types] + ['TA→Enterocyte']
    )],
    sec_to_sec.merge(cc_df, on='batch_plate_well', how='left'),
    hybrid_sec_to_ent.merge(cc_df, on='batch_plate_well', how='left')
    .assign(group='Hybrid(Sec)→Enterocyte'),
], ignore_index=True)

cc_order = (['Secretory→Secretory'] +
            [f'{s}→Enterocyte' for s in secretory_types] +
            ['Hybrid(Sec)→Enterocyte', 'TA→Enterocyte'])
cc_palette = {
    'Secretory→Secretory':    '#95a5a6',
    'Paneth→Enterocyte':      '#DC143C',
    'Goblet→Enterocyte':      '#2E8B57',
    'Tuft→Enterocyte':        '#FF1493',
    'EEC→Enterocyte':         '#4169E1',
    'Hybrid(Sec)→Enterocyte': '#FFD700',
    'TA→Enterocyte':          '#3498db',
}
plot_cellcycle(
    cc_combined, 'group', cc_order, cc_palette, 'TA→Enterocyte',
    title_str = ('Cell Cycle — Secretory → Enterocyte\n'
                 'Neg ctrl: Secretory→Secretory  |  Pos ctrl: TA→Enterocyte'),
    save_path = os.path.join(output_dir_cc, 'cellcycle_secretory_to_enterocyte_full.pdf'),
)

# ─────────────────────────────────────────────
# STEP 8: HYBRID DCM → ENTEROCYTE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: HYBRID DCM → ENTEROCYTE")
print("=" * 60)

plot_umap(
    hybrid_sec_to_ent,
    title_str = f'Hybrid DCM (secretory) → Enterocyte  (n={len(hybrid_sec_to_ent)})',
    save_path = os.path.join(output_dir_umap, 'umap_hybrid_dcm_to_enterocyte.pdf'),
    hybrid_dcm=True,
)

plot_heatmap(
    hybrid_sec_to_ent,
    title_str = f'Z-score — Hybrid DCM → Enterocyte  (n={len(hybrid_sec_to_ent)})',
    save_path = os.path.join(output_dir_heat, 'heatmap_hybrid_dcm_to_enterocyte.pdf'),
)
cc_hybrid = pd.concat([
    hybrid_sec_to_ent.merge(cc_df, on='batch_plate_well', how='left')
    .assign(group='Hybrid(Sec)→Enterocyte'),
    plot1_cc_clear[plot1_cc_clear['group'] == 'TA→Enterocyte'],
], ignore_index=True)
plot_cellcycle(
    cc_hybrid, 'group',
    ['Hybrid(Sec)→Enterocyte', 'TA→Enterocyte'],
    {'Hybrid(Sec)→Enterocyte': '#FFD700', 'TA→Enterocyte': '#3498db'},
    'TA→Enterocyte',
    'Cell Cycle — Hybrid DCM (secretory) → Enterocyte',
    os.path.join(output_dir_cc, 'cellcycle_hybrid_dcm_to_enterocyte.pdf'),
)

# ─────────────────────────────────────────────
# STEP 9: CLEAR SECRETORY → ENTEROCYTE HYBRID RNA
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 9: CLEAR SECRETORY → ENTEROCYTE HYBRID RNA")
print("=" * 60)

plot_umap(
    sec_to_ent_hybrid_rna,
    title_str = (f'Clear Secretory → Enterocyte-hybrid RNA  (n={len(sec_to_ent_hybrid_rna)})\n'
                 f'RNA Enterocyte with secretory z-score ≥ {secretory_zscore_threshold}'),
    save_path = os.path.join(output_dir_umap, 'umap_clear_secretory_to_hybrid_rna_enterocyte.pdf'),
)

plot_heatmap(
    sec_to_ent_hybrid_rna,
    title_str = f'Z-score — Clear Secretory → Enterocyte-hybrid RNA  (n={len(sec_to_ent_hybrid_rna)})',
    save_path = os.path.join(output_dir_heat, 'heatmap_clear_secretory_to_hybrid_rna_enterocyte.pdf'),
)
cc_hrna_palette = {f'{s}→EntHybrid': cell_type_colors[s] for s in secretory_types}
cc_hrna_palette['TA→Enterocyte'] = '#3498db'
cc_hrna = pd.concat([
    sec_to_ent_hybrid_rna.merge(cc_df, on='batch_plate_well', how='left'),
    plot1_cc_clear[plot1_cc_clear['group'] == 'TA→Enterocyte'],
], ignore_index=True)
plot_cellcycle(
    cc_hrna, 'group',
    [f'{s}→EntHybrid' for s in secretory_types] + ['TA→Enterocyte'],
    cc_hrna_palette, 'TA→Enterocyte',
    (f'Cell Cycle — Clear Secretory → Enterocyte-hybrid RNA\n'
     f'RNA Enterocyte with secretory z-score ≥ {secretory_zscore_threshold}'),
    os.path.join(output_dir_cc, 'cellcycle_clear_secretory_to_hybrid_rna_enterocyte.pdf'),
)

# ─────────────────────────────────────────────
# SAVE TABLES
# ─────────────────────────────────────────────
conn_clear_df.to_csv(os.path.join(score_dir, 'connections_clear.csv'),          index=False)
conn_hybrid_df.to_csv(os.path.join(score_dir, 'connections_hybrid.csv'),         index=False)
sec_to_ent_clear.to_csv(os.path.join(score_dir, 'sec_to_ent_clear.csv'),         index=False)
hybrid_sec_to_ent.to_csv(os.path.join(score_dir, 'hybrid_dcm_to_enterocyte.csv'),index=False)
sec_to_ent_hybrid_rna.to_csv(os.path.join(score_dir, 'sec_to_hybrid_rna.csv'),   index=False)
print("\nAll tables saved")
print(f"\nOutputs in:\n  {output_dir_umap}\n  {output_dir_heat}\n  {output_dir_cc}")

In [ ]:
##############################################
# TRANSITION ANALYSIS — adata_combined
# Full port of the reference code adapted to
# the new adata_combined object.
#
# All needed columns are already in obs:
#   assay, batch_plate_well, leiden_annot_25,
#   ttest_gmm_label_confident, is_multipotent,
#   significant_celltypes, ttest_*_zscore,
#   S_score, G2M_score, phase
#
# NOTE: cell cycle scores are pre-computed —
# sc.tl.score_genes_cell_cycle is NOT re-run.
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns
from scipy import stats
from scipy.stats import gaussian_kde

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
root       = "/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py"
score_dir  = os.path.join(root, 'gene_scoring')
output_dir_umap = os.path.join(score_dir, 'gmm_085/cell_cycle3/umap_transitions')
output_dir_heat = os.path.join(score_dir, 'gmm_085/cell_cycle3/heatmap_transitions')
output_dir_cc   = os.path.join(score_dir, 'gmm_085/cell_cycle3/cellcycle_transitions')
for d in [output_dir_umap, output_dir_heat, output_dir_cc]:
    os.makedirs(d, exist_ok=True)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
cell_type_colors = {
    'Stem'      : '#9370DB',
    'TA'        : '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth'    : '#DC143C',
    'Goblet'    : '#2E8B57',
    'Tuft'      : '#FF1493',
    'EEC'       : '#4169E1',
    'ambiguous' : '#7f7f7f',
    'hybrid'    : '#FFD700',
}

label_order      = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']
secretory_types  = ['Paneth', 'Goblet', 'Tuft', 'EEC']

BW_METHOD  = 0.5
LEVEL_FRAC = 0.08

# ─────────────────────────────────────────────
# STEP 1: BUILD plot_df
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: BUILDING PLOT DATAFRAME")
print("=" * 60)

adata = adata_combined
umap  = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1':                   umap[:, 0],
    'UMAP2':                   umap[:, 1],
    'assay':                   adata.obs['assay'].values,
    'batch_plate_well':        adata.obs['batch_plate_well'].values,
    'leiden_annot_25':         adata.obs['leiden_annot_25'].astype(str).values,
    'ttest_gmm_label_confident': adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'is_multipotent':          adata.obs['is_multipotent'].astype(str).values,
    'significant_celltypes':   adata.obs['significant_celltypes'].astype(str).values,
    'cell_index':              adata.obs.index.values,
}, index=adata.obs_names)

# Strip numeric prefix from leiden/gmm labels (e.g. '3_Enterocyte' → 'Enterocyte')
plot_df['leiden_annot_25']           = plot_df['leiden_annot_25'].str.split('_').str[-1]
plot_df['ttest_gmm_label_confident'] = plot_df['ttest_gmm_label_confident'].str.split('_').str[-1]

# Attach ttest zscore columns
ttest_cols = [c for c in adata.obs.columns
              if c.startswith('ttest_') and c.endswith('_zscore')
              and 'max' not in c.lower()]
for col in ttest_cols:
    plot_df[col] = adata.obs[col].values

print(f"ttest zscore columns: {ttest_cols}")
print(f"Assay values: {sorted(plot_df['assay'].unique())}")

# ─────────────────────────────────────────────
# STEP 2: CELL CYCLE — already pre-scored
# Just pull S_score, G2M_score, phase from obs
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: CELL CYCLE (pre-scored, reading from obs)")
print("=" * 60)

dcm_obs = adata.obs[adata.obs['assay'] == 'pDCM']
cc_df   = pd.DataFrame({
    'batch_plate_well': dcm_obs['batch_plate_well'].values,
    'S_score':          dcm_obs['S_score'].values,
    'G2M_score':        dcm_obs['G2M_score'].values,
    'phase':            dcm_obs['phase'].values,
}, index=dcm_obs.index)
cc_df['combined_score'] = cc_df['S_score'] + cc_df['G2M_score']
print(f"DCM cells for cell cycle: {len(cc_df)}")

# Add combined_score as a new column in adata_combined for all cells
adata.obs['combined_score'] = adata.obs['S_score'].astype(float) + adata.obs['G2M_score'].astype(float)
print(f"Added 'combined_score' to adata.obs (all {adata.n_obs} cells)")

# ─────────────────────────────────────────────
# STEP 3: SEPARATE CLEAR vs HYBRID DCM
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: SEPARATING CLEAR vs HYBRID DCM CELLS")
print("=" * 60)

dcm_clear_mask = (
    (plot_df['assay'] == 'pDCM') &
    (plot_df['ttest_gmm_label_confident'].isin(label_order)) &
    (plot_df['is_multipotent'].str.lower().isin(['false', '0', 'no']))
)
dcm_clear  = plot_df[dcm_clear_mask].copy()

dcm_hybrid_mask = (
    (plot_df['assay'] == 'pDCM') &
    (plot_df['is_multipotent'].str.lower().isin(['true', '1', 'yes']))
)
dcm_hybrid = plot_df[dcm_hybrid_mask].copy()

rna_mask   = (
    (plot_df['assay'] == 'pRNA') &
    (plot_df['leiden_annot_25'].isin(label_order))
)
rna_cells  = plot_df[rna_mask].copy()

print(f"Clear DCM  : {len(dcm_clear)}")
print(f"Hybrid DCM : {len(dcm_hybrid)}")
print(f"RNA cells  : {len(rna_cells)}")

# ─────────────────────────────────────────────
# STEP 4: 1:1 MATCHING
# ─────────────────────────────────────────────
def match_one_to_one(dcm_df, rna_df):
    common = set(dcm_df['batch_plate_well']) & set(rna_df['batch_plate_well'])
    dcm_s  = set(dcm_df[dcm_df['batch_plate_well'].isin(common)]
                 ['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
    rna_s  = set(rna_df[rna_df['batch_plate_well'].isin(common)]
                 ['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
    return dcm_s & rna_s

def build_connections(dcm_df, rna_df, matched_wells, is_hybrid=False):
    rows = []
    for bpw in matched_wells:
        dr = dcm_df[dcm_df['batch_plate_well'] == bpw].iloc[0]
        rr = rna_df[rna_df['batch_plate_well'] == bpw].iloc[0]
        row = {
            'batch_plate_well': bpw,
            'dcm_umap1': dr['UMAP1'], 'dcm_umap2': dr['UMAP2'],
            'rna_umap1': rr['UMAP1'], 'rna_umap2': rr['UMAP2'],
            'dcm_cell_type':      dr['ttest_gmm_label_confident'],
            'rna_cell_type':      rr['leiden_annot_25'],
            'dcm_sig_celltypes':  dr['significant_celltypes'],
            'is_hybrid':          is_hybrid,
            'transition': f"{dr['ttest_gmm_label_confident']} -> {rr['leiden_annot_25']}",
        }
        for col in ttest_cols:
            ct = col.replace('ttest_', '').replace('_zscore', '')
            row[f'dcm_{ct}_zscore'] = dr[col]
            row[f'rna_{ct}_zscore'] = rr[col]
        rows.append(row)
    return pd.DataFrame(rows)

both_clear  = match_one_to_one(dcm_clear,  rna_cells)
both_hybrid = match_one_to_one(dcm_hybrid, rna_cells)
print(f"\nClear 1:1 matched wells : {len(both_clear)}")
print(f"Hybrid 1:1 matched wells: {len(both_hybrid)}")

conn_clear_df  = build_connections(dcm_clear,  rna_cells, both_clear,  is_hybrid=False)
conn_hybrid_df = build_connections(dcm_hybrid, rna_cells, both_hybrid, is_hybrid=True)

# ─────────────────────────────────────────────
# STEP 5: TRANSITION GROUPS
# ─────────────────────────────────────────────
sec_to_ent_clear = conn_clear_df[
    conn_clear_df['dcm_cell_type'].isin(secretory_types) &
    (conn_clear_df['rna_cell_type'] == 'Enterocyte')
].copy()
sec_to_ent_clear['group']      = sec_to_ent_clear['dcm_cell_type'] + '→Enterocyte'
sec_to_ent_clear['cell_class'] = 'clear'

ta_ent_clear = conn_clear_df[
    (conn_clear_df['dcm_cell_type'] == 'TA') &
    (conn_clear_df['rna_cell_type'] == 'Enterocyte')
].copy()
ta_ent_clear['group']      = 'TA→Enterocyte'
ta_ent_clear['cell_class'] = 'clear_control'

def has_secretory(sig_str):
    if pd.isna(sig_str) or str(sig_str) in ['none', 'None', '']:
        return False
    return any(s in str(sig_str) for s in secretory_types)

hybrid_sec_to_ent = conn_hybrid_df[
    conn_hybrid_df['dcm_sig_celltypes'].apply(has_secretory) &
    (conn_hybrid_df['rna_cell_type'] == 'Enterocyte')
].copy()
hybrid_sec_to_ent['group']      = 'Hybrid(Sec)→Enterocyte'
hybrid_sec_to_ent['cell_class'] = 'hybrid_dcm'

secretory_zscore_threshold = 1.5
ct_names = [c.replace('ttest_', '').replace('_zscore', '') for c in ttest_cols]

def rna_is_enterocyte_hybrid(row):
    if row['rna_cell_type'] != 'Enterocyte':
        return False
    return any(row.get(f'rna_{s}_zscore', 0) >= secretory_zscore_threshold
               for s in secretory_types)

sec_to_ent_hybrid_rna = conn_clear_df[
    conn_clear_df['dcm_cell_type'].isin(secretory_types)
].copy()
sec_to_ent_hybrid_rna = sec_to_ent_hybrid_rna[
    sec_to_ent_hybrid_rna.apply(rna_is_enterocyte_hybrid, axis=1)
].copy()
sec_to_ent_hybrid_rna['group']      = sec_to_ent_hybrid_rna['dcm_cell_type'] + '→EntHybrid'
sec_to_ent_hybrid_rna['cell_class'] = 'hybrid_rna'

print(f"\nTransition group sizes:")
print(f"  Clear secretory→Enterocyte     : {len(sec_to_ent_clear)}")
print(f"  TA→Enterocyte (control)        : {len(ta_ent_clear)}")
print(f"  Hybrid DCM sec→Enterocyte      : {len(hybrid_sec_to_ent)}")
print(f"  Clear sec→Enterocyte hybrid RNA: {len(sec_to_ent_hybrid_rna)}")

plot1_cc_clear = pd.concat([sec_to_ent_clear, ta_ent_clear], ignore_index=True)
plot1_cc_clear = plot1_cc_clear.merge(cc_df, on='batch_plate_well', how='left')

# ─────────────────────────────────────────────
# STEP 6: KDE STORE FOR BACKGROUND
# Use all cells for contours (pDCM + pRNA)
# ─────────────────────────────────────────────
ref_mask = plot_df['assay'].isin(['pDCM', 'pRNA'])

def build_kde(ct, bw=BW_METHOD):
    sub = plot_df[ref_mask & (plot_df['leiden_annot_25'] == ct)]
    if len(sub) < 10:
        return None
    return gaussian_kde(np.vstack([sub['UMAP1'].values, sub['UMAP2'].values]),
                        bw_method=bw)

kde_store = {ct: build_kde(ct) for ct in label_order}

# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────
def draw_background(ax):
    ref_df = plot_df[ref_mask]
    for ct in label_order:
        sub = ref_df[ref_df['leiden_annot_25'] == ct]
        if len(sub):
            ax.scatter(sub['UMAP1'], sub['UMAP2'], s=8,
                       c=cell_type_colors[ct], alpha=0.15,
                       marker='o', zorder=1, rasterized=True)
    other = ref_df[~ref_df['leiden_annot_25'].isin(label_order)]
    if len(other):
        ax.scatter(other['UMAP1'], other['UMAP2'], s=8,
                   c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)

    for ct in label_order:
        kde = kde_store.get(ct)
        if kde is None:
            continue
        sub = ref_df[ref_df['leiden_annot_25'] == ct]
        x, y = sub['UMAP1'].values, sub['UMAP2'].values
        xg = np.linspace(x.min()-0.5, x.max()+0.5, 200)
        yg = np.linspace(y.min()-0.5, y.max()+0.5, 200)
        xx, yy = np.meshgrid(xg, yg)
        zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        ax.contour(xx, yy, zz, levels=[zz.max() * LEVEL_FRAC],
                   colors=[cell_type_colors[ct]],
                   linewidths=2, linestyles='--', alpha=0.8, zorder=2)
        peak = np.unravel_index(zz.argmax(), zz.shape)
        ax.text(xx[peak], yy[peak], ct, ha='center', va='center',
                fontsize=10, fontweight='bold', color=cell_type_colors[ct], zorder=5)


def plot_umap(subset_df, title_str, save_path, hybrid_dcm=False):
    if len(subset_df) == 0:
        print(f"  Skipping UMAP — no cells"); return

    fig, ax = plt.subplots(figsize=(12, 10))

    # ── Layer 1: ALL cells coloured by cluster (background) ──
    for ct in label_order:
        all_ct = plot_df[plot_df['leiden_annot_25'] == ct]
        if len(all_ct):
            ax.scatter(all_ct['UMAP1'], all_ct['UMAP2'],
                       s=6, c=cell_type_colors.get(ct, '#7f7f7f'),
                       alpha=0.15, marker='o', zorder=1, rasterized=True)
    other_all = plot_df[~plot_df['leiden_annot_25'].isin(label_order)]
    if len(other_all):
        ax.scatter(other_all['UMAP1'], other_all['UMAP2'],
                   s=6, c='lightgray', alpha=0.08,
                   marker='o', zorder=1, rasterized=True)

    # ── Layer 2: KDE contours + labels ──
    for ct in label_order:
        kde = kde_store.get(ct)
        if kde is None:
            continue
        sub = plot_df[plot_df['leiden_annot_25'] == ct]
        x, y = sub['UMAP1'].values, sub['UMAP2'].values
        xg = np.linspace(x.min()-0.5, x.max()+0.5, 200)
        yg = np.linspace(y.min()-0.5, y.max()+0.5, 200)
        xx, yy = np.meshgrid(xg, yg)
        zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        ax.contour(xx, yy, zz, levels=[zz.max() * LEVEL_FRAC],
                   colors=[cell_type_colors[ct]],
                   linewidths=2, linestyles='--', alpha=0.8, zorder=2)
        peak = np.unravel_index(zz.argmax(), zz.shape)
        ax.text(xx[peak], yy[peak], ct, ha='center', va='center',
                fontsize=10, fontweight='bold', color=cell_type_colors[ct], zorder=5)

    # ── Layer 3: Transition arrows (colour = RNA destination) ──
    all_bpw = set(subset_df['batch_plate_well'])
    for _, conn in subset_df.iterrows():
        color = cell_type_colors.get(conn['rna_cell_type'], 'black')
        ax.add_patch(FancyArrowPatch(
            (conn['dcm_umap1'], conn['dcm_umap2']),
            (conn['rna_umap1'], conn['rna_umap2']),
            arrowstyle='-|>', mutation_scale=15,
            color=color, alpha=0.7, linewidth=1.5, zorder=3,
        ))

    # ── Layer 4: Highlighted DCM cells (squares) ──
    for ct in label_order:
        if hybrid_dcm:
            pts = plot_df[(plot_df['assay'] == 'pDCM') &
                          (plot_df['batch_plate_well'].isin(all_bpw)) &
                          (plot_df['is_multipotent'].str.lower().isin(['true','1','yes']))]
            c, m = cell_type_colors['hybrid'], 'D'
        else:
            pts = plot_df[(plot_df['assay'] == 'pDCM') &
                          (plot_df['batch_plate_well'].isin(all_bpw)) &
                          (plot_df['ttest_gmm_label_confident'] == ct)]
            c, m = cell_type_colors.get(ct, '#7f7f7f'), 's'
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100, c=c, alpha=1.0,
                       edgecolors='black', linewidths=1.5,
                       marker=m, zorder=4, rasterized=True)
        if hybrid_dcm:
            break

    # ── Layer 5: Highlighted RNA cells (circles) ──
    dest_cts = subset_df['rna_cell_type'].unique()
    for ct in dest_cts:
        pts = plot_df[(plot_df['assay'] == 'pRNA') &
                      (plot_df['batch_plate_well'].isin(all_bpw)) &
                      (plot_df['leiden_annot_25'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, '#7f7f7f'), alpha=1.0,
                       edgecolors='black', linewidths=1.5,
                       marker='o', zorder=4, rasterized=True)

    # ── Legend ──
    legend = []
    # All-cells background entries
    for ct in label_order:
        legend.append(mlines.Line2D([], [], color=cell_type_colors.get(ct, '#7f7f7f'),
                                     marker='o', linestyle='None', markersize=7,
                                     alpha=0.4, label=f'All cells ({ct})'))
    if hybrid_dcm:
        legend.append(mlines.Line2D([], [], color=cell_type_colors['hybrid'],
                                     marker='D', linestyle='None', markersize=10,
                                     markeredgecolor='black', label='DCM Hybrid'))
    else:
        for ct in subset_df['dcm_cell_type'].unique():
            legend.append(mlines.Line2D([], [], color=cell_type_colors.get(ct,'gray'),
                                         marker='s', linestyle='None', markersize=10,
                                         markeredgecolor='black', label=f'DCM {ct}'))
    for ct in dest_cts:
        c = cell_type_colors.get(ct, '#7f7f7f')
        legend += [
            mlines.Line2D([], [], color=c, marker='o', linestyle='None',
                           markersize=10, markeredgecolor='black', label=f'RNA {ct}'),
            mlines.Line2D([], [], color=c, linestyle='-', linewidth=2,
                           alpha=0.8, marker='>', markersize=8, label=f'→ {ct}'),
        ]
    ax.legend(handles=legend, loc='center left',
              bbox_to_anchor=(1, 0.5), frameon=True, fontsize=8)
    ax.set_title(title_str, fontsize=13, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


def plot_umap_cellcycle(title_str, save_path):
    """
    UMAP coloured by combined cell cycle score (S + G2M) for ALL cells.
    Scores are read directly from adata_combined.obs which has
    S_score, G2M_score, phase pre-computed for every cell.
    """
    # Use the pre-computed combined_score column added to adata.obs
    scores_aligned = (adata.obs.reindex(plot_df.index)['combined_score']
                      .fillna(0).values)

    fig, ax = plt.subplots(figsize=(12, 10))

    sc = ax.scatter(
        plot_df['UMAP1'], plot_df['UMAP2'],
        s=8, c=scores_aligned, cmap='RdYlBu_r',
        vmin=np.percentile(scores_aligned, 2),
        vmax=np.percentile(scores_aligned, 98),
        alpha=0.7, rasterized=True, zorder=2,
    )
    cbar = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02, shrink=0.6)
    cbar.set_label('Combined cell cycle score (S + G2M)', fontsize=9)

    ax.set_title(title_str, fontsize=13, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


def plot_heatmap(subset_df, title_str, save_path, group_col='dcm_cell_type'):
    if len(subset_df) == 0:
        print(f"  Skipping heatmap — no cells"); return

    norm_dcm  = TwoSlopeNorm(vmin=-2.5, vcenter=1.0, vmax=3.5)  # DCM: active threshold ~1
    norm_rna  = TwoSlopeNorm(vmin=-2.5, vcenter=0.0, vmax=3.5)  # RNA: centred on 0
    n_cells   = len(subset_df)
    fig_width = max(14, n_cells * 0.18)

    df = subset_df.copy()
    df['_sort'] = df[group_col].map({ct: i for i, ct in enumerate(label_order)}).fillna(99)
    df['_self_z'] = df.apply(
        lambda r: r.get(f'dcm_{r["dcm_cell_type"]}_zscore', 0)
        if r['dcm_cell_type'] in ct_names else 0, axis=1
    )
    df = df.sort_values(['_sort', '_self_z'], ascending=[True, False]).reset_index(drop=True)

    dcm_mat = np.array([[row.get(f'dcm_{ct}_zscore', np.nan) for ct in ct_names]
                         for _, row in df.iterrows()]).T
    rna_mat = np.array([[row.get(f'rna_{ct}_zscore', np.nan) for ct in ct_names]
                         for _, row in df.iterrows()]).T

    fig, axes = plt.subplots(2, 1, figsize=(fig_width, 10))
    for ax, matrix, label, norm, vcenter_val in [
        (axes[0], dcm_mat, 'DCM (GMM label)',         norm_dcm, 1.0),
        (axes[1], rna_mat, 'RNA (leiden_annot_25)',    norm_rna, 0.0),
    ]:
        im = ax.imshow(matrix, aspect='auto', cmap=plt.cm.RdBu_r, norm=norm)
        ax.set_yticks(range(len(ct_names)))
        ax.set_yticklabels(ct_names, fontsize=9)
        ax.set_xticks([])
        ax.set_ylabel('Module', fontsize=11, fontweight='bold')
        ax.set_title(label, fontsize=12, fontweight='bold')

        changes = df[group_col].ne(df[group_col].shift()).to_numpy().nonzero()[0]
        for x in changes:
            ax.axvline(x - 0.5, color='white', lw=1.5, alpha=0.8)
        prev = 0
        for x in list(changes) + [n_cells]:
            mid = (prev + x) / 2
            ct  = df.iloc[prev][group_col]
            ax.text(mid, -0.8, ct, ha='center', va='top', fontsize=7,
                    fontweight='bold', color=cell_type_colors.get(ct, 'black'))
            prev = x
        cbar = plt.colorbar(im, ax=ax, fraction=0.01, pad=0.01)
        cbar.set_label('Z-score', fontsize=9)
        cbar.ax.axhline(y=vcenter_val, color='black', lw=1, ls='--',
                        label=f'vcenter={vcenter_val}')

    fig.suptitle(title_str, fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


def get_significance(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'


def plot_cellcycle(df, group_col, group_order, palette,
                   control_group, title_str, save_path):
    if df.empty:
        print(f"  Skipping cell cycle — no cells"); return
    group_order = [g for g in group_order if g in df[group_col].values]
    if not group_order:
        return

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    scores  = ['combined_score', 'S_score', 'G2M_score']
    titles  = ['Combined (S+G2M)', 'S Phase Score', 'G2M Phase Score']
    ylabels = ['Combined Cell Cycle Score\n(S + G2M)', 'S Phase Score', 'G2M Phase Score']
    ctrl_vals = (df[df[group_col] == control_group]['combined_score']
                 if control_group in df[group_col].values else pd.Series(dtype=float))

    for ax, score, ttl, ylabel in zip(axes, scores, titles, ylabels):
        sns.boxplot(data=df, x=group_col, y=score,
                    order=group_order, palette=palette, ax=ax)
        ax.set_xlabel('Transition', fontsize=11, fontweight='bold')
        ax.set_ylabel(ylabel,       fontsize=11, fontweight='bold')
        ax.set_title(ttl,           fontsize=12, fontweight='bold')
        ax.tick_params(axis='x', rotation=45, labelsize=9)
        if score == 'combined_score':
            ymax = ax.get_ylim()[1]
            for i, grp in enumerate(group_order):
                n = len(df[df[group_col] == grp])
                ax.text(i, ymax * 0.97, f'n={n}', ha='center', fontsize=8, fontweight='bold')
                if grp != control_group and len(ctrl_vals) > 0:
                    gv = df[df[group_col] == grp][score]
                    if len(gv) > 1:
                        _, pval = stats.ttest_ind(gv, ctrl_vals)
                        if pval < 0.05:
                            ax.text(i, ymax * 0.89, get_significance(pval),
                                    ha='center', fontsize=12, fontweight='bold')
    plt.suptitle(title_str, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

# ─────────────────────────────────────────────
# STEP 7: CLEAR SECRETORY → ENTEROCYTE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: CLEAR SECRETORY → ENTEROCYTE")
print("=" * 60)

plot_umap(
    sec_to_ent_clear,
    title_str = f'Clear secretory → Enterocyte  (n={len(sec_to_ent_clear)})\nArrow colour = RNA destination',
    save_path = os.path.join(output_dir_umap, 'umap_clear_secretory_to_enterocyte.pdf'),
)
plot_umap_cellcycle(
    title_str = 'All cells — cell cycle score',
    save_path = os.path.join(output_dir_umap, 'umap_cellcycle_all_cells.pdf'),
)
plot_heatmap(
    sec_to_ent_clear,
    title_str = f'Z-score — clear secretory → Enterocyte  (n={len(sec_to_ent_clear)})',
    save_path = os.path.join(output_dir_heat, 'heatmap_clear_secretory_to_enterocyte.pdf'),
    group_col = 'dcm_cell_type',
)

sec_to_sec = conn_clear_df[
    conn_clear_df['dcm_cell_type'].isin(secretory_types) &
    conn_clear_df['rna_cell_type'].isin(secretory_types)
].copy()
sec_to_sec['group'] = 'Secretory→Secretory'

cc_combined = pd.concat([
    plot1_cc_clear[plot1_cc_clear['group'].isin(
        [f'{s}→Enterocyte' for s in secretory_types] + ['TA→Enterocyte']
    )],
    sec_to_sec.merge(cc_df, on='batch_plate_well', how='left'),
    hybrid_sec_to_ent.merge(cc_df, on='batch_plate_well', how='left')
    .assign(group='Hybrid(Sec)→Enterocyte'),
], ignore_index=True)

cc_order = (['Secretory→Secretory'] +
            [f'{s}→Enterocyte' for s in secretory_types] +
            ['Hybrid(Sec)→Enterocyte', 'TA→Enterocyte'])
cc_palette = {
    'Secretory→Secretory':    '#95a5a6',
    'Paneth→Enterocyte':      '#DC143C',
    'Goblet→Enterocyte':      '#2E8B57',
    'Tuft→Enterocyte':        '#FF1493',
    'EEC→Enterocyte':         '#4169E1',
    'Hybrid(Sec)→Enterocyte': '#FFD700',
    'TA→Enterocyte':          '#3498db',
}
plot_cellcycle(
    cc_combined, 'group', cc_order, cc_palette, 'TA→Enterocyte',
    title_str = ('Cell Cycle — Secretory → Enterocyte\n'
                 'Neg ctrl: Secretory→Secretory  |  Pos ctrl: TA→Enterocyte'),
    save_path = os.path.join(output_dir_cc, 'cellcycle_secretory_to_enterocyte_full.pdf'),
)

# ─────────────────────────────────────────────
# STEP 8: HYBRID DCM → ENTEROCYTE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: HYBRID DCM → ENTEROCYTE")
print("=" * 60)

plot_umap(
    hybrid_sec_to_ent,
    title_str = f'Hybrid DCM (secretory) → Enterocyte  (n={len(hybrid_sec_to_ent)})',
    save_path = os.path.join(output_dir_umap, 'umap_hybrid_dcm_to_enterocyte.pdf'),
    hybrid_dcm=True,
)

plot_heatmap(
    hybrid_sec_to_ent,
    title_str = f'Z-score — Hybrid DCM → Enterocyte  (n={len(hybrid_sec_to_ent)})',
    save_path = os.path.join(output_dir_heat, 'heatmap_hybrid_dcm_to_enterocyte.pdf'),
)
cc_hybrid = pd.concat([
    hybrid_sec_to_ent.merge(cc_df, on='batch_plate_well', how='left')
    .assign(group='Hybrid(Sec)→Enterocyte'),
    plot1_cc_clear[plot1_cc_clear['group'] == 'TA→Enterocyte'],
], ignore_index=True)
plot_cellcycle(
    cc_hybrid, 'group',
    ['Hybrid(Sec)→Enterocyte', 'TA→Enterocyte'],
    {'Hybrid(Sec)→Enterocyte': '#FFD700', 'TA→Enterocyte': '#3498db'},
    'TA→Enterocyte',
    'Cell Cycle — Hybrid DCM (secretory) → Enterocyte',
    os.path.join(output_dir_cc, 'cellcycle_hybrid_dcm_to_enterocyte.pdf'),
)

# ─────────────────────────────────────────────
# STEP 9: CLEAR SECRETORY → ENTEROCYTE HYBRID RNA
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 9: CLEAR SECRETORY → ENTEROCYTE HYBRID RNA")
print("=" * 60)

plot_umap(
    sec_to_ent_hybrid_rna,
    title_str = (f'Clear Secretory → Enterocyte-hybrid RNA  (n={len(sec_to_ent_hybrid_rna)})\n'
                 f'RNA Enterocyte with secretory z-score ≥ {secretory_zscore_threshold}'),
    save_path = os.path.join(output_dir_umap, 'umap_clear_secretory_to_hybrid_rna_enterocyte.pdf'),
)

plot_heatmap(
    sec_to_ent_hybrid_rna,
    title_str = f'Z-score — Clear Secretory → Enterocyte-hybrid RNA  (n={len(sec_to_ent_hybrid_rna)})',
    save_path = os.path.join(output_dir_heat, 'heatmap_clear_secretory_to_hybrid_rna_enterocyte.pdf'),
)
cc_hrna_palette = {f'{s}→EntHybrid': cell_type_colors[s] for s in secretory_types}
cc_hrna_palette['TA→Enterocyte'] = '#3498db'
cc_hrna = pd.concat([
    sec_to_ent_hybrid_rna.merge(cc_df, on='batch_plate_well', how='left'),
    plot1_cc_clear[plot1_cc_clear['group'] == 'TA→Enterocyte'],
], ignore_index=True)
plot_cellcycle(
    cc_hrna, 'group',
    [f'{s}→EntHybrid' for s in secretory_types] + ['TA→Enterocyte'],
    cc_hrna_palette, 'TA→Enterocyte',
    (f'Cell Cycle — Clear Secretory → Enterocyte-hybrid RNA\n'
     f'RNA Enterocyte with secretory z-score ≥ {secretory_zscore_threshold}'),
    os.path.join(output_dir_cc, 'cellcycle_clear_secretory_to_hybrid_rna_enterocyte.pdf'),
)

# ─────────────────────────────────────────────
# SAVE TABLES
# ─────────────────────────────────────────────
conn_clear_df.to_csv(os.path.join(score_dir, 'connections_clear.csv'),          index=False)
conn_hybrid_df.to_csv(os.path.join(score_dir, 'connections_hybrid.csv'),         index=False)
sec_to_ent_clear.to_csv(os.path.join(score_dir, 'sec_to_ent_clear.csv'),         index=False)
hybrid_sec_to_ent.to_csv(os.path.join(score_dir, 'hybrid_dcm_to_enterocyte.csv'),index=False)
sec_to_ent_hybrid_rna.to_csv(os.path.join(score_dir, 'sec_to_hybrid_rna.csv'),   index=False)
print("\nAll tables saved")
print(f"\nOutputs in:\n  {output_dir_umap}\n  {output_dir_heat}\n  {output_dir_cc}")

In [ ]:
#Cell cycle split by dox to enterocyte

In [ ]:
##############################################
# TRANSITION ANALYSIS — adata_combined
# Full port of the reference code adapted to
# the new adata_combined object.
#
# All needed columns are already in obs:
#   assay, batch_plate_well, leiden_annot_25,
#   ttest_gmm_label_confident, is_multipotent,
#   significant_celltypes, ttest_*_zscore,
#   S_score, G2M_score, phase
#
# NOTE: cell cycle scores are pre-computed —
# sc.tl.score_genes_cell_cycle is NOT re-run.
#
# STATS: pairwise t-tests vs Secretory→Secretory
# shown on all three score panels + saved to CSV.
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns
from scipy import stats
from scipy.stats import gaussian_kde

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
root       = "/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py"
score_dir  = os.path.join(root, 'gene_scoring')
output_dir_umap = os.path.join(score_dir, 'gmm_085/cell_cycle3/umap_transitions') # gmm_085/cell_cycle3/umap_transitions
output_dir_heat = os.path.join(score_dir, 'gmm_085/cell_cycle3/heatmap_transitions') #gmm_085/cell_cycle3/heatmap_transitions
output_dir_cc   = os.path.join(score_dir, 'gmm_085/cell_cycle3/cellcycle_transitions') # gmm_085/cell_cycle3/cellcycle_transitions
for d in [output_dir_umap, output_dir_heat, output_dir_cc]:
    os.makedirs(d, exist_ok=True)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
cell_type_colors = {
    'Stem'      : '#9370DB',
    'TA'        : '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth'    : '#DC143C',
    'Goblet'    : '#2E8B57',
    'Tuft'      : '#FF1493',
    'EEC'       : '#4169E1',
    'ambiguous' : '#7f7f7f',
    'hybrid'    : '#FFD700',
}

label_order      = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']
secretory_types  = ['Paneth', 'Goblet', 'Tuft', 'EEC']

BW_METHOD  = 0.5
LEVEL_FRAC = 0.08

# Reference group for all pairwise t-tests
SEC_SEC_GROUP = 'Secretory→Secretory'

# ─────────────────────────────────────────────
# STEP 1: BUILD plot_df
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: BUILDING PLOT DATAFRAME")
print("=" * 60)

adata = adata_combined
umap  = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1':                     umap[:, 0],
    'UMAP2':                     umap[:, 1],
    'assay':                     adata.obs['assay'].values,
    'batch_plate_well':          adata.obs['batch_plate_well'].values,
    'leiden_annot_25':           adata.obs['leiden_annot_25'].astype(str).values,
    'ttest_gmm_label_confident': adata.obs['ttest_gmm_label_confident_085'].astype(str).values, #ttest_gmm_label_confident_085
    'is_multipotent':            adata.obs['is_multipotent'].astype(str).values,
    'significant_celltypes':     adata.obs['significant_celltypes'].astype(str).values,
    'Dox':                       adata.obs['Dox'].astype(str).values,
    'cell_index':                adata.obs.index.values,
}, index=adata.obs_names)

# Strip numeric prefix from leiden/gmm labels (e.g. '3_Enterocyte' → 'Enterocyte')
plot_df['leiden_annot_25']           = plot_df['leiden_annot_25'].str.split('_').str[-1]
plot_df['ttest_gmm_label_confident'] = plot_df['ttest_gmm_label_confident'].str.split('_').str[-1]

# Attach ttest zscore columns
ttest_cols = [c for c in adata.obs.columns
              if c.startswith('ttest_') and c.endswith('_zscore')
              and 'max' not in c.lower()]
for col in ttest_cols:
    plot_df[col] = adata.obs[col].values

print(f"ttest zscore columns: {ttest_cols}")
print(f"Assay values: {sorted(plot_df['assay'].unique())}")
print(f"Dox values  : {sorted(plot_df['Dox'].unique())}")

DOX_VALUES = sorted(plot_df['Dox'].dropna().unique())

# ─────────────────────────────────────────────
# STEP 2: CELL CYCLE — already pre-scored
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: CELL CYCLE (pre-scored, reading from obs)")
print("=" * 60)

dcm_obs = adata.obs[adata.obs['assay'] == 'pDCM']
cc_df   = pd.DataFrame({
    'batch_plate_well': dcm_obs['batch_plate_well'].values,
    'Dox':              dcm_obs['Dox'].astype(str).values,
    'S_score':          dcm_obs['S_score'].values,
    'G2M_score':        dcm_obs['G2M_score'].values,
    'phase':            dcm_obs['phase'].values,
}, index=dcm_obs.index)
cc_df['combined_score'] = cc_df['S_score'] + cc_df['G2M_score']
print(f"DCM cells for cell cycle: {len(cc_df)}")

adata.obs['combined_score'] = (adata.obs['S_score'].astype(float) +
                                adata.obs['G2M_score'].astype(float))
print(f"Added 'combined_score' to adata.obs (all {adata.n_obs} cells)")

# ─────────────────────────────────────────────
# STEP 3: SEPARATE CLEAR vs HYBRID DCM
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: SEPARATING CLEAR vs HYBRID DCM CELLS")
print("=" * 60)

dcm_clear_mask = (
    (plot_df['assay'] == 'pDCM') &
    (plot_df['ttest_gmm_label_confident'].isin(label_order)) &
    (plot_df['is_multipotent'].str.lower().isin(['false', '0', 'no']))
)
dcm_clear  = plot_df[dcm_clear_mask].copy()

dcm_hybrid_mask = (
    (plot_df['assay'] == 'pDCM') &
    (plot_df['is_multipotent'].str.lower().isin(['true', '1', 'yes']))
)
dcm_hybrid = plot_df[dcm_hybrid_mask].copy()

rna_mask   = (
    (plot_df['assay'] == 'pRNA') &
    (plot_df['leiden_annot_25'].isin(label_order))
)
rna_cells  = plot_df[rna_mask].copy()

print(f"Clear DCM  : {len(dcm_clear)}")
print(f"Hybrid DCM : {len(dcm_hybrid)}")
print(f"RNA cells  : {len(rna_cells)}")

# ─────────────────────────────────────────────
# STEP 4: 1:1 MATCHING
# ─────────────────────────────────────────────
def match_one_to_one(dcm_df, rna_df):
    common = set(dcm_df['batch_plate_well']) & set(rna_df['batch_plate_well'])
    dcm_s  = set(dcm_df[dcm_df['batch_plate_well'].isin(common)]
                 ['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
    rna_s  = set(rna_df[rna_df['batch_plate_well'].isin(common)]
                 ['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
    return dcm_s & rna_s

def build_connections(dcm_df, rna_df, matched_wells, is_hybrid=False):
    rows = []
    for bpw in matched_wells:
        dr = dcm_df[dcm_df['batch_plate_well'] == bpw].iloc[0]
        rr = rna_df[rna_df['batch_plate_well'] == bpw].iloc[0]
        row = {
            'batch_plate_well': bpw,
            'dcm_umap1': dr['UMAP1'], 'dcm_umap2': dr['UMAP2'],
            'rna_umap1': rr['UMAP1'], 'rna_umap2': rr['UMAP2'],
            'dcm_cell_type':     dr['ttest_gmm_label_confident'],
            'rna_cell_type':     rr['leiden_annot_25'],
            'dcm_sig_celltypes': dr['significant_celltypes'],
            'Dox':               dr['Dox'],
            'is_hybrid':         is_hybrid,
            'transition': f"{dr['ttest_gmm_label_confident']} -> {rr['leiden_annot_25']}",
        }
        for col in ttest_cols:
            ct = col.replace('ttest_', '').replace('_zscore', '')
            row[f'dcm_{ct}_zscore'] = dr[col]
            row[f'rna_{ct}_zscore'] = rr[col]
        rows.append(row)
    return pd.DataFrame(rows)

both_clear  = match_one_to_one(dcm_clear,  rna_cells)
both_hybrid = match_one_to_one(dcm_hybrid, rna_cells)
print(f"\nClear 1:1 matched wells : {len(both_clear)}")
print(f"Hybrid 1:1 matched wells: {len(both_hybrid)}")

conn_clear_df  = build_connections(dcm_clear,  rna_cells, both_clear,  is_hybrid=False)
conn_hybrid_df = build_connections(dcm_hybrid, rna_cells, both_hybrid, is_hybrid=True)

# ─────────────────────────────────────────────
# STEP 5: TRANSITION GROUPS
# ─────────────────────────────────────────────
sec_to_ent_clear = conn_clear_df[
    conn_clear_df['dcm_cell_type'].isin(secretory_types) &
    (conn_clear_df['rna_cell_type'] == 'Enterocyte')
].copy()
sec_to_ent_clear['group']      = sec_to_ent_clear['dcm_cell_type'] + '→Enterocyte'
sec_to_ent_clear['cell_class'] = 'clear'

ta_ent_clear = conn_clear_df[
    (conn_clear_df['dcm_cell_type'] == 'TA') &
    (conn_clear_df['rna_cell_type'] == 'Enterocyte')
].copy()
ta_ent_clear['group']      = 'TA→Enterocyte'
ta_ent_clear['cell_class'] = 'clear_control'

def has_secretory(sig_str):
    if pd.isna(sig_str) or str(sig_str) in ['none', 'None', '']:
        return False
    return any(s in str(sig_str) for s in secretory_types)

hybrid_sec_to_ent = conn_hybrid_df[
    conn_hybrid_df['dcm_sig_celltypes'].apply(has_secretory) &
    (conn_hybrid_df['rna_cell_type'] == 'Enterocyte')
].copy()
hybrid_sec_to_ent['group']      = 'Hybrid(Sec)→Enterocyte'
hybrid_sec_to_ent['cell_class'] = 'hybrid_dcm'

secretory_zscore_threshold = 1.5
ct_names = [c.replace('ttest_', '').replace('_zscore', '') for c in ttest_cols]

def rna_is_enterocyte_hybrid(row):
    if row['rna_cell_type'] != 'Enterocyte':
        return False
    return any(row.get(f'rna_{s}_zscore', 0) >= secretory_zscore_threshold
               for s in secretory_types)

sec_to_ent_hybrid_rna = conn_clear_df[
    conn_clear_df['dcm_cell_type'].isin(secretory_types)
].copy()
sec_to_ent_hybrid_rna = sec_to_ent_hybrid_rna[
    sec_to_ent_hybrid_rna.apply(rna_is_enterocyte_hybrid, axis=1)
].copy()
sec_to_ent_hybrid_rna['group']      = sec_to_ent_hybrid_rna['dcm_cell_type'] + '→EntHybrid'
sec_to_ent_hybrid_rna['cell_class'] = 'hybrid_rna'

print(f"\nTransition group sizes:")
print(f"  Clear secretory→Enterocyte     : {len(sec_to_ent_clear)}")
print(f"  TA→Enterocyte (control)        : {len(ta_ent_clear)}")
print(f"  Hybrid DCM sec→Enterocyte      : {len(hybrid_sec_to_ent)}")
print(f"  Clear sec→Enterocyte hybrid RNA: {len(sec_to_ent_hybrid_rna)}")

plot1_cc_clear = pd.concat([sec_to_ent_clear, ta_ent_clear], ignore_index=True)
# cc_df also has a Dox column — drop it before merging to avoid Dox_x/Dox_y
plot1_cc_clear = plot1_cc_clear.merge(
    cc_df.drop(columns=['Dox']),
    on='batch_plate_well', how='left'
)

# ─────────────────────────────────────────────
# STEP 6: KDE STORE FOR BACKGROUND
# ─────────────────────────────────────────────
ref_mask = plot_df['assay'].isin(['pDCM', 'pRNA'])

def build_kde(ct, bw=BW_METHOD):
    sub = plot_df[ref_mask & (plot_df['leiden_annot_25'] == ct)]
    if len(sub) < 10:
        return None
    return gaussian_kde(np.vstack([sub['UMAP1'].values, sub['UMAP2'].values]),
                        bw_method=bw)

kde_store = {ct: build_kde(ct) for ct in label_order}

# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────
def plot_umap(subset_df, title_str, save_path, hybrid_dcm=False):
    if len(subset_df) == 0:
        print(f"  Skipping UMAP — no cells"); return

    fig, ax = plt.subplots(figsize=(12, 10))

    for ct in label_order:
        all_ct = plot_df[plot_df['leiden_annot_25'] == ct]
        if len(all_ct):
            ax.scatter(all_ct['UMAP1'], all_ct['UMAP2'],
                       s=6, c=cell_type_colors.get(ct, '#7f7f7f'),
                       alpha=0.15, marker='o', zorder=1, rasterized=True)
    other_all = plot_df[~plot_df['leiden_annot_25'].isin(label_order)]
    if len(other_all):
        ax.scatter(other_all['UMAP1'], other_all['UMAP2'],
                   s=6, c='lightgray', alpha=0.08,
                   marker='o', zorder=1, rasterized=True)

    for ct in label_order:
        kde = kde_store.get(ct)
        if kde is None:
            continue
        sub = plot_df[plot_df['leiden_annot_25'] == ct]
        x, y = sub['UMAP1'].values, sub['UMAP2'].values
        xg = np.linspace(x.min()-0.5, x.max()+0.5, 200)
        yg = np.linspace(y.min()-0.5, y.max()+0.5, 200)
        xx, yy = np.meshgrid(xg, yg)
        zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        ax.contour(xx, yy, zz, levels=[zz.max() * LEVEL_FRAC],
                   colors=[cell_type_colors[ct]],
                   linewidths=2, linestyles='--', alpha=0.8, zorder=2)
        peak = np.unravel_index(zz.argmax(), zz.shape)
        ax.text(xx[peak], yy[peak], ct, ha='center', va='center',
                fontsize=10, fontweight='bold', color=cell_type_colors[ct], zorder=5)

    all_bpw = set(subset_df['batch_plate_well'])
    for _, conn in subset_df.iterrows():
        color = cell_type_colors.get(conn['rna_cell_type'], 'black')
        ax.add_patch(FancyArrowPatch(
            (conn['dcm_umap1'], conn['dcm_umap2']),
            (conn['rna_umap1'], conn['rna_umap2']),
            arrowstyle='-|>', mutation_scale=15,
            color=color, alpha=0.7, linewidth=1.5, zorder=3,
        ))

    for ct in label_order:
        if hybrid_dcm:
            pts = plot_df[(plot_df['assay'] == 'pDCM') &
                          (plot_df['batch_plate_well'].isin(all_bpw)) &
                          (plot_df['is_multipotent'].str.lower().isin(['true','1','yes']))]
            c, m = cell_type_colors['hybrid'], 'D'
        else:
            pts = plot_df[(plot_df['assay'] == 'pDCM') &
                          (plot_df['batch_plate_well'].isin(all_bpw)) &
                          (plot_df['ttest_gmm_label_confident'] == ct)]
            c, m = cell_type_colors.get(ct, '#7f7f7f'), 's'
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100, c=c, alpha=1.0,
                       edgecolors='black', linewidths=1.5,
                       marker=m, zorder=4, rasterized=True)
        if hybrid_dcm:
            break

    dest_cts = subset_df['rna_cell_type'].unique()
    for ct in dest_cts:
        pts = plot_df[(plot_df['assay'] == 'pRNA') &
                      (plot_df['batch_plate_well'].isin(all_bpw)) &
                      (plot_df['leiden_annot_25'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, '#7f7f7f'), alpha=1.0,
                       edgecolors='black', linewidths=1.5,
                       marker='o', zorder=4, rasterized=True)

    legend = []
    for ct in label_order:
        legend.append(mlines.Line2D([], [], color=cell_type_colors.get(ct, '#7f7f7f'),
                                     marker='o', linestyle='None', markersize=7,
                                     alpha=0.4, label=f'All cells ({ct})'))
    if hybrid_dcm:
        legend.append(mlines.Line2D([], [], color=cell_type_colors['hybrid'],
                                     marker='D', linestyle='None', markersize=10,
                                     markeredgecolor='black', label='DCM Hybrid'))
    else:
        for ct in subset_df['dcm_cell_type'].unique():
            legend.append(mlines.Line2D([], [], color=cell_type_colors.get(ct,'gray'),
                                         marker='s', linestyle='None', markersize=10,
                                         markeredgecolor='black', label=f'DCM {ct}'))
    for ct in dest_cts:
        c = cell_type_colors.get(ct, '#7f7f7f')
        legend += [
            mlines.Line2D([], [], color=c, marker='o', linestyle='None',
                           markersize=10, markeredgecolor='black', label=f'RNA {ct}'),
            mlines.Line2D([], [], color=c, linestyle='-', linewidth=2,
                           alpha=0.8, marker='>', markersize=8, label=f'→ {ct}'),
        ]
    ax.legend(handles=legend, loc='center left',
              bbox_to_anchor=(1, 0.5), frameon=True, fontsize=8)
    ax.set_title(title_str, fontsize=13, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


def plot_umap_cellcycle(title_str, save_path):
    scores_aligned = (adata.obs.reindex(plot_df.index)['combined_score']
                      .fillna(0).values)
    fig, ax = plt.subplots(figsize=(12, 10))
    sc = ax.scatter(
        plot_df['UMAP1'], plot_df['UMAP2'],
        s=8, c=scores_aligned, cmap='RdYlBu_r',
        vmin=np.percentile(scores_aligned, 2),
        vmax=np.percentile(scores_aligned, 98),
        alpha=0.7, rasterized=True, zorder=2,
    )
    cbar = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02, shrink=0.6)
    cbar.set_label('Combined cell cycle score (S + G2M)', fontsize=9)
    ax.set_title(title_str, fontsize=13, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


def plot_heatmap(subset_df, title_str, save_path, group_col='dcm_cell_type'):
    if len(subset_df) == 0:
        print(f"  Skipping heatmap — no cells"); return

    norm_dcm  = TwoSlopeNorm(vmin=-2.5, vcenter=1.0, vmax=3.5)
    norm_rna  = TwoSlopeNorm(vmin=-2.5, vcenter=0.0, vmax=3.5)
    n_cells   = len(subset_df)
    fig_width = max(14, n_cells * 0.18)

    df = subset_df.copy()
    df['_sort'] = df[group_col].map({ct: i for i, ct in enumerate(label_order)}).fillna(99)
    df['_self_z'] = df.apply(
        lambda r: r.get(f'dcm_{r["dcm_cell_type"]}_zscore', 0)
        if r['dcm_cell_type'] in ct_names else 0, axis=1
    )
    df = df.sort_values(['_sort', '_self_z'], ascending=[True, False]).reset_index(drop=True)

    dcm_mat = np.array([[row.get(f'dcm_{ct}_zscore', np.nan) for ct in ct_names]
                         for _, row in df.iterrows()]).T
    rna_mat = np.array([[row.get(f'rna_{ct}_zscore', np.nan) for ct in ct_names]
                         for _, row in df.iterrows()]).T

    fig, axes = plt.subplots(2, 1, figsize=(fig_width, 10))
    for ax, matrix, label, norm, vcenter_val in [
        (axes[0], dcm_mat, 'DCM (GMM label)',       norm_dcm, 1.0),
        (axes[1], rna_mat, 'RNA (leiden_annot_25)',  norm_rna, 0.0),
    ]:
        im = ax.imshow(matrix, aspect='auto', cmap=plt.cm.RdBu_r, norm=norm)
        ax.set_yticks(range(len(ct_names)))
        ax.set_yticklabels(ct_names, fontsize=9)
        ax.set_xticks([])
        ax.set_ylabel('Module', fontsize=11, fontweight='bold')
        ax.set_title(label, fontsize=12, fontweight='bold')

        changes = df[group_col].ne(df[group_col].shift()).to_numpy().nonzero()[0]
        for x in changes:
            ax.axvline(x - 0.5, color='white', lw=1.5, alpha=0.8)
        prev = 0
        for x in list(changes) + [n_cells]:
            mid  = (prev + x) / 2
            ct   = df.iloc[prev][group_col]
            ax.text(mid, -0.8, ct, ha='center', va='top', fontsize=7,
                    fontweight='bold', color=cell_type_colors.get(ct, 'black'))
            prev = x
        cbar = plt.colorbar(im, ax=ax, fraction=0.01, pad=0.01)
        cbar.set_label('Z-score', fontsize=9)
        cbar.ax.axhline(y=vcenter_val, color='black', lw=1, ls='--',
                        label=f'vcenter={vcenter_val}')

    fig.suptitle(title_str, fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


# ─────────────────────────────────────────────
# STATS HELPERS
# ─────────────────────────────────────────────
def get_significance(p):
    if np.isnan(p):   return ''
    if p < 0.001:     return '***'
    if p < 0.01:      return '**'
    if p < 0.05:      return '*'
    return 'ns'


def compute_stats(df, group_col, group_order, context_label, dox_label='ALL'):
    """
    Tidy DataFrame: descriptive stats + t-test p-values vs SEC_SEC_GROUP
    for every group × score.
    """
    rows     = []
    scores   = ['combined_score', 'S_score', 'G2M_score']
    ref_data = {s: df[df[group_col] == SEC_SEC_GROUP][s].dropna() for s in scores}

    for grp in group_order:
        sub = df[df[group_col] == grp]
        for score in scores:
            vals = sub[score].dropna()
            ref  = ref_data[score]
            if grp != SEC_SEC_GROUP and len(vals) > 1 and len(ref) > 1:
                _, pval = stats.ttest_ind(vals, ref)
            else:
                pval = np.nan
            rows.append({
                'context':              context_label,
                'Dox':                  dox_label,
                'group':                grp,
                'score':                score,
                'N':                    len(vals),
                'mean':                 vals.mean()   if len(vals) else np.nan,
                'median':               vals.median() if len(vals) else np.nan,
                'std':                  vals.std()    if len(vals) else np.nan,
                'ttest_vs_SecSec_pval': pval,
                'significance':         get_significance(pval),
            })
    return pd.DataFrame(rows)


def _draw_brackets(ax, df, group_col, group_order, score, y_top):
    """
    Draw t-test bracket lines + stars vs SEC_SEC_GROUP on a single axis.
    Uses two staggered y-levels to avoid overlap when there are many groups.
    Returns the highest y used so the caller can extend ylim.
    """
    if SEC_SEC_GROUP not in group_order or SEC_SEC_GROUP not in df[group_col].values:
        return y_top

    ref_vals = df[df[group_col] == SEC_SEC_GROUP][score].dropna()
    if len(ref_vals) <= 1:
        return y_top

    ref_i      = group_order.index(SEC_SEC_GROUP)
    # Stagger brackets across two rows to avoid label collisions
    y_levels   = [y_top * 1.08, y_top * 1.16]
    y_max_used = y_levels[0]

    for k, grp in enumerate(g for g in group_order if g != SEC_SEC_GROUP):
        i    = group_order.index(grp)
        gv   = df[df[group_col] == grp][score].dropna()
        if len(gv) <= 1:
            continue
        _, pval = stats.ttest_ind(gv, ref_vals)
        stars   = get_significance(pval)
        y_lvl   = y_levels[k % 2]
        y_max_used = max(y_max_used, y_lvl)
        x_mid   = (i + ref_i) / 2

        ax.annotate(
            '', xy=(i, y_lvl), xytext=(ref_i, y_lvl),
            arrowprops=dict(arrowstyle='-', color='#444444', lw=0.9),
        )
        color = 'black' if stars not in ('ns', '') else '#888888'
        ax.text(x_mid, y_lvl, stars,
                ha='center', va='bottom', fontsize=10,
                fontweight='bold', color=color)

    return y_max_used


def plot_cellcycle(df, group_col, group_order, palette,
                   control_group, title_str, save_path,
                   context_label='', dox_label='ALL'):
    """
    Boxplots with pairwise t-test brackets vs Secretory→Secretory on
    ALL THREE score panels. Returns a stats DataFrame.
    """
    if df.empty:
        print(f"  Skipping cell cycle — no cells")
        return pd.DataFrame()

    group_order = [g for g in group_order if g in df[group_col].values]
    if not group_order:
        return pd.DataFrame()

    scores  = ['combined_score', 'S_score', 'G2M_score']
    titles  = ['Combined (S+G2M)', 'S Phase Score', 'G2M Phase Score']
    ylabels = ['Combined Cell Cycle Score\n(S + G2M)', 'S Phase Score', 'G2M Phase Score']

    fig, axes = plt.subplots(1, 3, figsize=(22, 8))

    for ax, score, ttl, ylabel in zip(axes, scores, titles, ylabels):
        # ── boxplot (seaborn ≥0.13 compatible) ──
        sns.boxplot(
            data=df, x=group_col, y=score,
            hue=group_col, order=group_order,
            palette=palette, legend=False, ax=ax,
        )
        ax.set_xlabel('Transition', fontsize=11, fontweight='bold')
        ax.set_ylabel(ylabel,       fontsize=11, fontweight='bold')
        ax.set_title(ttl,           fontsize=12, fontweight='bold')
        ax.tick_params(axis='x', rotation=45, labelsize=9)

        ylim  = ax.get_ylim()
        y_top = ylim[1]

        # n labels just above each box
        for i, grp in enumerate(group_order):
            n = len(df[df[group_col] == grp])
            ax.text(i, y_top * 1.01, f'n={n}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

        # pairwise brackets vs Secretory→Secretory
        y_max = _draw_brackets(ax, df, group_col, group_order, score, y_top)

        ax.set_ylim(ylim[0], y_max * 1.10)

        # footnote
        if SEC_SEC_GROUP in group_order:
            ax.text(0.01, 0.99,
                    f'— t-test vs {SEC_SEC_GROUP}',
                    transform=ax.transAxes, fontsize=6,
                    color='gray', va='top', ha='left')

    plt.suptitle(title_str, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

    return compute_stats(df, group_col, group_order, context_label, dox_label)


# Accumulator for stats across all plot_cellcycle calls
_all_stats = []

# ─────────────────────────────────────────────
# HELPER: safe filename tag from Dox value
# ─────────────────────────────────────────────
def dox_tag(dox):
    return str(dox).replace(' ', '_').replace('/', '-')


# ─────────────────────────────────────────────
# STEP 7: CLEAR SECRETORY → ENTEROCYTE
# ALL + per-Dox split
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: CLEAR SECRETORY → ENTEROCYTE")
print("=" * 60)

# UMAPs and heatmaps — ALL only (UMAP coords are global)
plot_umap(
    sec_to_ent_clear,
    title_str = f'Clear secretory → Enterocyte  (n={len(sec_to_ent_clear)})\nArrow colour = RNA destination',
    save_path = os.path.join(output_dir_umap, 'umap_clear_secretory_to_enterocyte.pdf'),
)
plot_umap_cellcycle(
    title_str = 'All cells — cell cycle score',
    save_path = os.path.join(output_dir_umap, 'umap_cellcycle_all_cells.pdf'),
)
plot_heatmap(
    sec_to_ent_clear,
    title_str = f'Z-score — clear secretory → Enterocyte  (n={len(sec_to_ent_clear)})',
    save_path = os.path.join(output_dir_heat, 'heatmap_clear_secretory_to_enterocyte.pdf'),
    group_col = 'dcm_cell_type',
)

sec_to_sec = conn_clear_df[
    conn_clear_df['dcm_cell_type'].isin(secretory_types) &
    conn_clear_df['rna_cell_type'].isin(secretory_types)
].copy()
sec_to_sec['group'] = SEC_SEC_GROUP

cc_order = (
    [SEC_SEC_GROUP] +
    [f'{s}→Enterocyte' for s in secretory_types] +
    ['Hybrid(Sec)→Enterocyte', 'TA→Enterocyte']
)
cc_palette = {
    SEC_SEC_GROUP:                '#95a5a6',
    'Paneth→Enterocyte':          '#DC143C',
    'Goblet→Enterocyte':          '#2E8B57',
    'Tuft→Enterocyte':            '#FF1493',
    'EEC→Enterocyte':             '#4169E1',
    'Hybrid(Sec)→Enterocyte':     '#FFD700',
    'TA→Enterocyte':              '#3498db',
}

for dox_label in ['ALL'] + list(DOX_VALUES):
    tag = dox_tag(dox_label)
    print(f"\n  Dox={dox_label}")

    # sec_to_sec and TA→Enterocyte always from ALL conditions (reference groups)
    sub_sec_sec = sec_to_sec
    if dox_label == 'ALL':
        sub_hybrid  = hybrid_sec_to_ent
        sec_ent_sub = plot1_cc_clear[plot1_cc_clear['group'].isin(
                          [f'{s}→Enterocyte' for s in secretory_types])]
    else:
        sub_hybrid  = hybrid_sec_to_ent[hybrid_sec_to_ent['Dox'] == dox_label]
        sec_ent_sub = plot1_cc_clear[
                          (plot1_cc_clear['group'].isin(
                              [f'{s}→Enterocyte' for s in secretory_types])) &
                          (plot1_cc_clear['Dox'] == dox_label)]

    ta_all = plot1_cc_clear[plot1_cc_clear['group'] == 'TA→Enterocyte']  # ALL Dox

    cc_combined = pd.concat([
        sec_ent_sub,
        ta_all,
        sub_sec_sec.merge(cc_df[['batch_plate_well', 'S_score', 'G2M_score', 'combined_score']],
                          on='batch_plate_well', how='left'),
        sub_hybrid.merge(cc_df[['batch_plate_well', 'S_score', 'G2M_score', 'combined_score']],
                         on='batch_plate_well', how='left')
                  .assign(group='Hybrid(Sec)→Enterocyte'),
    ], ignore_index=True)

    _stats = plot_cellcycle(
        cc_combined, 'group', cc_order, cc_palette, 'TA→Enterocyte',
        title_str     = (f'Cell Cycle — Secretory → Enterocyte | Dox = {dox_label}\n'
                         f't-tests vs {SEC_SEC_GROUP}'),
        save_path     = os.path.join(output_dir_cc,
                                     f'cellcycle_secretory_to_enterocyte_full_Dox_{tag}.pdf'),
        context_label = 'SecretoryToEnterocyte',
        dox_label     = dox_label,
    )
    _all_stats.append(_stats)

# ─────────────────────────────────────────────
# STEP 8: HYBRID DCM → ENTEROCYTE
# ALL + per-Dox split
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: HYBRID DCM → ENTEROCYTE")
print("=" * 60)

plot_umap(
    hybrid_sec_to_ent,
    title_str = f'Hybrid DCM (secretory) → Enterocyte  (n={len(hybrid_sec_to_ent)})',
    save_path = os.path.join(output_dir_umap, 'umap_hybrid_dcm_to_enterocyte.pdf'),
    hybrid_dcm=True,
)
plot_heatmap(
    hybrid_sec_to_ent,
    title_str = f'Z-score — Hybrid DCM → Enterocyte  (n={len(hybrid_sec_to_ent)})',
    save_path = os.path.join(output_dir_heat, 'heatmap_hybrid_dcm_to_enterocyte.pdf'),
)

for dox_label in ['ALL'] + list(DOX_VALUES):
    tag = dox_tag(dox_label)
    print(f"\n  Dox={dox_label}")

    # sec_to_sec and TA→Enterocyte always from ALL conditions (reference groups)
    sub_sec_sec = sec_to_sec
    sub_ta_ent  = plot1_cc_clear[plot1_cc_clear['group'] == 'TA→Enterocyte']  # ALL Dox
    if dox_label == 'ALL':
        sub_hybrid = hybrid_sec_to_ent
    else:
        sub_hybrid = hybrid_sec_to_ent[hybrid_sec_to_ent['Dox'] == dox_label]

    cc_hybrid = pd.concat([
        sub_hybrid.merge(cc_df[['batch_plate_well', 'S_score', 'G2M_score', 'combined_score']],
                         on='batch_plate_well', how='left')
                  .assign(group='Hybrid(Sec)→Enterocyte'),
        sub_ta_ent,
        sub_sec_sec.merge(cc_df[['batch_plate_well', 'S_score', 'G2M_score', 'combined_score']],
                          on='batch_plate_well', how='left'),
    ], ignore_index=True)

    _stats = plot_cellcycle(
        cc_hybrid, 'group',
        [SEC_SEC_GROUP, 'Hybrid(Sec)→Enterocyte', 'TA→Enterocyte'],
        {SEC_SEC_GROUP: '#95a5a6',
         'Hybrid(Sec)→Enterocyte': '#FFD700',
         'TA→Enterocyte': '#3498db'},
        'TA→Enterocyte',
        f'Cell Cycle — Hybrid DCM (secretory) → Enterocyte | Dox = {dox_label}\nt-tests vs {SEC_SEC_GROUP}',
        os.path.join(output_dir_cc,
                     f'cellcycle_hybrid_dcm_to_enterocyte_Dox_{tag}.pdf'),
        context_label = 'HybridDCMToEnterocyte',
        dox_label     = dox_label,
    )
    _all_stats.append(_stats)

# ─────────────────────────────────────────────
# STEP 9: CLEAR SECRETORY → ENTEROCYTE HYBRID RNA
# ALL + per-Dox split
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 9: CLEAR SECRETORY → ENTEROCYTE HYBRID RNA")
print("=" * 60)

plot_umap(
    sec_to_ent_hybrid_rna,
    title_str = (f'Clear Secretory → Enterocyte-hybrid RNA  (n={len(sec_to_ent_hybrid_rna)})\n'
                 f'RNA Enterocyte with secretory z-score ≥ {secretory_zscore_threshold}'),
    save_path = os.path.join(output_dir_umap,
                             'umap_clear_secretory_to_hybrid_rna_enterocyte.pdf'),
)
plot_heatmap(
    sec_to_ent_hybrid_rna,
    title_str = (f'Z-score — Clear Secretory → Enterocyte-hybrid RNA  '
                 f'(n={len(sec_to_ent_hybrid_rna)})'),
    save_path = os.path.join(output_dir_heat,
                             'heatmap_clear_secretory_to_hybrid_rna_enterocyte.pdf'),
)

cc_hrna_palette = {f'{s}→EntHybrid': cell_type_colors[s] for s in secretory_types}
cc_hrna_palette['TA→Enterocyte'] = '#3498db'
cc_hrna_palette[SEC_SEC_GROUP]   = '#95a5a6'
cc_hrna_order = (
    [SEC_SEC_GROUP] +
    [f'{s}→EntHybrid' for s in secretory_types] +
    ['TA→Enterocyte']
)

for dox_label in ['ALL'] + list(DOX_VALUES):
    tag = dox_tag(dox_label)
    print(f"\n  Dox={dox_label}")

    # sec_to_sec and TA→Enterocyte always from ALL conditions (reference groups)
    sub_sec_sec = sec_to_sec
    sub_ta_ent  = plot1_cc_clear[plot1_cc_clear['group'] == 'TA→Enterocyte']  # ALL Dox
    if dox_label == 'ALL':
        sub_hrna = sec_to_ent_hybrid_rna
    else:
        sub_hrna = sec_to_ent_hybrid_rna[sec_to_ent_hybrid_rna['Dox'] == dox_label]

    cc_hrna = pd.concat([
        sub_hrna.merge(cc_df[['batch_plate_well', 'S_score', 'G2M_score', 'combined_score']],
                       on='batch_plate_well', how='left'),
        sub_ta_ent,
        sub_sec_sec.merge(cc_df[['batch_plate_well', 'S_score', 'G2M_score', 'combined_score']],
                          on='batch_plate_well', how='left'),
    ], ignore_index=True)

    _stats = plot_cellcycle(
        cc_hrna, 'group',
        cc_hrna_order,
        cc_hrna_palette, 'TA→Enterocyte',
        (f'Cell Cycle — Clear Secretory → Enterocyte-hybrid RNA | Dox = {dox_label}\n'
         f'RNA Enterocyte with secretory z-score ≥ {secretory_zscore_threshold} | '
         f't-tests vs {SEC_SEC_GROUP}'),
        os.path.join(output_dir_cc,
                     f'cellcycle_clear_secretory_to_hybrid_rna_enterocyte_Dox_{tag}.pdf'),
        context_label = 'SecToHybridRNAEnterocyte',
        dox_label     = dox_label,
    )
    _all_stats.append(_stats)

# ─────────────────────────────────────────────
# SAVE TABLES
# ─────────────────────────────────────────────
conn_clear_df.to_csv(os.path.join(score_dir, 'connections_clear.csv'),           index=False)
conn_hybrid_df.to_csv(os.path.join(score_dir, 'connections_hybrid.csv'),          index=False)
sec_to_ent_clear.to_csv(os.path.join(score_dir, 'sec_to_ent_clear.csv'),          index=False)
hybrid_sec_to_ent.to_csv(os.path.join(score_dir, 'hybrid_dcm_to_enterocyte.csv'), index=False)
sec_to_ent_hybrid_rna.to_csv(os.path.join(score_dir, 'sec_to_hybrid_rna.csv'),    index=False)

# Stats CSV — all contexts × Dox × score
stats_df  = pd.concat([s for s in _all_stats if not s.empty], ignore_index=True)
stats_csv = os.path.join(output_dir_cc, 'cellcycle_stats_vs_SecSec.csv')
stats_df.to_csv(stats_csv, index=False)
print(f"\nStats CSV saved: {stats_csv}")
print(stats_df.to_string(index=False))

print("\nAll tables saved")
print(f"\nOutputs in:\n  {output_dir_umap}\n  {output_dir_heat}\n  {output_dir_cc}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scanpy as sc
from scipy import stats
from itertools import combinations
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/'
output_dir = score_dir

# Cell cycle genes from Tirosh et al. 2016
s_genes = ['MCM5', 'PCNA', 'TYMS', 'FEN1', 'MCM2', 'MCM4', 'RRM1', 'UNG', 'GINS2', 'MCM6', 
           'CDCA7', 'DTL', 'PRIM1', 'UHRF1', 'MLF1IP', 'HELLS', 'RFC2', 'RPA2', 'NASP', 
           'RAD51AP1', 'GMNN', 'WDR76', 'SLBP', 'CCNE2', 'UBR7', 'POLD3', 'MSH2', 'ATAD2', 
           'RAD51', 'RRM2', 'CDC45', 'CDC6', 'EXO1', 'TIPIN', 'DSCC1', 'BLM', 'CASP8AP2', 
           'USP1', 'CLSPN', 'POLA1', 'CHAF1B', 'BRIP1', 'E2F8']

g2m_genes = ['HMGB2', 'CDK1', 'NUSAP1', 'UBE2C', 'BIRC5', 'TPX2', 'TOP2A', 'NDC80', 'CKS2', 
             'NUF2', 'CKS1B', 'MKI67', 'TMPO', 'CENPF', 'TACC3', 'FAM64A', 'SMC4', 'CCNB2', 
             'CKAP2L', 'CKAP2', 'AURKB', 'BUB1', 'KIF11', 'ANP32E', 'TUBB4B', 'GTSE1', 'KIF20B', 
             'HJURP', 'CDCA3', 'HN1', 'CDC20', 'TTK', 'CDC25C', 'KIF2C', 'RANGAP1', 'NCAPD2', 
             'DLGAP5', 'CDCA2', 'CDCA8', 'ECT2', 'KIF23', 'HMMR', 'AURKA', 'PSRC1', 'ANLN', 
             'LBR', 'CKAP5', 'CENPE', 'CTCF', 'NEK2', 'G2E3', 'GAS2L3', 'CBX5', 'CENPA']

# Convert to mouse gene names (capitalize first letter only)
s_genes_mouse = [gene.capitalize() for gene in s_genes]
g2m_genes_mouse = [gene.capitalize() for gene in g2m_genes]

# Define cell type groups
secretory_types = ['Paneth', 'Goblet', 'Tuft', 'EEC']
absorptive_types = ['Enterocyte']
stem_ta_types = ['Stem', 'TA']

print("="*80)
print("CALCULATING CELL CYCLE SCORES WITH BACKGROUND CONTROL")
print("="*80)

# -----------------------------
# CALCULATE CELL CYCLE SCORES ON DCM DATA
# -----------------------------
print("\n" + "="*80)
print("CALCULATING CELL CYCLE SCORES FROM DCM CELLS")
print("="*80)

# Filter to only DCM cells
adata_dcm = adata_combined[adata_combined.obs['assay'] == 'pDCM'].copy()
print(f"\nTotal DCM cells: {len(adata_dcm)}")

# Calculate cell cycle scores using scanpy
print("\nRunning scanpy cell cycle scoring...")
sc.tl.score_genes_cell_cycle(adata_dcm, s_genes=s_genes_mouse, g2m_genes=g2m_genes_mouse)

print(f"\nCell cycle scores calculated:")
print(f"  S_score - Mean: {adata_dcm.obs['S_score'].mean():.4f}, "
      f"Range: [{adata_dcm.obs['S_score'].min():.4f}, {adata_dcm.obs['S_score'].max():.4f}]")
print(f"  G2M_score - Mean: {adata_dcm.obs['G2M_score'].mean():.4f}, "
      f"Range: [{adata_dcm.obs['G2M_score'].min():.4f}, {adata_dcm.obs['G2M_score'].max():.4f}]")

# Check which genes were found
s_genes_found = [g for g in s_genes_mouse if g in adata_dcm.var_names]
g2m_genes_found = [g for g in g2m_genes_mouse if g in adata_dcm.var_names]
print(f"\nGenes used:")
print(f"  S phase: {len(s_genes_found)}/{len(s_genes_mouse)} genes")
print(f"  G2M phase: {len(g2m_genes_found)}/{len(g2m_genes_mouse)} genes")

# -----------------------------
# PREPARE DATA
# -----------------------------
print("\n" + "="*80)
print("PREPARING PAIRED DATA")
print("="*80)

# Create batch_plate_well identifier
adata_combined.obs['batch_plate_well'] = (
    adata_combined.obs['Batch'].astype(str) + '_' + 
    adata_combined.obs['plate_well'].astype(str)
)

# Get DCM data with cell cycle scores
pDCM_data = pd.DataFrame({
    'batch_plate_well': adata_dcm.obs['batch_plate_well'].values,
    'dcm_celltype': adata_dcm.obs['ttest_gmm_label_confident'].astype(str).values,
    'S_score': adata_dcm.obs['S_score'].values,
    'G2M_score': adata_dcm.obs['G2M_score'].values,
    'phase': adata_dcm.obs['phase'].values
})

# Extract cell type from labels
pDCM_data['dcm_celltype'] = pDCM_data['dcm_celltype'].str.split('_').str[-1]

# Combined score
pDCM_data['combined_score'] = pDCM_data['S_score'] + pDCM_data['G2M_score']

# Get RNA data
rna_mask = adata_combined.obs['assay'] == 'pRNA'
pRNA_data = pd.DataFrame({
    'batch_plate_well': adata_combined.obs[rna_mask]['batch_plate_well'].values,
    'rna_celltype': adata_combined.obs[rna_mask]['leiden_annot_25'].astype(str).values
})

# Extract cell type from labels
pRNA_data['rna_celltype'] = pRNA_data['rna_celltype'].str.split('_').str[-1]

# Merge to pair cells
paired_cells = pDCM_data.merge(pRNA_data, on='batch_plate_well', how='inner')

print(f"Total paired cells: {len(paired_cells)}")

print("\nCell cycle phase distribution in DCM cells:")
print(paired_cells['phase'].value_counts())

# =============================================================================
# PLOT 1: SECRETORY TRANSITIONS (EXCLUDING STEM→ENTEROCYTE)
# =============================================================================

print("\n" + "="*80)
print("PREPARING PLOT 1: SECRETORY TRANSITIONS")
print("="*80)

def classify_transition_plot1(row):
    dcm = row['dcm_celltype']
    rna = row['rna_celltype']
    
    # Secretory → Secretory (control)
    if dcm in secretory_types and rna in secretory_types:
        return 'Secretory→Secretory', f'{dcm}→{rna}'
    
    # Individual Secretory → Enterocyte
    elif dcm == 'Paneth' and rna == 'Enterocyte':
        return 'Paneth→Enterocyte', 'Paneth→Enterocyte'
    elif dcm == 'Goblet' and rna == 'Enterocyte':
        return 'Goblet→Enterocyte', 'Goblet→Enterocyte'
    elif dcm == 'Tuft' and rna == 'Enterocyte':
        return 'Tuft→Enterocyte', 'Tuft→Enterocyte'
    elif dcm == 'EEC' and rna == 'Enterocyte':
        return 'EEC→Enterocyte', 'EEC→Enterocyte'
    
    # TA → Enterocyte (control)
    elif dcm == 'TA' and rna == 'Enterocyte':
        return 'TA→Enterocyte', 'TA→Enterocyte'
    
    else:
        return 'Other', f'{dcm}→{rna}'

paired_cells['transition_plot1'], paired_cells['specific_transition_plot1'] = zip(
    *paired_cells.apply(classify_transition_plot1, axis=1)
)

# Filter to relevant groups for plot 1 (EXCLUDING Stem→Enterocyte)
relevant_groups_plot1 = ['Secretory→Secretory', 'Paneth→Enterocyte', 'Goblet→Enterocyte', 
                         'Tuft→Enterocyte', 'EEC→Enterocyte', 'TA→Enterocyte']
filtered_cells_plot1 = paired_cells[paired_cells['transition_plot1'].isin(relevant_groups_plot1)].copy()

print(f"\nFiltered to {len(filtered_cells_plot1)} cells for Plot 1")
print("\nTransition counts:")
print(filtered_cells_plot1['transition_plot1'].value_counts())

# Create Plot 1
print("\nCreating Plot 1...")

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Define colors
transition_colors_plot1 = {
    'Secretory→Secretory': '#95a5a6',  # Gray (control)
    'Paneth→Enterocyte': '#DC143C',    # Red
    'Goblet→Enterocyte': '#2E8B57',    # Green
    'Tuft→Enterocyte': '#FF1493',      # Pink
    'EEC→Enterocyte': '#4169E1',       # Blue
    'TA→Enterocyte': '#3498db'         # Light Blue (control)
}

# Order for plot 1
transition_order_plot1 = ['Secretory→Secretory', 'Paneth→Enterocyte', 'Goblet→Enterocyte', 
                          'Tuft→Enterocyte', 'EEC→Enterocyte', 'TA→Enterocyte']

# Plot 1: Combined score
sns.boxplot(data=filtered_cells_plot1, x='transition_plot1', y='combined_score',
            order=transition_order_plot1, palette=transition_colors_plot1, ax=axes[0])
axes[0].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Combined Cell Cycle Score\n(S + G2M)', fontsize=11, fontweight='bold')
axes[0].set_title('Combined Cell Cycle Score', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45, labelsize=9)

# Add n values
for i, trans in enumerate(transition_order_plot1):
    n = len(filtered_cells_plot1[filtered_cells_plot1['transition_plot1'] == trans])
    axes[0].text(i, axes[0].get_ylim()[1] * 0.95, f'n={n}', 
                ha='center', fontsize=8, fontweight='bold')

# T-test vs Secretory→Secretory control
sec_sec_vals = filtered_cells_plot1[filtered_cells_plot1['transition_plot1'] == 'Secretory→Secretory']['combined_score']
for i, trans in enumerate(transition_order_plot1[1:], start=1):  # Skip the control itself
    trans_vals = filtered_cells_plot1[filtered_cells_plot1['transition_plot1'] == trans]['combined_score']
    if len(trans_vals) > 0 and len(sec_sec_vals) > 0:
        tstat, pval = stats.ttest_ind(trans_vals, sec_sec_vals)
        if pval < 0.05:
            sig_marker = '***' if pval < 0.001 else '**' if pval < 0.01 else '*'
            axes[0].text(i, axes[0].get_ylim()[1] * 0.88, sig_marker, 
                        ha='center', fontsize=12, fontweight='bold')

# Plot 2: S score
sns.boxplot(data=filtered_cells_plot1, x='transition_plot1', y='S_score',
            order=transition_order_plot1, palette=transition_colors_plot1, ax=axes[1])
axes[1].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[1].set_ylabel('S Phase Score', fontsize=11, fontweight='bold')
axes[1].set_title('S Phase Score', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45, labelsize=9)

# Plot 3: G2M score
sns.boxplot(data=filtered_cells_plot1, x='transition_plot1', y='G2M_score',
            order=transition_order_plot1, palette=transition_colors_plot1, ax=axes[2])
axes[2].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[2].set_ylabel('G2M Phase Score', fontsize=11, fontweight='bold')
axes[2].set_title('G2M Phase Score', fontsize=12, fontweight='bold')
axes[2].tick_params(axis='x', rotation=45, labelsize=9)

plt.suptitle('Cell Cycle Scores: Secretory→Enterocyte Transitions (Scanpy Scoring, DCM Expression)', 
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{output_dir}/cell_cycle_scores_secretory_plot1.pdf', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: cell_cycle_scores_secretory_plot1.pdf")

# =============================================================================
# PLOT 2: TUFT-FOCUSED TRANSITIONS
# =============================================================================

print("\n" + "="*80)
print("PREPARING PLOT 2: TUFT-FOCUSED TRANSITIONS")
print("="*80)

def classify_transition_plot2(row):
    dcm = row['dcm_celltype']
    rna = row['rna_celltype']
    
    # Tuft to Stem
    if dcm == 'Tuft' and rna == 'Stem':
        return 'Tuft→Stem', 'Tuft→Stem'
    
    # Tuft to TA
    elif dcm == 'Tuft' and rna == 'TA':
        return 'Tuft→TA', 'Tuft→TA'
    
    # Tuft to Enterocyte
    elif dcm == 'Tuft' and rna == 'Enterocyte':
        return 'Tuft→Enterocyte', 'Tuft→Enterocyte'
    
    # Tuft to other secretory types
    elif dcm == 'Tuft' and rna == 'Paneth':
        return 'Tuft→Paneth', 'Tuft→Paneth'
    elif dcm == 'Tuft' and rna == 'Goblet':
        return 'Tuft→Goblet', 'Tuft→Goblet'
    elif dcm == 'Tuft' and rna == 'Tuft':
        return 'Tuft→Tuft', 'Tuft→Tuft'
    elif dcm == 'Tuft' and rna == 'EEC':
        return 'Tuft→EEC', 'Tuft→EEC'
    
    # TA → Enterocyte (control)
    elif dcm == 'TA' and rna == 'Enterocyte':
        return 'TA→Enterocyte', 'TA→Enterocyte'
    
    else:
        return 'Other', f'{dcm}→{rna}'

paired_cells['transition_plot2'], paired_cells['specific_transition_plot2'] = zip(
    *paired_cells.apply(classify_transition_plot2, axis=1)
)

# Define the exact order for plot 2
transition_order_plot2 = [
    'Tuft→Stem', 
    'Tuft→TA', 
    'Tuft→Enterocyte', 
    'Tuft→Paneth', 
    'Tuft→Goblet', 
    'Tuft→Tuft', 
    'Tuft→EEC', 
    'TA→Enterocyte'
]

# Filter to these transitions
filtered_cells_plot2 = paired_cells[paired_cells['transition_plot2'].isin(transition_order_plot2)].copy()

# Only include transitions that exist in the data
transition_order_plot2 = [t for t in transition_order_plot2 if t in filtered_cells_plot2['transition_plot2'].values]

print(f"\nFiltered to {len(filtered_cells_plot2)} cells for Plot 2")
print("\nTransition counts:")
print(filtered_cells_plot2['transition_plot2'].value_counts())

# Create Plot 2
print("\nCreating Plot 2...")

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Define colors for plot 2
transition_colors_plot2 = {
    'Tuft→Stem': '#9370DB',        # Purple
    'Tuft→TA': '#D2B48C',          # Tan
    'Tuft→Enterocyte': '#FF8C00',  # Orange
    'Tuft→Paneth': '#DC143C',      # Red
    'Tuft→Goblet': '#2E8B57',      # Green
    'Tuft→Tuft': '#FF1493',        # Pink
    'Tuft→EEC': '#4169E1',         # Blue
    'TA→Enterocyte': '#3498db'     # Light Blue (control)
}

# Plot 1: Combined score
sns.boxplot(data=filtered_cells_plot2, x='transition_plot2', y='combined_score',
            order=transition_order_plot2, palette=transition_colors_plot2, ax=axes[0])
axes[0].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Combined Cell Cycle Score\n(S + G2M)', fontsize=11, fontweight='bold')
axes[0].set_title('Combined Cell Cycle Score', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45, labelsize=9)

# Add n values
for i, trans in enumerate(transition_order_plot2):
    n = len(filtered_cells_plot2[filtered_cells_plot2['transition_plot2'] == trans])
    axes[0].text(i, axes[0].get_ylim()[1] * 0.95, f'n={n}', 
                ha='center', fontsize=8, fontweight='bold')

# T-test vs TA→Enterocyte control
ta_ent_vals = filtered_cells_plot2[filtered_cells_plot2['transition_plot2'] == 'TA→Enterocyte']['combined_score']
for i, trans in enumerate(transition_order_plot2[:-1]):  # Skip the control itself
    trans_vals = filtered_cells_plot2[filtered_cells_plot2['transition_plot2'] == trans]['combined_score']
    if len(trans_vals) > 0 and len(ta_ent_vals) > 0:
        tstat, pval = stats.ttest_ind(trans_vals, ta_ent_vals)
        if pval < 0.05:
            sig_marker = '***' if pval < 0.001 else '**' if pval < 0.01 else '*'
            axes[0].text(i, axes[0].get_ylim()[1] * 0.88, sig_marker, 
                        ha='center', fontsize=12, fontweight='bold')

# Plot 2: S score
sns.boxplot(data=filtered_cells_plot2, x='transition_plot2', y='S_score',
            order=transition_order_plot2, palette=transition_colors_plot2, ax=axes[1])
axes[1].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[1].set_ylabel('S Phase Score', fontsize=11, fontweight='bold')
axes[1].set_title('S Phase Score', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45, labelsize=9)

# Plot 3: G2M score
sns.boxplot(data=filtered_cells_plot2, x='transition_plot2', y='G2M_score',
            order=transition_order_plot2, palette=transition_colors_plot2, ax=axes[2])
axes[2].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[2].set_ylabel('G2M Phase Score', fontsize=11, fontweight='bold')
axes[2].set_title('G2M Phase Score', fontsize=12, fontweight='bold')
axes[2].tick_params(axis='x', rotation=45, labelsize=9)

plt.suptitle('Cell Cycle Scores: Tuft Cell Transitions (Scanpy Scoring, DCM Expression)', 
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{output_dir}/cell_cycle_scores_tuft_plot2.pdf', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: cell_cycle_scores_tuft_plot2.pdf")

# =============================================================================
# PLOT 3: TRANSITIONS TO TA
# =============================================================================

print("\n" + "="*80)
print("PREPARING PLOT 3: TRANSITIONS TO TA")
print("="*80)

def classify_transition_plot3(row):
    dcm = row['dcm_celltype']
    rna = row['rna_celltype']
    
    # All transitions to TA
    if rna == 'TA':
        if dcm == 'Stem':
            return 'Stem→TA', 'Stem→TA'
        elif dcm == 'TA':
            return 'TA→TA', 'TA→TA'
        elif dcm == 'Enterocyte':
            return 'Enterocyte→TA', 'Enterocyte→TA'
        elif dcm == 'Paneth':
            return 'Paneth→TA', 'Paneth→TA'
        elif dcm == 'Goblet':
            return 'Goblet→TA', 'Goblet→TA'
        elif dcm == 'Tuft':
            return 'Tuft→TA', 'Tuft→TA'
        elif dcm == 'EEC':
            return 'EEC→TA', 'EEC→TA'
    
    # TA → Enterocyte (control)
    elif dcm == 'TA' and rna == 'Enterocyte':
        return 'TA→Enterocyte', 'TA→Enterocyte'
    
    return 'Other', f'{dcm}→{rna}'

paired_cells['transition_plot3'], paired_cells['specific_transition_plot3'] = zip(
    *paired_cells.apply(classify_transition_plot3, axis=1)
)

# Define the exact order for plot 3 - NOW INCLUDING TA→Enterocyte
transition_order_plot3 = [
    'Stem→TA',
    'TA→TA',
    'Enterocyte→TA',
    'Paneth→TA',
    'Goblet→TA',
    'Tuft→TA',
    'EEC→TA',
    'TA→Enterocyte'
]

# Filter to these transitions
filtered_cells_plot3 = paired_cells[paired_cells['transition_plot3'].isin(transition_order_plot3)].copy()

# Only include transitions that exist in the data
transition_order_plot3 = [t for t in transition_order_plot3 if t in filtered_cells_plot3['transition_plot3'].values]

print(f"\nFiltered to {len(filtered_cells_plot3)} cells for Plot 3")
print("\nTransition counts:")
print(filtered_cells_plot3['transition_plot3'].value_counts())

# Create Plot 3
print("\nCreating Plot 3...")

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Define colors for plot 3
transition_colors_plot3 = {
    'Stem→TA': '#9370DB',          # Purple
    'TA→TA': '#D2B48C',            # Tan
    'Enterocyte→TA': '#FF8C00',    # Orange
    'Paneth→TA': '#DC143C',        # Red
    'Goblet→TA': '#2E8B57',        # Green
    'Tuft→TA': '#FF1493',          # Pink
    'EEC→TA': '#4169E1',           # Blue
    'TA→Enterocyte': '#3498db'     # Light Blue (control)
}

# Plot 1: Combined score
sns.boxplot(data=filtered_cells_plot3, x='transition_plot3', y='combined_score',
            order=transition_order_plot3, palette=transition_colors_plot3, ax=axes[0])
axes[0].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Combined Cell Cycle Score\n(S + G2M)', fontsize=11, fontweight='bold')
axes[0].set_title('Combined Cell Cycle Score', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45, labelsize=9)

# Add n values
for i, trans in enumerate(transition_order_plot3):
    n = len(filtered_cells_plot3[filtered_cells_plot3['transition_plot3'] == trans])
    axes[0].text(i, axes[0].get_ylim()[1] * 0.95, f'n={n}', 
                ha='center', fontsize=8, fontweight='bold')

# T-test vs TA→Enterocyte control
ta_ent_vals_plot3 = filtered_cells_plot3[filtered_cells_plot3['transition_plot3'] == 'TA→Enterocyte']['combined_score']
for i, trans in enumerate(transition_order_plot3[:-1]):  # Skip the control itself
    trans_vals = filtered_cells_plot3[filtered_cells_plot3['transition_plot3'] == trans]['combined_score']
    if len(trans_vals) > 0 and len(ta_ent_vals_plot3) > 0:
        tstat, pval = stats.ttest_ind(trans_vals, ta_ent_vals_plot3)
        if pval < 0.05:
            sig_marker = '***' if pval < 0.001 else '**' if pval < 0.01 else '*'
            axes[0].text(i, axes[0].get_ylim()[1] * 0.88, sig_marker, 
                        ha='center', fontsize=12, fontweight='bold')

# Plot 2: S score
sns.boxplot(data=filtered_cells_plot3, x='transition_plot3', y='S_score',
            order=transition_order_plot3, palette=transition_colors_plot3, ax=axes[1])
axes[1].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[1].set_ylabel('S Phase Score', fontsize=11, fontweight='bold')
axes[1].set_title('S Phase Score', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45, labelsize=9)

# Plot 3: G2M score
sns.boxplot(data=filtered_cells_plot3, x='transition_plot3', y='G2M_score',
            order=transition_order_plot3, palette=transition_colors_plot3, ax=axes[2])
axes[2].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[2].set_ylabel('G2M Phase Score', fontsize=11, fontweight='bold')
axes[2].set_title('G2M Phase Score', fontsize=12, fontweight='bold')
axes[2].tick_params(axis='x', rotation=45, labelsize=9)

plt.suptitle('Cell Cycle Scores: Transitions to TA (Scanpy Scoring, DCM Expression)', 
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{output_dir}/cell_cycle_scores_to_TA_plot3.pdf', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: cell_cycle_scores_to_TA_plot3.pdf")

# -----------------------------
# STATISTICS FOR ALL PLOTS
# -----------------------------
print("\n" + "="*80)
print("CALCULATING STATISTICS")
print("="*80)

def get_significance(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return 'ns'

# Statistics for Plot 1
print("\nPlot 1 Statistics:")
plot1_stats = []
sec_sec_vals = filtered_cells_plot1[filtered_cells_plot1['transition_plot1'] == 'Secretory→Secretory']['combined_score']

for trans in transition_order_plot1:
    subset = filtered_cells_plot1[filtered_cells_plot1['transition_plot1'] == trans]
    
    if len(subset) > 0:
        # T-test vs Secretory→Secretory if not the control itself
        if trans != 'Secretory→Secretory' and len(sec_sec_vals) > 0:
            tstat, pval = stats.ttest_ind(subset['combined_score'], sec_sec_vals)
        else:
            tstat, pval = np.nan, np.nan
        
        plot1_stats.append({
            'Transition': trans,
            'N': len(subset),
            'Combined_Mean': subset['combined_score'].mean(),
            'Combined_Median': subset['combined_score'].median(),
            'Combined_Std': subset['combined_score'].std(),
            'S_Mean': subset['S_score'].mean(),
            'G2M_Mean': subset['G2M_score'].mean(),
            't_stat_vs_control': tstat,
            'p_value_vs_control': pval,
            'significance': get_significance(pval) if not np.isnan(pval) else '-'
        })

plot1_stats_df = pd.DataFrame(plot1_stats)
plot1_stats_df.to_csv(f'{output_dir}/cell_cycle_plot1_statistics.csv', index=False)
print(f"✅ Saved: {output_dir}/cell_cycle_plot1_statistics.csv")

# Statistics for Plot 2
print("\nPlot 2 Statistics:")
plot2_stats = []
ta_ent_vals = filtered_cells_plot2[filtered_cells_plot2['transition_plot2'] == 'TA→Enterocyte']['combined_score']

for trans in transition_order_plot2:
    subset = filtered_cells_plot2[filtered_cells_plot2['transition_plot2'] == trans]
    
    if len(subset) > 0:
        # T-test vs TA→Enterocyte if not the control itself
        if trans != 'TA→Enterocyte' and len(ta_ent_vals) > 0:
            tstat, pval = stats.ttest_ind(subset['combined_score'], ta_ent_vals)
        else:
            tstat, pval = np.nan, np.nan
        
        plot2_stats.append({
            'Transition': trans,
            'N': len(subset),
            'Combined_Mean': subset['combined_score'].mean(),
            'Combined_Median': subset['combined_score'].median(),
            'Combined_Std': subset['combined_score'].std(),
            'S_Mean': subset['S_score'].mean(),
            'G2M_Mean': subset['G2M_score'].mean(),
            't_stat_vs_control': tstat,
            'p_value_vs_control': pval,
            'significance': get_significance(pval) if not np.isnan(pval) else '-'
        })

plot2_stats_df = pd.DataFrame(plot2_stats)
plot2_stats_df.to_csv(f'{output_dir}/cell_cycle_plot2_statistics.csv', index=False)
print(f"✅ Saved: {output_dir}/cell_cycle_plot2_statistics.csv")

# Statistics for Plot 3
print("\nPlot 3 Statistics:")
plot3_stats = []

# Use TA→Enterocyte as control
ta_ent_vals_plot3 = filtered_cells_plot3[filtered_cells_plot3['transition_plot3'] == 'TA→Enterocyte']['combined_score']

for trans in transition_order_plot3:
    subset = filtered_cells_plot3[filtered_cells_plot3['transition_plot3'] == trans]
    
    if len(subset) > 0:
        # T-test vs TA→Enterocyte if not the control itself
        if trans != 'TA→Enterocyte' and len(ta_ent_vals_plot3) > 0:
            tstat, pval = stats.ttest_ind(subset['combined_score'], ta_ent_vals_plot3)
        else:
            tstat, pval = np.nan, np.nan
        
        plot3_stats.append({
            'Transition': trans,
            'N': len(subset),
            'Combined_Mean': subset['combined_score'].mean(),
            'Combined_Median': subset['combined_score'].median(),
            'Combined_Std': subset['combined_score'].std(),
            'S_Mean': subset['S_score'].mean(),
            'G2M_Mean': subset['G2M_score'].mean(),
            't_stat_vs_control': tstat,
            'p_value_vs_control': pval,
            'significance': get_significance(pval) if not np.isnan(pval) else '-'
        })

plot3_stats_df = pd.DataFrame(plot3_stats)
plot3_stats_df.to_csv(f'{output_dir}/cell_cycle_plot3_statistics.csv', index=False)
print(f"✅ Saved: {output_dir}/cell_cycle_plot3_statistics.csv")

# Save data
filtered_cells_plot1.to_csv(f'{output_dir}/cell_cycle_plot1_data.csv', index=False)
filtered_cells_plot2.to_csv(f'{output_dir}/cell_cycle_plot2_data.csv', index=False)
filtered_cells_plot3.to_csv(f'{output_dir}/cell_cycle_plot3_data.csv', index=False)
print(f"✅ Saved: {output_dir}/cell_cycle_plot1_data.csv")
print(f"✅ Saved: {output_dir}/cell_cycle_plot2_data.csv")
print(f"✅ Saved: {output_dir}/cell_cycle_plot3_data.csv")

# Save gene lists used
gene_info = pd.DataFrame({
    'Gene': s_genes_found + g2m_genes_found,
    'Phase': ['S']*len(s_genes_found) + ['G2M']*len(g2m_genes_found)
})
gene_info.to_csv(f'{output_dir}/cell_cycle_genes_used.csv', index=False)
print(f"✅ Saved: {output_dir}/cell_cycle_genes_used.csv")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print("\nPlot 1 - Secretory Transitions:")
print(plot1_stats_df[['Transition', 'N', 'Combined_Mean', 'p_value_vs_control', 'significance']])

print("\nPlot 2 - Tuft Transitions:")
print(plot2_stats_df[['Transition', 'N', 'Combined_Mean', 'p_value_vs_control', 'significance']])

print("\nPlot 3 - Transitions to TA:")
print(plot3_stats_df[['Transition', 'N', 'Combined_Mean', 'p_value_vs_control', 'significance']])

print("\n" + "="*80)
print("DONE!")
print("="*80)

## Cell cycle genes in TA marker genes??

In [ ]:
# Cell cycle genes from Tirosh et al. 2016
s_genes = ['MCM5', 'PCNA', 'TYMS', 'FEN1', 'MCM2', 'MCM4', 'RRM1', 'UNG', 'GINS2', 'MCM6', 
           'CDCA7', 'DTL', 'PRIM1', 'UHRF1', 'MLF1IP', 'HELLS', 'RFC2', 'RPA2', 'NASP', 
           'RAD51AP1', 'GMNN', 'WDR76', 'SLBP', 'CCNE2', 'UBR7', 'POLD3', 'MSH2', 'ATAD2', 
           'RAD51', 'RRM2', 'CDC45', 'CDC6', 'EXO1', 'TIPIN', 'DSCC1', 'BLM', 'CASP8AP2', 
           'USP1', 'CLSPN', 'POLA1', 'CHAF1B', 'BRIP1', 'E2F8']

g2m_genes = ['HMGB2', 'CDK1', 'NUSAP1', 'UBE2C', 'BIRC5', 'TPX2', 'TOP2A', 'NDC80', 'CKS2', 
             'NUF2', 'CKS1B', 'MKI67', 'TMPO', 'CENPF', 'TACC3', 'FAM64A', 'SMC4', 'CCNB2', 
             'CKAP2L', 'CKAP2', 'AURKB', 'BUB1', 'KIF11', 'ANP32E', 'TUBB4B', 'GTSE1', 'KIF20B', 
             'HJURP', 'CDCA3', 'HN1', 'CDC20', 'TTK', 'CDC25C', 'KIF2C', 'RANGAP1', 'NCAPD2', 
             'DLGAP5', 'CDCA2', 'CDCA8', 'ECT2', 'KIF23', 'HMMR', 'AURKA', 'PSRC1', 'ANLN', 
             'LBR', 'CKAP5', 'CENPE', 'CTCF', 'NEK2', 'G2E3', 'GAS2L3', 'CBX5', 'CENPA']

In [ ]:
s_genes = [gene.capitalize() for gene in s_genes]
g2m_genes = [gene.capitalize() for gene in g2m_genes]

In [ ]:
ttest_markers = pd.read_csv(
    "/media/erlantz/71DE167A75E99BB4/Pool32/Combi/Python/integration/gene_scoring/marker_gene_sets/ttest_markers_13012026.csv"
)
lit_markers = pd.read_csv(
    "/media/erlantz/71DE167A75E99BB4/Pool32/Combi/Python/integration/gene_scoring/marker_gene_sets/literature_top_markers_13012026.csv"
)

In [ ]:
cell_cycle_genes = set(s_genes) | set(g2m_genes)

for cluster in ttest_markers['cell_type'].unique():
    cluster_genes = ttest_markers[ttest_markers['cell_type'] == cluster]['gene'].values
    total_cluster_genes = len(cluster_genes)
    
    overlap = len(cell_cycle_genes & set(cluster_genes))
    
    # % of cell cycle genes covered
    cc_pct = (overlap / len(cell_cycle_genes)) * 100
    
    # % of cluster markers that are cell cycle genes
    cluster_pct = (overlap / total_cluster_genes) * 100
    
    print(f"Cluster: {cluster} ({total_cluster_genes} marker genes)")
    print(f"  Cell cycle overlap: {overlap}/{len(cell_cycle_genes)} cell cycle genes ({cc_pct:.1f}%) | {overlap}/{total_cluster_genes} cluster markers ({cluster_pct:.1f}%)")
    print()

In [ ]:
cell_cycle_genes = set(s_genes) | set(g2m_genes)

for cluster in ttest_marker_adata['cell_type'].unique():
    cluster_genes = ttest_marker_adata[ttest_marker_adata['cell_type'] == cluster]['gene'].values
    total_cluster_genes = len(cluster_genes)
    
    overlap = len(cell_cycle_genes & set(cluster_genes))
    
    # % of cell cycle genes covered
    cc_pct = (overlap / len(cell_cycle_genes)) * 100
    
    # % of cluster markers that are cell cycle genes
    cluster_pct = (overlap / total_cluster_genes) * 100
    
    print(f"Cluster: {cluster} ({total_cluster_genes} marker genes)")
    print(f"  Cell cycle overlap: {overlap}/{len(cell_cycle_genes)} cell cycle genes ({cc_pct:.1f}%) | {overlap}/{total_cluster_genes} cluster markers ({cluster_pct:.1f}%)")
    print()

In [ ]:
cell_cycle_genes = set(s_genes) | set(g2m_genes)

for cluster in lit_markers['cell_type'].unique():
    cluster_genes = lit_markers[lit_markers['cell_type'] == cluster]['gene'].values
    total_cluster_genes = len(cluster_genes)
    
    overlap = len(cell_cycle_genes & set(cluster_genes))
    
    # % of cell cycle genes covered
    cc_pct = (overlap / len(cell_cycle_genes)) * 100
    
    # % of cluster markers that are cell cycle genes
    cluster_pct = (overlap / total_cluster_genes) * 100
    
    print(f"Cluster: {cluster} ({total_cluster_genes} marker genes)")
    print(f"  Cell cycle overlap: {overlap}/{len(cell_cycle_genes)} cell cycle genes ({cc_pct:.1f}%) | {overlap}/{total_cluster_genes} cluster markers ({cluster_pct:.1f}%)")
    print()

## Cell cycle GOs

In [ ]:
import gseapy as gp

go_results = {}

for cluster in ttest_markers['cell_type'].unique():
    cluster_genes = ttest_markers[ttest_markers['cell_type'] == cluster]['gene'].tolist()
    
    enr = gp.enrichr(
        gene_list=cluster_genes,
        gene_sets=['GO_Biological_Process_2023'],  # can also use 'KEGG_2019_Mouse', 'Reactome_2022'
        organism='mouse',
        outdir=None,
    )
    
    go_results[cluster] = enr.results
    
    print(f"\n=== Cluster: {cluster} ===")
    top = enr.results.sort_values('Adjusted P-value').head(10)
    for _, row in top.iterrows():
        print(f"  {row['Term']} | P-adj: {row['Adjusted P-value']:.2e} | Genes: {row['Overlap']}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(go_results), 1, figsize=(12, 5 * len(go_results)))

for ax, (cluster, res) in zip(axes, go_results.items()):
    top = res.sort_values('Adjusted P-value').head(10).iloc[::-1]
    ax.barh(top['Term'], -np.log10(top['Adjusted P-value']))
    ax.set_title(f'Cluster: {cluster}')
    ax.set_xlabel('-log10(Adjusted P-value)')

plt.tight_layout()
plt.show()

In [ ]:
gene_sets = [
    'GO_Biological_Process_2023',
    'GO_Molecular_Function_2023',
    'GO_Cellular_Component_2023',
    'KEGG_2021_Human',
    'Reactome_2022',
]

In [ ]:
# Get genes from adata
adata_genes = set(adata_combined.var_names)

# Filter ttest_markers to only include genes present in adata
ttest_marker_adata = ttest_markers[ttest_markers['gene'].isin(adata_genes)].copy()

print(f"Original ttest_markers: {len(ttest_markers)} entries")
print(f"After intersection: {len(ttest_marker_adata)} entries")
print(f"Genes removed: {len(ttest_markers) - len(ttest_marker_adata)}")

In [ ]:
print(ttest_markers.groupby('cell_type')['gene'].count())


In [ ]:
import gseapy as gp

go_results = {}

for cluster in ttest_marker_adata['cell_type'].unique():
    cluster_genes = ttest_marker_adata[ttest_marker_adata['cell_type'] == cluster]['gene'].tolist()
    
    enr = gp.enrichr(
        gene_list=cluster_genes,
        gene_sets=['GO_Biological_Process_2023'],  # can also use 'KEGG_2019_Mouse', 'Reactome_2022'
        organism='mouse',
        outdir=None,
    )
    
    go_results[cluster] = enr.results
    
    print(f"\n=== Cluster: {cluster} ===")
    top = enr.results.sort_values('Adjusted P-value').head(10)
    for _, row in top.iterrows():
        print(f"  {row['Term']} | P-adj: {row['Adjusted P-value']:.2e} | Genes: {row['Overlap']}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

for cluster, res in go_results.items():
    fig, ax = plt.subplots(figsize=(14, 6))
    
    top = res.sort_values('Adjusted P-value').head(10).iloc[::-1]
    bars = ax.barh(top['Term'], -np.log10(top['Adjusted P-value']))
    
    for bar, overlap in zip(bars, top['Overlap']):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, 
                f' {overlap}', va='center', fontsize=9)
    
    ax.set_title(f'Cluster: {cluster}')
    ax.set_xlabel('-log10(Adjusted P-value)')
    plt.tight_layout()
    
    # Clean cluster name for filename (remove spaces/special chars)
    clean_name = cluster.replace(' ', '_').replace('/', '_')
    fig.savefig(f'GO_enrichment_{clean_name}.pdf', dpi=300, bbox_inches='tight')
    plt.close(fig)
    
    print(f"Saved: GO_enrichment_{clean_name}.pdf")

In [ ]:
import gseapy as gp

go_results = {}

for cluster in lit_markers['cell_type'].unique():
    cluster_genes = lit_markers[lit_markers['cell_type'] == cluster]['gene'].tolist()
    
    enr = gp.enrichr(
        gene_list=cluster_genes,
        gene_sets=['GO_Biological_Process_2023'],  # can also use 'KEGG_2019_Mouse', 'Reactome_2022'
        organism='mouse',
        outdir=None,
    )
    
    go_results[cluster] = enr.results
    
    print(f"\n=== Cluster: {cluster} ===")
    top = enr.results.sort_values('Adjusted P-value').head(10)
    for _, row in top.iterrows():
        print(f"  {row['Term']} | P-adj: {row['Adjusted P-value']:.2e} | Genes: {row['Overlap']}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

for cluster, res in go_results.items():
    fig, ax = plt.subplots(figsize=(14, 6))
    
    top = res.sort_values('Adjusted P-value').head(10).iloc[::-1]
    bars = ax.barh(top['Term'], -np.log10(top['Adjusted P-value']))
    
    for bar, overlap in zip(bars, top['Overlap']):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, 
                f' {overlap}', va='center', fontsize=9)
    
    ax.set_title(f'Cluster: {cluster}')
    ax.set_xlabel('-log10(Adjusted P-value)')
    plt.tight_layout()
    
    # Clean cluster name for filename (remove spaces/special chars)
    clean_name = cluster.replace(' ', '_').replace('/', '_')
    fig.savefig(f'GO_enrichment_literature_{clean_name}.pdf', dpi=300, bbox_inches='tight')
    plt.close(fig)
    
    print(f"Saved: GO_enrichment_literature_{clean_name}.pdf")

In [ ]:
# Step 1: Get ISC cells from DCM assay (ttest_gmm_label_confident)
isc_dcm = adata_combined.obs[
    (adata_combined.obs['assay'] == 'pDCM') & 
    (adata_combined.obs['ttest_gmm_label_confident'] == 'Stem')
]

# Step 2: Get ALL cells in RNA assay (leiden_annot_25)
rna_cells = adata_combined.obs[
    (adata_combined.obs['assay'] == 'pRNA')
]

# Step 3: Find batch_plate_well IDs that are ISC in DCM and present in RNA
transit_ids = set(isc_dcm['batch_plate_well']) & set(rna_cells['batch_plate_well'])

print(f"ISC cells in DCM: {len(isc_dcm['batch_plate_well'].unique())}")
print(f"Total cells in RNA: {len(rna_cells['batch_plate_well'].unique())}")
print(f"ISC (DCM) cells found in RNA: {len(transit_ids)}")

# Filter to matching cells
transit_dcm = isc_dcm[isc_dcm['batch_plate_well'].isin(transit_ids)]
transit_rna = rna_cells[rna_cells['batch_plate_well'].isin(transit_ids)]

# Show what cell types they are in RNA
print("\nRNA cell type distribution of DCM-ISC cells:")
print(transit_rna['leiden_annot_25'].value_counts())

In [ ]:
# ISC
isc_dcm = adata_combined.obs[
    (adata_combined.obs['assay'] == 'pDCM') & 
    (adata_combined.obs['ttest_gmm_label_confident'] == 'Stem')
]

# TA
ta_dcm = adata_combined.obs[
    (adata_combined.obs['assay'] == 'pDCM') & 
    (adata_combined.obs['ttest_gmm_label_confident'] == 'TA')
]

# RNA cells
rna_cells = adata_combined.obs[
    (adata_combined.obs['assay'] == 'pRNA')
]

# --- ISC ---
isc_transit_ids = set(isc_dcm['batch_plate_well']) & set(rna_cells['batch_plate_well'])
isc_dcm_match = isc_dcm[isc_dcm['batch_plate_well'].isin(isc_transit_ids)][['batch_plate_well', 'ttest_gmm_label_confident']].drop_duplicates()
isc_rna_match = rna_cells[rna_cells['batch_plate_well'].isin(isc_transit_ids)][['batch_plate_well', 'leiden_annot_25']].drop_duplicates()

isc_table = isc_dcm_match.merge(isc_rna_match, on='batch_plate_well')
isc_table.columns = ['batch_plate_well', 'DCM_label', 'RNA_label']

# --- TA ---
ta_transit_ids = set(ta_dcm['batch_plate_well']) & set(rna_cells['batch_plate_well'])
ta_dcm_match = ta_dcm[ta_dcm['batch_plate_well'].isin(ta_transit_ids)][['batch_plate_well', 'ttest_gmm_label_confident']].drop_duplicates()
ta_rna_match = rna_cells[rna_cells['batch_plate_well'].isin(ta_transit_ids)][['batch_plate_well', 'leiden_annot_25']].drop_duplicates()

ta_table = ta_dcm_match.merge(ta_rna_match, on='batch_plate_well')
ta_table.columns = ['batch_plate_well', 'DCM_label', 'RNA_label']

# Combine both
transit_table = pd.concat([isc_table, ta_table], ignore_index=True)

print(f"ISC matches: {len(isc_table)}")
print(f"TA matches: {len(ta_table)}")
print(f"Total: {len(transit_table)}")
print()
print(transit_table)

# Summary crosstab
print("\nCrosstab DCM vs RNA labels:")
print(pd.crosstab(transit_table['DCM_label'], transit_table['RNA_label']))

transit_table.to_csv('/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/CellRank/DCM_RNA_transit_table_stem_TA.csv', index=False)
print("Saved: DCM_RNA_transit_table.csv")

In [ ]:
###FDR

In [ ]:
root = "/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/"

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.interpolate import interp1d

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/FDR/'
output_dir = score_dir

# -----------------------------
# EXTRACT POSTERIOR COLUMNS
# -----------------------------
print("="*80)
print("EXTRACTING POSTERIOR PROBABILITIES FROM DCM CELLS")
print("="*80)

# Filter to DCM cells only
adata_dcm = adata_combined[adata_combined.obs['assay'] == 'pDCM'].copy()
print(f"\nTotal DCM cells: {len(adata_dcm)}")

# Find all posterior columns, excluding 'max'
posterior_cols = [col for col in adata_dcm.obs.columns 
                 if col.startswith('ttest_') and col.endswith('_posterior') 
                 and 'max' not in col]

print(f"\nPosterior columns found ({len(posterior_cols)}):")
for col in posterior_cols:
    print(f"  {col}")

# Extract cell type names from column names
cell_types = [col.replace('ttest_', '').replace('_posterior', '') for col in posterior_cols]
print(f"\nCell types: {cell_types}")

# Build posterior DataFrame
posterior_df = adata_dcm.obs[posterior_cols].copy()
posterior_df.columns = cell_types
posterior_df.index = adata_dcm.obs.index

# Max posterior and class per cell
max_posteriors = posterior_df.max(axis=1)
max_classes = posterior_df.idxmax(axis=1)

# -----------------------------
# FIND THE 4.99% ERROR RATE THRESHOLD
# -----------------------------
print("\n" + "="*80)
print("FINDING 4.99% ERROR RATE THRESHOLD")
print("="*80)

thresh_sweep = np.arange(0.50, 1.0, 0.001)  # Fine-grained sweep
sweep_results = []

for t in thresh_sweep:
    mask = max_posteriors >= t
    n = mask.sum()
    if n > 0:
        expected_error = (1 - max_posteriors[mask]).mean() * 100
        sweep_results.append({
            'threshold': t,
            'n_calls': n,
            'expected_error': expected_error
        })

sweep_df = pd.DataFrame(sweep_results)

# Interpolate to find exact threshold where error = 4.99%
interp_func = interp1d(
    sweep_df['expected_error'].values[::-1],  # Reverse: error is decreasing
    sweep_df['threshold'].values[::-1],
    kind='linear',
    bounds_error=False,
    fill_value='extrapolate'
)

threshold_499 = float(interp_func(4.99))
# Get n_calls at that threshold
n_calls_499 = (max_posteriors >= threshold_499).sum()
# Verify the error rate
actual_error_499 = (1 - max_posteriors[max_posteriors >= threshold_499]).mean() * 100

print(f"\n*** Threshold for 4.99% expected error rate: {threshold_499:.4f} ***")
print(f"    Cells passing this threshold: {n_calls_499} ({n_calls_499/len(max_posteriors)*100:.1f}%)")
print(f"    Actual error rate at this threshold: {actual_error_499:.2f}%")

# Also print the standard thresholds for comparison
print(f"\nComparison of thresholds:")
print(f"{'Threshold':<14} {'N cells':<12} {'% retained':<14} {'Expected error'}")
print("-" * 56)
for t in [threshold_499, 0.90, 0.95, 0.99]:
    mask = max_posteriors >= t
    n = mask.sum()
    err = (1 - max_posteriors[mask]).mean() * 100 if n > 0 else 0
    marker = " <-- 4.99% error" if t == threshold_499 else ""
    print(f"{t:<14.4f} {n:<12} {n/len(max_posteriors)*100:<14.1f} {err:.2f}%{marker}")

# -----------------------------
# PLOT 1: POSTERIOR DIAGNOSTICS (with 4.99% threshold)
# -----------------------------
print("\n" + "="*80)
print("CREATING POSTERIOR DIAGNOSTICS PLOT")
print("="*80)

cell_type_colors = {
    'Stem': '#9370DB', 'TA': '#D2B48C', 'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C', 'Goblet': '#2E8B57', 'Tuft': '#FF1493', 'EEC': '#4169E1'
}

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Plot 1: Distribution of max posteriors
axes[0, 0].hist(max_posteriors, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(threshold_499, color='green', linestyle='-', linewidth=2.5, 
                   label=f'4.99% error (thresh={threshold_499:.3f})')
axes[0, 0].axvline(0.95, color='red', linestyle='--', linewidth=1.5, label='thresh=0.95')
axes[0, 0].axvline(0.90, color='orange', linestyle='--', linewidth=1.5, label='thresh=0.90')
axes[0, 0].set_xlabel('Max Posterior Probability', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Number of Cells', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Distribution of Max Posterior Probabilities', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(axis='y', alpha=0.3)

# Plot 2: Max posterior per cell type
bp_data = []
bp_labels = []
bp_colors = []
for ct in cell_types:
    ct_mask = max_classes == ct
    if ct_mask.sum() > 0:
        bp_data.append(max_posteriors[ct_mask].values)
        bp_labels.append(f"{ct}\n(n={ct_mask.sum()})")
        bp_colors.append(cell_type_colors.get(ct, 'gray'))

bp = axes[0, 1].boxplot(bp_data, labels=bp_labels, patch_artist=True)
for patch, color in zip(bp['boxes'], bp_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0, 1].axhline(threshold_499, color='green', linestyle='-', linewidth=2.5, 
                   label=f'4.99% error (thresh={threshold_499:.3f})')
axes[0, 1].axhline(0.95, color='red', linestyle='--', linewidth=1.5, label='thresh=0.95')
axes[0, 1].set_ylabel('Max Posterior Probability', fontsize=11, fontweight='bold')
axes[0, 1].set_xlabel('Cell Type', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Max Posterior per Cell Type', fontsize=12, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(axis='y', alpha=0.3)

# Plot 3: Error rate vs threshold trade-off
ax_twin = axes[1, 0].twinx()
axes[1, 0].plot(sweep_df['threshold'], sweep_df['expected_error'], 
                color='steelblue', linewidth=2, label='Expected error rate')
ax_twin.plot(sweep_df['threshold'], sweep_df['n_calls'], 
             color='orange', linewidth=2, linestyle='--', label='N calls')

# 5% FDR reference line
axes[1, 0].axhline(5, color='red', linestyle=':', linewidth=1.5, label='5% FDR reference')

# 4.99% threshold marker
axes[1, 0].axvline(threshold_499, color='green', linestyle='-', linewidth=2.5, 
                   label=f'4.99% error (thresh={threshold_499:.3f})')
axes[1, 0].plot(threshold_499, 4.99, 'o', color='green', markersize=10, zorder=5)
axes[1, 0].annotate(f'  {threshold_499:.3f}\n  n={n_calls_499}', 
                    xy=(threshold_499, 4.99), fontsize=9, fontweight='bold', color='green')

axes[1, 0].set_xlabel('Posterior Threshold', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Expected Error Rate (%)', fontsize=11, fontweight='bold', color='steelblue')
ax_twin.set_ylabel('Number of Calls', fontsize=11, fontweight='bold', color='orange')
axes[1, 0].set_title('Error Rate vs Threshold Trade-off', fontsize=12, fontweight='bold')
axes[1, 0].set_ylim(0, 10)

lines1, labels1 = axes[1, 0].get_legend_handles_labels()
lines2, labels2 = ax_twin.get_legend_handles_labels()
axes[1, 0].legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)
axes[1, 0].grid(axis='y', alpha=0.3)

# Plot 4: Expected error rate per posterior bin
n_bins = 20
posterior_bins = pd.cut(max_posteriors, bins=n_bins)
calibration_data = pd.DataFrame({
    'posterior': max_posteriors,
    'bin': posterior_bins
}).groupby('bin').agg(
    mean_posterior=('posterior', 'mean'),
    expected_error=('posterior', lambda x: (1 - x).mean()),
    n_cells=('posterior', 'count')
).reset_index()

axes[1, 1].bar(range(len(calibration_data)), 
               calibration_data['expected_error'] * 100,
               color='steelblue', edgecolor='black', alpha=0.7)
axes[1, 1].axhline(4.99, color='green', linestyle='-', linewidth=2.5, label=f'4.99% error (thresh={threshold_499:.3f})')
axes[1, 1].axhline(5, color='red', linestyle='--', linewidth=1.5, label='5% FDR reference')

for i, row in calibration_data.iterrows():
    if row['n_cells'] > 0:
        axes[1, 1].text(i, row['expected_error'] * 100 + 0.5, 
                       f"n={int(row['n_cells'])}", ha='center', fontsize=7)

axes[1, 1].set_xticks(range(len(calibration_data)))
axes[1, 1].set_xticklabels(
    [f"{b.left:.2f}" for b in calibration_data['bin']], 
    rotation=45, ha='right', fontsize=7
)
axes[1, 1].set_xlabel('Posterior Probability Bin (left edge)', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Expected Error Rate (%)', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Expected Error Rate per Posterior Bin', fontsize=12, fontweight='bold')
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(axis='y', alpha=0.3)

plt.suptitle('Posterior Probability Diagnostics (DCM Cells)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{output_dir}/posterior_diagnostics.pdf', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: posterior_diagnostics.pdf")

# -----------------------------
# PLOT 2: PER CELL TYPE ANALYSIS WITH 4.99% THRESHOLD
# -----------------------------
print("\n" + "="*80)
print("PER CELL TYPE ANALYSIS")
print("="*80)

# Find per-cell-type 4.99% thresholds too
thresholds_standard = [0.90, 0.95, 0.99]
celltype_results = []
celltype_499_thresholds = {}

for ct in cell_types:
    ct_mask = max_classes == ct
    n_total = ct_mask.sum()
    if n_total == 0:
        continue
    
    ct_posteriors = max_posteriors[ct_mask]
    
    # Sweep to find per-cell-type 4.99% threshold
    ct_sweep = []
    for t in thresh_sweep:
        t_mask = ct_posteriors >= t
        n = t_mask.sum()
        if n > 0:
            ct_sweep.append({
                'threshold': t,
                'n_calls': n,
                'expected_error': (1 - ct_posteriors[t_mask]).mean() * 100
            })
    
    ct_sweep_df = pd.DataFrame(ct_sweep)
    
    # Try to interpolate for 4.99% threshold
    try:
        # Only interpolate if 4.99% is within the range
        if ct_sweep_df['expected_error'].min() < 4.99 < ct_sweep_df['expected_error'].max():
            ct_interp = interp1d(
                ct_sweep_df['expected_error'].values[::-1],
                ct_sweep_df['threshold'].values[::-1],
                kind='linear', bounds_error=False, fill_value='extrapolate'
            )
            ct_thresh_499 = float(ct_interp(4.99))
            celltype_499_thresholds[ct] = ct_thresh_499
        else:
            celltype_499_thresholds[ct] = None
    except:
        celltype_499_thresholds[ct] = None

    # Collect results at standard thresholds + 4.99% threshold
    all_thresholds = thresholds_standard.copy()
    if celltype_499_thresholds[ct] is not None:
        all_thresholds.append(celltype_499_thresholds[ct])
    
    for thresh in all_thresholds:
        t_mask = ct_posteriors >= thresh
        n_pass = t_mask.sum()
        if n_pass > 0:
            exp_err = (1 - ct_posteriors[t_mask]).mean()
            celltype_results.append({
                'Cell_type': ct,
                'Threshold': round(thresh, 4),
                'Is_499': thresh == celltype_499_thresholds[ct],
                'N_total': n_total,
                'N_pass': n_pass,
                'N_pass_pct': n_pass / n_total * 100,
                'Expected_error_rate': exp_err
            })

celltype_df = pd.DataFrame(celltype_results)

# Print summary
print(f"\n{'Cell Type':<14} {'4.99% Threshold':<18} {'N cells at 4.99%':<18} {'N cells at 0.95'}")
print("-" * 70)
for ct in cell_types:
    t499 = celltype_499_thresholds.get(ct)
    if t499 is not None:
        n_499 = (max_posteriors[max_classes == ct] >= t499).sum()
        n_95 = (max_posteriors[max_classes == ct] >= 0.95).sum()
        print(f"{ct:<14} {t499:<18.4f} {n_499:<18} {n_95}")
    else:
        n_95 = (max_posteriors[max_classes == ct] >= 0.95).sum()
        print(f"{ct:<14} {'N/A':<18} {'N/A':<18} {n_95}")

# -----------------------------
# PLOT 3: PER CELL TYPE ERROR CURVES WITH 4.99% MARKERS
# -----------------------------

fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# Plot 1: Error curves per cell type
for ct in cell_types:
    ct_mask = max_classes == ct
    if ct_mask.sum() == 0:
        continue
    
    ct_posteriors = max_posteriors[ct_mask]
    ct_errors = []
    ct_thresholds = []
    ct_n = []
    
    for t in thresh_sweep:
        t_mask = ct_posteriors >= t
        n = t_mask.sum()
        if n > 0:
            ct_errors.append((1 - ct_posteriors[t_mask]).mean() * 100)
            ct_thresholds.append(t)
            ct_n.append(n)
    
    axes[0].plot(ct_thresholds, ct_errors, linewidth=2, 
                color=cell_type_colors.get(ct, 'gray'), label=ct)
    
    # Mark the 4.99% threshold point
    if celltype_499_thresholds.get(ct) is not None:
        axes[0].plot(celltype_499_thresholds[ct], 4.99, 'o', 
                    color=cell_type_colors.get(ct, 'gray'), markersize=8, zorder=5)

# Reference lines
axes[0].axhline(4.99, color='green', linestyle='-', linewidth=2.5, 
               label=f'4.99% error target', alpha=0.7)
axes[0].axvline(threshold_499, color='green', linestyle=':', linewidth=1.5, alpha=0.5)
axes[0].axvline(0.95, color='red', linestyle='--', linewidth=1.5, alpha=0.5, label='thresh=0.95')

axes[0].set_xlabel('Posterior Threshold', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Expected Error Rate (%)', fontsize=11, fontweight='bold')
axes[0].set_title('Expected Error Rate per Cell Type', fontsize=12, fontweight='bold')
axes[0].set_ylim(0, 15)
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
axes[0].grid(alpha=0.3)

# Plot 2: N cells retained per cell type at key thresholds
key_thresholds = [threshold_499, 0.90, 0.95, 0.99]
key_labels = [f'{threshold_499:.3f}\n(4.99% err)', '0.90', '0.95', '0.99']

ct_list = [ct for ct in cell_types if (max_classes == ct).sum() > 0]
x_pos = np.arange(len(ct_list))
bar_width = 0.2

for i, (thresh, label) in enumerate(zip(key_thresholds, key_labels)):
    values = []
    for ct in ct_list:
        n = (max_posteriors[max_classes == ct] >= thresh).sum()
        values.append(n)
    
    color = '#2ecc71' if thresh == threshold_499 else plt.cm.viridis(i / len(key_thresholds))
    axes[1].bar(x_pos + i * bar_width, values, bar_width, label=label, 
               color=color, edgecolor='black', linewidth=0.5, alpha=0.8)

axes[1].set_xticks(x_pos + bar_width * 1.5)
axes[1].set_xticklabels(ct_list, rotation=45, ha='right')
axes[1].set_ylabel('Number of Cells Passing Threshold', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Cell Type', fontsize=11, fontweight='bold')
axes[1].set_title('Cells Retained per Cell Type at Different Thresholds', fontsize=12, fontweight='bold')
axes[1].legend(title='Threshold', fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{output_dir}/celltype_error_breakdown.pdf', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: celltype_error_breakdown.pdf")

# -----------------------------
# SAVE RESULTS
# -----------------------------
celltype_df.to_csv(f'{output_dir}/celltype_error_breakdown.csv', index=False)

# Save threshold summary
threshold_summary = pd.DataFrame([
    {
        'Threshold': t,
        'Label': '4.99% error' if t == threshold_499 else f'Standard ({t})',
        'N_calls': (max_posteriors >= t).sum(),
        'N_calls_pct': (max_posteriors >= t).sum() / len(max_posteriors) * 100,
        'Expected_error_rate_pct': (1 - max_posteriors[max_posteriors >= t]).mean() * 100
    }
    for t in [threshold_499, 0.90, 0.95, 0.99]
])
threshold_summary.to_csv(f'{output_dir}/threshold_summary.csv', index=False)

print(f"\n✅ Saved: celltype_error_breakdown.csv")
print(f"✅ Saved: threshold_summary.csv")

# -----------------------------
# FINAL SUMMARY
# -----------------------------
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)
print(f"\n*** Global 4.99% error threshold: {threshold_499:.4f} ***")
print(f"    Cells passing: {n_calls_499} ({n_calls_499/len(max_posteriors)*100:.1f}%)")
print(f"    Actual error rate: {actual_error_499:.2f}%")

print(f"\n*** Per cell type 4.99% thresholds: ***")
for ct in cell_types:
    t = celltype_499_thresholds.get(ct)
    if t is not None:
        n = (max_posteriors[max_classes == ct] >= t).sum()
        print(f"    {ct}: threshold={t:.4f}, n={n}")
    else:
        print(f"    {ct}: cannot reach 4.99% error (all posteriors too low or too few cells)")

print("\n✅ Output files:")
print(f"  {output_dir}/posterior_diagnostics.pdf")
print(f"  {output_dir}/celltype_error_breakdown.pdf")
print(f"  {output_dir}/threshold_summary.csv")
print(f"  {output_dir}/celltype_error_breakdown.csv")

print("\n" + "="*80)
print("DONE!")
print("="*80)

In [ ]:
import sys
print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Scanpy: {sc.__version__}")
print(f"Matplotlib: {mpl.__version__}")
print(f"Seaborn: {sns.__version__}")
print(f"SciPy: {scipy.__version__}")

In [ ]:
import bbknn
print(bbknn.__version__)

In [ ]:
adata_combined.obsm['X_pca']

In [ ]:
umap_coordinates = pd.DataFrame(
    adata_combined.obsm['X_umap'],
    index=adata_combined.obs_names,
    columns=['umap1', 'umap2']
)
umap_coordinates.to_csv('/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/integration/plots_paper/umapCoordinates_adata_combined_SI_20012026.h5ad')

In [ ]:
pca_columns = [f'pca{i}' for i in range(1, 51)]

# Create DataFrame with all 50 PCA components
pca_coordinates = pd.DataFrame(
    adata_combined.obsm['X_pca'][:, :50],  # Select first 50 components
    index=adata_combined.obs_names,
    columns=pca_columns
)
pca_coordinates.to_csv('/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/integration/plots_paper/pcaCoordinates_adata_combined_SI_20012026.h5ad')

In [ ]:
## Sorted vs RNA heatmap

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scanpy as sc

# Set the figure directory
sc.settings.figdir = '/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/integration/plots_paper/'

# Get pRNA data only
pRNA_data = adata_combined.obs[adata_combined.obs['assay'] == 'pRNA'][
    ['Sorted_population', 'leiden_annot_25']
].copy()

# Create contingency table (cross-tabulation)
contingency_table = pd.crosstab(
    pRNA_data['Sorted_population'], 
    pRNA_data['leiden_annot_25']
)

# Calculate percentages (normalize by row - each sorted population sums to 100%)
contingency_pct = contingency_table.div(contingency_table.sum(axis=1), axis=0) * 100

# Define ordering
leiden_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']
sorted_pop_order = ['Enterocytes', 'ISC-Goblet', 'Paneth', 'Tuft']

# Reorder the contingency tables
# Filter to only include existing categories
leiden_order_existing = [ct for ct in leiden_order if ct in contingency_table.columns]
sorted_pop_order_existing = [sp for sp in sorted_pop_order if sp in contingency_table.index]

contingency_table_ordered = contingency_table.loc[sorted_pop_order_existing, leiden_order_existing]
contingency_pct_ordered = contingency_pct.loc[sorted_pop_order_existing, leiden_order_existing]

# Create figure with two heatmaps
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Heatmap 1: Absolute counts
sns.heatmap(
    contingency_table_ordered,
    annot=True,
    fmt='g',
    cmap='RdBu_r',
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Number of Cells'},
    ax=axes[0]
)
axes[0].set_title('pRNA: Sorted Population vs Leiden Annotation (Cell Counts)', 
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Leiden Annotation (leiden_annot_25)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Sorted Population', fontsize=11, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Heatmap 2: Percentages
sns.heatmap(
    contingency_pct_ordered,
    annot=True,
    fmt='.1f',
    cmap='RdBu_r',
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Percentage (%)'},
    ax=axes[1]
)
axes[1].set_title('pRNA: Sorted Population vs Leiden Annotation (Percentage per Sorted Pop)', 
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Leiden Annotation (leiden_annot_25)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Sorted Population', fontsize=11, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'{sc.settings.figdir}pRNA_sorted_vs_leiden_heatmap.pdf', dpi=300, bbox_inches='tight')
plt.show()

print("="*100)
print("SORTED POPULATION vs LEIDEN ANNOTATION (pRNA only)")
print("="*100)
print("\nAbsolute Counts:")
print(contingency_table_ordered)
print("\nPercentages (by Sorted Population):")
print(contingency_pct_ordered.round(1))

# Print summary statistics
print("\n" + "="*100)
print("SUMMARY STATISTICS")
print("="*100)
print(f"\nTotal pRNA cells analyzed: {len(pRNA_data)}")
print(f"\nNumber of Sorted Populations: {pRNA_data['Sorted_population'].nunique()}")
print(f"Sorted Populations: {sorted(pRNA_data['Sorted_population'].unique())}")
print(f"\nNumber of Leiden Annotations: {pRNA_data['leiden_annot_25'].nunique()}")
print(f"Leiden Annotations: {sorted(pRNA_data['leiden_annot_25'].unique())}")

# Save tables to CSV
output_dir = sc.settings.figdir
contingency_table_ordered.to_csv(f'{output_dir}pRNA_sorted_vs_leiden_counts.csv')
contingency_pct_ordered.to_csv(f'{output_dir}pRNA_sorted_vs_leiden_percentages.csv')

print("\n" + "="*100)
print("Files saved:")
print(f"  - pRNA_sorted_vs_leiden_heatmap.pdf")
print(f"  - pRNA_sorted_vs_leiden_counts.csv")
print(f"  - pRNA_sorted_vs_leiden_percentages.csv")
print("="*100)

In [ ]:
### UMAP with most clear transitions

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
import os

# -----------------------------
# SETTINGS
# -----------------------------
sc.settings.figdir = '/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/gene_scoring/gmm_085/'

output_dir = sc.settings.figdir
os.makedirs(output_dir, exist_ok=True)

adata = adata_combined

cell_type_colors = {
    'Stem'      : '#9370DB',
    'TA'        : '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth'    : '#DC143C',
    'Goblet'    : '#2E8B57',
    'Tuft'      : '#FF1493',
    'EEC'       : '#4169E1',
    'ambiguous' : '#7f7f7f'
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# -----------------------------
# EXTRACT UMAP AND METADATA
# -----------------------------
umap_coords = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1'                          : umap_coords[:, 0],
    'UMAP2'                          : umap_coords[:, 1],
    'assay'                          : adata.obs['assay'].values,
    'batch_plate_well'               : adata.obs['batch_plate_well'].values,
    'leiden_annot_25'                : adata.obs['leiden_annot_25'].astype(str).values,
    'ttest_gmm_label_confident_085'  : adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'cell_index'                     : adata.obs.index.values
})

# -----------------------------
# STEP 1: FILTER CLEAR DCM CELLS
# -----------------------------
print("\n" + "="*60)
print("STEP 1: FILTERING DCM CELLS WITH CLEAR TRANSITIONS")
print("="*60)

dcm_mask = plot_df['assay'] == 'pDCM'
dcm_clear_mask = (
    dcm_mask &
    (plot_df['ttest_gmm_label_confident_085'].isin(label_order)) &
    (plot_df['ttest_gmm_label_confident_085'] == plot_df['leiden_annot_25'])
)

dcm_cells = plot_df[dcm_clear_mask].copy()
print(f"Total pDCM cells: {dcm_mask.sum()}")
print(f"DCM cells with CLEAR labels (GMM == leiden_annot_25): {dcm_clear_mask.sum()}")

# -----------------------------
# STEP 2: FILTER RNA CELLS
# -----------------------------
print("\n" + "="*60)
print("STEP 2: FILTERING RNA CELLS")
print("="*60)

rna_mask  = plot_df['assay'] == 'pRNA'
rna_cells = plot_df[rna_mask].copy()
print(f"Total pRNA cells: {rna_mask.sum()}")

# -----------------------------
# STEP 3: 1:1 MATCHING
# -----------------------------
print("\n" + "="*60)
print("STEP 3: FINDING CONNECTIONS USING BATCH_PLATE_WELL")
print("="*60)

common_bpw  = set(dcm_cells['batch_plate_well']) & set(rna_cells['batch_plate_well'])
dcm_single  = set(dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
                  ['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
rna_single  = set(rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]
                  ['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
both_single = dcm_single & rna_single
print(f"1:1 matched batch_plate_wells: {len(both_single)}")

connections = []
for bpw in both_single:
    dcm_row = dcm_cells[dcm_cells['batch_plate_well'] == bpw].iloc[0]
    rna_row = rna_cells[rna_cells['batch_plate_well'] == bpw].iloc[0]
    connections.append({
        'batch_plate_well': bpw,
        'dcm_umap1'       : dcm_row['UMAP1'],
        'dcm_umap2'       : dcm_row['UMAP2'],
        'rna_umap1'       : rna_row['UMAP1'],
        'rna_umap2'       : rna_row['UMAP2'],
        'dcm_cell_type'   : dcm_row['ttest_gmm_label_confident_085'],
        'rna_cell_type'   : rna_row['leiden_annot_25'],
        'transition'      : f"{dcm_row['ttest_gmm_label_confident_085']} -> {rna_row['leiden_annot_25']}"
    })

connections_df = pd.DataFrame(connections)
print(f"Total clear connections: {len(connections_df)}")

# -----------------------------
# STEP 3b: KEEP ONLY SWITCHING CELLS (DCM ≠ RNA)
# -----------------------------
connections_df = connections_df[
    connections_df['dcm_cell_type'] != connections_df['rna_cell_type']
].reset_index(drop=True)
print(f"Switching transitions (DCM ≠ RNA): {len(connections_df)}")

# -----------------------------
# STEP 4: PLOT
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("PLOTTING SWITCHING TRANSITIONS")
    print("="*60)

    fig, ax = plt.subplots(figsize=(12, 10))

    # Background
    ref_mask = (plot_df['assay'] == 'Ref') if 'Ref' in plot_df['assay'].values \
               else plot_df['assay'].isin(['pDCM', 'pRNA'])

    for ct in label_order:
        ref_ct = plot_df[ref_mask & (plot_df['leiden_annot_25'] == ct)]
        if len(ref_ct) > 0:
            ax.scatter(ref_ct['UMAP1'], ref_ct['UMAP2'],
                       s=8, c=cell_type_colors[ct], alpha=0.15,
                       marker='o', zorder=1, rasterized=True)

    ref_ambig = plot_df[ref_mask & (~plot_df['leiden_annot_25'].isin(label_order))]
    if len(ref_ambig) > 0:
        ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                   s=8, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)

        # Draw circles around each cluster based on background reference cells
    from scipy.stats import gaussian_kde
    import numpy as np
    
    for ct in label_order:
        ref_ct = plot_df[ref_mask & (plot_df['leiden_annot_25'] == ct)]
        if len(ref_ct) < 10:
            continue
    
        x = ref_ct['UMAP1'].values
        y = ref_ct['UMAP2'].values
    
        # KDE over a grid
        kde = gaussian_kde(np.vstack([x, y]), bw_method=0.3)
        x_grid = np.linspace(x.min() - 0.5, x.max() + 0.5, 200)
        y_grid = np.linspace(y.min() - 0.5, y.max() + 0.5, 200)
        xx, yy = np.meshgrid(x_grid, y_grid)
        zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    
        # Draw contour at a given density level
        ax.contour(xx, yy, zz,
                   levels=[zz.max() * 0.15],   # 15% of peak density — adjust to taste
                   colors=[cell_type_colors[ct]],
                   linewidths=2,
                   linestyles='--',
                   alpha=0.8,
                   zorder=2)
    
        # Label at the density peak
        peak_idx = np.unravel_index(zz.argmax(), zz.shape)
        ax.text(xx[peak_idx], yy[peak_idx], ct,
                ha='center', va='center', fontsize=10, fontweight='bold',
                color=cell_type_colors[ct], zorder=5)
        # Arrows
            # Arrows — colored by destination (RNA) cell type
        for _, conn in connections_df.iterrows():
            arrow = FancyArrowPatch(
                (conn['dcm_umap1'], conn['dcm_umap2']),
                (conn['rna_umap1'], conn['rna_umap2']),
                arrowstyle='-|>', mutation_scale=15,
                color=cell_type_colors.get(conn['rna_cell_type'], 'black'),
                alpha=0.7, linewidth=1.5, zorder=3
            )
            ax.add_patch(arrow)
    
        all_bpw = set(connections_df['batch_plate_well'])

    # DCM squares
    for ct in label_order:
        dcm_plot = plot_df[
            (plot_df['assay'] == 'pDCM') &
            (plot_df['batch_plate_well'].isin(all_bpw)) &
            (plot_df['ttest_gmm_label_confident_085'] == ct)
        ]
        if len(dcm_plot) > 0:
            ax.scatter(dcm_plot['UMAP1'], dcm_plot['UMAP2'],
                       s=80, c=cell_type_colors[ct], alpha=0.9,
                       edgecolors='black', linewidths=1, marker='s', zorder=4, rasterized=True)

    # RNA circles
    for ct in label_order:
        rna_plot = plot_df[
            (plot_df['assay'] == 'pRNA') &
            (plot_df['batch_plate_well'].isin(all_bpw)) &
            (plot_df['leiden_annot_25'] == ct)
        ]
        if len(rna_plot) > 0:
            ax.scatter(rna_plot['UMAP1'], rna_plot['UMAP2'],
                       s=80, c=cell_type_colors[ct], alpha=0.9,
                       edgecolors='black', linewidths=1, marker='o', zorder=4, rasterized=True)

       # Legend
    legend_elements = []
    for ct in label_order:
        legend_elements.append(
            mlines.Line2D([], [], color=cell_type_colors[ct], marker='s', linestyle='None',
                          markersize=10, markeredgecolor='black', label=f'DCM {ct}')
        )
    for ct in label_order:
        legend_elements.append(
            mlines.Line2D([], [], color=cell_type_colors[ct], marker='o', linestyle='None',
                          markersize=10, markeredgecolor='black', label=f'RNA {ct}')
        )
    # Arrow legend entries — one per destination cell type
    for ct in label_order:
        legend_elements.append(
            mlines.Line2D([], [], color=cell_type_colors[ct], linestyle='-', linewidth=2,
                          alpha=0.8, marker='>', markersize=8, label=f'→ {ct}')
        )
    
    ax.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1, 0.5),
              frameon=True, fontsize=9, ncol=2)
    ax.set_title(
        f'Switching Transitions: DCM → RNA  (n={len(connections_df)} cells)\n'
        f'Only cells where DCM cell type ≠ RNA cell type\n'
        f'Arrow color = destination (RNA) cell type',
        fontsize=14, fontweight='bold'
    )
    ax.set_aspect('equal')
    ax.axis('off')
    
    plt.tight_layout()
    save_path = os.path.join(output_dir, 'umap_switching_transitions.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"✅ Saved: {save_path}")

    connections_df.to_csv(os.path.join(output_dir, 'DCM_RNA_switching_connections.csv'), index=False)
    print(f"✅ Saved: DCM_RNA_switching_connections.csv")

else:
    print("No switching transitions found!")

# -----------------------------
# SUMMARY
# -----------------------------
print("\n" + "="*60)
print("SUMMARY BY SWITCHING TRANSITION")
print("="*60)

if len(connections_df) > 0:
    print(connections_df['transition'].value_counts())

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
from scipy.stats import gaussian_kde
import os

# -----------------------------
# SETTINGS
# -----------------------------
sc.settings.figdir = '/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/gene_scoring/gmm_085/'

output_dir = sc.settings.figdir
os.makedirs(output_dir, exist_ok=True)

adata = adata_combined

cell_type_colors = {
    'Stem'      : '#9370DB',
    'TA'        : '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth'    : '#DC143C',
    'Goblet'    : '#2E8B57',
    'Tuft'      : '#FF1493',
    'EEC'       : '#4169E1',
    'ambiguous' : '#7f7f7f'
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# -----------------------------
# EXTRACT UMAP AND METADATA
# -----------------------------
umap_coords = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1'                         : umap_coords[:, 0],
    'UMAP2'                         : umap_coords[:, 1],
    'assay'                         : adata.obs['assay'].values,
    'batch_plate_well'              : adata.obs['batch_plate_well'].values,
    'leiden_annot_25'               : adata.obs['leiden_annot_25'].astype(str).values,
    'ttest_gmm_label_confident_085' : adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'cell_index'                    : adata.obs.index.values
})

# -----------------------------
# STEP 1: ALL DCM CELLS — exclude ambiguous GMM labels
# -----------------------------
print("\n" + "="*60)
print("STEP 1: ALL DCM CELLS (excluding ambiguous GMM)")
print("="*60)

dcm_mask = (
    (plot_df['assay'] == 'pDCM') &
    (plot_df['leiden_annot_25'].isin(label_order)) &
    (plot_df['ttest_gmm_label_confident_085'] != 'ambiguous')   # <-- only change
)
dcm_cells = plot_df[dcm_mask].copy()
print(f"Total pDCM cells in label_order           : {(plot_df['assay'] == 'pDCM').sum()}")
print(f"DCM cells after removing ambiguous GMM    : {len(dcm_cells)}")
print(dcm_cells['leiden_annot_25'].value_counts())

# -----------------------------
# STEP 3: 1:1 MATCHING
# -----------------------------
print("\n" + "="*60)
print("STEP 3: FINDING CONNECTIONS USING BATCH_PLATE_WELL")
print("="*60)

common_bpw  = set(dcm_cells['batch_plate_well']) & set(rna_cells['batch_plate_well'])
dcm_single  = set(dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
                  ['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
rna_single  = set(rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]
                  ['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
both_single = dcm_single & rna_single
print(f"1:1 matched batch_plate_wells: {len(both_single)}")

connections = []
for bpw in both_single:
    dcm_row = dcm_cells[dcm_cells['batch_plate_well'] == bpw].iloc[0]
    rna_row = rna_cells[rna_cells['batch_plate_well'] == bpw].iloc[0]
    connections.append({
        'batch_plate_well': bpw,
        'dcm_umap1'       : dcm_row['UMAP1'],
        'dcm_umap2'       : dcm_row['UMAP2'],
        'rna_umap1'       : rna_row['UMAP1'],
        'rna_umap2'       : rna_row['UMAP2'],
        'dcm_cell_type'   : dcm_row['leiden_annot_25'],   # now using leiden for DCM too
        'rna_cell_type'   : rna_row['leiden_annot_25'],
        'transition'      : f"{dcm_row['leiden_annot_25']} -> {rna_row['leiden_annot_25']}"
    })

connections_df = pd.DataFrame(connections)
print(f"Total connections: {len(connections_df)}")

# -----------------------------
# STEP 3b: KEEP ONLY SWITCHING CELLS (DCM ≠ RNA)
# -----------------------------
connections_df = connections_df[
    connections_df['dcm_cell_type'] != connections_df['rna_cell_type']
].reset_index(drop=True)
print(f"Switching transitions (DCM ≠ RNA): {len(connections_df)}")

# -----------------------------
# STEP 3c: BUILD KDE PER CLUSTER & FILTER TO CONTOUR
# -----------------------------
print("\n" + "="*60)
print("STEP 3c: FILTERING TO CELLS WITHIN CLUSTER CONTOURS")
print("="*60)

ref_mask = (plot_df['assay'] == 'Ref') if 'Ref' in plot_df['assay'].values \
           else plot_df['assay'].isin(['pDCM', 'pRNA'])

bw_method  = 0.5
level_frac = 0.08
margin     = 0.0

def build_kde(plot_df, ref_mask, ct, bw_method=0.3):
    ref_ct = plot_df[ref_mask & (plot_df['leiden_annot_25'] == ct)]
    if len(ref_ct) < 10:
        return None, None
    x   = ref_ct['UMAP1'].values
    y   = ref_ct['UMAP2'].values
    kde = gaussian_kde(np.vstack([x, y]), bw_method=bw_method)
    return kde, (x, y)

def get_threshold(kde, x, y, level_frac, margin):
    densities = kde(np.vstack([x, y]))
    return densities.max() * level_frac - margin

def point_in_contour(kde, threshold, u1, u2):
    if kde is None:
        return False
    return kde(np.array([[u1], [u2]]))[0] >= threshold

# Pre-compute KDE + thresholds
kde_store = {}
for ct in label_order:
    kde, coords = build_kde(plot_df, ref_mask, ct, bw_method)
    if kde is not None:
        threshold = get_threshold(kde, coords[0], coords[1], level_frac, margin)
        kde_store[ct] = (kde, threshold)
    else:
        kde_store[ct] = (None, None)

# Filter connections
def is_valid(row):
    dcm_kde, dcm_thresh = kde_store.get(row['dcm_cell_type'], (None, None))
    rna_kde, rna_thresh = kde_store.get(row['rna_cell_type'],  (None, None))
    return (point_in_contour(dcm_kde, dcm_thresh, row['dcm_umap1'], row['dcm_umap2']) and
            point_in_contour(rna_kde, rna_thresh, row['rna_umap1'], row['rna_umap2']))

connections_df_raw = connections_df.copy()
connections_df     = connections_df[connections_df.apply(is_valid, axis=1)].reset_index(drop=True)

print(f"Connections before contour filter : {len(connections_df_raw)}")
print(f"Connections after contour filter  : {len(connections_df)}")

# -----------------------------
# STEP 4: PLOT
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("PLOTTING CONTOUR-FILTERED SWITCHING TRANSITIONS")
    print("="*60)

    fig, ax = plt.subplots(figsize=(12, 10))

    # --- Background scatter ---
    for ct in label_order:
        ref_ct = plot_df[ref_mask & (plot_df['leiden_annot_25'] == ct)]
        if len(ref_ct) > 0:
            ax.scatter(ref_ct['UMAP1'], ref_ct['UMAP2'],
                       s=8, c=cell_type_colors[ct], alpha=0.15,
                       marker='o', zorder=1, rasterized=True)

    ref_ambig = plot_df[ref_mask & (~plot_df['leiden_annot_25'].isin(label_order))]
    if len(ref_ambig) > 0:
        ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                   s=8, c='lightgray', alpha=0.03,
                   marker='o', zorder=1, rasterized=True)

    # --- KDE contours ---
    for ct in label_order:
        ref_ct = plot_df[ref_mask & (plot_df['leiden_annot_25'] == ct)]
        if len(ref_ct) < 10:
            continue
        x = ref_ct['UMAP1'].values
        y = ref_ct['UMAP2'].values
        kde, coords = build_kde(plot_df, ref_mask, ct, bw_method)
        x_grid = np.linspace(x.min() - 0.5, x.max() + 0.5, 200)
        y_grid = np.linspace(y.min() - 0.5, y.max() + 0.5, 200)
        xx, yy = np.meshgrid(x_grid, y_grid)
        zz     = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)

        ax.contour(xx, yy, zz,
                   levels=[zz.max() * level_frac],
                   colors=[cell_type_colors[ct]],
                   linewidths=2, linestyles='--', alpha=0.8, zorder=2)

        peak_idx = np.unravel_index(zz.argmax(), zz.shape)
        ax.text(xx[peak_idx], yy[peak_idx], ct,
                ha='center', va='center', fontsize=10, fontweight='bold',
                color=cell_type_colors[ct], zorder=5)

    # --- Arrows colored by destination (RNA) cell type ---
    for _, conn in connections_df.iterrows():
        arrow = FancyArrowPatch(
            (conn['dcm_umap1'], conn['dcm_umap2']),
            (conn['rna_umap1'], conn['rna_umap2']),
            arrowstyle='-|>', mutation_scale=15,
            color=cell_type_colors.get(conn['rna_cell_type'], 'black'),
            alpha=0.7, linewidth=1.5, zorder=3
        )
        ax.add_patch(arrow)

    all_bpw = set(connections_df['batch_plate_well'])

    # --- DCM squares ---
    for ct in label_order:
        dcm_plot = plot_df[
            (plot_df['assay'] == 'pDCM') &
            (plot_df['batch_plate_well'].isin(all_bpw)) &
            (plot_df['leiden_annot_25'] == ct)
        ]
        if len(dcm_plot) > 0:
            ax.scatter(dcm_plot['UMAP1'], dcm_plot['UMAP2'],
                       s=80, c=cell_type_colors[ct], alpha=0.9,
                       edgecolors='black', linewidths=1,
                       marker='s', zorder=4, rasterized=True)

    # --- RNA circles ---
    for ct in label_order:
        rna_plot = plot_df[
            (plot_df['assay'] == 'pRNA') &
            (plot_df['batch_plate_well'].isin(all_bpw)) &
            (plot_df['leiden_annot_25'] == ct)
        ]
        if len(rna_plot) > 0:
            ax.scatter(rna_plot['UMAP1'], rna_plot['UMAP2'],
                       s=80, c=cell_type_colors[ct], alpha=0.9,
                       edgecolors='black', linewidths=1,
                       marker='o', zorder=4, rasterized=True)

    # --- Legend ---
    legend_elements = []
    for ct in label_order:
        legend_elements.append(
            mlines.Line2D([], [], color=cell_type_colors[ct], marker='s', linestyle='None',
                          markersize=10, markeredgecolor='black', label=f'DCM {ct}')
        )
    for ct in label_order:
        legend_elements.append(
            mlines.Line2D([], [], color=cell_type_colors[ct], marker='o', linestyle='None',
                          markersize=10, markeredgecolor='black', label=f'RNA {ct}')
        )
    for ct in label_order:
        legend_elements.append(
            mlines.Line2D([], [], color=cell_type_colors[ct], linestyle='-', linewidth=2,
                          alpha=0.8, marker='>', markersize=8, label=f'→ {ct}')
        )

    ax.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1, 0.5),
              frameon=True, fontsize=9, ncol=2)

    ax.set_title(
        f'Switching Transitions: DCM → RNA  (n={len(connections_df)} cells)\n'
        f'All DCM cells (leiden-based) — contour-filtered, DCM ≠ RNA\n'
        f'Arrow color = destination (RNA) cell type',
        fontsize=14, fontweight='bold'
    )
    ax.set_aspect('equal')
    ax.axis('off')

    plt.tight_layout()
    save_path = os.path.join(output_dir, 'umap_switching_transitions_all_dcm_contour_filtered2.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"✅ Saved: {save_path}")

    connections_df.to_csv(
        os.path.join(output_dir, 'DCM_RNA_switching_connections_all_dcm_contour_filtered2.csv'),
        index=False
    )
    print(f"✅ Saved: DCM_RNA_switching_connections_all_dcm_contour_filtered.csv")

else:
    print("No switching transitions found after contour filtering!")

# -----------------------------
# SUMMARY
# -----------------------------
print("\n" + "="*60)
print("SUMMARY BY SWITCHING TRANSITION")
print("="*60)

if len(connections_df) > 0:
    print(connections_df['transition'].value_counts())

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
connections_df = pd.read_csv(os.path.join(output_dir, 'DCM_RNA_switching_connections_all_dcm_contour_filtered.csv'))

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import gaussian_kde
import os

# -----------------------------
# SETTINGS
# -----------------------------
sc.settings.figdir = '/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/gene_scoring/gmm_085/'

output_dir      = sc.settings.figdir
output_dir_umap = os.path.join(output_dir, 'umap_per_transition')
output_dir_heat = os.path.join(output_dir, 'heatmap_per_transition')
os.makedirs(output_dir_umap, exist_ok=True)
os.makedirs(output_dir_heat, exist_ok=True)

adata = adata_combined

cell_type_colors = {
    'Stem'      : '#9370DB',
    'TA'        : '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth'    : '#DC143C',
    'Goblet'    : '#2E8B57',
    'Tuft'      : '#FF1493',
    'EEC'       : '#4169E1',
    'ambiguous' : '#7f7f7f'
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# -----------------------------
# EXTRACT UMAP AND METADATA
# -----------------------------
umap_coords = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1'                         : umap_coords[:, 0],
    'UMAP2'                         : umap_coords[:, 1],
    'assay'                         : adata.obs['assay'].values,
    'batch_plate_well'              : adata.obs['batch_plate_well'].values,
    'leiden_annot_25'               : adata.obs['leiden_annot_25'].astype(str).values,
    'ttest_gmm_label_confident_085' : adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'cell_index'                    : adata.obs.index.values,
    'is_multipotent'                : adata.obs['is_multipotent'].astype(str).values
})

plot_df['leiden_annot_25']               = plot_df['leiden_annot_25'].str.split('_').str[-1]
plot_df['ttest_gmm_label_confident_085'] = plot_df['ttest_gmm_label_confident_085'].str.split('_').str[-1]

ttest_cols = [col for col in adata.obs.columns
              if 'ttest_' in col and '_zscore' in col and 'max' not in col.lower()]
for col in ttest_cols:
    plot_df[col] = adata.obs[col].values

print(f"Found {len(ttest_cols)} ttest z-score columns: {ttest_cols}")

# -----------------------------
# STEP 1: DCM CELLS
# -----------------------------
print("\n" + "="*60)
print("STEP 1: ALL DCM CELLS (GMM label, excluding ambiguous)")
print("="*60)

dcm_mask = (
    (plot_df['assay'] == 'pDCM') &
    (plot_df['ttest_gmm_label_confident_085'].isin(label_order))
)
dcm_cells = plot_df[dcm_mask].copy()
print(f"Total pDCM cells                  : {(plot_df['assay'] == 'pDCM').sum()}")
print(f"DCM cells with confident GMM label: {len(dcm_cells)}")
print(dcm_cells['ttest_gmm_label_confident_085'].value_counts())

# -----------------------------
# STEP 2: RNA CELLS
# -----------------------------
print("\n" + "="*60)
print("STEP 2: ALL RNA CELLS (leiden label)")
print("="*60)

rna_mask = (
    (plot_df['assay'] == 'pRNA') &
    (plot_df['leiden_annot_25'].isin(label_order))
)
rna_cells = plot_df[rna_mask].copy()
print(f"Total pRNA cells in label_order: {len(rna_cells)}")
print(rna_cells['leiden_annot_25'].value_counts())

# 🔥 Enforce biological trajectory order for RNA
rna_cells['leiden_annot_25'] = pd.Categorical(
    rna_cells['leiden_annot_25'],
    categories=label_order,
    ordered=True
)
# -----------------------------
# STEP 3: 1:1 MATCHING
# -----------------------------
print("\n" + "="*60)
print("STEP 3: FINDING CONNECTIONS USING BATCH_PLATE_WELL")
print("="*60)

common_bpw  = set(dcm_cells['batch_plate_well']) & set(rna_cells['batch_plate_well'])
dcm_single  = set(dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
                  ['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
rna_single  = set(rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]
                  ['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
both_single = dcm_single & rna_single
print(f"1:1 matched batch_plate_wells: {len(both_single)}")

connections = []
for bpw in both_single:
    dcm_row = dcm_cells[dcm_cells['batch_plate_well'] == bpw].iloc[0]
    rna_row = rna_cells[rna_cells['batch_plate_well'] == bpw].iloc[0]
    conn = {
        'batch_plate_well': bpw,
        'dcm_umap1'       : dcm_row['UMAP1'],
        'dcm_umap2'       : dcm_row['UMAP2'],
        'rna_umap1'       : rna_row['UMAP1'],
        'rna_umap2'       : rna_row['UMAP2'],
        'dcm_cell_type'   : dcm_row['ttest_gmm_label_confident_085'],
        'rna_cell_type'   : rna_row['leiden_annot_25'],
        'transition'          : f"{dcm_row['ttest_gmm_label_confident_085']} -> {rna_row['leiden_annot_25']}",
        'dcm_is_multipotent'  : str(dcm_row.get('is_multipotent', 'False')).lower() in ('true', '1', 'yes'),
        'rna_is_multipotent'  : str(rna_row.get('is_multipotent', 'False')).lower() in ('true', '1', 'yes'),
    }
    for col in ttest_cols:
        ct = col.replace('ttest_', '').replace('_zscore', '')
        conn[f'dcm_{ct}_zscore'] = dcm_row[col]
        conn[f'rna_{ct}_zscore'] = rna_row[col]
    connections.append(conn)

connections_df = pd.DataFrame(connections)
# 🔥 Enforce trajectory order in connections
connections_df['rna_cell_type'] = pd.Categorical(
    connections_df['rna_cell_type'],
    categories=label_order,
    ordered=True
)

# 🔥 Keep a FULL version (including self-transitions)
connections_df_all = connections_df.copy()

# Original behaviour (switching only)
connections_df = connections_df[
    connections_df['dcm_cell_type'] != connections_df['rna_cell_type']
].reset_index(drop=True)

print(f"Switching transitions (DCM ≠ RNA): {len(connections_df)}")
print(f"All transitions (including self): {len(connections_df_all)}")

# -----------------------------
# STEP 3c: KDE CONTOUR FILTER
# -----------------------------
print("\n" + "="*60)
print("STEP 3c: FILTERING TO CELLS WITHIN CLUSTER CONTOURS")
print("="*60)

bw_method  = 0.5
level_frac = 0.08
margin     = 0.0

def build_kde(plot_df, ref_mask, ct, bw_method=0.3):
    ref_ct = plot_df[ref_mask & (plot_df['leiden_annot_25'] == ct)]
    if len(ref_ct) < 10:
        return None, None
    x   = ref_ct['UMAP1'].values
    y   = ref_ct['UMAP2'].values
    kde = gaussian_kde(np.vstack([x, y]), bw_method=bw_method)
    return kde, (x, y)

def get_threshold(kde, x, y, level_frac, margin):
    return kde(np.vstack([x, y])).max() * level_frac - margin

def point_in_contour(kde, threshold, u1, u2):
    if kde is None:
        return False
    return kde(np.array([[u1], [u2]]))[0] >= threshold

ref_mask = (plot_df['assay'] == 'Ref') if 'Ref' in plot_df['assay'].values \
           else plot_df['assay'].isin(['pDCM', 'pRNA'])

kde_store = {}
for ct in label_order:
    kde, coords = build_kde(plot_df, ref_mask, ct, bw_method)
    if kde is not None:
        threshold = get_threshold(kde, coords[0], coords[1], level_frac, margin)
        kde_store[ct] = (kde, threshold)
    else:
        kde_store[ct] = (None, None)

def is_valid(row):
    dcm_kde, dcm_thresh = kde_store.get(row['dcm_cell_type'], (None, None))
    rna_kde, rna_thresh = kde_store.get(row['rna_cell_type'],  (None, None))
    return (point_in_contour(dcm_kde, dcm_thresh, row['dcm_umap1'], row['dcm_umap2']) and
            point_in_contour(rna_kde, rna_thresh, row['rna_umap1'], row['rna_umap2']))

 # RAW (before contour filter)
connections_df_raw = connections_df.copy()
connections_df_all_raw = connections_df_all.copy()

# FILTERED
connections_df     = connections_df[connections_df.apply(is_valid, axis=1)].reset_index(drop=True)
connections_df_all = connections_df_all[connections_df_all.apply(is_valid, axis=1)].reset_index(drop=True)
print(f"\nConnections before contour filter : {len(connections_df_raw)}")
print(f"Connections after contour filter  : {len(connections_df)}")

# Apply same ordering to raw table
connections_df_raw['rna_cell_type'] = pd.Categorical(
    connections_df_raw['rna_cell_type'],
    categories=label_order,
    ordered=True
)   
# =====================================================================
# HEATMAP HELPER WITH ANNOTATION BARS
# =====================================================================
def hex_to_rgb(h):
    h = h.lstrip('#')
    return [int(h[i:i+2], 16) / 255.0 for i in (0, 2, 4)]

def plot_heatmap_transition(subset_df, label_order, ttest_cols,
                             cell_type_colors, title_str, save_path):
    """
    Heatmap with coloured annotation bars above each panel.

    Layout (4 rows × 2 cols):
      row 0 — DCM annotation bar  (colour = dcm_cell_type)
      row 1 — DCM z-score heatmap
      row 2 — RNA annotation bar  (colour = rna_cell_type)
      row 3 — RNA z-score heatmap
      col 1 — colourbars for rows 1 and 3
    """
    if len(subset_df) == 0:
        print(f"  Skipping heatmap — no transitions")
        return

    vcenter   = 1.0
    vmin      = vcenter - 3
    vmax      = vcenter + 3
    norm      = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
    cmap      = plt.cm.RdBu_r
    n_cells   = len(subset_df)
    fig_width = max(14, n_cells * 0.18)

    # Sort by DCM cell type then self-zscore descending
    df = subset_df.copy()
    df['dcm_sort_order'] = df['dcm_cell_type'].map(
        {ct: i for i, ct in enumerate(label_order)}
    ).fillna(99)
    df['dcm_self_zscore'] = df.apply(
        lambda r: r.get(f'dcm_{r["dcm_cell_type"]}_zscore', 0), axis=1
    )
    # 🔥 Add RNA trajectory ordering
# 🔥 Convert to numeric safely before fillna
    df['rna_sort_order'] = df['rna_cell_type'].map(
        {ct: i for i, ct in enumerate(label_order)}
    )
    
    df['rna_sort_order'] = df['rna_sort_order'].astype(float).fillna(99)
    
    df = df.sort_values(
        ['dcm_sort_order', 'rna_sort_order', 'dcm_self_zscore'],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    dcm_matrix = np.array(
        [[row.get(f'dcm_{ct}_zscore', np.nan) for ct in label_order]
         for _, row in df.iterrows()]
    ).T
    rna_matrix = np.array(
        [[row.get(f'rna_{ct}_zscore', np.nan) for ct in label_order]
         for _, row in df.iterrows()]
    ).T

    # Build RGB annotation arrays
    def make_ann_rgb(col):
        colors = [cell_type_colors.get(v, '#cccccc') for v in df[col]]
        return np.array([hex_to_rgb(c) for c in colors])[np.newaxis, :, :]

    dcm_ann_rgb = make_ann_rgb('dcm_cell_type')
    rna_ann_rgb = make_ann_rgb('rna_cell_type')

    # Group boundary positions (based on DCM sort)
    changes = df['dcm_cell_type'].ne(
        df['dcm_cell_type'].shift()
    ).to_numpy().nonzero()[0]

    # ── Figure layout ─────────────────────────────────────────────
    fig = plt.figure(figsize=(fig_width, 14))
    gs  = fig.add_gridspec(
        4, 2,
        height_ratios=[0.04, 1, 0.04, 1],
        width_ratios=[1, 0.015],
        hspace=0.06,
        wspace=0.02,
    )
    ax_ann_dcm = fig.add_subplot(gs[0, 0])
    ax_dcm     = fig.add_subplot(gs[1, 0])
    ax_ann_rna = fig.add_subplot(gs[2, 0])
    ax_rna     = fig.add_subplot(gs[3, 0])
    ax_cb_dcm  = fig.add_subplot(gs[1, 1])
    ax_cb_rna  = fig.add_subplot(gs[3, 1])

    # ── Annotation bar helper ─────────────────────────────────────
    def draw_ann_bar(ax, ann_rgb, label_col, bar_title):
        ax.imshow(ann_rgb, aspect='auto', interpolation='nearest')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_ylabel(bar_title, fontsize=7, rotation=0, labelpad=32, va='center')
        for x in changes:
            ax.axvline(x - 0.5, color='white', lw=1.5)
        seen, patches = [], []
        for ct in df[label_col]:
            if ct not in seen:
                seen.append(ct)
                patches.append(mpatches.Patch(
                    facecolor=cell_type_colors.get(ct, '#cccccc'),
                    edgecolor='white', linewidth=0.5, label=ct,
                ))
        ax.legend(handles=patches, loc='upper left', bbox_to_anchor=(1.02, 1.0),
                  fontsize=7, frameon=True, title=bar_title, title_fontsize=7)

    draw_ann_bar(ax_ann_dcm, dcm_ann_rgb, 'dcm_cell_type', 'DCM type')
    draw_ann_bar(ax_ann_rna, rna_ann_rgb, 'rna_cell_type', 'RNA type')

    # ── Heatmap panels ────────────────────────────────────────────
    for ax, ax_cb, matrix, assay_label, vcenter_val, multipotent_col in [
        (ax_dcm, ax_cb_dcm, dcm_matrix, 'pDCM (GMM label)',   1.0, 'dcm_is_multipotent'),
        (ax_rna, ax_cb_rna, rna_matrix, 'pRNA (leiden label)', 0.0, 'rna_is_multipotent'),
    ]:
        # Re-normalise per panel so vcenter is correct
        _norm = TwoSlopeNorm(vmin=vcenter_val - 3,
                             vcenter=vcenter_val,
                             vmax=vcenter_val + 3)
        im = ax.imshow(matrix, aspect='auto', cmap=cmap, norm=_norm)
        ax.set_yticks(range(len(label_order)))
        ax.set_yticklabels(label_order, fontsize=10)
        ax.set_xticks([])
        ax.set_ylabel('Module', fontsize=11, fontweight='bold')
        ax.set_title(assay_label, fontsize=12, fontweight='bold')

        # Dividers
        for x in changes:
            ax.axvline(x - 0.5, color='white', lw=1.5, alpha=0.8)

        # Group labels below heatmap
        prev = 0
        for x in list(changes) + [n_cells]:
            mid = (prev + x) / 2
            ct  = df.iloc[prev]['dcm_cell_type']
            ax.text(mid, -0.8, ct, ha='center', va='top', fontsize=7,
                    fontweight='bold', color=cell_type_colors.get(ct, 'black'))
            prev = x

        # ── Multipotent markers: '*' above each hybrid cell ──────────
        # multipotent_col is set per panel in the loop below
        for cell_idx, row in df.iterrows():
            if row.get(multipotent_col, False):
                ax.text(cell_idx, -0.3, '*',
                        ha='center', va='bottom', fontsize=8,
                        fontweight='bold', color='#333333')

        cb = plt.colorbar(im, cax=ax_cb)
        cb.set_label('Z-Score', fontsize=9)
        cb.ax.axhline(y=vcenter_val, color='black', lw=1, ls='--')

    fig.suptitle(title_str, fontsize=13, fontweight='bold', y=1.01)
    fig.text(0.01, 0.01, '* = multipotent cell (is_multipotent=True)',
             fontsize=7, color='#555555', va='bottom')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {save_path}")


# =====================================================================
# UMAP HELPER  (unchanged from doc 13)
# =====================================================================
def plot_umap_transition(subset_df, plot_df, ref_mask,
                          cell_type_colors, label_order, bw_method, level_frac,
                          highlight_source_ct, highlight_dest_ct,
                          title_str, save_path):
    if len(subset_df) == 0:
        print(f"  Skipping — no transitions found")
        return

    fig, ax = plt.subplots(figsize=(12, 10))

    for ct in label_order:
        ref_ct = plot_df[ref_mask & (plot_df['leiden_annot_25'] == ct)]
        if len(ref_ct) > 0:
            ax.scatter(ref_ct['UMAP1'], ref_ct['UMAP2'],
                       s=8, c=cell_type_colors[ct], alpha=0.15,
                       marker='o', zorder=1, rasterized=True)

    ref_ambig = plot_df[ref_mask & (~plot_df['leiden_annot_25'].isin(label_order))]
    if len(ref_ambig) > 0:
        ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                   s=8, c='lightgray', alpha=0.03,
                   marker='o', zorder=1, rasterized=True)

    for ct in label_order:
        ref_ct = plot_df[ref_mask & (plot_df['leiden_annot_25'] == ct)]
        if len(ref_ct) < 10:
            continue
        x = ref_ct['UMAP1'].values
        y = ref_ct['UMAP2'].values
        kde, _ = build_kde(plot_df, ref_mask, ct, bw_method)
        x_grid = np.linspace(x.min() - 0.5, x.max() + 0.5, 200)
        y_grid = np.linspace(y.min() - 0.5, y.max() + 0.5, 200)
        xx, yy = np.meshgrid(x_grid, y_grid)
        zz     = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        ax.contour(xx, yy, zz,
                   levels=[zz.max() * level_frac],
                   colors=[cell_type_colors[ct]],
                   linewidths=2, linestyles='--', alpha=0.8, zorder=2)
        peak_idx = np.unravel_index(zz.argmax(), zz.shape)
        ax.text(xx[peak_idx], yy[peak_idx], ct,
                ha='center', va='center', fontsize=10, fontweight='bold',
                color=cell_type_colors[ct], zorder=5)

    for _, conn in subset_df.iterrows():
        arrow = FancyArrowPatch(
            (conn['dcm_umap1'], conn['dcm_umap2']),
            (conn['rna_umap1'], conn['rna_umap2']),
            arrowstyle='-|>', mutation_scale=15,
            color=cell_type_colors.get(conn['rna_cell_type'], 'black'),
            alpha=0.7, linewidth=1.5, zorder=3
        )
        ax.add_patch(arrow)

    source_cts = subset_df['dcm_cell_type'].unique()
    dest_cts   = subset_df['rna_cell_type'].unique()

    for _, conn in subset_df.iterrows():
        ct = conn['dcm_cell_type']
        size = 140 if ct == highlight_source_ct else 80
        ax.scatter(conn['dcm_umap1'], conn['dcm_umap2'],
                   s=size, c=cell_type_colors.get(ct, 'gray'), alpha=1.0,
                   edgecolors='black', linewidths=1.5,
                   marker='s', zorder=4)

    for _, conn in subset_df.iterrows():
        ct = conn['rna_cell_type']
        size = 140 if ct == highlight_dest_ct else 80
        ax.scatter(conn['rna_umap1'], conn['rna_umap2'],
                   s=size, c=cell_type_colors.get(ct, 'gray'), alpha=1.0,
                   edgecolors='black', linewidths=1.5,
                   marker='o', zorder=4)

    legend_elements = []
    for ct in source_cts:
        legend_elements.append(
            mlines.Line2D([], [], color=cell_type_colors[ct], marker='s',
                          linestyle='None', markersize=10, markeredgecolor='black',
                          label=f'DCM {ct} (GMM)'))
    for ct in dest_cts:
        legend_elements.append(
            mlines.Line2D([], [], color=cell_type_colors[ct], marker='o',
                          linestyle='None', markersize=10, markeredgecolor='black',
                          label=f'RNA {ct} (leiden)'))
    for ct in dest_cts:
        legend_elements.append(
            mlines.Line2D([], [], color=cell_type_colors[ct], linestyle='-',
                          linewidth=2, alpha=0.8, marker='>', markersize=8,
                          label=f'→ {ct}'))

    ax.legend(handles=legend_elements, loc='center left',
              bbox_to_anchor=(1, 0.5), frameon=True, fontsize=9)
    ax.set_title(title_str, fontsize=13, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {save_path}")


# =====================================================================
# STEP 4a: PER SOURCE DCM CELL TYPE
# Heatmaps produced BOTH before and after contour filter
# =====================================================================
print("\n" + "="*60)
print("STEP 4a: PLOTS PER SOURCE DCM CELL TYPE (GMM label)")
print("="*60)

for source_ct in label_order:
    # ── post-filter (default) ──────────────────────────────────────
    subset     = connections_df[connections_df['dcm_cell_type'] == source_ct].copy()
    # ── pre-filter ────────────────────────────────────────────────
    subset_raw = connections_df_raw[connections_df_raw['dcm_cell_type'] == source_ct].copy()

    print(f"\n{source_ct} → *  :  raw={len(subset_raw)}  filtered={len(subset)}")
    if len(subset_raw) == 0:
        continue

    trans_str = '  |  '.join(
        [f'{t}: {c}' for t, c in subset['transition'].value_counts().items()]
    ) if len(subset) > 0 else 'no cells after filter'

    # UMAP (post-filter only — coordinates meaningful only in-contour)
    if len(subset) > 0:
        plot_umap_transition(
            subset_df           = subset,
            plot_df             = plot_df,
            ref_mask            = ref_mask,
            cell_type_colors    = cell_type_colors,
            label_order         = label_order,
            bw_method           = bw_method,
            level_frac          = level_frac,
            highlight_source_ct = source_ct,
            highlight_dest_ct   = None,
            title_str           = (f'Transitions FROM {source_ct} [GMM]  (n={len(subset)})\n'
                                   f'{trans_str}\nArrow color = destination (RNA leiden)'),
            save_path           = os.path.join(output_dir_umap, f'umap_from_{source_ct}.pdf')
        )

    # Heatmap — PRE-filter
    plot_heatmap_transition(
        subset_df        = subset_raw,
        label_order      = label_order,
        ttest_cols       = ttest_cols,
        cell_type_colors = cell_type_colors,
        title_str        = (f'Z-score Heatmap (PRE-filter) — FROM {source_ct} [GMM]  '
                            f'(n={len(subset_raw)})'),
        save_path        = os.path.join(output_dir_heat,
                                        f'heatmap_from_{source_ct}_prefilter.pdf')
    )

    # Heatmap — POST-filter
    plot_heatmap_transition(
        subset_df        = subset,
        label_order      = label_order,
        ttest_cols       = ttest_cols,
        cell_type_colors = cell_type_colors,
        title_str        = (f'Z-score Heatmap (POST-filter) — FROM {source_ct} [GMM]  '
                            f'(n={len(subset)})'),
        save_path        = os.path.join(output_dir_heat,
                                        f'heatmap_from_{source_ct}_postfilter.pdf')
    )

# =====================================================================
# STEP 4b: PER DESTINATION RNA CELL TYPE
# =====================================================================
print("\n" + "="*60)
print("STEP 4b: PLOTS PER DESTINATION RNA CELL TYPE (leiden label)")
print("="*60)

for dest_ct in label_order:
    subset     = connections_df[connections_df['rna_cell_type'] == dest_ct].copy()
    subset_raw = connections_df_raw[connections_df_raw['rna_cell_type'] == dest_ct].copy()

    print(f"\n* → {dest_ct}  :  raw={len(subset_raw)}  filtered={len(subset)}")
    if len(subset_raw) == 0:
        continue

    trans_str = '  |  '.join(
        [f'{t}: {c}' for t, c in subset['transition'].value_counts().items()]
    ) if len(subset) > 0 else 'no cells after filter'

    if len(subset) > 0:
        plot_umap_transition(
            subset_df           = subset,
            plot_df             = plot_df,
            ref_mask            = ref_mask,
            cell_type_colors    = cell_type_colors,
            label_order         = label_order,
            bw_method           = bw_method,
            level_frac          = level_frac,
            highlight_source_ct = None,
            highlight_dest_ct   = dest_ct,
            title_str           = (f'Transitions TO {dest_ct} [leiden]  (n={len(subset)})\n'
                                   f'{trans_str}\nArrow color = destination (RNA leiden)'),
            save_path           = os.path.join(output_dir_umap, f'umap_to_{dest_ct}.pdf')
        )

    # Heatmap — PRE-filter
    plot_heatmap_transition(
        subset_df        = subset_raw,
        label_order      = label_order,
        ttest_cols       = ttest_cols,
        cell_type_colors = cell_type_colors,
        title_str        = (f'Z-score Heatmap (PRE-filter) — TO {dest_ct} [leiden]  '
                            f'(n={len(subset_raw)})'),
        save_path        = os.path.join(output_dir_heat,
                                        f'heatmap_to_{dest_ct}_prefilter.pdf')
    )

    # Heatmap — POST-filter
    plot_heatmap_transition(
        subset_df        = subset,
        label_order      = label_order,
        ttest_cols       = ttest_cols,
        cell_type_colors = cell_type_colors,
        title_str        = (f'Z-score Heatmap (POST-filter) — TO {dest_ct} [leiden]  '
                            f'(n={len(subset)})'),
        save_path        = os.path.join(output_dir_heat,
                                        f'heatmap_to_{dest_ct}_postfilter.pdf')
    )

# =====================================================================
# STEP 4c: ALL TRANSITIONS COMBINED
# =====================================================================
print("\n" + "="*60)
print("STEP 4c: ALL TRANSITIONS COMBINED")
print("="*60)

trans_str = '  |  '.join(
    [f'{t}: {c}' for t, c in connections_df['transition'].value_counts().items()]
)

plot_umap_transition(
    subset_df           = connections_df,
    plot_df             = plot_df,
    ref_mask            = ref_mask,
    cell_type_colors    = cell_type_colors,
    label_order         = label_order,
    bw_method           = bw_method,
    level_frac          = level_frac,
    highlight_source_ct = None,
    highlight_dest_ct   = None,
    title_str           = (f'All Switching Transitions: DCM [GMM] → RNA [leiden]  '
                           f'(n={len(connections_df)})\nArrow color = destination (RNA leiden)'),
    save_path           = os.path.join(output_dir_umap, 'umap_all_transitions.pdf')
)

# Heatmap — PRE-filter (all)
plot_heatmap_transition(
    subset_df        = connections_df_raw,
    label_order      = label_order,
    ttest_cols       = ttest_cols,
    cell_type_colors = cell_type_colors,
    title_str        = (f'Z-score Heatmap (PRE-filter) — All Switching Transitions  '
                        f'(n={len(connections_df_raw)})\n'
                        f'DCM [GMM] → RNA [leiden]  |  grouped by DCM cell type'),
    save_path        = os.path.join(output_dir_heat,
                                    'heatmap_all_transitions_prefilter.pdf')
)

# Heatmap — POST-filter (all)
plot_heatmap_transition(
    subset_df        = connections_df,
    label_order      = label_order,
    ttest_cols       = ttest_cols,
    cell_type_colors = cell_type_colors,
    title_str        = (f'Z-score Heatmap (POST-filter) — All Switching Transitions  '
                        f'(n={len(connections_df)})\n'
                        f'DCM [GMM] → RNA [leiden]  |  grouped by DCM cell type'),
    save_path        = os.path.join(output_dir_heat,
                                    'heatmap_all_transitions_postfilter.pdf')
)
# =====================================================================
# STEP 4d: ALL TRANSITIONS INCLUDING SELF (WITH OWN TRANSITION)
# =====================================================================
print("\n" + "="*60)
print("STEP 4d: ALL TRANSITIONS INCLUDING SELF (WITH OWN TRANSITION)")
print("="*60)

trans_str_all = '  |  '.join(
    [f'{t}: {c}' for t, c in connections_df_all['transition'].value_counts().items()]
)

# ── UMAP ─────────────────────────────────────────────────────────────
plot_umap_transition(
    subset_df           = connections_df_all,
    plot_df             = plot_df,
    ref_mask            = ref_mask,
    cell_type_colors    = cell_type_colors,
    label_order         = label_order,
    bw_method           = bw_method,
    level_frac          = level_frac,
    highlight_source_ct = None,
    highlight_dest_ct   = None,
    title_str           = (f'All Transitions INCLUDING SELF (n={len(connections_df_all)})\n'
                           f'DCM [GMM] → RNA [leiden]'),
    save_path           = os.path.join(output_dir_umap, 'umap_all_transitions_with_own_transition.pdf')
)

# ── Heatmap PRE-filter ───────────────────────────────────────────────
plot_heatmap_transition(
    subset_df        = connections_df_all_raw,
    label_order      = label_order,
    ttest_cols       = ttest_cols,
    cell_type_colors = cell_type_colors,
    title_str        = (f'Heatmap (PRE-filter) WITH OWN TRANSITIONS  '
                        f'(n={len(connections_df_all_raw)})'),
    save_path        = os.path.join(output_dir_heat,
                                    'heatmap_all_transitions_with_own_transition_prefilter.pdf')
)

# ── Heatmap POST-filter ──────────────────────────────────────────────
plot_heatmap_transition(
    subset_df        = connections_df_all,
    label_order      = label_order,
    ttest_cols       = ttest_cols,
    cell_type_colors = cell_type_colors,
    title_str        = (f'Heatmap (POST-filter) WITH OWN TRANSITIONS  '
                        f'(n={len(connections_df_all)})'),
    save_path        = os.path.join(output_dir_heat,
                                    'heatmap_all_transitions_with_own_transition_postfilter.pdf')
)
# =====================================================================
# CUSTOM: Transitions TO Stem or TA combined
# =====================================================================
print("\n" + "="*60)
print("CUSTOM: Transitions TO Stem or TA")
print("="*60)

subset_stem_ta = connections_df[
    (connections_df['rna_cell_type'].isin(['Stem', 'TA'])) &
    ~((connections_df['dcm_cell_type'] == 'Stem') & (connections_df['rna_cell_type'] == 'TA'))
].copy()
print(f"Transitions to Stem or TA: {len(subset_stem_ta)}")

if len(subset_stem_ta) > 0:
    trans_str = '  |  '.join(
        [f'{t}: {c}' for t, c in subset_stem_ta['transition'].value_counts().items()]
    )

    plot_umap_transition(
        subset_df           = subset_stem_ta,
        plot_df             = plot_df,
        ref_mask            = ref_mask,
        cell_type_colors    = cell_type_colors,
        label_order         = label_order,
        bw_method           = bw_method,
        level_frac          = level_frac,
        highlight_source_ct = None,
        highlight_dest_ct   = None,
        title_str           = (f'Transitions TO Stem / TA [leiden]  (n={len(subset_stem_ta)})\n'
                               f'{trans_str}\nArrow color = destination (RNA leiden)'),
        save_path           = os.path.join(output_dir_umap, 'umap_to_Stem_TA.pdf')
    )
 # Heatmap — PRE-filter
    subset_stem_ta_raw = connections_df_raw[
        (connections_df_raw['rna_cell_type'].isin(['Stem', 'TA'])) &
        ~((connections_df_raw['dcm_cell_type'] == 'Stem') & (connections_df_raw['rna_cell_type'] == 'TA'))
    ].copy()

    plot_heatmap_transition(
        subset_df        = subset_stem_ta_raw,
        label_order      = label_order,
        ttest_cols       = ttest_cols,
        cell_type_colors = cell_type_colors,
        title_str        = (f'Z-score Heatmap (PRE-filter) — TO Stem / TA  '
                            f'(n={len(subset_stem_ta_raw)})'),
        save_path        = os.path.join(output_dir_heat,
                                        'heatmap_to_Stem_TA_prefilter.pdf')
    )

    # Heatmap — POST-filter
    plot_heatmap_transition(
        subset_df        = subset_stem_ta,
        label_order      = label_order,
        ttest_cols       = ttest_cols,
        cell_type_colors = cell_type_colors,
        title_str        = (f'Z-score Heatmap (POST-filter) — TO Stem / TA  '
                            f'(n={len(subset_stem_ta)})'),
        save_path        = os.path.join(output_dir_heat,
                                        'heatmap_to_Stem_TA_postfilter.pdf')
    )
# =====================================================================
# SAVE FULL TABLE & SUMMARY
# =====================================================================
connections_df.to_csv(
    os.path.join(output_dir, 'DCM_RNA_switching_connections_gmm_leiden.csv'),
    index=False
)
connections_df_raw.to_csv(
    os.path.join(output_dir, 'DCM_RNA_switching_connections_gmm_leiden_raw.csv'),
    index=False
)
print(f"\n✅ Saved connection tables")

print("\n" + "="*60)
print("SUMMARY BY TRANSITION (post-filter)")
print("="*60)
print(connections_df['transition'].value_counts())

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
cellrank_prob = pd.read_csv("/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/CellRank/Nature/Combi_kernel/probabilities_tostemta.csv")

In [ ]:
subset_stem_ta['cellid'] = subset_stem_ta['rna_cell_type'].astype(str) + '_' + subset_stem_ta['batch_plate_well'].astype(str)
print('\n'.join(subset_stem_ta['cellid'].tolist()))

In [ ]:
cellrank_prob2 = cellrank_prob[cellrank_prob['leiden_annot_25_CR'].isin(subset_stem_ta['cellid'])]

In [ ]:
cellrank_prob2

In [ ]:
subset_stem_ta2 =  subset_stem_ta[subset_stem_ta['cellid'].isin(cellrank_prob2['leiden_annot_25_CR'])]

In [ ]:
cellrank_prob2 = cellrank_prob2.merge(
    subset_stem_ta2[['cellid','dcm_cell_type','rna_cell_type']],
    left_on='leiden_annot_25_CR',
    right_on='cellid',
    how='inner'
)

In [ ]:

cell_annotations = {
    row['cellid']: (row['dcm_cell_type'], row['rna_cell_type'])
    for _, row in cellrank_prob2.iterrows()
}

dcm_order      = ["Stem", "TA", "Enterocyte", "Paneth", "Goblet", "Tuft", "EEC"]
main_cell_types = ["Stem", "TA", "Enterocyte", "Paneth", "Goblet", "Tuft", "EEC"]
desired_ordery  = ["Stem", "TA", "Enterocyte", "Paneth", "Goblet", "Tuft", "EEC"]

color_palette   = ["#9370DB", "#D2B48C", "#FF8C00", "#DC143C", "#2E8B57", "#FF1493", '#4169E1']
cell_type_colors = dict(zip(main_cell_types, color_palette))

def get_sort_key(cell_id):
    dcm_type, rna_type = cell_annotations[cell_id]
    dcm_rank = dcm_order.index(dcm_type)
    rna_rank = main_cell_types.index(rna_type)
    return (dcm_rank, rna_rank)

individual_cells_sorted = sorted(cell_annotations.keys(), key=get_sort_key)
def get_sort_key(cell_id):
    dcm_type, rna_type = cell_annotations[cell_id]
    dcm_rank = dcm_order.index(dcm_type)
    rna_rank = main_cell_types.index(rna_type)
    return (dcm_rank, rna_rank)

individual_cells_sorted = sorted(cell_annotations.keys(), key=get_sort_key)
# Sort by DCM then RNA order
individual_cells_sorted = sorted(cell_annotations.keys(), key=get_sort_key)

# Reindex probabilities to match your cells
summed_probabilities_individuals = cellrank_prob2.set_index('cellid')[
    desired_ordery
].reindex(index=individual_cells_sorted)

fig2, ax2 = plt.subplots(figsize=(12, 6.5))
sns.heatmap(
    summed_probabilities_individuals.T,
    annot=False, cmap="RdBu_r",
    cbar_kws={'label': 'Probability'},
    vmin=0, vmax=1, ax=ax2
)

for i, cell_id in enumerate(individual_cells_sorted):
    dcm_type, rna_type = cell_annotations[cell_id]
    ax2.add_patch(plt.Rectangle((i, -1.4), 1, 0.5,
                                facecolor=cell_type_colors[dcm_type],
                                edgecolor='black', linewidth=0.5, clip_on=False))
    ax2.add_patch(plt.Rectangle((i, -0.7), 1, 0.5,
                                facecolor=cell_type_colors[rna_type],
                                edgecolor='black', linewidth=0.5, clip_on=False))

ax2.text(-1.5, -1.15, 'DCM', fontsize=10, weight='bold', va='center', ha='right')
ax2.text(-1.5, -0.45, 'RNA', fontsize=10, weight='bold', va='center', ha='right')

ax2.set_title("RNA Velocity Probabilities - Transitioning Cells", fontsize=16, weight='bold', pad=15)
ax2.set_xlabel("Source Individual Cells", fontsize=14, weight='bold')
ax2.set_ylabel("Destination Cell Type", fontsize=14, weight='bold')
ax2.tick_params(axis='both', which='major', labelsize=10)
plt.xticks(rotation=90, ha='right', fontsize=8)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('probability_heatmap_transitioning_cells.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#Heatmap of clear transitions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/gmm_085/'
output_dir = score_dir
os.makedirs(output_dir, exist_ok=True)

ttest_cols = [col for col in adata_combined.obs.columns
              if 'ttest_' in col and '_zscore' in col and 'max' not in col.lower()]
print(f"Found {len(ttest_cols)} ttest z-score columns: {ttest_cols}")

cell_type_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

cell_type_colors = {
    'Stem'      : '#9370DB',
    'TA'        : '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth'    : '#DC143C',
    'Goblet'    : '#2E8B57',
    'Tuft'      : '#FF1493',
    'EEC'       : '#4169E1',
}

# -----------------------------
# STEP 1: BUILD PLOT_DF
# -----------------------------
adata_combined.obs['batch_plate_well'] = (
    adata_combined.obs['Batch'].astype(str) + '_' +
    adata_combined.obs['plate_well'].astype(str)
)

plot_df = pd.DataFrame({
    'assay'           : adata_combined.obs['assay'].values,
    'batch_plate_well': adata_combined.obs['batch_plate_well'].values,
    'leiden_annot_25' : adata_combined.obs['leiden_annot_25'].astype(str).values,
    'gmm_label'       : adata_combined.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'Dox'             : adata_combined.obs['Dox'].astype(str).values
                        if 'Dox' in adata_combined.obs.columns else 'Unknown',
    'cell_index'      : adata_combined.obs.index.values,
})
plot_df['leiden_annot_25'] = plot_df['leiden_annot_25'].str.split('_').str[-1]
plot_df['gmm_label']       = plot_df['gmm_label'].str.split('_').str[-1]

for col in ttest_cols:
    plot_df[col] = adata_combined.obs[col].values

# -----------------------------
# STEP 2: FILTER CLEAR DCM & RNA CELLS
# -----------------------------
dcm_cells = plot_df[
    (plot_df['assay'] == 'pDCM') &
    (plot_df['gmm_label'].isin(cell_type_order)) &
    (plot_df['gmm_label'] == plot_df['leiden_annot_25']) &
    (plot_df['Dox'] != 'Minus')
].copy()

rna_cells = plot_df[
    (plot_df['assay'] == 'pRNA') &
    (plot_df['leiden_annot_25'].isin(cell_type_order)) &
    (plot_df['Dox'] != 'Minus')
].copy()

print(f"Clear DCM cells : {len(dcm_cells)}")
print(f"RNA cells       : {len(rna_cells)}")

# -----------------------------
# STEP 3: 1:1 MATCHING
# -----------------------------
common_bpw  = set(dcm_cells['batch_plate_well']) & set(rna_cells['batch_plate_well'])
dcm_single  = set(dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
                  ['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
rna_single  = set(rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]
                  ['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
both_single = dcm_single & rna_single
print(f"1:1 matched wells: {len(both_single)}")

connections = []
for bpw in both_single:
    dcm_row = dcm_cells[dcm_cells['batch_plate_well'] == bpw].iloc[0]
    rna_row = rna_cells[rna_cells['batch_plate_well'] == bpw].iloc[0]
    conn = {
        'batch_plate_well': bpw,
        'dcm_cell_type'   : dcm_row['gmm_label'],
        'rna_cell_type'   : rna_row['leiden_annot_25'],
    }
    for col in ttest_cols:
        ct = col.replace('ttest_', '').replace('_zscore', '')
        conn[f'dcm_{ct}_zscore'] = dcm_row[col]
        conn[f'rna_{ct}_zscore'] = rna_row[col]
    connections.append(conn)

connections_df = pd.DataFrame(connections)
print(f"Total clear transitions: {len(connections_df)}")

# -----------------------------
# STEP 3b: KEEP ONLY SWITCHING CELLS (DCM cell type ≠ RNA cell type)
# -----------------------------
connections_df = connections_df[
    connections_df['dcm_cell_type'] != connections_df['rna_cell_type']
].reset_index(drop=True)
print(f"Switching transitions (DCM ≠ RNA): {len(connections_df)}")

# -----------------------------
# STEP 4: SORT — group by DCM cell type, then by DCM source z-score
# -----------------------------
connections_df['dcm_sort_order'] = connections_df['dcm_cell_type'].map(
    {ct: i for i, ct in enumerate(cell_type_order)}
)
connections_df['dcm_self_zscore'] = connections_df.apply(
    lambda r: r[f'dcm_{r["dcm_cell_type"]}_zscore'], axis=1
)
connections_df = connections_df.sort_values(
    ['dcm_sort_order', 'dcm_self_zscore'], ascending=[True, False]
).reset_index(drop=True)

# -----------------------------
# STEP 5: BUILD MATRICES
# -----------------------------
dcm_matrix = np.array(
    [[row[f'dcm_{ct}_zscore'] for ct in cell_type_order]
     for _, row in connections_df.iterrows()]
).T  # shape: (n_cell_types, n_cells)

rna_matrix = np.array(
    [[row[f'rna_{ct}_zscore'] for ct in cell_type_order]
     for _, row in connections_df.iterrows()]
).T

# -----------------------------
# STEP 6: PLOT
# -----------------------------
vcenter = 1.0
vmin    = vcenter - 3
vmax    = vcenter + 3
norm    = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
cmap    = plt.cm.RdBu_r

n_cells    = len(connections_df)
fig_width  = max(14, n_cells * 0.18)
fig, axes  = plt.subplots(2, 1, figsize=(fig_width, 10))

for ax, matrix, label_col, title_str in [
    (axes[0], dcm_matrix, 'dcm_cell_type', 'pDCM (clear label: GMM == leiden_annot_25)'),
    (axes[1], rna_matrix, 'rna_cell_type', 'pRNA'),
]:
    im = ax.imshow(matrix, aspect='auto', cmap=cmap, norm=norm)

    ax.set_yticks(range(len(cell_type_order)))
    ax.set_yticklabels(cell_type_order, fontsize=10)
    ax.set_xticks([])
    ax.set_ylabel('Module', fontsize=11, fontweight='bold')
    ax.set_title(title_str, fontsize=12, fontweight='bold')

    # Vertical dividers between DCM cell type groups
    group_changes = connections_df['dcm_cell_type'].ne(
        connections_df['dcm_cell_type'].shift()
    ).to_numpy().nonzero()[0]

    for x in group_changes:
        ax.axvline(x - 0.5, color='white', lw=1.5, alpha=0.8)

    # Colour-coded cell type labels along the x-axis at the group midpoints
    prev = 0
    for x in list(group_changes) + [n_cells]:
        mid = (prev + x) / 2
        ct  = connections_df.iloc[prev]['dcm_cell_type']
        ax.text(mid, -0.8, ct,
                ha='center', va='top', fontsize=7, fontweight='bold',
                color=cell_type_colors[ct])
        prev = x

    # Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.01, pad=0.01)
    cbar.set_label('Z-Score', fontsize=9)
    cbar.ax.axhline(y=vcenter, color='black', lw=1, ls='--')

fig.suptitle(
    f'Switching Transitions: pDCM → pRNA  (n={n_cells} cells)\n'
    f'Only cells where DCM cell type ≠ RNA cell type\n'
    f'Grouped by DCM cell type, sorted by source z-score',
    fontsize=13, fontweight='bold'
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
save_path = os.path.join(output_dir, 'heatmap_switching_transitions.pdf')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print(f"✅ Saved: {save_path}")

# Save data
connections_df.to_csv(os.path.join(output_dir, 'switching_transitions_with_zscores.csv'), index=False)
print("✅ Saved: switching_transitions_with_zscores.csv")

print("\nDone!")

In [ ]:
#Transitions per cell type gmm 0.85

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/gmm_085/'
transition_dir = os.path.join(score_dir, 'transition_umaps_clear')
os.makedirs(transition_dir, exist_ok=True)

adata = adata_combined

# Small intestine cell types and colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': '#7f7f7f'
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata.obs['batch_plate_well'] = (
    adata.obs['Batch'].astype(str) + '_' + 
    adata.obs['plate_well'].astype(str)
)

# -----------------------------
# EXTRACT UMAP AND METADATA
# -----------------------------
print("\n" + "="*60)
print("PREPARING DATA")
print("="*60)

umap_coords = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1': umap_coords[:, 0],
    'UMAP2': umap_coords[:, 1],
    'assay': adata.obs['assay'].values,
    'plate_well': adata.obs['plate_well'].values,
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'Batch': adata.obs['Batch'].values,
    'cell_type_broad': adata.obs['leiden_annot_25'].astype(str).values,
    'module_max_label': adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'cell_index': adata.obs.index.values
})

if 'Dox' in adata.obs.columns:
    plot_df['Dox'] = adata.obs['Dox'].values

# -----------------------------
# FILTER DCM CELLS - CLEAR TRANSITIONS ONLY (GMM == leiden_annot_25)
# -----------------------------
print("\n" + "="*60)
print("FILTERING CELLS")
print("="*60)

dcm_mask = plot_df['assay'] == 'pDCM'
dcm_known_mask = (dcm_mask & 
                  plot_df['module_max_label'].isin(label_order) &
                  (plot_df['module_max_label'] == plot_df['cell_type_broad']))  # CLEAR ONLY

# Exclude Minus Dox
if 'Dox' in plot_df.columns:
    dcm_known_mask = dcm_known_mask & (plot_df['Dox'] != 'Minus')

dcm_cells = plot_df[dcm_known_mask].copy()

# RNA cells with known labels
rna_mask = plot_df['assay'] == 'pRNA'
rna_known_mask = rna_mask & plot_df['cell_type_broad'].isin(label_order)

# Exclude Minus Dox
if 'Dox' in plot_df.columns:
    rna_known_mask = rna_known_mask & (plot_df['Dox'] != 'Minus')

rna_cells = plot_df[rna_known_mask].copy()

print(f"Total pDCM cells: {dcm_mask.sum()}")
print(f"DCM cells with CLEAR labels (GMM == leiden_annot_25, excluding Minus Dox): {len(dcm_cells)}")
print(f"RNA cells with known labels (excluding Minus Dox): {len(rna_cells)}")

print("\nDCM clear label distribution:")
print(dcm_cells['module_max_label'].value_counts())

# -----------------------------
# MATCH BY BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("FINDING CONNECTIONS USING BATCH_PLATE_WELL")
print("="*60)

dcm_bpw = set(dcm_cells['batch_plate_well'])
rna_bpw = set(rna_cells['batch_plate_well'])
common_bpw = dcm_bpw.intersection(rna_bpw)
print(f"Common batch_plate_wells: {len(common_bpw)}")

dcm_common = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
rna_common = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]

dcm_counts = dcm_common['batch_plate_well'].value_counts()
rna_counts = rna_common['batch_plate_well'].value_counts()

dcm_single = set(dcm_counts[dcm_counts == 1].index)
rna_single = set(rna_counts[rna_counts == 1].index)
both_single = dcm_single.intersection(rna_single)

print(f"1:1 matched batch_plate_wells: {len(both_single)}")

# Build connections
connections = []
for bpw in both_single:
    dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
    rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
    
    if len(dcm_cell) == 1 and len(rna_cell) == 1:
        dcm_row = dcm_cell.iloc[0]
        rna_row = rna_cell.iloc[0]
        
        connections.append({
            'batch_plate_well': bpw,
            'batch': dcm_row['Batch'],
            'plate_well': dcm_row['plate_well'],
            'dcm_umap1': dcm_row['UMAP1'],
            'dcm_umap2': dcm_row['UMAP2'],
            'rna_umap1': rna_row['UMAP1'],
            'rna_umap2': rna_row['UMAP2'],
            'dcm_cell_type': dcm_row['module_max_label'],
            'rna_cell_type': rna_row['cell_type_broad'],
            'transition': f"{dcm_row['module_max_label']} -> {rna_row['cell_type_broad']}",
            'is_stable': dcm_row['module_max_label'] == rna_row['cell_type_broad']
        })

connections_df = pd.DataFrame(connections)
print(f"Total valid clear connections: {len(connections_df)}")

# -----------------------------
# FUNCTION TO PLOT TRANSITIONS FROM ONE DCM CELL TYPE (COLORED ARROWS)
# -----------------------------
def plot_transitions_from_dcm_type(dcm_cell_type, connections_df, plot_df,
                                    cell_type_colors, label_order, save_dir):
    print(f"\n{'='*60}")
    print(f"PLOTTING TRANSITIONS FROM DCM {dcm_cell_type.upper()} (clear only)")
    print(f"{'='*60}")

    dcm_connections = connections_df[connections_df['dcm_cell_type'] == dcm_cell_type].copy()

    if len(dcm_connections) == 0:
        print(f"  No connections found for DCM {dcm_cell_type}")
        return None

    print(f"  Total connections: {len(dcm_connections)}")
    print(f"  RNA destinations:")
    for rna_ct, count in dcm_connections['rna_cell_type'].value_counts().items():
        print(f"    -> {rna_ct}: {count}")

    trans_bpw = set(dcm_connections['batch_plate_well'])

    fig, ax = plt.subplots(figsize=(12, 10))

    # Reference background
    for ct in label_order:
        ref_cells = plot_df[(plot_df['assay'] == 'Ref') & (plot_df['cell_type_broad'] == ct)]
        if len(ref_cells) > 0:
            ax.scatter(ref_cells['UMAP1'], ref_cells['UMAP2'],
                       s=8, c=cell_type_colors[ct], alpha=0.08, marker='o',
                       zorder=1, rasterized=True)

    ref_ambig = plot_df[(plot_df['assay'] == 'Ref') & (~plot_df['cell_type_broad'].isin(label_order))]
    if len(ref_ambig) > 0:
        ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                   s=8, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)

    # Colored arrows based on RNA destination
    for _, conn in dcm_connections.iterrows():
        rna_ct = conn['rna_cell_type']
        arrow_color = cell_type_colors.get(rna_ct, '#7f7f7f')

        arrow = FancyArrowPatch(
            (conn['dcm_umap1'], conn['dcm_umap2']),
            (conn['rna_umap1'], conn['rna_umap2']),
            arrowstyle='-|>', mutation_scale=15,
            color=arrow_color, alpha=0.7, linewidth=2, zorder=3
        )
        ax.add_patch(arrow)

    # DCM cells - squares
    dcm_cells_plot = plot_df[(plot_df['assay'] == 'pDCM') &
                             (plot_df['batch_plate_well'].isin(trans_bpw)) &
                             (plot_df['module_max_label'] == dcm_cell_type)]
    if len(dcm_cells_plot) > 0:
        ax.scatter(dcm_cells_plot['UMAP1'], dcm_cells_plot['UMAP2'],
                   s=120, c=cell_type_colors[dcm_cell_type], alpha=0.9,
                   edgecolors='black', linewidths=1.5, marker='s', zorder=4, rasterized=True)

    # RNA cells - circles
    for rna_ct in label_order:
        rna_ct_connections = dcm_connections[dcm_connections['rna_cell_type'] == rna_ct]
        if len(rna_ct_connections) == 0:
            continue

        rna_bpw = set(rna_ct_connections['batch_plate_well'])
        rna_cells_plot = plot_df[(plot_df['assay'] == 'pRNA') &
                                 (plot_df['batch_plate_well'].isin(rna_bpw)) &
                                 (plot_df['cell_type_broad'] == rna_ct)]

        if len(rna_cells_plot) > 0:
            ax.scatter(rna_cells_plot['UMAP1'], rna_cells_plot['UMAP2'],
                       s=120, c=cell_type_colors[rna_ct], alpha=0.9,
                       edgecolors='black', linewidths=1.5, marker='o', zorder=4, rasterized=True)

    # Legend
    legend_elements = [
        mlines.Line2D([], [], color=cell_type_colors[dcm_cell_type], marker='s',
                     linestyle='None', markersize=12, markeredgecolor='black',
                     label=f'DCM: {dcm_cell_type} (n={len(dcm_connections)})')
    ]

    for rna_ct in label_order:
        count = (dcm_connections['rna_cell_type'] == rna_ct).sum()
        if count > 0:
            legend_elements.append(
                mlines.Line2D([], [], color=cell_type_colors[rna_ct], marker='o',
                             linestyle='None', markersize=10, markeredgecolor='black',
                             label=f'RNA: {rna_ct} (n={count})')
            )

    legend_elements.append(
        mlines.Line2D([], [], color='gray', linestyle='-', linewidth=2, alpha=0.7,
                     marker='>', markersize=8, label='DCM → RNA')
    )

    ax.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=9,
             fancybox=True, framealpha=0.9)

    n_stable = dcm_connections['is_stable'].sum()
    n_unstable = len(dcm_connections) - n_stable
    pct_stable = (n_stable / len(dcm_connections) * 100) if len(dcm_connections) > 0 else 0

    ax.set_title(f'Clear Transitions from DCM {dcm_cell_type} (GMM == leiden_annot_25)\n'
                f'Total: {len(dcm_connections)} | Stable: {n_stable} ({pct_stable:.1f}%) | Unstable: {n_unstable}',
                fontsize=14, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')

    plt.tight_layout()

    save_path = os.path.join(save_dir, f'umap_clear_from_DCM_{dcm_cell_type}_to_all.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  Saved: {save_path}")

    return dcm_connections


# -----------------------------
# FUNCTION TO PLOT TRANSITIONS TO ONE RNA CELL TYPE (COLORED ARROWS)
# -----------------------------
def plot_transitions_to_rna_type(rna_cell_type, connections_df, plot_df,
                                  cell_type_colors, label_order, save_dir):
    print(f"\n{'='*60}")
    print(f"PLOTTING TRANSITIONS TO RNA {rna_cell_type.upper()} (clear only)")
    print(f"{'='*60}")

    rna_connections = connections_df[connections_df['rna_cell_type'] == rna_cell_type].copy()

    if len(rna_connections) == 0:
        print(f"  No connections found for RNA {rna_cell_type}")
        return None

    print(f"  Total connections: {len(rna_connections)}")
    print(f"  DCM sources:")
    for dcm_ct, count in rna_connections['dcm_cell_type'].value_counts().items():
        print(f"    <- {dcm_ct}: {count}")

    trans_bpw = set(rna_connections['batch_plate_well'])

    fig, ax = plt.subplots(figsize=(12, 10))

    # Reference background
    for ct in label_order:
        ref_cells = plot_df[(plot_df['assay'] == 'Ref') & (plot_df['cell_type_broad'] == ct)]
        if len(ref_cells) > 0:
            ax.scatter(ref_cells['UMAP1'], ref_cells['UMAP2'],
                       s=8, c=cell_type_colors[ct], alpha=0.08, marker='o',
                       zorder=1, rasterized=True)

    ref_ambig = plot_df[(plot_df['assay'] == 'Ref') & (~plot_df['cell_type_broad'].isin(label_order))]
    if len(ref_ambig) > 0:
        ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                   s=8, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)

    # Colored arrows based on DCM source
    for _, conn in rna_connections.iterrows():
        dcm_ct = conn['dcm_cell_type']
        arrow_color = cell_type_colors.get(dcm_ct, '#7f7f7f')

        arrow = FancyArrowPatch(
            (conn['dcm_umap1'], conn['dcm_umap2']),
            (conn['rna_umap1'], conn['rna_umap2']),
            arrowstyle='-|>', mutation_scale=15,
            color=arrow_color, alpha=0.7, linewidth=2, zorder=3
        )
        ax.add_patch(arrow)

    # DCM cells - squares
    for dcm_ct in label_order:
        dcm_ct_connections = rna_connections[rna_connections['dcm_cell_type'] == dcm_ct]
        if len(dcm_ct_connections) == 0:
            continue

        dcm_bpw = set(dcm_ct_connections['batch_plate_well'])
        dcm_cells_plot = plot_df[(plot_df['assay'] == 'pDCM') &
                                 (plot_df['batch_plate_well'].isin(dcm_bpw)) &
                                 (plot_df['module_max_label'] == dcm_ct)]

        if len(dcm_cells_plot) > 0:
            ax.scatter(dcm_cells_plot['UMAP1'], dcm_cells_plot['UMAP2'],
                       s=120, c=cell_type_colors[dcm_ct], alpha=0.9,
                       edgecolors='black', linewidths=1.5, marker='s', zorder=4, rasterized=True)

    # RNA cells - circles
    rna_cells_plot = plot_df[(plot_df['assay'] == 'pRNA') &
                             (plot_df['batch_plate_well'].isin(trans_bpw)) &
                             (plot_df['cell_type_broad'] == rna_cell_type)]
    if len(rna_cells_plot) > 0:
        ax.scatter(rna_cells_plot['UMAP1'], rna_cells_plot['UMAP2'],
                   s=120, c=cell_type_colors[rna_cell_type], alpha=0.9,
                   edgecolors='black', linewidths=1.5, marker='o', zorder=4, rasterized=True)

    # Legend
    legend_elements = []

    for dcm_ct in label_order:
        count = (rna_connections['dcm_cell_type'] == dcm_ct).sum()
        if count > 0:
            legend_elements.append(
                mlines.Line2D([], [], color=cell_type_colors[dcm_ct], marker='s',
                             linestyle='None', markersize=10, markeredgecolor='black',
                             label=f'DCM: {dcm_ct} (n={count})')
            )

    legend_elements.append(
        mlines.Line2D([], [], color=cell_type_colors[rna_cell_type], marker='o',
                     linestyle='None', markersize=12, markeredgecolor='black',
                     label=f'RNA: {rna_cell_type} (n={len(rna_connections)})')
    )

    legend_elements.append(
        mlines.Line2D([], [], color='gray', linestyle='-', linewidth=2, alpha=0.7,
                     marker='>', markersize=8, label='DCM → RNA')
    )

    ax.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=9,
             fancybox=True, framealpha=0.9)

    n_stable = rna_connections['is_stable'].sum()
    n_unstable = len(rna_connections) - n_stable
    pct_stable = (n_stable / len(rna_connections) * 100) if len(rna_connections) > 0 else 0

    ax.set_title(f'Clear Transitions to RNA {rna_cell_type} (GMM == leiden_annot_25)\n'
                f'Total: {len(rna_connections)} | Stable: {n_stable} ({pct_stable:.1f}%) | Unstable: {n_unstable}',
                fontsize=14, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')

    plt.tight_layout()

    save_path = os.path.join(save_dir, f'umap_clear_to_RNA_{rna_cell_type}_from_all.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  Saved: {save_path}")

    return rna_connections


# -----------------------------
# CREATE UMAPS FOR EACH DCM CELL TYPE
# -----------------------------
print("\n" + "="*60)
print("CREATING UMAPS: FROM EACH DCM CELL TYPE TO ALL RNA")
print("="*60)

dcm_summary = {}

for dcm_ct in label_order:
    if dcm_ct in connections_df['dcm_cell_type'].unique():
        result = plot_transitions_from_dcm_type(
            dcm_ct, connections_df, plot_df,
            cell_type_colors, label_order, transition_dir
        )
        if result is not None:
            dcm_summary[dcm_ct] = result['rna_cell_type'].value_counts().to_dict()

# -----------------------------
# CREATE UMAPS FOR EACH RNA CELL TYPE
# -----------------------------
print("\n" + "="*60)
print("CREATING UMAPS: TO EACH RNA CELL TYPE FROM ALL DCM")
print("="*60)

rna_summary = {}

for rna_ct in label_order:
    if rna_ct in connections_df['rna_cell_type'].unique():
        result = plot_transitions_to_rna_type(
            rna_ct, connections_df, plot_df,
            cell_type_colors, label_order, transition_dir
        )
        if result is not None:
            rna_summary[rna_ct] = result['dcm_cell_type'].value_counts().to_dict()

# -----------------------------
# SUMMARY TABLE
# -----------------------------
print("\n" + "="*60)
print("TRANSITION SUMMARY")
print("="*60)

print("\n--- From DCM to RNA ---")
for dcm_ct, destinations in dcm_summary.items():
    total = sum(destinations.values())
    print(f"\n{dcm_ct} (n={total}):")
    for rna_ct, count in sorted(destinations.items(), key=lambda x: -x[1]):
        pct = (count / total * 100) if total > 0 else 0
        stable = "STABLE" if dcm_ct == rna_ct else ""
        print(f"  -> {rna_ct}: {count} ({pct:.1f}%) {stable}")

print("\n--- To RNA from DCM ---")
for rna_ct, sources in rna_summary.items():
    total = sum(sources.values())
    print(f"\n{rna_ct} (n={total}):")
    for dcm_ct, count in sorted(sources.items(), key=lambda x: -x[1]):
        pct = (count / total * 100) if total > 0 else 0
        stable = "STABLE" if dcm_ct == rna_ct else ""
        print(f"  <- {dcm_ct}: {count} ({pct:.1f}%) {stable}")

# -----------------------------
# SUMMARY HEATMAPS
# -----------------------------
print("\n" + "="*60)
print("CREATING SUMMARY HEATMAPS")
print("="*60)

transition_matrix = pd.crosstab(
    connections_df['dcm_cell_type'],
    connections_df['rna_cell_type'],
    rownames=['DCM Cell Type'],
    colnames=['RNA Cell Type']
)

transition_matrix = transition_matrix.reindex(
    index=[l for l in label_order if l in transition_matrix.index],
    columns=[l for l in label_order if l in transition_matrix.columns]
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax1 = axes[0]
sns.heatmap(transition_matrix, annot=True, fmt='d', cmap='viridis',
            cbar_kws={'label': 'Number of cells'}, ax=ax1)
ax1.set_title('Clear Transitions: DCM → RNA (Counts)\n(DCM: GMM == leiden_annot_25)',
             fontsize=12, fontweight='bold')
ax1.set_xlabel('RNA Cell Type', fontsize=11)
ax1.set_ylabel('DCM Cell Type', fontsize=11)

ax2 = axes[1]
transition_pct = transition_matrix.div(transition_matrix.sum(axis=1), axis=0) * 100
sns.heatmap(transition_pct, annot=True, fmt='.1f', cmap='viridis',
            cbar_kws={'label': 'Percentage (%)'}, ax=ax2)
ax2.set_title('Clear Transitions: DCM → RNA (% of DCM type)\n(DCM: GMM == leiden_annot_25)',
             fontsize=12, fontweight='bold')
ax2.set_xlabel('RNA Cell Type', fontsize=11)
ax2.set_ylabel('DCM Cell Type', fontsize=11)

plt.tight_layout()
save_path = os.path.join(transition_dir, 'clear_transition_summary_heatmaps.pdf')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print(f"Saved: {save_path}")

# -----------------------------
# SAVE DATA
# -----------------------------
connections_df.to_csv(os.path.join(transition_dir, 'clear_transitions_batch_plate_well.csv'), index=False)
transition_matrix.to_csv(os.path.join(transition_dir, 'clear_transition_matrix_counts.csv'))
transition_pct.to_csv(os.path.join(transition_dir, 'clear_transition_matrix_percentages.csv'))

print(f"\nSaved:")
print(f"  - {os.path.join(transition_dir, 'clear_transitions_batch_plate_well.csv')}")
print(f"  - {os.path.join(transition_dir, 'clear_transition_matrix_counts.csv')}")
print(f"  - {os.path.join(transition_dir, 'clear_transition_matrix_percentages.csv')}")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
from adjustText import adjust_text
import os

# -----------------------------
# SETTINGS
# -----------------------------
#sc.settings.figdir = '/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/integration/plots_paper/'
sc.settings.figdir = '/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/gene_scoring/gmm_085/'

output_dir = sc.settings.figdir
os.makedirs(output_dir, exist_ok=True)

adata = adata_combined

# Small intestine cell types and colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': '#7f7f7f'
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# Cells to exclude
cells_to_exclude = [
    'Batch_1_Plate_1_P07',
    'Batch_1_Plate_1_I06',
    'Batch_2_Plate_2_H13',
    'Batch_1_Plate_2_D12',
    'Batch_1_Plate_1_E22',
    'Batch_2_Plate_3_C22',
    'Batch_1_Plate_1_B23'
]

# -----------------------------
# EXTRACT UMAP AND METADATA
# -----------------------------
umap_coords = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1': umap_coords[:, 0],
    'UMAP2': umap_coords[:, 1],
    'assay': adata.obs['assay'].values,
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'leiden_annot_25': adata.obs['leiden_annot_25'].astype(str).values,
    'ttest_gmm_label_confident': adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'cell_index': adata.obs.index.values
})

# -----------------------------
# FILTER DCM CELLS WITH CLEAR LABELS (GMM matches leiden_annot_25)
# -----------------------------
print("\n" + "="*60)
print("STEP 1: FILTERING DCM CELLS WITH CLEAR TRANSITIONS")
print("="*60)

dcm_mask = plot_df['assay'] == 'pDCM'
# Only keep DCM cells where GMM label matches leiden annotation (clear transitions)
dcm_clear_mask = (dcm_mask & 
                  (plot_df['ttest_gmm_label_confident'].isin(label_order)) &
                  (plot_df['ttest_gmm_label_confident'] == plot_df['leiden_annot_25']))

dcm_cells = plot_df[dcm_clear_mask].copy()

print(f"Total pDCM cells: {dcm_mask.sum()}")
print(f"DCM cells with CLEAR labels (GMM == leiden_annot_25): {dcm_clear_mask.sum()}")
print("\nDCM clear label distribution:")
print(dcm_cells['ttest_gmm_label_confident'].value_counts())

# -----------------------------
# FILTER RNA CELLS
# -----------------------------
print("\n" + "="*60)
print("STEP 2: FILTERING RNA CELLS")
print("="*60)

rna_mask = plot_df['assay'] == 'pRNA'
rna_cells = plot_df[rna_mask].copy()
print(f"Total pRNA cells: {rna_mask.sum()}")
print("\nRNA cell type distribution:")
print(rna_cells['leiden_annot_25'].value_counts())

# -----------------------------
# MATCH BY BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("STEP 3: FINDING CONNECTIONS USING BATCH_PLATE_WELL")
print("="*60)

dcm_bpw = set(dcm_cells['batch_plate_well'])
rna_bpw = set(rna_cells['batch_plate_well'])
common_bpw = dcm_bpw.intersection(rna_bpw)
print(f"Common batch_plate_wells: {len(common_bpw)}")

# Filter to common
dcm_common = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
rna_common = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]

# Find 1:1 matches
dcm_counts = dcm_common['batch_plate_well'].value_counts()
rna_counts = rna_common['batch_plate_well'].value_counts()

dcm_single = set(dcm_counts[dcm_counts == 1].index)
rna_single = set(rna_counts[rna_counts == 1].index)
both_single = dcm_single.intersection(rna_single)

print(f"1:1 matched batch_plate_wells: {len(both_single)}")

# Build connections
connections = []
for bpw in both_single:
    dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
    rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
    
    if len(dcm_cell) == 1 and len(rna_cell) == 1:
        dcm_row = dcm_cell.iloc[0]
        rna_row = rna_cell.iloc[0]
        
        connections.append({
            'batch_plate_well': bpw,
            'dcm_umap1': dcm_row['UMAP1'],
            'dcm_umap2': dcm_row['UMAP2'],
            'rna_umap1': rna_row['UMAP1'],
            'rna_umap2': rna_row['UMAP2'],
            'dcm_cell_type': dcm_row['ttest_gmm_label_confident'],
            'rna_cell_type': rna_row['leiden_annot_25'],
            'transition': f"{dcm_row['ttest_gmm_label_confident']} -> {rna_row['leiden_annot_25']}"
        })

connections_df = pd.DataFrame(connections)
print(f"Total valid connections with clear DCM labels: {len(connections_df)}")

# -----------------------------
# FILTER OUT STABLE CELLS (SAME CELL TYPE)
# -----------------------------
print("\n" + "="*60)
print("STEP 4: FILTERING OUT STABLE CELLS (DCM == RNA)")
print("="*60)

connections_df['stable'] = connections_df['dcm_cell_type'] == connections_df['rna_cell_type']
n_stable = connections_df['stable'].sum()
n_changed = (~connections_df['stable']).sum()

print(f"Total connections before filtering: {len(connections_df)}")
print(f"Stable (same type): {n_stable} ({n_stable/len(connections_df)*100:.1f}%)")
print(f"Changed (different type): {n_changed} ({n_changed/len(connections_df)*100:.1f}%)")

# Keep only cells that changed cell type
connections_df = connections_df[~connections_df['stable']].copy()
print(f"\nConnections after removing stable cells: {len(connections_df)}")

# -----------------------------
# REMOVE SPECIFIED CELLS
# -----------------------------
print("\n" + "="*60)
print("STEP 5: REMOVING SPECIFIED CELLS")
print("="*60)

print(f"Cells to exclude: {cells_to_exclude}")
print(f"Connections before exclusion: {len(connections_df)}")

connections_df = connections_df[~connections_df['batch_plate_well'].isin(cells_to_exclude)].copy()

print(f"Connections after exclusion: {len(connections_df)}")

# -----------------------------
# PLOT ALL TRANSITIONS - WITH LABELS
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("PLOTTING CELLS WITH LABELS")
    print("="*60)
    
    fig, ax = plt.subplots(figsize=(14, 12))

    # Reference background
    ref_mask = (plot_df['assay'] == 'Ref') if 'Ref' in plot_df['assay'].values else (plot_df['assay'].isin(['pDCM', 'pRNA']))
    
    for ct in label_order:
        ref_cells = plot_df[ref_mask & (plot_df['leiden_annot_25'] == ct)]
        if len(ref_cells) > 0:
            ax.scatter(ref_cells['UMAP1'], ref_cells['UMAP2'],
                       s=8, c=cell_type_colors[ct], alpha=0.05, marker='o', zorder=1, rasterized=True)

    # Ambiguous reference
    ref_ambig = plot_df[ref_mask & (~plot_df['leiden_annot_25'].isin(label_order))]
    if len(ref_ambig) > 0:
        ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                   s=8, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)

    # Draw all arrows
    for _, conn in connections_df.iterrows():
        arrow = FancyArrowPatch((conn['dcm_umap1'], conn['dcm_umap2']),
                                (conn['rna_umap1'], conn['rna_umap2']),
                                arrowstyle='-|>', mutation_scale=15,
                                color='black', alpha=0.6, linewidth=1.5, zorder=3)
        ax.add_patch(arrow)

    # Get all unique batch_plate_wells from connections
    all_bpw = set(connections_df['batch_plate_well'])

    # Plot all DCM cells - squares
    for ct in label_order:
        dcm_cells_plot = plot_df[(plot_df['assay'] == 'pDCM') &
                                 (plot_df['batch_plate_well'].isin(all_bpw)) &
                                 (plot_df['ttest_gmm_label_confident'] == ct)]
        if len(dcm_cells_plot) > 0:
            ax.scatter(dcm_cells_plot['UMAP1'], dcm_cells_plot['UMAP2'],
                       s=80, c=cell_type_colors[ct], alpha=0.9,
                       edgecolors='black', linewidths=1, marker='s', zorder=4, rasterized=True)

    # Plot all RNA cells - circles
    for ct in label_order:
        rna_cells_plot = plot_df[(plot_df['assay'] == 'pRNA') &
                                 (plot_df['batch_plate_well'].isin(all_bpw)) &
                                 (plot_df['leiden_annot_25'] == ct)]
        if len(rna_cells_plot) > 0:
            ax.scatter(rna_cells_plot['UMAP1'], rna_cells_plot['UMAP2'],
                       s=80, c=cell_type_colors[ct], alpha=0.9,
                       edgecolors='black', linewidths=1, marker='o', zorder=4, rasterized=True)

    # Add text labels for DCM cells only
    texts = []
    dcm_labeled_cells = plot_df[(plot_df['assay'] == 'pDCM') &
                                 (plot_df['batch_plate_well'].isin(all_bpw))]
    
    for _, cell in dcm_labeled_cells.iterrows():
        text = ax.text(cell['UMAP1'], cell['UMAP2'], 
                      cell['batch_plate_well'],
                      fontsize=7, fontweight='normal',
                      ha='center', va='center',
                      bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                               edgecolor='gray', alpha=0.8, linewidth=0.5),
                      zorder=5)
        texts.append(text)
    
    # Adjust text positions to avoid overlap
    if len(texts) > 0:
        adjust_text(texts, 
                   arrowprops=dict(arrowstyle='-', color='gray', lw=0.5, alpha=0.5),
                   expand_points=(1.5, 1.5),
                   expand_text=(1.2, 1.2),
                   force_points=(0.5, 0.5),
                   force_text=(0.5, 0.5),
                   ax=ax)

    # Create custom legend
    legend_elements = []
    for ct in label_order:
        legend_elements.append(
            mlines.Line2D([], [], color=cell_type_colors[ct], marker='s', linestyle='None', 
                         markersize=10, markeredgecolor='black', label=f'DCM {ct}')
        )
    for ct in label_order:
        legend_elements.append(
            mlines.Line2D([], [], color=cell_type_colors[ct], marker='o', linestyle='None', 
                         markersize=10, markeredgecolor='black', label=f'RNA {ct}')
        )
    legend_elements.append(
        mlines.Line2D([], [], color='black', linestyle='-', linewidth=2, alpha=0.8,
                     marker='>', markersize=8, label='DCM → RNA')
    )
    
    ax.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1, 0.5), 
             frameon=True, fontsize=9, ncol=2)

    ax.set_title(f'Cell Type Transitions: DCM → RNA (Changed Cells Only)\n(n = {len(connections_df)} cells, DCM: GMM==leiden_annot_25)', 
                fontsize=14, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')

    plt.tight_layout()
    save_path = os.path.join(output_dir, 'umap_changed_transitions_with_labels.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved: {save_path}")

# -----------------------------
# PLOT ALL TRANSITIONS - WITHOUT LABELS
# -----------------------------
if len(connections_df) > 0:
    print("\n" + "="*60)
    print("PLOTTING CELLS WITHOUT LABELS")
    print("="*60)
    
    fig, ax = plt.subplots(figsize=(12, 10))

    # Reference background
    ref_mask = (plot_df['assay'] == 'Ref') if 'Ref' in plot_df['assay'].values else (plot_df['assay'].isin(['pDCM', 'pRNA']))
    
    for ct in label_order:
        ref_cells = plot_df[ref_mask & (plot_df['leiden_annot_25'] == ct)]
        if len(ref_cells) > 0:
            ax.scatter(ref_cells['UMAP1'], ref_cells['UMAP2'],
                       s=8, c=cell_type_colors[ct], alpha=0.05, marker='o', zorder=1, rasterized=True)

    # Ambiguous reference
    ref_ambig = plot_df[ref_mask & (~plot_df['leiden_annot_25'].isin(label_order))]
    if len(ref_ambig) > 0:
        ax.scatter(ref_ambig['UMAP1'], ref_ambig['UMAP2'],
                   s=8, c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)

    # Draw all arrows
    for _, conn in connections_df.iterrows():
        arrow = FancyArrowPatch((conn['dcm_umap1'], conn['dcm_umap2']),
                                (conn['rna_umap1'], conn['rna_umap2']),
                                arrowstyle='-|>', mutation_scale=15,
                                color='black', alpha=0.6, linewidth=1.5, zorder=3)
        ax.add_patch(arrow)

    # Get all unique batch_plate_wells from connections
    all_bpw = set(connections_df['batch_plate_well'])

    # Plot all DCM cells - squares
    for ct in label_order:
        dcm_cells_plot = plot_df[(plot_df['assay'] == 'pDCM') &
                                 (plot_df['batch_plate_well'].isin(all_bpw)) &
                                 (plot_df['ttest_gmm_label_confident'] == ct)]
        if len(dcm_cells_plot) > 0:
            ax.scatter(dcm_cells_plot['UMAP1'], dcm_cells_plot['UMAP2'],
                       s=80, c=cell_type_colors[ct], alpha=0.9,
                       edgecolors='black', linewidths=1, marker='s', zorder=4, rasterized=True)

    # Plot all RNA cells - circles
    for ct in label_order:
        rna_cells_plot = plot_df[(plot_df['assay'] == 'pRNA') &
                                 (plot_df['batch_plate_well'].isin(all_bpw)) &
                                 (plot_df['leiden_annot_25'] == ct)]
        if len(rna_cells_plot) > 0:
            ax.scatter(rna_cells_plot['UMAP1'], rna_cells_plot['UMAP2'],
                       s=80, c=cell_type_colors[ct], alpha=0.9,
                       edgecolors='black', linewidths=1, marker='o', zorder=4, rasterized=True)

    # Create custom legend
    legend_elements = []
    for ct in label_order:
        legend_elements.append(
            mlines.Line2D([], [], color=cell_type_colors[ct], marker='s', linestyle='None', 
                         markersize=10, markeredgecolor='black', label=f'DCM {ct}')
        )
    for ct in label_order:
        legend_elements.append(
            mlines.Line2D([], [], color=cell_type_colors[ct], marker='o', linestyle='None', 
                         markersize=10, markeredgecolor='black', label=f'RNA {ct}')
        )
    legend_elements.append(
        mlines.Line2D([], [], color='black', linestyle='-', linewidth=2, alpha=0.8,
                     marker='>', markersize=8, label='DCM → RNA')
    )
    
    ax.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1, 0.5), 
             frameon=True, fontsize=9, ncol=2)

    ax.set_title(f'Cell Type Transitions: DCM → RNA (Changed Cells Only)\n(n = {len(connections_df)} cells, DCM: GMM==leiden_annot_25)', 
                fontsize=14, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')

    plt.tight_layout()
    save_path = os.path.join(output_dir, 'umap_changed_transitions_no_labels.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved: {save_path}")

    # Save connections summary
    connections_df.to_csv(os.path.join(output_dir, 'DCM_RNA_changed_connections_filtered.csv'), index=False)
    print(f"\nSaved connections: {os.path.join(output_dir, 'DCM_RNA_changed_connections_filtered.csv')}")

else:
    print("No changed connections found!")

# -----------------------------
# SUMMARY BY TRANSITION
# -----------------------------
print("\n" + "="*60)
print("SUMMARY OF CELL TYPE CHANGES")
print("="*60)

if len(connections_df) > 0:
    transition_counts = connections_df['transition'].value_counts()
    print(transition_counts)
    print(f"\nTotal cells that changed cell type: {len(connections_df)}")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/'
os.makedirs(score_dir, exist_ok=True)

# Extract columns for z-scores and posteriors, EXCLUDING 'max' columns
ttest_zscore_cols = [col for col in adata_combined.obs.columns 
                     if 'ttest_' in col and '_zscore' in col and 'max' not in col.lower()]

ttest_posterior_cols = [col for col in adata_combined.obs.columns 
                        if 'ttest_' in col and '_posterior' in col and 'max' not in col.lower()]

print(f"Found {len(ttest_zscore_cols)} z-score columns")
print(f"Found {len(ttest_posterior_cols)} posterior columns")

# Define cell type order
cell_type_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# Cell type colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1'
}

# Cells to exclude (from the provided code)
cells_to_exclude = [
    'Batch_1_Plate_1_P07',
    'Batch_1_Plate_1_I06',
    'Batch_2_Plate_2_H13',
    'Batch_1_Plate_2_D12',
    'Batch_1_Plate_1_E22',
    'Batch_2_Plate_3_C22',
    'Batch_1_Plate_1_B23'
]

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("\n" + "="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata_combined.obs['batch_plate_well'] = (
    adata_combined.obs['Batch'].astype(str) + '_' + 
    adata_combined.obs['plate_well'].astype(str)
)

# -----------------------------
# PREPARE DATA - FOLLOWING PROVIDED CODE LOGIC
# -----------------------------
print("\n" + "="*60)
print("PREPARING DATA WITH FILTERING LOGIC FROM PROVIDED CODE")
print("="*60)

plot_df = pd.DataFrame({
    'assay': adata_combined.obs['assay'].values,
    'plate_well': adata_combined.obs['plate_well'].values,
    'batch_plate_well': adata_combined.obs['batch_plate_well'].values,
    'Batch': adata_combined.obs['Batch'].values,
    'leiden_annot_25': adata_combined.obs['leiden_annot_25'].astype(str).values,
    'ttest_gmm_label_confident': adata_combined.obs['ttest_gmm_label_confident'].astype(str).values,
    'cell_index': adata_combined.obs.index.values
})

# Extract cell type from labels (remove number prefix if present)
plot_df['leiden_annot_25'] = plot_df['leiden_annot_25'].str.split('_').str[-1]
plot_df['ttest_gmm_label_confident'] = plot_df['ttest_gmm_label_confident'].str.split('_').str[-1]

# Add z-score and posterior columns
for col in ttest_zscore_cols:
    plot_df[col] = adata_combined.obs[col].values

for col in ttest_posterior_cols:
    plot_df[col] = adata_combined.obs[col].values

# -----------------------------
# FILTER DCM CELLS WITH CLEAR LABELS (GMM matches leiden_annot_25)
# -----------------------------
print("\n" + "="*60)
print("FILTERING DCM CELLS WITH CLEAR TRANSITIONS")
print("="*60)

dcm_mask = plot_df['assay'] == 'pDCM'
dcm_clear_mask = (dcm_mask & 
                  (plot_df['ttest_gmm_label_confident'].isin(cell_type_order)) &
                  (plot_df['ttest_gmm_label_confident'] == plot_df['leiden_annot_25']))

dcm_cells = plot_df[dcm_clear_mask].copy()

print(f"Total pDCM cells: {dcm_mask.sum()}")
print(f"DCM cells with CLEAR labels (GMM == leiden_annot_25): {len(dcm_cells)}")

# -----------------------------
# FILTER RNA CELLS
# -----------------------------
print("\n" + "="*60)
print("FILTERING RNA CELLS")
print("="*60)

rna_mask = plot_df['assay'] == 'pRNA'
rna_cells = plot_df[rna_mask & plot_df['leiden_annot_25'].isin(cell_type_order)].copy()
print(f"Total pRNA cells: {len(rna_cells)}")

# -----------------------------
# MATCH BY BATCH_PLATE_WELL (1:1)
# -----------------------------
print("\n" + "="*60)
print("MATCHING CELLS BY BATCH_PLATE_WELL")
print("="*60)

dcm_bpw = set(dcm_cells['batch_plate_well'])
rna_bpw = set(rna_cells['batch_plate_well'])
common_bpw = dcm_bpw.intersection(rna_bpw)

print(f"Common batch_plate_wells: {len(common_bpw)}")

dcm_common = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
rna_common = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]

dcm_counts = dcm_common['batch_plate_well'].value_counts()
rna_counts = rna_common['batch_plate_well'].value_counts()

dcm_single = set(dcm_counts[dcm_counts == 1].index)
rna_single = set(rna_counts[rna_counts == 1].index)
both_single = dcm_single.intersection(rna_single)

print(f"1:1 matched batch_plate_wells: {len(both_single)}")

# Create connections
connections = []
for bpw in both_single:
    dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
    rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
    
    if len(dcm_cell) == 1 and len(rna_cell) == 1:
        dcm_row = dcm_cell.iloc[0]
        rna_row = rna_cell.iloc[0]
        
        conn = {
            'batch_plate_well': bpw,
            'dcm_cell_type': dcm_row['ttest_gmm_label_confident'],
            'rna_cell_type': rna_row['leiden_annot_25'],
            'transition': f"{dcm_row['ttest_gmm_label_confident']} -> {rna_row['leiden_annot_25']}",
            'dcm_index': dcm_row['cell_index'],
            'rna_index': rna_row['cell_index']
        }
        
        # Add z-scores
        for col in ttest_zscore_cols:
            ct = col.replace('ttest_', '').replace('_zscore', '')
            conn[f'dcm_{ct}_zscore'] = dcm_row[col]
            conn[f'rna_{ct}_zscore'] = rna_row[col]
        
        # Add posteriors
        for col in ttest_posterior_cols:
            ct = col.replace('ttest_', '').replace('_posterior', '')
            conn[f'dcm_{ct}_posterior'] = dcm_row[col]
            conn[f'rna_{ct}_posterior'] = rna_row[col]
        
        connections.append(conn)

connections_df = pd.DataFrame(connections)
print(f"\nTotal connections: {len(connections_df)}")

# -----------------------------
# FILTER OUT STABLE CELLS (SAME CELL TYPE)
# -----------------------------
print("\n" + "="*60)
print("FILTERING OUT STABLE CELLS")
print("="*60)

connections_df['stable'] = connections_df['dcm_cell_type'] == connections_df['rna_cell_type']
connections_df = connections_df[~connections_df['stable']].copy()

# -----------------------------
# REMOVE SPECIFIED CELLS
# -----------------------------
print("\n" + "="*60)
print("REMOVING SPECIFIED CELLS")
print("="*60)

print(f"Connections before exclusion: {len(connections_df)}")
connections_df = connections_df[~connections_df['batch_plate_well'].isin(cells_to_exclude)].copy()
print(f"Connections after exclusion: {len(connections_df)}")

if len(connections_df) == 0:
    print("\nNo transitions found after filtering!")
else:
    # -----------------------------
    # SORT BY DCM CELL TYPE, THEN RNA CELL TYPE
    # -----------------------------
    print("\n" + "="*60)
    print("ORGANIZING CELLS BY DCM CELL TYPE")
    print("="*60)
    
    connections_df['dcm_cell_type'] = pd.Categorical(connections_df['dcm_cell_type'], 
                                                      categories=cell_type_order, 
                                                      ordered=True)
    connections_df['rna_cell_type'] = pd.Categorical(connections_df['rna_cell_type'], 
                                                      categories=cell_type_order, 
                                                      ordered=True)
    
    connections_df = connections_df.sort_values(['dcm_cell_type', 'rna_cell_type'])
    
    # -----------------------------
    # CREATE Z-SCORE HEATMAPS WITH COLOR ANNOTATIONS
    # -----------------------------
    print("\n" + "="*60)
    print("CREATING Z-SCORE HEATMAPS WITH COLOR ANNOTATIONS")
    print("="*60)
    
    # Prepare z-score matrices
    dcm_zscore_data = []
    rna_zscore_data = []
    
    for _, row in connections_df.iterrows():
        dcm_zscores = [row[f'dcm_{ct}_zscore'] for ct in cell_type_order]
        rna_zscores = [row[f'rna_{ct}_zscore'] for ct in cell_type_order]
        dcm_zscore_data.append(dcm_zscores)
        rna_zscore_data.append(rna_zscores)
    
    dcm_zscore_matrix = np.array(dcm_zscore_data).T
    rna_zscore_matrix = np.array(rna_zscore_data).T
    
    # Create color annotation arrays
    dcm_numeric = np.array([[cell_type_order.index(ct) for ct in connections_df['dcm_cell_type']]])
    rna_numeric = np.array([[cell_type_order.index(ct) for ct in connections_df['rna_cell_type']]])
    
    colors_list = [cell_type_colors[ct] for ct in cell_type_order]
    cmap_annot = ListedColormap(colors_list)
    
    # Create figure
    fig = plt.figure(figsize=(max(18, len(connections_df) * 0.2), 14))
    gs = fig.add_gridspec(4, 2, width_ratios=[1, 0.03], 
                         height_ratios=[0.3, 10, 0.3, 10],
                         hspace=0.05, wspace=0.05)
    
    vmin, vmax = -3, 3
    cmap = plt.cm.RdBu_r
    
    # ===== DCM ANNOTATION BAR =====
    ax_dcm_annot = fig.add_subplot(gs[0, 0])
    ax_dcm_annot.imshow(dcm_numeric, aspect='auto', cmap=cmap_annot,
                        vmin=0, vmax=len(cell_type_order)-1, interpolation='nearest')
    ax_dcm_annot.set_xticks([])
    ax_dcm_annot.set_yticks([])
    ax_dcm_annot.set_ylabel('DCM\nType', fontsize=9, fontweight='bold', rotation=0, 
                            ha='right', va='center')
    
    # ===== DCM Z-SCORE HEATMAP =====
    ax_dcm = fig.add_subplot(gs[1, 0])
    im_dcm = ax_dcm.imshow(dcm_zscore_matrix, aspect='auto', cmap=cmap, 
                           vmin=vmin, vmax=vmax, interpolation='nearest')
    
    ax_dcm.set_yticks(range(len(cell_type_order)))
    ax_dcm.set_yticklabels(cell_type_order, fontsize=10)
    ax_dcm.set_xticks([])
    ax_dcm.set_title('DCM Module Z-Scores', fontsize=12, fontweight='bold', pad=10)
    ax_dcm.set_ylabel('Module', fontsize=11, fontweight='bold')
    
    # Add dashed boxes around DCM cell type rows
    for i, cell in enumerate(connections_df.iterrows()):
        dcm_ct = cell[1]['dcm_cell_type']
        dcm_idx = cell_type_order.index(dcm_ct)
        rect = Rectangle((i-0.5, dcm_idx-0.5), 1, 1, 
                        fill=False, edgecolor='black', linewidth=1.5, 
                        linestyle='--', alpha=0.7)
        ax_dcm.add_patch(rect)
    
    ax_dcm_cbar = fig.add_subplot(gs[1, 1])
    cbar_dcm = fig.colorbar(im_dcm, cax=ax_dcm_cbar)
    cbar_dcm.set_label('Z-Score', fontsize=10, fontweight='bold')
    
    # ===== RNA ANNOTATION BAR =====
    ax_rna_annot = fig.add_subplot(gs[2, 0])
    ax_rna_annot.imshow(rna_numeric, aspect='auto', cmap=cmap_annot,
                        vmin=0, vmax=len(cell_type_order)-1, interpolation='nearest')
    ax_rna_annot.set_xticks([])
    ax_rna_annot.set_yticks([])
    ax_rna_annot.set_ylabel('RNA\nType', fontsize=9, fontweight='bold', rotation=0, 
                            ha='right', va='center')
    
    # ===== RNA Z-SCORE HEATMAP =====
    ax_rna = fig.add_subplot(gs[3, 0])
    im_rna = ax_rna.imshow(rna_zscore_matrix, aspect='auto', cmap=cmap, 
                           vmin=vmin, vmax=vmax, interpolation='nearest')
    
    ax_rna.set_yticks(range(len(cell_type_order)))
    ax_rna.set_yticklabels(cell_type_order, fontsize=10)
    ax_rna.set_xticks([])
    ax_rna.set_xlabel('Individual Cells (sorted by DCM type, then RNA type)', 
                     fontsize=11, fontweight='bold')
    ax_rna.set_title('RNA Module Z-Scores', fontsize=12, fontweight='bold', pad=10)
    ax_rna.set_ylabel('Module', fontsize=11, fontweight='bold')
    
    # Add dashed boxes around RNA cell type rows
    for i, cell in enumerate(connections_df.iterrows()):
        rna_ct = cell[1]['rna_cell_type']
        rna_idx = cell_type_order.index(rna_ct)
        rect = Rectangle((i-0.5, rna_idx-0.5), 1, 1, 
                        fill=False, edgecolor='black', linewidth=1.5, 
                        linestyle='--', alpha=0.7)
        ax_rna.add_patch(rect)
    
    ax_rna_cbar = fig.add_subplot(gs[3, 1])
    cbar_rna = fig.colorbar(im_rna, cax=ax_rna_cbar)
    cbar_rna.set_label('Z-Score', fontsize=10, fontweight='bold')
    
    # Add legend for cell types
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=cell_type_colors[ct], label=ct) 
                      for ct in cell_type_order]
    ax_rna.legend(handles=legend_elements, title='Cell Type', 
                 bbox_to_anchor=(1.15, 0.5), loc='center left', frameon=True, fontsize=9)
    
    fig.suptitle(f'Module Z-Scores for Changed Cells\n'
                f'(n = {len(connections_df)} cells, DCM: GMM==leiden_annot_25, Stable cells excluded)', 
                fontsize=14, fontweight='bold', y=0.98)
    
    plt.tight_layout()
    save_path = os.path.join(score_dir, 'heatmap_zscore_changed_cells_annotated.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"✅ Saved: {save_path}")
    
    # -----------------------------
    # CREATE POSTERIOR PROBABILITY HEATMAPS WITH COLOR ANNOTATIONS
    # -----------------------------
    print("\n" + "="*60)
    print("CREATING POSTERIOR PROBABILITY HEATMAPS WITH COLOR ANNOTATIONS")
    print("="*60)
    
    # Prepare posterior matrices
    dcm_posterior_data = []
    rna_posterior_data = []
    
    for _, row in connections_df.iterrows():
        dcm_posteriors = [row[f'dcm_{ct}_posterior'] for ct in cell_type_order]
        rna_posteriors = [row[f'rna_{ct}_posterior'] for ct in cell_type_order]
        dcm_posterior_data.append(dcm_posteriors)
        rna_posterior_data.append(rna_posteriors)
    
    dcm_posterior_matrix = np.array(dcm_posterior_data).T
    rna_posterior_matrix = np.array(rna_posterior_data).T
    
    # Check actual posterior ranges
    print(f"\nPosterior ranges:")
    print(f"  DCM min: {dcm_posterior_matrix.min():.4f}, max: {dcm_posterior_matrix.max():.4f}")
    print(f"  RNA min: {rna_posterior_matrix.min():.4f}, max: {rna_posterior_matrix.max():.4f}")
    
    # Create figure
    fig = plt.figure(figsize=(max(18, len(connections_df) * 0.2), 14))
    gs = fig.add_gridspec(4, 2, width_ratios=[1, 0.03], 
                         height_ratios=[0.3, 10, 0.3, 10],
                         hspace=0.05, wspace=0.05)
    
    # Blue-to-red gradient for posterior probabilities
    cmap_posterior = plt.cm.RdBu_r
    posterior_vmin = 0.9
    posterior_vmax = 1.0
    
    # ===== DCM ANNOTATION BAR =====
    ax_dcm_annot = fig.add_subplot(gs[0, 0])
    ax_dcm_annot.imshow(dcm_numeric, aspect='auto', cmap=cmap_annot,
                        vmin=0, vmax=len(cell_type_order)-1, interpolation='nearest')
    ax_dcm_annot.set_xticks([])
    ax_dcm_annot.set_yticks([])
    ax_dcm_annot.set_ylabel('DCM\nType', fontsize=9, fontweight='bold', rotation=0, 
                            ha='right', va='center')
    
    # ===== DCM POSTERIOR HEATMAP =====
    ax_dcm_post = fig.add_subplot(gs[1, 0])
    im_dcm_post = ax_dcm_post.imshow(dcm_posterior_matrix, aspect='auto', cmap=cmap_posterior, 
                                     vmin=posterior_vmin, vmax=posterior_vmax, interpolation='nearest')
    
    ax_dcm_post.set_yticks(range(len(cell_type_order)))
    ax_dcm_post.set_yticklabels(cell_type_order, fontsize=10)
    ax_dcm_post.set_xticks([])
    ax_dcm_post.set_title('DCM Posterior Probabilities', fontsize=12, fontweight='bold', pad=10)
    ax_dcm_post.set_ylabel('Cell Type', fontsize=11, fontweight='bold')
    
    # Add dashed boxes around DCM cell type rows
    for i, cell in enumerate(connections_df.iterrows()):
        dcm_ct = cell[1]['dcm_cell_type']
        dcm_idx = cell_type_order.index(dcm_ct)
        rect = Rectangle((i-0.5, dcm_idx-0.5), 1, 1, 
                        fill=False, edgecolor='black', linewidth=1.5, 
                        linestyle='--', alpha=0.7)
        ax_dcm_post.add_patch(rect)
    
    ax_dcm_post_cbar = fig.add_subplot(gs[1, 1])
    cbar_dcm_post = fig.colorbar(im_dcm_post, cax=ax_dcm_post_cbar)
    cbar_dcm_post.set_label('Posterior', fontsize=10, fontweight='bold')
    
    # ===== RNA ANNOTATION BAR =====
    ax_rna_annot = fig.add_subplot(gs[2, 0])
    ax_rna_annot.imshow(rna_numeric, aspect='auto', cmap=cmap_annot,
                        vmin=0, vmax=len(cell_type_order)-1, interpolation='nearest')
    ax_rna_annot.set_xticks([])
    ax_rna_annot.set_yticks([])
    ax_rna_annot.set_ylabel('RNA\nType', fontsize=9, fontweight='bold', rotation=0, 
                            ha='right', va='center')
    
    # ===== RNA POSTERIOR HEATMAP =====
    ax_rna_post = fig.add_subplot(gs[3, 0])
    im_rna_post = ax_rna_post.imshow(rna_posterior_matrix, aspect='auto', cmap=cmap_posterior, 
                                     vmin=posterior_vmin, vmax=posterior_vmax, interpolation='nearest')
    
    ax_rna_post.set_yticks(range(len(cell_type_order)))
    ax_rna_post.set_yticklabels(cell_type_order, fontsize=10)
    ax_rna_post.set_xticks([])
    ax_rna_post.set_xlabel('Individual Cells (sorted by DCM type, then RNA type)', 
                          fontsize=11, fontweight='bold')
    ax_rna_post.set_title('RNA Posterior Probabilities', fontsize=12, fontweight='bold', pad=10)
    ax_rna_post.set_ylabel('Cell Type', fontsize=11, fontweight='bold')
    
    # Add dashed boxes around RNA cell type rows
    for i, cell in enumerate(connections_df.iterrows()):
        rna_ct = cell[1]['rna_cell_type']
        rna_idx = cell_type_order.index(rna_ct)
        rect = Rectangle((i-0.5, rna_idx-0.5), 1, 1, 
                        fill=False, edgecolor='black', linewidth=1.5, 
                        linestyle='--', alpha=0.7)
        ax_rna_post.add_patch(rect)
    
    ax_rna_post_cbar = fig.add_subplot(gs[3, 1])
    cbar_rna_post = fig.colorbar(im_rna_post, cax=ax_rna_post_cbar)
    cbar_rna_post.set_label('Posterior', fontsize=10, fontweight='bold')
    
    # Add legend
    legend_elements = [Patch(facecolor=cell_type_colors[ct], label=ct) 
                      for ct in cell_type_order]
    ax_rna_post.legend(handles=legend_elements, title='Cell Type', 
                      bbox_to_anchor=(1.15, 0.5), loc='center left', frameon=True, fontsize=9)
    
    fig.suptitle(f'Posterior Probabilities for Changed Cells (Range: 0.9-1.0)\n'
                f'(n = {len(connections_df)} cells, DCM: GMM==leiden_annot_25, Stable cells excluded)', 
                fontsize=14, fontweight='bold', y=0.98)
    
    plt.tight_layout()
    save_path = os.path.join(score_dir, 'heatmap_posterior_changed_cells_annotated.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"✅ Saved: {save_path}")
    
    # -----------------------------
    # SAVE DATA
    # -----------------------------
    print("\n" + "="*60)
    print("SAVING DATA")
    print("="*60)
    
    connections_df.to_csv(os.path.join(score_dir, 'changed_cells_filtered_annotated.csv'), index=False)
    print(f"✅ Saved: {os.path.join(score_dir, 'changed_cells_filtered_annotated.csv')}")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
### Figure S5 plots###

In [ ]:
stats_RNA = pd.read_csv("/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/gDNA/Stats_intestine_batch1_batch2_RNA.csv")
stats_gDNA = pd.read_csv("/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/gDNA/Stats_intestine_batch1_batch2_gDNA.csv")
stats_RNA_gDNA = pd.read_csv("/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/gDNA/Stats_intestine_batch1_batch2_gDNA_RNA.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/'

# Colors for Dox conditions
dox_colors = {
    'Minus': '#A50F15',
    '48hDox': '#377EB8',
    '48hDox-72hChase': '#4DAF4A'
}

# Set style
plt.style.use('seaborn-v0_8-whitegrid')

# -----------------------------
# FILTER DATA
# -----------------------------
print("="*60)
print("FILTERING DATA")
print("="*60)

threshold = 0.5

print(f"Original shape: {stats_gDNA.shape}")
print(f"Cells with DCM/CpG > {threshold}: {(stats_gDNA['DCM/CpG'] > threshold).sum()}")

# Filter
plot_data = stats_gDNA[stats_gDNA['DCM/CpG'] <= threshold].copy()

print(f"After filtering (DCM/CpG <= {threshold}): {plot_data.shape}")
print(f"Cells removed: {len(stats_gDNA) - len(plot_data)} ({(len(stats_gDNA) - len(plot_data))/len(stats_gDNA)*100:.1f}%)")

# Orders
dox_order = ['Minus', '48hDox', '48hDox-72hChase']
dox_order = [d for d in dox_order if d in plot_data['Dox'].unique()]

# -----------------------------
# VERSION 1: SIMPLE BOXPLOT BY DOX (NO BATCH)
# -----------------------------
print("\n" + "="*60)
print("VERSION 1: BOXPLOT BY DOX (NO BATCH)")
print("="*60)

fig, ax = plt.subplots(figsize=(8, 6))

sns.boxplot(
    data=plot_data,
    x='Dox',
    y='DCM/CpG',
    order=dox_order,
    palette=dox_colors,
    ax=ax,
    width=0.5,
    linewidth=1.5,
    fliersize=3,
    flierprops={'alpha': 0.4},
    boxprops={'alpha': 0.8},
    medianprops={'color': 'black', 'linewidth': 2}
)

ax.set_xlabel('Dox Condition', fontsize=12, fontweight='bold')
ax.set_ylabel('DCM/CpG', fontsize=12, fontweight='bold')
ax.set_title(f'DCM/CpG by Dox Condition\n(filtered: DCM/CpG ≤ {threshold})', fontsize=14, fontweight='bold')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add sample sizes
for i, dox in enumerate(dox_order):
    n = len(plot_data[plot_data['Dox'] == dox])
    ax.text(i, ax.get_ylim()[0] - 0.01, f'n={n}', ha='center', va='top', fontsize=10)

plt.tight_layout()

save_path = f'{score_dir}/boxplot_DCM_CpG_by_Dox_filtered.pdf'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print(f"Saved: {save_path}")

# -----------------------------
# VERSION 2: BOXPLOT BY DOX WITH BATCH SEPARATION
# -----------------------------
print("\n" + "="*60)
print("VERSION 2: BOXPLOT BY DOX WITH BATCH SEPARATION")
print("="*60)

fig, ax = plt.subplots(figsize=(12, 6))

sns.boxplot(
    data=plot_data,
    x='Dox',
    y='DCM/CpG',
    hue='Batch',
    order=dox_order,
    ax=ax,
    width=0.6,
    linewidth=1.5,
    fliersize=3,
    flierprops={'alpha': 0.4},
    boxprops={'alpha': 0.8},
    medianprops={'color': 'black', 'linewidth': 2}
)

ax.set_xlabel('Dox Condition', fontsize=12, fontweight='bold')
ax.set_ylabel('DCM/CpG', fontsize=12, fontweight='bold')
ax.set_title(f'DCM/CpG by Dox Condition and Batch\n(filtered: DCM/CpG ≤ {threshold})', fontsize=14, fontweight='bold')
ax.legend(title='Batch', loc='upper right', fontsize=10)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

save_path = f'{score_dir}/boxplot_DCM_CpG_by_Dox_Batch_filtered.pdf'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print(f"Saved: {save_path}")

# -----------------------------
# SUMMARY STATISTICS
# -----------------------------
print("\n" + "="*60)
print("SUMMARY STATISTICS (FILTERED DATA)")
print("="*60)

print("\nBy Dox only:")
summary_dox = plot_data.groupby('Dox')['DCM/CpG'].agg(['count', 'median', 'mean', 'std']).round(4)
print(summary_dox)

print("\nBy Dox and Batch:")
summary_batch = plot_data.groupby(['Dox', 'Batch'])['DCM/CpG'].agg(['count', 'median', 'mean', 'std']).round(4)
print(summary_batch)

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/'

# Assuming stats_gDNA is already loaded as a pandas DataFrame
# If not, load it:
# stats_gDNA = pd.read_csv('path_to_your_file.csv')

# -----------------------------
# PREPARE DATA
# -----------------------------
print("="*60)
print("PREPARING DATA")
print("="*60)

# Check columns
print(f"Columns: {stats_gDNA.columns.tolist()}")
print(f"Shape: {stats_gDNA.shape}")

# Check Dox values
print(f"\nDox values: {stats_gDNA['Dox'].unique()}")

# Filter out zeros or negative values (can't log transform)
plot_data = stats_gDNA.copy()
plot_data = plot_data[(plot_data['CpG_unique'] > 0) & (plot_data['Aligned'] > 0)]

print(f"\nCells after filtering zeros: {len(plot_data)}")

# -----------------------------
# DEFINE COLORS
# -----------------------------
dox_colors = {
    'Minus': '#A50F15',    # Green
    '48hDox': '#377EB8',   # Red
    '48hDox-72hChase' : '#4DAF4A'
    # Add more if needed
}

# Get unique Dox values and assign colors if not predefined
unique_dox = plot_data['Dox'].unique()
for dox in unique_dox:
    if dox not in dox_colors:
        # Assign a color from a colormap
        cmap = plt.cm.tab10
        idx = list(unique_dox).index(dox)
        dox_colors[dox] = cmap(idx)

print(f"Dox colors: {dox_colors}")

# -----------------------------
# CREATE SCATTERPLOT
# -----------------------------
print("\n" + "="*60)
print("CREATING SCATTERPLOT")
print("="*60)

fig, ax = plt.subplots(figsize=(10, 8))

# Plot each Dox condition
for dox in unique_dox:
    subset = plot_data[plot_data['Dox'] == dox]
    
    ax.scatter(
        np.log10(subset['Aligned']),
        np.log10(subset['CpG_unique']),
        c=dox_colors[dox],
        label=f'{dox} (n={len(subset)})',
        alpha=0.6,
        s=30,
        edgecolors='none'
    )

# Add diagonal line (y = x)
min_val = min(np.log10(plot_data['Aligned'].min()), np.log10(plot_data['CpG_unique'].min()))
max_val = max(np.log10(plot_data['Aligned'].max()), np.log10(plot_data['CpG_unique'].max()))
ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='y = x')

# Labels and title
ax.set_xlabel('log10(Aligned)', fontsize=12, fontweight='bold')
ax.set_ylabel('log10(CpG_unique)', fontsize=12, fontweight='bold')
ax.set_title('CpG Unique vs Aligned Reads by Dox Condition', fontsize=14, fontweight='bold')

# Legend
ax.legend(loc='lower right', fontsize=10)

# Grid
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()

# Save
save_path = f'{score_dir}/scatterplot_CpG_unique_vs_Aligned_by_Dox.pdf'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print(f"Saved: {save_path}")

# -----------------------------
# SUMMARY STATISTICS
# -----------------------------
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

for dox in unique_dox:
    subset = plot_data[plot_data['Dox'] == dox]
    print(f"\n{dox}:")
    print(f"  n = {len(subset)}")
    print(f"  Aligned: median = {subset['Aligned'].median():.0f}, mean = {subset['Aligned'].mean():.0f}")
    print(f"  CpG_unique: median = {subset['CpG_unique'].median():.0f}, mean = {subset['CpG_unique'].mean():.0f}")
    print(f"  Ratio (CpG_unique/Aligned): median = {(subset['CpG_unique']/subset['Aligned']).median():.4f}")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/'

# Assuming stats_RNA is already loaded as a pandas DataFrame
# If not, load it:
# stats_RNA = pd.read_csv('path_to_your_file.csv')

# -----------------------------
# PREPARE DATA
# -----------------------------
print("="*60)
print("PREPARING DATA")
print("="*60)

# Check columns
print(f"Columns: {stats_RNA.columns.tolist()}")
print(f"Shape: {stats_RNA.shape}")

# Check Dox values
print(f"\nDox values: {stats_RNA['Dox'].unique()}")

# Filter out zeros or negative values (can't log transform)
plot_data = stats_RNA.copy()
plot_data = plot_data[(plot_data['UMI'] > 0) & (plot_data['PE_aligned'] > 0)]

print(f"\nCells after filtering zeros: {len(plot_data)}")

# -----------------------------
# DEFINE COLORS
# -----------------------------
dox_colors = {
    'Minus': '#A50F15',    # Green
    '48hDox': '#377EB8',   # Red
    '48hDox-72hChase' : '#4DAF4A'
    # Add more if needed
}

# Get unique Dox values and assign colors if not predefined
unique_dox = plot_data['Dox'].unique()
for dox in unique_dox:
    if dox not in dox_colors:
        # Assign a color from a colormap
        cmap = plt.cm.tab10
        idx = list(unique_dox).index(dox)
        dox_colors[dox] = cmap(idx)

print(f"Dox colors: {dox_colors}")

# -----------------------------
# CREATE SCATTERPLOT
# -----------------------------
print("\n" + "="*60)
print("CREATING SCATTERPLOT")
print("="*60)

fig, ax = plt.subplots(figsize=(10, 8))

# Plot each Dox condition
for dox in unique_dox:
    subset = plot_data[plot_data['Dox'] == dox]
    
    ax.scatter(
        np.log10(subset['PE_aligned']),
        np.log10(subset['UMI']),
        c=dox_colors[dox],
        label=f'{dox} (n={len(subset)})',
        alpha=0.6,
        s=30,
        edgecolors='none'
    )

# Add diagonal line (y = x)
min_val = min(np.log10(plot_data['PE_aligned'].min()), np.log10(plot_data['UMI'].min()))
max_val = max(np.log10(plot_data['PE_aligned'].max()), np.log10(plot_data['UMI'].max()))
ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='y = x')

# Labels and title
ax.set_xlabel('log10(PE_aligned)', fontsize=12, fontweight='bold')
ax.set_ylabel('log10(RNA UMI)', fontsize=12, fontweight='bold')
ax.set_title('RNA UMI vs PE_aligned Reads by Dox Condition', fontsize=14, fontweight='bold')

# Legend
ax.legend(loc='lower right', fontsize=10)

# Grid
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()

# Save
save_path = f'{score_dir}/scatterplot_RNA_UMI_vs_Aligned_by_Dox.pdf'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print(f"Saved: {save_path}")

# -----------------------------
# SUMMARY STATISTICS
# -----------------------------
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

for dox in unique_dox:
    subset = plot_data[plot_data['Dox'] == dox]
    print(f"\n{dox}:")
    print(f"  n = {len(subset)}")
    print(f"  Aligned: median = {subset['PE_aligned'].median():.0f}, mean = {subset['PE_aligned'].mean():.0f}")
    print(f"  UMI: median = {subset['UMI'].median():.0f}, mean = {subset['UMI'].mean():.0f}")
    print(f"  Ratio (UMI/Aligned): median = {(subset['UMI']/subset['PE_aligned']).median():.4f}")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/'

# Colors for Dox conditions
dox_colors = {
    'Minus': '#A50F15',
    '48hDox': '#377EB8',
    '48hDox-72hChase': '#4DAF4A'
}

# Set style (R-like)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("paper", font_scale=1.2)

# -----------------------------
# PREPARE DATA
# -----------------------------
print("="*60)
print("PREPARING DATA")
print("="*60)

# Filter cells with >= 200 genes
plot_data = stats_RNA_gDNA[stats_RNA_gDNA['Genes'] >= 200].copy()

print(f"After filtering (Genes >= 200): {plot_data.shape}")

# Reshape data
genes_df = plot_data[['Dox', 'Biotype', 'Genes']].copy()
genes_df['Metric'] = 'Genes'
genes_df['Value'] = genes_df['Genes']
genes_df['Category'] = 'Genes ' + genes_df['Biotype']

umi_df = plot_data[['Dox', 'Biotype', 'UMI']].copy()
umi_df['Metric'] = 'UMI'
umi_df['Value'] = umi_df['UMI']
umi_df['Category'] = 'UMI ' + umi_df['Biotype']

combined_df = pd.concat([genes_df[['Dox', 'Biotype', 'Metric', 'Value', 'Category']], 
                         umi_df[['Dox', 'Biotype', 'Metric', 'Value', 'Category']]], 
                        ignore_index=True)

combined_df = combined_df[combined_df['Value'] > 0]

# Orders
dox_order = ['Minus', '48hDox', '48hDox-72hChase']
dox_order = [d for d in dox_order if d in combined_df['Dox'].unique()]

biotypes = plot_data['Biotype'].unique()
category_order = []
for metric in ['Genes', 'UMI']:
    for biotype in biotypes:
        cat = f'{metric} {biotype}'
        if cat in combined_df['Category'].unique():
            category_order.append(cat)

# -----------------------------
# VERSION 1: R-STYLE VIOLIN + BOXPLOT OVERLAY
# -----------------------------
print("\n" + "="*60)
print("VERSION 1: R-STYLE VIOLIN + BOXPLOT")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, metric in enumerate(['Genes', 'UMI']):
    ax = axes[idx]
    metric_df = combined_df[combined_df['Metric'] == metric]
    metric_categories = [c for c in category_order if metric in c]
    
    # Violin plot (transparent, wide)
    sns.violinplot(
        data=metric_df,
        x='Category',
        y='Value',
        hue='Dox',
        hue_order=dox_order,
        palette=dox_colors,
        order=metric_categories,
        ax=ax,
        inner=None,
        cut=0,
        linewidth=0,
        alpha=0.3,
        saturation=1
    )
    
    # Boxplot overlay (narrow)
    sns.boxplot(
        data=metric_df,
        x='Category',
        y='Value',
        hue='Dox',
        hue_order=dox_order,
        palette=dox_colors,
        order=metric_categories,
        ax=ax,
        width=0.15,
        linewidth=1.5,
        fliersize=2,
        flierprops={'alpha': 0.5}
    )
    
    ax.set_yscale('log')
    ax.set_xlabel('', fontsize=12)
    ax.set_ylabel(f'Number of {metric} (log10)', fontsize=12, fontweight='bold')
    ax.set_title(metric, fontsize=14, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    
    # Remove duplicate legend
    handles, labels = ax.get_legend_handles_labels()
    n_groups = len(dox_order)
    ax.legend(handles[:n_groups], labels[:n_groups], title='Dox', 
             loc='upper right', fontsize=9, framealpha=0.9)

plt.suptitle('Genes and UMI by Biotype and Dox Condition (Genes ≥ 200)', 
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

save_path = f'{score_dir}/violin_boxplot_Genes_UMI_Rstyle.pdf'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print(f"Saved: {save_path}")

# -----------------------------
# VERSION 2: CLEAN BOXPLOT (GGPLOT2 STYLE)
# -----------------------------
print("\n" + "="*60)
print("VERSION 2: CLEAN BOXPLOT (GGPLOT2 STYLE)")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, metric in enumerate(['Genes', 'UMI']):
    ax = axes[idx]
    metric_df = combined_df[combined_df['Metric'] == metric]
    metric_categories = [c for c in category_order if metric in c]
    
    sns.boxplot(
        data=metric_df,
        x='Category',
        y='Value',
        hue='Dox',
        hue_order=dox_order,
        palette=dox_colors,
        order=metric_categories,
        ax=ax,
        width=0.6,
        linewidth=1.5,
        fliersize=3,
        flierprops={'alpha': 0.4, 'markeredgecolor': 'gray'},
        boxprops={'alpha': 0.8},
        medianprops={'color': 'black', 'linewidth': 2}
    )
    
    ax.set_yscale('log')
    ax.set_xlabel('', fontsize=12)
    ax.set_ylabel(f'Number of {metric} (log10)', fontsize=12, fontweight='bold')
    ax.set_title(metric, fontsize=14, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(title='Dox', loc='upper right', fontsize=9, framealpha=0.9)
    
    # Clean up spines (R/ggplot2 style)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Genes and UMI by Biotype and Dox Condition (Genes ≥ 200)', 
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

save_path = f'{score_dir}/boxplot_Genes_UMI_ggplot_style.pdf'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print(f"Saved: {save_path}")

# -----------------------------
# VERSION 3: BOXPLOT WITH JITTERED POINTS (MOST R-LIKE)
# -----------------------------
print("\n" + "="*60)
print("VERSION 3: BOXPLOT WITH JITTERED POINTS")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, metric in enumerate(['Genes', 'UMI']):
    ax = axes[idx]
    metric_df = combined_df[combined_df['Metric'] == metric]
    metric_categories = [c for c in category_order if metric in c]
    
    # Boxplot
    sns.boxplot(
        data=metric_df,
        x='Category',
        y='Value',
        hue='Dox',
        hue_order=dox_order,
        palette=dox_colors,
        order=metric_categories,
        ax=ax,
        width=0.6,
        linewidth=1.5,
        fliersize=0,  # Hide outliers, will show with stripplot
        boxprops={'alpha': 0.7},
        medianprops={'color': 'black', 'linewidth': 2}
    )
    
    # Jittered points (sample to avoid overplotting)
    sample_df = metric_df.groupby(['Category', 'Dox']).apply(
        lambda x: x.sample(min(50, len(x)), random_state=42)
    ).reset_index(drop=True)
    
    sns.stripplot(
        data=sample_df,
        x='Category',
        y='Value',
        hue='Dox',
        hue_order=dox_order,
        palette=dox_colors,
        order=metric_categories,
        ax=ax,
        dodge=True,
        size=2,
        alpha=0.4,
        jitter=0.15
    )
    
    ax.set_yscale('log')
    ax.set_xlabel('', fontsize=12)
    ax.set_ylabel(f'Number of {metric} (log10)', fontsize=12, fontweight='bold')
    ax.set_title(metric, fontsize=14, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    
    # Fix legend (remove duplicates)
    handles, labels = ax.get_legend_handles_labels()
    n_groups = len(dox_order)
    ax.legend(handles[:n_groups], labels[:n_groups], title='Dox', 
             loc='upper right', fontsize=9, framealpha=0.9)
    
    # Clean spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Genes and UMI by Biotype and Dox Condition (Genes ≥ 200)', 
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

save_path = f'{score_dir}/boxplot_jitter_Genes_UMI_Rstyle.pdf'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print(f"Saved: {save_path}")

# -----------------------------
# VERSION 4: ALL IN ONE PLOT
# -----------------------------
print("\n" + "="*60)
print("VERSION 4: ALL IN ONE PLOT")
print("="*60)

fig, ax = plt.subplots(figsize=(16, 7))

sns.boxplot(
    data=combined_df,
    x='Category',
    y='Value',
    hue='Dox',
    hue_order=dox_order,
    palette=dox_colors,
    order=category_order,
    ax=ax,
    width=0.6,
    linewidth=1.5,
    fliersize=2,
    flierprops={'alpha': 0.3},
    boxprops={'alpha': 0.8},
    medianprops={'color': 'black', 'linewidth': 2}
)

ax.set_yscale('log')

# Add separator line between Genes and UMI
n_biotypes = len(biotypes)
ax.axvline(x=n_biotypes - 0.5, color='gray', linestyle='--', linewidth=1.5, alpha=0.7)

# Section labels
ylim = ax.get_ylim()
ax.text(n_biotypes/2 - 0.5, ylim[1] * 1.5, 'GENES', ha='center', va='bottom', 
       fontsize=14, fontweight='bold', style='italic')
ax.text(n_biotypes + n_biotypes/2 - 0.5, ylim[1] * 1.5, 'UMI', ha='center', va='bottom', 
       fontsize=14, fontweight='bold', style='italic')

ax.set_xlabel('', fontsize=12)
ax.set_ylabel('Count (log10 scale)', fontsize=12, fontweight='bold')
ax.set_title('Genes and UMI by Biotype and Dox Condition (Genes ≥ 200)', 
            fontsize=14, fontweight='bold')
ax.legend(title='Dox', loc='upper right', fontsize=10, framealpha=0.9)
ax.tick_params(axis='x', rotation=45)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

save_path = f'{score_dir}/boxplot_combined_all_Rstyle.pdf'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print(f"Saved: {save_path}")

print("\n" + "="*60)
print("DONE!")
print("="*60)

# PlooootSSSSS

In [ ]:
import scanpy as sc 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import pandas as pd

# hybrid analysis

In [ ]:
adata_combined.obs.groupby('assay')['is_multipotent'].value_counts()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# Get pDCM and pRNA obs separately
ttest_cols = [col for col in adata_combined.obs.columns 
              if 'ttest_' in col and '_zscore' in col and 'max' not in col.lower()]
print(f"Found z-score columns: {ttest_cols}")

dcm_obs = adata_combined.obs[adata_combined.obs['assay'] == 'pDCM'][
    ['batch_plate_well', 'is_multipotent', 'significant_celltypes'] + ttest_cols].copy()
rna_obs = adata_combined.obs[adata_combined.obs['assay'] == 'pRNA'][
    ['batch_plate_well', 'is_multipotent', 'significant_celltypes'] + ttest_cols].copy()

# Rename columns to distinguish
dcm_obs = dcm_obs.rename(columns={
    'is_multipotent': 'dcm_is_multipotent',
    'significant_celltypes': 'dcm_significant_celltypes',
    **{col: f'dcm_{col}' for col in ttest_cols}
})
rna_obs = rna_obs.rename(columns={
    'is_multipotent': 'rna_is_multipotent',
    'significant_celltypes': 'rna_significant_celltypes',
    **{col: f'rna_{col}' for col in ttest_cols}
})

# Merge on batch_plate_well
merged = dcm_obs.merge(rna_obs, on='batch_plate_well', how='inner')
print(f"Total matched cells: {len(merged)}")

# Find cells that are multipotent in BOTH assays
both_multipotent = merged[
    (merged['dcm_is_multipotent'] == True) &
    (merged['rna_is_multipotent'] == True)
].copy()

print(f"Cells multipotent in BOTH pDCM and pRNA: {len(both_multipotent)}")
print("\nSample:")
print(both_multipotent[['batch_plate_well',
                          'dcm_is_multipotent', 'dcm_significant_celltypes',
                          'rna_is_multipotent', 'rna_significant_celltypes']].head(20))

# Summary of cell type combinations
print("\nDCM significant cell types in multipotent cells:")
print(both_multipotent['dcm_significant_celltypes'].value_counts())
print("\nRNA significant cell types in multipotent cells:")
print(both_multipotent['rna_significant_celltypes'].value_counts())

# Crosstab of DCM vs RNA cell types for multipotent cells
print("\nCrosstab DCM vs RNA significant cell types (multipotent in both):")
print(pd.crosstab(both_multipotent['dcm_significant_celltypes'],
                  both_multipotent['rna_significant_celltypes']))

################################################################################
# HEATMAP OF Z-SCORES FOR MULTIPOTENT CELLS
################################################################################

if len(both_multipotent) > 0:

    # Extract cell type names from ttest columns
    cell_types = [col.replace('ttest_', '').replace('_zscore', '') for col in ttest_cols]

    # Build DCM and RNA z-score matrices
    dcm_zcols = [f'dcm_{col}' for col in ttest_cols]
    rna_zcols = [f'rna_{col}' for col in ttest_cols]

    # Sort cells by DCM significant celltypes for cleaner visualization
    both_multipotent_sorted = both_multipotent.sort_values(
        by=['dcm_significant_celltypes', 'rna_significant_celltypes']
    ).copy()

    dcm_matrix = both_multipotent_sorted[dcm_zcols].values.T
    rna_matrix = both_multipotent_sorted[rna_zcols].values.T

    # Color scale centered at 1.65
    vcenter = 1.65
    vmin = vcenter - 3
    vmax = vcenter + 3
    norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
    cmap = plt.cm.RdBu_r

    # Cell type boundary lines
    dcm_groups = both_multipotent_sorted['dcm_significant_celltypes'].values
    boundaries = np.where(np.array(dcm_groups[:-1]) != np.array(dcm_groups[1:]))[0] + 1

    fig = plt.figure(figsize=(max(10, len(both_multipotent_sorted) * 0.25), 10))
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 0.03], height_ratios=[1, 1],
                          hspace=0.35, wspace=0.05)

    # --- DCM heatmap ---
    ax_dcm = fig.add_subplot(gs[0, 0])
    im_dcm = ax_dcm.imshow(dcm_matrix, aspect='auto', cmap=cmap, norm=norm)

    for b in boundaries:
        ax_dcm.axvline(x=b - 0.5, color='white', linewidth=2)

    ax_dcm.set_yticks(range(len(cell_types)))
    ax_dcm.set_yticklabels(cell_types, fontsize=10)
    ax_dcm.set_xticks([])
    ax_dcm.set_title('DCM Module Z-Scores\n(multipotent in both assays)',
                     fontsize=12, fontweight='bold')
    ax_dcm.set_ylabel('Module', fontsize=11, fontweight='bold')

    # Add DCM cell type labels at top
    prev = 0
    for b in list(boundaries) + [len(dcm_groups)]:
        mid = (prev + b) / 2
        label = dcm_groups[prev]
        ax_dcm.text(mid, -0.8, label, ha='center', va='bottom', fontsize=8,
                    fontweight='bold', rotation=45)
        prev = b

    ax_dcm_cbar = fig.add_subplot(gs[0, 1])
    cbar_dcm = fig.colorbar(im_dcm, cax=ax_dcm_cbar)
    cbar_dcm.set_label('Z-Score', fontsize=9)
    cbar_dcm.ax.axhline(y=vcenter, color='black', linewidth=1, linestyle='--')

    # --- RNA heatmap ---
    ax_rna = fig.add_subplot(gs[1, 0])
    im_rna = ax_rna.imshow(rna_matrix, aspect='auto', cmap=cmap, norm=norm)
    rna_groups = both_multipotent_sorted['rna_significant_celltypes'].values
    rna_boundaries = np.where(np.array(rna_groups[:-1]) != np.array(rna_groups[1:]))[0] + 1

    prev = 0
    for b in list(rna_boundaries) + [len(rna_groups)]:
        mid = (prev + b) / 2
        label = rna_groups[prev]
        ax_rna.text(mid, -0.8, label, ha='center', va='bottom', fontsize=8,
                    fontweight='bold', rotation=45)
        prev = b

    ax_rna.set_yticks(range(len(cell_types)))
    ax_rna.set_yticklabels(cell_types, fontsize=10)
    ax_rna.set_xticks(range(len(both_multipotent_sorted)))
    ax_rna.set_xticklabels(both_multipotent_sorted['batch_plate_well'].values, 
                            rotation=90, fontsize=7, ha='center')
    ax_rna.set_xlabel('Cells (sorted by DCM significant celltypes)', fontsize=11, fontweight='bold')
    ax_rna.set_title('RNA Module Z-Scores\n(multipotent in both assays)',
                     fontsize=12, fontweight='bold')
    ax_rna.set_ylabel('Module', fontsize=11, fontweight='bold')

    ax_rna_cbar = fig.add_subplot(gs[1, 1])
    cbar_rna = fig.colorbar(im_rna, cax=ax_rna_cbar)
    cbar_rna.set_label('Z-Score', fontsize=9)
    cbar_rna.ax.axhline(y=vcenter, color='black', linewidth=1, linestyle='--')

    fig.suptitle(f'Z-Scores for Multipotent Cells in Both pDCM and pRNA\n'
                 f'(n = {len(both_multipotent_sorted)} cells | Z-score center: {vcenter})',
                 fontsize=13, fontweight='bold', y=1.01)

    plt.tight_layout()
    plt.savefig('multipotent_zscore_heatmap.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print("✅ Saved: multipotent_zscore_heatmap.pdf")

else:
    print("No multipotent cells found in both assays, skipping heatmap")

In [ ]:
either_multipotent_sorted

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# Get pDCM and pRNA obs separately - include all cells
dcm_obs = adata_combined.obs[adata_combined.obs['assay'] == 'pDCM'][
    ['batch_plate_well', 'is_multipotent', 'significant_celltypes'] + ttest_cols].copy()
rna_obs = adata_combined.obs[adata_combined.obs['assay'] == 'pRNA'][
    ['batch_plate_well', 'is_multipotent', 'significant_celltypes'] + ttest_cols].copy()

# Rename columns to distinguish
dcm_obs = dcm_obs.rename(columns={
    'is_multipotent': 'dcm_is_multipotent',
    'significant_celltypes': 'dcm_significant_celltypes',
    **{col: f'dcm_{col}' for col in ttest_cols}
})
rna_obs = rna_obs.rename(columns={
    'is_multipotent': 'rna_is_multipotent',
    'significant_celltypes': 'rna_significant_celltypes',
    **{col: f'rna_{col}' for col in ttest_cols}
})

# Merge on batch_plate_well
merged = dcm_obs.merge(rna_obs, on='batch_plate_well', how='inner')
print(f"Total matched cells: {len(merged)}")

# Find cells that are multipotent in EITHER pDCM OR pRNA
either_multipotent = merged[
    (merged['dcm_is_multipotent'] == True) |
    (merged['rna_is_multipotent'] == True)
].copy()

print(f"Cells multipotent in EITHER pDCM or pRNA: {len(either_multipotent)}")
print(f"  - Multipotent in DCM only: {((either_multipotent['dcm_is_multipotent'] == True) & (either_multipotent['rna_is_multipotent'] != True)).sum()}")
print(f"  - Multipotent in RNA only: {((either_multipotent['rna_is_multipotent'] == True) & (either_multipotent['dcm_is_multipotent'] != True)).sum()}")
print(f"  - Multipotent in BOTH:     {((either_multipotent['dcm_is_multipotent'] == True) & (either_multipotent['rna_is_multipotent'] == True)).sum()}")

################################################################################
# FILTER OUT 'none' IN SIGNIFICANT CELLTYPES
################################################################################

def contains_none_str(val):
    """Check if a significant_celltypes value contains 'none'."""
    if pd.isna(val) or val is None:
        return True
    parts = [p.strip().lower() for p in str(val).split(';')]
    return any(p == 'none' or p == '' or p == 'nan' for p in parts)

before = len(either_multipotent)
either_multipotent = either_multipotent[
    ~either_multipotent['dcm_significant_celltypes'].apply(contains_none_str) &
    ~either_multipotent['rna_significant_celltypes'].apply(contains_none_str)
].copy()
after = len(either_multipotent)
print(f"\nRemoved {before - after} rows with 'none' in significant_celltypes")
print(f"Remaining cells: {after}")

print("\nDCM significant cell types:")
print(either_multipotent['dcm_significant_celltypes'].value_counts())
print("\nRNA significant cell types:")
print(either_multipotent['rna_significant_celltypes'].value_counts())

print("\nCrosstab DCM vs RNA significant cell types:")
print(pd.crosstab(either_multipotent['dcm_significant_celltypes'],
                  either_multipotent['rna_significant_celltypes']))

################################################################################
# HEATMAP
################################################################################

if len(either_multipotent) > 0:

    # Sort by multipotent status first, then cell types
    either_multipotent['multipotent_status'] = 'none'
    either_multipotent.loc[
        (either_multipotent['dcm_is_multipotent'] == True) &
        (either_multipotent['rna_is_multipotent'] != True),
        'multipotent_status'] = 'DCM_only'
    either_multipotent.loc[
        (either_multipotent['rna_is_multipotent'] == True) &
        (either_multipotent['dcm_is_multipotent'] != True),
        'multipotent_status'] = 'RNA_only'
    either_multipotent.loc[
        (either_multipotent['dcm_is_multipotent'] == True) &
        (either_multipotent['rna_is_multipotent'] == True),
        'multipotent_status'] = 'BOTH'

    either_multipotent_sorted = either_multipotent.sort_values(
        by=['multipotent_status', 'dcm_significant_celltypes', 'rna_significant_celltypes']
    ).copy()

    # -----------------------------
    # SECOND FILTER - after sorting, remove any remaining none
    # (catches edge cases from multipotent_status assignment)
    # -----------------------------
    either_multipotent_sorted = either_multipotent_sorted[
        ~either_multipotent_sorted['dcm_significant_celltypes'].apply(contains_none_str) &
        ~either_multipotent_sorted['rna_significant_celltypes'].apply(contains_none_str) &
        (either_multipotent_sorted['multipotent_status'] != 'none')
    ].copy()

    print(f"\nFinal cells after all filters: {len(either_multipotent_sorted)}")
    print("\nStatus distribution:")
    print(either_multipotent_sorted['multipotent_status'].value_counts())

    # Get z-score columns from actual dataframe
    dcm_zcols = [col for col in either_multipotent_sorted.columns
                 if col.startswith('dcm_') and 'zscore' in col]
    rna_zcols = [col for col in either_multipotent_sorted.columns
                 if col.startswith('rna_') and 'zscore' in col]
    cell_types = [col.replace('dcm_ttest_', '').replace('_zscore', '') for col in dcm_zcols]

    print(f"\nDCM z-score columns: {dcm_zcols}")
    print(f"RNA z-score columns: {rna_zcols}")
    print(f"Cell types: {cell_types}")

    if len(dcm_zcols) == 0:
        print("No z-score columns found! Check column names:")
        print(either_multipotent_sorted.columns.tolist())
    else:
        dcm_matrix = either_multipotent_sorted[dcm_zcols].values.T
        rna_matrix = either_multipotent_sorted[rna_zcols].values.T

        # Color scale centered at 1.65
        vcenter = 1.65
        vmin = vcenter - 3
        vmax = vcenter + 3
        norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
        cmap = plt.cm.RdBu_r

        # Boundaries based on multipotent_status groups
        status_groups = either_multipotent_sorted['multipotent_status'].values
        status_boundaries = np.where(
            np.array(status_groups[:-1]) != np.array(status_groups[1:]))[0] + 1

        fig = plt.figure(figsize=(max(10, len(either_multipotent_sorted) * 0.25), 12))
        gs = fig.add_gridspec(2, 2, width_ratios=[1, 0.03], height_ratios=[1, 1],
                              hspace=0.8, wspace=0.05)

        # --- DCM heatmap ---
        ax_dcm = fig.add_subplot(gs[0, 0])
        im_dcm = ax_dcm.imshow(dcm_matrix, aspect='auto', cmap=cmap, norm=norm)

        for b in status_boundaries:
            ax_dcm.axvline(x=b - 0.5, color='white', linewidth=3)

        ax_dcm.set_yticks(range(len(cell_types)))
        ax_dcm.set_yticklabels(cell_types, fontsize=10)
        ax_dcm.set_xticks([])
        ax_dcm.set_ylabel('Module', fontsize=11, fontweight='bold')

        # DCM cell type labels on top via twin axis
        ax_dcm_top = ax_dcm.twiny()
        ax_dcm_top.set_xlim(ax_dcm.get_xlim())
        ax_dcm_top.set_xticks(range(len(either_multipotent_sorted)))
        ax_dcm_top.set_xticklabels(
            either_multipotent_sorted['dcm_significant_celltypes'].values,
            rotation=90, fontsize=7, ha='center'
        )
        ax_dcm_top.tick_params(axis='x', length=0)
        ax_dcm_top.set_title('DCM Module Z-Scores\n(multipotent in DCM, RNA, or both)\n',
                             fontsize=12, fontweight='bold', pad=60)

        # Multipotent status colored labels
        prev = 0
        status_colors = {'DCM_only': '#DC143C', 'RNA_only': '#4169E1', 'BOTH': '#2E8B57'}
        for b in list(status_boundaries) + [len(status_groups)]:
            mid = (prev + b) / 2
            label = status_groups[prev]
            ax_dcm_top.text(mid, 1.35, label, ha='center', va='bottom', fontsize=9,
                            fontweight='bold', rotation=0,
                            color=status_colors.get(label, 'black'),
                            transform=ax_dcm_top.get_xaxis_transform())
            prev = b

        ax_dcm_cbar = fig.add_subplot(gs[0, 1])
        cbar_dcm = fig.colorbar(im_dcm, cax=ax_dcm_cbar)
        cbar_dcm.set_label('Z-Score', fontsize=9)
        cbar_dcm.ax.axhline(y=vcenter, color='black', linewidth=1, linestyle='--')

        # --- RNA heatmap ---
        ax_rna = fig.add_subplot(gs[1, 0])
        im_rna = ax_rna.imshow(rna_matrix, aspect='auto', cmap=cmap, norm=norm)

        for b in status_boundaries:
            ax_rna.axvline(x=b - 0.5, color='white', linewidth=3)

        ax_rna.set_yticks(range(len(cell_types)))
        ax_rna.set_yticklabels(cell_types, fontsize=10)

        # batch_plate_well names on x-axis
        ax_rna.set_xticks(range(len(either_multipotent_sorted)))
        ax_rna.set_xticklabels(either_multipotent_sorted['batch_plate_well'].values,
                                rotation=90, fontsize=7, ha='center')
        ax_rna.set_xlabel('Cells (sorted by multipotent status, DCM and RNA cell types)',
                          fontsize=11, fontweight='bold')
        ax_rna.set_ylabel('Module', fontsize=11, fontweight='bold')

        # RNA cell type labels on top via twin axis
        ax_rna_top = ax_rna.twiny()
        ax_rna_top.set_xlim(ax_rna.get_xlim())
        ax_rna_top.set_xticks(range(len(either_multipotent_sorted)))
        ax_rna_top.set_xticklabels(
            either_multipotent_sorted['rna_significant_celltypes'].values,
            rotation=90, fontsize=7, ha='center'
        )
        ax_rna_top.tick_params(axis='x', length=0)
        ax_rna_top.set_title('RNA Module Z-Scores\n(multipotent in DCM, RNA, or both)\n',
                             fontsize=12, fontweight='bold', pad=60)

        ax_rna_cbar = fig.add_subplot(gs[1, 1])
        cbar_rna = fig.colorbar(im_rna, cax=ax_rna_cbar)
        cbar_rna.set_label('Z-Score', fontsize=9)
        cbar_rna.ax.axhline(y=vcenter, color='black', linewidth=1, linestyle='--')

        fig.suptitle(f'Z-Scores for Cells Multipotent in Either pDCM or pRNA\n'
                     f'(n = {len(either_multipotent_sorted)} cells | '
                     f'DCM only: {(status_groups == "DCM_only").sum()} | '
                     f'RNA only: {(status_groups == "RNA_only").sum()} | '
                     f'Both: {(status_groups == "BOTH").sum()})',
                     fontsize=13, fontweight='bold', y=1.05)

        plt.tight_layout()
        plt.savefig('either_multipotent_zscore_heatmap.pdf', dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        print("✅ Saved: either_multipotent_zscore_heatmap.pdf")

else:
    print("No multipotent cells found after filtering, skipping heatmap")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import matplotlib.patheffects as pe
from collections import Counter

# -----------------------------
# SETTINGS
# -----------------------------
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': '#7f7f7f'
}

cell_type_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

################################################################################
# STEP 1: EXTRACT SIGNIFICANT CELL TYPES AND Z-SCORES
################################################################################
def parse_significant_celltypes(sig_str):
    """Parse significant_celltypes string into a list - using ; separator."""
    if pd.isna(sig_str) or sig_str == '' or sig_str == 'nan':
        return []
    sig_str = str(sig_str).strip("[]'\" ")
    celltypes = [ct.strip().strip("'\" ") for ct in sig_str.split(';')]  # FIX: ; not ,
    return [ct for ct in celltypes if ct != '' and ct != 'nan']

def get_top2_by_zscore(sig_celltypes, zscore_row, prefix, ttest_cols):
    """
    Given a list of significant cell types and z-score values,
    return top 2 ordered by z-score descending.
    prefix should be 'dcm_' or 'rna_'
    """
    if len(sig_celltypes) == 0:
        return []
    
    ct_scores = {}
    for ct in sig_celltypes:
        col = f'{prefix}ttest_{ct}_zscore'  # e.g. dcm_ttest_Stem_zscore
        if col in zscore_row.index and not pd.isna(zscore_row[col]):
            ct_scores[ct] = zscore_row[col]
    
    sorted_cts = sorted(ct_scores.items(), key=lambda x: -x[1])
    return [ct for ct, _ in sorted_cts[:2]]
################################################################################
# BUILD SCHEMES - ALL APPROACHES
################################################################################

def get_ordered_by_zscore(sig_celltypes, zscore_row, prefix, top_n=2, ascending=True):
    """Order significant cell types by z-score. Default ascending (lowest left)."""
    if len(sig_celltypes) == 0:
        return []
    ct_scores = {}
    for ct in sig_celltypes:
        col = f'{prefix}ttest_{ct}_zscore'
        if col in zscore_row.index and not pd.isna(zscore_row[col]):
            ct_scores[ct] = zscore_row[col]
    # ascending=True: lowest z-score first (left), highest last (right)
    sorted_cts = sorted(ct_scores.items(), key=lambda x: x[1], reverse=not ascending)
    return [ct for ct, _ in sorted_cts[:top_n]]


def build_match_based_scheme(dcm_sig, rna_sig, row):
    """
    Match-based approach:
    Layout: DCM_first → shared → RNA_unique
    If DCM=[TA] and RNA=[Entero, TA]:
        DCM node (TA) → shared node (TA) → RNA unique (Entero)
    The DCM node comes first, then shared, then RNA unique.
    """
    dcm_set = set(dcm_sig)
    rna_set = set(rna_sig)
    overlap = dcm_set.intersection(rna_set)

    if len(overlap) > 0:
        # Pick shared node with highest combined z-score
        overlap_scores = {}
        for ct in overlap:
            dcm_col = f'dcm_ttest_{ct}_zscore'
            rna_col = f'rna_ttest_{ct}_zscore'
            dcm_z = row[dcm_col] if dcm_col in row.index else 0
            rna_z = row[rna_col] if rna_col in row.index else 0
            overlap_scores[ct] = (dcm_z + rna_z) / 2
        shared_ct = max(overlap_scores, key=overlap_scores.get)

        # DCM node: the non-shared DCM cell type (or shared if DCM only has one)
        dcm_unique = [ct for ct in dcm_sig if ct != shared_ct]
        dcm_unique_ordered = get_ordered_by_zscore(dcm_unique, row, 'dcm_', ascending=False)
        dcm_first = dcm_unique_ordered[0] if dcm_unique_ordered else None

        # RNA unique: non-shared RNA cell type
        rna_unique = [ct for ct in rna_sig if ct != shared_ct]
        rna_unique_ordered = get_ordered_by_zscore(rna_unique, row, 'rna_', ascending=False)
        rna_last = rna_unique_ordered[0] if rna_unique_ordered else None

        # Build: DCM_first → shared → RNA_unique
        nodes = []
        labels = []
        if dcm_first:
            nodes.append(dcm_first)
            labels.append('DCM')
        nodes.append(shared_ct)
        labels.append('shared')
        if rna_last:
            nodes.append(rna_last)
            labels.append('RNA')

        scheme_str = ' → '.join(nodes)
        return scheme_str, labels, 'with_overlap'

    else:
        # No overlap: DCM | RNA two-row with dashed connection
        dcm_ordered = get_ordered_by_zscore(dcm_sig, row, 'dcm_', ascending=False)
        rna_ordered = get_ordered_by_zscore(rna_sig, row, 'rna_', ascending=False)

        dcm_str = ' → '.join(dcm_ordered) if dcm_ordered else None
        rna_str = ' → '.join(rna_ordered) if rna_ordered else None

        if dcm_str and rna_str:
            return f"DCM: {dcm_str} | RNA: {rna_str}", None, 'no_overlap'
        elif dcm_str:
            return f"DCM: {dcm_str}", None, 'no_overlap'
        elif rna_str:
            return f"RNA: {rna_str}", None, 'no_overlap'
        return None, None, 'no_overlap'


def build_zscore_scheme(dcm_sig, rna_sig, row, dcm_multipotent, rna_multipotent):
    """
    Z-score based: lowest z-score LEFT, highest RIGHT (ascending).
    Always show both assays as two rows connected by dashed arrow.
    Non-multipotent assay shows only top 1 cell type.
    """
    dcm_ordered = get_ordered_by_zscore(
        dcm_sig, row, 'dcm_', top_n=2 if dcm_multipotent else 1, ascending=True)
    rna_ordered = get_ordered_by_zscore(
        rna_sig, row, 'rna_', top_n=2 if rna_multipotent else 1, ascending=True)

    dcm_str = ' → '.join(dcm_ordered) if dcm_ordered else None
    rna_str = ' → '.join(rna_ordered) if rna_ordered else None

    if dcm_str and rna_str:
        return f"DCM: {dcm_str} | RNA: {rna_str}"
    elif dcm_str:
        return f"DCM: {dcm_str}"
    elif rna_str:
        return f"RNA: {rna_str}"
    return None


def build_approach2_scheme(dcm_sig, rna_sig, row,
                            dcm_multipotent, rna_multipotent):
    """Original approach 2: overlap as middle node in linear scheme."""
    dcm_set = set(dcm_sig)
    rna_set = set(rna_sig)
    overlap = dcm_set.intersection(rna_set)

    if dcm_multipotent and rna_multipotent and len(overlap) > 0:
        overlap_scores = {}
        for ct in overlap:
            dcm_col = f'dcm_ttest_{ct}_zscore'
            rna_col = f'rna_ttest_{ct}_zscore'
            dcm_z = row[dcm_col] if dcm_col in row.index else 0
            rna_z = row[rna_col] if rna_col in row.index else 0
            overlap_scores[ct] = (dcm_z + rna_z) / 2
        middle_ct = max(overlap_scores, key=overlap_scores.get)

        dcm_unique = get_ordered_by_zscore(
            [ct for ct in dcm_sig if ct != middle_ct], row, 'dcm_', ascending=False)
        rna_unique = get_ordered_by_zscore(
            [ct for ct in rna_sig if ct != middle_ct], row, 'rna_', ascending=False)

        dcm_first = dcm_unique[0] if dcm_unique else None
        rna_last = rna_unique[0] if rna_unique else None

        if dcm_first and rna_last:
            return f"{dcm_first} → {middle_ct} → {rna_last}", 'with_overlap'
        elif dcm_first:
            return f"{dcm_first} → {middle_ct}", 'with_overlap'
        elif rna_last:
            return f"{middle_ct} → {rna_last}", 'with_overlap'
        return None, 'with_overlap'

    else:
        # No overlap or single assay
        scheme = build_single_assay_scheme(
            row,
            get_ordered_by_zscore(dcm_sig, row, 'dcm_', ascending=False),
            get_ordered_by_zscore(rna_sig, row, 'rna_', ascending=False),
            dcm_multipotent, rna_multipotent)

        if dcm_multipotent and rna_multipotent:
            return scheme, 'no_overlap'
        elif dcm_multipotent:
            return scheme, 'dcm_only'
        else:
            return scheme, 'rna_only'


################################################################################
# REBUILD ALL RECORDS
################################################################################

records = []

for _, row in either_multipotent_sorted.iterrows():

    dcm_sig = parse_significant_celltypes(row['dcm_significant_celltypes'])
    rna_sig = parse_significant_celltypes(row['rna_significant_celltypes'])
    dcm_multipotent = row['dcm_is_multipotent'] == True
    rna_multipotent = row['rna_is_multipotent'] == True
    status = row['multipotent_status']

    # Approach 2
    approach2_scheme, approach2_type = build_approach2_scheme(
        dcm_sig, rna_sig, row, dcm_multipotent, rna_multipotent)

    # Approach 3 (match-based)
    approach3_scheme, approach3_labels, approach3_type = build_match_based_scheme(
        dcm_sig, rna_sig, row)

    # Approach 4 (z-score ascending)
    approach4_scheme = build_zscore_scheme(
        dcm_sig, rna_sig, row, dcm_multipotent, rna_multipotent)

    records.append({
        'batch_plate_well': row['batch_plate_well'],
        'multipotent_status': status,
        'dcm_sig': dcm_sig,
        'rna_sig': rna_sig,
        'approach2_scheme': approach2_scheme,
        'approach2_type': approach2_type,
        'approach3_scheme': approach3_scheme,
        'approach3_labels': approach3_labels,
        'approach3_type': approach3_type,
        'approach4_scheme': approach4_scheme,
    })

schemes_df = pd.DataFrame(records)
schemes_df = schemes_df[
    schemes_df['approach2_scheme'].notna() |
    schemes_df['approach3_scheme'].notna() |
    schemes_df['approach4_scheme'].notna()
]

print(f"Total valid schemes: {len(schemes_df)}")
print("\nAPPROACH 2 (overlap middle):")
print(schemes_df['approach2_scheme'].value_counts())
print("\nAPPROACH 3 (match-based, DCM → shared → RNA):")
print(schemes_df['approach3_scheme'].value_counts())
print("\nAPPROACH 4 (z-score ascending):")
print(schemes_df['approach4_scheme'].value_counts())

# Build node_labels_dict for approach 3
node_labels_dict = {}
for rec in records:
    if rec['approach3_labels'] and rec['approach3_scheme']:
        node_labels_dict[rec['approach3_scheme']] = rec['approach3_labels']

################################################################################
# COUNTERS
################################################################################

approach2_overlap    = Counter(schemes_df[schemes_df['approach2_type'] == 'with_overlap']['approach2_scheme'].dropna())
approach2_no_overlap = Counter(schemes_df[schemes_df['approach2_type'] == 'no_overlap']['approach2_scheme'].dropna())
approach2_single     = Counter(schemes_df[schemes_df['approach2_type'].isin(['dcm_only', 'rna_only'])]['approach2_scheme'].dropna())

approach3_overlap    = Counter(schemes_df[schemes_df['approach3_type'] == 'with_overlap']['approach3_scheme'].dropna())
approach3_no_overlap = Counter(schemes_df[schemes_df['approach3_type'] == 'no_overlap']['approach3_scheme'].dropna())

approach4_counts     = Counter(schemes_df['approach4_scheme'].dropna())

################################################################################
# PDF 1: APPROACH 2 (original 3-panel)
################################################################################

fig, axes = plt.subplots(1, 3, figsize=(24, 14))

draw_arrow_diagram(
    approach2_overlap,
    f'Approach 2: Overlap as middle\n'
    f'(BOTH multipotent + overlap, '
    f'n={len(schemes_df[schemes_df["approach2_type"] == "with_overlap"])})',
    axes[0], cell_type_colors)

draw_arrow_diagram(
    approach2_no_overlap,
    f'Approach 2: No overlap\n'
    f'(BOTH multipotent, no shared, '
    f'n={len(schemes_df[schemes_df["approach2_type"] == "no_overlap"])})',
    axes[1], cell_type_colors)

draw_arrow_diagram(
    approach2_single,
    f'Approach 2: Single assay multipotent\n'
    f'(DCM only: {len(schemes_df[schemes_df["approach2_type"] == "dcm_only"])} | '
    f'RNA only: {len(schemes_df[schemes_df["approach2_type"] == "rna_only"])})',
    axes[2], cell_type_colors)

plt.suptitle('Approach 2: Overlap as Middle Node',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('hybrid_schemes_approach2.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print("✅ Saved: hybrid_schemes_approach2.pdf")

################################################################################
# PDF 2: APPROACH 3 (match-based: DCM → shared → RNA)
################################################################################

fig, axes = plt.subplots(1, 2, figsize=(18, 14))

draw_arrow_diagram(
    approach3_overlap,
    f'Approach 3 (match-based): With overlap\n'
    f'DCM node → shared → RNA unique\n'
    f'(n={len(schemes_df[schemes_df["approach3_type"] == "with_overlap"])})',
    axes[0], cell_type_colors,
    node_labels_dict=node_labels_dict)

draw_arrow_diagram(
    approach3_no_overlap,
    f'Approach 3 (match-based): No overlap\n'
    f'(n={len(schemes_df[schemes_df["approach3_type"] == "no_overlap"])})',
    axes[1], cell_type_colors)

plt.suptitle('Approach 3: Match-Based (DCM node → shared → RNA unique)\n'
             'If DCM=[TA] and RNA=[Entero,TA]: TA(DCM) → TA(shared) → Entero(RNA)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('hybrid_schemes_approach3_matchbased.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print("✅ Saved: hybrid_schemes_approach3_matchbased.pdf")

################################################################################
# PDF 3: APPROACH 4 (z-score ascending: lowest left, highest right)
################################################################################

fig, ax = plt.subplots(1, 1, figsize=(12, 14))

draw_arrow_diagram(
    approach4_counts,
    f'Approach 4: Z-score ordered (ascending)\n'
    f'Lowest z-score LEFT → Highest z-score RIGHT\n'
    f'(n={len(schemes_df[schemes_df["approach4_scheme"].notna()])})',
    ax, cell_type_colors)

plt.suptitle('Approach 4: Z-score Ordered (Ascending per Assay)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('hybrid_schemes_approach4_zscore.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print("✅ Saved: hybrid_schemes_approach4_zscore.pdf")

# Save summary CSV
schemes_df.to_csv('hybrid_direction_schemes_all.csv', index=False)
print("✅ Saved: hybrid_direction_schemes_all.csv")

In [ ]:
#Transitions DCM and RNA with hybrids

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from collections import Counter
import re
import os

################################################################################
# SETTINGS
################################################################################

cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': '#7f7f7f'
}

cell_type_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

schemes_dir = os.path.join(score_dir, 'hybrid_schemes')
os.makedirs(schemes_dir, exist_ok=True)

################################################################################
# HELPER FUNCTIONS
################################################################################

def is_valid_celltype(ct):
    """Check if cell type is valid."""
    if ct is None or ct == '' or str(ct).lower() in ['none', 'nan', 'unknown']:
        return False
    return True


def parse_significant_celltypes(sig_str):
    """Parse significant_celltypes string - semicolon separated, exclude none."""
    if pd.isna(sig_str) or sig_str == '' or sig_str == 'nan':
        return []
    sig_str = str(sig_str).strip("[]'\" ")
    celltypes = [ct.strip().strip("'\" ") for ct in sig_str.split(';')]
    return [ct for ct in celltypes
            if ct != '' and ct != 'nan' and str(ct).lower() != 'none']


def get_ordered_by_zscore(sig_celltypes, zscore_row, prefix, top_n=2, ascending=True):
    """Order significant cell types by z-score, excluding invalid cell types."""
    if len(sig_celltypes) == 0:
        return []
    ct_scores = {}
    for ct in sig_celltypes:
        if not is_valid_celltype(ct):
            continue
        col = f'{prefix}ttest_{ct}_zscore'
        if col in zscore_row.index and not pd.isna(zscore_row[col]):
            ct_scores[ct] = zscore_row[col]
    sorted_cts = sorted(ct_scores.items(), key=lambda x: x[1], reverse=not ascending)
    return [ct for ct, _ in sorted_cts[:top_n]]


def contains_none_celltype(scheme_str):
    """Check if scheme contains 'none' as a cell type node."""
    if pd.isna(scheme_str) or scheme_str is None:
        return False
    nodes = re.split(r'→|\|', str(scheme_str).lower())
    for node in nodes:
        node_clean = node.strip().replace('dcm:', '').replace('rna:', '').strip()
        if node_clean == 'none':
            return True
    return False


def build_single_assay_scheme(row, dcm_ordered, rna_ordered,
                               dcm_multipotent, rna_multipotent):
    """Build scheme string for single-assay multipotent cells."""
    if dcm_multipotent and not rna_multipotent:
        if len(dcm_ordered) >= 2:
            dcm_part = f"DCM: {dcm_ordered[0]} → {dcm_ordered[1]}"
        elif len(dcm_ordered) == 1:
            dcm_part = f"DCM: {dcm_ordered[0]}"
        else:
            return None
        rna_sig = parse_significant_celltypes(row['rna_significant_celltypes'])
        rna_valid = [ct for ct in rna_sig if is_valid_celltype(ct)]
        rna_part = f"RNA: {rna_valid[0]}" if rna_valid else None
        if rna_part:
            return f"{dcm_part} | {rna_part}"
        return dcm_part

    elif rna_multipotent and not dcm_multipotent:
        dcm_sig = parse_significant_celltypes(row['dcm_significant_celltypes'])
        dcm_valid = [ct for ct in dcm_sig if is_valid_celltype(ct)]
        dcm_part = f"DCM: {dcm_valid[0]}" if dcm_valid else None
        if len(rna_ordered) >= 2:
            rna_part = f"RNA: {rna_ordered[0]} → {rna_ordered[1]}"
        elif len(rna_ordered) == 1:
            rna_part = f"RNA: {rna_ordered[0]}"
        else:
            return None
        if dcm_part:
            return f"{dcm_part} | {rna_part}"
        return rna_part
    return None


def build_approach2_scheme(dcm_sig, rna_sig, row,
                            dcm_multipotent, rna_multipotent):
    """Approach 2: overlap as middle node in linear scheme."""
    dcm_set = set(dcm_sig)
    rna_set = set(rna_sig)
    overlap = dcm_set.intersection(rna_set)

    if dcm_multipotent and rna_multipotent and len(overlap) > 0:
        overlap_scores = {}
        for ct in overlap:
            if not is_valid_celltype(ct):
                continue
            dcm_col = f'dcm_ttest_{ct}_zscore'
            rna_col = f'rna_ttest_{ct}_zscore'
            dcm_z = row[dcm_col] if dcm_col in row.index else 0
            rna_z = row[rna_col] if rna_col in row.index else 0
            overlap_scores[ct] = (dcm_z + rna_z) / 2

        if len(overlap_scores) == 0:
            return None, 'with_overlap'

        middle_ct = max(overlap_scores, key=overlap_scores.get)

        dcm_unique = get_ordered_by_zscore(
            [ct for ct in dcm_sig if ct != middle_ct and is_valid_celltype(ct)],
            row, 'dcm_', ascending=False)
        rna_unique = get_ordered_by_zscore(
            [ct for ct in rna_sig if ct != middle_ct and is_valid_celltype(ct)],
            row, 'rna_', ascending=False)

        dcm_first = dcm_unique[0] if dcm_unique else None
        rna_last = rna_unique[0] if rna_unique else None

        if dcm_first and rna_last:
            return f"{dcm_first} → {middle_ct} → {rna_last}", 'with_overlap'
        elif dcm_first:
            return f"{dcm_first} → {middle_ct}", 'with_overlap'
        elif rna_last:
            return f"{middle_ct} → {rna_last}", 'with_overlap'
        return None, 'with_overlap'

    else:
        scheme = build_single_assay_scheme(
            row,
            get_ordered_by_zscore(dcm_sig, row, 'dcm_', ascending=False),
            get_ordered_by_zscore(rna_sig, row, 'rna_', ascending=False),
            dcm_multipotent, rna_multipotent)

        if dcm_multipotent and rna_multipotent:
            return scheme, 'no_overlap'
        elif dcm_multipotent:
            return scheme, 'dcm_only'
        else:
            return scheme, 'rna_only'


def build_match_based_scheme(dcm_sig, rna_sig, row):
    """
    Approach 3: match-based.
    Layout: DCM_first → shared → RNA_unique
    """
    dcm_set = set([ct for ct in dcm_sig if is_valid_celltype(ct)])
    rna_set = set([ct for ct in rna_sig if is_valid_celltype(ct)])
    overlap = dcm_set.intersection(rna_set)

    if len(overlap) > 0:
        overlap_scores = {}
        for ct in overlap:
            dcm_col = f'dcm_ttest_{ct}_zscore'
            rna_col = f'rna_ttest_{ct}_zscore'
            dcm_z = row[dcm_col] if dcm_col in row.index else 0
            rna_z = row[rna_col] if rna_col in row.index else 0
            overlap_scores[ct] = (dcm_z + rna_z) / 2

        if len(overlap_scores) == 0:
            return None, None, 'with_overlap'

        shared_ct = max(overlap_scores, key=overlap_scores.get)

        dcm_unique = get_ordered_by_zscore(
            [ct for ct in dcm_sig if ct != shared_ct and is_valid_celltype(ct)],
            row, 'dcm_', ascending=False)
        rna_unique = get_ordered_by_zscore(
            [ct for ct in rna_sig if ct != shared_ct and is_valid_celltype(ct)],
            row, 'rna_', ascending=False)

        dcm_first = dcm_unique[0] if dcm_unique else None
        rna_last = rna_unique[0] if rna_unique else None

        nodes = []
        labels = []
        if dcm_first:
            nodes.append(dcm_first)
            labels.append('DCM')
        nodes.append(shared_ct)
        labels.append('shared')
        if rna_last:
            nodes.append(rna_last)
            labels.append('RNA')

        return ' → '.join(nodes), labels, 'with_overlap'

    else:
        dcm_ordered = get_ordered_by_zscore(
            [ct for ct in dcm_sig if is_valid_celltype(ct)],
            row, 'dcm_', ascending=False)
        rna_ordered = get_ordered_by_zscore(
            [ct for ct in rna_sig if is_valid_celltype(ct)],
            row, 'rna_', ascending=False)

        dcm_str = ' → '.join(dcm_ordered) if dcm_ordered else None
        rna_str = ' → '.join(rna_ordered) if rna_ordered else None

        if dcm_str and rna_str:
            return f"DCM: {dcm_str} | RNA: {rna_str}", None, 'no_overlap'
        elif dcm_str:
            return f"DCM: {dcm_str}", None, 'no_overlap'
        elif rna_str:
            return f"RNA: {rna_str}", None, 'no_overlap'
        return None, None, 'no_overlap'


def build_zscore_scheme(dcm_sig, rna_sig, row, dcm_multipotent, rna_multipotent):
    """Approach 4: z-score ascending (lowest left, highest right)."""
    dcm_ordered = get_ordered_by_zscore(
        [ct for ct in dcm_sig if is_valid_celltype(ct)],
        row, 'dcm_', top_n=2 if dcm_multipotent else 1, ascending=True)
    rna_ordered = get_ordered_by_zscore(
        [ct for ct in rna_sig if is_valid_celltype(ct)],
        row, 'rna_', top_n=2 if rna_multipotent else 1, ascending=True)

    dcm_str = ' → '.join(dcm_ordered) if dcm_ordered else None
    rna_str = ' → '.join(rna_ordered) if rna_ordered else None

    if dcm_str and rna_str:
        return f"DCM: {dcm_str} | RNA: {rna_str}"
    elif dcm_str:
        return f"DCM: {dcm_str}"
    elif rna_str:
        return f"RNA: {rna_str}"
    return None


################################################################################
# ADD DOX TO either_multipotent_sorted
################################################################################

if 'Dox' not in either_multipotent_sorted.columns:
    dox_info = adata_combined.obs[
        adata_combined.obs['assay'] == 'pDCM'
    ][['batch_plate_well', 'Dox']].drop_duplicates('batch_plate_well')
    either_multipotent_sorted = either_multipotent_sorted.merge(
        dox_info, on='batch_plate_well', how='left')

print("Dox values:", either_multipotent_sorted['Dox'].unique())

################################################################################
# BUILD ALL RECORDS
################################################################################

records = []

for _, row in either_multipotent_sorted.iterrows():

    dcm_sig = parse_significant_celltypes(row['dcm_significant_celltypes'])
    rna_sig = parse_significant_celltypes(row['rna_significant_celltypes'])
    dcm_multipotent = row['dcm_is_multipotent'] == True
    rna_multipotent = row['rna_is_multipotent'] == True
    status = row['multipotent_status']
    dox = row['Dox'] if 'Dox' in row.index else 'Unknown'

    # Approach 2
    approach2_scheme, approach2_type = build_approach2_scheme(
        dcm_sig, rna_sig, row, dcm_multipotent, rna_multipotent)

    # Approach 3
    approach3_scheme, approach3_labels, approach3_type = build_match_based_scheme(
        dcm_sig, rna_sig, row)

    # Approach 4
    approach4_scheme = build_zscore_scheme(
        dcm_sig, rna_sig, row, dcm_multipotent, rna_multipotent)

    records.append({
        'batch_plate_well': row['batch_plate_well'],
        'multipotent_status': status,
        'Dox': dox,
        'dcm_sig': dcm_sig,
        'rna_sig': rna_sig,
        'approach2_scheme': approach2_scheme,
        'approach2_type': approach2_type,
        'approach3_scheme': approach3_scheme,
        'approach3_labels': approach3_labels,
        'approach3_type': approach3_type,
        'approach4_scheme': approach4_scheme,
    })

schemes_df = pd.DataFrame(records)

# -----------------------------
# EXCLUDE 'none' CELL TYPES
# -----------------------------
before = len(schemes_df)
schemes_df = schemes_df[
    ~schemes_df['approach2_scheme'].apply(contains_none_celltype) &
    ~schemes_df['approach3_scheme'].apply(contains_none_celltype) &
    ~schemes_df['approach4_scheme'].apply(contains_none_celltype)
]
schemes_df = schemes_df[
    schemes_df['approach2_scheme'].notna() |
    schemes_df['approach3_scheme'].notna() |
    schemes_df['approach4_scheme'].notna()
]
after = len(schemes_df)
print(f"Removed {before - after} cells with 'none' cell type")
print(f"Remaining: {after} cells")
print("\nDox distribution:")
print(schemes_df['Dox'].value_counts())

# Build node_labels_dict for approach 3
node_labels_dict = {}
for rec in records:
    if rec['approach3_labels'] and rec['approach3_scheme']:
        if not contains_none_celltype(rec['approach3_scheme']):
            node_labels_dict[rec['approach3_scheme']] = rec['approach3_labels']

dox_values = sorted(schemes_df['Dox'].dropna().unique())
print(f"\nDox values: {dox_values}")

################################################################################
# DRAW FUNCTION
################################################################################

def parse_scheme_to_nodes(scheme_str):
    """Parse scheme string into nodes with assay labels."""
    if pd.isna(scheme_str) or scheme_str is None:
        return []
    scheme_str = str(scheme_str)
    if 'DCM:' in scheme_str and 'RNA:' in scheme_str:
        parts = scheme_str.split('|')
        dcm_part = parts[0].replace('DCM:', '').strip()
        rna_part = parts[1].replace('RNA:', '').strip()
        dcm_nodes = [n.strip() for n in dcm_part.split('→') if is_valid_celltype(n.strip())]
        rna_nodes = [n.strip() for n in rna_part.split('→') if is_valid_celltype(n.strip())]
        return [('DCM', dcm_nodes), ('RNA', rna_nodes)]
    elif 'DCM:' in scheme_str:
        dcm_part = scheme_str.replace('DCM:', '').strip()
        return [('DCM', [n.strip() for n in dcm_part.split('→')
                         if is_valid_celltype(n.strip())])]
    elif 'RNA:' in scheme_str:
        rna_part = scheme_str.replace('RNA:', '').strip()
        return [('RNA', [n.strip() for n in rna_part.split('→')
                         if is_valid_celltype(n.strip())])]
    else:
        return [('BOTH', [n.strip() for n in scheme_str.split('→')
                          if is_valid_celltype(n.strip())])]


def draw_arrow_diagram(schemes_counts, title, ax, cell_type_colors,
                       max_show=15, node_labels_dict=None):
    """
    Draw arrow diagram with all nodes on the same horizontal line.
    - Excludes RNA-only schemes (no DCM node)
    - Consistent node size across all scheme types
    - Fixed xlim to prevent label overflow
    """
    # Filter out RNA-only schemes and none cell types
    top_schemes = []
    for s, c in schemes_counts.most_common(max_show * 2):  # fetch more to account for filtering
        if contains_none_celltype(s):
            continue
        # Exclude RNA-only (schemes that have RNA: but no DCM:)
        if 'RNA:' in str(s) and 'DCM:' not in str(s):
            continue
        top_schemes.append((s, c))
        if len(top_schemes) >= max_show:
            break

    if len(top_schemes) == 0:
        ax.text(0.5, 0.5, 'No schemes found', ha='center', va='center',
                transform=ax.transAxes, fontsize=12)
        ax.axis('off')
        return

    n_schemes = len(top_schemes)
    row_height = 1.2
    ax.set_xlim(-1.5, 10)
    ax.set_ylim(-0.8, n_schemes * row_height)
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=15)

    node_radius = 0.28  # consistent across all scheme types
    node_spacing = 1.6  # consistent spacing

    label_colors = {
        'DCM': '#DC143C', 'RNA': '#4169E1',
        'shared': '#2E8B57', 'none': 'black'
    }

    for i, (scheme, count) in enumerate(top_schemes):
        y = (n_schemes - i - 1) * row_height + row_height / 2
        parsed = parse_scheme_to_nodes(scheme)

        if len(parsed) == 0:
            continue

        if parsed[0][0] == 'BOTH':
            # -------------------------------------------------------
            # LINEAR SCHEME: always show 3 nodes even if shared
            # Node 1: DCM, Node 2: shared (shown twice), Node 3: RNA
            # -------------------------------------------------------
            nodes = [n for n in parsed[0][1] if is_valid_celltype(n)]
            n_nodes = len(nodes)
            if n_nodes == 0:
                continue

            # Get per-node labels
            per_node_labels = None
            if node_labels_dict and scheme in node_labels_dict:
                per_node_labels = node_labels_dict[scheme]

            # If only 2 nodes (e.g. DCM → shared, no RNA unique),
            # duplicate the shared node to always show 3
            if n_nodes == 2 and per_node_labels is not None:
                if 'shared' in per_node_labels:
                    shared_idx = per_node_labels.index('shared')
                    shared_node = nodes[shared_idx]
                    if per_node_labels[0] == 'DCM':
                        # DCM → shared, add shared as RNA too
                        nodes = [nodes[0], shared_node, shared_node]
                        per_node_labels = ['DCM', 'shared', 'RNA']
                    elif per_node_labels[-1] == 'RNA':
                        # shared → RNA, add shared as DCM too
                        nodes = [shared_node, shared_node, nodes[-1]]
                        per_node_labels = ['DCM', 'shared', 'RNA']
            elif n_nodes == 1:
                # Only shared node: show as DCM → shared → RNA
                nodes = [nodes[0], nodes[0], nodes[0]]
                per_node_labels = ['DCM', 'shared', 'RNA']

            n_nodes = len(nodes)
            x_positions = [j * node_spacing for j in range(n_nodes)]

            for j, (ct, x) in enumerate(zip(nodes, x_positions)):
                color = cell_type_colors.get(ct, '#cccccc')

                # Determine label
                if per_node_labels and j < len(per_node_labels):
                    lbl = per_node_labels[j]
                else:
                    if n_nodes == 3:
                        lbl = ['DCM', 'shared', 'RNA'][j]
                    elif n_nodes == 2:
                        lbl = ['DCM', 'RNA'][j]
                    else:
                        lbl = ''

                lc = label_colors.get(lbl, 'black')

                # Border color based on assay
                if lbl == 'DCM':
                    border_color = '#DC143C'
                elif lbl == 'RNA':
                    border_color = '#4169E1'
                elif lbl == 'shared':
                    border_color = '#2E8B57'
                else:
                    border_color = 'black'

                ax.text(x, y + node_radius + 0.25, lbl,
                       ha='center', va='bottom', fontsize=7,
                       fontweight='bold', color=lc)

                circle = plt.Circle((x, y), node_radius, color=color,
                                   zorder=3, linewidth=2, ec=border_color)
                ax.add_patch(circle)
                ax.text(x, y, ct[:3], ha='center', va='center',
                       fontsize=7, fontweight='bold', color='white', zorder=4)

                if j < n_nodes - 1:
                    x_next = x_positions[j + 1]

                    # Solid arrow: DCM → shared
                    # Dashed arrow: shared → RNA
                    if lbl == 'shared':
                        arrow_color = '#4169E1'
                        ls = '--'
                    elif lbl == 'DCM':
                        arrow_color = '#DC143C'
                        ls = '-'
                    else:
                        arrow_color = 'black'
                        ls = '-'

                    ax.annotate('',
                               xy=(x_next - node_radius - 0.05, y),
                               xytext=(x + node_radius + 0.05, y),
                               arrowprops=dict(arrowstyle='-|>',
                                             color=arrow_color,
                                             lw=1.5, mutation_scale=15,
                                             linestyle=ls),
                               zorder=2)

            x_end = x_positions[-1] if x_positions else 0
            ax.text(x_end + 0.7, y, f'n={count}',
                   ha='left', va='center', fontsize=9,
                   fontweight='bold', color='black')

################################################################################
# COUNTER HELPER
################################################################################

def get_counters_for_dox(df, dox=None):
    """Get all approach counters for a specific Dox value or all combined."""
    sub = df[df['Dox'] == dox] if dox is not None else df

    return {
        'a2_overlap':    Counter(sub[sub['approach2_type'] == 'with_overlap']['approach2_scheme'].dropna()),
        'a2_no_overlap': Counter(sub[sub['approach2_type'] == 'no_overlap']['approach2_scheme'].dropna()),
        'a2_single':     Counter(sub[sub['approach2_type'].isin(['dcm_only', 'rna_only'])]['approach2_scheme'].dropna()),
        'a3_overlap':    Counter(sub[sub['approach3_type'] == 'with_overlap']['approach3_scheme'].dropna()),
        'a3_no_overlap': Counter(sub[sub['approach3_type'] == 'no_overlap']['approach3_scheme'].dropna()),
        'a4':            Counter(sub['approach4_scheme'].dropna()),
        'n_total':       len(sub),
        'n_dcm_only':    len(sub[sub['approach2_type'] == 'dcm_only']),
        'n_rna_only':    len(sub[sub['approach2_type'] == 'rna_only']),
        'n_a2_overlap':  len(sub[sub['approach2_type'] == 'with_overlap']),
        'n_a2_no_overlap': len(sub[sub['approach2_type'] == 'no_overlap']),
        'n_a3_overlap':  len(sub[sub['approach3_type'] == 'with_overlap']),
        'n_a3_no_overlap': len(sub[sub['approach3_type'] == 'no_overlap']),
        'n_a4':          len(sub[sub['approach4_scheme'].notna()]),
    }


################################################################################
# PLOT FUNCTION PER DOX
################################################################################

def plot_all_approaches_for_dox(schemes_df, dox, cell_type_colors,
                                 node_labels_dict, save_dir='.'):
    """Generate all 3 PDFs for a specific Dox value or combined."""
    dox_label = 'ALL' if dox is None else str(dox).replace(' ', '_').replace('/', '-')
    c = get_counters_for_dox(schemes_df, dox if dox != 'ALL' else None)

    print(f"\n{'='*60}")
    print(f"PLOTTING DOX: {dox_label} (n={c['n_total']})")
    print(f"{'='*60}")

    # --- PDF 1: Approach 2 ---
    fig, axes = plt.subplots(1, 3, figsize=(24, 14))
    draw_arrow_diagram(
        c['a2_overlap'],
        f'Approach 2: Overlap as middle\n(BOTH + overlap, n={c["n_a2_overlap"]})',
        axes[0], cell_type_colors)
    draw_arrow_diagram(
        c['a2_no_overlap'],
        f'Approach 2: No overlap\n(BOTH, no shared, n={c["n_a2_no_overlap"]})',
        axes[1], cell_type_colors)
    draw_arrow_diagram(
        c['a2_single'],
        f'Approach 2: Single assay multipotent\n'
        f'(DCM only: {c["n_dcm_only"]} | RNA only: {c["n_rna_only"]})',
        axes[2], cell_type_colors)
    plt.suptitle(f'Approach 2: Overlap as Middle Node | Dox = {dox_label}',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    path = os.path.join(save_dir, f'hybrid_schemes_approach2_Dox_{dox_label}.pdf')
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  ✅ Saved: {path}")

    # --- PDF 2: Approach 3 ---
    fig, axes = plt.subplots(1, 2, figsize=(18, 14))
    draw_arrow_diagram(
        c['a3_overlap'],
        f'Approach 3 (match-based): With overlap\n'
        f'DCM node → shared → RNA unique\n(n={c["n_a3_overlap"]})',
        axes[0], cell_type_colors,
        node_labels_dict=node_labels_dict)
    draw_arrow_diagram(
        c['a3_no_overlap'],
        f'Approach 3 (match-based): No overlap\n(n={c["n_a3_no_overlap"]})',
        axes[1], cell_type_colors)
    plt.suptitle(f'Approach 3: Match-Based | Dox = {dox_label}',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    path = os.path.join(save_dir, f'hybrid_schemes_approach3_Dox_{dox_label}.pdf')
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  ✅ Saved: {path}")

    # --- PDF 3: Approach 4 ---
    fig, ax = plt.subplots(1, 1, figsize=(12, 14))
    draw_arrow_diagram(
        c['a4'],
        f'Approach 4: Z-score ordered (ascending)\n'
        f'Lowest z-score LEFT → Highest z-score RIGHT\n(n={c["n_a4"]})',
        ax, cell_type_colors)
    plt.suptitle(f'Approach 4: Z-score Ordered | Dox = {dox_label}',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    path = os.path.join(save_dir, f'hybrid_schemes_approach4_Dox_{dox_label}.pdf')
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  ✅ Saved: {path}")


################################################################################
# RUN: COMBINED + PER DOX
################################################################################

# Combined
plot_all_approaches_for_dox(schemes_df, dox='ALL',
                             cell_type_colors=cell_type_colors,
                             node_labels_dict=node_labels_dict,
                             save_dir=schemes_dir)

# Per Dox
for dox in dox_values:
    plot_all_approaches_for_dox(schemes_df, dox=dox,
                                 cell_type_colors=cell_type_colors,
                                 node_labels_dict=node_labels_dict,
                                 save_dir=schemes_dir)

################################################################################
# SAVE SUMMARY
################################################################################

schemes_df.to_csv(os.path.join(schemes_dir, 'hybrid_direction_schemes_all.csv'), index=False)
print(f"\n✅ Saved: {os.path.join(schemes_dir, 'hybrid_direction_schemes_all.csv')}")

dox_summary = schemes_df.groupby('Dox').agg(
    n_total=('batch_plate_well', 'count'),
    n_both=('multipotent_status', lambda x: (x == 'BOTH').sum()),
    n_dcm_only=('multipotent_status', lambda x: (x == 'DCM_only').sum()),
    n_rna_only=('multipotent_status', lambda x: (x == 'RNA_only').sum()),
).reset_index()
print("\nDox summary:")
print(dox_summary)
dox_summary.to_csv(os.path.join(schemes_dir, 'hybrid_schemes_dox_summary.csv'), index=False)
print(f"✅ Saved: {os.path.join(schemes_dir, 'hybrid_schemes_dox_summary.csv')}")

print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
## RNA cell cycling across different cells:
#1. Cells that reverse back to Stem/TA (RNA)
#2 Cells that are hybrid in DCM and RNA (3 cells)
#3. cells that are only hybrid in RNA

In [ ]:
adata_combined = sc.read_h5ad('/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/integration/adata_combined_gmm_23032026.h5ad')

## gmm085

## Cell cycle mature to inmature

In [ ]:
### adata_combined_gmm_23032026
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import scanpy as sc

root      = "/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/"
score_dir = f'{root}/gene_scoring/gmm_085/cell_cycle/'
import os; os.makedirs(score_dir, exist_ok=True)

# ─────────────────────────────────────────────
# SETTINGS
# ─────────────────────────────────────────────
s_genes = ['MCM5','PCNA','TYMS','FEN1','MCM2','MCM4','RRM1','UNG','GINS2','MCM6',
           'CDCA7','DTL','PRIM1','UHRF1','MLF1IP','HELLS','RFC2','RPA2','NASP',
           'RAD51AP1','GMNN','WDR76','SLBP','CCNE2','UBR7','POLD3','MSH2','ATAD2',
           'RAD51','RRM2','CDC45','CDC6','EXO1','TIPIN','DSCC1','BLM','CASP8AP2',
           'USP1','CLSPN','POLA1','CHAF1B','BRIP1','E2F8']
g2m_genes = ['HMGB2','CDK1','NUSAP1','UBE2C','BIRC5','TPX2','TOP2A','NDC80','CKS2',
             'NUF2','CKS1B','MKI67','TMPO','CENPF','TACC3','FAM64A','SMC4','CCNB2',
             'CKAP2L','CKAP2','AURKB','BUB1','KIF11','ANP32E','TUBB4B','GTSE1','KIF20B',
             'HJURP','CDCA3','HN1','CDC20','TTK','CDC25C','KIF2C','RANGAP1','NCAPD2',
             'DLGAP5','CDCA2','CDCA8','ECT2','KIF23','HMMR','AURKA','PSRC1','ANLN',
             'LBR','CKAP5','CENPE','CTCF','NEK2','G2E3','GAS2L3','CBX5','CENPA']

s_genes_mouse   = [g.capitalize() for g in s_genes]
g2m_genes_mouse = [g.capitalize() for g in g2m_genes]

differentiated_types = ['Paneth','Goblet','Tuft','EEC','Enterocyte']

# ── Control labels ──
REF_CTRL_LABEL = 'Positive Control: Ref cycling (Stem/TA, S)'
REF_NEG_LABEL  = 'Negative Control: Ref non-cycling (Stem/TA, G1)'

GROUP_ORDER = [
    REF_CTRL_LABEL,
    REF_NEG_LABEL,
   # 'Negative Control: RNA = Differentiated',
    'Group 1: Diff. DCM → Stem RNA (single)',
    'Group 2: Diff. DCM → TA RNA (single)',
    'Group 3: Diff. DCM → RNA hybrid w/ Stem',
    'Group 4: Diff. DCM → RNA hybrid w/ TA',
]

GROUP_COLORS = {
    REF_CTRL_LABEL:                            '#D2B48C',
    REF_NEG_LABEL:                             '#B0C4DE',
   # 'Negative Control: RNA = Differentiated':  '#95a5a6',
    'Group 1: Diff. DCM → Stem RNA (single)':  '#9370DB',
    'Group 2: Diff. DCM → TA RNA (single)':    '#C39BD3',
    'Group 3: Diff. DCM → RNA hybrid w/ Stem': '#e74c3c',
    'Group 4: Diff. DCM → RNA hybrid w/ TA':   '#3498db',
}

# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────
def ensure_bpw(adata):
    if 'batch_plate_well' not in adata.obs.columns:
        adata.obs['batch_plate_well'] = (
            adata.obs['Batch'].astype(str) + '_' +
            adata.obs['plate_well'].astype(str)
        )

def mean_expr(adata, genes):
    idx = [list(adata.var_names).index(g) for g in genes]
    X   = adata.X[:, idx]
    if hasattr(X, 'toarray'):
        X = X.toarray()
    return X.mean(axis=1).flatten()

def parse_sig(sig_str):
    if pd.isna(sig_str) or str(sig_str).strip() in ('', 'nan'):
        return []
    return [ct.strip().strip("'\" ")
            for ct in str(sig_str).split(';')
            if ct.strip().strip("'\" ") not in ('', 'nan', 'none', 'None')]

# ─────────────────────────────────────────────
# STEP 1: CELL CYCLE SCORES
# ─────────────────────────────────────────────
print("=" * 70)
print("STEP 1: CELL CYCLE SCORES")
print("=" * 70)

ensure_bpw(adata_combined)

# -- pDCM --
adata_dcm = adata_combined[adata_combined.obs['assay'] == 'pDCM'].copy()
ensure_bpw(adata_dcm)
s_dcm   = [g for g in s_genes_mouse   if g in adata_dcm.var_names]
g2m_dcm = [g for g in g2m_genes_mouse if g in adata_dcm.var_names]
print(f"pDCM  — S: {len(s_dcm)}, G2M: {len(g2m_dcm)}")

dcm_scores_df = pd.DataFrame({
    'batch_plate_well':   adata_dcm.obs['batch_plate_well'].values,
    'dcm_celltype':       adata_dcm.obs['ttest_gmm_label_confident_085']
                                   .astype(str).str.split('_').str[-1].values,
    'dcm_is_multipotent': adata_dcm.obs['is_multipotent'].values,
    'dcm_sig_celltypes':  adata_dcm.obs['significant_celltypes'].values,
    'Dox':                adata_dcm.obs['Dox'].values,
    'dcm_S':              mean_expr(adata_dcm, s_dcm),
    'dcm_G2M':            mean_expr(adata_dcm, g2m_dcm),
})
dcm_scores_df['dcm_combined'] = dcm_scores_df['dcm_S'] + dcm_scores_df['dcm_G2M']
print(f"pDCM scores: {len(dcm_scores_df)} cells")

# -- pRNA --
adata_rna = adata_combined[adata_combined.obs['assay'] == 'pRNA'].copy()
ensure_bpw(adata_rna)
s_rna   = [g for g in s_genes_mouse   if g in adata_rna.var_names]
g2m_rna = [g for g in g2m_genes_mouse if g in adata_rna.var_names]
print(f"pRNA  — S: {len(s_rna)}, G2M: {len(g2m_rna)}")

rna_scores_df = pd.DataFrame({
    'batch_plate_well':   adata_rna.obs['batch_plate_well'].values,
    'rna_celltype':       adata_rna.obs['leiden_annot_25']
                                   .astype(str).str.split('_').str[-1].values,
    'rna_is_multipotent': adata_rna.obs['is_multipotent'].values,
    'rna_sig_celltypes':  adata_rna.obs['significant_celltypes'].values,
    'rna_phase':          adata_rna.obs['phase'].astype(str).values
                          if 'phase' in adata_rna.obs.columns else 'Unknown',
    'rna_S':              mean_expr(adata_rna, s_rna),
    'rna_G2M':            mean_expr(adata_rna, g2m_rna),
})
rna_scores_df['rna_combined'] = rna_scores_df['rna_S'] + rna_scores_df['rna_G2M']
print(f"pRNA scores: {len(rna_scores_df)} cells")

# ─────────────────────────────────────────────
# STEP 2: BUILD REFERENCE CONTROLS
# Positive: pRNA Stem/TA cells in S phase
# Negative: pRNA Stem/TA cells in G1 phase
# Both are independent of pairing.
# ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 2: BUILDING REFERENCE CONTROLS")
print("=" * 70)

sc.tl.score_genes_cell_cycle(
    adata_rna, s_genes=s_genes_mouse, g2m_genes=g2m_genes_mouse
)

ref_ctrl_df = pd.DataFrame({
    'batch_plate_well': adata_rna.obs['batch_plate_well'].values,
    'rna_celltype':     adata_rna.obs['leiden_annot_25']
                                 .astype(str).str.split('_').str[-1].values,
    'phase':            adata_rna.obs['phase'].astype(str).values,
    'S_score':          adata_rna.obs['S_score'].values,
    'G2M_score':        adata_rna.obs['G2M_score'].values,
})
ref_ctrl_df['combined_score'] = ref_ctrl_df['S_score'] + ref_ctrl_df['G2M_score']

# Positive control: Stem/TA, S phase only
ref_ctrl_cycling = ref_ctrl_df[
    ref_ctrl_df['rna_celltype'].isin(['Stem', 'TA']) &
    ref_ctrl_df['phase'].isin(['S'])
].copy()
ref_ctrl_cycling['group']             = REF_CTRL_LABEL
ref_ctrl_cycling['dcm_celltype']      = 'ref'
ref_ctrl_cycling['rna_celltype_orig'] = ref_ctrl_cycling['rna_celltype']
ref_ctrl_cycling['Dox']               = 'ALL'

print(f"Positive control (S phase): {len(ref_ctrl_cycling)} cells")
print(ref_ctrl_cycling['rna_celltype'].value_counts().to_string())
print(ref_ctrl_cycling['phase'].value_counts().to_string())

# Negative control: Stem/TA, G1 phase
ref_ctrl_g1 = ref_ctrl_df[
    ref_ctrl_df['rna_celltype'].isin(['Stem', 'TA']) &
    ref_ctrl_df['phase'].isin(['G1'])
].copy()
ref_ctrl_g1['group']             = REF_NEG_LABEL
ref_ctrl_g1['dcm_celltype']      = 'ref'
ref_ctrl_g1['rna_celltype_orig'] = ref_ctrl_g1['rna_celltype']
ref_ctrl_g1['Dox']               = 'ALL'

print(f"\nNegative control (G1 phase): {len(ref_ctrl_g1)} cells")
print(ref_ctrl_g1['rna_celltype'].value_counts().to_string())
print(ref_ctrl_g1['phase'].value_counts().to_string())

# ─────────────────────────────────────────────
# STEP 3: BUILD PAIRED TABLE
# ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 3: BUILDING PAIRED TABLE")
print("=" * 70)

paired = dcm_scores_df.merge(rna_scores_df, on='batch_plate_well', how='inner')
print(f"Total paired cells: {len(paired)}")

# ─────────────────────────────────────────────
# STEP 4: ASSIGN TRANSITION GROUPS
# ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 4: ASSIGNING GROUPS")
print("=" * 70)

def get_groups(row):
    groups = []
    rna_ct    = row['rna_celltype']
    dcm_ct    = row['dcm_celltype']
    dcm_multi = row['dcm_is_multipotent'] == True
    rna_multi = row['rna_is_multipotent'] == True
    rna_sig   = parse_sig(row['rna_sig_celltypes'])

    rna_is_ta_single   = (not rna_multi) and (rna_ct == 'TA')
    rna_is_stem_single = (not rna_multi) and (rna_ct == 'Stem')
    rna_is_diff_single = (not rna_multi) and (rna_ct in differentiated_types)
    dcm_is_diff_single = (not dcm_multi) and (dcm_ct in differentiated_types)

   # if rna_is_diff_single:
   #     groups.append('Negative Control: RNA = Differentiated')
    if dcm_is_diff_single and rna_is_stem_single:
        groups.append('Group 1: Diff. DCM → Stem RNA (single)')
    if dcm_is_diff_single and rna_is_ta_single:
        groups.append('Group 2: Diff. DCM → TA RNA (single)')
    if dcm_is_diff_single and rna_multi and 'Stem' in rna_sig:
        groups.append('Group 3: Diff. DCM → RNA hybrid w/ Stem')
    if dcm_is_diff_single and rna_multi and 'TA' in rna_sig:
        groups.append('Group 4: Diff. DCM → RNA hybrid w/ TA')

    return groups

paired['groups']   = paired.apply(get_groups, axis=1)
paired['n_groups'] = paired['groups'].apply(len)

long_df = paired.explode('groups').rename(columns={'groups': 'group'})
long_df = long_df[long_df['group'].notna() & (long_df['group'] != '')].copy()

print("Group counts:")
print(long_df['group'].value_counts().to_string())

# Build DCM and RNA long tables
dcm_long = long_df.rename(columns={
    'dcm_S': 'S_score', 'dcm_G2M': 'G2M_score', 'dcm_combined': 'combined_score'
})[['batch_plate_well','dcm_celltype','rna_celltype',
    'group','Dox','S_score','G2M_score','combined_score']].copy()

rna_long = long_df.rename(columns={
    'rna_S': 'S_score', 'rna_G2M': 'G2M_score', 'rna_combined': 'combined_score'
})[['batch_plate_well','dcm_celltype','rna_celltype',
    'group','Dox','S_score','G2M_score','combined_score']].copy()

# Append both reference controls to both assay tables
ref_for_append = ref_ctrl_cycling[
    ['batch_plate_well','group','S_score','G2M_score','combined_score']
].copy()
ref_for_append['dcm_celltype'] = 'ref'
ref_for_append['rna_celltype'] = ref_ctrl_cycling['rna_celltype_orig'].values
ref_for_append['Dox']          = 'ALL'

ref_neg_for_append = ref_ctrl_g1[
    ['batch_plate_well','group','S_score','G2M_score','combined_score']
].copy()
ref_neg_for_append['dcm_celltype'] = 'ref'
ref_neg_for_append['rna_celltype'] = ref_ctrl_g1['rna_celltype_orig'].values
ref_neg_for_append['Dox']          = 'ALL'

dcm_long = pd.concat([dcm_long, ref_for_append, ref_neg_for_append], ignore_index=True)
rna_long = pd.concat([rna_long, ref_for_append, ref_neg_for_append], ignore_index=True)

dcm_long = dcm_long[dcm_long['combined_score'].notna()].copy()
rna_long = rna_long[rna_long['combined_score'].notna()].copy()

# ─────────────────────────────────────────────
# STEP 5: PLOT FUNCTION
# ─────────────────────────────────────────────
score_info = {
    'S_score':        'S Phase\nMean Expression',
    'G2M_score':      'G2M Phase\nMean Expression',
    'combined_score': 'Combined (S + G2M)\nMean Expression',
}

pos_ctrl_group = REF_CTRL_LABEL
neg_ctrl_group = REF_NEG_LABEL

def make_boxplot_figure(plot_df, assay_label, dox_label, filename):
    present_groups = [g for g in GROUP_ORDER if g in plot_df['group'].values]
    if not present_groups:
        print(f"  No data for {assay_label} Dox={dox_label}, skipping.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(24, 8))
    for ax, (col, ylabel) in zip(axes, score_info.items()):
        sns.boxplot(
            data=plot_df, x='group', y=col,
            order=present_groups, palette=GROUP_COLORS,
            ax=ax, width=0.55,
            flierprops=dict(marker='o', markersize=2, alpha=0.4),
        )
        ylim  = ax.get_ylim()
        y_top = ylim[1]

        # n labels
        for i, grp in enumerate(present_groups):
            n = len(plot_df[plot_df['group'] == grp])
            ax.text(i, y_top * 1.01, f'n={n}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

        pos_vals = plot_df[plot_df['group'] == pos_ctrl_group][col]
        neg_vals = plot_df[plot_df['group'] == neg_ctrl_group][col]

        # Two annotation rows: top vs positive ctrl, middle vs negative ctrl
        y_annot_pos = y_top * 1.10
        y_annot_neg = y_top * 1.19

        for i, grp in enumerate(present_groups):
            if grp in (pos_ctrl_group, neg_ctrl_group):
                continue
            test_vals = plot_df[plot_df['group'] == grp][col]
            if len(test_vals) <= 1:
                continue

            # vs positive control
            if pos_ctrl_group in present_groups and len(pos_vals) > 1:
                pos_i   = present_groups.index(pos_ctrl_group)
                _, pval = stats.ttest_ind(test_vals, pos_vals)
                stars   = ('***' if pval < 0.001 else '**' if pval < 0.01
                           else '*' if pval < 0.05 else 'ns')
                x_mid   = (i + pos_i) / 2
                ax.annotate('', xy=(i, y_annot_pos), xytext=(pos_i, y_annot_pos),
                            arrowprops=dict(arrowstyle='-', color='#8B4513', lw=0.8))
                ax.text(x_mid, y_annot_pos, stars,
                        ha='center', va='bottom', fontsize=9,
                        fontweight='bold', color='#8B4513')

            # vs negative control (G1)
            if neg_ctrl_group in present_groups and len(neg_vals) > 1:
                neg_i   = present_groups.index(neg_ctrl_group)
                _, pval = stats.ttest_ind(test_vals, neg_vals)
                stars   = ('***' if pval < 0.001 else '**' if pval < 0.01
                           else '*' if pval < 0.05 else 'ns')
                x_mid   = (i + neg_i) / 2
                ax.annotate('', xy=(i, y_annot_neg), xytext=(neg_i, y_annot_neg),
                            arrowprops=dict(arrowstyle='-', color='#4682B4', lw=0.8))
                ax.text(x_mid, y_annot_neg, stars,
                        ha='center', va='bottom', fontsize=9,
                        fontweight='bold', color='#4682B4')

        ax.set_xlabel('')
        ax.set_ylabel(ylabel, fontsize=11, fontweight='bold')
        ax.set_title(ylabel.replace('\n', ' '), fontsize=12, fontweight='bold')
        ax.tick_params(axis='x', rotation=35, labelsize=8)
        ax.set_ylim(ylim[0], y_annot_neg * 1.10)

        # Legend for annotation colours
        ax.text(0.01, 0.99,
                '— vs Positive ctrl (S phase)   — vs Negative ctrl (G1)',
                transform=ax.transAxes, fontsize=6,
                color='gray', va='top', ha='left')
        ax.text(0.01, 0.95,
                '* cells may appear in multiple groups',
                transform=ax.transAxes, fontsize=6,
                color='gray', va='top', ha='left')

    plt.suptitle(
        f'Cell Cycle Gene Expression by Group\n'
        f'({assay_label} counts | Dox = {dox_label} | '
        f't-tests vs Pos ctrl (S) and Neg ctrl (G1))',
        fontsize=13, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.savefig(f'{score_dir}/{filename}', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {filename}")

# ─────────────────────────────────────────────
# STEP 6: GENERATE FIGURES
# Always include BOTH reference controls in every per-Dox subset
# ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 6: GENERATING FIGURES")
print("=" * 70)

dox_values = sorted(long_df['Dox'].dropna().unique())

for assay_label, plot_df_full in [('pDCM', dcm_long), ('pRNA', rna_long)]:
    print(f"\n--- {assay_label} ---")
    make_boxplot_figure(
        plot_df_full, assay_label, 'ALL',
        f'cell_cycle_groups_{assay_label}_Dox_ALL_095.pdf'
    )
    for dox in dox_values:
        subset = pd.concat([
            plot_df_full[plot_df_full['Dox'] == dox],
            plot_df_full[plot_df_full['group'] == pos_ctrl_group],  # positive ctrl
            plot_df_full[plot_df_full['group'] == neg_ctrl_group],  # negative ctrl (G1)
        ], ignore_index=True).drop_duplicates()
        dox_str = str(dox).replace(' ', '_').replace('/', '-')
        make_boxplot_figure(
            subset, assay_label, str(dox),
            f'cell_cycle_groups_{assay_label}_Dox_{dox_str}_095.pdf'
        )

# ─────────────────────────────────────────────
# STEP 7: SUMMARY STATISTICS
# t-tests vs both positive and negative controls
# ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 7: SUMMARY STATISTICS")
print("=" * 70)

for assay_label, plot_df_full in [('DCM', dcm_long), ('RNA', rna_long)]:
    all_rows = []
    for dox_label, subset in (
        [('ALL', plot_df_full)] +
        [(str(d), pd.concat([
            plot_df_full[plot_df_full['Dox'] == d],
            plot_df_full[plot_df_full['group'] == pos_ctrl_group],
            plot_df_full[plot_df_full['group'] == neg_ctrl_group],
        ], ignore_index=True).drop_duplicates())
         for d in dox_values]
    ):
        pos_vals = subset[subset['group'] == pos_ctrl_group]['combined_score']
        neg_vals = subset[subset['group'] == neg_ctrl_group]['combined_score']

        for grp in GROUP_ORDER:
            sub = subset[subset['group'] == grp]
            if len(sub) == 0:
                continue

            # t-test vs positive control
            if grp != pos_ctrl_group and len(sub) > 1 and len(pos_vals) > 1:
                _, pval_pos = stats.ttest_ind(sub['combined_score'], pos_vals)
            else:
                pval_pos = np.nan

            # t-test vs negative control (G1)
            if grp != neg_ctrl_group and len(sub) > 1 and len(neg_vals) > 1:
                _, pval_neg = stats.ttest_ind(sub['combined_score'], neg_vals)
            else:
                pval_neg = np.nan

            all_rows.append({
                'Dox':                        dox_label,
                'Group':                      grp,
                'N':                          len(sub),
                'S_mean':                     sub['S_score'].mean(),
                'S_median':                   sub['S_score'].median(),
                'G2M_mean':                   sub['G2M_score'].mean(),
                'G2M_median':                 sub['G2M_score'].median(),
                'Combined_mean':              sub['combined_score'].mean(),
                'Combined_median':            sub['combined_score'].median(),
                'Combined_std':               sub['combined_score'].std(),
                'ttest_vs_pos_ctrl_pval':     pval_pos,
                'ttest_vs_neg_ctrl_G1_pval':  pval_neg,
            })

    summary_df = pd.DataFrame(all_rows)
    print(f"\n--- {assay_label} ---")
    print(summary_df.to_string(index=False))
    summary_df.to_csv(f'{score_dir}/cell_cycle_groups_{assay_label}_stats_095.csv', index=False)
    plot_df_full.to_csv(f'{score_dir}/cell_cycle_groups_{assay_label}_data_095.csv', index=False)

paired.to_csv(f'{score_dir}/cell_cycle_groups_master_095.csv', index=False)
print("\nDone!")

## Cell_Cycle_Secretory_Absorptive

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scanpy as sc
from scipy import stats
from itertools import combinations
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/gmm_085/cell_cycle/'
output_dir = score_dir

# Cell cycle genes from Tirosh et al. 2016
s_genes = ['MCM5', 'PCNA', 'TYMS', 'FEN1', 'MCM2', 'MCM4', 'RRM1', 'UNG', 'GINS2', 'MCM6', 
           'CDCA7', 'DTL', 'PRIM1', 'UHRF1', 'MLF1IP', 'HELLS', 'RFC2', 'RPA2', 'NASP', 
           'RAD51AP1', 'GMNN', 'WDR76', 'SLBP', 'CCNE2', 'UBR7', 'POLD3', 'MSH2', 'ATAD2', 
           'RAD51', 'RRM2', 'CDC45', 'CDC6', 'EXO1', 'TIPIN', 'DSCC1', 'BLM', 'CASP8AP2', 
           'USP1', 'CLSPN', 'POLA1', 'CHAF1B', 'BRIP1', 'E2F8']

g2m_genes = ['HMGB2', 'CDK1', 'NUSAP1', 'UBE2C', 'BIRC5', 'TPX2', 'TOP2A', 'NDC80', 'CKS2', 
             'NUF2', 'CKS1B', 'MKI67', 'TMPO', 'CENPF', 'TACC3', 'FAM64A', 'SMC4', 'CCNB2', 
             'CKAP2L', 'CKAP2', 'AURKB', 'BUB1', 'KIF11', 'ANP32E', 'TUBB4B', 'GTSE1', 'KIF20B', 
             'HJURP', 'CDCA3', 'HN1', 'CDC20', 'TTK', 'CDC25C', 'KIF2C', 'RANGAP1', 'NCAPD2', 
             'DLGAP5', 'CDCA2', 'CDCA8', 'ECT2', 'KIF23', 'HMMR', 'AURKA', 'PSRC1', 'ANLN', 
             'LBR', 'CKAP5', 'CENPE', 'CTCF', 'NEK2', 'G2E3', 'GAS2L3', 'CBX5', 'CENPA']

# Convert to mouse gene names (capitalize first letter only)
s_genes_mouse = [gene.capitalize() for gene in s_genes]
g2m_genes_mouse = [gene.capitalize() for gene in g2m_genes]

# Define cell type groups
secretory_types = ['Paneth', 'Goblet', 'Tuft', 'EEC']
absorptive_types = ['Enterocyte']
stem_ta_types = ['Stem', 'TA']

print("="*80)
print("CALCULATING CELL CYCLE SCORES WITH BACKGROUND CONTROL")
print("="*80)

# -----------------------------
# CALCULATE CELL CYCLE SCORES ON DCM DATA
# -----------------------------
print("\n" + "="*80)
print("CALCULATING CELL CYCLE SCORES FROM DCM CELLS")
print("="*80)

# Filter to only DCM cells
adata_dcm = adata_combined[adata_combined.obs['assay'] == 'pDCM'].copy()
print(f"\nTotal DCM cells: {len(adata_dcm)}")

# Calculate cell cycle scores using scanpy
print("\nRunning scanpy cell cycle scoring...")
sc.tl.score_genes_cell_cycle(adata_dcm, s_genes=s_genes_mouse, g2m_genes=g2m_genes_mouse)

print(f"\nCell cycle scores calculated:")
print(f"  S_score - Mean: {adata_dcm.obs['S_score'].mean():.4f}, "
      f"Range: [{adata_dcm.obs['S_score'].min():.4f}, {adata_dcm.obs['S_score'].max():.4f}]")
print(f"  G2M_score - Mean: {adata_dcm.obs['G2M_score'].mean():.4f}, "
      f"Range: [{adata_dcm.obs['G2M_score'].min():.4f}, {adata_dcm.obs['G2M_score'].max():.4f}]")

# Check which genes were found
s_genes_found = [g for g in s_genes_mouse if g in adata_dcm.var_names]
g2m_genes_found = [g for g in g2m_genes_mouse if g in adata_dcm.var_names]
print(f"\nGenes used:")
print(f"  S phase: {len(s_genes_found)}/{len(s_genes_mouse)} genes")
print(f"  G2M phase: {len(g2m_genes_found)}/{len(g2m_genes_mouse)} genes")

# -----------------------------
# PREPARE DATA
# -----------------------------
print("\n" + "="*80)
print("PREPARING PAIRED DATA")
print("="*80)

# Create batch_plate_well identifier
adata_combined.obs['batch_plate_well'] = (
    adata_combined.obs['Batch'].astype(str) + '_' + 
    adata_combined.obs['plate_well'].astype(str)
)

# Get DCM data with cell cycle scores
pDCM_data = pd.DataFrame({
    'batch_plate_well': adata_dcm.obs['batch_plate_well'].values,
    'dcm_celltype': adata_dcm.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'S_score': adata_dcm.obs['S_score'].values,
    'G2M_score': adata_dcm.obs['G2M_score'].values,
    'phase': adata_dcm.obs['phase'].values
})

# Extract cell type from labels
pDCM_data['dcm_celltype'] = pDCM_data['dcm_celltype'].str.split('_').str[-1]

# Combined score
pDCM_data['combined_score'] = pDCM_data['S_score'] + pDCM_data['G2M_score']

# Get RNA data
rna_mask = adata_combined.obs['assay'] == 'pRNA'
pRNA_data = pd.DataFrame({
    'batch_plate_well': adata_combined.obs[rna_mask]['batch_plate_well'].values,
    'rna_celltype': adata_combined.obs[rna_mask]['leiden_annot_25'].astype(str).values
})

# Extract cell type from labels
pRNA_data['rna_celltype'] = pRNA_data['rna_celltype'].str.split('_').str[-1]

# Merge to pair cells
paired_cells = pDCM_data.merge(pRNA_data, on='batch_plate_well', how='inner')

print(f"Total paired cells: {len(paired_cells)}")

print("\nCell cycle phase distribution in DCM cells:")
print(paired_cells['phase'].value_counts())

# =============================================================================
# PLOT 1: SECRETORY TRANSITIONS (EXCLUDING STEM→ENTEROCYTE)
# =============================================================================

print("\n" + "="*80)
print("PREPARING PLOT 1: SECRETORY TRANSITIONS")
print("="*80)

def classify_transition_plot1(row):
    dcm = row['dcm_celltype']
    rna = row['rna_celltype']
    
    # Secretory → Secretory (control)
    if dcm in secretory_types and rna in secretory_types:
        return 'Secretory→Secretory', f'{dcm}→{rna}'
    
    # Individual Secretory → Enterocyte
    elif dcm == 'Paneth' and rna == 'Enterocyte':
        return 'Paneth→Enterocyte', 'Paneth→Enterocyte'
    elif dcm == 'Goblet' and rna == 'Enterocyte':
        return 'Goblet→Enterocyte', 'Goblet→Enterocyte'
    elif dcm == 'Tuft' and rna == 'Enterocyte':
        return 'Tuft→Enterocyte', 'Tuft→Enterocyte'
    elif dcm == 'EEC' and rna == 'Enterocyte':
        return 'EEC→Enterocyte', 'EEC→Enterocyte'
    
    # TA → Enterocyte (control)
    elif dcm == 'TA' and rna == 'Enterocyte':
        return 'TA→Enterocyte', 'TA→Enterocyte'
    
    else:
        return 'Other', f'{dcm}→{rna}'

paired_cells['transition_plot1'], paired_cells['specific_transition_plot1'] = zip(
    *paired_cells.apply(classify_transition_plot1, axis=1)
)

# Filter to relevant groups for plot 1 (EXCLUDING Stem→Enterocyte)
relevant_groups_plot1 = ['Secretory→Secretory', 'Paneth→Enterocyte', 'Goblet→Enterocyte', 
                         'Tuft→Enterocyte', 'EEC→Enterocyte', 'TA→Enterocyte']
filtered_cells_plot1 = paired_cells[paired_cells['transition_plot1'].isin(relevant_groups_plot1)].copy()

print(f"\nFiltered to {len(filtered_cells_plot1)} cells for Plot 1")
print("\nTransition counts:")
print(filtered_cells_plot1['transition_plot1'].value_counts())

# Create Plot 1
print("\nCreating Plot 1...")

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Define colors
transition_colors_plot1 = {
    'Secretory→Secretory': '#95a5a6',  # Gray (control)
    'Paneth→Enterocyte': '#DC143C',    # Red
    'Goblet→Enterocyte': '#2E8B57',    # Green
    'Tuft→Enterocyte': '#FF1493',      # Pink
    'EEC→Enterocyte': '#4169E1',       # Blue
    'TA→Enterocyte': '#3498db'         # Light Blue (control)
}

# Order for plot 1
transition_order_plot1 = ['Secretory→Secretory', 'Paneth→Enterocyte', 'Goblet→Enterocyte', 
                          'Tuft→Enterocyte', 'EEC→Enterocyte', 'TA→Enterocyte']

# Plot 1: Combined score
sns.boxplot(data=filtered_cells_plot1, x='transition_plot1', y='combined_score',
            order=transition_order_plot1, palette=transition_colors_plot1, ax=axes[0])
axes[0].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Combined Cell Cycle Score\n(S + G2M)', fontsize=11, fontweight='bold')
axes[0].set_title('Combined Cell Cycle Score', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45, labelsize=9)

# Add n values
for i, trans in enumerate(transition_order_plot1):
    n = len(filtered_cells_plot1[filtered_cells_plot1['transition_plot1'] == trans])
    axes[0].text(i, axes[0].get_ylim()[1] * 0.95, f'n={n}', 
                ha='center', fontsize=8, fontweight='bold')

# T-test vs Secretory→Secretory control
sec_sec_vals = filtered_cells_plot1[filtered_cells_plot1['transition_plot1'] == 'Secretory→Secretory']['combined_score']
for i, trans in enumerate(transition_order_plot1[1:], start=1):  # Skip the control itself
    trans_vals = filtered_cells_plot1[filtered_cells_plot1['transition_plot1'] == trans]['combined_score']
    if len(trans_vals) > 0 and len(sec_sec_vals) > 0:
        tstat, pval = stats.ttest_ind(trans_vals, sec_sec_vals)
        if pval < 0.05:
            sig_marker = '***' if pval < 0.001 else '**' if pval < 0.01 else '*'
            axes[0].text(i, axes[0].get_ylim()[1] * 0.88, sig_marker, 
                        ha='center', fontsize=12, fontweight='bold')

# Plot 2: S score
sns.boxplot(data=filtered_cells_plot1, x='transition_plot1', y='S_score',
            order=transition_order_plot1, palette=transition_colors_plot1, ax=axes[1])
axes[1].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[1].set_ylabel('S Phase Score', fontsize=11, fontweight='bold')
axes[1].set_title('S Phase Score', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45, labelsize=9)

# Plot 3: G2M score
sns.boxplot(data=filtered_cells_plot1, x='transition_plot1', y='G2M_score',
            order=transition_order_plot1, palette=transition_colors_plot1, ax=axes[2])
axes[2].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[2].set_ylabel('G2M Phase Score', fontsize=11, fontweight='bold')
axes[2].set_title('G2M Phase Score', fontsize=12, fontweight='bold')
axes[2].tick_params(axis='x', rotation=45, labelsize=9)

plt.suptitle('Cell Cycle Scores: Secretory→Enterocyte Transitions (Scanpy Scoring, DCM Expression)', 
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{output_dir}/cell_cycle_scores_secretory_plot1.pdf', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: cell_cycle_scores_secretory_plot1.pdf")

# =============================================================================
# PLOT 2: TUFT-FOCUSED TRANSITIONS
# =============================================================================

print("\n" + "="*80)
print("PREPARING PLOT 2: TUFT-FOCUSED TRANSITIONS")
print("="*80)

def classify_transition_plot2(row):
    dcm = row['dcm_celltype']
    rna = row['rna_celltype']
    
    # Tuft to Stem
    if dcm == 'Tuft' and rna == 'Stem':
        return 'Tuft→Stem', 'Tuft→Stem'
    
    # Tuft to TA
    elif dcm == 'Tuft' and rna == 'TA':
        return 'Tuft→TA', 'Tuft→TA'
    
    # Tuft to Enterocyte
    elif dcm == 'Tuft' and rna == 'Enterocyte':
        return 'Tuft→Enterocyte', 'Tuft→Enterocyte'
    
    # Tuft to other secretory types
    elif dcm == 'Tuft' and rna == 'Paneth':
        return 'Tuft→Paneth', 'Tuft→Paneth'
    elif dcm == 'Tuft' and rna == 'Goblet':
        return 'Tuft→Goblet', 'Tuft→Goblet'
    elif dcm == 'Tuft' and rna == 'Tuft':
        return 'Tuft→Tuft', 'Tuft→Tuft'
    elif dcm == 'Tuft' and rna == 'EEC':
        return 'Tuft→EEC', 'Tuft→EEC'
    
    # TA → Enterocyte (control)
    elif dcm == 'TA' and rna == 'Enterocyte':
        return 'TA→Enterocyte', 'TA→Enterocyte'
    
    else:
        return 'Other', f'{dcm}→{rna}'

paired_cells['transition_plot2'], paired_cells['specific_transition_plot2'] = zip(
    *paired_cells.apply(classify_transition_plot2, axis=1)
)

# Define the exact order for plot 2
transition_order_plot2 = [
    'Tuft→Stem', 
    'Tuft→TA', 
    'Tuft→Enterocyte', 
    'Tuft→Paneth', 
    'Tuft→Goblet', 
    'Tuft→Tuft', 
    'Tuft→EEC', 
    'TA→Enterocyte'
]

# Filter to these transitions
filtered_cells_plot2 = paired_cells[paired_cells['transition_plot2'].isin(transition_order_plot2)].copy()

# Only include transitions that exist in the data
transition_order_plot2 = [t for t in transition_order_plot2 if t in filtered_cells_plot2['transition_plot2'].values]

print(f"\nFiltered to {len(filtered_cells_plot2)} cells for Plot 2")
print("\nTransition counts:")
print(filtered_cells_plot2['transition_plot2'].value_counts())

# Create Plot 2
print("\nCreating Plot 2...")

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Define colors for plot 2
transition_colors_plot2 = {
    'Tuft→Stem': '#9370DB',        # Purple
    'Tuft→TA': '#D2B48C',          # Tan
    'Tuft→Enterocyte': '#FF8C00',  # Orange
    'Tuft→Paneth': '#DC143C',      # Red
    'Tuft→Goblet': '#2E8B57',      # Green
    'Tuft→Tuft': '#FF1493',        # Pink
    'Tuft→EEC': '#4169E1',         # Blue
    'TA→Enterocyte': '#3498db'     # Light Blue (control)
}

# Plot 1: Combined score
sns.boxplot(data=filtered_cells_plot2, x='transition_plot2', y='combined_score',
            order=transition_order_plot2, palette=transition_colors_plot2, ax=axes[0])
axes[0].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Combined Cell Cycle Score\n(S + G2M)', fontsize=11, fontweight='bold')
axes[0].set_title('Combined Cell Cycle Score', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45, labelsize=9)

# Add n values
for i, trans in enumerate(transition_order_plot2):
    n = len(filtered_cells_plot2[filtered_cells_plot2['transition_plot2'] == trans])
    axes[0].text(i, axes[0].get_ylim()[1] * 0.95, f'n={n}', 
                ha='center', fontsize=8, fontweight='bold')

# T-test vs TA→Enterocyte control
ta_ent_vals = filtered_cells_plot2[filtered_cells_plot2['transition_plot2'] == 'TA→Enterocyte']['combined_score']
for i, trans in enumerate(transition_order_plot2[:-1]):  # Skip the control itself
    trans_vals = filtered_cells_plot2[filtered_cells_plot2['transition_plot2'] == trans]['combined_score']
    if len(trans_vals) > 0 and len(ta_ent_vals) > 0:
        tstat, pval = stats.ttest_ind(trans_vals, ta_ent_vals)
        if pval < 0.05:
            sig_marker = '***' if pval < 0.001 else '**' if pval < 0.01 else '*'
            axes[0].text(i, axes[0].get_ylim()[1] * 0.88, sig_marker, 
                        ha='center', fontsize=12, fontweight='bold')

# Plot 2: S score
sns.boxplot(data=filtered_cells_plot2, x='transition_plot2', y='S_score',
            order=transition_order_plot2, palette=transition_colors_plot2, ax=axes[1])
axes[1].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[1].set_ylabel('S Phase Score', fontsize=11, fontweight='bold')
axes[1].set_title('S Phase Score', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45, labelsize=9)

# Plot 3: G2M score
sns.boxplot(data=filtered_cells_plot2, x='transition_plot2', y='G2M_score',
            order=transition_order_plot2, palette=transition_colors_plot2, ax=axes[2])
axes[2].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[2].set_ylabel('G2M Phase Score', fontsize=11, fontweight='bold')
axes[2].set_title('G2M Phase Score', fontsize=12, fontweight='bold')
axes[2].tick_params(axis='x', rotation=45, labelsize=9)

plt.suptitle('Cell Cycle Scores: Tuft Cell Transitions (Scanpy Scoring, DCM Expression)', 
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{output_dir}/cell_cycle_scores_tuft_plot2.pdf', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: cell_cycle_scores_tuft_plot2.pdf")

# =============================================================================
# PLOT 3: TRANSITIONS TO TA
# =============================================================================

print("\n" + "="*80)
print("PREPARING PLOT 3: TRANSITIONS TO TA")
print("="*80)

def classify_transition_plot3(row):
    dcm = row['dcm_celltype']
    rna = row['rna_celltype']
    
    # All transitions to TA
    if rna == 'TA':
        if dcm == 'Stem':
            return 'Stem→TA', 'Stem→TA'
        elif dcm == 'TA':
            return 'TA→TA', 'TA→TA'
        elif dcm == 'Enterocyte':
            return 'Enterocyte→TA', 'Enterocyte→TA'
        elif dcm == 'Paneth':
            return 'Paneth→TA', 'Paneth→TA'
        elif dcm == 'Goblet':
            return 'Goblet→TA', 'Goblet→TA'
        elif dcm == 'Tuft':
            return 'Tuft→TA', 'Tuft→TA'
        elif dcm == 'EEC':
            return 'EEC→TA', 'EEC→TA'
    
    # TA → Enterocyte (control)
    elif dcm == 'TA' and rna == 'Enterocyte':
        return 'TA→Enterocyte', 'TA→Enterocyte'
    
    return 'Other', f'{dcm}→{rna}'

paired_cells['transition_plot3'], paired_cells['specific_transition_plot3'] = zip(
    *paired_cells.apply(classify_transition_plot3, axis=1)
)

# Define the exact order for plot 3 - NOW INCLUDING TA→Enterocyte
transition_order_plot3 = [
    'Stem→TA',
    'TA→TA',
    'Enterocyte→TA',
    'Paneth→TA',
    'Goblet→TA',
    'Tuft→TA',
    'EEC→TA',
    'TA→Enterocyte'
]

# Filter to these transitions
filtered_cells_plot3 = paired_cells[paired_cells['transition_plot3'].isin(transition_order_plot3)].copy()

# Only include transitions that exist in the data
transition_order_plot3 = [t for t in transition_order_plot3 if t in filtered_cells_plot3['transition_plot3'].values]

print(f"\nFiltered to {len(filtered_cells_plot3)} cells for Plot 3")
print("\nTransition counts:")
print(filtered_cells_plot3['transition_plot3'].value_counts())

# Create Plot 3
print("\nCreating Plot 3...")

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Define colors for plot 3
transition_colors_plot3 = {
    'Stem→TA': '#9370DB',          # Purple
    'TA→TA': '#D2B48C',            # Tan
    'Enterocyte→TA': '#FF8C00',    # Orange
    'Paneth→TA': '#DC143C',        # Red
    'Goblet→TA': '#2E8B57',        # Green
    'Tuft→TA': '#FF1493',          # Pink
    'EEC→TA': '#4169E1',           # Blue
    'TA→Enterocyte': '#3498db'     # Light Blue (control)
}

# Plot 1: Combined score
sns.boxplot(data=filtered_cells_plot3, x='transition_plot3', y='combined_score',
            order=transition_order_plot3, palette=transition_colors_plot3, ax=axes[0])
axes[0].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Combined Cell Cycle Score\n(S + G2M)', fontsize=11, fontweight='bold')
axes[0].set_title('Combined Cell Cycle Score', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45, labelsize=9)

# Add n values
for i, trans in enumerate(transition_order_plot3):
    n = len(filtered_cells_plot3[filtered_cells_plot3['transition_plot3'] == trans])
    axes[0].text(i, axes[0].get_ylim()[1] * 0.95, f'n={n}', 
                ha='center', fontsize=8, fontweight='bold')

# T-test vs TA→Enterocyte control
ta_ent_vals_plot3 = filtered_cells_plot3[filtered_cells_plot3['transition_plot3'] == 'TA→Enterocyte']['combined_score']
for i, trans in enumerate(transition_order_plot3[:-1]):  # Skip the control itself
    trans_vals = filtered_cells_plot3[filtered_cells_plot3['transition_plot3'] == trans]['combined_score']
    if len(trans_vals) > 0 and len(ta_ent_vals_plot3) > 0:
        tstat, pval = stats.ttest_ind(trans_vals, ta_ent_vals_plot3)
        if pval < 0.05:
            sig_marker = '***' if pval < 0.001 else '**' if pval < 0.01 else '*'
            axes[0].text(i, axes[0].get_ylim()[1] * 0.88, sig_marker, 
                        ha='center', fontsize=12, fontweight='bold')

# Plot 2: S score
sns.boxplot(data=filtered_cells_plot3, x='transition_plot3', y='S_score',
            order=transition_order_plot3, palette=transition_colors_plot3, ax=axes[1])
axes[1].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[1].set_ylabel('S Phase Score', fontsize=11, fontweight='bold')
axes[1].set_title('S Phase Score', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45, labelsize=9)

# Plot 3: G2M score
sns.boxplot(data=filtered_cells_plot3, x='transition_plot3', y='G2M_score',
            order=transition_order_plot3, palette=transition_colors_plot3, ax=axes[2])
axes[2].set_xlabel('Transition', fontsize=11, fontweight='bold')
axes[2].set_ylabel('G2M Phase Score', fontsize=11, fontweight='bold')
axes[2].set_title('G2M Phase Score', fontsize=12, fontweight='bold')
axes[2].tick_params(axis='x', rotation=45, labelsize=9)

plt.suptitle('Cell Cycle Scores: Transitions to TA (Scanpy Scoring, DCM Expression)', 
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{output_dir}/cell_cycle_scores_to_TA_plot3.pdf', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: cell_cycle_scores_to_TA_plot3.pdf")

# -----------------------------
# STATISTICS FOR ALL PLOTS
# -----------------------------
print("\n" + "="*80)
print("CALCULATING STATISTICS")
print("="*80)

def get_significance(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return 'ns'

# Statistics for Plot 1
print("\nPlot 1 Statistics:")
plot1_stats = []
sec_sec_vals = filtered_cells_plot1[filtered_cells_plot1['transition_plot1'] == 'Secretory→Secretory']['combined_score']

for trans in transition_order_plot1:
    subset = filtered_cells_plot1[filtered_cells_plot1['transition_plot1'] == trans]
    
    if len(subset) > 0:
        # T-test vs Secretory→Secretory if not the control itself
        if trans != 'Secretory→Secretory' and len(sec_sec_vals) > 0:
            tstat, pval = stats.ttest_ind(subset['combined_score'], sec_sec_vals)
        else:
            tstat, pval = np.nan, np.nan
        
        plot1_stats.append({
            'Transition': trans,
            'N': len(subset),
            'Combined_Mean': subset['combined_score'].mean(),
            'Combined_Median': subset['combined_score'].median(),
            'Combined_Std': subset['combined_score'].std(),
            'S_Mean': subset['S_score'].mean(),
            'G2M_Mean': subset['G2M_score'].mean(),
            't_stat_vs_control': tstat,
            'p_value_vs_control': pval,
            'significance': get_significance(pval) if not np.isnan(pval) else '-'
        })

plot1_stats_df = pd.DataFrame(plot1_stats)
plot1_stats_df.to_csv(f'{output_dir}/cell_cycle_plot1_statistics.csv', index=False)
print(f"✅ Saved: {output_dir}/cell_cycle_plot1_statistics.csv")

# Statistics for Plot 2
print("\nPlot 2 Statistics:")
plot2_stats = []
ta_ent_vals = filtered_cells_plot2[filtered_cells_plot2['transition_plot2'] == 'TA→Enterocyte']['combined_score']

for trans in transition_order_plot2:
    subset = filtered_cells_plot2[filtered_cells_plot2['transition_plot2'] == trans]
    
    if len(subset) > 0:
        # T-test vs TA→Enterocyte if not the control itself
        if trans != 'TA→Enterocyte' and len(ta_ent_vals) > 0:
            tstat, pval = stats.ttest_ind(subset['combined_score'], ta_ent_vals)
        else:
            tstat, pval = np.nan, np.nan
        
        plot2_stats.append({
            'Transition': trans,
            'N': len(subset),
            'Combined_Mean': subset['combined_score'].mean(),
            'Combined_Median': subset['combined_score'].median(),
            'Combined_Std': subset['combined_score'].std(),
            'S_Mean': subset['S_score'].mean(),
            'G2M_Mean': subset['G2M_score'].mean(),
            't_stat_vs_control': tstat,
            'p_value_vs_control': pval,
            'significance': get_significance(pval) if not np.isnan(pval) else '-'
        })

plot2_stats_df = pd.DataFrame(plot2_stats)
plot2_stats_df.to_csv(f'{output_dir}/cell_cycle_plot2_statistics.csv', index=False)
print(f"✅ Saved: {output_dir}/cell_cycle_plot2_statistics.csv")

# Statistics for Plot 3
print("\nPlot 3 Statistics:")
plot3_stats = []

# Use TA→Enterocyte as control
ta_ent_vals_plot3 = filtered_cells_plot3[filtered_cells_plot3['transition_plot3'] == 'TA→Enterocyte']['combined_score']

for trans in transition_order_plot3:
    subset = filtered_cells_plot3[filtered_cells_plot3['transition_plot3'] == trans]
    
    if len(subset) > 0:
        # T-test vs TA→Enterocyte if not the control itself
        if trans != 'TA→Enterocyte' and len(ta_ent_vals_plot3) > 0:
            tstat, pval = stats.ttest_ind(subset['combined_score'], ta_ent_vals_plot3)
        else:
            tstat, pval = np.nan, np.nan
        
        plot3_stats.append({
            'Transition': trans,
            'N': len(subset),
            'Combined_Mean': subset['combined_score'].mean(),
            'Combined_Median': subset['combined_score'].median(),
            'Combined_Std': subset['combined_score'].std(),
            'S_Mean': subset['S_score'].mean(),
            'G2M_Mean': subset['G2M_score'].mean(),
            't_stat_vs_control': tstat,
            'p_value_vs_control': pval,
            'significance': get_significance(pval) if not np.isnan(pval) else '-'
        })

plot3_stats_df = pd.DataFrame(plot3_stats)
plot3_stats_df.to_csv(f'{output_dir}/cell_cycle_plot3_statistics.csv', index=False)
print(f"✅ Saved: {output_dir}/cell_cycle_plot3_statistics.csv")

# Save data
filtered_cells_plot1.to_csv(f'{output_dir}/cell_cycle_plot1_data.csv', index=False)
filtered_cells_plot2.to_csv(f'{output_dir}/cell_cycle_plot2_data.csv', index=False)
filtered_cells_plot3.to_csv(f'{output_dir}/cell_cycle_plot3_data.csv', index=False)
print(f"✅ Saved: {output_dir}/cell_cycle_plot1_data.csv")
print(f"✅ Saved: {output_dir}/cell_cycle_plot2_data.csv")
print(f"✅ Saved: {output_dir}/cell_cycle_plot3_data.csv")

# Save gene lists used
gene_info = pd.DataFrame({
    'Gene': s_genes_found + g2m_genes_found,
    'Phase': ['S']*len(s_genes_found) + ['G2M']*len(g2m_genes_found)
})
gene_info.to_csv(f'{output_dir}/cell_cycle_genes_used.csv', index=False)
print(f"✅ Saved: {output_dir}/cell_cycle_genes_used.csv")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print("\nPlot 1 - Secretory Transitions:")
print(plot1_stats_df[['Transition', 'N', 'Combined_Mean', 'p_value_vs_control', 'significance']])

print("\nPlot 2 - Tuft Transitions:")
print(plot2_stats_df[['Transition', 'N', 'Combined_Mean', 'p_value_vs_control', 'significance']])

print("\nPlot 3 - Transitions to TA:")
print(plot3_stats_df[['Transition', 'N', 'Combined_Mean', 'p_value_vs_control', 'significance']])

print("\n" + "="*80)
print("DONE!")
print("="*80)

## Module score for DCM based on cells that reverse vs stable

In [ ]:
adata_combined.obs['is_multipotent']

In [ ]:
##############################################
# STABLE vs UNSTABLE TRANSITION ANALYSIS
# adata_combined
#
# For each DCM cell type, cells are split into:
#   Stable   — DCM label == RNA label
#   Unstable — DCM label != RNA label
#
# The module score (ttest_<CellType>_zscore) for
# that cell type is plotted per group, per Dox,
# and combined (ALL).
#
# t-test: Unstable vs Stable within each cell type.
# CSV output with all stats.
#
# ADDED: barplots comparing Stable cell counts
# between pulse vs chase, and Unstable cell counts
# between pulse vs chase (binomial test).
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import binomtest, chi2_contingency

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
root       = "/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py"
score_dir  = os.path.join(root, 'gene_scoring', 'gmm_085', 'MS_stable_unstable')
os.makedirs(score_dir, exist_ok=True)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
cell_type_colors = {
    'Stem'      : '#9370DB',
    'TA'        : '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth'    : '#DC143C',
    'Goblet'    : '#2E8B57',
    'Tuft'      : '#FF1493',
    'EEC'       : '#4169E1',
}

STABLE_COLOR   = '#2ecc71'   # green
UNSTABLE_COLOR = '#e74c3c'   # red

DOX_COLORS = {
    '48hDox'         : '#4A90D9',   # blue  — pulse
    '48hDox-72hChase': '#E87722',   # orange — chase
}
DOX_ORDER  = ['48hDox', '48hDox-72hChase']
DOX_LABELS = {'48hDox': 'Pulse (48hDox)', '48hDox-72hChase': 'Chase (48hDox-72hChase)'}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# ─────────────────────────────────────────────
# STEP 1: BUILD DATAFRAME
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: BUILDING DATAFRAME")
print("=" * 60)

adata = adata_combined

ttest_cols = [c for c in adata.obs.columns
              if c.startswith('ttest_') and c.endswith('_zscore')
              and 'max' not in c.lower()]

ct_from_col = {col: col.replace('ttest_', '').replace('_zscore', '')
               for col in ttest_cols}
col_from_ct = {v: k for k, v in ct_from_col.items()}

print(f"Module score columns found: {ttest_cols}")

base = pd.DataFrame({
    'batch_plate_well':  adata.obs['batch_plate_well'].values,
    'assay':             adata.obs['assay'].values,
    'Dox':               adata.obs['Dox'].astype(str).values,
    'leiden_annot_25':   adata.obs['leiden_annot_25'].astype(str)
                                  .str.split('_').str[-1].values,
    'ttest_gmm_label':   adata.obs['ttest_gmm_label_confident_085'].astype(str)
                                  .str.split('_').str[-1].values,
    'is_multipotent':    adata.obs['is_multipotent'].astype(str).values,
}, index=adata.obs_names)

for col in ttest_cols:
    base[col] = adata.obs[col].values

print(f"Total cells: {len(base)}")
print(f"Assay values: {sorted(base['assay'].unique())}")
print(f"Dox values  : {sorted(base['Dox'].unique())}")

DOX_VALUES = sorted(base['Dox'].dropna().unique())

# ─────────────────────────────────────────────
# STEP 2: PAIR pDCM ↔ pRNA
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: PAIRING pDCM ↔ pRNA")
print("=" * 60)

dcm_df = base[base['assay'] == 'pDCM'].copy()
rna_df = base[base['assay'] == 'pRNA'][
    ['batch_plate_well', 'leiden_annot_25']
].copy().rename(columns={'leiden_annot_25': 'rna_celltype'})

def match_one_to_one(df_a, df_b):
    common = set(df_a['batch_plate_well']) & set(df_b['batch_plate_well'])
    a_s = set(df_a[df_a['batch_plate_well'].isin(common)]
              ['batch_plate_well'].value_counts()
              .pipe(lambda s: s[s == 1].index))
    b_s = set(df_b[df_b['batch_plate_well'].isin(common)]
              ['batch_plate_well'].value_counts()
              .pipe(lambda s: s[s == 1].index))
    return a_s & b_s

matched_wells = match_one_to_one(dcm_df, rna_df)
print(f"1:1 matched wells: {len(matched_wells)}")

dcm_matched = dcm_df[dcm_df['batch_plate_well'].isin(matched_wells)].copy()
rna_matched = rna_df[rna_df['batch_plate_well'].isin(matched_wells)].copy()

paired = dcm_matched.merge(rna_matched, on='batch_plate_well', how='inner')
print(f"Paired cells: {len(paired)}")

# ─────────────────────────────────────────────
# STEP 3: CLASSIFY STABLE vs UNSTABLE
# ─────────────────────────────────────────────
paired = paired[paired['ttest_gmm_label'].isin(label_order)].copy()

n_before = len(paired)
paired = paired[
    ~paired['is_multipotent'].str.lower().isin(['true', '1', 'yes'])
].copy()
print(f'Removed {n_before - len(paired)} hybrid/multipotent cells '
      f'({n_before} → {len(paired)})')

paired['stability'] = np.where(
    paired['ttest_gmm_label'] == paired['rna_celltype'],
    'Stable', 'Unstable'
)

def get_self_module(row):
    col = col_from_ct.get(row['ttest_gmm_label'])
    if col is None or col not in row.index:
        return np.nan
    return row[col]

paired['self_module_score'] = paired.apply(get_self_module, axis=1)

print("\nStability counts (ALL Dox):")
print(paired['stability'].value_counts().to_string())
print("\nPer cell type:")
print(paired.groupby(['ttest_gmm_label', 'stability']).size()
           .unstack(fill_value=0).to_string())
print(f"\nSelf-module score coverage: "
      f"{paired['self_module_score'].notna().sum()}/{len(paired)} cells")

# ─────────────────────────────────────────────
# STATS HELPERS
# ─────────────────────────────────────────────
def get_sig(p):
    if pd.isna(p):    return ''
    if p < 0.001:     return '***'
    if p < 0.01:      return '**'
    if p < 0.05:      return '*'
    return 'ns'

def compute_stats_one(df, ct, dox_label):
    col   = col_from_ct.get(ct)
    rows  = []
    if col is None:
        return pd.DataFrame()

    for stab in ['Stable', 'Unstable']:
        vals = df[df['stability'] == stab][col].dropna()
        rows.append({
            'cell_type':  ct,
            'Dox':        dox_label,
            'stability':  stab,
            'module_col': col,
            'N':          len(vals),
            'mean':       vals.mean()   if len(vals) else np.nan,
            'median':     vals.median() if len(vals) else np.nan,
            'std':        vals.std()    if len(vals) else np.nan,
        })

    stat_df = pd.DataFrame(rows)
    s_vals  = df[df['stability'] == 'Stable'  ][col].dropna()
    u_vals  = df[df['stability'] == 'Unstable'][col].dropna()

    if len(s_vals) > 1 and len(u_vals) > 1:
        _, pval = stats.ttest_ind(u_vals, s_vals)
    else:
        pval = np.nan

    stat_df['ttest_unstable_vs_stable_pval'] = pval
    stat_df['significance'] = get_sig(pval)
    return stat_df

# ─────────────────────────────────────────────
# BARPLOT HELPERS — Pulse vs Chase comparison
# ─────────────────────────────────────────────
def _count_by_dox(df_subset, stability_label):
    """
    Returns counts of cells with stability == stability_label
    per Dox condition, in DOX_ORDER.
    """
    counts = {}
    for dox in DOX_ORDER:
        sub = df_subset[df_subset['Dox'] == dox]
        counts[dox] = int((sub['stability'] == stability_label).sum())
    return counts

def _binom_between_dox(n1, total1, n2, total2):
    """
    Fisher-style comparison of proportions between two Dox conditions.
    Uses binomtest on the larger pool: tests whether the proportion
    n1/(n1+n2) equals 0.5 (i.e. equal in both conditions).
    Returns p-value.
    """
    total = n1 + n2
    if total == 0:
        return np.nan
    return binomtest(n1, total, p=0.5, alternative='two-sided').pvalue

def make_dox_count_barplot(df, stability_label, color,
                            title_str, save_path,
                            per_celltype=False):
    """
    Barplot: number of stable (or unstable) cells per Dox condition.
    If per_celltype=True, one grouped subplot per cell type.
    Binomial test compares pulse vs chase counts.
    """
    dox_present = [d for d in DOX_ORDER if d in df['Dox'].unique()]
    if len(dox_present) < 2:
        print(f"  Skipping — fewer than 2 Dox conditions present")
        return pd.DataFrame()

    if not per_celltype:
        # ── Overall (all cell types pooled) ──────────────────────
        sub = df[df['stability'] == stability_label]
        counts = {dox: int((sub['Dox'] == dox).sum()) for dox in dox_present}
        totals = {dox: int((df['Dox'] == dox).sum()) for dox in dox_present}

        n1, n2   = counts[dox_present[0]], counts[dox_present[1]]
        tot1, tot2 = totals[dox_present[0]], totals[dox_present[1]]
        pval     = _binom_between_dox(n1, tot1, n2, tot2)
        stars    = get_sig(pval)

        fig, ax = plt.subplots(figsize=(6, 6))
        x_pos   = np.arange(len(dox_present))
        bar_vals = [counts[d] for d in dox_present]
        bar_cols = [DOX_COLORS.get(d, color) for d in dox_present]

        bars = ax.bar(x_pos, bar_vals, color=bar_cols,
                      edgecolor='black', linewidth=1, width=0.5)

        for bar, val in zip(bars, bar_vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(bar_vals) * 0.02,
                    str(val), ha='center', va='bottom',
                    fontsize=11, fontweight='bold')

        # bracket between bars
        y_max  = max(bar_vals)
        y_brac = y_max * 1.12
        ax.plot([0, 0, 1, 1],
                [y_brac - y_max*0.03, y_brac,
                 y_brac, y_brac - y_max*0.03],
                color='black', lw=1.5)
        ax.text(0.5, y_brac + y_max*0.02, stars,
                ha='center', va='bottom', fontsize=14, fontweight='bold',
                color='black' if stars != 'ns' else '#888888')
        pval_txt = f'p={pval:.2e}' if pval < 0.001 else f'p={pval:.3f}'
        ax.text(0.5, y_brac + y_max*0.09, pval_txt,
                ha='center', va='bottom', fontsize=8, style='italic')

        ax.set_xticks(x_pos)
        ax.set_xticklabels([DOX_LABELS.get(d, d) for d in dox_present],
                           fontsize=10)
        ax.set_ylabel(f'Number of {stability_label} cells', fontsize=11,
                      fontweight='bold')
        ax.set_title(title_str, fontsize=12, fontweight='bold')
        ax.set_ylim(0, y_brac * 1.20)
        ax.text(0.5, -0.12,
                'Binomial test: are counts equal between conditions?',
                ha='center', transform=ax.transAxes,
                fontsize=8, color='gray', style='italic')

        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {save_path}")

        rows = []
        for dox in dox_present:
            rows.append({
                'stability':    stability_label,
                'cell_type':    'ALL',
                'Dox':          dox,
                'count':        counts[dox],
                'total_in_dox': totals[dox],
            })
        rows_df = pd.DataFrame(rows)
        rows_df['binom_p_pulse_vs_chase'] = pval
        rows_df['significance']           = stars
        return rows_df

    else:
        # ── Per cell type ─────────────────────────────────────────
        ct_present = [ct for ct in label_order
                      if ct in df['ttest_gmm_label'].unique()]
        n_ct = len(ct_present)
        if n_ct == 0:
            return pd.DataFrame()

        fig, axes = plt.subplots(1, n_ct, figsize=(4 * n_ct, 6),
                                 sharey=False)
        if n_ct == 1:
            axes = [axes]

        all_rows = []

        for ax, ct in zip(axes, ct_present):
            sub    = df[(df['stability'] == stability_label) &
                        (df['ttest_gmm_label'] == ct)]
            totals = {dox: int((df[df['ttest_gmm_label'] == ct]['Dox'] == dox).sum())
                      for dox in dox_present}
            counts = {dox: int((sub['Dox'] == dox).sum()) for dox in dox_present}

            n1, n2     = counts[dox_present[0]], counts[dox_present[1]]
            tot1, tot2 = totals[dox_present[0]], totals[dox_present[1]]
            pval       = _binom_between_dox(n1, tot1, n2, tot2)
            stars      = get_sig(pval)

            bar_vals = [counts[d] for d in dox_present]
            bar_cols = [DOX_COLORS.get(d, color) for d in dox_present]

            bars = ax.bar(np.arange(len(dox_present)), bar_vals,
                          color=bar_cols, edgecolor='black',
                          linewidth=1, width=0.5)

            for bar, val in zip(bars, bar_vals):
                ax.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + max(bar_vals + [1]) * 0.04,
                        str(val), ha='center', va='bottom',
                        fontsize=9, fontweight='bold')

            y_max  = max(bar_vals + [1])
            y_brac = y_max * 1.15
            ax.plot([0, 0, 1, 1],
                    [y_brac - y_max*0.04, y_brac,
                     y_brac, y_brac - y_max*0.04],
                    color='black', lw=1.2)
            pval_txt = f'p={pval:.2e}' if pval < 0.001 else f'p={pval:.3f}'
            ax.text(0.5, y_brac + y_max*0.03,
                    f'{stars}\n{pval_txt}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold',
                    color='black' if stars != 'ns' else '#888888')

            ax.set_xticks(np.arange(len(dox_present)))
            ax.set_xticklabels(['Pulse', 'Chase'], fontsize=9)
            ax.set_title(ct, fontsize=11, fontweight='bold',
                         color=cell_type_colors.get(ct, 'black'))
            ax.set_ylabel('Cell count', fontsize=9)
            ax.set_ylim(0, (y_brac + y_max*0.15) * 1.1)

            for dox in dox_present:
                all_rows.append({
                    'stability':    stability_label,
                    'cell_type':    ct,
                    'Dox':          dox,
                    'count':        counts[dox],
                    'total_in_dox': totals[dox],
                    'binom_p_pulse_vs_chase': pval,
                    'significance':           stars,
                })

        plt.suptitle(title_str, fontsize=13, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {save_path}")

        return pd.DataFrame(all_rows)


# ─────────────────────────────────────────────
# COMBINED PLOT HELPER
# ─────────────────────────────────────────────
def make_combined_plot(df, dox_label, save_path):
    sub = df[df['self_module_score'].notna()][
        ['stability', 'self_module_score', 'ttest_gmm_label']
    ].copy()

    if sub.empty or sub['stability'].nunique() < 2:
        print(f"  Skipping combined plot Dox={dox_label} — insufficient data")
        return pd.DataFrame()

    fig, ax = plt.subplots(figsize=(6, 6))

    palette_box = {'Stable': STABLE_COLOR, 'Unstable': UNSTABLE_COLOR}

    sns.boxplot(
        data=sub, x='stability', y='self_module_score',
        hue='stability', order=['Stable', 'Unstable'],
        palette=palette_box, legend=False, ax=ax,
        flierprops=dict(marker='o', markersize=2, alpha=0.3),
        width=0.5,
    )
    sns.stripplot(
        data=sub, x='stability', y='self_module_score',
        hue='ttest_gmm_label', order=['Stable', 'Unstable'],
        palette=cell_type_colors, dodge=False, alpha=0.5,
        size=3, jitter=True, ax=ax,
    )
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, title='Cell type',
              bbox_to_anchor=(1.01, 1), loc='upper left',
              fontsize=8, frameon=True)

    ylim  = ax.get_ylim()
    y_top = ylim[1]

    for i, stab in enumerate(['Stable', 'Unstable']):
        n = len(sub[sub['stability'] == stab])
        ax.text(i, y_top * 1.01, f'n={n}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

    s_vals = sub[sub['stability'] == 'Stable'  ]['self_module_score'].dropna()
    u_vals = sub[sub['stability'] == 'Unstable']['self_module_score'].dropna()
    pval   = np.nan
    if len(s_vals) > 1 and len(u_vals) > 1:
        _, pval = stats.ttest_ind(u_vals, s_vals)
        stars   = get_sig(pval)
        y_ann   = y_top * 1.09
        ax.annotate('', xy=(1, y_ann), xytext=(0, y_ann),
                    arrowprops=dict(arrowstyle='-', color='#333333', lw=1.2))
        ax.text(0.5, y_ann, stars,
                ha='center', va='bottom', fontsize=13, fontweight='bold',
                color='black' if stars not in ('ns', '') else '#888888')
        ax.set_ylim(ylim[0], y_ann * 1.13)

    ax.set_xlabel('')
    ax.set_ylabel('Self-module score (own DCM cell-type module)', fontsize=10)
    ax.set_title(f'All cell types combined — Stable vs Unstable | Dox = {dox_label}',
                 fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

    rows = []
    for stab in ['Stable', 'Unstable']:
        vals = sub[sub['stability'] == stab]['self_module_score'].dropna()
        rows.append({
            'cell_type':  'ALL_combined',
            'Dox':        dox_label,
            'stability':  stab,
            'module_col': 'self_module_score',
            'N':          len(vals),
            'mean':       vals.mean()   if len(vals) else np.nan,
            'median':     vals.median() if len(vals) else np.nan,
            'std':        vals.std()    if len(vals) else np.nan,
            'ttest_unstable_vs_stable_pval': pval,
            'significance': get_sig(pval),
        })
    return pd.DataFrame(rows)

# ─────────────────────────────────────────────
# PER-CELLTYPE BOXPLOT HELPER
# ─────────────────────────────────────────────
def make_stability_plot(df, dox_label, save_path):
    ct_present = []
    for ct in label_order:
        sub = df[df['ttest_gmm_label'] == ct]
        if (len(sub[sub['stability'] == 'Stable']) > 0 and
                len(sub[sub['stability'] == 'Unstable']) > 0):
            ct_present.append(ct)

    if not ct_present:
        print(f"  Skipping Dox={dox_label} — no cell types with both groups")
        return pd.DataFrame()

    n_ct      = len(ct_present)
    fig, axes = plt.subplots(1, n_ct, figsize=(4 * n_ct, 6), sharey=False)
    if n_ct == 1:
        axes = [axes]

    all_stats = []

    for ax, ct in zip(axes, ct_present):
        col = col_from_ct.get(ct)
        if col is None:
            ax.set_visible(False)
            continue

        sub = df[df['ttest_gmm_label'] == ct][[col, 'stability']].copy()
        sub = sub.rename(columns={col: 'module_score'})

        palette = {'Stable': STABLE_COLOR, 'Unstable': UNSTABLE_COLOR}

        sns.boxplot(
            data=sub, x='stability', y='module_score',
            hue='stability', order=['Stable', 'Unstable'],
            palette=palette, legend=False, ax=ax,
            flierprops=dict(marker='o', markersize=2, alpha=0.4),
        )

        ylim  = ax.get_ylim()
        y_top = ylim[1]
        for i, stab in enumerate(['Stable', 'Unstable']):
            n = len(sub[sub['stability'] == stab])
            ax.text(i, y_top * 1.01, f'n={n}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

        s_vals = sub[sub['stability'] == 'Stable'  ]['module_score'].dropna()
        u_vals = sub[sub['stability'] == 'Unstable']['module_score'].dropna()
        if len(s_vals) > 1 and len(u_vals) > 1:
            _, pval = stats.ttest_ind(u_vals, s_vals)
            stars   = get_sig(pval)
            y_ann   = y_top * 1.09
            ax.annotate('', xy=(1, y_ann), xytext=(0, y_ann),
                        arrowprops=dict(arrowstyle='-', color='#333333', lw=1.0))
            ax.text(0.5, y_ann, stars,
                    ha='center', va='bottom', fontsize=11, fontweight='bold',
                    color='black' if stars not in ('ns', '') else '#888888')
            ax.set_ylim(ylim[0], y_ann * 1.12)

        ax.set_title(ct, fontsize=12, fontweight='bold',
                     color=cell_type_colors.get(ct, 'black'))
        ax.set_xlabel('')
        ax.set_ylabel(f'{ct} module score\n(ttest score)', fontsize=9)
        ax.tick_params(axis='x', labelsize=10)

        all_stats.append(compute_stats_one(
            df[df['ttest_gmm_label'] == ct], ct, dox_label
        ))

    plt.suptitle(
        f'Module Scores — Stable vs Unstable Transitions | Dox = {dox_label}\n'
        f'Each panel: DCM cell type module score split by identity stability',
        fontsize=13, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

    return pd.concat(all_stats, ignore_index=True) if all_stats else pd.DataFrame()

# ─────────────────────────────────────────────
# STEP 4: GENERATE MODULE SCORE PLOTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: GENERATING MODULE SCORE PLOTS")
print("=" * 60)

collected_stats = []

print(f"\n  Dox = ALL")
_s = make_stability_plot(
    paired, dox_label='ALL',
    save_path=os.path.join(score_dir, 'stable_unstable_modules_Dox_ALL.pdf'),
)
collected_stats.append(_s)

_s_comb = make_combined_plot(
    paired, dox_label='ALL',
    save_path=os.path.join(score_dir, 'stable_unstable_combined_Dox_ALL.pdf'),
)
collected_stats.append(_s_comb)

for dox in DOX_VALUES:
    tag = str(dox).replace(' ', '_').replace('/', '-')
    print(f"\n  Dox = {dox}")
    sub = paired[paired['Dox'] == dox]
    _s  = make_stability_plot(
        sub, dox_label=str(dox),
        save_path=os.path.join(score_dir, f'stable_unstable_modules_Dox_{tag}.pdf'),
    )
    collected_stats.append(_s)

    _s_comb = make_combined_plot(
        sub, dox_label=str(dox),
        save_path=os.path.join(score_dir, f'stable_unstable_combined_Dox_{tag}.pdf'),
    )
    collected_stats.append(_s_comb)

# ─────────────────────────────────────────────
# STEP 5: BARPLOTS — Pulse vs Chase cell counts
# One pair of plots for Stable, one for Unstable.
# Each pair: overall (all cell types pooled) +
#            per cell type.
# Binomial test: are counts equal between Dox?
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: BARPLOTS — PULSE vs CHASE CELL COUNTS")
print("=" * 60)

barplot_stats = []

# Filter to only the two Dox conditions of interest
paired_dox = paired[paired['Dox'].isin(DOX_ORDER)].copy()

for stab_label, bar_color in [('Stable', STABLE_COLOR),
                                ('Unstable', UNSTABLE_COLOR)]:
    print(f"\n  --- {stab_label} ---")

    # Overall (all cell types)
    _bp = make_dox_count_barplot(
        df             = paired_dox,
        stability_label= stab_label,
        color          = bar_color,
        title_str      = (f'{stab_label} cells — Pulse vs Chase\n'
                          f'(all cell types pooled, binomial test)'),
        save_path      = os.path.join(score_dir,
                                      f'barplot_{stab_label.lower()}_pulse_vs_chase_overall.pdf'),
        per_celltype   = False,
    )
    barplot_stats.append(_bp)

    # Per cell type
    _bp = make_dox_count_barplot(
        df             = paired_dox,
        stability_label= stab_label,
        color          = bar_color,
        title_str      = (f'{stab_label} cells per cell type — Pulse vs Chase\n'
                          f'(binomial test per cell type)'),
        save_path      = os.path.join(score_dir,
                                      f'barplot_{stab_label.lower()}_pulse_vs_chase_per_celltype.pdf'),
        per_celltype   = True,
    )
    barplot_stats.append(_bp)

# ─────────────────────────────────────────────
# STEP 6: SAVE CSV OUTPUTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: SAVING CSV OUTPUTS")
print("=" * 60)

# Module score stats
stats_df  = pd.concat([s for s in collected_stats if not s.empty],
                      ignore_index=True)
stats_csv = os.path.join(score_dir, 'stable_unstable_stats.csv')
stats_df.to_csv(stats_csv, index=False)
print(f"Stats CSV        : {stats_csv}")
print(stats_df.to_string(index=False))

# Barplot stats (pulse vs chase counts)
bp_stats_df  = pd.concat([s for s in barplot_stats if s is not None and not s.empty],
                          ignore_index=True)
bp_stats_csv = os.path.join(score_dir, 'pulse_vs_chase_count_stats.csv')
bp_stats_df.to_csv(bp_stats_csv, index=False)
print(f"\nBarplot stats CSV: {bp_stats_csv}")
print(bp_stats_df.to_string(index=False))

# Full paired data
out_cols = (['batch_plate_well', 'Dox', 'ttest_gmm_label', 'is_multipotent',
             'rna_celltype', 'stability', 'self_module_score'] + ttest_cols)
data_csv = os.path.join(score_dir, 'stable_unstable_paired_data.csv')
paired[out_cols].to_csv(data_csv, index=False)
print(f"Data CSV         : {data_csv}")

print("\nDone!")
print(f"Outputs in  : {score_dir}")

In [ ]:
##Stable vs programmed cells to TA/Stem

In [ ]:
##############################################
# STABLE vs REPROGRAMMED-TO-STEM/TA ANALYSIS
# adata_combined
#
# For each DCM cell type (secretory / absorptive),
# cells are split into three groups:
#   Stable      — DCM label == RNA label
#   To_Stem     — DCM label != RNA label AND RNA == Stem
#   To_TA       — DCM label != RNA label AND RNA == TA
#   To_StemOrTA — combined Stem + TA destination
#
# The module score (ttest_<CellType>_module) for
# that cell type is plotted per group, per Dox,
# and combined (ALL).
#
# Hybrid / multipotent DCM cells are excluded.
# t-test: each unstable group vs Stable.
# CSV output with all stats.
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
root      = "/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py"
score_dir = os.path.join(root, 'gene_scoring','gmm_085', 'MS_stable_vs_reprogrammed')
os.makedirs(score_dir, exist_ok=True)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
cell_type_colors = {
    'Stem'      : '#9370DB',
    'TA'        : '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth'    : '#DC143C',
    'Goblet'    : '#2E8B57',
    'Tuft'      : '#FF1493',
    'EEC'       : '#4169E1',
}

# Colours for the three transition groups
GROUP_COLORS = {
    'Stable':              '#2ecc71',   # green
    'To_Stem':             '#9370DB',   # purple  (matches Stem)
    'To_TA':               '#D2B48C',   # tan     (matches TA)
    'To_StemOrTA':         '#5B4A8A',   # dark purple (combined)
    'Stable_Stem':         '#2ecc71',   # green  (stable Stem/TA reference)
    'Stable_TA':           '#27ae60',   # dark green
    'Reversed_to_Stem':    '#9370DB',   # purple
    'Reversed_to_TA':      '#b8860b',   # dark goldenrod
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

DOX_COLORS = {
    '48hDox'         : '#4A90D9',   # blue  — pulse
    '48hDox-72hChase': '#E87722',   # orange — chase
}
DOX_ORDER  = ['48hDox', '48hDox-72hChase']
DOX_LABELS = {'48hDox': 'Pulse (48hDox)', '48hDox-72hChase': 'Chase (48hDox-72hChase)'}

# Cell types to analyse — exclude Stem/TA themselves since
# "going back to Stem/TA" only makes biological sense for
# more differentiated cells.
ANALYSIS_TYPES = ['Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# ─────────────────────────────────────────────
# STEP 1: BUILD DATAFRAME
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: BUILDING DATAFRAME")
print("=" * 60)

adata = adata_combined

ttest_cols = [c for c in adata.obs.columns
              if c.startswith('ttest_') and c.endswith('_module')
              and 'max' not in c.lower()]

ct_from_col = {col: col.replace('ttest_', '').replace('_module', '')
               for col in ttest_cols}
col_from_ct = {v: k for k, v in ct_from_col.items()}

print(f"Module score columns: {ttest_cols}")

base = pd.DataFrame({
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'assay':            adata.obs['assay'].values,
    'Dox':              adata.obs['Dox'].astype(str).values,
    'leiden_annot_25':  adata.obs['leiden_annot_25'].astype(str)
                                 .str.split('_').str[-1].values,
    'ttest_gmm_label':  adata.obs['ttest_gmm_label_confident_085'].astype(str)
                                 .str.split('_').str[-1].values,
    'is_multipotent':   adata.obs['is_multipotent'].astype(str).values,
}, index=adata.obs_names)

for col in ttest_cols:
    base[col] = adata.obs[col].values

print(f"Total cells : {len(base)}")
print(f"Assay values: {sorted(base['assay'].unique())}")
print(f"Dox values  : {sorted(base['Dox'].unique())}")

DOX_VALUES = sorted(base['Dox'].dropna().unique())

# ─────────────────────────────────────────────
# STEP 2: PAIR pDCM ↔ pRNA  (1:1)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: PAIRING pDCM ↔ pRNA")
print("=" * 60)

dcm_df = base[base['assay'] == 'pDCM'].copy()
rna_df = (base[base['assay'] == 'pRNA']
          [['batch_plate_well', 'leiden_annot_25']]
          .copy()
          .rename(columns={'leiden_annot_25': 'rna_celltype'}))

def match_one_to_one(df_a, df_b):
    common = set(df_a['batch_plate_well']) & set(df_b['batch_plate_well'])
    a_s = set(df_a[df_a['batch_plate_well'].isin(common)]
              ['batch_plate_well'].value_counts()
              .pipe(lambda s: s[s == 1].index))
    b_s = set(df_b[df_b['batch_plate_well'].isin(common)]
              ['batch_plate_well'].value_counts()
              .pipe(lambda s: s[s == 1].index))
    return a_s & b_s

matched_wells = match_one_to_one(dcm_df, rna_df)
print(f"1:1 matched wells: {len(matched_wells)}")

paired = (dcm_df[dcm_df['batch_plate_well'].isin(matched_wells)]
          .merge(rna_df[rna_df['batch_plate_well'].isin(matched_wells)],
                 on='batch_plate_well', how='inner'))
print(f"Paired cells: {len(paired)}")

# ─────────────────────────────────────────────
# STEP 3: FILTER AND CLASSIFY
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: CLASSIFYING")
print("=" * 60)

# Keep only known cell types, exclude hybrids
paired = paired[paired['ttest_gmm_label'].isin(label_order)].copy()
n_before = len(paired)
paired = paired[
    ~paired['is_multipotent'].str.lower().isin(['true', '1', 'yes'])
].copy()
print(f"Removed {n_before - len(paired)} hybrid cells ({n_before} → {len(paired)})")

# Keep only ANALYSIS_TYPES as DCM origin
paired = paired[paired['ttest_gmm_label'].isin(ANALYSIS_TYPES)].copy()
print(f"Cells with DCM in ANALYSIS_TYPES: {len(paired)}")

# Assign transition group
def classify(row):
    dcm = row['ttest_gmm_label']
    rna = row['rna_celltype']
    if dcm == rna:
        return 'Stable'
    if rna == 'Stem':
        return 'To_Stem'
    if rna == 'TA':
        return 'To_TA'
    return 'Other'   # different but not Stem/TA

paired['transition'] = paired.apply(classify, axis=1)

# Also add a combined To_StemOrTA label for the pooled comparison
# (kept as a separate column so both views are available)
paired['transition_combined'] = paired['transition'].replace(
    {'To_Stem': 'To_StemOrTA', 'To_TA': 'To_StemOrTA'}
)

print("\nTransition counts (ALL Dox):")
print(paired['transition'].value_counts().to_string())
print("\nPer DCM cell type:")
print(paired.groupby(['ttest_gmm_label', 'transition']).size()
           .unstack(fill_value=0).to_string())

# ── Self-module score ─────────────────────────────────────────────
def get_self_module(row):
    col = col_from_ct.get(row['ttest_gmm_label'])
    return row[col] if (col is not None and col in row.index) else np.nan

paired['self_module_score'] = paired.apply(get_self_module, axis=1)

# ─────────────────────────────────────────────
# BUILD STEM/TA REFERENCE DATAFRAME
# Two sources pooled per destination (Stem, TA):
#   Origin A — DCM == Stem/TA and RNA == Stem/TA  (Stable_StemTA)
#   Origin B — DCM in ANALYSIS_TYPES and RNA == Stem/TA (Reversed)
# The score used is the Stem or TA module respectively.
# ─────────────────────────────────────────────
# Re-pair full paired set (all DCM cell types, no hybrid filter yet)
# so we can extract Stem/TA stable cells.
_all_paired_raw = (dcm_df[dcm_df['batch_plate_well'].isin(matched_wells)]
                   .merge(rna_df[rna_df['batch_plate_well'].isin(matched_wells)],
                          on='batch_plate_well', how='inner'))

# Remove hybrids
_all_paired_raw = _all_paired_raw[
    ~_all_paired_raw['is_multipotent'].str.lower().isin(['true', '1', 'yes'])
].copy()

rows_stemta = []
for dest_ct in ['Stem', 'TA']:
    mod_col = col_from_ct.get(dest_ct)
    if mod_col is None:
        continue

    # Stable: DCM == dest_ct AND RNA == dest_ct
    stable_sub = _all_paired_raw[
        (_all_paired_raw['ttest_gmm_label'] == dest_ct) &
        (_all_paired_raw['rna_celltype']    == dest_ct)
    ].copy()
    stable_sub['stemta_group']  = f'Stable_{dest_ct}'
    stable_sub['stemta_dest']   = dest_ct
    stable_sub['stemta_score']  = stable_sub[mod_col]

    # Reversed: DCM in ANALYSIS_TYPES AND RNA == dest_ct
    reversed_sub = paired[paired['rna_celltype'] == dest_ct].copy()
    reversed_sub['stemta_group']  = f'Reversed_to_{dest_ct}'
    reversed_sub['stemta_dest']   = dest_ct
    reversed_sub['stemta_score']  = reversed_sub[mod_col]

    rows_stemta.append(stable_sub)
    rows_stemta.append(reversed_sub)

paired_stemta = pd.concat(rows_stemta, ignore_index=True)

print(f"\nStem/TA comparison dataframe: {len(paired_stemta)} rows")
print(paired_stemta.groupby(['stemta_dest','stemta_group']).size().to_string())

# ─────────────────────────────────────────────
# STATS HELPERS
# ─────────────────────────────────────────────
def get_sig(p):
    if pd.isna(p):  return ''
    if p < 0.001:   return '***'
    if p < 0.01:    return '**'
    if p < 0.05:    return '*'
    return 'ns'

def ttest_vs_stable(df, score_col, group_col, test_group):
    s = df[df[group_col] == 'Stable'    ][score_col].dropna()
    u = df[df[group_col] == test_group  ][score_col].dropna()
    if len(s) > 1 and len(u) > 1:
        return stats.ttest_ind(u, s)[1]
    return np.nan

def compute_stats(df, score_col, group_col, dox_label, ct, context):
    rows = []
    for grp in df[group_col].unique():
        vals = df[df[group_col] == grp][score_col].dropna()
        pval = ttest_vs_stable(df, score_col, group_col, grp) if grp != 'Stable' else np.nan
        rows.append({
            'context':                    context,
            'cell_type':                  ct,
            'Dox':                        dox_label,
            'group':                      grp,
            'score_col':                  score_col,
            'N':                          len(vals),
            'mean':                       vals.mean()   if len(vals) else np.nan,
            'median':                     vals.median() if len(vals) else np.nan,
            'std':                        vals.std()    if len(vals) else np.nan,
            'ttest_vs_Stable_pval':       pval,
            'significance':               get_sig(pval),
        })
    return pd.DataFrame(rows)

# ─────────────────────────────────────────────
# PLOT HELPERS
# ─────────────────────────────────────────────
def draw_bracket(ax, group_order, i_test, i_ref, y_lvl, stars):
    ax.annotate('', xy=(i_test, y_lvl), xytext=(i_ref, y_lvl),
                arrowprops=dict(arrowstyle='-', color='#333333', lw=0.9))
    x_mid = (i_test + i_ref) / 2
    ax.text(x_mid, y_lvl, stars,
            ha='center', va='bottom', fontsize=10, fontweight='bold',
            color='black' if stars not in ('ns', '') else '#888888')


def make_per_celltype_plot(df, group_col, group_order_full,
                           palette, dox_label, save_path, context):
    """
    One subplot per cell type; each subplot shows the self-module
    score split by transition group.
    """
    ct_present = []
    for ct in ANALYSIS_TYPES:
        sub = df[df['ttest_gmm_label'] == ct]
        # Need Stable + at least one other group
        grps = sub[group_col].unique()
        if 'Stable' in grps and len(grps) > 1:
            ct_present.append(ct)

    if not ct_present:
        print(f"  Skipping per-celltype Dox={dox_label} — no data")
        return pd.DataFrame()

    n_ct  = len(ct_present)
    fig, axes = plt.subplots(1, n_ct, figsize=(4.5 * n_ct, 6), sharey=False)
    if n_ct == 1:
        axes = [axes]

    all_stats = []

    for ax, ct in zip(axes, ct_present):
        col = col_from_ct.get(ct)
        if col is None:
            ax.set_visible(False)
            continue

        sub = df[df['ttest_gmm_label'] == ct].copy()
        present = [g for g in group_order_full if g in sub[group_col].values]

        plot_sub = sub[[col, group_col]].rename(columns={col: 'score'})

        sns.boxplot(
            data=plot_sub, x=group_col, y='score',
            hue=group_col, order=present,
            palette=palette, legend=False, ax=ax,
            flierprops=dict(marker='o', markersize=2, alpha=0.4),
        )

        ylim  = ax.get_ylim()
        y_top = ylim[1]

        # n labels
        for i, grp in enumerate(present):
            n = len(plot_sub[plot_sub[group_col] == grp])
            ax.text(i, y_top * 1.01, f'n={n}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

        # brackets vs Stable
        if 'Stable' in present:
            ref_i    = present.index('Stable')
            y_levels = [y_top * 1.09, y_top * 1.17]
            y_max    = y_levels[0]
            k = 0
            for grp in present:
                if grp == 'Stable':
                    continue
                pval  = ttest_vs_stable(plot_sub, 'score', group_col, grp)
                stars = get_sig(pval)
                y_lvl = y_levels[k % 2]
                y_max = max(y_max, y_lvl)
                draw_bracket(ax, present, present.index(grp), ref_i, y_lvl, stars)
                k += 1
            ax.set_ylim(ylim[0], y_max * 1.12)

        ax.set_title(ct, fontsize=12, fontweight='bold',
                     color=cell_type_colors.get(ct, 'black'))
        ax.set_xlabel('')
        ax.set_ylabel(f'{ct} module score', fontsize=9)
        ax.tick_params(axis='x', rotation=20, labelsize=9)

        all_stats.append(compute_stats(sub, col, group_col,
                                       dox_label, ct, context))

    plt.suptitle(
        f'Module Scores — Stable vs Reprogrammed to Stem/TA | Dox = {dox_label}\n'
        f't-tests vs Stable within each cell type',
        fontsize=13, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

    return pd.concat(all_stats, ignore_index=True) if all_stats else pd.DataFrame()


def make_combined_plot(df, group_col, group_order_full,
                       palette, dox_label, save_path, context):
    """
    All cell types pooled; each cell uses its own self_module_score.
    Stripplot overlay colours points by DCM cell type.
    """
    sub = df[df['self_module_score'].notna()].copy()
    present = [g for g in group_order_full if g in sub[group_col].values]

    if len(present) < 2 or 'Stable' not in present:
        print(f"  Skipping combined Dox={dox_label} — insufficient groups")
        return pd.DataFrame()

    fig, ax = plt.subplots(figsize=(max(5, 2 * len(present)), 6))

    sns.boxplot(
        data=sub, x=group_col, y='self_module_score',
        hue=group_col, order=present,
        palette=palette, legend=False, ax=ax,
        flierprops=dict(marker='o', markersize=2, alpha=0.3),
        width=0.5,
    )
    sns.stripplot(
        data=sub, x=group_col, y='self_module_score',
        hue='ttest_gmm_label', order=present,
        palette=cell_type_colors, dodge=False,
        alpha=0.5, size=3, jitter=True, ax=ax,
    )
    h, l = ax.get_legend_handles_labels()
    ax.legend(h, l, title='Cell type',
              bbox_to_anchor=(1.01, 1), loc='upper left',
              fontsize=8, frameon=True)

    ylim  = ax.get_ylim()
    y_top = ylim[1]

    for i, grp in enumerate(present):
        n = len(sub[sub[group_col] == grp])
        ax.text(i, y_top * 1.01, f'n={n}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

    if 'Stable' in present:
        ref_i    = present.index('Stable')
        y_levels = [y_top * 1.09, y_top * 1.17]
        y_max    = y_levels[0]
        k = 0
        for grp in present:
            if grp == 'Stable':
                continue
            pval  = ttest_vs_stable(sub, 'self_module_score', group_col, grp)
            stars = get_sig(pval)
            y_lvl = y_levels[k % 2]
            y_max = max(y_max, y_lvl)
            draw_bracket(ax, present, present.index(grp), ref_i, y_lvl, stars)
            k += 1
        ax.set_ylim(ylim[0], y_max * 1.13)

    ax.set_xlabel('')
    ax.set_ylabel('Self-module score (own DCM cell-type module)', fontsize=10)
    ax.set_title(
        f'All cell types combined — Stable vs Reprogrammed to Stem/TA\n'
        f'Dox = {dox_label}',
        fontsize=12, fontweight='bold'
    )
    ax.tick_params(axis='x', rotation=20, labelsize=10)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

    return compute_stats(sub, 'self_module_score', group_col,
                         dox_label, 'ALL_combined', context)


# ─────────────────────────────────────────────
# STEM/TA COMPARISON PLOT HELPER
# One subplot per destination (Stem, TA).
# Each subplot: Stable_<dest> vs Reversed_to_<dest>
# using the <dest> module score.
# ─────────────────────────────────────────────
def make_stemta_plot(df, dox_label, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(10, 6))
    all_stats = []

    for ax, dest_ct in zip(axes, ['Stem', 'TA']):
        mod_col  = col_from_ct.get(dest_ct)
        if mod_col is None:
            ax.set_visible(False)
            continue

        sub = df[df['stemta_dest'] == dest_ct][
            ['stemta_group', 'stemta_score', 'ttest_gmm_label']
        ].copy()

        stbl   = f'Stable_{dest_ct}'
        rev    = f'Reversed_to_{dest_ct}'
        present = [g for g in [stbl, rev] if g in sub['stemta_group'].values]

        if len(present) < 2:
            ax.set_visible(False)
            continue

        pal = {stbl: GROUP_COLORS[stbl], rev: GROUP_COLORS[rev]}

        sns.boxplot(
            data=sub, x='stemta_group', y='stemta_score',
            hue='stemta_group', order=present,
            palette=pal, legend=False, ax=ax,
            flierprops=dict(marker='o', markersize=2, alpha=0.4),
        )
        sns.stripplot(
            data=sub, x='stemta_group', y='stemta_score',
            hue='ttest_gmm_label', order=present,
            palette=cell_type_colors, dodge=False,
            alpha=0.5, size=3, jitter=True, ax=ax,
        )
        h, l = ax.get_legend_handles_labels()
        ax.legend(h, l, title='DCM cell type',
                  bbox_to_anchor=(1.01, 1), loc='upper left',
                  fontsize=7, frameon=True)

        ylim  = ax.get_ylim()
        y_top = ylim[1]

        for i, grp in enumerate(present):
            n = len(sub[sub['stemta_group'] == grp])
            ax.text(i, y_top * 1.01, f'n={n}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

        # t-test: Reversed vs Stable
        s_vals = sub[sub['stemta_group'] == stbl]['stemta_score'].dropna()
        r_vals = sub[sub['stemta_group'] == rev ]['stemta_score'].dropna()
        if len(s_vals) > 1 and len(r_vals) > 1:
            _, pval = stats.ttest_ind(r_vals, s_vals)
            stars   = get_sig(pval)
            y_ann   = y_top * 1.09
            ax.annotate('', xy=(1, y_ann), xytext=(0, y_ann),
                        arrowprops=dict(arrowstyle='-', color='#333333', lw=1.0))
            ax.text(0.5, y_ann, stars,
                    ha='center', va='bottom', fontsize=11, fontweight='bold',
                    color='black' if stars not in ('ns', '') else '#888888')
            ax.set_ylim(ylim[0], y_ann * 1.13)
        else:
            pval = np.nan

        ax.set_title(f'{dest_ct} module score', fontsize=12, fontweight='bold',
                     color=cell_type_colors.get(dest_ct, 'black'))
        ax.set_xlabel('')
        ax.set_ylabel(f'{dest_ct} module score (ttest zscore)', fontsize=9)
        ax.tick_params(axis='x', rotation=15, labelsize=9)

        # Stats rows
        for grp in present:
            vals = sub[sub['stemta_group'] == grp]['stemta_score'].dropna()
            all_stats.append({
                'context':                  'StemTA_StableVsReversed',
                'cell_type':               dest_ct,
                'Dox':                     dox_label,
                'group':                   grp,
                'score_col':               mod_col,
                'N':                       len(vals),
                'mean':                    vals.mean()   if len(vals) else np.nan,
                'median':                  vals.median() if len(vals) else np.nan,
                'std':                     vals.std()    if len(vals) else np.nan,
                'ttest_vs_Stable_pval':    pval if grp == rev else np.nan,
                'significance':            get_sig(pval) if grp == rev else '',
            })

    plt.suptitle(
        f'Stem/TA Module Score: Stable Stem/TA vs Reversed from Differentiated\n'
        f'Dox = {dox_label} | t-test: Reversed vs Stable',
        fontsize=13, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

    return pd.DataFrame(all_stats) if all_stats else pd.DataFrame()


# ─────────────────────────────────────────────
# BARPLOT HELPERS — Pulse vs Chase comparison
# ─────────────────────────────────────────────
def _count_by_dox(df_subset, stability_label):
    """
    Returns counts of cells with stability == stability_label
    per Dox condition, in DOX_ORDER.
    """
    counts = {}
    for dox in DOX_ORDER:
        sub = df_subset[df_subset['Dox'] == dox]
        counts[dox] = int((sub['stability'] == stability_label).sum())
    return counts

def _binom_between_dox(n1, total1, n2, total2):
    """
    Fisher-style comparison of proportions between two Dox conditions.
    Uses binomtest on the larger pool: tests whether the proportion
    n1/(n1+n2) equals 0.5 (i.e. equal in both conditions).
    Returns p-value.
    """
    total = n1 + n2
    if total == 0:
        return np.nan
    return binomtest(n1, total, p=0.5, alternative='two-sided').pvalue

def make_dox_count_barplot(df, stability_label, color,
                            title_str, save_path,
                            per_celltype=False):
    """
    Barplot: number of stable (or unstable) cells per Dox condition.
    If per_celltype=True, one grouped subplot per cell type.
    Binomial test compares pulse vs chase counts.
    """
    dox_present = [d for d in DOX_ORDER if d in df['Dox'].unique()]
    if len(dox_present) < 2:
        print(f"  Skipping — fewer than 2 Dox conditions present")
        return pd.DataFrame()

    if not per_celltype:
        # ── Overall (all cell types pooled) ──────────────────────
        sub = df[df['stability'] == stability_label]
        counts = {dox: int((sub['Dox'] == dox).sum()) for dox in dox_present}
        totals = {dox: int((df['Dox'] == dox).sum()) for dox in dox_present}

        n1, n2   = counts[dox_present[0]], counts[dox_present[1]]
        tot1, tot2 = totals[dox_present[0]], totals[dox_present[1]]
        pval     = _binom_between_dox(n1, tot1, n2, tot2)
        stars    = get_sig(pval)

        fig, ax = plt.subplots(figsize=(6, 6))
        x_pos   = np.arange(len(dox_present))
        bar_vals = [counts[d] for d in dox_present]
        bar_cols = [DOX_COLORS.get(d, color) for d in dox_present]

        bars = ax.bar(x_pos, bar_vals, color=bar_cols,
                      edgecolor='black', linewidth=1, width=0.5)

        for bar, val in zip(bars, bar_vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(bar_vals) * 0.02,
                    str(val), ha='center', va='bottom',
                    fontsize=11, fontweight='bold')

        # bracket between bars
        y_max  = max(bar_vals)
        y_brac = y_max * 1.12
        ax.plot([0, 0, 1, 1],
                [y_brac - y_max*0.03, y_brac,
                 y_brac, y_brac - y_max*0.03],
                color='black', lw=1.5)
        ax.text(0.5, y_brac + y_max*0.02, stars,
                ha='center', va='bottom', fontsize=14, fontweight='bold',
                color='black' if stars != 'ns' else '#888888')
        pval_txt = f'p={pval:.2e}' if pval < 0.001 else f'p={pval:.3f}'
        ax.text(0.5, y_brac + y_max*0.09, pval_txt,
                ha='center', va='bottom', fontsize=8, style='italic')

        ax.set_xticks(x_pos)
        ax.set_xticklabels([DOX_LABELS.get(d, d) for d in dox_present],
                           fontsize=10)
        ax.set_ylabel(f'Number of {stability_label} cells', fontsize=11,
                      fontweight='bold')
        ax.set_title(title_str, fontsize=12, fontweight='bold')
        ax.set_ylim(0, y_brac * 1.20)
        ax.text(0.5, -0.12,
                'Binomial test: are counts equal between conditions?',
                ha='center', transform=ax.transAxes,
                fontsize=8, color='gray', style='italic')

        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {save_path}")

        rows = []
        for dox in dox_present:
            rows.append({
                'stability':    stability_label,
                'cell_type':    'ALL',
                'Dox':          dox,
                'count':        counts[dox],
                'total_in_dox': totals[dox],
            })
        rows_df = pd.DataFrame(rows)
        rows_df['binom_p_pulse_vs_chase'] = pval
        rows_df['significance']           = stars
        return rows_df

    else:
        # ── Per cell type ─────────────────────────────────────────
        ct_present = [ct for ct in label_order
                      if ct in df['ttest_gmm_label'].unique()]
        n_ct = len(ct_present)
        if n_ct == 0:
            return pd.DataFrame()

        fig, axes = plt.subplots(1, n_ct, figsize=(4 * n_ct, 6),
                                 sharey=False)
        if n_ct == 1:
            axes = [axes]

        all_rows = []

        for ax, ct in zip(axes, ct_present):
            sub    = df[(df['stability'] == stability_label) &
                        (df['ttest_gmm_label'] == ct)]
            totals = {dox: int((df[df['ttest_gmm_label'] == ct]['Dox'] == dox).sum())
                      for dox in dox_present}
            counts = {dox: int((sub['Dox'] == dox).sum()) for dox in dox_present}

            n1, n2     = counts[dox_present[0]], counts[dox_present[1]]
            tot1, tot2 = totals[dox_present[0]], totals[dox_present[1]]
            pval       = _binom_between_dox(n1, tot1, n2, tot2)
            stars      = get_sig(pval)

            bar_vals = [counts[d] for d in dox_present]
            bar_cols = [DOX_COLORS.get(d, color) for d in dox_present]

            bars = ax.bar(np.arange(len(dox_present)), bar_vals,
                          color=bar_cols, edgecolor='black',
                          linewidth=1, width=0.5)

            for bar, val in zip(bars, bar_vals):
                ax.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + max(bar_vals + [1]) * 0.04,
                        str(val), ha='center', va='bottom',
                        fontsize=9, fontweight='bold')

            y_max  = max(bar_vals + [1])
            y_brac = y_max * 1.15
            ax.plot([0, 0, 1, 1],
                    [y_brac - y_max*0.04, y_brac,
                     y_brac, y_brac - y_max*0.04],
                    color='black', lw=1.2)
            pval_txt = f'p={pval:.2e}' if pval < 0.001 else f'p={pval:.3f}'
            ax.text(0.5, y_brac + y_max*0.03,
                    f'{stars}\n{pval_txt}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold',
                    color='black' if stars != 'ns' else '#888888')

            ax.set_xticks(np.arange(len(dox_present)))
            ax.set_xticklabels(['Pulse', 'Chase'], fontsize=9)
            ax.set_title(ct, fontsize=11, fontweight='bold',
                         color=cell_type_colors.get(ct, 'black'))
            ax.set_ylabel('Cell count', fontsize=9)
            ax.set_ylim(0, (y_brac + y_max*0.15) * 1.1)

            for dox in dox_present:
                all_rows.append({
                    'stability':    stability_label,
                    'cell_type':    ct,
                    'Dox':          dox,
                    'count':        counts[dox],
                    'total_in_dox': totals[dox],
                    'binom_p_pulse_vs_chase': pval,
                    'significance':           stars,
                })

        plt.suptitle(title_str, fontsize=13, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {save_path}")

        return pd.DataFrame(all_rows)


# ─────────────────────────────────────────────
# STEP 4: GENERATE PLOTS
# Two views per Dox condition:
#   A) Separate To_Stem / To_TA groups
#   B) Combined To_StemOrTA group
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: GENERATING PLOTS")
print("=" * 60)

# Group orders and palettes for the two views
ORDER_SPLIT    = ['Stable', 'To_Stem', 'To_TA']
PALETTE_SPLIT  = {k: GROUP_COLORS[k] for k in ORDER_SPLIT}

ORDER_COMBINED   = ['Stable', 'To_StemOrTA']
PALETTE_COMBINED = {k: GROUP_COLORS[k] for k in ORDER_COMBINED}

collected_stats = []

def dox_tag(d):
    return str(d).replace(' ', '_').replace('/', '-')

for dox_label, subset in (
    [('ALL', paired)] +
    [(str(d), paired[paired['Dox'] == str(d)]) for d in DOX_VALUES]
):
    tag = dox_tag(dox_label)
    print(f"\n  Dox = {dox_label}  (n={len(subset)})")

    # ── A: Separate Stem / TA destinations ──────────────────────
    # Per cell type
    _s = make_per_celltype_plot(
        subset[subset['transition'].isin(ORDER_SPLIT)],
        group_col      = 'transition',
        group_order_full = ORDER_SPLIT,
        palette        = PALETTE_SPLIT,
        dox_label      = dox_label,
        save_path      = os.path.join(score_dir,
                         f'per_celltype_split_Dox_{tag}.pdf'),
        context        = 'SplitStemTA_PerCelltype',
    )
    collected_stats.append(_s)

    # Combined
    _s = make_combined_plot(
        subset[subset['transition'].isin(ORDER_SPLIT)],
        group_col      = 'transition',
        group_order_full = ORDER_SPLIT,
        palette        = PALETTE_SPLIT,
        dox_label      = dox_label,
        save_path      = os.path.join(score_dir,
                         f'combined_split_Dox_{tag}.pdf'),
        context        = 'SplitStemTA_Combined',
    )
    collected_stats.append(_s)

    # ── B: Merged Stem+TA destination ───────────────────────────
    # Per cell type
    _s = make_per_celltype_plot(
        subset[subset['transition_combined'].isin(ORDER_COMBINED)],
        group_col        = 'transition_combined',
        group_order_full = ORDER_COMBINED,
        palette          = PALETTE_COMBINED,
        dox_label        = dox_label,
        save_path        = os.path.join(score_dir,
                           f'per_celltype_merged_Dox_{tag}.pdf'),
        context          = 'MergedStemTA_PerCelltype',
    )
    collected_stats.append(_s)

    # Combined
    _s = make_combined_plot(
        subset[subset['transition_combined'].isin(ORDER_COMBINED)],
        group_col        = 'transition_combined',
        group_order_full = ORDER_COMBINED,
        palette          = PALETTE_COMBINED,
        dox_label        = dox_label,
        save_path        = os.path.join(score_dir,
                           f'combined_merged_Dox_{tag}.pdf'),
        context          = 'MergedStemTA_Combined',
    )
    collected_stats.append(_s)

    # ── C: Stem/TA stable vs reversed ──────────────────────────
    stemta_subset = paired_stemta if dox_label == 'ALL' else (
        paired_stemta[paired_stemta['Dox'] == str(dox_label)]
    )
    _s = make_stemta_plot(
        stemta_subset,
        dox_label = dox_label,
        save_path = os.path.join(score_dir,
                    f'stemta_stable_vs_reversed_Dox_{tag}.pdf'),
    )
    collected_stats.append(_s)


# ─────────────────────────────────────────────
# COMBINED DOX BARPLOT HELPER
# Shows multiple transition groups side by side,
# each split into Pulse vs Chase bars.
# Layout: one group-pair per cluster of bars.
# Binomial test within each group (pulse vs chase).
# ─────────────────────────────────────────────
def make_combined_dox_barplot(df, group_col, groups, colors,
                               dox_present, title_str, save_path,
                               per_celltype=False):
    """
    For each group in `groups`, draw two adjacent bars (Pulse, Chase).
    Groups are separated by a small gap.
    Binomial test within each group.
    `df` must have columns: group_col, 'Dox', 'ttest_gmm_label'.
    """
    if len(dox_present) < 2:
        print(f"  Skipping — fewer than 2 Dox conditions")
        return pd.DataFrame()

    d1, d2 = dox_present[0], dox_present[1]

    if not per_celltype:
        # ── Overall (pooled) ──────────────────────────────────────
        fig, ax = plt.subplots(figsize=(max(6, 2.2 * len(groups)), 6))

        group_width = 0.35   # width of each individual bar
        gap         = 0.15   # extra gap between groups
        all_rows    = []

        for g_idx, (grp, color) in enumerate(zip(groups, colors)):
            sub   = df[df[group_col] == grp]
            n1    = int((sub['Dox'] == d1).sum())
            n2    = int((sub['Dox'] == d2).sum())
            tot1  = int((df['Dox'] == d1).sum())
            tot2  = int((df['Dox'] == d2).sum())
            pval  = _binom_between_dox(n1, tot1, n2, tot2)
            stars = get_sig(pval)

            x_centre = g_idx * (2 * group_width + gap + 0.1)
            x1 = x_centre - group_width / 2
            x2 = x_centre + group_width / 2

            b1 = ax.bar(x1, n1, group_width,
                        color=DOX_COLORS.get(d1, color),
                        edgecolor='black', linewidth=0.8, label=d1 if g_idx == 0 else '')
            b2 = ax.bar(x2, n2, group_width,
                        color=DOX_COLORS.get(d2, color),
                        edgecolor='black', linewidth=0.8,
                        hatch='//', label=d2 if g_idx == 0 else '')

            for bar, val in [(b1, n1), (b2, n2)]:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + max(n1, n2, 1) * 0.04,
                        str(val), ha='center', va='bottom',
                        fontsize=9, fontweight='bold')

            y_max  = max(n1, n2, 1)
            y_brac = y_max * 1.18
            ax.plot([x1, x1, x2, x2],
                    [y_brac - y_max * 0.04, y_brac,
                     y_brac, y_brac - y_max * 0.04],
                    color='black', lw=1.2)
            pval_txt = f'p={pval:.2e}' if pval < 0.001 else f'p={pval:.3f}'
            ax.text(x_centre, y_brac + y_max * 0.04,
                    f'{stars}\n{pval_txt}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold',
                    color='black' if stars != 'ns' else '#888888')

            ax.text(x_centre, -max(n1, n2, 1) * 0.08, grp,
                    ha='center', va='top', fontsize=9, fontweight='bold',
                    color=color)

            for dox_val, cnt, total in [(d1, n1, tot1), (d2, n2, tot2)]:
                all_rows.append({
                    'group':          grp,
                    'cell_type':      'ALL',
                    'Dox':            dox_val,
                    'count':          cnt,
                    'total_in_dox':   total,
                    'binom_p_pulse_vs_chase': pval,
                    'significance':   stars,
                })

        ax.set_xticks([])
        ax.set_ylabel('Number of cells', fontsize=11, fontweight='bold')
        ax.set_title(title_str, fontsize=12, fontweight='bold')
        handles = [
            plt.Rectangle((0,0), 1, 1,
                           facecolor=DOX_COLORS.get(d1, '#4A90D9'),
                           edgecolor='black', label=DOX_LABELS.get(d1, d1)),
            plt.Rectangle((0,0), 1, 1,
                           facecolor=DOX_COLORS.get(d2, '#E87722'),
                           edgecolor='black', hatch='//',
                           label=DOX_LABELS.get(d2, d2)),
        ]
        ax.legend(handles=handles, loc='upper right', fontsize=9)
        ax.set_ylim(bottom=0)

        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {save_path}")
        return pd.DataFrame(all_rows)

    else:
        # ── Per cell type ─────────────────────────────────────────
        ct_present = [ct for ct in label_order
                      if ct in df['ttest_gmm_label'].unique()]
        if not ct_present:
            return pd.DataFrame()

        n_ct      = len(ct_present)
        fig, axes = plt.subplots(1, n_ct, figsize=(3.5 * n_ct, 6), sharey=False)
        if n_ct == 1:
            axes = [axes]

        all_rows  = []
        group_width = 0.3
        gap         = 0.1

        for ax, ct in zip(axes, ct_present):
            df_ct = df[df['ttest_gmm_label'] == ct]

            for g_idx, (grp, color) in enumerate(zip(groups, colors)):
                sub  = df_ct[df_ct[group_col] == grp]
                tot1 = int((df_ct['Dox'] == d1).sum())
                tot2 = int((df_ct['Dox'] == d2).sum())
                n1   = int((sub['Dox'] == d1).sum())
                n2   = int((sub['Dox'] == d2).sum())
                pval = _binom_between_dox(n1, tot1, n2, tot2)
                stars= get_sig(pval)

                x_centre = g_idx * (2 * group_width + gap + 0.08)
                x1 = x_centre - group_width / 2
                x2 = x_centre + group_width / 2

                b1 = ax.bar(x1, n1, group_width,
                            color=DOX_COLORS.get(d1, color),
                            edgecolor='black', linewidth=0.7)
                b2 = ax.bar(x2, n2, group_width,
                            color=DOX_COLORS.get(d2, color),
                            edgecolor='black', linewidth=0.7, hatch='//')

                y_max  = max(n1, n2, 1)
                for bar, val in [(b1, n1), (b2, n2)]:
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            bar.get_height() + y_max * 0.05,
                            str(val), ha='center', va='bottom', fontsize=7)

                y_brac = y_max * 1.20
                ax.plot([x1, x1, x2, x2],
                        [y_brac - y_max*0.05, y_brac,
                         y_brac, y_brac - y_max*0.05],
                        color='black', lw=1.0)
                pval_txt = f'p={pval:.2e}' if pval < 0.001 else f'p={pval:.3f}'
                ax.text(x_centre, y_brac + y_max * 0.05,
                        f'{stars}\n{pval_txt}',
                        ha='center', va='bottom', fontsize=7,
                        color='black' if stars != 'ns' else '#888888')

                ax.text(x_centre, -y_max * 0.10, grp,
                        ha='center', va='top', fontsize=7,
                        color=color, fontweight='bold')

                for dox_val, cnt, total in [(d1, n1, tot1), (d2, n2, tot2)]:
                    all_rows.append({
                        'group':        grp,
                        'cell_type':    ct,
                        'Dox':          dox_val,
                        'count':        cnt,
                        'total_in_dox': total,
                        'binom_p_pulse_vs_chase': pval,
                        'significance': stars,
                    })

            ax.set_xticks([])
            ax.set_title(ct, fontsize=10, fontweight='bold',
                         color=cell_type_colors.get(ct, 'black'))
            ax.set_ylabel('Cell count', fontsize=8)
            ax.set_ylim(bottom=0)

        plt.suptitle(title_str, fontsize=12, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {save_path}")
        return pd.DataFrame(all_rows)

# ─────────────────────────────────────────────
# STEP 6: BARPLOTS — Pulse vs Chase cell counts
# One plot per transition group:
#   • Stable, To_Stem, To_TA, To_StemOrTA
#     (from the main `paired` df — column `transition` / `transition_combined`)
#   • Reversed_to_Stem, Reversed_to_TA
#     (from `paired_stemta`)
# Per-celltype variants also produced.
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: BARPLOTS — PULSE vs CHASE CELL COUNTS")
print("=" * 60)

barplot_stats = []

paired_dox        = paired[paired['Dox'].isin(DOX_ORDER)].copy()
paired_stemta_dox = paired_stemta[paired_stemta['Dox'].isin(DOX_ORDER)].copy()
dox_present       = [d for d in DOX_ORDER if d in paired_dox['Dox'].unique()]

# ── A: Stable + To_Stem + To_TA  (split view) ─────────────────────
barplot_stats.append(make_combined_dox_barplot(
    df          = paired_dox,
    group_col   = 'transition',
    groups      = ['Stable', 'To_Stem', 'To_TA'],
    colors      = ['#2ecc71', '#9370DB', '#D2B48C'],
    dox_present = dox_present,
    title_str   = 'Stable / To_Stem / To_TA — Pulse vs Chase (binomial test within each group)',
    save_path   = os.path.join(score_dir, 'barplot_stable_toStem_toTA_overall.pdf'),
    per_celltype= False,
))
barplot_stats.append(make_combined_dox_barplot(
    df          = paired_dox,
    group_col   = 'transition',
    groups      = ['Stable', 'To_Stem', 'To_TA'],
    colors      = ['#2ecc71', '#9370DB', '#D2B48C'],
    dox_present = dox_present,
    title_str   = 'Stable / To_Stem / To_TA per cell type — Pulse vs Chase',
    save_path   = os.path.join(score_dir, 'barplot_stable_toStem_toTA_per_celltype.pdf'),
    per_celltype= True,
))

# ── B: Stable + To_StemOrTA  (merged view) ────────────────────────
barplot_stats.append(make_combined_dox_barplot(
    df          = paired_dox,
    group_col   = 'transition_combined',
    groups      = ['Stable', 'To_StemOrTA'],
    colors      = ['#2ecc71', '#5B4A8A'],
    dox_present = dox_present,
    title_str   = 'Stable / To_StemOrTA — Pulse vs Chase (binomial test within each group)',
    save_path   = os.path.join(score_dir, 'barplot_stable_toStemOrTA_overall.pdf'),
    per_celltype= False,
))
barplot_stats.append(make_combined_dox_barplot(
    df          = paired_dox,
    group_col   = 'transition_combined',
    groups      = ['Stable', 'To_StemOrTA'],
    colors      = ['#2ecc71', '#5B4A8A'],
    dox_present = dox_present,
    title_str   = 'Stable / To_StemOrTA per cell type — Pulse vs Chase',
    save_path   = os.path.join(score_dir, 'barplot_stable_toStemOrTA_per_celltype.pdf'),
    per_celltype= True,
))

# ── C: Stable_Stem/TA vs Reversed_to_Stem/TA  (stemta view) ───────
for dest_ct, stbl_grp, rev_grp, stbl_col, rev_col in [
    ('Stem', 'Stable_Stem', 'Reversed_to_Stem', '#2ecc71', '#9370DB'),
    ('TA',   'Stable_TA',   'Reversed_to_TA',   '#27ae60', '#b8860b'),
]:
    sub = paired_stemta_dox[paired_stemta_dox['stemta_dest'] == dest_ct].copy()
    sub['transition']     = sub['stemta_group']
    sub['ttest_gmm_label']= sub.get('ttest_gmm_label',
                                     sub.get('dcm_cell_type', 'unknown'))
    barplot_stats.append(make_combined_dox_barplot(
        df          = sub,
        group_col   = 'transition',
        groups      = [stbl_grp, rev_grp],
        colors      = [stbl_col, rev_col],
        dox_present = dox_present,
        title_str   = (f'{stbl_grp} vs {rev_grp} — Pulse vs Chase\n'
                       f'(binomial test within each group)'),
        save_path   = os.path.join(score_dir,
                                   f'barplot_{dest_ct}_stable_vs_reversed_overall.pdf'),
        per_celltype= False,
    ))
    barplot_stats.append(make_combined_dox_barplot(
        df          = sub,
        group_col   = 'transition',
        groups      = [stbl_grp, rev_grp],
        colors      = [stbl_col, rev_col],
        dox_present = dox_present,
        title_str   = (f'{stbl_grp} vs {rev_grp} per cell type — Pulse vs Chase'),
        save_path   = os.path.join(score_dir,
                                   f'barplot_{dest_ct}_stable_vs_reversed_per_celltype.pdf'),
        per_celltype= True,
    ))

# Save barplot stats CSV
bp_stats_df  = pd.concat(
    [s for s in barplot_stats if s is not None and not s.empty],
    ignore_index=True
)
bp_stats_csv = os.path.join(score_dir, 'pulse_vs_chase_count_stats.csv')
bp_stats_df.to_csv(bp_stats_csv, index=False)
print(f"\nBarplot stats CSV: {bp_stats_csv}")
print(bp_stats_df.to_string(index=False))


# ─────────────────────────────────────────────
# STEP 5: CSV OUTPUTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: SAVING CSV OUTPUTS")
print("=" * 60)

stats_df  = pd.concat([s for s in collected_stats if not s.empty],
                      ignore_index=True)
stats_csv = os.path.join(score_dir, 'stable_vs_reprogrammed_stats.csv')
stats_df.to_csv(stats_csv, index=False)
print(f"Stats CSV : {stats_csv}")
print(stats_df.to_string(index=False))

out_cols = (['batch_plate_well', 'Dox', 'ttest_gmm_label', 'is_multipotent',
             'rna_celltype', 'transition', 'transition_combined',
             'self_module_score'] + ttest_cols)
data_csv = os.path.join(score_dir, 'stable_vs_reprogrammed_data.csv')
paired[out_cols].to_csv(data_csv, index=False)
print(f"Data CSV  : {data_csv}")

print("\nDone!")
print(f"Outputs in: {score_dir}")

In [ ]:
#Negative control on module scores

In [ ]:
score_dir = os.path.join(root, 'gene_scoring','gmm_085', 'MS_stable_vs_reprogrammed','Neg_ctrl')


In [ ]:
##############################################
# STABLE vs REPROGRAMMED-TO-STEM/TA ANALYSIS
# adata_combined
#
# For each DCM cell type (secretory / absorptive),
# cells are split into three groups:
#   Stable      — DCM label == RNA label
#   To_Stem     — DCM label != RNA label AND RNA == Stem
#   To_TA       — DCM label != RNA label AND RNA == TA
#   To_StemOrTA — combined Stem + TA destination
#
# The module score (ttest_<CellType>_module) for
# that cell type is plotted per group, per Dox,
# and combined (ALL).
#
# Hybrid / multipotent DCM cells are excluded.
# t-test: each unstable group vs Stable.
# CSV output with all stats.
#
# NEGATIVE CONTROLS (new):
#   per_celltype_merged plots:
#     Enterocyte subplot → adds NegCtrl = Stable secretory cells
#                          (Paneth+Goblet+Tuft+EEC) scored on Enterocyte module
#     Secretory subplots → adds NegCtrl = Stable Enterocyte cells
#                          scored on that secretory module
#   stemta_stable_vs_reversed plots:
#     Adds NegCtrl_Absorptive = Stable Enterocytes
#     Adds NegCtrl_Secretory  = Stable Paneth+Goblet+Tuft+EEC
#     Both scored on the Stem/TA module respectively
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
root      = "/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py"
score_dir = os.path.join(root, 'gene_scoring','gmm_085', 'MS_stable_vs_reprogrammed','Neg_ctrl')
os.makedirs(score_dir, exist_ok=True)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
cell_type_colors = {
    'Stem'      : '#9370DB',
    'TA'        : '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth'    : '#DC143C',
    'Goblet'    : '#2E8B57',
    'Tuft'      : '#FF1493',
    'EEC'       : '#4169E1',
}

GROUP_COLORS = {
    'Stable':              '#2ecc71',
    'To_Stem':             '#9370DB',
    'To_TA':               '#D2B48C',
    'To_StemOrTA':         '#5B4A8A',
    'Stable_Stem':         '#2ecc71',
    'Stable_TA':           '#27ae60',
    'Reversed_to_Stem':    '#9370DB',
    'Reversed_to_TA':      '#b8860b',
    'NegCtrl':             '#AAAAAA',   # grey — negative control
    'NegCtrl_Absorptive':  '#FF8C00',   # orange tint — Enterocyte-based neg ctrl
    'NegCtrl_Secretory':   '#DC143C',   # crimson tint — secretory-based neg ctrl
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# Cell-type class membership
ABSORPTIVE_TYPES = ['Enterocyte']
SECRETORY_TYPES  = ['Paneth', 'Goblet', 'Tuft', 'EEC']

DOX_COLORS = {
    '48hDox'         : '#4A90D9',
    '48hDox-72hChase': '#E87722',
}
DOX_ORDER  = ['48hDox', '48hDox-72hChase']
DOX_LABELS = {'48hDox': 'Pulse (48hDox)', '48hDox-72hChase': 'Chase (48hDox-72hChase)'}

ANALYSIS_TYPES = ['Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# ─────────────────────────────────────────────
# STEP 1: BUILD DATAFRAME
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: BUILDING DATAFRAME")
print("=" * 60)

adata = adata_combined

ttest_cols = [c for c in adata.obs.columns
              if c.startswith('ttest_') and c.endswith('_module')
              and 'max' not in c.lower()]

ct_from_col = {col: col.replace('ttest_', '').replace('_module', '')
               for col in ttest_cols}
col_from_ct = {v: k for k, v in ct_from_col.items()}

print(f"Module score columns: {ttest_cols}")

base = pd.DataFrame({
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'assay':            adata.obs['assay'].values,
    'Dox':              adata.obs['Dox'].astype(str).values,
    'leiden_annot_25':  adata.obs['leiden_annot_25'].astype(str)
                                 .str.split('_').str[-1].values,
    'ttest_gmm_label':  adata.obs['ttest_gmm_label_confident_085'].astype(str)
                                 .str.split('_').str[-1].values,
    'is_multipotent':   adata.obs['is_multipotent'].astype(str).values,
}, index=adata.obs_names)

for col in ttest_cols:
    base[col] = adata.obs[col].values

print(f"Total cells : {len(base)}")
print(f"Assay values: {sorted(base['assay'].unique())}")
print(f"Dox values  : {sorted(base['Dox'].unique())}")

DOX_VALUES = sorted(base['Dox'].dropna().unique())

# ─────────────────────────────────────────────
# STEP 2: PAIR pDCM ↔ pRNA  (1:1)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: PAIRING pDCM ↔ pRNA")
print("=" * 60)

dcm_df = base[base['assay'] == 'pDCM'].copy()
rna_df = (base[base['assay'] == 'pRNA']
          [['batch_plate_well', 'leiden_annot_25']]
          .copy()
          .rename(columns={'leiden_annot_25': 'rna_celltype'}))

def match_one_to_one(df_a, df_b):
    common = set(df_a['batch_plate_well']) & set(df_b['batch_plate_well'])
    a_s = set(df_a[df_a['batch_plate_well'].isin(common)]
              ['batch_plate_well'].value_counts()
              .pipe(lambda s: s[s == 1].index))
    b_s = set(df_b[df_b['batch_plate_well'].isin(common)]
              ['batch_plate_well'].value_counts()
              .pipe(lambda s: s[s == 1].index))
    return a_s & b_s

matched_wells = match_one_to_one(dcm_df, rna_df)
print(f"1:1 matched wells: {len(matched_wells)}")

paired = (dcm_df[dcm_df['batch_plate_well'].isin(matched_wells)]
          .merge(rna_df[rna_df['batch_plate_well'].isin(matched_wells)],
                 on='batch_plate_well', how='inner'))
print(f"Paired cells: {len(paired)}")

# ─────────────────────────────────────────────
# STEP 3: FILTER AND CLASSIFY
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: CLASSIFYING")
print("=" * 60)

paired = paired[paired['ttest_gmm_label'].isin(label_order)].copy()
n_before = len(paired)
paired = paired[
    ~paired['is_multipotent'].str.lower().isin(['true', '1', 'yes'])
].copy()
print(f"Removed {n_before - len(paired)} hybrid cells ({n_before} → {len(paired)})")

paired = paired[paired['ttest_gmm_label'].isin(ANALYSIS_TYPES)].copy()
print(f"Cells with DCM in ANALYSIS_TYPES: {len(paired)}")

def classify(row):
    dcm = row['ttest_gmm_label']
    rna = row['rna_celltype']
    if dcm == rna:
        return 'Stable'
    if rna == 'Stem':
        return 'To_Stem'
    if rna == 'TA':
        return 'To_TA'
    return 'Other'

paired['transition'] = paired.apply(classify, axis=1)
paired['transition_combined'] = paired['transition'].replace(
    {'To_Stem': 'To_StemOrTA', 'To_TA': 'To_StemOrTA'}
)

print("\nTransition counts (ALL Dox):")
print(paired['transition'].value_counts().to_string())
print("\nPer DCM cell type:")
print(paired.groupby(['ttest_gmm_label', 'transition']).size()
           .unstack(fill_value=0).to_string())

def get_self_module(row):
    col = col_from_ct.get(row['ttest_gmm_label'])
    return row[col] if (col is not None and col in row.index) else np.nan

paired['self_module_score'] = paired.apply(get_self_module, axis=1)

# ─────────────────────────────────────────────
# BUILD STEM/TA REFERENCE DATAFRAME
# ─────────────────────────────────────────────
_all_paired_raw = (dcm_df[dcm_df['batch_plate_well'].isin(matched_wells)]
                   .merge(rna_df[rna_df['batch_plate_well'].isin(matched_wells)],
                          on='batch_plate_well', how='inner'))

_all_paired_raw = _all_paired_raw[
    ~_all_paired_raw['is_multipotent'].str.lower().isin(['true', '1', 'yes'])
].copy()

rows_stemta = []
for dest_ct in ['Stem', 'TA']:
    mod_col = col_from_ct.get(dest_ct)
    if mod_col is None:
        continue

    stable_sub = _all_paired_raw[
        (_all_paired_raw['ttest_gmm_label'] == dest_ct) &
        (_all_paired_raw['rna_celltype']    == dest_ct)
    ].copy()
    stable_sub['stemta_group']  = f'Stable_{dest_ct}'
    stable_sub['stemta_dest']   = dest_ct
    stable_sub['stemta_score']  = stable_sub[mod_col]

    reversed_sub = paired[paired['rna_celltype'] == dest_ct].copy()
    reversed_sub['stemta_group']  = f'Reversed_to_{dest_ct}'
    reversed_sub['stemta_dest']   = dest_ct
    reversed_sub['stemta_score']  = reversed_sub[mod_col]

    rows_stemta.append(stable_sub)
    rows_stemta.append(reversed_sub)

paired_stemta = pd.concat(rows_stemta, ignore_index=True)

print(f"\nStem/TA comparison dataframe: {len(paired_stemta)} rows")
print(paired_stemta.groupby(['stemta_dest','stemta_group']).size().to_string())

# ─────────────────────────────────────────────
# NEGATIVE CONTROL HELPERS
# ─────────────────────────────────────────────

def get_neg_ctrl_class(ct):
    """
    Return the opposite class label for a given cell type.
    Enterocyte → 'Secretory'; Paneth/Goblet/Tuft/EEC → 'Absorptive'
    """
    if ct in ABSORPTIVE_TYPES:
        return 'Secretory'
    return 'Absorptive'

def build_neg_ctrl_for_celltype(paired_all, ct, dox_subset=None):
    """
    For a given focal cell type `ct`, extract Stable cells of the
    opposite class and return their score on the `ct` module.

    `paired_all` should be the full `paired` df (all ANALYSIS_TYPES,
    already filtered for non-hybrids) so that cells of the opposite
    class are available even when the caller is iterating over a
    single focal cell type.

    Returns a small DataFrame with columns:
        group, score, ttest_gmm_label
    where group == 'NegCtrl (<class>)'.
    """
    mod_col = col_from_ct.get(ct)
    if mod_col is None:
        return pd.DataFrame()

    if ct in ABSORPTIVE_TYPES:
        ctrl_types = SECRETORY_TYPES
        ctrl_label = 'NegCtrl (Secretory)'
    else:
        ctrl_types = ABSORPTIVE_TYPES
        ctrl_label = 'NegCtrl (Absorptive)'

    # Stable cells of the opposite class
    src = paired_all if dox_subset is None else paired_all[
        paired_all['Dox'].isin(dox_subset)
    ]
    ctrl = src[
        (src['ttest_gmm_label'].isin(ctrl_types)) &
        (src['transition'] == 'Stable')
    ].copy()

    if ctrl.empty:
        return pd.DataFrame()

    ctrl['group'] = ctrl_label
    ctrl['score'] = ctrl[mod_col]
    return ctrl[['group', 'score', 'ttest_gmm_label']]


def build_neg_ctrl_stemta(paired_all_raw, dest_ct, dox_subset=None):
    """
    Build a single merged negative-control group for a Stem/TA module score plot:
      NegCtrl — Stable Absorptive (Enterocytes) + Stable Secretory
                (Paneth+Goblet+Tuft+EEC) pooled together, scored on dest_ct module.

    `paired_all_raw` = _all_paired_raw (full paired df before ANALYSIS_TYPES filter,
                        hybrids already removed).
    Returns a DataFrame with columns: stemta_group, stemta_score, ttest_gmm_label
    """
    mod_col = col_from_ct.get(dest_ct)
    if mod_col is None:
        return pd.DataFrame()

    src = paired_all_raw if dox_subset is None else paired_all_raw[
        paired_all_raw['Dox'].isin(dox_subset)
    ]

    all_ctrl_types = ABSORPTIVE_TYPES + SECRETORY_TYPES
    ctrl = src[
        (src['ttest_gmm_label'].isin(all_ctrl_types)) &
        (src['rna_celltype'] == src['ttest_gmm_label'])   # Stable: DCM == RNA
    ].copy()

    if ctrl.empty:
        return pd.DataFrame()

    ctrl['stemta_group'] = 'NegCtrl'
    ctrl['stemta_dest']  = dest_ct
    ctrl['stemta_score'] = ctrl[mod_col]
    return ctrl[['stemta_group', 'stemta_dest', 'stemta_score',
                 'ttest_gmm_label', 'Dox']]


# ─────────────────────────────────────────────
# STATS HELPERS
# ─────────────────────────────────────────────
def get_sig(p):
    if pd.isna(p):  return ''
    if p < 0.001:   return '***'
    if p < 0.01:    return '**'
    if p < 0.05:    return '*'
    return 'ns'

def ttest_vs_ref(df, score_col, group_col, test_group, ref_group='Stable'):
    s = df[df[group_col] == ref_group  ][score_col].dropna()
    u = df[df[group_col] == test_group ][score_col].dropna()
    if len(s) > 1 and len(u) > 1:
        return stats.ttest_ind(u, s)[1]
    return np.nan

# Keep old name as alias for backward compat
def ttest_vs_stable(df, score_col, group_col, test_group):
    return ttest_vs_ref(df, score_col, group_col, test_group, ref_group='Stable')

def compute_stats(df, score_col, group_col, dox_label, ct, context):
    rows = []
    for grp in df[group_col].unique():
        vals = df[df[group_col] == grp][score_col].dropna()
        pval = ttest_vs_stable(df, score_col, group_col, grp) if grp != 'Stable' else np.nan
        rows.append({
            'context':              context,
            'cell_type':            ct,
            'Dox':                  dox_label,
            'group':                grp,
            'score_col':            score_col,
            'N':                    len(vals),
            'mean':                 vals.mean()   if len(vals) else np.nan,
            'median':               vals.median() if len(vals) else np.nan,
            'std':                  vals.std()    if len(vals) else np.nan,
            'ttest_vs_Stable_pval': pval,
            'significance':         get_sig(pval),
        })
    return pd.DataFrame(rows)

# ─────────────────────────────────────────────
# PLOT HELPERS
# ─────────────────────────────────────────────
def draw_bracket(ax, group_order, i_test, i_ref, y_lvl, stars):
    ax.annotate('', xy=(i_test, y_lvl), xytext=(i_ref, y_lvl),
                arrowprops=dict(arrowstyle='-', color='#333333', lw=0.9))
    x_mid = (i_test + i_ref) / 2
    ax.text(x_mid, y_lvl, stars,
            ha='center', va='bottom', fontsize=10, fontweight='bold',
            color='black' if stars not in ('ns', '') else '#888888')


def make_per_celltype_plot(df, group_col, group_order_full,
                           palette, dox_label, save_path, context):
    """
    One subplot per cell type; each subplot shows the self-module
    score split by transition group.
    """
    ct_present = []
    for ct in ANALYSIS_TYPES:
        sub = df[df['ttest_gmm_label'] == ct]
        grps = sub[group_col].unique()
        if 'Stable' in grps and len(grps) > 1:
            ct_present.append(ct)

    if not ct_present:
        print(f"  Skipping per-celltype Dox={dox_label} — no data")
        return pd.DataFrame()

    n_ct  = len(ct_present)
    fig, axes = plt.subplots(1, n_ct, figsize=(4.5 * n_ct, 6), sharey=False)
    if n_ct == 1:
        axes = [axes]

    all_stats = []

    for ax, ct in zip(axes, ct_present):
        col = col_from_ct.get(ct)
        if col is None:
            ax.set_visible(False)
            continue

        sub = df[df['ttest_gmm_label'] == ct].copy()
        present = [g for g in group_order_full if g in sub[group_col].values]

        plot_sub = sub[[col, group_col]].rename(columns={col: 'score'})

        sns.boxplot(
            data=plot_sub, x=group_col, y='score',
            hue=group_col, order=present,
            palette=palette, legend=False, ax=ax,
            flierprops=dict(marker='o', markersize=2, alpha=0.4),
        )

        ylim  = ax.get_ylim()
        y_top = ylim[1]

        for i, grp in enumerate(present):
            n = len(plot_sub[plot_sub[group_col] == grp])
            ax.text(i, y_top * 1.01, f'n={n}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

        if 'Stable' in present:
            ref_i    = present.index('Stable')
            y_levels = [y_top * 1.09, y_top * 1.17]
            y_max    = y_levels[0]
            k = 0
            for grp in present:
                if grp == 'Stable':
                    continue
                pval  = ttest_vs_stable(plot_sub, 'score', group_col, grp)
                stars = get_sig(pval)
                y_lvl = y_levels[k % 2]
                y_max = max(y_max, y_lvl)
                draw_bracket(ax, present, present.index(grp), ref_i, y_lvl, stars)
                k += 1
            ax.set_ylim(ylim[0], y_max * 1.12)

        ax.set_title(ct, fontsize=12, fontweight='bold',
                     color=cell_type_colors.get(ct, 'black'))
        ax.set_xlabel('')
        ax.set_ylabel(f'{ct} module score', fontsize=9)
        ax.tick_params(axis='x', rotation=20, labelsize=9)

        all_stats.append(compute_stats(sub, col, group_col,
                                       dox_label, ct, context))

    plt.suptitle(
        f'Module Scores — Stable vs Reprogrammed to Stem/TA | Dox = {dox_label}\n'
        f't-tests vs Stable within each cell type',
        fontsize=13, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

    return pd.concat(all_stats, ignore_index=True) if all_stats else pd.DataFrame()


def make_per_celltype_merged_plot_with_negctrl(
        df, paired_full, group_col, group_order_full,
        palette, dox_label, save_path, context):
    """
    Like make_per_celltype_plot but for the merged (Stable vs To_StemOrTA) view,
    with a negative control boxplot appended to each cell type subplot.

    Negative control:
      Enterocyte subplot → Stable secretory cells (Paneth+Goblet+Tuft+EEC)
                           scored on the Enterocyte module
      Secretory subplots → Stable Enterocyte cells
                           scored on that secretory module
    The NegCtrl box sits to the right of the existing groups, separated
    by a dashed vertical line, and is NOT connected by a bracket to Stable.
    """
    ct_present = []
    for ct in ANALYSIS_TYPES:
        sub = df[df['ttest_gmm_label'] == ct]
        grps = sub[group_col].unique()
        if 'Stable' in grps and len(grps) > 1:
            ct_present.append(ct)

    if not ct_present:
        print(f"  Skipping per-celltype-merged+negctrl Dox={dox_label} — no data")
        return pd.DataFrame()

    # Determine which Dox values are active in this subset
    active_dox = list(df['Dox'].unique()) if dox_label != 'ALL' else None

    n_ct  = len(ct_present)
    fig, axes = plt.subplots(1, n_ct, figsize=(5 * n_ct, 6), sharey=False)
    if n_ct == 1:
        axes = [axes]

    all_stats = []

    NEGCTRL_COLOR = '#BBBBBB'   # uniform grey for all neg-ctrl boxes

    for ax, ct in zip(axes, ct_present):
        col = col_from_ct.get(ct)
        if col is None:
            ax.set_visible(False)
            continue

        sub = df[df['ttest_gmm_label'] == ct].copy()
        present = [g for g in group_order_full if g in sub[group_col].values]

        # Build neg-ctrl data
        nc_df = build_neg_ctrl_for_celltype(paired_full, ct,
                                             dox_subset=active_dox)
        neg_ctrl_label = nc_df['group'].iloc[0] if not nc_df.empty else None

        # Concatenate main + neg-ctrl for a single boxplot call
        main_plot = sub[[col, group_col]].rename(
            columns={col: 'score', group_col: 'group'})
        main_plot['is_negctrl'] = False

        if neg_ctrl_label is not None:
            nc_plot = nc_df[['score', 'group']].copy()
            nc_plot['is_negctrl'] = True
            all_data = pd.concat([main_plot, nc_plot], ignore_index=True)
        else:
            all_data = main_plot

        full_order  = present + ([neg_ctrl_label] if neg_ctrl_label else [])
        full_palette = {g: palette.get(g, NEGCTRL_COLOR) for g in full_order}
        if neg_ctrl_label:
            full_palette[neg_ctrl_label] = NEGCTRL_COLOR

        sns.boxplot(
            data=all_data, x='group', y='score',
            hue='group', order=full_order,
            palette=full_palette, legend=False, ax=ax,
            flierprops=dict(marker='o', markersize=2, alpha=0.4),
        )

        ylim  = ax.get_ylim()
        y_top = ylim[1]

        # n labels
        for i, grp in enumerate(full_order):
            n = int((all_data['group'] == grp).sum())
            ax.text(i, y_top * 1.01, f'n={n}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

        # Brackets vs Stable (only for the main groups, not NegCtrl)
        y_max_bracket = y_top
        if 'Stable' in present:
            ref_i    = full_order.index('Stable')
            y_levels = [y_top * 1.09, y_top * 1.17]
            y_max_bracket = y_levels[0]
            k = 0
            for grp in present:
                if grp == 'Stable':
                    continue
                pval  = ttest_vs_ref(all_data, 'score', 'group', grp,
                                     ref_group='Stable')
                stars = get_sig(pval)
                y_lvl = y_levels[k % 2]
                y_max_bracket = max(y_max_bracket, y_lvl)
                draw_bracket(ax, full_order, full_order.index(grp),
                             ref_i, y_lvl, stars)
                k += 1

        # Dashed separator before NegCtrl column
        if neg_ctrl_label and len(present) > 0:
            sep_x = len(present) - 0.5
            ax.axvline(sep_x, color='#666666', lw=1.2,
                       linestyle='--', alpha=0.7)
            # t-test: NegCtrl vs Stable (shown below separator, no bracket line)
            pval_nc = ttest_vs_ref(all_data, 'score', 'group',
                                   neg_ctrl_label, ref_group='Stable')
            stars_nc = get_sig(pval_nc)
            ax.text(len(full_order) - 1, y_max_bracket * 1.05,
                    f'vs Stable:\n{stars_nc}',
                    ha='center', va='bottom', fontsize=7,
                    color='#555555', style='italic')

        ax.set_ylim(ylim[0], max(y_max_bracket, y_top) * 1.22)

        ax.set_title(ct, fontsize=12, fontweight='bold',
                     color=cell_type_colors.get(ct, 'black'))
        ax.set_xlabel('')
        ax.set_ylabel(f'{ct} module score', fontsize=9)
        ax.tick_params(axis='x', rotation=25, labelsize=8)

        all_stats.append(compute_stats(
            all_data, 'score', 'group', dox_label, ct, context))

    plt.suptitle(
        f'Module Scores — Stable vs To_StemOrTA | Dox = {dox_label}\n'
        f'Grey box = negative control (opposite-class Stable cells)\n'
        f't-tests vs Stable; dashed line separates neg ctrl',
        fontsize=12, fontweight='bold', y=1.03
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

    return pd.concat(all_stats, ignore_index=True) if all_stats else pd.DataFrame()


def make_combined_plot(df, group_col, group_order_full,
                       palette, dox_label, save_path, context):
    """
    All cell types pooled; each cell uses its own self_module_score.
    """
    sub = df[df['self_module_score'].notna()].copy()
    present = [g for g in group_order_full if g in sub[group_col].values]

    if len(present) < 2 or 'Stable' not in present:
        print(f"  Skipping combined Dox={dox_label} — insufficient groups")
        return pd.DataFrame()

    fig, ax = plt.subplots(figsize=(max(5, 2 * len(present)), 6))

    sns.boxplot(
        data=sub, x=group_col, y='self_module_score',
        hue=group_col, order=present,
        palette=palette, legend=False, ax=ax,
        flierprops=dict(marker='o', markersize=2, alpha=0.3),
        width=0.5,
    )
    sns.stripplot(
        data=sub, x=group_col, y='self_module_score',
        hue='ttest_gmm_label', order=present,
        palette=cell_type_colors, dodge=False,
        alpha=0.5, size=3, jitter=True, ax=ax,
    )
    h, l = ax.get_legend_handles_labels()
    ax.legend(h, l, title='Cell type',
              bbox_to_anchor=(1.01, 1), loc='upper left',
              fontsize=8, frameon=True)

    ylim  = ax.get_ylim()
    y_top = ylim[1]

    for i, grp in enumerate(present):
        n = len(sub[sub[group_col] == grp])
        ax.text(i, y_top * 1.01, f'n={n}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

    if 'Stable' in present:
        ref_i    = present.index('Stable')
        y_levels = [y_top * 1.09, y_top * 1.17]
        y_max    = y_levels[0]
        k = 0
        for grp in present:
            if grp == 'Stable':
                continue
            pval  = ttest_vs_stable(sub, 'self_module_score', group_col, grp)
            stars = get_sig(pval)
            y_lvl = y_levels[k % 2]
            y_max = max(y_max, y_lvl)
            draw_bracket(ax, present, present.index(grp), ref_i, y_lvl, stars)
            k += 1
        ax.set_ylim(ylim[0], y_max * 1.13)

    ax.set_xlabel('')
    ax.set_ylabel('Self-module score (own DCM cell-type module)', fontsize=10)
    ax.set_title(
        f'All cell types combined — Stable vs Reprogrammed to Stem/TA\n'
        f'Dox = {dox_label}',
        fontsize=12, fontweight='bold'
    )
    ax.tick_params(axis='x', rotation=20, labelsize=10)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

    return compute_stats(sub, 'self_module_score', group_col,
                         dox_label, 'ALL_combined', context)


# ─────────────────────────────────────────────
# STEM/TA COMPARISON PLOT WITH NEGATIVE CONTROLS
# ─────────────────────────────────────────────
def make_stemta_plot(df, dox_label, save_path,
                     paired_all_raw=None):
    """
    One subplot per destination (Stem, TA).
    Groups shown:
        Stable_<dest>       — DCM==dest and RNA==dest
        Reversed_to_<dest>  — DCM in ANALYSIS_TYPES, RNA==dest
        NegCtrl_Absorptive  — Stable Enterocytes scored on dest module
        NegCtrl_Secretory   — Stable Paneth+Goblet+Tuft+EEC scored on dest module

    NegCtrl groups are separated by a dashed vertical line and annotated
    with their t-test vs Stable_<dest> significance (no bracket line drawn).
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    all_stats = []

    # Determine active Dox for neg-ctrl filtering
    active_dox = list(df['Dox'].unique()) if dox_label != 'ALL' else None

    for ax, dest_ct in zip(axes, ['Stem', 'TA']):
        mod_col = col_from_ct.get(dest_ct)
        if mod_col is None:
            ax.set_visible(False)
            continue

        sub = df[df['stemta_dest'] == dest_ct][
            ['stemta_group', 'stemta_score', 'ttest_gmm_label', 'Dox']
        ].copy()

        stbl   = f'Stable_{dest_ct}'
        rev    = f'Reversed_to_{dest_ct}'
        main_order = [g for g in [stbl, rev] if g in sub['stemta_group'].values]

        if len(main_order) < 2:
            ax.set_visible(False)
            continue

        # Build merged neg-ctrl group (Stable Absorptive + Stable Secretory pooled)
        nc_combined = pd.DataFrame()
        if paired_all_raw is not None:
            nc_df = build_neg_ctrl_stemta(paired_all_raw, dest_ct,
                                          dox_subset=active_dox)
            if not nc_df.empty:
                nc_combined = nc_df[['stemta_group', 'stemta_score',
                                     'ttest_gmm_label']].copy()

        nc_labels = ['NegCtrl'] if not nc_combined.empty else []
        full_order = main_order + nc_labels

        pal = {stbl: GROUP_COLORS[stbl], rev: GROUP_COLORS[rev]}
        if nc_labels:
            pal['NegCtrl'] = '#AAAAAA'   # single grey box

        # Combine for plotting
        plot_df = sub[['stemta_group', 'stemta_score', 'ttest_gmm_label']].copy()
        if not nc_combined.empty:
            plot_df = pd.concat([plot_df, nc_combined], ignore_index=True)

        sns.boxplot(
            data=plot_df, x='stemta_group', y='stemta_score',
            hue='stemta_group', order=full_order,
            palette=pal, legend=False, ax=ax,
            flierprops=dict(marker='o', markersize=2, alpha=0.4),
        )
        sns.stripplot(
            data=plot_df, x='stemta_group', y='stemta_score',
            hue='ttest_gmm_label', order=full_order,
            palette=cell_type_colors, dodge=False,
            alpha=0.5, size=3, jitter=True, ax=ax,
        )
        h, l = ax.get_legend_handles_labels()
        ax.legend(h, l, title='DCM cell type',
                  bbox_to_anchor=(1.01, 1), loc='upper left',
                  fontsize=7, frameon=True)

        ylim  = ax.get_ylim()
        y_top = ylim[1]

        # n labels
        for i, grp in enumerate(full_order):
            n = int((plot_df['stemta_group'] == grp).sum())
            ax.text(i, y_top * 1.01, f'n={n}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

        # Bracket: Reversed vs Stable (main comparison)
        s_vals = plot_df[plot_df['stemta_group'] == stbl]['stemta_score'].dropna()
        r_vals = plot_df[plot_df['stemta_group'] == rev ]['stemta_score'].dropna()
        y_brac = y_top
        if len(s_vals) > 1 and len(r_vals) > 1:
            _, pval = stats.ttest_ind(r_vals, s_vals)
            stars   = get_sig(pval)
            y_ann   = y_top * 1.09
            ax.annotate('', xy=(full_order.index(rev), y_ann),
                        xytext=(full_order.index(stbl), y_ann),
                        arrowprops=dict(arrowstyle='-', color='#333333', lw=1.0))
            ax.text((full_order.index(stbl) + full_order.index(rev)) / 2,
                    y_ann, stars,
                    ha='center', va='bottom', fontsize=11, fontweight='bold',
                    color='black' if stars not in ('ns', '') else '#888888')
            y_brac = y_ann
        else:
            pval = np.nan

        # Dashed separator + neg-ctrl annotations
        if nc_labels:
            sep_x = len(main_order) - 0.5
            ax.axvline(sep_x, color='#666666', lw=1.2,
                       linestyle='--', alpha=0.7)
            ax.text(sep_x + 0.02, ylim[0] + (ylim[1]-ylim[0])*0.02,
                    'Neg. controls', fontsize=7, color='#555555',
                    rotation=90, va='bottom', ha='left', style='italic')

            nc_i = full_order.index('NegCtrl')
            nc_v = plot_df[plot_df['stemta_group'] == 'NegCtrl']['stemta_score'].dropna()

            # t-test: NegCtrl vs Stable_<dest>
            pval_nc_vs_stable = np.nan
            if len(s_vals) > 1 and len(nc_v) > 1:
                _, pval_nc_vs_stable = stats.ttest_ind(nc_v, s_vals)
            stars_nc_vs_stable = get_sig(pval_nc_vs_stable)
            ax.text(nc_i, y_brac * 1.05,
                    f'vs {stbl}:\n{stars_nc_vs_stable}',
                    ha='center', va='bottom', fontsize=7,
                    color='#444444', style='italic')

            # t-test: Reversed vs NegCtrl — drawn as a bracket above the plot
            pval_nc = np.nan
            rev_i = full_order.index(rev)
            if len(r_vals) > 1 and len(nc_v) > 1:
                _, pval_nc = stats.ttest_ind(r_vals, nc_v)
            stars_rev_nc = get_sig(pval_nc)
            y_rev_nc = max(y_brac, y_top) * 1.17
            ax.annotate('', xy=(nc_i, y_rev_nc), xytext=(rev_i, y_rev_nc),
                        arrowprops=dict(arrowstyle='-', color='#333333', lw=0.9))
            ax.text((rev_i + nc_i) / 2, y_rev_nc, stars_rev_nc,
                    ha='center', va='bottom', fontsize=10, fontweight='bold',
                    color='black' if stars_rev_nc not in ('ns', '') else '#888888')
            # extend ylim to fit new bracket
            ax.set_ylim(ylim[0], y_rev_nc * 1.15)

        ax.set_ylim(ylim[0], max(y_brac, y_top) * 1.25)

        ax.set_title(f'{dest_ct} module score', fontsize=12, fontweight='bold',
                     color=cell_type_colors.get(dest_ct, 'black'))
        ax.set_xlabel('')
        ax.set_ylabel(f'{dest_ct} module score (ttest zscore)', fontsize=9)
        ax.tick_params(axis='x', rotation=20, labelsize=8)

        # Stats rows — main groups
        for grp in main_order:
            vals = plot_df[plot_df['stemta_group'] == grp]['stemta_score'].dropna()
            pv   = pval if grp == rev else np.nan
            all_stats.append({
                'context':              'StemTA_StableVsReversed',
                'cell_type':            dest_ct,
                'Dox':                  dox_label,
                'group':                grp,
                'score_col':            mod_col,
                'N':                    len(vals),
                'mean':                 vals.mean()   if len(vals) else np.nan,
                'median':               vals.median() if len(vals) else np.nan,
                'std':                  vals.std()    if len(vals) else np.nan,
                'ttest_vs_Stable_pval': pv,
                'significance':         get_sig(pv) if grp == rev else '',
            })
        # Stats rows — merged neg-ctrl
        if nc_labels:
            nc_v = plot_df[plot_df['stemta_group'] == 'NegCtrl']['stemta_score'].dropna()
            pval_nc_vs_stable = np.nan
            if len(s_vals) > 1 and len(nc_v) > 1:
                _, pval_nc_vs_stable = stats.ttest_ind(nc_v, s_vals)
            pval_rev_vs_nc = np.nan
            if len(r_vals) > 1 and len(nc_v) > 1:
                _, pval_rev_vs_nc = stats.ttest_ind(r_vals, nc_v)
            all_stats.append({
                'context':                    'StemTA_NegCtrl',
                'cell_type':                  dest_ct,
                'Dox':                        dox_label,
                'group':                      'NegCtrl (Absorptive+Secretory)',
                'score_col':                  mod_col,
                'N':                          len(nc_v),
                'mean':                       nc_v.mean()   if len(nc_v) else np.nan,
                'median':                     nc_v.median() if len(nc_v) else np.nan,
                'std':                        nc_v.std()    if len(nc_v) else np.nan,
                'ttest_vs_Stable_pval':       pval_nc_vs_stable,
                'significance_vs_Stable':     get_sig(pval_nc_vs_stable),
                'ttest_Reversed_vs_NegCtrl':  pval_rev_vs_nc,
                'significance_Rev_vs_NC':     get_sig(pval_rev_vs_nc),
            })

    plt.suptitle(
        f'Stem/TA Module Score: Stable vs Reversed from Differentiated | Dox = {dox_label}\n'
        f'Grey = Stable Absorptive + Secretory pooled (Neg Ctrl)\n'
        f'Dashed line separates negative control',
        fontsize=12, fontweight='bold', y=1.03
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

    return pd.DataFrame(all_stats) if all_stats else pd.DataFrame()


# ─────────────────────────────────────────────
# BARPLOT HELPERS — Pulse vs Chase comparison
# ─────────────────────────────────────────────
def _binom_between_dox(n1, total1, n2, total2):
    total = n1 + n2
    if total == 0:
        return np.nan
    return binomtest(n1, total, p=0.5, alternative='two-sided').pvalue

def make_combined_dox_barplot(df, group_col, groups, colors,
                               dox_present, title_str, save_path,
                               per_celltype=False):
    if len(dox_present) < 2:
        print(f"  Skipping — fewer than 2 Dox conditions")
        return pd.DataFrame()

    d1, d2 = dox_present[0], dox_present[1]

    if not per_celltype:
        fig, ax = plt.subplots(figsize=(max(6, 2.2 * len(groups)), 6))

        group_width = 0.35
        gap         = 0.15
        all_rows    = []

        for g_idx, (grp, color) in enumerate(zip(groups, colors)):
            sub   = df[df[group_col] == grp]
            n1    = int((sub['Dox'] == d1).sum())
            n2    = int((sub['Dox'] == d2).sum())
            tot1  = int((df['Dox'] == d1).sum())
            tot2  = int((df['Dox'] == d2).sum())
            pval  = _binom_between_dox(n1, tot1, n2, tot2)
            stars = get_sig(pval)

            x_centre = g_idx * (2 * group_width + gap + 0.1)
            x1 = x_centre - group_width / 2
            x2 = x_centre + group_width / 2

            b1 = ax.bar(x1, n1, group_width,
                        color=DOX_COLORS.get(d1, color),
                        edgecolor='black', linewidth=0.8,
                        label=d1 if g_idx == 0 else '')
            b2 = ax.bar(x2, n2, group_width,
                        color=DOX_COLORS.get(d2, color),
                        edgecolor='black', linewidth=0.8,
                        hatch='//', label=d2 if g_idx == 0 else '')

            for bar, val in [(b1, n1), (b2, n2)]:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + max(n1, n2, 1) * 0.04,
                        str(val), ha='center', va='bottom',
                        fontsize=9, fontweight='bold')

            y_max  = max(n1, n2, 1)
            y_brac = y_max * 1.18
            ax.plot([x1, x1, x2, x2],
                    [y_brac - y_max * 0.04, y_brac,
                     y_brac, y_brac - y_max * 0.04],
                    color='black', lw=1.2)
            pval_txt = f'p={pval:.2e}' if pval < 0.001 else f'p={pval:.3f}'
            ax.text(x_centre, y_brac + y_max * 0.04,
                    f'{stars}\n{pval_txt}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold',
                    color='black' if stars != 'ns' else '#888888')

            ax.text(x_centre, -max(n1, n2, 1) * 0.08, grp,
                    ha='center', va='top', fontsize=9, fontweight='bold',
                    color=color)

            for dox_val, cnt, total in [(d1, n1, tot1), (d2, n2, tot2)]:
                all_rows.append({
                    'group':          grp,
                    'cell_type':      'ALL',
                    'Dox':            dox_val,
                    'count':          cnt,
                    'total_in_dox':   total,
                    'binom_p_pulse_vs_chase': pval,
                    'significance':   stars,
                })

        ax.set_xticks([])
        ax.set_ylabel('Number of cells', fontsize=11, fontweight='bold')
        ax.set_title(title_str, fontsize=12, fontweight='bold')
        handles = [
            plt.Rectangle((0,0), 1, 1,
                           facecolor=DOX_COLORS.get(d1, '#4A90D9'),
                           edgecolor='black', label=DOX_LABELS.get(d1, d1)),
            plt.Rectangle((0,0), 1, 1,
                           facecolor=DOX_COLORS.get(d2, '#E87722'),
                           edgecolor='black', hatch='//',
                           label=DOX_LABELS.get(d2, d2)),
        ]
        ax.legend(handles=handles, loc='upper right', fontsize=9)
        ax.set_ylim(bottom=0)

        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {save_path}")
        return pd.DataFrame(all_rows)

    else:
        ct_present = [ct for ct in label_order
                      if ct in df['ttest_gmm_label'].unique()]
        if not ct_present:
            return pd.DataFrame()

        n_ct      = len(ct_present)
        fig, axes = plt.subplots(1, n_ct, figsize=(3.5 * n_ct, 6), sharey=False)
        if n_ct == 1:
            axes = [axes]

        all_rows  = []
        group_width = 0.3
        gap         = 0.1

        for ax, ct in zip(axes, ct_present):
            df_ct = df[df['ttest_gmm_label'] == ct]

            for g_idx, (grp, color) in enumerate(zip(groups, colors)):
                sub  = df_ct[df_ct[group_col] == grp]
                tot1 = int((df_ct['Dox'] == d1).sum())
                tot2 = int((df_ct['Dox'] == d2).sum())
                n1   = int((sub['Dox'] == d1).sum())
                n2   = int((sub['Dox'] == d2).sum())
                pval = _binom_between_dox(n1, tot1, n2, tot2)
                stars= get_sig(pval)

                x_centre = g_idx * (2 * group_width + gap + 0.08)
                x1 = x_centre - group_width / 2
                x2 = x_centre + group_width / 2

                b1 = ax.bar(x1, n1, group_width,
                            color=DOX_COLORS.get(d1, color),
                            edgecolor='black', linewidth=0.7)
                b2 = ax.bar(x2, n2, group_width,
                            color=DOX_COLORS.get(d2, color),
                            edgecolor='black', linewidth=0.7, hatch='//')

                y_max  = max(n1, n2, 1)
                for bar, val in [(b1, n1), (b2, n2)]:
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            bar.get_height() + y_max * 0.05,
                            str(val), ha='center', va='bottom', fontsize=7)

                y_brac = y_max * 1.20
                ax.plot([x1, x1, x2, x2],
                        [y_brac - y_max*0.05, y_brac,
                         y_brac, y_brac - y_max*0.05],
                        color='black', lw=1.0)
                pval_txt = f'p={pval:.2e}' if pval < 0.001 else f'p={pval:.3f}'
                ax.text(x_centre, y_brac + y_max * 0.05,
                        f'{stars}\n{pval_txt}',
                        ha='center', va='bottom', fontsize=7,
                        color='black' if stars != 'ns' else '#888888')

                ax.text(x_centre, -y_max * 0.10, grp,
                        ha='center', va='top', fontsize=7,
                        color=color, fontweight='bold')

                for dox_val, cnt, total in [(d1, n1, tot1), (d2, n2, tot2)]:
                    all_rows.append({
                        'group':        grp,
                        'cell_type':    ct,
                        'Dox':          dox_val,
                        'count':        cnt,
                        'total_in_dox': total,
                        'binom_p_pulse_vs_chase': pval,
                        'significance': stars,
                    })

            ax.set_xticks([])
            ax.set_title(ct, fontsize=10, fontweight='bold',
                         color=cell_type_colors.get(ct, 'black'))
            ax.set_ylabel('Cell count', fontsize=8)
            ax.set_ylim(bottom=0)

        plt.suptitle(title_str, fontsize=12, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {save_path}")

        return pd.DataFrame(all_rows)


# ─────────────────────────────────────────────
# STEP 4: GENERATE PLOTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: GENERATING PLOTS")
print("=" * 60)

ORDER_SPLIT    = ['Stable', 'To_Stem', 'To_TA']
PALETTE_SPLIT  = {k: GROUP_COLORS[k] for k in ORDER_SPLIT}

ORDER_COMBINED   = ['Stable', 'To_StemOrTA']
PALETTE_COMBINED = {k: GROUP_COLORS[k] for k in ORDER_COMBINED}

collected_stats = []

def dox_tag(d):
    return str(d).replace(' ', '_').replace('/', '-')

for dox_label, subset in (
    [('ALL', paired)] +
    [(str(d), paired[paired['Dox'] == str(d)]) for d in DOX_VALUES]
):
    tag = dox_tag(dox_label)
    print(f"\n  Dox = {dox_label}  (n={len(subset)})")

    # ── A: Separate Stem / TA destinations (unchanged) ──────────
    _s = make_per_celltype_plot(
        subset[subset['transition'].isin(ORDER_SPLIT)],
        group_col        = 'transition',
        group_order_full = ORDER_SPLIT,
        palette          = PALETTE_SPLIT,
        dox_label        = dox_label,
        save_path        = os.path.join(score_dir,
                           f'per_celltype_split_Dox_{tag}.pdf'),
        context          = 'SplitStemTA_PerCelltype',
    )
    collected_stats.append(_s)

    # Combined (unchanged)
    _s = make_combined_plot(
        subset[subset['transition'].isin(ORDER_SPLIT)],
        group_col        = 'transition',
        group_order_full = ORDER_SPLIT,
        palette          = PALETTE_SPLIT,
        dox_label        = dox_label,
        save_path        = os.path.join(score_dir,
                           f'combined_split_Dox_{tag}.pdf'),
        context          = 'SplitStemTA_Combined',
    )
    collected_stats.append(_s)

    # ── B: Merged Stem+TA destination — NOW WITH NEG CTRL ────────
    _s = make_per_celltype_merged_plot_with_negctrl(
        df           = subset[subset['transition_combined'].isin(ORDER_COMBINED)],
        paired_full  = paired,           # full df for neg-ctrl extraction
        group_col    = 'transition_combined',
        group_order_full = ORDER_COMBINED,
        palette      = PALETTE_COMBINED,
        dox_label    = dox_label,
        save_path    = os.path.join(score_dir,
                       f'per_celltype_merged_negctrl_Dox_{tag}.pdf'),
        context      = 'MergedStemTA_PerCelltype_NegCtrl',
    )
    collected_stats.append(_s)

    # Combined merged (unchanged)
    _s = make_combined_plot(
        subset[subset['transition_combined'].isin(ORDER_COMBINED)],
        group_col        = 'transition_combined',
        group_order_full = ORDER_COMBINED,
        palette          = PALETTE_COMBINED,
        dox_label        = dox_label,
        save_path        = os.path.join(score_dir,
                           f'combined_merged_Dox_{tag}.pdf'),
        context          = 'MergedStemTA_Combined',
    )
    collected_stats.append(_s)

    # ── C: Stem/TA stable vs reversed — NOW WITH NEG CTRL ────────
    stemta_subset = paired_stemta if dox_label == 'ALL' else (
        paired_stemta[paired_stemta['Dox'] == str(dox_label)]
    )
    _s = make_stemta_plot(
        df              = stemta_subset,
        dox_label       = dox_label,
        save_path       = os.path.join(score_dir,
                          f'stemta_stable_vs_reversed_negctrl_Dox_{tag}.pdf'),
        paired_all_raw  = _all_paired_raw,   # <-- passed for neg-ctrl building
    )
    collected_stats.append(_s)


# ─────────────────────────────────────────────
# STEP 6: BARPLOTS — Pulse vs Chase cell counts
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: BARPLOTS — PULSE vs CHASE CELL COUNTS")
print("=" * 60)

barplot_stats = []

paired_dox        = paired[paired['Dox'].isin(DOX_ORDER)].copy()
paired_stemta_dox = paired_stemta[paired_stemta['Dox'].isin(DOX_ORDER)].copy()
dox_present       = [d for d in DOX_ORDER if d in paired_dox['Dox'].unique()]

# ── A: Stable + To_Stem + To_TA  (split view) ─────────────────────
barplot_stats.append(make_combined_dox_barplot(
    df=paired_dox, group_col='transition',
    groups=['Stable', 'To_Stem', 'To_TA'],
    colors=['#2ecc71', '#9370DB', '#D2B48C'],
    dox_present=dox_present,
    title_str='Stable / To_Stem / To_TA — Pulse vs Chase (binomial test within each group)',
    save_path=os.path.join(score_dir, 'barplot_stable_toStem_toTA_overall.pdf'),
    per_celltype=False,
))
barplot_stats.append(make_combined_dox_barplot(
    df=paired_dox, group_col='transition',
    groups=['Stable', 'To_Stem', 'To_TA'],
    colors=['#2ecc71', '#9370DB', '#D2B48C'],
    dox_present=dox_present,
    title_str='Stable / To_Stem / To_TA per cell type — Pulse vs Chase',
    save_path=os.path.join(score_dir, 'barplot_stable_toStem_toTA_per_celltype.pdf'),
    per_celltype=True,
))

# ── B: Stable + To_StemOrTA  (merged view) ────────────────────────
barplot_stats.append(make_combined_dox_barplot(
    df=paired_dox, group_col='transition_combined',
    groups=['Stable', 'To_StemOrTA'],
    colors=['#2ecc71', '#5B4A8A'],
    dox_present=dox_present,
    title_str='Stable / To_StemOrTA — Pulse vs Chase (binomial test within each group)',
    save_path=os.path.join(score_dir, 'barplot_stable_toStemOrTA_overall.pdf'),
    per_celltype=False,
))
barplot_stats.append(make_combined_dox_barplot(
    df=paired_dox, group_col='transition_combined',
    groups=['Stable', 'To_StemOrTA'],
    colors=['#2ecc71', '#5B4A8A'],
    dox_present=dox_present,
    title_str='Stable / To_StemOrTA per cell type — Pulse vs Chase',
    save_path=os.path.join(score_dir, 'barplot_stable_toStemOrTA_per_celltype.pdf'),
    per_celltype=True,
))

# ── C: Stable_Stem/TA vs Reversed_to_Stem/TA  (stemta view) ───────
for dest_ct, stbl_grp, rev_grp, stbl_col, rev_col in [
    ('Stem', 'Stable_Stem', 'Reversed_to_Stem', '#2ecc71', '#9370DB'),
    ('TA',   'Stable_TA',   'Reversed_to_TA',   '#27ae60', '#b8860b'),
]:
    sub = paired_stemta_dox[paired_stemta_dox['stemta_dest'] == dest_ct].copy()
    sub['transition']      = sub['stemta_group']
    sub['ttest_gmm_label'] = sub.get('ttest_gmm_label',
                                      sub.get('dcm_cell_type', 'unknown'))
    barplot_stats.append(make_combined_dox_barplot(
        df=sub, group_col='transition',
        groups=[stbl_grp, rev_grp],
        colors=[stbl_col, rev_col],
        dox_present=dox_present,
        title_str=(f'{stbl_grp} vs {rev_grp} — Pulse vs Chase\n'
                   f'(binomial test within each group)'),
        save_path=os.path.join(score_dir,
                               f'barplot_{dest_ct}_stable_vs_reversed_overall.pdf'),
        per_celltype=False,
    ))
    barplot_stats.append(make_combined_dox_barplot(
        df=sub, group_col='transition',
        groups=[stbl_grp, rev_grp],
        colors=[stbl_col, rev_col],
        dox_present=dox_present,
        title_str=f'{stbl_grp} vs {rev_grp} per cell type — Pulse vs Chase',
        save_path=os.path.join(score_dir,
                               f'barplot_{dest_ct}_stable_vs_reversed_per_celltype.pdf'),
        per_celltype=True,
    ))

bp_stats_df  = pd.concat(
    [s for s in barplot_stats if s is not None and not s.empty],
    ignore_index=True
)
bp_stats_csv = os.path.join(score_dir, 'pulse_vs_chase_count_stats.csv')
bp_stats_df.to_csv(bp_stats_csv, index=False)
print(f"\nBarplot stats CSV: {bp_stats_csv}")
print(bp_stats_df.to_string(index=False))


# ─────────────────────────────────────────────
# STEP 5: CSV OUTPUTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: SAVING CSV OUTPUTS")
print("=" * 60)

stats_df  = pd.concat([s for s in collected_stats if s is not None and not s.empty],
                      ignore_index=True)
stats_csv = os.path.join(score_dir, 'stable_vs_reprogrammed_stats.csv')
stats_df.to_csv(stats_csv, index=False)
print(f"Stats CSV : {stats_csv}")
print(stats_df.to_string(index=False))

out_cols = (['batch_plate_well', 'Dox', 'ttest_gmm_label', 'is_multipotent',
             'rna_celltype', 'transition', 'transition_combined',
             'self_module_score'] + ttest_cols)
data_csv = os.path.join(score_dir, 'stable_vs_reprogrammed_data.csv')
paired[out_cols].to_csv(data_csv, index=False)
print(f"Data CSV  : {data_csv}")

print("\nDone!")
print(f"Outputs in: {score_dir}")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ─────────────────────────────────────────────
# PATHS & CONFIG
# ─────────────────────────────────────────────
root    = "/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py"
out_dir = os.path.join(root, 'gene_scoring', 'gmm_085',
                       'MS_stable_vs_reprogrammed', 'Neg_ctrl',
                       'module_score_boxplots')
os.makedirs(out_dir, exist_ok=True)

si_cell_types  = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']
ANALYSIS_TYPES = ['Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']
TRANSITIONS    = ['Stable', 'To_StemOrTA']

cell_type_colors = {
    'Stem'      : '#9370DB',
    'TA'        : '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth'    : '#DC143C',
    'Goblet'    : '#2E8B57',
    'Tuft'      : '#FF1493',
    'EEC'       : '#4169E1',
}

TRANSITION_COLORS = {
    'Stable'      : '#2ecc71',
    'To_StemOrTA' : '#5B4A8A',
}

# ─────────────────────────────────────────────
# STEP 1: DETECT LITERATURE MODULE COLUMNS
# ─────────────────────────────────────────────
adata = adata_combined

lit_mod_cols = [c for c in adata.obs.columns
                if c.startswith('literature_') and c.endswith('_module')]

# {'Enterocyte': 'literature_Enterocyte_module', ...}
col_from_ct = {
    col.replace('literature_', '').replace('_module', ''): col
    for col in lit_mod_cols
}
print(f"Module columns found: {col_from_ct}")

# ─────────────────────────────────────────────
# STEP 2: BUILD BASE + PAIRED TABLE
# ─────────────────────────────────────────────
base = pd.DataFrame({
    'batch_plate_well' : adata.obs['batch_plate_well'].values,
    'assay'            : adata.obs['assay'].values,
    'Dox'              : adata.obs['Dox'].astype(str).values,
    'ttest_gmm_label'  : adata.obs['ttest_gmm_label_confident_085'].astype(str)
                                .str.split('_').str[-1].values,
    'leiden_annot_25'  : adata.obs['leiden_annot_25'].astype(str)
                                .str.split('_').str[-1].values,
    'is_multipotent'   : adata.obs['is_multipotent'].astype(str).values,
}, index=adata.obs_names)

# Add all literature module scores to base
for col in lit_mod_cols:
    base[col] = adata.obs[col].values

# Pair pDCM ↔ pRNA 1:1
dcm_df = base[base['assay'] == 'pDCM'].copy()
rna_df = (base[base['assay'] == 'pRNA']
          [['batch_plate_well', 'leiden_annot_25']]
          .copy()
          .rename(columns={'leiden_annot_25': 'rna_celltype'}))

def match_one_to_one(df_a, df_b):
    common = set(df_a['batch_plate_well']) & set(df_b['batch_plate_well'])
    a_s = set(df_a[df_a['batch_plate_well'].isin(common)]['batch_plate_well']
              .value_counts().pipe(lambda s: s[s == 1].index))
    b_s = set(df_b[df_b['batch_plate_well'].isin(common)]['batch_plate_well']
              .value_counts().pipe(lambda s: s[s == 1].index))
    return a_s & b_s

matched_wells = match_one_to_one(dcm_df, rna_df)
print(f"1:1 matched wells: {len(matched_wells)}")

paired = (dcm_df[dcm_df['batch_plate_well'].isin(matched_wells)]
          .merge(rna_df[rna_df['batch_plate_well'].isin(matched_wells)],
                 on='batch_plate_well', how='inner'))

# Remove hybrids, keep ANALYSIS_TYPES
paired = paired[paired['ttest_gmm_label'].isin(si_cell_types)].copy()
paired = paired[~paired['is_multipotent'].str.lower().isin(['true','1','yes'])].copy()
paired = paired[paired['ttest_gmm_label'].isin(ANALYSIS_TYPES)].copy()

def classify(row):
    dcm = row['ttest_gmm_label']
    rna = row['rna_celltype']
    if dcm == rna:              return 'Stable'
    if rna in ('Stem', 'TA'):   return 'To_StemOrTA'
    return 'Other'

paired['transition_combined'] = paired.apply(classify, axis=1)
paired = paired[paired['transition_combined'].isin(TRANSITIONS)].copy()

print(f"Stable: {(paired['transition_combined']=='Stable').sum()}, "
      f"To_StemOrTA: {(paired['transition_combined']=='To_StemOrTA').sum()}")

# ─────────────────────────────────────────────
# STEP 3: HELPERS
# ─────────────────────────────────────────────
def get_sig(p):
    if pd.isna(p): return ''
    if p < 0.001:  return '***'
    if p < 0.01:   return '**'
    if p < 0.05:   return '*'
    return 'ns'

def draw_bracket(ax, x0, x1, y, stars):
    ax.plot([x0, x0, x1, x1],
            [y * 0.998, y, y, y * 0.998],
            color='#333333', lw=0.9)
    ax.text((x0 + x1) / 2, y, stars,
            ha='center', va='bottom', fontsize=11, fontweight='bold',
            color='black' if stars != 'ns' else '#888888')

# ─────────────────────────────────────────────
# STEP 4: PLOT — one PDF per cell type
# Only cells with DCM label == that cell type
# ─────────────────────────────────────────────
for ct in ANALYSIS_TYPES:
    mod_col = col_from_ct.get(ct)
    if mod_col is None:
        print(f"  {ct}: no module column found, skipping")
        continue

    # Filter: only cells whose DCM label is this cell type
    sub = paired[paired['ttest_gmm_label'] == ct].copy()
    sub = sub[sub[mod_col].notna()].copy()

    if sub.empty:
        print(f"  {ct}: no cells, skipping")
        continue

    present = [t for t in TRANSITIONS if t in sub['transition_combined'].values]
    if len(present) < 2:
        print(f"  {ct}: only one transition group present, skipping")
        continue

    print(f"\n── {ct} | n={len(sub)} | groups: {sub['transition_combined'].value_counts().to_dict()}")

    fig, ax = plt.subplots(figsize=(5, 6))

    plot_data = sub[['transition_combined', 'ttest_gmm_label', mod_col]].copy()
    plot_data = plot_data.rename(columns={mod_col: 'score'})

    sns.boxplot(
        data       = plot_data,
        x          = 'transition_combined',
        y          = 'score',
        hue        = 'transition_combined',
        order      = present,
        palette    = TRANSITION_COLORS,
        legend     = False,
        ax         = ax,
        width      = 0.5,
        flierprops = dict(marker='o', markersize=2, alpha=0.3),
        linewidth  = 0.9,
    )
    sns.stripplot(
        data    = plot_data,
        x       = 'transition_combined',
        y       = 'score',
        hue     = 'ttest_gmm_label',   # all same ct here, but kept for consistency
        order   = present,
        palette = cell_type_colors,
        dodge   = False,
        alpha   = 0.6,
        size    = 4,
        jitter  = True,
        ax      = ax,
    )
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()

    # t-test
    ylim  = ax.get_ylim()
    y_top = ylim[1]
    s_v   = plot_data[plot_data['transition_combined'] == 'Stable'     ]['score'].dropna()
    t_v   = plot_data[plot_data['transition_combined'] == 'To_StemOrTA']['score'].dropna()
    if len(s_v) > 1 and len(t_v) > 1:
        _, pval = stats.ttest_ind(t_v, s_v)
        stars   = get_sig(pval)
        y_ann   = y_top * 1.10
        draw_bracket(ax, 0, 1, y_ann, stars)
        pval_str = f'p={pval:.2e}' if pval < 0.001 else f'p={pval:.3f}'
        ax.text(0.5, y_ann * 1.01, pval_str,
                ha='center', va='bottom', fontsize=8, color='#444444')
        ax.set_ylim(ylim[0], y_ann * 1.20)

    # n labels
    cur_top = ax.get_ylim()[1]
    for i, grp in enumerate(present):
        n = int((plot_data['transition_combined'] == grp).sum())
        ax.text(i, cur_top * 0.98, f'n={n}',
                ha='center', va='top', fontsize=9, fontweight='bold')

    # Background tint
    hex_c = cell_type_colors.get(ct, '#EEEEEE').lstrip('#')
    rgb   = tuple(int(hex_c[i:i+2], 16) / 255 for i in (0, 2, 4))
    ax.set_facecolor((*rgb, 0.07))

    ax.set_xlabel('')
    ax.set_ylabel(f'literature_{ct}_module score', fontsize=10)
    ax.set_xticklabels(
        [t.replace('To_StemOrTA', 'To Stem/TA') for t in present],
        fontsize=10
    )
    ax.set_title(
        f'{ct} cells\nliterature_{ct}_module score\nStable vs To_StemOrTA',
        fontsize=12, fontweight='bold',
        color=cell_type_colors.get(ct, 'black')
    )

    plt.tight_layout()
    save_path = os.path.join(out_dir, f'module_score_{ct}_stable_vs_ToStemOrTA.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

print(f"\nAll module score PDFs saved to: {out_dir}")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ─────────────────────────────────────────────
# PATHS & CONFIG
# ─────────────────────────────────────────────
root      = "/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py"
out_dir   = os.path.join(root, 'gene_scoring', 'gmm_085',
                         'MS_stable_vs_reprogrammed', 'Neg_ctrl',
                         'expression_boxplots')
os.makedirs(out_dir, exist_ok=True)

si_cell_types = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

cell_type_colors = {
    'Stem'      : '#9370DB',
    'TA'        : '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth'    : '#DC143C',
    'Goblet'    : '#2E8B57',
    'Tuft'      : '#FF1493',
    'EEC'       : '#4169E1',
}

TRANSITION_COLORS = {
    'Stable'      : '#2ecc71',
    'To_StemOrTA' : '#5B4A8A',
}

TRANSITIONS    = ['Stable', 'To_StemOrTA']
ANALYSIS_TYPES = ['Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# ─────────────────────────────────────────────
# LOAD MARKER GENES
# ─────────────────────────────────────────────
lit_markers = pd.read_csv(
    "/media/erlantz/71DE167A75E99BB4/Pool32/Combi/Python/integration/"
    "gene_scoring/marker_gene_sets/literature_top_markers_13012026.csv"
)

group_col = 'cell_type' if 'cell_type' in lit_markers.columns else 'cluster'
gene_col  = 'gene'

genes_per_ct = {}
for ct in si_cell_types:
    sub = lit_markers[lit_markers[group_col] == ct]
    genes_per_ct[ct] = sub[gene_col].tolist()

all_marker_genes = []
for ct in si_cell_types:
    for g in genes_per_ct[ct]:
        if g not in all_marker_genes:
            all_marker_genes.append(g)

print(f"Total marker genes: {len(all_marker_genes)}")

# ─────────────────────────────────────────────
# STEP 1: REBUILD PAIRED TABLE (pDCM ↔ pRNA)
# ─────────────────────────────────────────────
adata = adata_combined

base = pd.DataFrame({
    'batch_plate_well'     : adata.obs['batch_plate_well'].values,
    'assay'                : adata.obs['assay'].values,
    'Dox'                  : adata.obs['Dox'].astype(str).values,
    'ttest_gmm_label'      : adata.obs['ttest_gmm_label_confident_085'].astype(str)
                                    .str.split('_').str[-1].values,
    'ttest_gmm_label_full' : adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'leiden_annot_25'      : adata.obs['leiden_annot_25'].astype(str)
                                    .str.split('_').str[-1].values,
    'cell_type_broad'      : adata.obs['cell_type_broad'].astype(str).values,
    'is_multipotent'       : adata.obs['is_multipotent'].astype(str).values,
}, index=adata.obs_names)

# Pair pDCM ↔ pRNA 1:1
dcm_df = base[base['assay'] == 'pDCM'].copy()
rna_df = (base[base['assay'] == 'pRNA']
          [['batch_plate_well', 'leiden_annot_25', 'cell_type_broad']]
          .copy()
          .rename(columns={'leiden_annot_25' : 'rna_celltype',
                           'cell_type_broad'  : 'rna_cell_type_broad'}))

def match_one_to_one(df_a, df_b):
    common = set(df_a['batch_plate_well']) & set(df_b['batch_plate_well'])
    a_s = set(df_a[df_a['batch_plate_well'].isin(common)]['batch_plate_well']
              .value_counts().pipe(lambda s: s[s == 1].index))
    b_s = set(df_b[df_b['batch_plate_well'].isin(common)]['batch_plate_well']
              .value_counts().pipe(lambda s: s[s == 1].index))
    return a_s & b_s

matched_wells = match_one_to_one(dcm_df, rna_df)
print(f"1:1 matched wells: {len(matched_wells)}")

paired = (dcm_df[dcm_df['batch_plate_well'].isin(matched_wells)]
          .merge(rna_df[rna_df['batch_plate_well'].isin(matched_wells)],
                 on='batch_plate_well', how='inner'))

# Remove hybrids, keep ANALYSIS_TYPES
paired = paired[paired['ttest_gmm_label'].isin(si_cell_types)].copy()
paired = paired[~paired['is_multipotent'].str.lower().isin(['true', '1', 'yes'])].copy()
paired = paired[paired['ttest_gmm_label'].isin(ANALYSIS_TYPES)].copy()

def classify(row):
    dcm = row['ttest_gmm_label']
    rna = row['rna_celltype']
    if dcm == rna:
        return 'Stable'
    if rna in ('Stem', 'TA'):
        return 'To_StemOrTA'
    return 'Other'

paired['transition_combined'] = paired.apply(classify, axis=1)
paired = paired[paired['transition_combined'].isin(TRANSITIONS)].copy()

print(f"Stable: {(paired['transition_combined']=='Stable').sum()}, "
      f"To_StemOrTA: {(paired['transition_combined']=='To_StemOrTA').sum()}")

# ─────────────────────────────────────────────
# STEP 2: EXTRACT LOGNORM EXPRESSION
# Use batch_plate_well to map wells → obs_names
# for both pDCM and pRNA
# ─────────────────────────────────────────────

def extract_lognorm(adata_full, assay_name, obs_names, genes):
    mask      = adata_full.obs['assay'] == assay_name
    adata_sub = adata_full[mask].copy()

    genes_avail = [g for g in genes if g in adata_sub.var_names]
    missing     = [g for g in genes if g not in adata_sub.var_names]
    if missing:
        print(f"  [{assay_name}] {len(missing)} genes not in var_names, skipping.")

    adata_sub = adata_sub[:, genes_avail]

    if 'lognorm' in adata_sub.layers:
        X = adata_sub.layers['lognorm']
    else:
        print(f"  [{assay_name}] No 'lognorm' layer — using .X")
        X = adata_sub.X

    expr_df = pd.DataFrame(
        np.asarray(X.todense() if hasattr(X, 'todense') else X),
        index   = adata_sub.obs_names,
        columns = genes_avail,
    )
    obs_present = [o for o in obs_names if o in expr_df.index]
    return expr_df.loc[obs_present]

# pDCM: map well → pDCM obs_name
dcm_obs_map = (base[base['assay'] == 'pDCM']
               .reset_index()
               .rename(columns={'index': 'obs_name'})
               [['batch_plate_well', 'obs_name']]
               .drop_duplicates('batch_plate_well')
               .set_index('batch_plate_well')['obs_name']
               .to_dict())

paired['dcm_obs_name'] = paired['batch_plate_well'].map(dcm_obs_map)
dcm_obs_names = paired['dcm_obs_name'].dropna().tolist()

print("Extracting pDCM lognorm expression...")
expr_dcm = extract_lognorm(adata, 'pDCM', dcm_obs_names, all_marker_genes)
print(f"  expr_dcm shape: {expr_dcm.shape}")

# pRNA: map well → pRNA obs_name
rna_obs_map = (base[base['assay'] == 'pRNA']
               .reset_index()
               .rename(columns={'index': 'obs_name'})
               [['batch_plate_well', 'obs_name']]
               .drop_duplicates('batch_plate_well')
               .set_index('batch_plate_well')['obs_name']
               .to_dict())

paired['rna_obs_name'] = paired['batch_plate_well'].map(rna_obs_map)
rna_obs_names = paired['rna_obs_name'].dropna().tolist()

print("Extracting pRNA lognorm expression...")
expr_rna = extract_lognorm(adata, 'pRNA', rna_obs_names, all_marker_genes)
print(f"  expr_rna shape: {expr_rna.shape}")

# ─────────────────────────────────────────────
# STEP 3: PLOTTING FUNCTION
# ─────────────────────────────────────────────

def get_sig(p):
    if pd.isna(p): return ''
    if p < 0.001:  return '***'
    if p < 0.01:   return '**'
    if p < 0.05:   return '*'
    return 'ns'

def make_expression_boxplot_figure(
        paired_df, expr_df, obs_col,
        genes, genes_per_ct, si_cell_types,
        cell_type_colors, transition_colors,
        title_prefix, save_path):

    gene_to_ct = {}
    for ct in si_cell_types:
        for g in genes_per_ct.get(ct, []):
            if g not in gene_to_ct:
                gene_to_ct[g] = ct

    # Resolve obs_names and align meta ↔ expr
    paired_df = paired_df.copy()
    if obs_col is None:
        paired_df['_obs'] = paired_df.index
    else:
        paired_df['_obs'] = paired_df[obs_col]

    paired_df = paired_df[paired_df['_obs'].isin(expr_df.index)].copy()
    expr_sub  = expr_df.loc[paired_df['_obs'].values].copy()
    expr_sub.index = paired_df.index   # align indices

    genes_plot = [g for g in genes if g in expr_df.columns]
    if not genes_plot:
        print(f"  No genes to plot for {title_prefix[:40]}...")
        return

    print(f"  Plotting {len(genes_plot)} genes | cells: {len(paired_df)}")

    n_genes   = len(genes_plot)
    fig_w     = max(16, 2.2 * n_genes)
    fig, axes = plt.subplots(1, n_genes, figsize=(fig_w, 6), sharey=False)
    if n_genes == 1:
        axes = [axes]

    for ax, gene in zip(axes, genes_plot):
        plot_data = paired_df[['transition_combined', 'ttest_gmm_label']].copy()
        plot_data['expression'] = expr_sub[gene].values

        present = [t for t in TRANSITIONS if t in plot_data['transition_combined'].values]

        sns.boxplot(
            data       = plot_data,
            x          = 'transition_combined',
            y          = 'expression',
            hue        = 'transition_combined',
            order      = present,
            palette    = transition_colors,
            legend     = False,
            ax         = ax,
            width      = 0.55,
            flierprops = dict(marker='o', markersize=1.5, alpha=0.2),
            linewidth  = 0.8,
        )

        sns.stripplot(
            data    = plot_data,
            x       = 'transition_combined',
            y       = 'expression',
            hue     = 'ttest_gmm_label',
            order   = present,
            palette = cell_type_colors,
            dodge   = False,
            alpha   = 0.55,
            size    = 3,
            jitter  = True,
            ax      = ax,
        )
        legend = ax.get_legend()
        if legend is not None:
            legend.remove()

        # t-test
        ylim  = ax.get_ylim()
        y_top = ylim[1]
        if 'Stable' in present and 'To_StemOrTA' in present:
            s_vals = plot_data[plot_data['transition_combined'] == 'Stable'     ]['expression'].dropna()
            t_vals = plot_data[plot_data['transition_combined'] == 'To_StemOrTA']['expression'].dropna()
            if len(s_vals) > 1 and len(t_vals) > 1:
                _, pval = stats.ttest_ind(t_vals, s_vals)
                stars   = get_sig(pval)
                y_ann   = y_top * 1.08
                ax.plot([0, 0, 1, 1],
                        [y_ann - (y_ann - ylim[0]) * 0.02,
                         y_ann, y_ann,
                         y_ann - (y_ann - ylim[0]) * 0.02],
                        color='#333333', lw=0.9)
                ax.text(0.5, y_ann, stars,
                        ha='center', va='bottom', fontsize=9, fontweight='bold',
                        color='black' if stars != 'ns' else '#888888')
                ax.set_ylim(ylim[0], y_ann * 1.18)

        # n labels
        cur_ylim = ax.get_ylim()
        for i, grp in enumerate(present):
            n = int((plot_data['transition_combined'] == grp).sum())
            ax.text(i, cur_ylim[1] * 0.98, f'n={n}',
                    ha='center', va='top', fontsize=7)

        # Gene title colored by marker cell type
        ct_color = cell_type_colors.get(gene_to_ct.get(gene, ''), 'black')
        ax.set_title(gene, fontsize=8, fontweight='bold', color=ct_color, pad=3)
        ax.set_xlabel('')
        ax.set_ylabel('lognorm expression' if gene == genes_plot[0] else '', fontsize=8)
        ax.tick_params(axis='x', rotation=30, labelsize=7)
        ax.tick_params(axis='y', labelsize=7)

        # Faint background tint by marker cell type
        ct = gene_to_ct.get(gene, '')
        hex_c = cell_type_colors.get(ct, '#EEEEEE').lstrip('#')
        rgb   = tuple(int(hex_c[i:i+2], 16) / 255 for i in (0, 2, 4))
        ax.set_facecolor((*rgb, 0.07))

    # Shared legend
    handles = [
        plt.Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=cell_type_colors.get(ct, 'grey'),
                   markersize=7, label=ct)
        for ct in si_cell_types
    ]
    fig.legend(handles=handles, title='DCM cell type',
               loc='lower center', ncol=len(si_cell_types),
               fontsize=8, frameon=True,
               bbox_to_anchor=(0.5, -0.06))

    fig.suptitle(title_prefix, fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


# ─────────────────────────────────────────────
# STEP 4: GENERATE FIGURES (one PDF per marker ct)
# ─────────────────────────────────────────────

print("\n" + "="*60)
print("GENERATING FIGURES")
print("="*60)

for ct in si_cell_types:
    ct_genes = [g for g in genes_per_ct.get(ct, [])
                if g in expr_dcm.columns or g in expr_rna.columns]
    if not ct_genes:
        print(f"\n  {ct}: no genes available, skipping")
        continue

    print(f"\n── {ct} markers ({len(ct_genes)} genes) ──")

    # Figure A: pDCM
    make_expression_boxplot_figure(
        paired_df        = paired,
        expr_df          = expr_dcm,
        obs_col          = 'dcm_obs_name',
        genes            = ct_genes,
        genes_per_ct     = {ct: ct_genes},
        si_cell_types    = si_cell_types,
        cell_type_colors = cell_type_colors,
        transition_colors= TRANSITION_COLORS,
        title_prefix     = (f'pDCM lognorm expression — {ct} marker genes\n'
                            f'Stable vs To_StemOrTA | dots = DCM cell type'),
        save_path        = os.path.join(out_dir, f'expr_pDCM_{ct}_markers.pdf'),
    )

    # Figure B: pRNA
    make_expression_boxplot_figure(
        paired_df        = paired,
        expr_df          = expr_rna,
        obs_col          = 'rna_obs_name',
        genes            = ct_genes,
        genes_per_ct     = {ct: ct_genes},
        si_cell_types    = si_cell_types,
        cell_type_colors = cell_type_colors,
        transition_colors= TRANSITION_COLORS,
        title_prefix     = (f'pRNA lognorm expression — {ct} marker genes\n'
                            f'Stable vs To_StemOrTA | dots = DCM cell type'),
        save_path        = os.path.join(out_dir, f'expr_pRNA_{ct}_markers.pdf'),
    )

print(f"\nAll figures saved to: {out_dir}")

In [ ]:
print("paired index sample:", paired.index[:5].tolist())
print("expr_dcm index sample:", expr_dcm.index[:5].tolist())
print("Overlap:", len(set(paired.index) & set(expr_dcm.index)))

In [ ]:
#Heatmap hybrid secretory to enterocyte and secretory to hybrid enterocytes

In [ ]:
##############################################
# TRANSITION ANALYSIS — adata_combined
# Full port of the reference code adapted to
# the new adata_combined object.
#
# All needed columns are already in obs:
#   assay, batch_plate_well, leiden_annot_25,
#   ttest_gmm_label_confident, is_multipotent,
#   significant_celltypes, ttest_*_zscore,
#   S_score, G2M_score, phase
#
# NOTE: cell cycle scores are pre-computed —
# sc.tl.score_genes_cell_cycle is NOT re-run.
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns
from scipy import stats
from scipy.stats import gaussian_kde

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
root       = "/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py"
score_dir  = os.path.join(root, 'gene_scoring')
output_dir_umap = os.path.join(score_dir, 'gmm_085/cell_cycle3/umap_transitions')
output_dir_heat = os.path.join(score_dir, 'gmm_085/cell_cycle3/heatmap_transitions')
output_dir_cc   = os.path.join(score_dir, 'gmm_085/cell_cycle3/cellcycle_transitions')
for d in [output_dir_umap, output_dir_heat, output_dir_cc]:
    os.makedirs(d, exist_ok=True)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
cell_type_colors = {
    'Stem'      : '#9370DB',
    'TA'        : '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth'    : '#DC143C',
    'Goblet'    : '#2E8B57',
    'Tuft'      : '#FF1493',
    'EEC'       : '#4169E1',
    'ambiguous' : '#7f7f7f',
    'hybrid'    : '#FFD700',
}

label_order      = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']
secretory_types  = ['Paneth', 'Goblet', 'Tuft', 'EEC']

BW_METHOD  = 0.5
LEVEL_FRAC = 0.08

# ─────────────────────────────────────────────
# STEP 1: BUILD plot_df
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: BUILDING PLOT DATAFRAME")
print("=" * 60)

adata = adata_combined
umap  = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1':                   umap[:, 0],
    'UMAP2':                   umap[:, 1],
    'assay':                   adata.obs['assay'].values,
    'batch_plate_well':        adata.obs['batch_plate_well'].values,
    'leiden_annot_25':         adata.obs['leiden_annot_25'].astype(str).values,
    'ttest_gmm_label_confident': adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'is_multipotent':          adata.obs['is_multipotent'].astype(str).values,
    'significant_celltypes':   adata.obs['significant_celltypes'].astype(str).values,
    'cell_index':              adata.obs.index.values,
}, index=adata.obs_names)

# Strip numeric prefix from leiden/gmm labels (e.g. '3_Enterocyte' → 'Enterocyte')
plot_df['leiden_annot_25']           = plot_df['leiden_annot_25'].str.split('_').str[-1]
plot_df['ttest_gmm_label_confident'] = plot_df['ttest_gmm_label_confident'].str.split('_').str[-1]

# Attach ttest zscore columns
ttest_cols = [c for c in adata.obs.columns
              if c.startswith('ttest_') and c.endswith('_zscore')
              and 'max' not in c.lower()]
for col in ttest_cols:
    plot_df[col] = adata.obs[col].values

print(f"ttest zscore columns: {ttest_cols}")
print(f"Assay values: {sorted(plot_df['assay'].unique())}")

# ─────────────────────────────────────────────
# STEP 2: CELL CYCLE — already pre-scored
# Just pull S_score, G2M_score, phase from obs
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: CELL CYCLE (pre-scored, reading from obs)")
print("=" * 60)

dcm_obs = adata.obs[adata.obs['assay'] == 'pDCM']
cc_df   = pd.DataFrame({
    'batch_plate_well': dcm_obs['batch_plate_well'].values,
    'S_score':          dcm_obs['S_score'].values,
    'G2M_score':        dcm_obs['G2M_score'].values,
    'phase':            dcm_obs['phase'].values,
}, index=dcm_obs.index)
cc_df['combined_score'] = cc_df['S_score'] + cc_df['G2M_score']
print(f"DCM cells for cell cycle: {len(cc_df)}")

# Add combined_score as a new column in adata_combined for all cells
adata.obs['combined_score'] = adata.obs['S_score'].astype(float) + adata.obs['G2M_score'].astype(float)
print(f"Added 'combined_score' to adata.obs (all {adata.n_obs} cells)")

# ─────────────────────────────────────────────
# STEP 3: SEPARATE CLEAR vs HYBRID DCM
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: SEPARATING CLEAR vs HYBRID DCM CELLS")
print("=" * 60)

dcm_clear_mask = (
    (plot_df['assay'] == 'pDCM') &
    (plot_df['ttest_gmm_label_confident'].isin(label_order)) &
    (plot_df['is_multipotent'].str.lower().isin(['false', '0', 'no']))
)
dcm_clear  = plot_df[dcm_clear_mask].copy()

dcm_hybrid_mask = (
    (plot_df['assay'] == 'pDCM') &
    (plot_df['is_multipotent'].str.lower().isin(['true', '1', 'yes']))
)
dcm_hybrid = plot_df[dcm_hybrid_mask].copy()

rna_mask   = (
    (plot_df['assay'] == 'pRNA') &
    (plot_df['leiden_annot_25'].isin(label_order))
)
rna_cells  = plot_df[rna_mask].copy()

print(f"Clear DCM  : {len(dcm_clear)}")
print(f"Hybrid DCM : {len(dcm_hybrid)}")
print(f"RNA cells  : {len(rna_cells)}")

# ─────────────────────────────────────────────
# STEP 4: 1:1 MATCHING
# ─────────────────────────────────────────────
def match_one_to_one(dcm_df, rna_df):
    common = set(dcm_df['batch_plate_well']) & set(rna_df['batch_plate_well'])
    dcm_s  = set(dcm_df[dcm_df['batch_plate_well'].isin(common)]
                 ['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
    rna_s  = set(rna_df[rna_df['batch_plate_well'].isin(common)]
                 ['batch_plate_well'].value_counts().pipe(lambda s: s[s == 1].index))
    return dcm_s & rna_s

def build_connections(dcm_df, rna_df, matched_wells, is_hybrid=False):
    rows = []
    for bpw in matched_wells:
        dr = dcm_df[dcm_df['batch_plate_well'] == bpw].iloc[0]
        rr = rna_df[rna_df['batch_plate_well'] == bpw].iloc[0]
        row = {
            'batch_plate_well': bpw,
            'dcm_umap1': dr['UMAP1'], 'dcm_umap2': dr['UMAP2'],
            'rna_umap1': rr['UMAP1'], 'rna_umap2': rr['UMAP2'],
            'dcm_cell_type':      dr['ttest_gmm_label_confident'],
            'rna_cell_type':      rr['leiden_annot_25'],
            'dcm_sig_celltypes':  dr['significant_celltypes'],
            'is_hybrid':          is_hybrid,
            'transition': f"{dr['ttest_gmm_label_confident']} -> {rr['leiden_annot_25']}",
        }
        for col in ttest_cols:
            ct = col.replace('ttest_', '').replace('_zscore', '')
            row[f'dcm_{ct}_zscore'] = dr[col]
            row[f'rna_{ct}_zscore'] = rr[col]
        rows.append(row)
    return pd.DataFrame(rows)

both_clear  = match_one_to_one(dcm_clear,  rna_cells)
both_hybrid = match_one_to_one(dcm_hybrid, rna_cells)
print(f"\nClear 1:1 matched wells : {len(both_clear)}")
print(f"Hybrid 1:1 matched wells: {len(both_hybrid)}")

conn_clear_df  = build_connections(dcm_clear,  rna_cells, both_clear,  is_hybrid=False)
conn_hybrid_df = build_connections(dcm_hybrid, rna_cells, both_hybrid, is_hybrid=True)

# ─────────────────────────────────────────────
# STEP 5: TRANSITION GROUPS
# ─────────────────────────────────────────────
sec_to_ent_clear = conn_clear_df[
    conn_clear_df['dcm_cell_type'].isin(secretory_types) &
    (conn_clear_df['rna_cell_type'] == 'Enterocyte')
].copy()
sec_to_ent_clear['group']      = sec_to_ent_clear['dcm_cell_type'] + '→Enterocyte'
sec_to_ent_clear['cell_class'] = 'clear'

ta_ent_clear = conn_clear_df[
    (conn_clear_df['dcm_cell_type'] == 'TA') &
    (conn_clear_df['rna_cell_type'] == 'Enterocyte')
].copy()
ta_ent_clear['group']      = 'TA→Enterocyte'
ta_ent_clear['cell_class'] = 'clear_control'

def has_secretory(sig_str):
    if pd.isna(sig_str) or str(sig_str) in ['none', 'None', '']:
        return False
    return any(s in str(sig_str) for s in secretory_types)

hybrid_sec_to_ent = conn_hybrid_df[
    conn_hybrid_df['dcm_sig_celltypes'].apply(has_secretory) &
    (conn_hybrid_df['rna_cell_type'] == 'Enterocyte')
].copy()
hybrid_sec_to_ent['group']      = 'Hybrid(Sec)→Enterocyte'
hybrid_sec_to_ent['cell_class'] = 'hybrid_dcm'

secretory_zscore_threshold = 1.5
ct_names = [c.replace('ttest_', '').replace('_zscore', '') for c in ttest_cols]

def rna_is_enterocyte_hybrid(row):
    if row['rna_cell_type'] != 'Enterocyte':
        return False
    return any(row.get(f'rna_{s}_zscore', 0) >= secretory_zscore_threshold
               for s in secretory_types)

sec_to_ent_hybrid_rna = conn_clear_df[
    conn_clear_df['dcm_cell_type'].isin(secretory_types)
].copy()
sec_to_ent_hybrid_rna = sec_to_ent_hybrid_rna[
    sec_to_ent_hybrid_rna.apply(rna_is_enterocyte_hybrid, axis=1)
].copy()
sec_to_ent_hybrid_rna['group']      = sec_to_ent_hybrid_rna['dcm_cell_type'] + '→EntHybrid'
sec_to_ent_hybrid_rna['cell_class'] = 'hybrid_rna'

print(f"\nTransition group sizes:")
print(f"  Clear secretory→Enterocyte     : {len(sec_to_ent_clear)}")
print(f"  TA→Enterocyte (control)        : {len(ta_ent_clear)}")
print(f"  Hybrid DCM sec→Enterocyte      : {len(hybrid_sec_to_ent)}")
print(f"  Clear sec→Enterocyte hybrid RNA: {len(sec_to_ent_hybrid_rna)}")

plot1_cc_clear = pd.concat([sec_to_ent_clear, ta_ent_clear], ignore_index=True)
plot1_cc_clear = plot1_cc_clear.merge(cc_df, on='batch_plate_well', how='left')

# ─────────────────────────────────────────────
# STEP 6: KDE STORE FOR BACKGROUND
# Use all cells for contours (pDCM + pRNA)
# ─────────────────────────────────────────────
ref_mask = plot_df['assay'].isin(['pDCM', 'pRNA'])

def build_kde(ct, bw=BW_METHOD):
    sub = plot_df[ref_mask & (plot_df['leiden_annot_25'] == ct)]
    if len(sub) < 10:
        return None
    return gaussian_kde(np.vstack([sub['UMAP1'].values, sub['UMAP2'].values]),
                        bw_method=bw)

kde_store = {ct: build_kde(ct) for ct in label_order}

# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────
def draw_background(ax):
    ref_df = plot_df[ref_mask]
    for ct in label_order:
        sub = ref_df[ref_df['leiden_annot_25'] == ct]
        if len(sub):
            ax.scatter(sub['UMAP1'], sub['UMAP2'], s=8,
                       c=cell_type_colors[ct], alpha=0.15,
                       marker='o', zorder=1, rasterized=True)
    other = ref_df[~ref_df['leiden_annot_25'].isin(label_order)]
    if len(other):
        ax.scatter(other['UMAP1'], other['UMAP2'], s=8,
                   c='lightgray', alpha=0.03, marker='o', zorder=1, rasterized=True)

    for ct in label_order:
        kde = kde_store.get(ct)
        if kde is None:
            continue
        sub = ref_df[ref_df['leiden_annot_25'] == ct]
        x, y = sub['UMAP1'].values, sub['UMAP2'].values
        xg = np.linspace(x.min()-0.5, x.max()+0.5, 200)
        yg = np.linspace(y.min()-0.5, y.max()+0.5, 200)
        xx, yy = np.meshgrid(xg, yg)
        zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        ax.contour(xx, yy, zz, levels=[zz.max() * LEVEL_FRAC],
                   colors=[cell_type_colors[ct]],
                   linewidths=2, linestyles='--', alpha=0.8, zorder=2)
        peak = np.unravel_index(zz.argmax(), zz.shape)
        ax.text(xx[peak], yy[peak], ct, ha='center', va='center',
                fontsize=10, fontweight='bold', color=cell_type_colors[ct], zorder=5)


def plot_umap(subset_df, title_str, save_path, hybrid_dcm=False):
    if len(subset_df) == 0:
        print(f"  Skipping UMAP — no cells"); return

    fig, ax = plt.subplots(figsize=(12, 10))

    # ── Layer 1: ALL cells coloured by cluster (background) ──
    for ct in label_order:
        all_ct = plot_df[plot_df['leiden_annot_25'] == ct]
        if len(all_ct):
            ax.scatter(all_ct['UMAP1'], all_ct['UMAP2'],
                       s=6, c=cell_type_colors.get(ct, '#7f7f7f'),
                       alpha=0.15, marker='o', zorder=1, rasterized=True)
    other_all = plot_df[~plot_df['leiden_annot_25'].isin(label_order)]
    if len(other_all):
        ax.scatter(other_all['UMAP1'], other_all['UMAP2'],
                   s=6, c='lightgray', alpha=0.08,
                   marker='o', zorder=1, rasterized=True)

    # ── Layer 2: KDE contours + labels ──
    for ct in label_order:
        kde = kde_store.get(ct)
        if kde is None:
            continue
        sub = plot_df[plot_df['leiden_annot_25'] == ct]
        x, y = sub['UMAP1'].values, sub['UMAP2'].values
        xg = np.linspace(x.min()-0.5, x.max()+0.5, 200)
        yg = np.linspace(y.min()-0.5, y.max()+0.5, 200)
        xx, yy = np.meshgrid(xg, yg)
        zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        ax.contour(xx, yy, zz, levels=[zz.max() * LEVEL_FRAC],
                   colors=[cell_type_colors[ct]],
                   linewidths=2, linestyles='--', alpha=0.8, zorder=2)
        peak = np.unravel_index(zz.argmax(), zz.shape)
        ax.text(xx[peak], yy[peak], ct, ha='center', va='center',
                fontsize=10, fontweight='bold', color=cell_type_colors[ct], zorder=5)

    # ── Layer 3: Transition arrows (colour = RNA destination) ──
    all_bpw = set(subset_df['batch_plate_well'])
    for _, conn in subset_df.iterrows():
        color = cell_type_colors.get(conn['rna_cell_type'], 'black')
        ax.add_patch(FancyArrowPatch(
            (conn['dcm_umap1'], conn['dcm_umap2']),
            (conn['rna_umap1'], conn['rna_umap2']),
            arrowstyle='-|>', mutation_scale=15,
            color=color, alpha=0.7, linewidth=1.5, zorder=3,
        ))

    # ── Layer 4: Highlighted DCM cells (squares) ──
    for ct in label_order:
        if hybrid_dcm:
            pts = plot_df[(plot_df['assay'] == 'pDCM') &
                          (plot_df['batch_plate_well'].isin(all_bpw)) &
                          (plot_df['is_multipotent'].str.lower().isin(['true','1','yes']))]
            c, m = cell_type_colors['hybrid'], 'D'
        else:
            pts = plot_df[(plot_df['assay'] == 'pDCM') &
                          (plot_df['batch_plate_well'].isin(all_bpw)) &
                          (plot_df['ttest_gmm_label_confident'] == ct)]
            c, m = cell_type_colors.get(ct, '#7f7f7f'), 's'
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100, c=c, alpha=1.0,
                       edgecolors='black', linewidths=1.5,
                       marker=m, zorder=4, rasterized=True)
        if hybrid_dcm:
            break

    # ── Layer 5: Highlighted RNA cells (circles) ──
    dest_cts = subset_df['rna_cell_type'].unique()
    for ct in dest_cts:
        pts = plot_df[(plot_df['assay'] == 'pRNA') &
                      (plot_df['batch_plate_well'].isin(all_bpw)) &
                      (plot_df['leiden_annot_25'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, '#7f7f7f'), alpha=1.0,
                       edgecolors='black', linewidths=1.5,
                       marker='o', zorder=4, rasterized=True)

    # ── Legend ──
    legend = []
    for ct in label_order:
        legend.append(mlines.Line2D([], [], color=cell_type_colors.get(ct, '#7f7f7f'),
                                     marker='o', linestyle='None', markersize=7,
                                     alpha=0.4, label=f'All cells ({ct})'))
    if hybrid_dcm:
        legend.append(mlines.Line2D([], [], color=cell_type_colors['hybrid'],
                                     marker='D', linestyle='None', markersize=10,
                                     markeredgecolor='black', label='DCM Hybrid'))
    else:
        for ct in subset_df['dcm_cell_type'].unique():
            legend.append(mlines.Line2D([], [], color=cell_type_colors.get(ct,'gray'),
                                         marker='s', linestyle='None', markersize=10,
                                         markeredgecolor='black', label=f'DCM {ct}'))
    for ct in dest_cts:
        c = cell_type_colors.get(ct, '#7f7f7f')
        legend += [
            mlines.Line2D([], [], color=c, marker='o', linestyle='None',
                           markersize=10, markeredgecolor='black', label=f'RNA {ct}'),
            mlines.Line2D([], [], color=c, linestyle='-', linewidth=2,
                           alpha=0.8, marker='>', markersize=8, label=f'→ {ct}'),
        ]
    ax.legend(handles=legend, loc='center left',
              bbox_to_anchor=(1, 0.5), frameon=True, fontsize=8)
    ax.set_title(title_str, fontsize=13, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


def plot_umap_cellcycle(title_str, save_path):
    scores_aligned = (adata.obs.reindex(plot_df.index)['combined_score']
                      .fillna(0).values)
    fig, ax = plt.subplots(figsize=(12, 10))
    sc = ax.scatter(
        plot_df['UMAP1'], plot_df['UMAP2'],
        s=8, c=scores_aligned, cmap='RdYlBu_r',
        vmin=np.percentile(scores_aligned, 2),
        vmax=np.percentile(scores_aligned, 98),
        alpha=0.7, rasterized=True, zorder=2,
    )
    cbar = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02, shrink=0.6)
    cbar.set_label('Combined cell cycle score (S + G2M)', fontsize=9)
    ax.set_title(title_str, fontsize=13, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


def plot_heatmap(subset_df, title_str, save_path, group_col='dcm_cell_type'):
    """
    Heatmap with a coloured annotation bar above EACH panel.

    Layout (4 rows):
      row 0 — DCM annotation bar  (colour = group_col / dcm_cell_type)
      row 1 — DCM z-score heatmap
      row 2 — RNA annotation bar  (colour = rna_cell_type)
      row 3 — RNA z-score heatmap

    Each annotation bar has its own legend to the right.
    """
    if len(subset_df) == 0:
        print(f"  Skipping heatmap — no cells"); return

    norm_dcm  = TwoSlopeNorm(vmin=-2.5, vcenter=1.0, vmax=3.5)
    norm_rna  = TwoSlopeNorm(vmin=-2.5, vcenter=0.0, vmax=3.5)
    n_cells   = len(subset_df)
    fig_width = max(14, n_cells * 0.18)

    # Sort cells
    df = subset_df.copy()
    df['_sort'] = df[group_col].map(
        {ct: i for i, ct in enumerate(label_order)}
    ).fillna(99)
    df['_self_z'] = df.apply(
        lambda r: r.get(f'dcm_{r["dcm_cell_type"]}_zscore', 0)
        if r['dcm_cell_type'] in ct_names else 0, axis=1
    )
    df = df.sort_values(['_sort', '_self_z'],
                        ascending=[True, False]).reset_index(drop=True)

    dcm_mat = np.array([
        [row.get(f'dcm_{ct}_zscore', np.nan) for ct in ct_names]
        for _, row in df.iterrows()
    ]).T
    rna_mat = np.array([
        [row.get(f'rna_{ct}_zscore', np.nan) for ct in ct_names]
        for _, row in df.iterrows()
    ]).T

    # ── Shared helper ────────────────────────────────────────────
    def hex_to_rgb(h):
        h = h.lstrip('#')
        return [int(h[i:i+2], 16) / 255.0 for i in (0, 2, 4)]

    def make_ann_rgb(color_col):
        """Build a (1, n_cells, 3) RGB array from a column of cell-type labels."""
        colors = [cell_type_colors.get(v, '#cccccc') for v in df[color_col]]
        return np.array([hex_to_rgb(c) for c in colors])[np.newaxis, :, :]

    # DCM annotation colours (grouped by group_col / dcm_cell_type)
    dcm_ann_rgb = make_ann_rgb(group_col)

    # RNA annotation colours (by rna_cell_type)
    rna_ann_rgb = make_ann_rgb('rna_cell_type')

    # Group boundaries (based on DCM sorting — same for both panels)
    changes = df[group_col].ne(df[group_col].shift()).to_numpy().nonzero()[0]

    # ── Figure layout ─────────────────────────────────────────────
    # 5 rows: DCM annot bar | DCM heatmap | RNA annot bar | RNA heatmap
    # 2 cols: heatmap | colourbar
    fig = plt.figure(figsize=(fig_width, 14))
    gs  = fig.add_gridspec(
        4, 2,
        height_ratios=[0.04, 1, 0.04, 1],
        width_ratios=[1, 0.015],
        hspace=0.06,
        wspace=0.02,
    )

    ax_ann_dcm = fig.add_subplot(gs[0, 0])   # DCM annotation bar
    ax_dcm     = fig.add_subplot(gs[1, 0])   # DCM heatmap
    ax_ann_rna = fig.add_subplot(gs[2, 0])   # RNA annotation bar
    ax_rna     = fig.add_subplot(gs[3, 0])   # RNA heatmap
    ax_cb1     = fig.add_subplot(gs[1, 1])   # colourbar DCM
    ax_cb2     = fig.add_subplot(gs[3, 1])   # colourbar RNA

    def draw_ann_bar(ax, ann_rgb, label_col, title):
        """Draw one annotation bar with dividers and a legend."""
        ax.imshow(ann_rgb, aspect='auto', interpolation='nearest')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_ylabel(title, fontsize=7, rotation=0, labelpad=32, va='center')
        for x in changes:
            ax.axvline(x - 0.5, color='white', lw=1.5)
        # Legend: unique cell types in appearance order
        seen, patches = [], []
        for ct in df[label_col]:
            if ct not in seen:
                seen.append(ct)
                patches.append(mpatches.Patch(
                    facecolor=cell_type_colors.get(ct, '#cccccc'),
                    edgecolor='white', linewidth=0.5, label=ct,
                ))
        ax.legend(handles=patches, loc='upper left', bbox_to_anchor=(1.02, 1.0),
                  fontsize=7, frameon=True, title=title, title_fontsize=7)

    draw_ann_bar(ax_ann_dcm, dcm_ann_rgb, group_col,      'DCM type')
    draw_ann_bar(ax_ann_rna, rna_ann_rgb, 'rna_cell_type', 'RNA type')

    # ── Heatmap panels ────────────────────────────────────────────
    for ax, ax_cb, matrix, label, norm, vcenter_val in [
        (ax_dcm, ax_cb1, dcm_mat, 'DCM (GMM label)',      norm_dcm, 1.0),
        (ax_rna, ax_cb2, rna_mat, 'RNA (leiden_annot_25)', norm_rna, 0.0),
    ]:
        im = ax.imshow(matrix, aspect='auto', cmap=plt.cm.RdBu_r, norm=norm)
        ax.set_yticks(range(len(ct_names)))
        ax.set_yticklabels(ct_names, fontsize=9)
        ax.set_xticks([])
        ax.set_ylabel('Module', fontsize=11, fontweight='bold')
        ax.set_title(label, fontsize=12, fontweight='bold')

        # Group dividers + labels
        for x in changes:
            ax.axvline(x - 0.5, color='white', lw=1.5, alpha=0.8)
        prev = 0
        for x in list(changes) + [n_cells]:
            mid = (prev + x) / 2
            ct  = df.iloc[prev][group_col]
            ax.text(mid, -0.8, ct, ha='center', va='top', fontsize=7,
                    fontweight='bold', color=cell_type_colors.get(ct, 'black'))
            prev = x

        cb = plt.colorbar(im, cax=ax_cb)
        cb.set_label('Z-score', fontsize=8)
        cb.ax.axhline(y=vcenter_val, color='black', lw=1, ls='--')

    fig.suptitle(title_str, fontsize=13, fontweight='bold', y=1.01)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


def get_significance(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'


def plot_cellcycle(df, group_col, group_order, palette,
                   control_group, title_str, save_path):
    if df.empty:
        print(f"  Skipping cell cycle — no cells"); return
    group_order = [g for g in group_order if g in df[group_col].values]
    if not group_order:
        return

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    scores  = ['combined_score', 'S_score', 'G2M_score']
    titles  = ['Combined (S+G2M)', 'S Phase Score', 'G2M Phase Score']
    ylabels = ['Combined Cell Cycle Score\n(S + G2M)', 'S Phase Score', 'G2M Phase Score']
    ctrl_vals = (df[df[group_col] == control_group]['combined_score']
                 if control_group in df[group_col].values else pd.Series(dtype=float))

    for ax, score, ttl, ylabel in zip(axes, scores, titles, ylabels):
        sns.boxplot(data=df, x=group_col, y=score,
                    order=group_order, palette=palette, ax=ax)
        ax.set_xlabel('Transition', fontsize=11, fontweight='bold')
        ax.set_ylabel(ylabel,       fontsize=11, fontweight='bold')
        ax.set_title(ttl,           fontsize=12, fontweight='bold')
        ax.tick_params(axis='x', rotation=45, labelsize=9)
        if score == 'combined_score':
            ymax = ax.get_ylim()[1]
            for i, grp in enumerate(group_order):
                n = len(df[df[group_col] == grp])
                ax.text(i, ymax * 0.97, f'n={n}', ha='center', fontsize=8, fontweight='bold')
                if grp != control_group and len(ctrl_vals) > 0:
                    gv = df[df[group_col] == grp][score]
                    if len(gv) > 1:
                        _, pval = stats.ttest_ind(gv, ctrl_vals)
                        if pval < 0.05:
                            ax.text(i, ymax * 0.89, get_significance(pval),
                                    ha='center', fontsize=12, fontweight='bold')
    plt.suptitle(title_str, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

# ─────────────────────────────────────────────
# STEP 7: CLEAR SECRETORY → ENTEROCYTE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: CLEAR SECRETORY → ENTEROCYTE")
print("=" * 60)

plot_umap(
    sec_to_ent_clear,
    title_str = f'Clear secretory → Enterocyte  (n={len(sec_to_ent_clear)})\nArrow colour = RNA destination',
    save_path = os.path.join(output_dir_umap, 'umap_clear_secretory_to_enterocyte.pdf'),
)
plot_umap_cellcycle(
    title_str = 'All cells — cell cycle score',
    save_path = os.path.join(output_dir_umap, 'umap_cellcycle_all_cells.pdf'),
)

# Combined heatmap: secretory→Enterocyte + TA→Enterocyte together
sec_plus_ta = pd.concat([sec_to_ent_clear, ta_ent_clear], ignore_index=True)
plot_heatmap(
    sec_plus_ta,
    title_str = (f'Z-score — secretory->Enterocyte + TA->Enterocyte  '
                 f'(n={len(sec_to_ent_clear)} + {len(ta_ent_clear)})'),
    save_path = os.path.join(output_dir_heat, 'heatmap_secretory_and_TA_to_enterocyte.pdf'),
    group_col = 'dcm_cell_type',
)

sec_to_sec = conn_clear_df[
    conn_clear_df['dcm_cell_type'].isin(secretory_types) &
    conn_clear_df['rna_cell_type'].isin(secretory_types)
].copy()
sec_to_sec['group'] = 'Secretory->Secretory'

# TA->Enterocyte UMAP (positive control)
plot_umap(
    ta_ent_clear,
    title_str = 'TA -> Enterocyte (positive control, n=' + str(len(ta_ent_clear)) + ')\nArrow colour = RNA destination',
    save_path = os.path.join(output_dir_umap, 'umap_TA_to_enterocyte.pdf'),
)

# Secretory->Secretory UMAP + heatmap (negative control)
plot_umap(
    sec_to_sec,
    title_str = 'Secretory -> Secretory (negative control, n=' + str(len(sec_to_sec)) + ')\nArrow colour = RNA destination',
    save_path = os.path.join(output_dir_umap, 'umap_secretory_to_secretory.pdf'),
)
plot_heatmap(
    sec_to_sec,
    title_str = 'Z-score -- Secretory -> Secretory (negative control, n=' + str(len(sec_to_sec)) + ')',
    save_path = os.path.join(output_dir_heat, 'heatmap_secretory_to_secretory.pdf'),
    group_col = 'dcm_cell_type',
)

cc_combined = pd.concat([
    plot1_cc_clear[plot1_cc_clear['group'].isin(
        [f'{s}→Enterocyte' for s in secretory_types] + ['TA→Enterocyte']
    )],
    sec_to_sec.merge(cc_df, on='batch_plate_well', how='left'),
    hybrid_sec_to_ent.merge(cc_df, on='batch_plate_well', how='left')
    .assign(group='Hybrid(Sec)→Enterocyte'),
], ignore_index=True)

cc_order = (['Secretory→Secretory'] +
            [f'{s}→Enterocyte' for s in secretory_types] +
            ['Hybrid(Sec)→Enterocyte', 'TA→Enterocyte'])
cc_palette = {
    'Secretory→Secretory':    '#95a5a6',
    'Paneth→Enterocyte':      '#DC143C',
    'Goblet→Enterocyte':      '#2E8B57',
    'Tuft→Enterocyte':        '#FF1493',
    'EEC→Enterocyte':         '#4169E1',
    'Hybrid(Sec)→Enterocyte': '#FFD700',
    'TA→Enterocyte':          '#3498db',
}
plot_cellcycle(
    cc_combined, 'group', cc_order, cc_palette, 'TA→Enterocyte',
    title_str = ('Cell Cycle — Secretory → Enterocyte\n'
                 'Neg ctrl: Secretory→Secretory  |  Pos ctrl: TA→Enterocyte'),
    save_path = os.path.join(output_dir_cc, 'cellcycle_secretory_to_enterocyte_full.pdf'),
)

# ─────────────────────────────────────────────
# STEP 8: HYBRID DCM → ENTEROCYTE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: HYBRID DCM → ENTEROCYTE")
print("=" * 60)

plot_umap(
    hybrid_sec_to_ent,
    title_str = f'Hybrid DCM (secretory) → Enterocyte  (n={len(hybrid_sec_to_ent)})',
    save_path = os.path.join(output_dir_umap, 'umap_hybrid_dcm_to_enterocyte.pdf'),
    hybrid_dcm=True,
)

plot_heatmap(
    hybrid_sec_to_ent,
    title_str = f'Z-score — Hybrid DCM → Enterocyte  (n={len(hybrid_sec_to_ent)})',
    save_path = os.path.join(output_dir_heat, 'heatmap_hybrid_dcm_to_enterocyte.pdf'),
)
cc_hybrid = pd.concat([
    hybrid_sec_to_ent.merge(cc_df, on='batch_plate_well', how='left')
    .assign(group='Hybrid(Sec)→Enterocyte'),
    plot1_cc_clear[plot1_cc_clear['group'] == 'TA→Enterocyte'],
], ignore_index=True)
plot_cellcycle(
    cc_hybrid, 'group',
    ['Hybrid(Sec)→Enterocyte', 'TA→Enterocyte'],
    {'Hybrid(Sec)→Enterocyte': '#FFD700', 'TA→Enterocyte': '#3498db'},
    'TA→Enterocyte',
    'Cell Cycle — Hybrid DCM (secretory) → Enterocyte',
    os.path.join(output_dir_cc, 'cellcycle_hybrid_dcm_to_enterocyte.pdf'),
)

# ─────────────────────────────────────────────
# STEP 9: CLEAR SECRETORY → ENTEROCYTE HYBRID RNA
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 9: CLEAR SECRETORY → ENTEROCYTE HYBRID RNA")
print("=" * 60)

plot_umap(
    sec_to_ent_hybrid_rna,
    title_str = (f'Clear Secretory → Enterocyte-hybrid RNA  (n={len(sec_to_ent_hybrid_rna)})\n'
                 f'RNA Enterocyte with secretory z-score ≥ {secretory_zscore_threshold}'),
    save_path = os.path.join(output_dir_umap, 'umap_clear_secretory_to_hybrid_rna_enterocyte.pdf'),
)

plot_heatmap(
    sec_to_ent_hybrid_rna,
    title_str = f'Z-score — Clear Secretory → Enterocyte-hybrid RNA  (n={len(sec_to_ent_hybrid_rna)})',
    save_path = os.path.join(output_dir_heat, 'heatmap_clear_secretory_to_hybrid_rna_enterocyte.pdf'),
)
cc_hrna_palette = {f'{s}→EntHybrid': cell_type_colors[s] for s in secretory_types}
cc_hrna_palette['TA→Enterocyte'] = '#3498db'
cc_hrna = pd.concat([
    sec_to_ent_hybrid_rna.merge(cc_df, on='batch_plate_well', how='left'),
    plot1_cc_clear[plot1_cc_clear['group'] == 'TA→Enterocyte'],
], ignore_index=True)
plot_cellcycle(
    cc_hrna, 'group',
    [f'{s}→EntHybrid' for s in secretory_types] + ['TA→Enterocyte'],
    cc_hrna_palette, 'TA→Enterocyte',
    (f'Cell Cycle — Clear Secretory → Enterocyte-hybrid RNA\n'
     f'RNA Enterocyte with secretory z-score ≥ {secretory_zscore_threshold}'),
    os.path.join(output_dir_cc, 'cellcycle_clear_secretory_to_hybrid_rna_enterocyte.pdf'),
)

# ─────────────────────────────────────────────
# SAVE TABLES
# ─────────────────────────────────────────────
conn_clear_df.to_csv(os.path.join(score_dir, 'connections_clear.csv'),          index=False)
conn_hybrid_df.to_csv(os.path.join(score_dir, 'connections_hybrid.csv'),         index=False)
sec_to_ent_clear.to_csv(os.path.join(score_dir, 'sec_to_ent_clear.csv'),         index=False)
hybrid_sec_to_ent.to_csv(os.path.join(score_dir, 'hybrid_dcm_to_enterocyte.csv'),index=False)
sec_to_ent_hybrid_rna.to_csv(os.path.join(score_dir, 'sec_to_hybrid_rna.csv'),   index=False)
print("\nAll tables saved")
print(f"\nOutputs in:\n  {output_dir_umap}\n  {output_dir_heat}\n  {output_dir_cc}")

## Back to Stem or TA analysis

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/gmm_085/backtoStem'
os.makedirs(score_dir, exist_ok=True)

adata = adata_combined

# Cell types and colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': '#7f7f7f'
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# ─────────────────────────────────────────────
# VCENTER SETTINGS
# DCM: scores reflect the cell's original identity
#      (active module ~1, so vcenter=1.0)
# RNA: scores reflect the new identity after
#      transition (centred on 0 like a z-score)
# Adjust these if your data is on a different scale.
# ─────────────────────────────────────────────
VCENTER_DCM = 0.25
VCENTER_RNA = 0.75
VRANGE      = 3.0   # +/- this many units from vcenter

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata.obs['batch_plate_well'] = (
    adata.obs['Batch'].astype(str) + '_' +
    adata.obs['plate_well'].astype(str)
)

# -----------------------------
# IDENTIFY MODULE SCORE COLUMNS
# -----------------------------
print("\n" + "="*60)
print("STEP 1: IDENTIFYING MODULE SCORE COLUMNS")
print("="*60)

module_cols = []
for ct in label_order:
    col_name = f'ttest_{ct}_module'
    if col_name in adata.obs.columns:
        module_cols.append(col_name)
        print(f"  Found: {col_name}")
    else:
        for alt in [f'ttest_{ct}_zscore', f'ttest_{ct}_score']:
            if alt in adata.obs.columns:
                module_cols.append(alt)
                print(f"  Found (alt): {alt}")
                break

print(f"\nTotal module columns found: {len(module_cols)}")

# -----------------------------
# PREPARE DATA
# -----------------------------
print("\n" + "="*60)
print("STEP 2: PREPARING DATA")
print("="*60)

umap_coords = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1':            umap_coords[:, 0],
    'UMAP2':            umap_coords[:, 1],
    'assay':            adata.obs['assay'].values,
    'plate_well':       adata.obs['plate_well'].values,
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'Batch':            adata.obs['Batch'].values,
    'cell_type_broad':  adata.obs['leiden_annot_25'].astype(str).values,
    'module_max_label': adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'cell_index':       adata.obs.index.values,
    'is_multipotent':   adata.obs['is_multipotent'].astype(str).values
})

if 'Dox' in adata.obs.columns:
    plot_df['Dox'] = adata.obs['Dox'].values

for col in module_cols:
    plot_df[col] = adata.obs[col].values

# -----------------------------
# FILTER AND MATCH CELLS
# -----------------------------
print("\n" + "="*60)
print("STEP 3: FILTERING AND MATCHING CELLS")
print("="*60)

dcm_mask = plot_df['assay'] == 'pDCM'
dcm_known_mask = dcm_mask & plot_df['module_max_label'].isin(label_order)
if 'Dox' in plot_df.columns:
    dcm_known_mask = dcm_known_mask & (plot_df['Dox'] != 'Minus')
dcm_cells = plot_df[dcm_known_mask].copy()

rna_mask = plot_df['assay'] == 'pRNA'
rna_known_mask = rna_mask & plot_df['cell_type_broad'].isin(label_order)
if 'Dox' in plot_df.columns:
    rna_known_mask = rna_known_mask & (plot_df['Dox'] != 'Minus')
rna_cells = plot_df[rna_known_mask].copy()

print(f"DCM cells (excluding Minus Dox): {len(dcm_cells)}")
print(f"RNA cells (excluding Minus Dox): {len(rna_cells)}")

# -----------------------------
# CHECK DUPLICATES
# -----------------------------
print("\n" + "="*60)
print("CHECKING DUPLICATES")
print("="*60)

print(f"DCM batch_plate_wells with >1 cell: "
      f"{(dcm_cells['batch_plate_well'].value_counts() > 1).sum()}")
print(f"RNA batch_plate_wells with >1 cell: "
      f"{(rna_cells['batch_plate_well'].value_counts() > 1).sum()}")

# -----------------------------
# MATCH BY BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("FINDING CONNECTIONS USING BATCH_PLATE_WELL")
print("="*60)

common_bpw  = set(dcm_cells['batch_plate_well']) & set(rna_cells['batch_plate_well'])
dcm_common  = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
rna_common  = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]

dcm_single  = set(dcm_common['batch_plate_well'].value_counts()
                  .pipe(lambda s: s[s == 1].index))
rna_single  = set(rna_common['batch_plate_well'].value_counts()
                  .pipe(lambda s: s[s == 1].index))
both_single = dcm_single & rna_single
print(f"1:1 matched batch_plate_wells: {len(both_single)}")

connections = []
for bpw in both_single:
    dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
    rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
    if len(dcm_cell) == 1 and len(rna_cell) == 1:
        dcm_row = dcm_cell.iloc[0]
        rna_row = rna_cell.iloc[0]
        conn = {
            'batch_plate_well': bpw,
            'batch':            dcm_row['Batch'],
            'plate_well':       dcm_row['plate_well'],
            'dcm_cell_type':    dcm_row['module_max_label'],
            'rna_cell_type':    rna_row['cell_type_broad'],
            'transition':       f"{dcm_row['module_max_label']} -> {rna_row['cell_type_broad']}",
            'dcm_index':        dcm_row['cell_index'],
            'rna_index':        rna_row['cell_index'],
            'dcm_is_multipotent': str(dcm_row.get('is_multipotent', 'False')).lower() in ('true', '1', 'yes'),
            'rna_is_multipotent': str(rna_row.get('is_multipotent', 'False')).lower() in ('true', '1', 'yes'),
        }
        for col in module_cols:
            ct = col.replace('ttest_', '').replace('_module', '') \
                    .replace('_zscore', '').replace('_score', '')
            conn[f'dcm_{ct}_score'] = dcm_row[col]
            conn[f'rna_{ct}_score'] = rna_row[col]
        connections.append(conn)

connections_df = pd.DataFrame(connections)
print(f"Total valid connections: {len(connections_df)}")

# -----------------------------
# FILTER: TO STEM OR TA ONLY
# -----------------------------
print("\n" + "="*60)
print("FILTERING FOR TRANSITIONS TO STEM OR TA")
print("="*60)

connections_df = connections_df[
    (connections_df['rna_cell_type'].isin(['Stem', 'TA'])) &
    (~connections_df['dcm_cell_type'].isin(['Stem', 'TA']))
].copy()

print(f"Connections to Stem or TA (from other cell types): {len(connections_df)}")
print("\nTransition counts:")
print(connections_df['transition'].value_counts())

if len(connections_df) == 0:
    print("\nNo transitions to Stem or TA found.")
else:
    # -----------------------------
    # Z-SCORE TRANSFORMATION
    # -----------------------------
    print("\n" + "="*60)
    print("STEP 4: Z-SCORE TRANSFORMATION")
    print("="*60)

    dcm_score_cols = [c for c in connections_df.columns
                      if c.startswith('dcm_') and c.endswith('_score')]
    rna_score_cols = [c for c in connections_df.columns
                      if c.startswith('rna_') and c.endswith('_score')]

    dcm_all = connections_df[dcm_score_cols].values.flatten()
    dcm_all = dcm_all[~np.isnan(dcm_all)]
    rna_all = connections_df[rna_score_cols].values.flatten()
    rna_all = rna_all[~np.isnan(rna_all)]

    print(f"Raw DCM: mean={dcm_all.mean():.2f}, std={dcm_all.std():.2f}")
    print(f"Raw RNA: mean={rna_all.mean():.2f}, std={rna_all.std():.2f}")

    dcm_mean, dcm_std = dcm_all.mean(), dcm_all.std()
    rna_mean, rna_std = rna_all.mean(), rna_all.std()

    for col in dcm_score_cols:
        connections_df[col.replace('_score', '_zscore')] = \
            (connections_df[col] - dcm_mean) / dcm_std

    for col in rna_score_cols:
        connections_df[col.replace('_score', '_zscore')] = \
            (connections_df[col] - rna_mean) / rna_std

    dcm_zscore_cols = [c for c in connections_df.columns
                       if c.startswith('dcm_') and c.endswith('_zscore')]
    rna_zscore_cols = [c for c in connections_df.columns
                       if c.startswith('rna_') and c.endswith('_zscore')]

    dcm_z = connections_df[dcm_zscore_cols].values.flatten()
    dcm_z = dcm_z[~np.isnan(dcm_z)]
    rna_z = connections_df[rna_zscore_cols].values.flatten()
    rna_z = rna_z[~np.isnan(rna_z)]

    print(f"Z-scored DCM: mean={dcm_z.mean():.4f}, std={dcm_z.std():.4f}")
    print(f"Z-scored RNA: mean={rna_z.mean():.4f}, std={rna_z.std():.4f}")

    # ─────────────────────────────────────────────────────────────
    # SHARED ANNOTATION HELPER
    # ─────────────────────────────────────────────────────────────
    def hex_to_rgb(h):
        h = h.lstrip('#')
        return [int(h[i:i+2], 16) / 255.0 for i in (0, 2, 4)]

    def draw_ann_bar(ax, labels, cell_type_colors, boundaries, bar_title):
        """
        Draw a coloured annotation bar.
        `labels` — list of cell-type strings, one per cell (column).
        `boundaries` — list of x positions where group changes.
        """
        rgb = np.array([hex_to_rgb(cell_type_colors.get(l, '#cccccc'))
                        for l in labels])[np.newaxis, :, :]
        ax.imshow(rgb, aspect='auto', interpolation='nearest')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_ylabel(bar_title, fontsize=7, rotation=0,
                      labelpad=32, va='center')
        for x in boundaries:
            ax.axvline(x - 0.5, color='white', lw=1.5)

        # Legend: unique cell types in order of first appearance
        seen, patches = [], []
        for ct in labels:
            if ct not in seen:
                seen.append(ct)
                patches.append(mpatches.Patch(
                    facecolor=cell_type_colors.get(ct, '#cccccc'),
                    edgecolor='white', linewidth=0.5, label=ct,
                ))
        ax.legend(handles=patches, loc='upper left',
                  bbox_to_anchor=(1.02, 1.0),
                  fontsize=7, frameon=True,
                  title=bar_title, title_fontsize=7)

    # ─────────────────────────────────────────────────────────────
    # HEATMAP FUNCTION
    # ─────────────────────────────────────────────────────────────
    def create_transition_to_target_heatmap(
        connections_df, target_ct, label_order, cell_type_colors,
        score_dir,
        vcenter_dcm=VCENTER_DCM, vcenter_rna=VCENTER_RNA, vrange=VRANGE,
        multipotent_col='dcm_is_multipotent',
    ):
        """
        Heatmap for all transitions TO target_ct.

        Layout (6 rows × 2 cols):
          row 0 — DCM annotation bar   (colour = dcm_cell_type)
          row 1 — DCM z-score heatmap
          row 2 — RNA annotation bar   (colour = rna_cell_type = target_ct)
          row 3 — RNA z-score heatmap
          col 1 — colourbars

        vcenter_dcm : diverging centre for DCM panel  (default 1.0)
        vcenter_rna : diverging centre for RNA panel  (default 0.0)
        vrange      : +/- range from vcenter          (default 3.0)
        """
        print(f"\n{'='*60}")
        print(f"HEATMAP — TRANSITIONS TO {target_ct.upper()}")
        print(f"  vcenter_dcm={vcenter_dcm}  vcenter_rna={vcenter_rna}  vrange={vrange}")
        print(f"{'='*60}")

        subset = connections_df[
            connections_df['rna_cell_type'] == target_ct
        ].copy()

        if len(subset) == 0:
            print(f"  No transitions found to {target_ct}")
            return

        print(f"  Total cells: {len(subset)}")
        print(f"  Source types: {subset['dcm_cell_type'].value_counts().to_dict()}")

        # ── Sort cells by source type then by self-zscore ─────────
        source_order = [ct for ct in label_order
                        if ct not in ['Stem', 'TA']
                        and ct in subset['dcm_cell_type'].unique()]

        all_dcm_rows  = []
        all_rna_rows  = []
        dcm_labels    = []   # per-cell DCM cell type (for annotation bar)
        rna_labels    = []   # per-cell RNA cell type (always target_ct)
        multi_flags   = []   # per-cell multipotent bool
        boundaries    = []
        trans_info    = []
        pos           = 0

        for dcm_ct in source_order:
            sub = subset[subset['dcm_cell_type'] == dcm_ct].copy()
            if len(sub) == 0:
                continue
            sub = sub.sort_values(
                by=[f'dcm_{dcm_ct}_zscore', f'rna_{target_ct}_zscore'],
                ascending=[False, False]
            )
            for _, row in sub.iterrows():
                all_dcm_rows.append([row.get(f'dcm_{ct}_zscore', np.nan)
                                     for ct in label_order])
                all_rna_rows.append([row.get(f'rna_{ct}_zscore', np.nan)
                                     for ct in label_order])
                dcm_labels.append(dcm_ct)
                rna_labels.append(target_ct)
                multi_flags.append(bool(row.get(multipotent_col, False)))

            trans_info.append((dcm_ct, len(sub), pos))
            pos += len(sub)
            boundaries.append(pos)

        if not all_dcm_rows:
            print(f"  No data to plot")
            return

        dcm_matrix = np.array(all_dcm_rows).T
        rna_matrix = np.array(all_rna_rows).T
        n_cells    = dcm_matrix.shape[1]

        # ── Norms ────────────────────────────────────────────────
        norm_dcm = TwoSlopeNorm(vmin=vcenter_dcm - vrange,
                                vcenter=vcenter_dcm,
                                vmax=vcenter_dcm + vrange)
        norm_rna = TwoSlopeNorm(vmin=vcenter_rna - vrange,
                                vcenter=vcenter_rna,
                                vmax=vcenter_rna + vrange)

        fig_width = max(14, n_cells * 0.15)

        # ── GridSpec: 4 rows (ann_dcm | hm_dcm | ann_rna | hm_rna)
        #              × 2 cols (heatmap | colorbar) ────────────────
        fig = plt.figure(figsize=(fig_width, 14))
        gs  = fig.add_gridspec(
            4, 2,
            height_ratios=[0.04, 1, 0.04, 1],
            width_ratios=[1, 0.015],
            hspace=0.06,
            wspace=0.02,
        )

        ax_ann_dcm = fig.add_subplot(gs[0, 0])
        ax_dcm     = fig.add_subplot(gs[1, 0])
        ax_ann_rna = fig.add_subplot(gs[2, 0])
        ax_rna     = fig.add_subplot(gs[3, 0])
        ax_cb_dcm  = fig.add_subplot(gs[1, 1])
        ax_cb_rna  = fig.add_subplot(gs[3, 1])

        # ── Annotation bars ───────────────────────────────────────
        draw_ann_bar(ax_ann_dcm, dcm_labels, cell_type_colors,
                     boundaries, 'DCM type')
        draw_ann_bar(ax_ann_rna, rna_labels, cell_type_colors,
                     boundaries, 'RNA type')

        # ── Heatmap panels ────────────────────────────────────────
        for ax, ax_cb, matrix, norm, vcenter_val, panel_label, highlight_ct in [
            (ax_dcm, ax_cb_dcm, dcm_matrix, norm_dcm,
             vcenter_dcm, f'DCM Module Z-Scores → {target_ct}', None),
            (ax_rna, ax_cb_rna, rna_matrix, norm_rna,
             vcenter_rna, f'RNA Module Z-Scores (after → {target_ct})', target_ct),
        ]:
            im = ax.imshow(matrix, aspect='auto',
                           cmap=plt.cm.RdBu_r, norm=norm)

            ax.set_yticks(range(len(label_order)))
            ax.set_yticklabels(label_order, fontsize=10)
            ax.set_xticks([])
            ax.set_ylabel('Module', fontsize=11, fontweight='bold')
            ax.set_title(panel_label, fontsize=12, fontweight='bold')

            # Group dividers
            for b in boundaries[:-1]:
                ax.axvline(b - 0.5, color='white', lw=2)

            # Group labels below panel
            prev = 0
            for dcm_ct, n_c, start in trans_info:
                mid = start + n_c / 2
                ax.text(mid, -0.8, f'{dcm_ct}\n(n={n_c})',
                        ha='center', va='bottom', fontsize=9,
                        fontweight='bold',
                        color=cell_type_colors.get(dcm_ct, 'black'))
                prev = start + n_c

            # Highlight target row on RNA panel
            if highlight_ct is not None and highlight_ct in label_order:
                idx = label_order.index(highlight_ct)
                ax.axhline(idx - 0.5, color='black', lw=2,
                           linestyle='--', alpha=0.8)
                ax.axhline(idx + 0.5, color='black', lw=2,
                           linestyle='--', alpha=0.8)

            # ── Multipotent * markers ──────────────────────
            for cell_idx, is_multi in enumerate(multi_flags):
                if is_multi:
                    ax.text(cell_idx, -0.3, '*',
                            ha='center', va='bottom', fontsize=8,
                            fontweight='bold', color='#333333')

            cb = plt.colorbar(im, cax=ax_cb)
            cb.set_label('Z-Score', fontsize=9)
            cb.ax.axhline(y=vcenter_val, color='black', lw=1, ls='--')

        fig.suptitle(
            f'Z-Score Module Scores — Transitions to {target_ct}\n'
            f'(n={n_cells} cells | vcenter DCM={vcenter_dcm}, '
            f'RNA={vcenter_rna} | vrange=±{vrange})',
            fontsize=13, fontweight='bold', y=1.01,
        )
        fig.text(0.01, 0.005,
                 'Dashed lines on RNA panel = target module row  |  * = multipotent cell',
                 fontsize=7, color='#555555')
        plt.tight_layout()
        save_path = os.path.join(score_dir,
                                 f'heatmap_zscore_transitions_to_{target_ct}.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {save_path}")

    # ── Generate heatmaps ─────────────────────────────────────────
    for target in ['Stem', 'TA']:
        if target in connections_df['rna_cell_type'].unique():
            create_transition_to_target_heatmap(
                connections_df, target, label_order,
                cell_type_colors, score_dir,
                vcenter_dcm=VCENTER_DCM,
                vcenter_rna=VCENTER_RNA,
                vrange=VRANGE,
            )


    # ─────────────────────────────────────────────────────────────
    # STABLE HEATMAP FUNCTION
    # For Stem→Stem and TA→TA (cells that keep their label)
    # Same layout as create_transition_to_target_heatmap but
    # source == target, so both annotation bars show the same colour.
    # ─────────────────────────────────────────────────────────────
    def create_stable_heatmap(
        all_connections_df, stable_ct, label_order, cell_type_colors,
        score_dir,
        vcenter_dcm=VCENTER_DCM, vcenter_rna=VCENTER_RNA, vrange=VRANGE,
        multipotent_col='dcm_is_multipotent',
    ):
        """
        Heatmap for stable transitions: DCM == stable_ct AND RNA == stable_ct.
        Same 4-row layout with annotation bars and * for multipotent cells.
        """
        print(f"\n{'='*60}")
        print(f"STABLE HEATMAP — {stable_ct.upper()} → {stable_ct.upper()}")
        print(f"  vcenter_dcm={vcenter_dcm}  vcenter_rna={vcenter_rna}  vrange={vrange}")
        print(f"{'='*60}")

        subset = all_connections_df[
            (all_connections_df['dcm_cell_type'] == stable_ct) &
            (all_connections_df['rna_cell_type']  == stable_ct)
        ].copy()

        if len(subset) == 0:
            print(f"  No stable {stable_ct}→{stable_ct} cells found")
            return

        print(f"  Total stable cells: {len(subset)}")

        # Sort by self-DCM zscore descending
        dcm_self_col = f'dcm_{stable_ct}_zscore'
        if dcm_self_col in subset.columns:
            subset = subset.sort_values(dcm_self_col, ascending=False)
        subset = subset.reset_index(drop=True)

        dcm_matrix = np.array(
            [[row.get(f'dcm_{ct}_zscore', np.nan) for ct in label_order]
             for _, row in subset.iterrows()]
        ).T
        rna_matrix = np.array(
            [[row.get(f'rna_{ct}_zscore', np.nan) for ct in label_order]
             for _, row in subset.iterrows()]
        ).T

        dcm_labels  = [stable_ct] * len(subset)
        rna_labels  = [stable_ct] * len(subset)
        multi_flags = [bool(row.get(multipotent_col, False))
                       for _, row in subset.iterrows()]
        n_cells     = len(subset)
        boundaries  = []   # no group splits for stable

        norm_dcm = TwoSlopeNorm(vmin=vcenter_dcm - vrange,
                                vcenter=vcenter_dcm,
                                vmax=vcenter_dcm + vrange)
        norm_rna = TwoSlopeNorm(vmin=vcenter_rna - vrange,
                                vcenter=vcenter_rna,
                                vmax=vcenter_rna + vrange)

        fig_width = max(14, n_cells * 0.15)

        fig = plt.figure(figsize=(fig_width, 14))
        gs  = fig.add_gridspec(
            4, 2,
            height_ratios=[0.04, 1, 0.04, 1],
            width_ratios=[1, 0.015],
            hspace=0.06,
            wspace=0.02,
        )
        ax_ann_dcm = fig.add_subplot(gs[0, 0])
        ax_dcm     = fig.add_subplot(gs[1, 0])
        ax_ann_rna = fig.add_subplot(gs[2, 0])
        ax_rna     = fig.add_subplot(gs[3, 0])
        ax_cb_dcm  = fig.add_subplot(gs[1, 1])
        ax_cb_rna  = fig.add_subplot(gs[3, 1])

        draw_ann_bar(ax_ann_dcm, dcm_labels, cell_type_colors,
                     boundaries, 'DCM type')
        draw_ann_bar(ax_ann_rna, rna_labels, cell_type_colors,
                     boundaries, 'RNA type')

        for ax, ax_cb, matrix, norm, vcenter_val, panel_label in [
            (ax_dcm, ax_cb_dcm, dcm_matrix, norm_dcm,
             vcenter_dcm, f'DCM Module Z-Scores ({stable_ct}→{stable_ct})'),
            (ax_rna, ax_cb_rna, rna_matrix, norm_rna,
             vcenter_rna, f'RNA Module Z-Scores ({stable_ct}→{stable_ct})'),
        ]:
            im = ax.imshow(matrix, aspect='auto',
                           cmap=plt.cm.RdBu_r, norm=norm)
            ax.set_yticks(range(len(label_order)))
            ax.set_yticklabels(label_order, fontsize=10)
            ax.set_xticks([])
            ax.set_ylabel('Module', fontsize=11, fontweight='bold')
            ax.set_title(panel_label, fontsize=12, fontweight='bold')

            # Highlight self-module row
            if stable_ct in label_order:
                idx = label_order.index(stable_ct)
                ax.axhline(idx - 0.5, color='black', lw=2,
                           linestyle='--', alpha=0.8)
                ax.axhline(idx + 0.5, color='black', lw=2,
                           linestyle='--', alpha=0.8)

            # n label below
            ax.text(n_cells / 2, -0.8, f'{stable_ct}→{stable_ct}  (n={n_cells})',
                    ha='center', va='bottom', fontsize=9, fontweight='bold',
                    color=cell_type_colors.get(stable_ct, 'black'))

            # * markers for multipotent cells
            for cell_idx, is_multi in enumerate(multi_flags):
                if is_multi:
                    ax.text(cell_idx, -0.3, '*',
                            ha='center', va='bottom', fontsize=8,
                            fontweight='bold', color='#333333')

            cb = plt.colorbar(im, cax=ax_cb)
            cb.set_label('Z-Score', fontsize=9)
            cb.ax.axhline(y=vcenter_val, color='black', lw=1, ls='--')

        fig.suptitle(
            f'Z-Score Module Scores — Stable {stable_ct}→{stable_ct}\n'
            f'(n={n_cells} cells | vcenter DCM={vcenter_dcm}, '
            f'RNA={vcenter_rna} | vrange=±{vrange})',
            fontsize=13, fontweight='bold', y=1.01,
        )
        fig.text(0.01, 0.005,
                 'Dashed lines = self-module row  |  * = multipotent cell',
                 fontsize=7, color='#555555')
        plt.tight_layout()
        save_path = os.path.join(score_dir,
                                 f'heatmap_stable_{stable_ct}_to_{stable_ct}.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {save_path}")

    # ── Build all-connections df for stable lookup (no filter applied) ──
    all_conn_df = pd.DataFrame(connections)   # raw connections before filter

    # Z-score columns are not on all_conn_df yet — compute them
    _dcm_sc = [c for c in all_conn_df.columns if c.startswith('dcm_') and c.endswith('_score')]
    _rna_sc = [c for c in all_conn_df.columns if c.startswith('rna_') and c.endswith('_score')]
    for c in _dcm_sc:
        all_conn_df[c.replace('_score', '_zscore')] = (all_conn_df[c] - dcm_mean) / dcm_std
    for c in _rna_sc:
        all_conn_df[c.replace('_score', '_zscore')] = (all_conn_df[c] - rna_mean) / rna_std

    # ─────────────────────────────────────────────────────────────
    # COMBINED STABLE HEATMAP FUNCTION
    # Stem→Stem and TA→TA in a single figure, separated by a white
    # divider. Also returns the sorted cell order as a DataFrame
    # so it can be saved to CSV.
    # ─────────────────────────────────────────────────────────────
    def create_stable_combined_heatmap(
        all_connections_df, label_order, cell_type_colors, score_dir,
        vcenter_dcm=VCENTER_DCM, vcenter_rna=VCENTER_RNA, vrange=VRANGE,
        multipotent_col='dcm_is_multipotent',
    ):
        """
        Single figure with Stem→Stem and TA→TA cells side by side.
        A white boundary divides the two groups.
        Returns a DataFrame with the cell order (batch_plate_well,
        stable_type, heatmap_position) for CSV export.
        """
        all_dcm_rows  = []
        all_rna_rows  = []
        all_labels    = []   # per-cell cell type label
        multi_flags   = []
        order_records = []   # for CSV output
        boundaries    = []
        group_info    = []   # (stable_ct, n_cells, start_pos)
        pos           = 0

        for stable_ct in ['Stem', 'TA']:
            subset = all_connections_df[
                (all_connections_df['dcm_cell_type'] == stable_ct) &
                (all_connections_df['rna_cell_type']  == stable_ct)
            ].copy()

            if len(subset) == 0:
                print(f"  No stable {stable_ct}→{stable_ct} cells — skipping")
                continue

            # Sort by self-DCM zscore descending
            dcm_self_col = f'dcm_{stable_ct}_zscore'
            if dcm_self_col in subset.columns:
                subset = subset.sort_values(dcm_self_col, ascending=False)
            subset = subset.reset_index(drop=True)

            for rank, (_, row) in enumerate(subset.iterrows()):
                all_dcm_rows.append([row.get(f'dcm_{ct}_zscore', np.nan)
                                     for ct in label_order])
                all_rna_rows.append([row.get(f'rna_{ct}_zscore', np.nan)
                                     for ct in label_order])
                all_labels.append(stable_ct)
                multi_flags.append(bool(row.get(multipotent_col, False)))
                order_records.append({
                    'heatmap_position':  pos + rank,
                    'stable_type':       stable_ct,
                    'batch_plate_well':  row.get('batch_plate_well', ''),
                    'is_multipotent':    bool(row.get(multipotent_col, False)),
                    f'dcm_{stable_ct}_zscore': row.get(dcm_self_col, np.nan),
                })

            group_info.append((stable_ct, len(subset), pos))
            pos += len(subset)
            boundaries.append(pos)   # divider after this group

        if not all_dcm_rows:
            print("  No stable Stem or TA cells found — skipping combined heatmap")
            return pd.DataFrame()

        dcm_matrix = np.array(all_dcm_rows).T
        rna_matrix = np.array(all_rna_rows).T
        n_cells    = dcm_matrix.shape[1]

        norm_dcm = TwoSlopeNorm(vmin=vcenter_dcm - vrange,
                                vcenter=vcenter_dcm,
                                vmax=vcenter_dcm + vrange)
        norm_rna = TwoSlopeNorm(vmin=vcenter_rna - vrange,
                                vcenter=vcenter_rna,
                                vmax=vcenter_rna + vrange)

        fig_width = max(14, n_cells * 0.15)
        fig = plt.figure(figsize=(fig_width, 14))
        gs  = fig.add_gridspec(
            4, 2,
            height_ratios=[0.04, 1, 0.04, 1],
            width_ratios=[1, 0.015],
            hspace=0.06,
            wspace=0.02,
        )
        ax_ann_dcm = fig.add_subplot(gs[0, 0])
        ax_dcm     = fig.add_subplot(gs[1, 0])
        ax_ann_rna = fig.add_subplot(gs[2, 0])
        ax_rna     = fig.add_subplot(gs[3, 0])
        ax_cb_dcm  = fig.add_subplot(gs[1, 1])
        ax_cb_rna  = fig.add_subplot(gs[3, 1])

        # Annotation bars — coloured by cell type
        draw_ann_bar(ax_ann_dcm, all_labels, cell_type_colors,
                     boundaries[:-1], 'DCM type')
        draw_ann_bar(ax_ann_rna, all_labels, cell_type_colors,
                     boundaries[:-1], 'RNA type')

        for ax, ax_cb, matrix, norm, vcenter_val, panel_label in [
            (ax_dcm, ax_cb_dcm, dcm_matrix, norm_dcm,
             vcenter_dcm, 'DCM Module Z-Scores (Stable Stem→Stem & TA→TA)'),
            (ax_rna, ax_cb_rna, rna_matrix, norm_rna,
             vcenter_rna, 'RNA Module Z-Scores (Stable Stem→Stem & TA→TA)'),
        ]:
            im = ax.imshow(matrix, aspect='auto',
                           cmap=plt.cm.RdBu_r, norm=norm)
            ax.set_yticks(range(len(label_order)))
            ax.set_yticklabels(label_order, fontsize=10)
            ax.set_xticks([])
            ax.set_ylabel('Module', fontsize=11, fontweight='bold')
            ax.set_title(panel_label, fontsize=12, fontweight='bold')

            # White group divider
            for b in boundaries[:-1]:
                ax.axvline(b - 0.5, color='white', lw=3)

            # Group labels + self-module row highlight per group
            for stable_ct, n_c, start in group_info:
                mid = start + n_c / 2
                ax.text(mid, -0.8, f'{stable_ct}→{stable_ct}  (n={n_c})',
                        ha='center', va='bottom', fontsize=9,
                        fontweight='bold',
                        color=cell_type_colors.get(stable_ct, 'black'))
                # Dashed highlight around self-module row
                if stable_ct in label_order:
                    idx = label_order.index(stable_ct)
                    ax.axhline(idx - 0.5, color=cell_type_colors.get(stable_ct, 'black'),
                               lw=1.5, linestyle='--', alpha=0.6,
                               xmin=start / n_cells,
                               xmax=(start + n_c) / n_cells)
                    ax.axhline(idx + 0.5, color=cell_type_colors.get(stable_ct, 'black'),
                               lw=1.5, linestyle='--', alpha=0.6,
                               xmin=start / n_cells,
                               xmax=(start + n_c) / n_cells)

            # * markers for multipotent cells
            for cell_idx, is_multi in enumerate(multi_flags):
                if is_multi:
                    ax.text(cell_idx, -0.3, '*',
                            ha='center', va='bottom', fontsize=8,
                            fontweight='bold', color='#333333')

            cb = plt.colorbar(im, cax=ax_cb)
            cb.set_label('Z-Score', fontsize=9)
            cb.ax.axhline(y=vcenter_val, color='black', lw=1, ls='--')

        n_stem = next((n for ct, n, _ in group_info if ct == 'Stem'), 0)
        n_ta   = next((n for ct, n, _ in group_info if ct == 'TA'),   0)
        fig.suptitle(
            f'Z-Score Module Scores — Stable Stem→Stem (n={n_stem}) & TA→TA (n={n_ta})\n'
            f'vcenter DCM={vcenter_dcm}, RNA={vcenter_rna} | vrange=±{vrange}',
            fontsize=13, fontweight='bold', y=1.01,
        )
        fig.text(0.01, 0.005,
                 'Dashed lines = self-module row per group  |  * = multipotent cell',
                 fontsize=7, color='#555555')
        plt.tight_layout()
        save_path = os.path.join(score_dir,
                                 'heatmap_stable_Stem_and_TA_combined.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {save_path}")

        return pd.DataFrame(order_records)

    # ── Generate stable heatmaps ─────────────────────────────────────
    print("\n" + "="*60)
    print("STABLE HEATMAPS — Stem→Stem and TA→TA")
    print("="*60)

    # Individual per-type heatmaps
    for stable_ct in ['Stem', 'TA']:
        create_stable_heatmap(
            all_conn_df, stable_ct, label_order,
            cell_type_colors, score_dir,
            vcenter_dcm=VCENTER_DCM,
            vcenter_rna=VCENTER_RNA,
            vrange=VRANGE,
        )

    # Combined Stem + TA heatmap + CSV cell order
    cell_order_df = create_stable_combined_heatmap(
        all_conn_df, label_order, cell_type_colors, score_dir,
        vcenter_dcm=VCENTER_DCM,
        vcenter_rna=VCENTER_RNA,
        vrange=VRANGE,
    )
    if not cell_order_df.empty:
        order_csv = os.path.join(score_dir,
                                 'stable_combined_cell_order.csv')
        cell_order_df.to_csv(order_csv, index=False)
        print(f"  Cell order CSV: {order_csv}")
        print(cell_order_df.to_string(index=False))

    # -----------------------------
    # SUMMARY TRANSITION HEATMAP
    # -----------------------------
    print("\n" + "="*60)
    print("CREATING SUMMARY HEATMAP")
    print("="*60)

    transition_matrix = pd.crosstab(
        connections_df['dcm_cell_type'],
        connections_df['rna_cell_type'],
        rownames=['DCM Cell Type'],
        colnames=['RNA Cell Type'],
    )
    dcm_order = [ct for ct in label_order
                 if ct not in ['Stem', 'TA']
                 and ct in transition_matrix.index]
    rna_order = [ct for ct in ['Stem', 'TA']
                 if ct in transition_matrix.columns]
    transition_matrix = transition_matrix.reindex(
        index=dcm_order, columns=rna_order
    ).fillna(0).astype(int)

    print("\nTransition Matrix (to Stem/TA):")
    print(transition_matrix)

    fig, ax = plt.subplots(figsize=(6, 8))
    sns.heatmap(transition_matrix, annot=True, fmt='d', cmap='YlOrRd',
                cbar_kws={'label': 'Number of cells'}, ax=ax,
                linewidths=0.5, linecolor='white')
    ax.set_title('Transitions to Stem/TA from Other Cell Types\n'
                 '(Matched using Batch + Plate_Well)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('RNA Cell Type', fontsize=11, fontweight='bold')
    ax.set_ylabel('DCM Cell Type', fontsize=11, fontweight='bold')
    plt.tight_layout()
    save_path = os.path.join(score_dir, 'heatmap_summary_to_stem_ta.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")

    # -----------------------------
    # Z-SCORE SUMMARY
    # -----------------------------
    print("\n" + "="*60)
    print("Z-SCORE SUMMARY")
    print("="*60)

    for name, arr, vcenter in [
        ('DCM', dcm_z, VCENTER_DCM),
        ('RNA', rna_z, VCENTER_RNA),
    ]:
        lo, hi = vcenter - VRANGE, vcenter + VRANGE
        pct = ((arr >= lo) & (arr <= hi)).mean() * 100
        print(f"\n{name}: min={arr.min():.2f}  max={arr.max():.2f}  "
              f"mean={arr.mean():.4f}  std={arr.std():.4f}  "
              f"% in [{lo:.1f},{hi:.1f}]={pct:.1f}%")

    # -----------------------------
    # SAVE DATA
    # -----------------------------
    print("\n" + "="*60)
    print("SAVING DATA")
    print("="*60)

    out_csv = os.path.join(score_dir, 'transitions_to_stem_ta_with_zscores.csv')
    connections_df.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")

    print("\n" + "="*60)
    print("DONE!")
    print("="*60)

In [ ]:
#Cell list of Stem and TA going back or stay stable

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
import os

# -----------------------------
# SETTINGS
# -----------------------------
score_dir = f'{root}/gene_scoring/gmm_085/backtoStem'
os.makedirs(score_dir, exist_ok=True)

adata = adata_combined

# Cell types and colors
cell_type_colors = {
    'Stem': '#9370DB',
    'TA': '#D2B48C',
    'Enterocyte': '#FF8C00',
    'Paneth': '#DC143C',
    'Goblet': '#2E8B57',
    'Tuft': '#FF1493',
    'EEC': '#4169E1',
    'ambiguous': '#7f7f7f'
}

label_order = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']

# ─────────────────────────────────────────────
# VCENTER SETTINGS
# DCM: scores reflect the cell's original identity
#      (active module ~1, so vcenter=1.0)
# RNA: scores reflect the new identity after
#      transition (centred on 0 like a z-score)
# Adjust these if your data is on a different scale.
# ─────────────────────────────────────────────
VCENTER_DCM = 0.25
VCENTER_RNA = 0.75
VRANGE      = 3.0   # +/- this many units from vcenter

# -----------------------------
# CREATE BATCH_PLATE_WELL IDENTIFIER
# -----------------------------
print("="*60)
print("CREATING BATCH_PLATE_WELL IDENTIFIER")
print("="*60)

adata.obs['batch_plate_well'] = (
    adata.obs['Batch'].astype(str) + '_' +
    adata.obs['plate_well'].astype(str)
)

# -----------------------------
# IDENTIFY MODULE SCORE COLUMNS
# -----------------------------
print("\n" + "="*60)
print("STEP 1: IDENTIFYING MODULE SCORE COLUMNS")
print("="*60)

module_cols = []
for ct in label_order:
    col_name = f'ttest_{ct}_module'
    if col_name in adata.obs.columns:
        module_cols.append(col_name)
        print(f"  Found: {col_name}")
    else:
        for alt in [f'ttest_{ct}_zscore', f'ttest_{ct}_score']:
            if alt in adata.obs.columns:
                module_cols.append(alt)
                print(f"  Found (alt): {alt}")
                break

print(f"\nTotal module columns found: {len(module_cols)}")

# -----------------------------
# PREPARE DATA
# -----------------------------
print("\n" + "="*60)
print("STEP 2: PREPARING DATA")
print("="*60)

umap_coords = adata.obsm['X_umap']

plot_df = pd.DataFrame({
    'UMAP1':            umap_coords[:, 0],
    'UMAP2':            umap_coords[:, 1],
    'assay':            adata.obs['assay'].values,
    'plate_well':       adata.obs['plate_well'].values,
    'batch_plate_well': adata.obs['batch_plate_well'].values,
    'Batch':            adata.obs['Batch'].values,
    'cell_type_broad':  adata.obs['leiden_annot_25'].astype(str).values,
    'module_max_label': adata.obs['ttest_gmm_label_confident_085'].astype(str).values,
    'cell_index':       adata.obs.index.values,
    'is_multipotent':   adata.obs['is_multipotent'].astype(str).values
})

if 'Dox' in adata.obs.columns:
    plot_df['Dox'] = adata.obs['Dox'].values

for col in module_cols:
    plot_df[col] = adata.obs[col].values

# -----------------------------
# FILTER AND MATCH CELLS
# -----------------------------
print("\n" + "="*60)
print("STEP 3: FILTERING AND MATCHING CELLS")
print("="*60)

dcm_mask = plot_df['assay'] == 'pDCM'
dcm_known_mask = dcm_mask & plot_df['module_max_label'].isin(label_order)
if 'Dox' in plot_df.columns:
    dcm_known_mask = dcm_known_mask & (plot_df['Dox'] != 'Minus')
dcm_cells = plot_df[dcm_known_mask].copy()

rna_mask = plot_df['assay'] == 'pRNA'
rna_known_mask = rna_mask & plot_df['cell_type_broad'].isin(label_order)
if 'Dox' in plot_df.columns:
    rna_known_mask = rna_known_mask & (plot_df['Dox'] != 'Minus')
rna_cells = plot_df[rna_known_mask].copy()

print(f"DCM cells (excluding Minus Dox): {len(dcm_cells)}")
print(f"RNA cells (excluding Minus Dox): {len(rna_cells)}")

# -----------------------------
# CHECK DUPLICATES
# -----------------------------
print("\n" + "="*60)
print("CHECKING DUPLICATES")
print("="*60)

print(f"DCM batch_plate_wells with >1 cell: "
      f"{(dcm_cells['batch_plate_well'].value_counts() > 1).sum()}")
print(f"RNA batch_plate_wells with >1 cell: "
      f"{(rna_cells['batch_plate_well'].value_counts() > 1).sum()}")

# -----------------------------
# MATCH BY BATCH_PLATE_WELL
# -----------------------------
print("\n" + "="*60)
print("FINDING CONNECTIONS USING BATCH_PLATE_WELL")
print("="*60)

common_bpw  = set(dcm_cells['batch_plate_well']) & set(rna_cells['batch_plate_well'])
dcm_common  = dcm_cells[dcm_cells['batch_plate_well'].isin(common_bpw)]
rna_common  = rna_cells[rna_cells['batch_plate_well'].isin(common_bpw)]

dcm_single  = set(dcm_common['batch_plate_well'].value_counts()
                  .pipe(lambda s: s[s == 1].index))
rna_single  = set(rna_common['batch_plate_well'].value_counts()
                  .pipe(lambda s: s[s == 1].index))
both_single = dcm_single & rna_single
print(f"1:1 matched batch_plate_wells: {len(both_single)}")

connections = []
for bpw in both_single:
    dcm_cell = dcm_cells[dcm_cells['batch_plate_well'] == bpw]
    rna_cell = rna_cells[rna_cells['batch_plate_well'] == bpw]
    if len(dcm_cell) == 1 and len(rna_cell) == 1:
        dcm_row = dcm_cell.iloc[0]
        rna_row = rna_cell.iloc[0]
        conn = {
            'batch_plate_well': bpw,
            'batch':            dcm_row['Batch'],
            'plate_well':       dcm_row['plate_well'],
            'dcm_cell_type':    dcm_row['module_max_label'],
            'rna_cell_type':    rna_row['cell_type_broad'],
            'transition':       f"{dcm_row['module_max_label']} -> {rna_row['cell_type_broad']}",
            'dcm_index':        dcm_row['cell_index'],
            'rna_index':        rna_row['cell_index'],
            'dcm_is_multipotent': str(dcm_row.get('is_multipotent', 'False')).lower() in ('true', '1', 'yes'),
            'rna_is_multipotent': str(rna_row.get('is_multipotent', 'False')).lower() in ('true', '1', 'yes'),
        }
        for col in module_cols:
            ct = col.replace('ttest_', '').replace('_module', '') \
                    .replace('_zscore', '').replace('_score', '')
            conn[f'dcm_{ct}_score'] = dcm_row[col]
            conn[f'rna_{ct}_score'] = rna_row[col]
        connections.append(conn)

connections_df = pd.DataFrame(connections)
print(f"Total valid connections: {len(connections_df)}")

# -----------------------------
# FILTER: TO STEM OR TA ONLY
# -----------------------------
print("\n" + "="*60)
print("FILTERING FOR TRANSITIONS TO STEM OR TA")
print("="*60)

connections_df = connections_df[
    (connections_df['rna_cell_type'].isin(['Stem', 'TA'])) &
    (~connections_df['dcm_cell_type'].isin(['Stem', 'TA']))
].copy()

print(f"Connections to Stem or TA (from other cell types): {len(connections_df)}")
print("\nTransition counts:")
print(connections_df['transition'].value_counts())

if len(connections_df) == 0:
    print("\nNo transitions to Stem or TA found.")
else:
    # -----------------------------
    # Z-SCORE TRANSFORMATION
    # -----------------------------
    print("\n" + "="*60)
    print("STEP 4: Z-SCORE TRANSFORMATION")
    print("="*60)

    dcm_score_cols = [c for c in connections_df.columns
                      if c.startswith('dcm_') and c.endswith('_score')]
    rna_score_cols = [c for c in connections_df.columns
                      if c.startswith('rna_') and c.endswith('_score')]

    dcm_all = connections_df[dcm_score_cols].values.flatten()
    dcm_all = dcm_all[~np.isnan(dcm_all)]
    rna_all = connections_df[rna_score_cols].values.flatten()
    rna_all = rna_all[~np.isnan(rna_all)]

    print(f"Raw DCM: mean={dcm_all.mean():.2f}, std={dcm_all.std():.2f}")
    print(f"Raw RNA: mean={rna_all.mean():.2f}, std={rna_all.std():.2f}")

    dcm_mean, dcm_std = dcm_all.mean(), dcm_all.std()
    rna_mean, rna_std = rna_all.mean(), rna_all.std()

    for col in dcm_score_cols:
        connections_df[col.replace('_score', '_zscore')] = \
            (connections_df[col] - dcm_mean) / dcm_std

    for col in rna_score_cols:
        connections_df[col.replace('_score', '_zscore')] = \
            (connections_df[col] - rna_mean) / rna_std

    dcm_zscore_cols = [c for c in connections_df.columns
                       if c.startswith('dcm_') and c.endswith('_zscore')]
    rna_zscore_cols = [c for c in connections_df.columns
                       if c.startswith('rna_') and c.endswith('_zscore')]

    dcm_z = connections_df[dcm_zscore_cols].values.flatten()
    dcm_z = dcm_z[~np.isnan(dcm_z)]
    rna_z = connections_df[rna_zscore_cols].values.flatten()
    rna_z = rna_z[~np.isnan(rna_z)]

    print(f"Z-scored DCM: mean={dcm_z.mean():.4f}, std={dcm_z.std():.4f}")
    print(f"Z-scored RNA: mean={rna_z.mean():.4f}, std={rna_z.std():.4f}")

    # ─────────────────────────────────────────────────────────────
    # SHARED ANNOTATION HELPER
    # ─────────────────────────────────────────────────────────────
    def hex_to_rgb(h):
        h = h.lstrip('#')
        return [int(h[i:i+2], 16) / 255.0 for i in (0, 2, 4)]

    def draw_ann_bar(ax, labels, cell_type_colors, boundaries, bar_title):
        """
        Draw a coloured annotation bar.
        `labels` — list of cell-type strings, one per cell (column).
        `boundaries` — list of x positions where group changes.
        """
        rgb = np.array([hex_to_rgb(cell_type_colors.get(l, '#cccccc'))
                        for l in labels])[np.newaxis, :, :]
        ax.imshow(rgb, aspect='auto', interpolation='nearest')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_ylabel(bar_title, fontsize=7, rotation=0,
                      labelpad=32, va='center')
        for x in boundaries:
            ax.axvline(x - 0.5, color='white', lw=1.5)

        # Legend: unique cell types in order of first appearance
        seen, patches = [], []
        for ct in labels:
            if ct not in seen:
                seen.append(ct)
                patches.append(mpatches.Patch(
                    facecolor=cell_type_colors.get(ct, '#cccccc'),
                    edgecolor='white', linewidth=0.5, label=ct,
                ))
        ax.legend(handles=patches, loc='upper left',
                  bbox_to_anchor=(1.02, 1.0),
                  fontsize=7, frameon=True,
                  title=bar_title, title_fontsize=7)

    # ─────────────────────────────────────────────────────────────
    # HEATMAP FUNCTION
    # ─────────────────────────────────────────────────────────────
    def create_transition_to_target_heatmap(
        connections_df, target_ct, label_order, cell_type_colors,
        score_dir,
        vcenter_dcm=VCENTER_DCM, vcenter_rna=VCENTER_RNA, vrange=VRANGE,
        multipotent_col='dcm_is_multipotent',
    ):
        """
        Heatmap for all transitions TO target_ct.

        Layout (6 rows × 2 cols):
          row 0 — DCM annotation bar   (colour = dcm_cell_type)
          row 1 — DCM z-score heatmap
          row 2 — RNA annotation bar   (colour = rna_cell_type = target_ct)
          row 3 — RNA z-score heatmap
          col 1 — colourbars

        vcenter_dcm : diverging centre for DCM panel  (default 1.0)
        vcenter_rna : diverging centre for RNA panel  (default 0.0)
        vrange      : +/- range from vcenter          (default 3.0)
        """
        print(f"\n{'='*60}")
        print(f"HEATMAP — TRANSITIONS TO {target_ct.upper()}")
        print(f"  vcenter_dcm={vcenter_dcm}  vcenter_rna={vcenter_rna}  vrange={vrange}")
        print(f"{'='*60}")

        subset = connections_df[
            connections_df['rna_cell_type'] == target_ct
        ].copy()

        if len(subset) == 0:
            print(f"  No transitions found to {target_ct}")
            return

        print(f"  Total cells: {len(subset)}")
        print(f"  Source types: {subset['dcm_cell_type'].value_counts().to_dict()}")

        # ── Sort cells by source type then by self-zscore ─────────
        source_order = [ct for ct in label_order
                        if ct not in ['Stem', 'TA']
                        and ct in subset['dcm_cell_type'].unique()]

        all_dcm_rows  = []
        all_rna_rows  = []
        dcm_labels    = []   # per-cell DCM cell type (for annotation bar)
        rna_labels    = []   # per-cell RNA cell type (always target_ct)
        multi_flags   = []   # per-cell multipotent bool
        boundaries    = []
        trans_info    = []
        pos           = 0

        for dcm_ct in source_order:
            sub = subset[subset['dcm_cell_type'] == dcm_ct].copy()
            if len(sub) == 0:
                continue
            sub = sub.sort_values(
                by=[f'dcm_{dcm_ct}_zscore', f'rna_{target_ct}_zscore'],
                ascending=[False, False]
            )
            for _, row in sub.iterrows():
                all_dcm_rows.append([row.get(f'dcm_{ct}_zscore', np.nan)
                                     for ct in label_order])
                all_rna_rows.append([row.get(f'rna_{ct}_zscore', np.nan)
                                     for ct in label_order])
                dcm_labels.append(dcm_ct)
                rna_labels.append(target_ct)
                multi_flags.append(bool(row.get(multipotent_col, False)))

            trans_info.append((dcm_ct, len(sub), pos))
            pos += len(sub)
            boundaries.append(pos)

        if not all_dcm_rows:
            print(f"  No data to plot")
            return

        dcm_matrix = np.array(all_dcm_rows).T
        rna_matrix = np.array(all_rna_rows).T
        n_cells    = dcm_matrix.shape[1]

        # ── Norms ────────────────────────────────────────────────
        norm_dcm = TwoSlopeNorm(vmin=vcenter_dcm - vrange,
                                vcenter=vcenter_dcm,
                                vmax=vcenter_dcm + vrange)
        norm_rna = TwoSlopeNorm(vmin=vcenter_rna - vrange,
                                vcenter=vcenter_rna,
                                vmax=vcenter_rna + vrange)

        fig_width = max(14, n_cells * 0.15)

        # ── GridSpec: 4 rows (ann_dcm | hm_dcm | ann_rna | hm_rna)
        #              × 2 cols (heatmap | colorbar) ────────────────
        fig = plt.figure(figsize=(fig_width, 14))
        gs  = fig.add_gridspec(
            4, 2,
            height_ratios=[0.04, 1, 0.04, 1],
            width_ratios=[1, 0.015],
            hspace=0.06,
            wspace=0.02,
        )

        ax_ann_dcm = fig.add_subplot(gs[0, 0])
        ax_dcm     = fig.add_subplot(gs[1, 0])
        ax_ann_rna = fig.add_subplot(gs[2, 0])
        ax_rna     = fig.add_subplot(gs[3, 0])
        ax_cb_dcm  = fig.add_subplot(gs[1, 1])
        ax_cb_rna  = fig.add_subplot(gs[3, 1])

        # ── Annotation bars ───────────────────────────────────────
        draw_ann_bar(ax_ann_dcm, dcm_labels, cell_type_colors,
                     boundaries, 'DCM type')
        draw_ann_bar(ax_ann_rna, rna_labels, cell_type_colors,
                     boundaries, 'RNA type')

        # ── Heatmap panels ────────────────────────────────────────
        for ax, ax_cb, matrix, norm, vcenter_val, panel_label, highlight_ct in [
            (ax_dcm, ax_cb_dcm, dcm_matrix, norm_dcm,
             vcenter_dcm, f'DCM Module Z-Scores → {target_ct}', None),
            (ax_rna, ax_cb_rna, rna_matrix, norm_rna,
             vcenter_rna, f'RNA Module Z-Scores (after → {target_ct})', target_ct),
        ]:
            im = ax.imshow(matrix, aspect='auto',
                           cmap=plt.cm.RdBu_r, norm=norm)

            ax.set_yticks(range(len(label_order)))
            ax.set_yticklabels(label_order, fontsize=10)
            ax.set_xticks([])
            ax.set_ylabel('Module', fontsize=11, fontweight='bold')
            ax.set_title(panel_label, fontsize=12, fontweight='bold')

            # Group dividers
            for b in boundaries[:-1]:
                ax.axvline(b - 0.5, color='white', lw=2)

            # Group labels below panel
            prev = 0
            for dcm_ct, n_c, start in trans_info:
                mid = start + n_c / 2
                ax.text(mid, -0.8, f'{dcm_ct}\n(n={n_c})',
                        ha='center', va='bottom', fontsize=9,
                        fontweight='bold',
                        color=cell_type_colors.get(dcm_ct, 'black'))
                prev = start + n_c

            # Highlight target row on RNA panel
            if highlight_ct is not None and highlight_ct in label_order:
                idx = label_order.index(highlight_ct)
                ax.axhline(idx - 0.5, color='black', lw=2,
                           linestyle='--', alpha=0.8)
                ax.axhline(idx + 0.5, color='black', lw=2,
                           linestyle='--', alpha=0.8)

            # ── Multipotent * markers ──────────────────────
            for cell_idx, is_multi in enumerate(multi_flags):
                if is_multi:
                    ax.text(cell_idx, -0.3, '*',
                            ha='center', va='bottom', fontsize=8,
                            fontweight='bold', color='#333333')

            cb = plt.colorbar(im, cax=ax_cb)
            cb.set_label('Z-Score', fontsize=9)
            cb.ax.axhline(y=vcenter_val, color='black', lw=1, ls='--')

        fig.suptitle(
            f'Z-Score Module Scores — Transitions to {target_ct}\n'
            f'(n={n_cells} cells | vcenter DCM={vcenter_dcm}, '
            f'RNA={vcenter_rna} | vrange=±{vrange})',
            fontsize=13, fontweight='bold', y=1.01,
        )
        fig.text(0.01, 0.005,
                 'Dashed lines on RNA panel = target module row  |  * = multipotent cell',
                 fontsize=7, color='#555555')
        plt.tight_layout()
        save_path = os.path.join(score_dir,
                                 f'heatmap_zscore_transitions_to_{target_ct}.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {save_path}")

    # ── Generate heatmaps ─────────────────────────────────────────
    for target in ['Stem', 'TA']:
        if target in connections_df['rna_cell_type'].unique():
            create_transition_to_target_heatmap(
                connections_df, target, label_order,
                cell_type_colors, score_dir,
                vcenter_dcm=VCENTER_DCM,
                vcenter_rna=VCENTER_RNA,
                vrange=VRANGE,
            )

    # ─────────────────────────────────────────────────────────────
    # SAVE SORTED CSV FOR TRANSITIONS (MATCHING HEATMAP ORDER)
    # ─────────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("SAVING SORTED TRANSITION CSV FILES")
    print("="*60)
    
    # Save sorted transitions for Stem
    if 'Stem' in connections_df['rna_cell_type'].unique():
        stem_subset = connections_df[connections_df['rna_cell_type'] == 'Stem'].copy()
        source_order = [ct for ct in label_order if ct not in ['Stem', 'TA'] and ct in stem_subset['dcm_cell_type'].unique()]
        
        sorted_stem_cells = []
        for dcm_ct in source_order:
            sub = stem_subset[stem_subset['dcm_cell_type'] == dcm_ct].copy()
            if len(sub) > 0:
                sub = sub.sort_values(by=[f'dcm_{dcm_ct}_zscore', f'rna_Stem_zscore'], 
                                      ascending=[False, False])
                sorted_stem_cells.append(sub)
        
        if sorted_stem_cells:
            sorted_stem_df = pd.concat(sorted_stem_cells, ignore_index=True)
            stem_csv = os.path.join(score_dir, 'transitions_to_Stem_sorted.csv')
            sorted_stem_df.to_csv(stem_csv, index=False)
            print(f"  Saved sorted Stem transitions (matches heatmap order): {stem_csv}")
            print(f"    Total cells: {len(sorted_stem_df)}")
    
    # Save sorted transitions for TA
    if 'TA' in connections_df['rna_cell_type'].unique():
        ta_subset = connections_df[connections_df['rna_cell_type'] == 'TA'].copy()
        source_order = [ct for ct in label_order if ct not in ['Stem', 'TA'] and ct in ta_subset['dcm_cell_type'].unique()]
        
        sorted_ta_cells = []
        for dcm_ct in source_order:
            sub = ta_subset[ta_subset['dcm_cell_type'] == dcm_ct].copy()
            if len(sub) > 0:
                sub = sub.sort_values(by=[f'dcm_{dcm_ct}_zscore', f'rna_TA_zscore'], 
                                      ascending=[False, False])
                sorted_ta_cells.append(sub)
        
        if sorted_ta_cells:
            sorted_ta_df = pd.concat(sorted_ta_cells, ignore_index=True)
            ta_csv = os.path.join(score_dir, 'transitions_to_TA_sorted.csv')
            sorted_ta_df.to_csv(ta_csv, index=False)
            print(f"  Saved sorted TA transitions (matches heatmap order): {ta_csv}")
            print(f"    Total cells: {len(sorted_ta_df)}")

    # ─────────────────────────────────────────────────────────────
    # STABLE HEATMAP FUNCTION
    # For Stem→Stem and TA→TA (cells that keep their label)
    # Same layout as create_transition_to_target_heatmap but
    # source == target, so both annotation bars show the same colour.
    # ─────────────────────────────────────────────────────────────
    def create_stable_heatmap(
        all_connections_df, stable_ct, label_order, cell_type_colors,
        score_dir,
        vcenter_dcm=VCENTER_DCM, vcenter_rna=VCENTER_RNA, vrange=VRANGE,
        multipotent_col='dcm_is_multipotent',
    ):
        """
        Heatmap for stable transitions: DCM == stable_ct AND RNA == stable_ct.
        Same 4-row layout with annotation bars and * for multipotent cells.
        """
        print(f"\n{'='*60}")
        print(f"STABLE HEATMAP — {stable_ct.upper()} → {stable_ct.upper()}")
        print(f"  vcenter_dcm={vcenter_dcm}  vcenter_rna={vcenter_rna}  vrange={vrange}")
        print(f"{'='*60}")

        subset = all_connections_df[
            (all_connections_df['dcm_cell_type'] == stable_ct) &
            (all_connections_df['rna_cell_type']  == stable_ct)
        ].copy()

        if len(subset) == 0:
            print(f"  No stable {stable_ct}→{stable_ct} cells found")
            return

        print(f"  Total stable cells: {len(subset)}")

        # Sort by self-DCM zscore descending
        dcm_self_col = f'dcm_{stable_ct}_zscore'
        if dcm_self_col in subset.columns:
            subset = subset.sort_values(dcm_self_col, ascending=False)
        subset = subset.reset_index(drop=True)

        dcm_matrix = np.array(
            [[row.get(f'dcm_{ct}_zscore', np.nan) for ct in label_order]
             for _, row in subset.iterrows()]
        ).T
        rna_matrix = np.array(
            [[row.get(f'rna_{ct}_zscore', np.nan) for ct in label_order]
             for _, row in subset.iterrows()]
        ).T

        dcm_labels  = [stable_ct] * len(subset)
        rna_labels  = [stable_ct] * len(subset)
        multi_flags = [bool(row.get(multipotent_col, False))
                       for _, row in subset.iterrows()]
        n_cells     = len(subset)
        boundaries  = []   # no group splits for stable

        norm_dcm = TwoSlopeNorm(vmin=vcenter_dcm - vrange,
                                vcenter=vcenter_dcm,
                                vmax=vcenter_dcm + vrange)
        norm_rna = TwoSlopeNorm(vmin=vcenter_rna - vrange,
                                vcenter=vcenter_rna,
                                vmax=vcenter_rna + vrange)

        fig_width = max(14, n_cells * 0.15)

        fig = plt.figure(figsize=(fig_width, 14))
        gs  = fig.add_gridspec(
            4, 2,
            height_ratios=[0.04, 1, 0.04, 1],
            width_ratios=[1, 0.015],
            hspace=0.06,
            wspace=0.02,
        )
        ax_ann_dcm = fig.add_subplot(gs[0, 0])
        ax_dcm     = fig.add_subplot(gs[1, 0])
        ax_ann_rna = fig.add_subplot(gs[2, 0])
        ax_rna     = fig.add_subplot(gs[3, 0])
        ax_cb_dcm  = fig.add_subplot(gs[1, 1])
        ax_cb_rna  = fig.add_subplot(gs[3, 1])

        draw_ann_bar(ax_ann_dcm, dcm_labels, cell_type_colors,
                     boundaries, 'DCM type')
        draw_ann_bar(ax_ann_rna, rna_labels, cell_type_colors,
                     boundaries, 'RNA type')

        for ax, ax_cb, matrix, norm, vcenter_val, panel_label in [
            (ax_dcm, ax_cb_dcm, dcm_matrix, norm_dcm,
             vcenter_dcm, f'DCM Module Z-Scores ({stable_ct}→{stable_ct})'),
            (ax_rna, ax_cb_rna, rna_matrix, norm_rna,
             vcenter_rna, f'RNA Module Z-Scores ({stable_ct}→{stable_ct})'),
        ]:
            im = ax.imshow(matrix, aspect='auto',
                           cmap=plt.cm.RdBu_r, norm=norm)
            ax.set_yticks(range(len(label_order)))
            ax.set_yticklabels(label_order, fontsize=10)
            ax.set_xticks([])
            ax.set_ylabel('Module', fontsize=11, fontweight='bold')
            ax.set_title(panel_label, fontsize=12, fontweight='bold')

            # Highlight self-module row
            if stable_ct in label_order:
                idx = label_order.index(stable_ct)
                ax.axhline(idx - 0.5, color='black', lw=2,
                           linestyle='--', alpha=0.8)
                ax.axhline(idx + 0.5, color='black', lw=2,
                           linestyle='--', alpha=0.8)

            # n label below
            ax.text(n_cells / 2, -0.8, f'{stable_ct}→{stable_ct}  (n={n_cells})',
                    ha='center', va='bottom', fontsize=9, fontweight='bold',
                    color=cell_type_colors.get(stable_ct, 'black'))

            # * markers for multipotent cells
            for cell_idx, is_multi in enumerate(multi_flags):
                if is_multi:
                    ax.text(cell_idx, -0.3, '*',
                            ha='center', va='bottom', fontsize=8,
                            fontweight='bold', color='#333333')

            cb = plt.colorbar(im, cax=ax_cb)
            cb.set_label('Z-Score', fontsize=9)
            cb.ax.axhline(y=vcenter_val, color='black', lw=1, ls='--')

        fig.suptitle(
            f'Z-Score Module Scores — Stable {stable_ct}→{stable_ct}\n'
            f'(n={n_cells} cells | vcenter DCM={vcenter_dcm}, '
            f'RNA={vcenter_rna} | vrange=±{vrange})',
            fontsize=13, fontweight='bold', y=1.01,
        )
        fig.text(0.01, 0.005,
                 'Dashed lines = self-module row  |  * = multipotent cell',
                 fontsize=7, color='#555555')
        plt.tight_layout()
        save_path = os.path.join(score_dir,
                                 f'heatmap_stable_{stable_ct}_to_{stable_ct}.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {save_path}")

    # ── Build all-connections df for stable lookup (no filter applied) ──
    all_conn_df = pd.DataFrame(connections)   # raw connections before filter

    # Z-score columns are not on all_conn_df yet — compute them
    _dcm_sc = [c for c in all_conn_df.columns if c.startswith('dcm_') and c.endswith('_score')]
    _rna_sc = [c for c in all_conn_df.columns if c.startswith('rna_') and c.endswith('_score')]
    for c in _dcm_sc:
        all_conn_df[c.replace('_score', '_zscore')] = (all_conn_df[c] - dcm_mean) / dcm_std
    for c in _rna_sc:
        all_conn_df[c.replace('_score', '_zscore')] = (all_conn_df[c] - rna_mean) / rna_std

    # ─────────────────────────────────────────────────────────────
    # SAVE SORTED STABLE CSV FILES (MATCHING INDIVIDUAL HEATMAPS)
    # ─────────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("SAVING SORTED STABLE CSV FILES")
    print("="*60)
    
    for stable_ct in ['Stem', 'TA']:
        stable_subset = all_conn_df[
            (all_conn_df['dcm_cell_type'] == stable_ct) &
            (all_conn_df['rna_cell_type'] == stable_ct)
        ].copy()
        
        if len(stable_subset) > 0:
            # Sort by self-DCM zscore descending (same as heatmap)
            dcm_self_col = f'dcm_{stable_ct}_zscore'
            if dcm_self_col in stable_subset.columns:
                stable_subset = stable_subset.sort_values(dcm_self_col, ascending=False)
            
            stable_csv = os.path.join(score_dir, f'stable_{stable_ct}_to_{stable_ct}_sorted.csv')
            stable_subset.to_csv(stable_csv, index=False)
            print(f"  Saved sorted stable {stable_ct} cells (matches heatmap order): {stable_csv}")
            print(f"    Total cells: {len(stable_subset)}")

    # ─────────────────────────────────────────────────────────────
    # COMBINED STABLE HEATMAP FUNCTION
    # Stem→Stem and TA→TA in a single figure, separated by a white
    # divider. Also returns the sorted cell order as a DataFrame
    # so it can be saved to CSV.
    # ─────────────────────────────────────────────────────────